# NeurIPS 2026 — Multi-Criteria Causal-Discovery Evaluation Pipeline

This notebook is the reproducibility artifact for the NeurIPS 2026 E&D Track paper *When Classical Inference Fails: A Multi-Criteria Diagnostic for Causal Discovery on Latent-Factor Panels*.

## What this notebook does

End-to-end pipeline reproducing every numerical claim in the paper:

- §3.2, Appendix K — null calibration (row-permuting + parametric AR(1))
- §4, Appendix O — synthetic validation across 13 regimes (R1, R1', R2, R2', R3, R3', R1-hetero, R1-bounded, R1-mixed, R1-heterogen, R1-vdemlike, R1-contaminated, R1-dynsplit)
- §5, Appendix P — real-data application (V-Dem-16 indices, V-Dem-60 indicators, WB-WPP demographics)
- §6.4, Appendix P — adversarial-robustness analysis
- Appendix M — signed-vs-unsigned ΔPR ablation
- Appendix Q — pre-application EDA

## Required setup

Designed for Google Colab with Drive mounted. Required directory structure under `DRIVE_DIR = /content/drive/MyDrive/NeurIPS2026_1296`:

```
data/
  my_data_75_years.csv      # V-Dem-16 indices (89 countries × 75 years × 16 indices)
  my_data_VDem_Ind.csv      # V-Dem-60 raw indicators (152 × 56 × 70)
  my_data_WB.csv            # WB-WPP demographic panel (173 × 56 × 20)
src/
  dynotears_panel.py        # DYNOTEARS panel implementation (vendored from causalnex)
  train_NAVAR.py            # NAVAR training (third-party)
  NAVAR.py                  # NAVAR architecture (third-party)
  dataloader.py             # NAVAR data loading (third-party)
  edge_ablation.py          # NAVAR edge-ablation DM tests (third-party)
  lsc_synthetic_validation.py   # Standalone version of Cell §1.1
```

External dependencies (pip-installed by the relevant cells):

- `tigramite` (for PCMCI)
- `factor_analyzer` (for some EDA checks)
- `torch`, `numpy`, `pandas`, `scipy`, `matplotlib` (standard scientific Python)

## How to run

Cells are self-contained per analysis. Most cells redefine the functions they need internally (PanelSpec, tvNAR fitter, CRPS scorer, method runners) so they can be run independently after mounting Drive. Cells that produce inputs for downstream cells are flagged in their annotations.

## Companion .py files

`src/` contains standalone .py versions of the canonical analysis scripts (`lsc_synthetic_validation.py`, `random_baselines_4method_corrected.py`, `adversary_recompute_snippet.py`). These are extracted from the corresponding cells and can be run as scripts. See `README.md`.

## Section 1 — §4 Synthetic validation

Reproduces the synthetic-validation results in §4 (paper) and Appendix O (full per-regime detail). The LSC synthetic validation cell defines all 13 regimes plus the 6 comparator methods plus the LSC machinery (rank-1 ALS, cov-rank-1, tetrad signature, components A/B/C, row-permuting null), and runs every (regime, seed, method) combination.

The OLS VAR substrate-diagnostic cell runs the simplest linear estimator across all regimes for the Layer-1 verdict (paper §4, Appendix O).

### Cell §1.1 — LSC synthetic validation: 13 regimes × 6 methods × 5 seeds

**Paper sections:** §4, Appendix O (synthetic validation: full results).

**Produces:** Table 1 (synthetic baseline) and the per-regime tables in Appendix O.

Self-contained cell: defines its own tvNAR fitter, CRPS scorer, all method runners (linear_var_granger, pcmci, cmlp, navar, dynotears, trivial_sparse), all 13 regime generators (R1, R1', R2, R2', R3, R3', R1-hetero, R1-bounded, R1-mixed, R1-heterogen, R1-vdemlike, R1-contaminated, R1-dynsplit), and the full LSC machinery.

**Inputs:** None (self-generates synthetic panels at U=139 units × T=75 timesteps each, matching V-Dem-16 dimensions).

**Outputs:** `synthetic_results.csv`, `synthetic_results_aggregated.csv`, summary figure.

**External deps:** `dynotears_panel.py`, `train_NAVAR.py`, `NAVAR.py`, `dataloader.py`, `edge_ablation.py` in `src/`. `tigramite` is pip-installed by §0.

**Runtime:** ~2 hours on Colab Pro (T4 GPU).

In [ ]:
# LSC Synthetic Validation — STANDALONE

import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

SRC_DIR = os.path.join(DRIVE_DIR, 'src')
OUT_DIR = os.path.join(DRIVE_DIR, 'lsc_synthetic_validation_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

sys.path = [p for p in sys.path if p != '/mnt/user-data/uploads']
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"SRC_DIR   = {SRC_DIR}  (exists: {os.path.isdir(SRC_DIR)})")
print(f"OUT_DIR   = {OUT_DIR}")

print("\nNAVAR codebase files:")
for f in ['NAVAR.py', 'dataloader.py', 'train_NAVAR.py', 'edge_ablation.py',
          'dynotears_panel.py']:
    p = os.path.join(SRC_DIR, f)
    print(f"  {f}: {'EXISTS' if os.path.exists(p) else 'MISSING'} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} B)")

# Import DYNOTEARS panel runner (vendored from causalnex).
try:
    from dynotears_panel import run_dynotears, fit_dynotears_panel
    print("dynotears_panel imported (run_dynotears, fit_dynotears_panel)")
except ImportError as e:
    raise ImportError(
        f"Could not import dynotears_panel from {SRC_DIR}. "
        f"Make sure dynotears_panel.py is in {SRC_DIR}/. ({e})"
    )


# Install tigramite (PCMCI)
try:
    import tigramite  # noqa: F401
    print("tigramite available")
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tigramite"])
    print("tigramite installed")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device={DEVICE}")


# §1. Global configuration


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, json, traceback, pickle
from itertools import combinations
from typing import Any, Dict, List, Optional, Tuple, Union
from dataclasses import dataclass

GLOBAL_SEED = 42
SEED = GLOBAL_SEED  # alias used by some legacy LSC functions

# Synthetic panel scale (matches V-Dem long_16var)
U_UNITS    = 139
T_PER_UNIT = 75

# LSC parameters
B_PERMS       = 100
ALS_MAX_ITERS = 50
ALS_TOL       = 1e-7
WEIGHT_R      = 1.0/3.0
WEIGHT_C      = 1.0/3.0
WEIGHT_F      = 1.0/3.0

N_SEEDS = 5
METHODS_TO_RUN = ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears', 'trivial_sparse']

# tvNAR / CRPS parameters (V-Dem rev 3)
H_HORIZON = 20
B_BOOT    = 300
GRID_SIZE = 50
RIDGE     = 1e-4
BANDWIDTH = 0.2
EPSILON   = 0.01

print(f"Synthetic panel: U={U_UNITS}, T_per_unit={T_PER_UNIT}, T_total={U_UNITS*T_PER_UNIT}")
print(f"Replication seeds: {N_SEEDS}")
print(f"Methods: {METHODS_TO_RUN}")
print(f"LSC permutations: B={B_PERMS}")


# §2. tvNAR fitter (kernel-weighted ridge on lag pairs, with optional projection)

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union

import numpy as np


ArrayLike = Union[np.ndarray]


def _as_2d_float(X: ArrayLike, name: str) -> np.ndarray:
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError(f"{name} must be 2D array-like, got shape {X.shape}")
    X = X.astype(float, copy=False)
    return X


def _split_lengths_to_slices(lengths: List[int]) -> List[slice]:
    if any(l <= 1 for l in lengths):
        raise ValueError("All series lengths must be >= 2 to form (t-1, t) pairs.")
    slices = []
    start = 0
    for L in lengths:
        slices.append(slice(start, start + L))
        start += L
    return slices


def _infer_split_lengths(
    T: int,
    split_timeseries: Union[bool, int, List[int], np.ndarray, Tuple[int, ...], None],
) -> Optional[List[int]]:
    """
    Normalize split_timeseries into a list of per-series lengths or None.

    - None/False: treat as a single series of length T
    - int: treat as equal-length series of this length (must divide T exactly)
    - list/array: treat as provided per-series lengths (must sum to T)
    """
    if split_timeseries is None or split_timeseries is False:
        return None
    if isinstance(split_timeseries, bool):
        # True without a length is ambiguous; treat as error to avoid silent leakage.
        raise ValueError("split_timeseries=True is ambiguous; provide int length or list of lengths.")
    if isinstance(split_timeseries, (int, np.integer)):
        L = int(split_timeseries)
        if L <= 1:
            raise ValueError("split_timeseries length must be >= 2.")
        if T % L != 0:
            raise ValueError(f"T={T} is not divisible by split_timeseries={L}.")
        return [L] * (T // L)
    # list-like
    lengths = [int(x) for x in list(split_timeseries)]
    if sum(lengths) != T:
        raise ValueError(f"split_timeseries lengths sum to {sum(lengths)} but T={T}.")
    return lengths


@dataclass
class PanelLagPairs:
    """
    Container for panel-safe (Y_{t-1}, Y_t) pairs.

    y_prev: (M, N)
    y_curr: (M, N)
    tau:    (M,)  within-series normalized time in [0, 1]
    series_id: (M,) integer id of originating series (country)
    """
    y_prev: np.ndarray
    y_curr: np.ndarray
    tau: np.ndarray
    series_id: np.ndarray


def build_panel_lag_pairs(
    Y: ArrayLike,
    split_timeseries: Union[bool, int, List[int], np.ndarray, Tuple[int, ...], None] = None,
    tau_mode: str = "within_series",
) -> PanelLagPairs:
    """
    Build (t-1, t) lag pairs from stacked panel data without crossing series boundaries.

    Parameters
    ----------
    Y : array (T, N)
        Stacked time series (e.g., country panels concatenated).
    split_timeseries : None/False, int, or list[int]
        - None/False: single series
        - int: equal-length series
        - list[int]: per-series lengths
    tau_mode : str
        - "within_series": tau = (k / (L-1)) for k=1..L-1 within each series
          (recommended for cross-country comparability when countries are "replicates")
        - "global_index": tau = global t / (T-1) (rarely what you want for panels)

    Returns
    -------
    PanelLagPairs
    """
    Y = _as_2d_float(Y, "Y")
    T, N = Y.shape
    lengths = _infer_split_lengths(T, split_timeseries)

    y_prev_list: List[np.ndarray] = []
    y_curr_list: List[np.ndarray] = []
    tau_list: List[np.ndarray] = []
    sid_list: List[np.ndarray] = []

    if lengths is None:
        # Single series
        y_prev = Y[:-1]
        y_curr = Y[1:]
        if tau_mode == "within_series":
            tau = np.arange(1, T) / (T - 1)
        elif tau_mode == "global_index":
            tau = np.arange(1, T) / (T - 1)
        else:
            raise ValueError("tau_mode must be 'within_series' or 'global_index'.")
        series_id = np.zeros(T - 1, dtype=int)
        return PanelLagPairs(y_prev=y_prev, y_curr=y_curr, tau=tau, series_id=series_id)

    slices = _split_lengths_to_slices(lengths)
    for sid, sl in enumerate(slices):
        Ys = Y[sl]  # (L, N)
        L = Ys.shape[0]
        y_prev = Ys[:-1]
        y_curr = Ys[1:]
        if tau_mode == "within_series":
            tau = np.arange(1, L) / (L - 1)
        elif tau_mode == "global_index":
            # Map local positions to global index range
            g_idx = np.arange(sl.start + 1, sl.stop)
            tau = g_idx / (T - 1)
        else:
            raise ValueError("tau_mode must be 'within_series' or 'global_index'.")
        y_prev_list.append(y_prev)
        y_curr_list.append(y_curr)
        tau_list.append(tau)
        sid_list.append(np.full(L - 1, sid, dtype=int))

    return PanelLagPairs(
        y_prev=np.vstack(y_prev_list),
        y_curr=np.vstack(y_curr_list),
        tau=np.concatenate(tau_list),
        series_id=np.concatenate(sid_list),
    )


def gaussian_kernel(u: np.ndarray) -> np.ndarray:
    #Gaussian kernel (unnormalized): exp(-0.5 u^2).
    return np.exp(-0.5 * (u ** 2))


def epanechnikov_kernel(u: np.ndarray) -> np.ndarray:
    #Epanechnikov kernel: 0.75*(1-u^2) for |u|<=1 else 0.
    w = np.zeros_like(u, dtype=float)
    m = np.abs(u) <= 1.0
    w[m] = 0.75 * (1.0 - u[m] ** 2)
    return w


def _get_kernel(name: str):
    name = name.lower()
    if name in ("gaussian", "normal"):
        return gaussian_kernel
    if name in ("epanechnikov", "epa"):
        return epanechnikov_kernel
    raise ValueError("kernel must be 'gaussian' or 'epanechnikov'.")


def row_normalize(W: np.ndarray, eps: float = 1e-12) -> np.ndarray:

    #Sign-preserving L1 row-normalize W (handles all-zero rows safely).

    W = np.asarray(W, dtype=float)
    rs = np.abs(W).sum(axis=1, keepdims=True)
    rs = np.where(rs < eps, 1.0, rs)
    return W / rs


def fit_tvNAR_lambda_paths(
    Y: ArrayLike,
    W: ArrayLike,
    split_timeseries: Union[bool, int, List[int], np.ndarray, Tuple[int, ...], None] = None,
    tau_grid: Optional[np.ndarray] = None,
    grid_size: int = 50,
    bandwidth: float = 0.15,
    kernel: str = "gaussian",
    ridge: float = 1e-4,
    tau_mode: str = "within_series",
    center: bool = False,
) -> Dict[str, np.ndarray]:
    """
    Estimate nodal influence trajectories lambda(tau) on a tau grid via kernel-weighted ridge regression.

    For each tau0 in tau_grid, solve:
        minimize_lambda  sum_m w_m(tau0) || y_curr[m] - (W+I) * (lambda ⊙ y_prev[m]) ||^2  + ridge*||lambda||^2
    """
    Y = _as_2d_float(Y, "Y")
    W = _as_2d_float(W, "W")
    T, N = Y.shape
    if W.shape != (N, N):
        raise ValueError(f"W must be shape (N,N) with N={N}, got {W.shape}")

    if center:
        Y = Y - Y.mean(axis=0, keepdims=True)

    pairs = build_panel_lag_pairs(Y, split_timeseries=split_timeseries, tau_mode=tau_mode)
    y_prev = pairs.y_prev  # (M,N)
    y_curr = pairs.y_curr  # (M,N)
    tau = pairs.tau        # (M,)
    M = y_prev.shape[0]
    if M == 0:
        raise ValueError("No usable (t-1,t) pairs. Check split_timeseries and data length.")

    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, int(grid_size))
    else:
        tau_grid = np.asarray(tau_grid, dtype=float).ravel()

    if not (0 < bandwidth <= 1.0):
        raise ValueError("bandwidth must be in (0, 1].")

    K = _get_kernel(kernel)
    WpI = W + np.eye(N, dtype=float)

    lambdas = np.zeros((tau_grid.size, N), dtype=float)

    # Weighted ridge at each tau0
    for g, tau0 in enumerate(tau_grid):
        u = (tau - tau0) / bandwidth
        w = K(u)

        # fallback if bandwidth too tight
        if w.sum() <= 1e-12:
            w = np.ones_like(w)

        A = np.zeros((N, N), dtype=float)
        b = np.zeros(N, dtype=float)

        for m in range(M):
            wm = w[m]
            if wm <= 0:
                continue
            # X_m = WpI * y_prev[m] (broadcast on columns)
            X = WpI * y_prev[m][None, :]  # (N,N)
            y = y_curr[m]                 # (N,)
            A += wm * (X.T @ X)
            b += wm * (X.T @ y)

        A += ridge * np.eye(N, dtype=float)

        try:
            lambdas[g] = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            lambdas[g] = np.linalg.lstsq(A, b, rcond=None)[0]

    return {
        "tau_grid": tau_grid,
        "lambda": lambdas,
        "W_plus_I": WpI,
        "meta": {
            "bandwidth": float(bandwidth),
            "kernel": kernel,
            "ridge": float(ridge),
            "tau_mode": tau_mode,
            "center": bool(center),
            "n_pairs": int(M),
        },
    }


#Adding code for flexible p paths for tvNAR



from typing import Any, Dict, Union, List, Tuple, Optional
import numpy as np


__all__ = [
    "fit_tvNAR_lambda_paths_constrained",
    "project_lambda_stable",
    "max_abs_lam",
]


# ---------------------------------------------------------------------------
# Low-level projection primitive
# ---------------------------------------------------------------------------

def _project_lambda_row(lam_row: np.ndarray, epsilon: float) -> Tuple[np.ndarray, bool]:
    """
    Global-rescale projection of a single lambda vector.

    If max(|lam_row|) > 1 - epsilon, rescale so max(|lam_row|) == 1 - epsilon.
    Otherwise, return unchanged.

    Returns (projected_lam, was_projected).
    """
    m = float(np.max(np.abs(lam_row)))
    boundary = 1.0 - epsilon
    if m > boundary:
        factor = boundary / m
        return lam_row * factor, True
    return lam_row, False


def project_lambda_stable(tv_result: Dict[str, Any], epsilon: float = 0.01) -> Dict[str, Any]:
    """
    Post-hoc stability projection of a tvNAR fit result.

    Parameters
    ----------
    tv_result : dict
        Output of fit_tvNAR_lambda_paths. Must contain a 'lambda' key
        holding an array of shape (grid_size, N).
    epsilon : float
        Numerical margin inside the unit boundary. Default 0.01 gives
        max(|lambda|) <= 0.99.

    Returns
    -------
    dict
        A *new* dict (tv_result is not mutated) with:
          - 'lambda' replaced by the projected array
          - 'lambda_raw' carrying the original pre-projection values
          - 'n_projected' number of grid rows that required rescaling
          - 'max_abs_lam_raw', 'max_abs_lam_projected' : scalar summaries
          - 'projection_epsilon' : the epsilon used
        All other keys from tv_result are copied through.
    """
    if "lambda" not in tv_result:
        raise KeyError("tv_result must contain 'lambda' key.")
    lam_raw = np.asarray(tv_result["lambda"], dtype=float)
    if lam_raw.ndim != 2:
        raise ValueError(f"lambda must be 2-D (grid_size, N); got shape {lam_raw.shape}.")
    if not 0.0 <= epsilon < 1.0:
        raise ValueError(f"epsilon must be in [0, 1); got {epsilon}.")

    lam_proj = np.empty_like(lam_raw)
    n_projected = 0
    for g in range(lam_raw.shape[0]):
        lam_g, was = _project_lambda_row(lam_raw[g], epsilon)
        lam_proj[g] = lam_g
        if was:
            n_projected += 1

    max_raw_per_row = np.max(np.abs(lam_raw), axis=1)
    max_proj_per_row = np.max(np.abs(lam_proj), axis=1)

    result = dict(tv_result)     # shallow copy; preserves all other keys
    result["lambda"] = lam_proj
    result["lambda_raw"] = lam_raw
    result["n_projected"] = int(n_projected)
    result["n_grid_rows"] = int(lam_raw.shape[0])
    result["max_abs_lam_raw"] = float(np.max(max_raw_per_row))
    result["max_abs_lam_projected"] = float(np.max(max_proj_per_row))
    result["projection_epsilon"] = float(epsilon)
    # Update meta if present
    if "meta" in result and isinstance(result["meta"], dict):
        result["meta"] = dict(result["meta"])
        result["meta"]["projected"] = True
        result["meta"]["projection_epsilon"] = float(epsilon)
    return result


# ---------------------------------------------------------------------------
# Top-level wrapper with constrain flag
# ---------------------------------------------------------------------------

def fit_tvNAR_lambda_paths_constrained(
    Y: np.ndarray,
    W: np.ndarray,
    *,
    constrain: bool = True,
    epsilon: float = 0.01,
    fit_fn=None,
    **kwargs,
) -> Dict[str, Any]:
    """
    tvNAR fit with optional stability projection.

    Thin wrapper around fit_tvNAR_lambda_paths. All additional kwargs are
    forwarded unchanged.

    Parameters
    ----------
    Y, W : see fit_tvNAR_lambda_paths
    constrain : bool
        If True, apply per-tau global-rescale projection to the fitted
        lambda array so max(|lambda|) <= 1 - epsilon at every grid row.
        If False, returns the unconstrained fit (same as calling
        fit_tvNAR_lambda_paths directly), with diagnostic fields still
        populated so downstream code can inspect stability.
    epsilon : float
        Numerical margin. Ignored when constrain=False.
    fit_fn : callable, optional
        The underlying ridge-solve function. Default: imported from
        tvNAR.fit_tvNAR_lambda_paths. Injecting fit_fn lets the wrapper
        be unit-tested without pulling the full tvNAR module.
    **kwargs :
        Forwarded to fit_fn (split_timeseries, tau_grid, grid_size,
        bandwidth, kernel, ridge, tau_mode, center).

    Returns
    -------
    dict
        Same keys as fit_tvNAR_lambda_paths plus (always):
          - 'constrain_requested' : bool
          - 'projection_epsilon' : float
        And if constrain=True, additionally:
          - 'lambda_raw', 'n_projected', 'n_grid_rows',
            'max_abs_lam_raw', 'max_abs_lam_projected'
        If constrain=False, the above are still populated (with
        n_projected=0 and lambda==lambda_raw) so downstream code can
        treat both cases uniformly.
    """
    if fit_fn is None:
        # Lazy import so this module is usable without tvNAR present
        # (e.g., in unit tests).
        # In the notebook, fit_tvNAR_lambda_paths is in globals()
        fit_fn = fit_tvNAR_lambda_paths

    tv_raw = fit_fn(Y=Y, W=W, **kwargs)

    # Always populate diagnostic fields so caller code is symmetric.
    if constrain:
        tv_out = project_lambda_stable(tv_raw, epsilon=epsilon)
    else:
        lam = np.asarray(tv_raw["lambda"], dtype=float)
        max_per_row = np.max(np.abs(lam), axis=1)
        tv_out = dict(tv_raw)
        tv_out["lambda_raw"] = lam.copy()
        tv_out["n_projected"] = 0
        tv_out["n_grid_rows"] = int(lam.shape[0])
        tv_out["max_abs_lam_raw"] = float(np.max(max_per_row))
        tv_out["max_abs_lam_projected"] = float(np.max(max_per_row))
        tv_out["projection_epsilon"] = float(epsilon)
        if "meta" in tv_out and isinstance(tv_out["meta"], dict):
            tv_out["meta"] = dict(tv_out["meta"])
            tv_out["meta"]["projected"] = False
            tv_out["meta"]["projection_epsilon"] = float(epsilon)

    tv_out["constrain_requested"] = bool(constrain)
    return tv_out


# ---------------------------------------------------------------------------
# Convenience diagnostic
# ---------------------------------------------------------------------------

def max_abs_lam(tv_result: Dict[str, Any]) -> float:
    """Return max(|lambda|) across the full tau grid."""
    lam = np.asarray(tv_result["lambda"], dtype=float)
    return float(np.max(np.abs(lam)))



# §3. CRPS scorer (one-step, residual-bootstrap)
#
# **Methodological note (uniform unsigned-magnitude convention):**
# The CRPS scorer treats `W_hat` as a matrix entering tvNAR's predictive mean
# `(W + I) · diag(λ) · y_{t-1}`. To make CRPS comparisons across methods
# apples-to-apples, we apply a UNIFORM unsigned-magnitude convention: every
# method's `W_hat` is non-negative.
#
#   * `pcmci`'s `W_hat` aggregates `|val_matrix|` (partial-correlation magnitudes).
#   * `cmlp`'s `W_hat` is the Frobenius norm of input-layer weight blocks
#     (strictly non-negative by construction).
#   * `navar`'s `W_hat` reports the SD over time of contributions
#     (strictly non-negative by construction).
#   * `linear_var_granger`'s `W_hat` is the absolute value of its OLS
#     coefficients after p-value thresholding, so it matches the convention
#     of the other three methods.
#
# When unsigned magnitudes are used as predictive coefficients, contributions
# from sources sum positively rather than canceling — biasing predictions
# toward the unconditional mean. The bias is UNIFORM across all four methods,
# so relative CRPS comparisons remain valid; absolute CRPS values are
# systematically inflated.
#
# A signed-coefficient extension (re-fitting OLS on the support selected by
# each method, or extracting average Jacobians from the trained NAVAR model)
# would remove this bias but is out of scope for this paper and is the focus
# of follow-up work.



from typing import Dict, Optional, Tuple, Any, List
import numpy as np


__all__ = [
    "crps_1d_from_samples",
    "mean_crps",
    "tvnar_mean_one_step",
    "score_W_against_panel",
]


# ---------------------------------------------------------------------------
# CRPS primitives
# ---------------------------------------------------------------------------

def crps_1d_from_samples(samples_1d: np.ndarray, y_true: float) -> float:
    """
    CRPS for empirical samples (sorted-sample closed form).

    CRPS(F, y) = E|X - y| - 0.5 * E|X - X'|
    where F is the empirical distribution of `samples_1d` and X, X' are iid ~ F.

    This is the standard sorted-sample formula; numerically identical to
    the integral form but O(n log n) instead of O(n^2).
    """
    x = np.sort(samples_1d.astype(float))
    Bn = x.size
    if Bn == 0:
        raise ValueError("CRPS requires at least one sample")
    term1 = float(np.mean(np.abs(x - y_true)))
    i = np.arange(1, Bn + 1, dtype=float)
    sum_abs = 2.0 * float(np.sum((2.0 * i - Bn - 1.0) * x))
    term2 = 0.5 * (sum_abs / (Bn * Bn))
    return term1 - term2


def mean_crps(y_true_vec: np.ndarray, samples_BN: np.ndarray) -> float:
    """
    Mean CRPS across N variables.

    Parameters
    ----------
    y_true_vec : (N,) observed values
    samples_BN : (B, N) predictive samples, one column per variable
    """
    y_true_vec = np.asarray(y_true_vec, dtype=float)
    samples_BN = np.asarray(samples_BN, dtype=float)
    if samples_BN.ndim != 2:
        raise ValueError(f"samples_BN must be 2D, got {samples_BN.shape}")
    if samples_BN.shape[1] != y_true_vec.size:
        raise ValueError(
            f"Column count mismatch: y_true has {y_true_vec.size} entries, "
            f"samples_BN has {samples_BN.shape[1]} columns"
        )
    return float(np.mean([
        crps_1d_from_samples(samples_BN[:, j], y_true_vec[j])
        for j in range(y_true_vec.size)
    ]))


# ---------------------------------------------------------------------------
# tvNAR one-step prediction
# ---------------------------------------------------------------------------

def tvnar_mean_one_step(W: np.ndarray, lam_vec: np.ndarray,
                        y_prev: np.ndarray) -> np.ndarray:
    """
    One-step mean prediction under tvNAR:

        y_hat = (W + I) * diag(lam_vec) * y_prev
              = (W + I broadcast-multiplied by y_prev across columns) @ lam_vec

    Parameters
    ----------
    W : (N, N) structural matrix (zero-diagonal convention)
    lam_vec : (N,) influence vector at a single tau
    y_prev : (N,) previous-step values

    Returns
    -------
    y_hat : (N,) predicted values
    """
    N = W.shape[0]
    W_plus_I = W + np.eye(N)
    return (W_plus_I * y_prev[None, :]) @ lam_vec


# ---------------------------------------------------------------------------
# Main scoring function
# ---------------------------------------------------------------------------

def score_W_against_panel(
    W: np.ndarray,
    Y: np.ndarray,
    split_spec: List[int],
    *,
    H: int = 10,
    B: int = 500,
    bandwidth: float = 0.2,
    ridge: float = 1e-4,
    grid_size: int = 30,
    kernel: str = "gaussian",
    lambda_policy: str = "last",
    constrain: bool = False,
    epsilon: float = 0.01,
    rng: Optional[np.random.Generator] = None,
    fit_fn: Optional[callable] = None,
) -> Dict[str, Any]:
    """
    Score a candidate W against panel Y using the two-level protocol.

    Fits tvNAR(Y, W) (optionally stability-constrained), then computes
    one-step CRPS on the last H points of Y using a pooled residual
    bootstrap.

    Parameters
    ----------
    W : (N, N) candidate structural matrix
    Y : (T_total, N) stacked panel
    split_spec : list of per-unit lengths
    H : number of final points to score CRPS on
    B : number of bootstrap samples per prediction
    bandwidth, ridge, grid_size, kernel : tvNAR hyperparameters
    lambda_policy : "last" or "mean" — which row of the lambda grid to use
    constrain : if True, apply stability projection to lambda
    epsilon : projection margin (used if constrain=True)
    rng : random generator for bootstrap sampling
    fit_fn : override for the tvNAR fit function (for testing)

    Returns
    -------
    dict with keys:
      status, CRPS, RMSE, MAE, MSE, nnz, max_abs_lam_raw,
      max_abs_lam_projected, n_projected, constrain_requested,
      protocol ("unconstrained" or "constrained"), epsilon
    """
    if rng is None:
        rng = np.random.default_rng(0)

    # Lazy import so this module is importable without tvNAR
    if fit_fn is None:
        import sys
        # In the notebook, fit_tvNAR_lambda_paths is in globals().
        fit_fn = fit_tvNAR_lambda_paths

    # fit_tvNAR_lambda_paths_constrained is in globals().

    T_panel, N = Y.shape
    if W.shape != (N, N):
        return {
            "status": f"skip: W shape {W.shape} != ({N}, {N})",
            "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
            "nnz": int((np.abs(W) > 1e-12).sum()),
            "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
            "n_projected": 0, "constrain_requested": bool(constrain),
            "protocol": "constrained" if constrain else "unconstrained",
            "epsilon": float(epsilon),
        }
    if np.allclose(W, 0.0):
        return {
            "status": "skip: all-zero W",
            "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
            "nnz": 0,
            "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
            "n_projected": 0, "constrain_requested": bool(constrain),
            "protocol": "constrained" if constrain else "unconstrained",
            "epsilon": float(epsilon),
        }

    # Fit tvNAR (with or without projection)
    try:
        tv = fit_tvNAR_lambda_paths_constrained(
            Y=Y, W=W,
            constrain=constrain, epsilon=epsilon,
            fit_fn=fit_fn,
            split_timeseries=split_spec,
            grid_size=grid_size, bandwidth=bandwidth,
            kernel=kernel, ridge=ridge,
            tau_mode="within_series", center=False,
        )
    except Exception as e:
        return {
            "status": f"fit_fail: {e}",
            "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
            "nnz": int((np.abs(W) > 1e-12).sum()),
            "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
            "n_projected": 0, "constrain_requested": bool(constrain),
            "protocol": "constrained" if constrain else "unconstrained",
            "epsilon": float(epsilon),
        }

    lam_grid = np.asarray(tv["lambda"], dtype=float)
    if lambda_policy == "last":
        lam_use = lam_grid[-1]
    elif lambda_policy == "mean":
        lam_use = lam_grid.mean(axis=0)
    else:
        raise ValueError(f"Unknown lambda_policy '{lambda_policy}'")

    # ------------------------------------------------------------------
    # Panel-safe CRPS evaluation.

    # Build panel-safe (t-1, t) lag pairs respecting unit boundaries.
    # `pairs.y_prev[m]`, `pairs.y_curr[m]`, `pairs.series_id[m]` are the
    # lag, current, and unit-id for the m-th pair.
    pairs = build_panel_lag_pairs(Y, split_timeseries=split_spec,
                                  tau_mode="within_series")
    M = pairs.y_prev.shape[0]
    if M == 0:
        return {
            "status": "skip: no valid lag pairs",
            "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
            "nnz": int((np.abs(W) > 1e-12).sum()),
            "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
            "n_projected": 0, "constrain_requested": bool(constrain),
            "protocol": "constrained" if constrain else "unconstrained",
            "epsilon": float(epsilon),
        }

    # Residual pool: residuals from all (t-1, t) WITHIN-UNIT pairs.
    resid_pool = np.zeros((M, N))
    for m in range(M):
        resid_pool[m] = pairs.y_curr[m] - tvnar_mean_one_step(W, lam_use,
                                                              pairs.y_prev[m])

    # Per-unit evaluation: score the last min(H, n_pairs_in_unit) pairs of
    # each unit; collect CRPS per pair; average across all collected pairs.
    crps_vals = []
    sq_errs = []
    abs_errs = []
    n_units_scored = 0
    for sid in np.unique(pairs.series_id):
        unit_mask = (pairs.series_id == sid)
        unit_pair_idxs = np.where(unit_mask)[0]
        if unit_pair_idxs.size == 0:
            continue
        # Last min(H, len) pairs of this unit
        h_eff = min(H, unit_pair_idxs.size)
        eval_idxs = unit_pair_idxs[-h_eff:]
        for m in eval_idxs:
            y_prev = pairs.y_prev[m]
            y_true = pairs.y_curr[m]
            mean_pred = tvnar_mean_one_step(W, lam_use, y_prev)
            # Bootstrap samples from the residual pool
            boot_idx = rng.integers(0, M, size=B)
            samples = mean_pred[None, :] + resid_pool[boot_idx]
            crps_vals.append(mean_crps(y_true, samples))
            sq_errs.append(np.mean((y_true - mean_pred) ** 2))
            abs_errs.append(np.mean(np.abs(y_true - mean_pred)))
        n_units_scored += 1

    if not crps_vals:
        return {
            "status": "skip: no eval pairs",
            "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
            "nnz": int((np.abs(W) > 1e-12).sum()),
            "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
            "n_projected": 0, "constrain_requested": bool(constrain),
            "protocol": "constrained" if constrain else "unconstrained",
            "epsilon": float(epsilon),
        }

    return {
        "status":                "ok",
        "CRPS":                  float(np.mean(crps_vals)),
        "RMSE":                  float(np.sqrt(np.mean(sq_errs))),
        "MAE":                   float(np.mean(abs_errs)),
        "MSE":                   float(np.mean(sq_errs)),
        "n_units_scored":        int(n_units_scored),
        "n_pairs_scored":        int(len(crps_vals)),
        "nnz":                   int((np.abs(W) > 1e-12).sum()),
        "max_abs_lam_raw":       float(tv["max_abs_lam_raw"]),
        "max_abs_lam_projected": float(tv["max_abs_lam_projected"]),
        "n_projected":           int(tv["n_projected"]),
        "constrain_requested":   bool(tv["constrain_requested"]),
        "protocol":              "constrained" if constrain else "unconstrained",
        "epsilon":               float(epsilon),
    }



# §4. Method runners
# All four causal-discovery methods. Defined fresh so signatures are guaranteed
# consistent between fit_* and run_*.


# ### §4.1 Linear VAR + Granger filtering



import time
from typing import Any, Dict, List, Optional

import numpy as np
from scipy import stats


def _unstack_panel(Y: np.ndarray, split_spec: List[int]) -> List[np.ndarray]:
    """Split a stacked panel (T_total, N) back into U per-unit arrays."""
    units = []
    cursor = 0
    for T_u in split_spec:
        units.append(Y[cursor:cursor + T_u])
        cursor += T_u
    return units


def fit_linear_var_granger(
    Y: np.ndarray,
    split_spec: List[int],
    *,
    alpha: float = 0.05,
    threshold_policy: str = "p_value",
) -> Dict[str, np.ndarray]:
    """
    Fit a pooled panel VAR(1): y_t = A y_{t-1} + ε_t.

    Per-variable OLS using all (t-1, t) pairs WITHIN each unit (never across
    unit boundaries — this is the panel-aware part).

    For each cross-coefficient A[i, j] (i ≠ j), compute a t-statistic and
    its two-sided p-value under the null that A[i, j] = 0. Threshold edges
    with p >= alpha to zero.

    Returns A_raw, A_thresh (p-value thresholded), F_stats (absolute
    t-values), p_values.
    """
    N = Y.shape[1]
    units = _unstack_panel(Y, split_spec)

    # Build pooled design: for each unit u and each t in 1..T_u-1, we have
    # one observation pair (Y_unit[t-1], Y_unit[t]). Concatenate across units.
    X_rows = []   # design matrix: Y_unit[t-1]
    Y_rows = []   # target: Y_unit[t]
    for unit in units:
        if len(unit) < 2:
            continue
        X_rows.append(unit[:-1])
        Y_rows.append(unit[1:])
    if not X_rows:
        raise ValueError("No valid (t-1, t) pairs across any unit.")
    X = np.concatenate(X_rows, axis=0)    # (n_obs, N)
    Yt = np.concatenate(Y_rows, axis=0)   # (n_obs, N)
    n_obs = X.shape[0]

    # Add an intercept column; useful for non-zero-mean panels
    X_design = np.concatenate([X, np.ones((n_obs, 1))], axis=1)  # (n_obs, N+1)

    # Per-variable OLS (one regression per target variable)
    A_raw = np.zeros((N, N))        # coefficient matrix, no intercept col
    F_stats = np.zeros((N, N))      # |t| for each coefficient
    p_values = np.ones((N, N))      # p-values; default 1.0 (non-significant)
    intercepts = np.zeros(N)

    # Use pinv for numerical stability on high-corr data
    XtX_inv = np.linalg.pinv(X_design.T @ X_design)
    for i in range(N):
        y_i = Yt[:, i]
        # Beta = (X'X)^-1 X'y
        beta = XtX_inv @ X_design.T @ y_i   # (N+1,)
        A_raw[i, :] = beta[:N]
        intercepts[i] = beta[N]
        # Residuals and sigma^2
        resid = y_i - X_design @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            # Underdetermined; can't compute standard errors
            continue
        sigma2 = float(resid @ resid) / dof
        # Covariance of beta: sigma^2 * (X'X)^-1
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        # t-statistics per coefficient
        t_stats = beta / se
        # Two-sided p-values
        p_vals = 2.0 * stats.t.sf(np.abs(t_stats), dof)
        F_stats[i, :] = np.abs(t_stats[:N])
        p_values[i, :] = p_vals[:N]

    # Thresholded W: zero out non-significant edges AND the diagonal.
    # We then take the ABSOLUTE VALUE so that lvg's W_hat uses the same
    # unsigned-magnitude convention as pcmci, cmlp, and navar. This makes
    # all four methods comparable when their W_hat is fed into the tvNAR
    # predictive equation as if it were a signed coefficient — the bias
    # is uniform across methods so relative CRPS comparisons remain valid.
    # See §5 setup of the paper for the full discussion.
    if threshold_policy == "p_value":
        A_thresh = A_raw.copy()
        A_thresh[p_values >= alpha] = 0.0
    elif threshold_policy == "none":
        A_thresh = A_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(A_thresh, 0.0)
    A_thresh = np.abs(A_thresh)   # uniform unsigned convention

    return {
        "A_raw": A_raw,
        "A_thresh": A_thresh,
        "F_stats": F_stats,
        "p_values": p_values,
        "intercepts": intercepts,
        "n_obs": n_obs,
        "dof": n_obs - (N + 1),
    }


def run_linear_var_granger(
    Y_train: np.ndarray,
    split_spec: List[int],
    *,
    variable_names: Optional[List[str]] = None,
    alpha: float = 0.05,
    seed: int = 0,
) -> Dict[str, Any]:
    """
    Thin wrapper producing the standard output contract.

    Args:
      Y_train: stacked panel (T_total, N)
      split_spec: per-unit time lengths
      variable_names: list of N names (optional; for metadata only)
      alpha: significance threshold for p-value-based edge selection
      seed: for consistency with other runners (OLS is deterministic, unused)

    Returns:
      Dict matching the standard runner output contract.
    """
    t0 = time.time()
    out = fit_linear_var_granger(Y_train, split_spec, alpha=alpha)
    elapsed = time.time() - t0

    N = Y_train.shape[1]
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    assert len(variable_names) == N

    return {
        "method": "linear_var_granger",
        "W_hat": out["A_thresh"],
        "W_hat_raw": out["A_raw"],
        "scores": out["F_stats"],
        "p_values": out["p_values"],
        "variable_names": list(variable_names),
        "method_config": {
            "alpha": alpha,
            "threshold_policy": "p_value",
            "include_intercept": True,
            "n_obs": int(out["n_obs"]),
            "dof": int(out["dof"]),
        },
        "training_time_sec": elapsed,
        "seed": seed,
    }


# ### §4.2 PCMCI



import time
from typing import Any, Dict, List, Optional

import numpy as np


def _build_unit_boundary_mask(split_spec: List[int], max_lag: int) -> np.ndarray:
    """
    Build a mask of shape (T_total,) that is True for valid (non-boundary)
    observations — i.e., observations for which all lagged predictors lie
    within the same unit.

    For unit u starting at cursor, the first max_lag observations cannot
    be used as targets (their lagged predictors would reach into the
    previous unit). Mask them out.
    """
    T_total = sum(split_spec)
    mask = np.ones(T_total, dtype=bool)
    cursor = 0
    for T_u in split_spec:
        # First max_lag observations of this unit: cannot be targets
        mask[cursor:cursor + max_lag] = False
        cursor += T_u
    return mask


def fit_pcmci(
    Y: np.ndarray,
    split_spec: List[int],
    *,
    tau_max: int = 1,
    pc_alpha: float = 0.2,
    alpha_level: float = 0.01,
    verbosity: int = 0,
) -> Dict[str, Any]:
    """
    Run PCMCI on a panel.

    Returns:
        val_matrix: (N, N, tau_max+1) tigramite's test statistic (partial
                    correlation magnitudes if ParCorr)
        p_matrix:   (N, N, tau_max+1) p-values
        W_raw:      (N, N) aggregated test statistic — max over tau=1..tau_max
        W_p:        (N, N) minimum p-value over tau=1..tau_max
    """
    try:
        from tigramite.pcmci import PCMCI
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite import data_processing as pp
    except ImportError:
        raise ImportError(
            "PCMCI runner requires tigramite. "
            "In a Colab cell: `!pip install tigramite`"
        )

    N = Y.shape[1]
    boundary_mask = _build_unit_boundary_mask(split_spec, max_lag=tau_max)

    # tigramite uses a DataFrame with optional missing_flag / mask
    # We use tigramite's DataFrame with a mask: positions where mask=True
    # are masked OUT from being used as a target.
    mask_2d = np.zeros_like(Y, dtype=int)
    mask_2d[~boundary_mask, :] = 1   # mask out boundary observations

    dataframe = pp.DataFrame(Y, mask=mask_2d)

    parcorr = ParCorr(mask_type="y")   # mask applies to target (y)
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr,
                  verbosity=verbosity)

    results = pcmci.run_pcmci(tau_max=tau_max, pc_alpha=pc_alpha,
                              alpha_level=alpha_level)
    # results["val_matrix"] shape: (N, N, tau_max+1)
    # results["p_matrix"] shape:   (N, N, tau_max+1)
    val = results["val_matrix"]
    p_mat = results["p_matrix"]

    # Aggregate over positive lags (tau=1..tau_max): strongest (highest |val|)
    # per (j->i) edge. Contemporaneous (tau=0) is excluded since tvNAR
    # uses only lag-1 dynamics.
    if tau_max < 1:
        raise ValueError("tau_max must be >= 1")
    # val.shape == (N, N, tau_max+1); tigramite's convention: val[i, j, tau]
    # is the association from Y_i at time (t-tau) to Y_j at time t, i.e.,
    # val[i, j, tau] asks: does past of i predict future of j?
    # So W[j_target, i_source] = max over tau>=1 of |val[i, j, tau]|.
    val_abs = np.abs(val[:, :, 1:tau_max + 1])   # (N, N, tau_max)
    W_raw_transposed = val_abs.max(axis=-1)      # (N, N) indexed [i, j] = i->j
    # Transpose to our convention: W[target, source]
    W_raw = W_raw_transposed.T   # (N, N) indexed [j, i] = i->j

    p_min_transposed = p_mat[:, :, 1:tau_max + 1].min(axis=-1)
    W_p = p_min_transposed.T     # (N, N)

    return {
        "W_raw": W_raw,
        "W_p": W_p,
        "val_matrix": val,
        "p_matrix": p_mat,
    }


def run_pcmci(
    Y_train: np.ndarray,
    split_spec: List[int],
    *,
    variable_names: Optional[List[str]] = None,
    tau_max: int = 1,
    # Runge et al. 2019 recommend pc_alpha relatively loose (0.1-0.2) for
    # variable selection in the PC-stage, with a stricter alpha_level for
    # final reporting. Tigramite's defaults are pc_alpha=0.2, alpha_level=0.05.
    # We tighten alpha_level to 0.01 to get sparser, more defensible W on
    # V-Dem. The previous setup (pc_alpha=0.05, alpha_level=tigramite default,
    # threshold_value=0.05) was conflating PC-stage and final-reporting
    # thresholds and producing dense W with marginal edges.
    pc_alpha: float = 0.2,
    alpha_level: float = 0.01,
    threshold_policy: str = "p_value",
    threshold_value: float = 0.01,   # default matches alpha_level
    seed: int = 0,
) -> Dict[str, Any]:
    """
    Run PCMCI and produce the standard runner output contract.

    threshold_policy:
      "p_value" (DEFAULT): keep edges with p < threshold_value (default 0.01,
                matching alpha_level). With alpha_level passed to PCMCI's
                final reporting stage, this is essentially redundant — we
                keep this layer for transparency.
      "absolute": keep edges with |val| > threshold_value
      "none"    : no thresholding (W_hat = W_raw with zero diagonal)
    """
    t0 = time.time()
    out = fit_pcmci(Y_train, split_spec, tau_max=tau_max,
                    pc_alpha=pc_alpha, alpha_level=alpha_level)
    elapsed = time.time() - t0

    N = Y_train.shape[1]
    W_raw = out["W_raw"]
    W_p = out["W_p"]
    scores = W_raw.copy()

    if threshold_policy == "p_value":
        W_hat = np.where(W_p < threshold_value, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)

    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    assert len(variable_names) == N

    return {
        "method": "pcmci",
        "W_hat": W_hat,
        "W_hat_raw": W_raw,
        "scores": scores,
        "p_values": W_p,
        "variable_names": list(variable_names),
        "method_config": {
            "tau_max": tau_max,
            "pc_alpha": pc_alpha,
            "alpha_level": alpha_level,
            "threshold_policy": threshold_policy,
            "threshold_value": threshold_value,
            "cond_ind_test": "ParCorr",
        },
        "training_time_sec": elapsed,
        "seed": seed,
    }


# ### §4.3 cMLP



import time
from typing import Any, Dict, List, Optional

import numpy as np

# Torch is imported lazily in fit_cmlp() so that the module can be
# IMPORTED on a CPU-only system without torch installed, as long as
# the user doesn't actually call run_method().


def _build_lagged_data(
    Y: np.ndarray,
    split_spec: List[int],
    lag: int,
) -> "tuple[np.ndarray, np.ndarray]":
    """
    Build (X, y) training data with the given number of lags.
    X shape: (n_samples, lag, N)
    y shape: (n_samples, N)

    Respects unit boundaries — samples only formed WITHIN each unit.
    """
    N = Y.shape[1]
    X_rows = []
    y_rows = []
    cursor = 0
    for T_u in split_spec:
        unit = Y[cursor:cursor + T_u]   # (T_u, N)
        cursor += T_u
        if T_u <= lag:
            continue
        for t in range(lag, T_u):
            X_rows.append(unit[t - lag:t])   # (lag, N)
            y_rows.append(unit[t])           # (N,)
    if not X_rows:
        raise ValueError(
            f"No samples could be built with lag={lag}. "
            f"All units have T <= lag."
        )
    X = np.stack(X_rows, axis=0)       # (n_samples, lag, N)
    y = np.stack(y_rows, axis=0)       # (n_samples, N)
    return X, y


def fit_cmlp(
    Y: np.ndarray,
    split_spec: List[int],
    *,
    lag: int = 8,
    hidden_dim: int = 10,
    n_epochs: int = 500,
    lr: float = 1e-3,
    lambda_group: float = 0.05,
    penalty: str = "hierarchical",
    device: str = "cpu",
    seed: int = 0,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Fit cMLP with a group-lasso-family penalty.

    penalty: "group_lasso" or "hierarchical"
      "group_lasso": standard Tank et al. GROUP penalty on input-layer weights.
                     Sums the Frobenius norm of each (input variable, all
                     lags) column block.
      "hierarchical" (DEFAULT): the HIER penalty from Tank et al. 2022,
                     which their own experiments show outperforms GROUP
                     across all K values (see their Table 3). Penalizes
                     nested subsets of lag weights — lag-k onward — for
                     each input variable, so later lags can only be
                     included if earlier lags are.

    For each target variable i, train an MLP:
        y_i(t) = f_i(Y(t-1), Y(t-2), ..., Y(t-lag))
    with the chosen penalty on the input-layer weights. The "group" for
    input variable j is the set of weights corresponding to all lags of Y_j.
    Weights zero out entire input variables.

    NOTE on optimization: we use Adam (not Tank et al.'s GISTA). GISTA
    guarantees convergence to a local minimum via proximal updates and
    produces exactly-zero weights for non-relevant inputs. Adam is faster
    but weights decay toward zero rather than becoming exactly zero, so
    we post-threshold. The paper's own README notes Adam "requires an
    additional thresholding parameter" — which is our `threshold_value`.
    This is a deliberate trade-off for simpler code and faster training.

    Returns a dict with:
        W_raw (N, N): Frobenius norm of input-layer weights per (j -> i)
        n_samples: int
    """
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        raise ImportError(
            "cMLP runner requires torch. "
            "In a Colab cell, run: !pip install torch"
        )

    N = Y.shape[1]
    X, yt = _build_lagged_data(Y, split_spec, lag)
    n_samples = X.shape[0]
    if verbose:
        print(f"[cMLP] N={N}, lag={lag}, n_samples={n_samples}")

    # Reshape X from (n_samples, lag, N) to (n_samples, lag*N) — flatten
    # lag-N pairs into an lag*N-dim vector. The first N entries are lag 1
    # (most recent), next N are lag 2, etc. This layout matters for the
    # group-lasso grouping below.
    X_flat = X.reshape(n_samples, lag * N)

    X_t = torch.tensor(X_flat, dtype=torch.float32, device=device)
    Y_t = torch.tensor(yt, dtype=torch.float32, device=device)

    # Component models: one MLP per target variable
    class ComponentMLP(nn.Module):
        def __init__(self, in_dim: int, hidden_dim: int):
            super().__init__()
            self.linear1 = nn.Linear(in_dim, hidden_dim)
            self.linear2 = nn.Linear(hidden_dim, 1)

        def forward(self, x):
            h = torch.relu(self.linear1(x))
            return self.linear2(h).squeeze(-1)

    # Group indices: for input variable j, the group of weight columns
    # in linear1 is [j, N+j, 2N+j, ...] (one column per lag).
    def group_indices(j):
        return [lag_idx * N + j for lag_idx in range(lag)]

    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Train one component MLP per target variable
    W_raw = np.zeros((N, N))
    losses_per_target = []

    for i in range(N):
        model = ComponentMLP(lag * N, hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        last_loss = None
        for epoch in range(n_epochs):
            optimizer.zero_grad()
            pred = model(X_t)
            mse = ((pred - Y_t[:, i]) ** 2).mean()

            # Structured penalty on input-layer weights.
            # W_input has shape (hidden_dim, lag*N) with column layout:
            #   columns [0..N-1]     = lag 1 (most recent)
            #   columns [N..2N-1]    = lag 2
            #   ...
            #   columns [(L-1)N..LN-1] = lag L
            # For input variable j, the "group" is [j, N+j, 2N+j, ...].
            W_input = model.linear1.weight
            if penalty == "group_lasso":
                # GROUP penalty: sum_j ||W_{:, group(j)}||_F
                group_norms = []
                for j in range(N):
                    idx = group_indices(j)
                    block = W_input[:, idx]
                    group_norms.append(torch.norm(block, p="fro"))
                pen = torch.stack(group_norms).sum()
            elif penalty == "hierarchical":
                # HIER penalty (Tank et al. eq 11):
                #   sum_j sum_{k=1..L} ||W_{:, group(j)[k-1:]}||_F
                # Nested: penalizes lag-k onward. Zeros out lag-k only if
                # all of lag-k, k+1, ..., L are zero.
                hier_norms = []
                for j in range(N):
                    idx = group_indices(j)   # [j, N+j, ..., (L-1)N+j]
                    for k in range(lag):
                        # Block is all weights for variable j from lag k+1 onward
                        sub_idx = idx[k:]
                        if len(sub_idx) == 0:
                            continue
                        block = W_input[:, sub_idx]
                        hier_norms.append(torch.norm(block, p="fro"))
                pen = torch.stack(hier_norms).sum()
            else:
                raise ValueError(
                    f"Unknown penalty '{penalty}'. Use 'group_lasso' or 'hierarchical'."
                )

            loss = mse + lambda_group * pen
            loss.backward()
            optimizer.step()
            last_loss = float(loss.item())
            if verbose and epoch % 100 == 0:
                print(f"[cMLP] target {i} epoch {epoch}: "
                      f"mse={float(mse):.4f}, pen={float(pen):.4f}")

        losses_per_target.append(last_loss)

        # Extract per-input-variable group norm as the Granger-score
        with torch.no_grad():
            W_in = model.linear1.weight.cpu().numpy()   # (hidden_dim, lag*N)
            for j in range(N):
                idx = group_indices(j)
                block = W_in[:, idx]   # (hidden_dim, lag)
                W_raw[i, j] = float(np.linalg.norm(block, ord="fro"))

    return {
        "W_raw": W_raw,
        "n_samples": n_samples,
        "losses_per_target": losses_per_target,
    }


def run_cmlp(
    Y_train: np.ndarray,
    split_spec: List[int],
    *,
    variable_names: Optional[List[str]] = None,
    # Hyperparameters sourced from Tank et al. 2022 "Neural Granger Causality"
    # and adapted for V-Dem-scale panel data:
    #   lag=8: matches our NAVAR wrapper (maxlags=8) for cross-method
    #     consistency. tvNAR downstream consumes only W, so lag mismatch with
    #     tvNAR's Markov-1 assumption does not matter. Tank et al. use K=5
    #     for VAR and K in {5,10,20} for Lorenz; 8 is in range.
    #   hidden_dim=10: Tank et al. Section 5 explicitly states "m = 10 hidden
    #     units for both methods" on their Lorenz-96 p=20, T=750 benchmark.
    #     V-Dem is p=16 with T=30-70 — less data means we need smaller
    #     capacity, not larger. Previous default of 64 was 6x too big.
    #   hidden_layers fixed at 1: "for all experiments we fix the number of
    #     hidden layers, L, to one" (Tank et al. 2022, Section 5).
    #   penalty="hierarchical": Tank et al. Table 3 shows HIER outperforms
    #     GROUP and MIXED across all tested K values. Default GROUP would
    #     underperform the paper's own published best setting.
    #   lambda_group=0.05: Tank et al. do not specify a single default —
    #     "Selecting the right regularization strength can be difficult
    #     and time consuming" (Neural-GC README). 0.05 is a reasonable
    #     starting point for our dataset scale; sweep {0.01, 0.05, 0.1,
    #     0.2} if the resulting W is too dense/sparse.
    #   n_epochs=500: the Neural-GC repo uses GISTA (proximal gradient);
    #     we use Adam, which converges faster per epoch. 500 should be
    #     sufficient; increase if loss is still decreasing.
    # These choices should be sanity-checked on V-Dem output density — if
    # the W matrix is all zero or fully dense, adjust lambda_group.
    #   lambda_group=0.1: bumped from 0.05. Tank et al. recommend
    #     "Selecting the right regularization strength can be difficult and
    #     time consuming" (Neural-GC README). 0.1 produces noticeably
    #     sparser W on V-Dem-scale data; sweep {0.05, 0.1, 0.2} if results
    #     are unsatisfactory.
    #   threshold_value=0.01 (relative): we emulate Tank et al.'s GISTA
    #     proximal-gradient, which drives weights for non-causal inputs
    #     exactly to zero. Adam (our optimizer) decays them to small but
    #     nonzero magnitudes. A 1% threshold relative to off-diagonal max
    #     approximates numerical-precision-zero. The previous 0.1 threshold
    #     was a default I picked without justification — too lax for this
    #     setup.
    lag: int = 8,
    hidden_dim: int = 10,
    n_epochs: int = 500,
    lr: float = 1e-3,
    lambda_group: float = 0.1,
    penalty: str = "hierarchical",
    threshold_policy: str = "relative",
    threshold_value: float = 0.01,
    device: str = "cpu",
    seed: int = 0,
) -> Dict[str, Any]:
    """
    Fit cMLP and produce the standard runner output contract.

    threshold_policy:
      "relative": keep edges whose score is above threshold_value * max(score)
      "absolute": keep edges whose score is above threshold_value
      "none"    : no thresholding (W_hat = W_raw with zero diagonal)
    """
    t0 = time.time()
    out = fit_cmlp(
        Y_train, split_spec,
        lag=lag, hidden_dim=hidden_dim, n_epochs=n_epochs, lr=lr,
        lambda_group=lambda_group, penalty=penalty,
        device=device, seed=seed,
    )
    elapsed = time.time() - t0

    N = Y_train.shape[1]
    W_raw = out["W_raw"]
    scores = W_raw.copy()

    if threshold_policy == "relative":
        # Compute threshold from OFF-DIAGONAL scores only.
        # The diagonal captures self-AR contributions which are typically
        # much larger than cross-variable Granger scores; using the full-
        # matrix max would dominate the cutoff and zero out all real
        # cross-variable edges.
        W_offdiag = W_raw.copy()
        np.fill_diagonal(W_offdiag, 0.0)
        cutoff = threshold_value * np.max(W_offdiag)
        W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)

    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    assert len(variable_names) == N

    return {
        "method": "cmlp",
        "W_hat": W_hat,
        "W_hat_raw": W_raw,
        "scores": scores,
        "variable_names": list(variable_names),
        "method_config": {
            "lag": lag,
            "hidden_dim": hidden_dim,
            "n_epochs": n_epochs,
            "lr": lr,
            "lambda_group": lambda_group,
            "penalty": penalty,
            "threshold_policy": threshold_policy,
            "threshold_value": threshold_value,
            "n_samples": int(out["n_samples"]),
        },
        "training_time_sec": elapsed,
        "seed": seed,
    }



# ### §4.4 NAVAR



import time
from typing import Any, Dict, List, Optional

import numpy as np


def _to_split_arg(split_spec: List[int]) -> "int | List[int]":
    """
    Convert our split_spec (always a list) to train_NAVAR's split_timeseries
    argument, which accepts int (uniform) or list.
    """
    # train_NAVAR interprets: int T_u → U units of length T_u.
    # But we pass a list for safety; their dataloader handles both.
    if len(set(split_spec)) == 1:
        # All units equal length — can use int or list. Use int for
        # backward-compatibility with older train_NAVAR versions.
        return int(split_spec[0])
    return list(split_spec)


def run_navar(
    Y_train: np.ndarray,
    split_spec: List[int],
    *,
    variable_names: Optional[List[str]] = None,
    # Hyperparameters below default to values used in the prior V-Dem
    # pipeline ((legacy preprocessing pipeline), (legacy preprocessing)). These are NOT the
    # library's train_NAVAR defaults — those were calibrated for large
    # synthetic benchmarks. V-Dem-calibrated values are:
    #   hidden_nodes=32  (not 256): V-Dem has N=16 and modest samples;
    #     256 hidden units would massively over-parameterize. 32 is what
    #     the working V-Dem pipeline uses and what produces sensible W.
    #   maxlags=8  (not 1): V-Dem is yearly panel data; 8-year lookback
    #     captures meaningful political-dynamics windows. The tvNAR
    #     downstream scorer consumes only the W matrix, not NAVAR's
    #     internal lag structure, so maxlags does not need to match
    #     tvNAR's lag-1 Markov assumption.
    #   lambda1=0.15  (not 0.0): sparsity penalty on contributions.
    #     Without this, high-capacity NAVAR overfits and produces dense
    #     contributions; with this, edges with weak Granger evidence are
    #     shrunk toward zero.
    #   learning_rate=3e-4  (not 1e-4): converges faster at lower capacity.
    # If these differ from what your prior work used, override them
    # explicitly in the call.
    maxlags: int = 8,
    hidden_nodes: int = 32,
    hidden_layers: int = 1,
    dropout: float = 0.0,
    n_epochs: int = 500,
    lr: float = 3e-4,
    batch_size: int = 300,
    lambda1: float = 0.15,
    weight_decay: float = 0.0,
    threshold_policy: str = "dm_significance",   # was "relative"; DM is the
                                                  # principled prior-pipeline choice
    threshold_value: float = 0.1,                 # only used by relative/absolute/top_k
    use_dm_ablation: bool = True,                 # was False; enable by default
    dm_alpha: float = 0.05,
    dm_val_proportion: float = 0.10,
    seed: int = 0,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Train NAVAR and produce the standard runner output contract.

    threshold_policy:
      "dm_significance" (DEFAULT): use DM (Diebold-Mariano) edge-ablation
                         significance testing. For each candidate edge i->j,
                         ablate the contribution f_ij in NAVAR's additive
                         decomposition and run a one-sided DM test on whether
                         ablation degrades held-out predictions. Edges with
                         p < dm_alpha are kept. This is the principled
                         threshold from the prior V-Dem NAVAR pipeline.
                         Requires use_dm_ablation=True (default) and
                         edge_ablation.py importable.
      "relative"       : keep edges with score >= threshold_value * max(score)
      "absolute"       : keep edges with score >= threshold_value
      "top_k"          : keep top-k edges; threshold_value interpreted as int k
      "none"           : no thresholding (W_hat = W_raw with zero diagonal)
    """
    try:
        import torch
    except ImportError:
        raise ImportError(
            "NAVAR runner requires torch. In a Colab cell, run: !pip install torch"
        )

    # Import train_NAVAR lazily so the module can be imported without
    # the NAVAR codebase being on the PYTHONPATH.
    #
    # Two-stage import: try direct import first; if that fails, fall back
    # to looking for `SRC_DIR` in the global namespace (set by the Colab
    # setup cell). Colab kernels sometimes restart between cells, wiping
    # sys.path modifications, so we re-add SRC_DIR defensively here.
    try:
        from train_NAVAR import train_NAVAR
    except ImportError as e:
        # Fallback 1: SRC_DIR from globals (set by setup cell)
        import sys
        src_dir = globals().get("SRC_DIR", None)
        if src_dir is not None and os.path.isdir(src_dir) and src_dir not in sys.path:
            sys.path.insert(0, src_dir)
            print(f"[NAVAR] Re-added {src_dir} to sys.path; retrying import...")
        # Fallback 2: try common Colab Drive locations
        for candidate in ["/content/drive/MyDrive/NeurIPS2026_1296/src",
                          "/content"]:
            if os.path.isdir(candidate) and candidate not in sys.path:
                sys.path.insert(0, candidate)
        try:
            from train_NAVAR import train_NAVAR
        except ImportError as e2:
            # Both attempts failed; emit a really clear error
            sp_str = "\n  ".join(sys.path[:10])
            raise ImportError(
                f"NAVAR runner could not find train_NAVAR.py.\n"
                f"  Original error: {e}\n"
                f"  Retry error:    {e2}\n"
                f"  sys.path (first 10 entries):\n  {sp_str}\n"
                f"  Globals SRC_DIR: {src_dir!r}\n"
                f"  Fix: ensure train_NAVAR.py is in a directory on sys.path. "
                f"In Colab: SRC_DIR='/content/drive/MyDrive/NeurIPS2026_1296/src' "
                f"and `import sys; sys.path.insert(0, SRC_DIR)`."
            )

    N = Y_train.shape[1]
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    assert len(variable_names) == N

    # Seed for determinism
    torch.manual_seed(seed)
    np.random.seed(seed)

    t0 = time.time()
    # train_NAVAR returns (causal_matrix, contributions, loss_val).
    # causal_matrix has the convention causal_matrix[i, j] = score for i -> j
    # (i.e., source->target).
    out = train_NAVAR(
        Y_train,
        maxlags=maxlags,
        hidden_nodes=hidden_nodes,
        hidden_layers=hidden_layers,
        dropout=dropout,
        epochs=n_epochs,
        learning_rate=lr,
        batch_size=batch_size,
        lambda1=lambda1,
        weight_decay=weight_decay,
        split_timeseries=_to_split_arg(split_spec),
        val_proportion=0.0,
        normalize=False,  # we've already standardized in M0
        check_every=10**9 if not verbose else max(1, n_epochs // 10),
    )
    # train_NAVAR returns (causal_matrix, contributions, val_loss, model).
    # causal_matrix[i, j] = score for i -> j (source->target convention).
    # Older versions returned only 3 elements; handle both.
    if isinstance(out, tuple):
        scores_src_tgt = out[0]
        navar_model = out[3] if len(out) >= 4 else None
    else:
        scores_src_tgt = out
        navar_model = None
    elapsed = time.time() - t0

    # Convert to our convention: W[target, source] so that W_hat[i, j]
    # is the score from j -> i (matches other runners).
    scores = np.asarray(scores_src_tgt).T.copy()
    scores = np.abs(scores)   # magnitudes only for thresholding

    # Threshold
    W_raw = scores.copy()
    dm_pvals = None    # only populated for threshold_policy='dm_significance'
    if threshold_policy == "relative":
        # Use OFF-DIAGONAL max so the threshold isn't dominated by self-AR
        # contributions on the diagonal (which get zeroed out anyway).
        W_offdiag = W_raw.copy()
        np.fill_diagonal(W_offdiag, 0.0)
        cutoff = threshold_value * np.max(W_offdiag)
        W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "top_k":
        k = int(threshold_value)
        flat = W_raw.copy()
        np.fill_diagonal(flat, -np.inf)   # exclude diagonal from top-k
        if k >= flat.size:
            W_hat = W_raw.copy()
        else:
            cutoff = np.partition(flat.flatten(), -k)[-k]
            W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "dm_significance":
        if not use_dm_ablation:
            raise ValueError(
                "threshold_policy='dm_significance' requires use_dm_ablation=True"
            )
        if navar_model is None:
            raise RuntimeError(
                "DM ablation requires the trained NAVAR model, but train_NAVAR "
                "returned only %d items (expected 4). Update train_NAVAR to "
                "return (causal_matrix, contributions, val_loss, model)."
                % len(out)
            )
        W_hat, dm_pvals = _apply_dm_ablation(
            model=navar_model,
            data=Y_train,
            variable_names=variable_names,
            split_spec=split_spec,
            scores_W=W_raw,
            dm_alpha=dm_alpha,
            maxlags=maxlags,
            val_proportion=dm_val_proportion,
            verbose=verbose,
        )
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)

    return {
        "method": "navar",
        "W_hat": W_hat,
        "W_hat_raw": W_raw,
        "scores": scores,
        "dm_pvals": dm_pvals,    # (N, N) p-values [target, source]; None if DM not used
        "variable_names": list(variable_names),
        "method_config": {
            "maxlags": maxlags,
            "hidden_nodes": hidden_nodes,
            "hidden_layers": hidden_layers,
            "dropout": dropout,
            "n_epochs": n_epochs,
            "lr": lr,
            "batch_size": batch_size,
            "lambda1": lambda1,
            "weight_decay": weight_decay,
            "threshold_policy": threshold_policy,
            "threshold_value": threshold_value,
            "use_dm_ablation": use_dm_ablation,
            "dm_alpha": dm_alpha if use_dm_ablation else None,
        },
        "training_time_sec": elapsed,
        "seed": seed,
    }


def _apply_dm_ablation(model, data, variable_names, split_spec, scores_W, *,
                       dm_alpha, maxlags, val_proportion, verbose):
    """
    Apply DM (Diebold-Mariano) edge-ablation significance testing to the
    trained NAVAR model. Edges with p < dm_alpha are kept (in W_hat with
    raw score magnitude); others are zeroed.

    Returns:
      W_hat   : (N, N) thresholded matrix in [target, source] orientation
      pvals_W : (N, N) p-values in [target, source] orientation, NaN for
                edges DM didn't test (e.g., diagonal)

    Conventions:
      edge_ablation_dm_all_pairs returns pvals_df with rows=source, cols=target.
      Our W convention is W[target, source]. We transpose accordingly.
    """
    try:
        from edge_ablation import edge_ablation_dm_all_pairs
    except ImportError:
        raise ImportError(
            "DM-significance thresholding requires edge_ablation.py from "
            "the prior pipeline to be on PYTHONPATH (e.g., in SRC_DIR). "
            "Either upload it or use threshold_policy='relative'."
        )

    pvals_df, dm_df, diff_df = edge_ablation_dm_all_pairs(
        model=model,
        data=data,
        variable_names=list(variable_names),
        maxlags=maxlags,
        split_timeseries=split_spec,
        val_proportion=val_proportion,
        normalize_for_eval=True,
        batch_size=512,
        include_diagonal=False,
        verbose=verbose,
    )
    # pvals_df: rows=source, cols=target.  Transpose to [target, source].
    pvals_W = pvals_df.to_numpy().T   # (N, N) [target, source]

    # Keep edges with p < dm_alpha; their value in W_hat is the raw NAVAR score
    # magnitude. NaN p-values (from the diagonal) are treated as not significant.
    keep_mask = (pvals_W < dm_alpha) & np.isfinite(pvals_W)
    W_hat = np.where(keep_mask, scores_W, 0.0)

    if verbose:
        n_kept = int(keep_mask.sum())
        n_total = int((pvals_W < 2.0).sum())   # all non-diagonal entries
        print(f"  [DM] {n_kept}/{n_total} edges significant at alpha={dm_alpha}")

    return W_hat, pvals_W



# §5. Trivial sparse baseline + dispatcher
#
# The trivial sparse baseline is NOT a competing method. It's a diagnostic
# stress-test: if its CRPS is competitive with real methods, that demonstrates
# CRPS can be achieved by aggressive shrinkage rather than by structure recovery.
# Should produce ΔPR ~ 0.


def fit_trivial_sparse(Y: np.ndarray, split_spec: List[int],
                       density: float = 0.05, weight_scale: float = 0.005,
                       seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    N = Y.shape[1]
    W = np.zeros((N, N))
    n_cells = N * (N - 1)
    n_nonzero = max(2, int(n_cells * density))
    off_diag_idx = [(i, j) for i in range(N) for j in range(N) if i != j]
    chosen = rng.choice(len(off_diag_idx), size=n_nonzero, replace=False)
    for c in chosen:
        i, j = off_diag_idx[c]
        W[i, j] = rng.uniform(-weight_scale, weight_scale)
    return W


def run_method(method_name: str, Y: np.ndarray, split_spec: List[int],
               variable_names: List[str], seed: int = 0,
               regime_name: Optional[str] = None) -> Dict:
    """Dispatcher. Returns {'W_hat', 'elapsed', 'error'}.
    Each runner is called with only universal arguments (Y, split_spec,
    variable_names, seed) so this works regardless of which optional kwargs
    each runner has.

    For DYNOTEARS, regime_name is required so the dispatcher can look up the
    per-regime tuned lambda_a from the global DYNOTEARS_LAMBDA_PER_REGIME dict
    (built in §8.5 below). All other methods ignore regime_name.
    """
    t0 = time.time()
    try:
        if method_name == 'linear_var_granger':
            out = run_linear_var_granger(Y, split_spec,
                                          variable_names=variable_names, seed=seed)
            W = out['W_hat']
        elif method_name == 'pcmci':
            out = run_pcmci(Y, split_spec, variable_names=variable_names, seed=seed)
            W = out['W_hat']
        elif method_name == 'cmlp':
            out = run_cmlp(Y, split_spec, variable_names=variable_names,
                           device=DEVICE, seed=seed)
            W = out['W_hat']
        elif method_name == 'navar':
            out = run_navar(Y, split_spec, variable_names=variable_names,
                            seed=seed, verbose=False)
            W = out['W_hat']
        elif method_name == 'dynotears':
            if regime_name is None:
                raise ValueError("dynotears requires regime_name (for per-regime "
                                 "lambda_a lookup); the dispatcher was called "
                                 "without it.")
            try:
                lambda_a = DYNOTEARS_LAMBDA_PER_REGIME[regime_name]
            except (NameError, KeyError):
                raise RuntimeError(
                    f"DYNOTEARS_LAMBDA_PER_REGIME missing entry for "
                    f"regime '{regime_name}'. Did §8.5 (per-regime tuning) run?"
                )
            out = run_dynotears(Y, split_spec, variable_names=variable_names,
                                lambda_a=lambda_a, seed=seed)
            W = out['W_hat']
        elif method_name == 'trivial_sparse':
            W = fit_trivial_sparse(Y, split_spec, seed=seed)
        else:
            raise ValueError(f"Unknown method {method_name}")
        return {'W_hat': W, 'elapsed': time.time() - t0, 'error': None}
    except Exception as e:
        return {'W_hat': None, 'elapsed': time.time() - t0,
                'error': f"{type(e).__name__}: {str(e)[:300]}"}


# §6. LSC machinery
#
# Off-diagonal rank-1 ALS, cov-rank-1 signature, tetrad signature, components
# (PR, RC, FC), LSC composite, row-permuting null. Same machinery as
# `lsc_diagnostic_vdem.py`.


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7, return_uv=False, seed=0):
    """Fit u, v minimizing sum_{i != j} (W_ij - u_i v_j)^2. Return explained-variance ratio.

    Parameters
    ----------
    W_block : (m, m) ndarray. Diagonal is set to 0 internally; only off-diagonal cells matter.
    max_iters : ALS iteration cap.
    tol : convergence tolerance on relative residual decrease.
    return_uv : if True, return (R, u, v) instead of just R.
    seed : random seed for fallback initialization.

    Returns
    -------
    R : float in [0, 1]. R=1 means off-diagonal cells exactly equal u_i v_j; R=0 means no rank-1 structure.
        If the off-diagonal sum-of-squares is 0, returns NaN (zero block, excluded from averaging).
    """
    W = W_block.astype(float).copy()
    m = W.shape[0]
    assert W.shape == (m, m)

    # Mask: 1 off-diagonal, 0 on diagonal
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return (np.nan, None, None) if return_uv else np.nan

    # Initialize u, v from SVD of diagonal-zeroed W (biased but reasonable starting point)
    W_zeroed = W.copy()
    np.fill_diagonal(W_zeroed, 0.0)
    try:
        U, S, Vt = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U[:, 0] * np.sqrt(S[0])
        v = Vt[0, :] * np.sqrt(S[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(seed)
        u = rng.standard_normal(m)
        v = rng.standard_normal(m)

    prev_res = np.inf
    for it in range(max_iters):
        # Update u given v: for each i, u_i = sum_{j != i} v_j W_ij / sum_{j != i} v_j^2
        # Vectorized: numerator[i] = sum_j (mask[i,j] * v_j * W_ij)
        #             denominator[i] = sum_j (mask[i,j] * v_j^2)
        num_u = (W * mask) @ v  # (m,)
        v2 = v * v
        den_u = mask @ v2  # (m,)
        # Guard zero denominators
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)

        # Update v given u_new
        num_v = (W * mask).T @ u_new
        u2 = u_new * u_new
        den_v = mask.T @ u2
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)

        u, v = u_new, v_new

        # Check residual on off-diagonal
        rank1 = np.outer(u, v)
        residual = float(np.sum(((W - rank1)[mask]) ** 2))

        if prev_res < np.inf:
            rel_decrease = abs(prev_res - residual) / max(prev_res, 1e-12)
            if rel_decrease < tol:
                break
        prev_res = residual

    R = max(0.0, 1.0 - residual / tss)
    R = min(R, 1.0)  # numerical safety
    if return_uv:
        return R, u, v
    return R




def sample_tetrad(cov, a, b, c, d):
    """tau_{abcd} = Cov(a,c) Cov(b,d) - Cov(a,d) Cov(b,c)"""
    return cov[a, c] * cov[b, d] - cov[a, d] * cov[b, c]


def cov_rank1_signature(panel_data, indices):
    """Test 1-factor reflective structure DIRECTLY via off-diagonal rank-1 fit on the indicator
    correlation matrix.

    Under 1-factor reflective measurement (linear, independent measurement errors), the population
    correlation matrix on the indicators of group C has off-diagonal entries
        Sigma_{ij} = lambda_i * lambda_j     (for i != j in C)
    after standardization, so the off-diagonal cells are rank-one. We fit a rank-one model to
    these off-diagonal cells using the same ALS solver as Component A of LSC, and return the
    explained-variance ratio R^cov_C in [0, 1].

    This is a stronger characterization of 1-factor reflective measurement than vanishing tetrads:
    tetrad vanishing is necessary but not sufficient (multi-factor models can also produce
    vanishing tetrads in particular subsets). Rank-1 covariance is essentially the defining
    property of 1-factor reflective measurement.

    Returns
    -------
    T_C : float in [0, 1] — fraction of off-diagonal correlation variance explained by rank-1 fit.
          High (>0.9) => strongly consistent with 1-factor reflective.
          Low (<0.5)  => clearly NOT 1-factor reflective.
    F_C : 1 - T_C
    """
    if len(indices) < 4:
        return np.nan, np.nan
    sub = panel_data[:, indices]
    sub_std = (sub - sub.mean(0)) / (sub.std(0) + 1e-12)
    corr = np.corrcoef(sub_std, rowvar=False)
    # Use the same off-diagonal ALS we use for Component A
    R = offdiag_rank1_fit(corr.copy(), max_iters=ALS_MAX_ITERS, tol=ALS_TOL)
    if np.isnan(R):
        return np.nan, np.nan
    return float(R), float(1.0 - R)


def tetrad_signature(panel_data, indices, z_cutoff=2.0, n_bootstrap=200, seed=0):
    """Compute T(C), F(C) for a candidate group C with |C| >= 4 via the standardized tetrad
    magnitude test.

    A tetrad is declared "vanishing" if |tau_hat / SD(tau_hat)| < z_cutoff, where SD is estimated
    by panel-bootstrap. The z_cutoff parameter is NOT a Type-I error level; it's a tolerance for
    "approximate vanishing." Larger z_cutoff is more permissive.

    NOTE: vanishing tetrads are NECESSARY for 1-factor reflective measurement but not SUFFICIENT
    (multi-factor models with the right correlation structure can also produce vanishing tetrads).
    This test is therefore best used alongside cov_rank1_signature() which directly checks rank-1.
    """
    from itertools import combinations
    if len(indices) < 4:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    T_total = panel_data.shape[0]
    sub = panel_data[:, indices]
    cov_full = np.cov(sub, rowvar=False, bias=True)
    quartets = list(combinations(range(len(indices)), 4))
    tau_pts = np.array([sample_tetrad(cov_full, a, b, c, d) for (a, b, c, d) in quartets])

    tau_boots = np.zeros((n_bootstrap, len(quartets)))
    for b in range(n_bootstrap):
        idx_resample = rng.integers(0, T_total, T_total)
        sub_b = sub[idx_resample]
        cov_b = np.cov(sub_b, rowvar=False, bias=True)
        for q, (a_, b_, c_, d_) in enumerate(quartets):
            tau_boots[b, q] = sample_tetrad(cov_b, a_, b_, c_, d_)

    sds = np.std(tau_boots, axis=0, ddof=1)
    sds = np.maximum(sds, 1e-12)
    z_scores = np.abs(tau_pts) / sds

    n_vanish = int(np.sum(z_scores < z_cutoff))
    T_C = n_vanish / len(quartets)
    return float(T_C), float(1.0 - T_C)


def cross_block_matrix(W, group_l_indices, group_k_indices):
    """Build E_{l -> k}[i, j] = W[j, i] for i in G_l (source), j in G_k (target).

    Returns matrix of shape (|G_l|, |G_k|).
    """
    # E[a, b] = W[group_k[b], group_l[a]]
    return W[np.ix_(group_k_indices, group_l_indices)].T


def component_PR(W, groups, min_size=3, restrict_to_reflective=True, return_per_group=False):
    """Within-group rank-1 coherence (off-diagonal ALS fit).

    Parameters
    ----------
    restrict_to_reflective : if True, average only over groups labeled 'reflective'
                             (this is the LSC metric proper as defined in the appendix).
                             If False, average over every group of size >= min_size with a
                             nonzero off-diagonal block, ignoring labels. This is the
                             "PR diagnostic" — useful when the data don't classify any group
                             as strictly reflective but we still want to inspect rank-1 structure.

    Returns
    -------
    PR : float or NaN
    per_group : list of dicts (if return_per_group=True), one per scored group
    """
    if restrict_to_reflective:
        eligible = [g for g in groups if g['type'] == 'reflective' and g['size'] >= min_size]
    else:
        eligible = [g for g in groups if g['size'] >= min_size]

    if not eligible:
        return (np.nan, []) if return_per_group else np.nan

    Rs = []
    per_group = []
    for g in eligible:
        idx = g['indices']
        W_block = W[np.ix_(idx, idx)].copy()
        # Zero diagonal (W_ii := 0 by tvNAR convention)
        np.fill_diagonal(W_block, 0.0)
        R = offdiag_rank1_fit(W_block, max_iters=ALS_MAX_ITERS, tol=ALS_TOL)
        if not np.isnan(R):
            Rs.append(R)
        per_group.append({
            'name': g['names'][:3] + ['...'] if len(g['names']) > 3 else g['names'],
            'size': g['size'], 'type': g['type'], 'T': g['T'], 'R_kk': R,
        })

    if not Rs:
        return (np.nan, per_group) if return_per_group else np.nan

    PR = float(np.mean(Rs))
    return (PR, per_group) if return_per_group else PR


def component_RC(W, groups, min_size=2, return_per_pair=False):
    """Cross-reflective-group rank-1 coherence.

    Returns average R_{l -> k} = sigma_1^2 / ||E||_F^2 over reflective-reflective cross-block pairs.
    """
    refl = [g for g in groups if g['type'] == 'reflective' and g['size'] >= min_size]
    if len(refl) < 2:
        return (np.nan, []) if return_per_pair else np.nan

    Rs = []
    per_pair = []
    for i, g_l in enumerate(refl):
        for j, g_k in enumerate(refl):
            if i == j:
                continue
            E = cross_block_matrix(W, g_l['indices'], g_k['indices'])
            fro2 = float(np.sum(E ** 2))
            if fro2 <= 0.0:
                continue
            try:
                s = np.linalg.svd(E, compute_uv=False)
                R = float(s[0] ** 2 / fro2)
            except np.linalg.LinAlgError:
                continue
            Rs.append(R)
            per_pair.append({'source': g_l['names'][:2] + ['...'] if len(g_l['names']) > 2 else g_l['names'],
                             'target': g_k['names'][:2] + ['...'] if len(g_k['names']) > 2 else g_k['names'],
                             'R': R})
    if not Rs:
        return (np.nan, per_pair) if return_per_pair else np.nan
    return (float(np.mean(Rs)), per_pair) if return_per_pair else float(np.mean(Rs))


def component_FC(W, groups, min_size=2, return_per_pair=False):
    """Formative-source heterogeneity: 1 - R_{l -> k} for l formative."""
    form = [g for g in groups if g['type'] == 'formative' and g['size'] >= min_size]
    if not form:
        return (np.nan, []) if return_per_pair else np.nan
    other = [g for g in groups if g['size'] >= min_size]
    if len(other) < 2:
        return (np.nan, []) if return_per_pair else np.nan

    Rs = []
    per_pair = []
    for g_l in form:
        for g_k in other:
            if g_l is g_k:
                continue
            E = cross_block_matrix(W, g_l['indices'], g_k['indices'])
            fro2 = float(np.sum(E ** 2))
            if fro2 <= 0.0:
                continue
            try:
                s = np.linalg.svd(E, compute_uv=False)
                R = float(s[0] ** 2 / fro2)
            except np.linalg.LinAlgError:
                continue
            FH = 1.0 - R
            Rs.append(FH)
            per_pair.append({'source': g_l['names'], 'target': g_k['names'], 'FH': FH})
    if not Rs:
        return (np.nan, per_pair) if return_per_pair else np.nan
    return (float(np.mean(Rs)), per_pair) if return_per_pair else float(np.mean(Rs))


def compute_LSC(W, groups, w_R=WEIGHT_R, w_C=WEIGHT_C, w_F=WEIGHT_F, return_components=False):
    """Combined LSC score with weight renormalization for absent components."""
    PR = component_PR(W, groups)
    RC = component_RC(W, groups)
    FC = component_FC(W, groups)

    weights = []
    values = []
    if not np.isnan(PR):
        weights.append(w_R); values.append(PR)
    if not np.isnan(RC):
        weights.append(w_C); values.append(RC)
    if not np.isnan(FC):
        weights.append(w_F); values.append(FC)

    if not weights:
        return (np.nan, {'PR': PR, 'RC': RC, 'FC': FC, 'weights_used': None}) if return_components else np.nan

    # Renormalize so weights sum to 1
    weights = np.array(weights) / sum(weights)
    LSC = float(np.dot(weights, values))

    if return_components:
        return LSC, {'PR': PR, 'RC': RC, 'FC': FC, 'weights_used': weights.tolist()}
    return LSC


# 7. Row-permuting null
#
# For each row $i$ independently, shuffle the off-diagonal column positions. This preserves the multiset of
# incoming-edge magnitudes per target (since row $i$ contains incoming edges to target $i$ under the
# convention $W_{ij}$ = source $j$ → target $i$) while randomizing which sources contribute to which targets.

def row_permute(W, rng):
    """Row-permuting null: for each row, shuffle off-diagonal column positions independently."""
    N = W.shape[0]
    W_perm = np.zeros_like(W)
    for i in range(N):
        # Off-diagonal positions for this row
        offdiag_cols = [j for j in range(N) if j != i]
        permuted_cols = rng.permutation(offdiag_cols)
        for src_j, tgt_j in zip(offdiag_cols, permuted_cols):
            W_perm[i, tgt_j] = W[i, src_j]
        # Diagonal stays 0
    return W_perm


def lsc_with_null(W, groups, B=B_PERMS, seed=SEED):
    """Compute LSC + PR_diagnostic and their row-permuting null distributions.

    PR_diagnostic differs from PR (LSC component A) in one respect: PR averages R_kk only over
    groups labeled 'reflective', while PR_diagnostic averages R_kk over EVERY group of size >= 3
    regardless of label. PR_diagnostic is therefore always defined when at least one group has
    size >= 3, while LSC may be NaN when no group passes the reflective threshold.
    """
    # LSC proper (uses group labels)
    LSC_obs, comps_obs = compute_LSC(W, groups, return_components=True)

    # PR_diagnostic (unconditional, always computed when a group of size >= 3 exists)
    PR_diag, _ = component_PR(W, groups, restrict_to_reflective=False, return_per_group=True)

    rng = np.random.default_rng(seed)
    null_LSCs = []
    null_PRs = []
    null_RCs = []
    null_FCs = []
    null_PR_diags = []
    for b in range(B):
        W_b = row_permute(W, rng)
        LSC_b, comps_b = compute_LSC(W_b, groups, return_components=True)
        PR_diag_b = component_PR(W_b, groups, restrict_to_reflective=False)
        null_LSCs.append(LSC_b)
        null_PRs.append(comps_b['PR'])
        null_RCs.append(comps_b['RC'])
        null_FCs.append(comps_b['FC'])
        null_PR_diags.append(PR_diag_b)

    null_LSCs = np.array(null_LSCs)
    null_PR_diags = np.array(null_PR_diags)

    def _z_p(obs, null_arr):
        finite = null_arr[np.isfinite(null_arr)]
        if len(finite) == 0 or np.isnan(obs):
            return np.nan, np.nan, np.nan, np.nan
        mu = float(np.mean(finite)); sd = float(np.std(finite))
        z = (obs - mu) / max(sd, 1e-12)
        # right-tailed empirical p
        p = (1 + np.sum(finite >= obs)) / (1 + len(finite))
        return mu, sd, float(z), float(p)

    lsc_mu, lsc_sd, z_lsc, p_lsc = _z_p(LSC_obs, null_LSCs)
    prd_mu, prd_sd, z_prd, p_prd = _z_p(PR_diag, null_PR_diags)
    pr_mu,  pr_sd,  z_pr,  p_pr  = _z_p(comps_obs['PR'], np.array(null_PRs))
    rc_mu,  rc_sd,  z_rc,  p_rc  = _z_p(comps_obs['RC'], np.array(null_RCs))
    fc_mu,  fc_sd,  z_fc,  p_fc  = _z_p(comps_obs['FC'], np.array(null_FCs))

    # Excess metrics (the headline numbers): observed minus null mean.
    # Sparsity-driven floors get subtracted out, so different methods become directly comparable.
    def _delta(obs, mu):
        return (obs - mu) if (np.isfinite(obs) and np.isfinite(mu)) else np.nan
    delta_LSC     = _delta(LSC_obs,           lsc_mu)
    delta_PR_diag = _delta(PR_diag,           prd_mu)
    delta_PR      = _delta(comps_obs['PR'],   pr_mu)
    delta_RC      = _delta(comps_obs['RC'],   rc_mu)
    delta_FC      = _delta(comps_obs['FC'],   fc_mu)

    return {
        'LSC': LSC_obs,
        'PR': comps_obs['PR'],
        'RC': comps_obs['RC'],
        'FC': comps_obs['FC'],
        'PR_diagnostic': PR_diag,
        # Excess-over-null (the headline metrics)
        'delta_LSC':     float(delta_LSC) if np.isfinite(delta_LSC) else np.nan,
        'delta_PR_diag': float(delta_PR_diag) if np.isfinite(delta_PR_diag) else np.nan,
        'delta_PR':      float(delta_PR) if np.isfinite(delta_PR) else np.nan,
        'delta_RC':      float(delta_RC) if np.isfinite(delta_RC) else np.nan,
        'delta_FC':      float(delta_FC) if np.isfinite(delta_FC) else np.nan,
        # Empirical p-values
        'p_LSC':         p_lsc,
        'p_PR_diag':     p_prd,
        'p_PR':          p_pr,
        'p_RC':          p_rc,
        'p_FC':          p_fc,
        # Null distributions and z-scores (descriptive)
        'null_LSC_mean': lsc_mu,    'null_LSC_sd': lsc_sd,    'z_LSC': z_lsc,
        'null_PR_diag_mean': prd_mu, 'null_PR_diag_sd': prd_sd, 'z_PR_diag': z_prd,
        'null_PR_mean':  pr_mu,     'null_PR_sd':  pr_sd,     'z_PR': z_pr,
        'null_RC_mean':  rc_mu,     'null_RC_sd':  rc_sd,     'z_RC': z_rc,
        'null_FC_mean':  fc_mu,     'null_FC_sd':  fc_sd,     'z_FC': z_fc,
        # Raw null arrays for downstream plotting
        'null_LSCs':     null_LSCs,
        'null_PRs':      np.array(null_PRs),
        'null_RCs':      np.array(null_RCs),
        'null_FCs':      np.array(null_FCs),
        'null_PR_diags': null_PR_diags,
    }





# §7. Synthetic DGPs
#
# Four regimes. R1 and R2 use **additive-tanh nonlinear latent dynamics** matching
# NAVAR's structural form. R3 and R3' are linear (formative / independent).


def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def generate_R1(seed, U=U_UNITS, T=T_PER_UNIT,
                rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                idiosyncratic_sd=0.3, burn_in=50):
    """R1: 1-factor reflective with tanh latent dynamics. Linear measurement.
    DGP: η_{t+1} = ρ tanh(s η_t) + ζ;  y_i = λ_i η + ε."""
    rng = np.random.default_rng(seed)
    N = 12  # was 6 — increased to match SEM identifiability conventions
    Lam = rng.uniform(0.6, 0.9, size=N)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            path[t] = Lam * eta + rng.normal(0, idiosyncratic_sd, size=N)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': 'R1',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd, 'nonlinear': True}}


def generate_R2(seed, U=U_UNITS, T=T_PER_UNIT,
                a_self=0.85, a_cross=0.10, tanh_scale=2.0,
                sigma_zeta=0.6, idiosyncratic_sd=0.3, burn_in=50):
    """R2: 2-factor reflective with NONLINEAR (additive-tanh) cross-block coupling."""
    rng = np.random.default_rng(seed)
    N = 16  # was 8 — increased to match SEM identifiability conventions (2 blocks × 8)
    block1 = list(range(8)); block2 = list(range(8, 16))
    Lam = rng.uniform(0.6, 0.9, size=N)
    block_of = np.array([0]*8 + [1]*8)
    unit_paths = []
    for u in range(U):
        eta = np.zeros(2)
        for _ in range(burn_in):
            n = np.zeros(2)
            n[0] = (a_self * np.tanh(tanh_scale * eta[0]) +
                    a_cross * np.tanh(tanh_scale * eta[1]) +
                    rng.normal(0, sigma_zeta))
            n[1] = (a_self * np.tanh(tanh_scale * eta[1]) +
                    a_cross * np.tanh(tanh_scale * eta[0]) +
                    rng.normal(0, sigma_zeta))
            eta = n
        path = np.zeros((T, N))
        for t in range(T):
            n = np.zeros(2)
            n[0] = (a_self * np.tanh(tanh_scale * eta[0]) +
                    a_cross * np.tanh(tanh_scale * eta[1]) +
                    rng.normal(0, sigma_zeta))
            n[1] = (a_self * np.tanh(tanh_scale * eta[1]) +
                    a_cross * np.tanh(tanh_scale * eta[0]) +
                    rng.normal(0, sigma_zeta))
            eta = n
            for i in range(N):
                path[t, i] = Lam[i] * eta[block_of[i]] + rng.normal(0, idiosyncratic_sd)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R2_v{i}' for i in range(N)]
    groups_true = [
        {'indices': block1, 'names': [var_names[i] for i in block1],
         'type': 'reflective', 'T': 1.0, 'F': 0.0,
         'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': 8},
        {'indices': block2, 'names': [var_names[i] for i in block2],
         'type': 'reflective', 'T': 1.0, 'F': 0.0,
         'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': 8},
    ]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': 'R2',
            'config_extras': {'a_self': a_self, 'a_cross': a_cross,
                              'tanh_scale': tanh_scale,
                              'sigma_zeta': sigma_zeta,
                              'idiosyncratic_sd': idiosyncratic_sd, 'nonlinear': True}}


def generate_R3(seed, U=U_UNITS, T=T_PER_UNIT, rho_innov=0.5, decay=1.0):
    """R3: covariance-formative. Independent AR(1) with banded innovation correlation."""
    rng = np.random.default_rng(seed)
    N = 12  # was 6 — increased to match SEM identifiability conventions
    rhos = rng.uniform(0.7, 0.95, size=N)
    Sigma_eps = np.eye(N)
    if rho_innov > 0:
        for i in range(N):
            for j in range(N):
                if i != j:
                    Sigma_eps[i, j] = rho_innov ** (decay * abs(i - j))
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-9:
        Sigma_eps = 0.99 * Sigma_eps + 0.01 * np.eye(N)
    L = np.linalg.cholesky(Sigma_eps)
    unit_paths = []
    for u in range(U):
        y = np.zeros(N)
        path = np.zeros((T, N))
        for t in range(T):
            eps = L @ rng.standard_normal(N)
            y = rhos * y + eps
            path[t] = y
        unit_paths.append(path)
    Y = np.vstack(unit_paths); split_spec = [T] * U
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R3_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'formative', 'T': 0.0, 'F': 1.0,
        'T_tetrad': 0.0, 'F_tetrad': 1.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)),
            'groups_true': groups_true, 'regime': 'R3',
            'config_extras': {'rho_innov': rho_innov, 'decay': decay,
                              'rhos': rhos.tolist(), 'nonlinear': False}}


def generate_R3prime(seed, U=U_UNITS, T=T_PER_UNIT):
    """R3': Independent indicators (rho_innov=0). Sanity check."""
    res = generate_R3(seed, U, T, rho_innov=0.0)
    res['regime'] = "R3'"
    return res


def generate_R1prime(seed, U=U_UNITS, T=T_PER_UNIT,
                     rho=0.95, sigma_zeta=0.6, alpha_cubic=0.05,
                     idiosyncratic_sd=0.3, burn_in=50):
    """R1': 1-factor reflective with CUBIC latent dynamics (form-robustness for R1).
    DGP: η_{t+1} = ρ·g(η_t) + ζ;  g(x) = x − α x³;  y_i = λ_i η + ε.

    The cubic g(x) = x - α x³ is monotone-saturating like tanh but has a
    polynomial functional form rather than hyperbolic-tangent. If method
    rankings are the same as on R1, the within-block rank-1 finding is
    form-robust within the single-factor reflective family. If rankings
    change, the R1 result is form-specific to tanh.
    """
    rng = np.random.default_rng(seed)
    N = 12  # was 6 — increased to match SEM identifiability conventions
    Lam = rng.uniform(0.6, 0.9, size=N)
    unit_paths = []
    # Bound η to a stable range. With α=0.05, |g(x)| < |x| once |x| > sqrt(20)≈4.5;
    # combined with ρ=0.95 and σ_ζ=0.6, η stays bounded in practice. We add a
    # soft clip at ±5 as a safety net to prevent rare numerical blowups.
    clip = 5.0
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            g_eta = eta - alpha_cubic * (eta ** 3)
            eta = rho * g_eta + rng.normal(0, sigma_zeta)
            eta = max(-clip, min(clip, eta))
        path = np.zeros((T, N))
        for t in range(T):
            g_eta = eta - alpha_cubic * (eta ** 3)
            eta = rho * g_eta + rng.normal(0, sigma_zeta)
            eta = max(-clip, min(clip, eta))
            path[t] = Lam * eta + rng.normal(0, idiosyncratic_sd, size=N)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1p_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': "R1'",
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'alpha_cubic': alpha_cubic,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'nonlinear': True, 'form': 'cubic'}}


def generate_R2prime(seed, U=U_UNITS, T=T_PER_UNIT,
                     a_self=0.85, a_cross=0.10, sigma_zeta=0.6,
                     idiosyncratic_sd=0.3, burn_in=50):
    """R2': 2-factor reflective with MULTIPLICATIVE (tanh-stabilized) cross-coupling.
    DGP for latents: η_{1,t+1} = a_self·η_{1,t} + a_cross·tanh(η_{1,t}·η_{2,t}) + ζ_1;
                     η_{2,t+1} = a_self·η_{2,t} + a_cross·tanh(η_{2,t}·η_{1,t}) + ζ_2.
    Linear measurement: y_i = λ_i·η_{block(i)} + ε.

    The cross-coupling here is η_1·η_2 (multiplicative) rather than R2's
    tanh(η_1) + tanh(η_2) (additive). NAVAR's parametric form is additive
    across sources: f_{i,j}(η_j) does not natively capture η_i·η_j
    interactions. So R2' tests whether NAVAR's R2 advantage requires the
    additive form, or generalizes to other nonlinear couplings.

    The outer tanh on the multiplicative term keeps the dynamics bounded.
    Without it the multiplicative cross-coupling can blow up.
    """
    rng = np.random.default_rng(seed)
    N = 16  # was 8 — increased to match SEM identifiability conventions (2 blocks × 8)
    block1 = list(range(8)); block2 = list(range(8, 16))
    Lam = rng.uniform(0.6, 0.9, size=N)
    block_of = np.array([0]*8 + [1]*8)
    unit_paths = []
    for u in range(U):
        eta = np.zeros(2)
        for _ in range(burn_in):
            n = np.zeros(2)
            cross_term = np.tanh(eta[0] * eta[1])
            n[0] = a_self * eta[0] + a_cross * cross_term + rng.normal(0, sigma_zeta)
            n[1] = a_self * eta[1] + a_cross * cross_term + rng.normal(0, sigma_zeta)
            eta = n
        path = np.zeros((T, N))
        for t in range(T):
            n = np.zeros(2)
            cross_term = np.tanh(eta[0] * eta[1])
            n[0] = a_self * eta[0] + a_cross * cross_term + rng.normal(0, sigma_zeta)
            n[1] = a_self * eta[1] + a_cross * cross_term + rng.normal(0, sigma_zeta)
            eta = n
            for i in range(N):
                path[t, i] = Lam[i] * eta[block_of[i]] + rng.normal(0, idiosyncratic_sd)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R2p_v{i}' for i in range(N)]
    groups_true = [
        {'indices': block1, 'names': [var_names[i] for i in block1],
         'type': 'reflective', 'T': 1.0, 'F': 0.0,
         'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': 8},
        {'indices': block2, 'names': [var_names[i] for i in block2],
         'type': 'reflective', 'T': 1.0, 'F': 0.0,
         'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': 8},
    ]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': "R2'",
            'config_extras': {'a_self': a_self, 'a_cross': a_cross,
                              'sigma_zeta': sigma_zeta,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'nonlinear': True, 'form': 'multiplicative_tanh'}}


# ============================================================================
# §7.5. V-Dem-realistic regimes (controlled experiment, single-mechanism)
# ============================================================================
#
# These regimes vary one V-Dem-likely property at a time relative to R1, with all
# other parameters held constant. Plus R1-vdemlike combines all four properties.
# Purpose: test whether each property changes which discovery methods recover
# rank-1 reflective structure.
#
# All 5 regimes share R1's latent dynamics (rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
# burn_in=50). All differences are in the measurement layer.
#
# Helper: deterministic σ_ε vector with exact 5× max/min ratio
def _make_heteroskedastic_sigma(N, regime_seed_int, mean_sigma=0.3, ratio=5.0):
    """Generate per-indicator σ_ε vector with exact max/min ratio = `ratio`,
    geometric mean = `mean_sigma`. Deterministic given regime_seed_int (i.e.,
    same σ_ε vector across all main-loop seeds).
    """
    rng = np.random.default_rng(regime_seed_int)
    # Draw log-σ from N(0, 1), recenter, then linearly rescale so max-min = log(ratio)
    log_sigma = rng.standard_normal(N)
    log_sigma = log_sigma - log_sigma.mean()
    spread = log_sigma.max() - log_sigma.min()
    if spread > 0:
        log_sigma = log_sigma * (np.log(ratio) / spread)
    # Apply geometric mean
    sigma_eps = mean_sigma * np.exp(log_sigma)
    return sigma_eps


# Helper: deterministic per-unit λ deviation matrix (multiplicative log-normal)
def _make_per_unit_lambda(Lam_pop, U, regime_seed_int, sigma_lam=0.10):
    """Generate per-unit loading matrix Lam_u of shape (U, N) where:
        Lam_u[u, i] = Lam_pop[i] * exp(N(0, sigma_lam))
    Deterministic given regime_seed_int (same Lam_u matrix across main-loop seeds).
    """
    rng = np.random.default_rng(regime_seed_int)
    log_dev = rng.standard_normal((U, len(Lam_pop))) * sigma_lam
    return Lam_pop[None, :] * np.exp(log_dev)


def generate_R1_hetero(seed, U=U_UNITS, T=T_PER_UNIT,
                       rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                       mean_sigma=0.3, sigma_ratio=5.0, burn_in=50):
    """R1-hetero: R1 with heteroskedastic σ_ε per indicator (max/min = 5×).
    Tests Mechanism 1 (heteroskedastic measurement error)."""
    rng = np.random.default_rng(seed)
    N = 12
    Lam = rng.uniform(0.6, 0.9, size=N)
    sigma_eps = _make_heteroskedastic_sigma(N, regime_seed_int=2026001,
                                            mean_sigma=mean_sigma, ratio=sigma_ratio)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            path[t] = Lam * eta + rng.normal(0, sigma_eps)  # vector noise
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1hetero_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': 'R1-hetero',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'mean_sigma': mean_sigma,
                              'sigma_ratio_target': sigma_ratio,
                              'sigma_eps_per_indicator': sigma_eps.tolist(),
                              'nonlinear': True,
                              'measurement_features': ['heteroskedastic']}}


def generate_R1_bounded(seed, U=U_UNITS, T=T_PER_UNIT,
                        rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                        idiosyncratic_sd=0.3, burn_in=50):
    """R1-bounded: R1 with tanh-bounded indicators (y_i ∈ (-1, 1)).
    Tests Mechanism 2 (bounded scale + measurement nonlinearity)."""
    rng = np.random.default_rng(seed)
    N = 12
    Lam = rng.uniform(0.6, 0.9, size=N)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            z = Lam * eta + rng.normal(0, idiosyncratic_sd, size=N)
            path[t] = np.tanh(z)  # bounded indicator
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1bounded_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': 'R1-bounded',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'nonlinear': True,
                              'measurement_features': ['bounded_tanh']}}


def generate_R1_mixed(seed, U=U_UNITS, T=T_PER_UNIT,
                      rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                      idiosyncratic_sd=0.3, burn_in=50,
                      n_singletons=3, rho_singleton=0.7,
                      weak_loading=0.2, sigma_singleton=0.5):
    """R1-mixed: R1 reflective block (size 12) + 3 singletons with own AR(1)
    dynamics and weak η-coupling. Total N = 15.
    Tests Mechanism 4 (cross-block contamination — mirrors V-Dem's
    Suffrage / Direct_vote / Regional_government structure)."""
    rng = np.random.default_rng(seed)
    N_block = 12
    N = N_block + n_singletons
    Lam = rng.uniform(0.6, 0.9, size=N_block)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        # Singleton initial conditions
        s_state = rng.normal(0, sigma_singleton / np.sqrt(1 - rho_singleton**2),
                             size=n_singletons)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            # Block (indices 0..N_block-1)
            path[t, :N_block] = Lam * eta + rng.normal(0, idiosyncratic_sd, size=N_block)
            # Singletons (indices N_block..N-1): own AR(1) + weak η-coupling
            s_state = (rho_singleton * s_state
                       + weak_loading * eta
                       + rng.normal(0, sigma_singleton, size=n_singletons))
            path[t, N_block:] = s_state
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = ([f'R1mixed_v{i}' for i in range(N_block)]
                 + [f'R1mixed_s{k}' for k in range(n_singletons)])
    # groups_true claims only the size-12 reflective block; singletons unmentioned
    groups_true = [{
        'indices': list(range(N_block)), 'names': var_names[:N_block],
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N_block,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam.tolist(),
            'groups_true': groups_true, 'regime': 'R1-mixed',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'n_singletons': n_singletons,
                              'rho_singleton': rho_singleton,
                              'weak_loading': weak_loading,
                              'sigma_singleton': sigma_singleton,
                              'nonlinear': True,
                              'measurement_features': ['weak_singletons']}}


def generate_R1_heterogen(seed, U=U_UNITS, T=T_PER_UNIT,
                          rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                          idiosyncratic_sd=0.3, sigma_lam=0.10, burn_in=50):
    """R1-heterogen: R1 with per-unit λ deviation (multiplicative log-normal).
    Tests Mechanism 5 (heterogeneous units — random-effects measurement model)."""
    rng = np.random.default_rng(seed)
    N = 12
    Lam_pop = rng.uniform(0.6, 0.9, size=N)
    # Per-unit λ matrix, deterministic per regime (same across main-loop seeds)
    Lam_u = _make_per_unit_lambda(Lam_pop, U, regime_seed_int=2026004,
                                  sigma_lam=sigma_lam)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        Lam_this_unit = Lam_u[u]  # length N
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            path[t] = Lam_this_unit * eta + rng.normal(0, idiosyncratic_sd, size=N)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1het_v{i}' for i in range(N)]
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam_pop.tolist(),
            'groups_true': groups_true, 'regime': 'R1-heterogen',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'sigma_lam': sigma_lam,
                              'lam_u_max_min_ratio': float(Lam_u.max() / Lam_u.min()),
                              'nonlinear': True,
                              'measurement_features': ['per_unit_lambda']}}


def generate_R1_vdemlike(seed, U=U_UNITS, T=T_PER_UNIT,
                         rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                         mean_sigma=0.3, sigma_ratio=5.0,
                         sigma_lam=0.10, burn_in=50,
                         n_singletons=3, rho_singleton=0.7,
                         weak_loading=0.2, sigma_singleton=0.5):
    """R1-vdemlike: ALL four V-Dem-likely measurement features combined.
        - Heteroskedastic σ_ε per block indicator (max/min = 5×)
        - tanh-bounded indicators (block + singletons)
        - 3 singletons with own AR(1) + weak η-coupling
        - Per-unit λ_u deviation (multiplicative log-normal, σ_lam=0.10)
    Singletons use homogeneous (non-perturbed) weak_loading per the design spec.
    Total N = 15. Diagnostic regime: tests whether combining V-Dem-like properties
    reproduces the V-Dem method-ranking inversion."""
    rng = np.random.default_rng(seed)
    N_block = 12
    N = N_block + n_singletons
    Lam_pop = rng.uniform(0.6, 0.9, size=N_block)
    sigma_eps = _make_heteroskedastic_sigma(N_block, regime_seed_int=2026005,
                                            mean_sigma=mean_sigma, ratio=sigma_ratio)
    Lam_u = _make_per_unit_lambda(Lam_pop, U, regime_seed_int=2026006,
                                  sigma_lam=sigma_lam)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        s_state = rng.normal(0, sigma_singleton / np.sqrt(1 - rho_singleton**2),
                             size=n_singletons)
        path = np.zeros((T, N))
        Lam_this_unit = Lam_u[u]
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            # Block: heteroskedastic σ_ε, per-unit λ, bounded
            z_block = Lam_this_unit * eta + rng.normal(0, sigma_eps)
            path[t, :N_block] = np.tanh(z_block)
            # Singletons: own AR(1) + weak η-coupling, then bounded
            s_state = (rho_singleton * s_state
                       + weak_loading * eta
                       + rng.normal(0, sigma_singleton, size=n_singletons))
            path[t, N_block:] = np.tanh(s_state)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = ([f'R1vdem_v{i}' for i in range(N_block)]
                 + [f'R1vdem_s{k}' for k in range(n_singletons)])
    groups_true = [{
        'indices': list(range(N_block)), 'names': var_names[:N_block],
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N_block,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam_pop.tolist(),
            'groups_true': groups_true, 'regime': 'R1-vdemlike',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'mean_sigma': mean_sigma,
                              'sigma_ratio_target': sigma_ratio,
                              'sigma_eps_per_indicator': sigma_eps.tolist(),
                              'sigma_lam': sigma_lam,
                              'lam_u_max_min_ratio': float(Lam_u.max() / Lam_u.min()),
                              'n_singletons': n_singletons,
                              'rho_singleton': rho_singleton,
                              'weak_loading': weak_loading,
                              'sigma_singleton': sigma_singleton,
                              'nonlinear': True,
                              'measurement_features': ['heteroskedastic',
                                                       'bounded_tanh',
                                                       'weak_singletons',
                                                       'per_unit_lambda']}}


def generate_R1_contaminated(seed, U=U_UNITS, T=T_PER_UNIT,
                             rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                             idiosyncratic_sd=0.3, burn_in=50,
                             rho_psi=0.5, sigma_zeta_psi=0.5,
                             secondary_loading_max=0.3,
                             n_secondary_indicators=7):
    """R1-contaminated: 13-indicator block with one DOMINANT latent η plus
    one WEAK SECONDARY latent ψ that loads on a subset of the block (first
    `n_secondary_indicators` indicators).

    DGP:
        η_t  = rho * tanh(s * η_{t-1}) + ζ_η          (main latent, rho=0.95)
        ψ_t  = rho_psi * ψ_{t-1}        + ζ_ψ          (secondary, rho_psi=0.5,
                                                        plain AR(1), no tanh)
        y_i  = Lam_main[i] * η + Lam_sec[i] * ψ + ε_i

    where Lam_sec[i] ∈ U[0, secondary_loading_max] for i < n_secondary_indicators
    and Lam_sec[i] = 0 otherwise.

    `groups_true` declares the FULL 13-indicator block as 'reflective' — this is
    intentionally misspecified, mirroring V-Dem's situation where a declared
    block plausibly contains subdimensions / weak secondary factors not modeled
    in the framework's group identification.

    Diagnostic purpose: tests whether secondary-factor contamination in a
    declared reflective block produces non-rank-1 within-block off-diagonals,
    which OLS VAR cannot fit but methods with appropriate inductive biases
    might handle differentially.

    Note on ρ_psi=0.5: deliberately chosen below the typical social-science
    range (~0.7-0.95) to break symmetry with η (rho=0.95). With matched
    persistence, ψ would behave like a "twin" of η and within-block
    off-diagonals could still look rank-1; the asymmetry forces ψ's
    contribution to the indicator covariance structure to be genuinely
    distinct from η's.
    """
    rng = np.random.default_rng(seed)
    N = 13
    Lam_main = rng.uniform(0.6, 0.9, size=N)
    # Secondary factor loadings: nonzero only on first n_secondary_indicators
    Lam_sec = np.zeros(N)
    Lam_sec[:n_secondary_indicators] = rng.uniform(0.0, secondary_loading_max,
                                                    size=n_secondary_indicators)

    unit_paths = []
    for u in range(U):
        # Burn-in η (tanh AR)
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        # Burn-in ψ (plain AR(1))
        psi = 0.0
        for _ in range(burn_in):
            psi = rho_psi * psi + rng.normal(0, sigma_zeta_psi)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            psi = rho_psi * psi + rng.normal(0, sigma_zeta_psi)
            path[t] = (Lam_main * eta + Lam_sec * psi
                       + rng.normal(0, idiosyncratic_sd, size=N))
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R1cont_v{i}' for i in range(N)]
    # Intentionally misspecified: claim full 13-indicator block is 'reflective'
    # even though Lam_sec contaminates the first n_secondary_indicators.
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)), 'Lambda': Lam_main.tolist(),
            'groups_true': groups_true, 'regime': 'R1-contaminated',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'rho_psi': rho_psi,
                              'sigma_zeta_psi': sigma_zeta_psi,
                              'secondary_loading_max': secondary_loading_max,
                              'n_secondary_indicators': n_secondary_indicators,
                              'Lam_secondary': Lam_sec.tolist(),
                              'nonlinear': True,
                              'measurement_features': ['secondary_weak_factor']}}

def generate_R1_dynsplit(seed, U=U_UNITS, T=T_PER_UNIT,
                         rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                         idiosyncratic_sd=0.3, burn_in=50,
                         r_latent=0.85,
                         n_block_A=6, n_block_B=6,
                         loading_low=0.7, loading_high=0.9):
    """R1-dynsplit (Phase 2 diagnostic regime).

    Tests the 'cross-sectionally reflective but dynamically multi-mechanistic'
    hypothesis that emerged from V-Dem dynamic-clustering diagnostics
    (vdem_dynamic_diagnostics.py).

    DGP:
        Two latent processes η_A, η_B with correlated innovations:
            ζ_t ~ N(0, Σ),   Σ = σ_ζ² [[1, r],[r, 1]],  r = r_latent
            η_A_t = ρ tanh(s η_A_{t-1}) + ζ_A_t
            η_B_t = ρ tanh(s η_B_{t-1}) + ζ_B_t
        Sub-block A (size n_block_A) loads ONLY on η_A: y_i = λ_A_i η_A + ε_i
        Sub-block B (size n_block_B) loads ONLY on η_B: y_j = λ_B_j η_B + ε_j
        ε ~ N(0, σ_ε² I) homoskedastic, no per-unit heterogeneity.

    The high latent correlation r_latent=0.85 makes the cross-sectional
    covariance approximately rank-1 (T_cov ≈ 0.97 in calibration tests),
    so the framework's group identification declares a single reflective
    block. But because η_A and η_B have INDEPENDENT lag-1 dynamics
    (only their innovations are correlated, not their lagged values),
    the lag-1 reduced-form W has block-diagonal structure rather than
    rank-1 outer-product structure.

    Calibration (3-seed average, see calibration log):
      r=0.85 → VIF=3.60, T_cov=0.968, PC1=0.763 (closest to V-Dem)

    `groups_true` claims a single 12-indicator reflective block (the entire
    panel) — same intentional misspecification as R1-contaminated. The
    framework's group-id step should still identify a single block, and
    ΔPR is then computed on that block. We expect ΔPR to FAIL or be
    much weaker than on R1 (where η_A, η_B are replaced by a single η).

    Locked parameters per pre-commit (do NOT tune to make discovery
    methods discriminate):
      r_latent       = 0.85   (calibrated for VIF<4 and T_cov≥0.95)
      n_block_A/B    = 6 / 6  (balanced; not matched to V-Dem 7/6 split)
      loading range  = [0.7, 0.9]  (SEM standard for 'good' loadings)
    """
    rng = np.random.default_rng(seed)
    N = n_block_A + n_block_B

    Lam_A = rng.uniform(loading_low, loading_high, size=n_block_A)
    Lam_B = rng.uniform(loading_low, loading_high, size=n_block_B)

    # Innovation covariance for (ζ_A, ζ_B): variance σ_ζ², correlation r_latent.
    cov_zeta = sigma_zeta ** 2 * np.array([[1.0, r_latent], [r_latent, 1.0]])
    L_zeta = np.linalg.cholesky(cov_zeta)

    unit_paths = []
    for u in range(U):
        # Burn-in both latents (correlated innovations applied at each step)
        eta_A, eta_B = 0.0, 0.0
        for _ in range(burn_in):
            zeta = L_zeta @ rng.standard_normal(2)
            eta_A = rho * np.tanh(tanh_scale * eta_A) + zeta[0]
            eta_B = rho * np.tanh(tanh_scale * eta_B) + zeta[1]
        path = np.zeros((T, N))
        for t in range(T):
            zeta = L_zeta @ rng.standard_normal(2)
            eta_A = rho * np.tanh(tanh_scale * eta_A) + zeta[0]
            eta_B = rho * np.tanh(tanh_scale * eta_B) + zeta[1]
            path[t, :n_block_A] = (Lam_A * eta_A
                                    + rng.normal(0, idiosyncratic_sd, n_block_A))
            path[t, n_block_A:] = (Lam_B * eta_B
                                    + rng.normal(0, idiosyncratic_sd, n_block_B))
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)

    var_names = ([f'R1ds_A{i}' for i in range(n_block_A)]
                 + [f'R1ds_B{j}' for j in range(n_block_B)])
    # Intentional misspecification: claim full 12-indicator block as one
    # reflective group. Framework's group-id should agree at T_cov ≈ 0.97,
    # so ΔPR is then tested on the entire 12-indicator block.
    groups_true = [{
        'indices': list(range(N)), 'names': var_names,
        'type': 'reflective', 'T': 1.0, 'F': 0.0,
        'T_tetrad': 1.0, 'F_tetrad': 0.0, 'size': N,
    }]
    return {'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
            'W_star': np.zeros((N, N)),
            'Lambda': np.concatenate([Lam_A, Lam_B]).tolist(),
            'groups_true': groups_true, 'regime': 'R1-dynsplit',
            'config_extras': {'rho': rho, 'sigma_zeta': sigma_zeta,
                              'tanh_scale': tanh_scale,
                              'idiosyncratic_sd': idiosyncratic_sd,
                              'r_latent': r_latent,
                              'vif': float(1.0 / (1.0 - r_latent ** 2)),
                              'n_block_A': n_block_A,
                              'n_block_B': n_block_B,
                              'loading_low': loading_low,
                              'loading_high': loading_high,
                              'nonlinear': True,
                              'measurement_features': ['cross_sectional_dynamic_split']}}


# §8. Convenience: CRPS wrapper (uses the score_W_against_panel from §3)


def compute_crps_for_W(W_hat: np.ndarray, Y: np.ndarray, split_spec: List[int],
                        seed: int = 0) -> float:
    """Run tvNAR scoring on the synthetic panel, V-Dem rev 3 hyperparameters,
    unconstrained protocol (matches §5.3 default). Returns mean CRPS."""
    if W_hat is None or np.allclose(W_hat, 0.0):
        return np.nan
    try:
        rng = np.random.default_rng(seed)
        result = score_W_against_panel(
            W=W_hat, Y=Y, split_spec=split_spec,
            H=H_HORIZON, B=B_BOOT, grid_size=GRID_SIZE,
            ridge=RIDGE, bandwidth=BANDWIDTH,
            kernel="gaussian", lambda_policy="last",
            constrain=False, epsilon=EPSILON,
            rng=rng,
        )
        return float(result.get('CRPS', np.nan))
    except Exception as e:
        print(f"  CRPS scoring failed: {type(e).__name__}: {str(e)[:150]}")
        return np.nan



# §8.5 Two additional evaluation criteria: Geweke + structure-aware fit
#
# **Criterion 4 — Geweke predictive improvement.** For a method's W_hat, refit
# its support pattern by OLS row-by-row and compare residual variance to a
# per-variable AR(1) baseline. Returns the log-ratio F = log(σ²_base / σ²_full)
# and its asymptotic χ² statistic (M·F ~ χ²(df) under H_0). This is the
# classical Geweke (1982) measure of Granger causality applied to W_hat's
# support, evaluating whether the cross-effects identified by the method
# improve predictive log-likelihood over per-variable autoregression.
#
# **Criterion 5 — Structure-aware fit indices.** A method's W_hat induces an
# AR(1) covariance structure; we evaluate how well that structure fits the
# empirical block-level covariance. Following SEM practice, the choice of fit
# index depends on the structural type of each block:
#
#   * **Reflective blocks** (group['type'] == 'reflective'): RMSEA, CFI, TLI.
#     These covariance-decomposition fit indices are appropriate for reflective
#     measurement models where indicators arise from a shared latent factor
#     (Hu & Bentler 1999). On reflective blocks, indicator covariances follow
#     a rank-1 structure (within an idiosyncratic component) so the implied
#     vs. empirical covariance comparison directly tests the latent assumption.
#   * **Formative blocks** (group['type'] == 'formative' or 'independent'):
#     SRMR, NFI. SRMR is residual-based (standardized average covariance
#     residual), not weighted by the implied covariance structure, so it is
#     more robust when the implied covariance does not have reflective form.
#     NFI is incremental, comparing full-model fit to a null model, also less
#     sensitive to the reflective parameterization.
#
# We compute fit indices per group, restricting both the implied and empirical
# covariance to the group's indices. Singletons (size-1 groups) are skipped:
# they have no within-block covariance to fit. Aggregation across groups is
# done within each fit-index family (e.g. mean RMSEA across reflective blocks).
#
# This is in line with SEM practice (Diamantopoulos & Winklhofer 2001;
# Diamantopoulos & Siguaw 2006) where formative measurement requires
# residual/predictive fit rather than covariance-decomposition fit, and is
# motivated by the observation that a method's W_hat may legitimately fit
# formative-block covariance well even when no reflective latent exists.
#
# **Caveats** (documented in the paper):
#  - With M ≈ 6000+ panel lag pairs (V-Dem) the χ² scaling makes RMSEA stricter
#    than its conventional thresholds (designed for n ≈ 200-500). Reported
#    RMSEA values may exceed Hu & Bentler's 0.08 threshold even for reasonable
#    fits.
#  - When a method's W_hat is dense enough that nnz(A_block) + p_block exceeds
#    p_block(p_block+1)/2, df becomes ≤ 0 and RMSEA / CFI / TLI become
#    undefined for that block. We return NaN with `identified=False`. SRMR
#    does not have this problem.


def _build_panel_lag_pairs_simple(Y, split_spec):
    """Build (y_prev, y_curr) lag pairs respecting unit boundaries."""
    y_prev_list, y_curr_list = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            y_prev_list.append(unit[:-1])
            y_curr_list.append(unit[1:])
        cursor += n_u
    if not y_prev_list:
        raise ValueError("No valid (t-1, t) pairs.")
    return np.vstack(y_prev_list), np.vstack(y_curr_list)


def geweke_improvement(W_hat, Y, split_spec, support_threshold=1e-10):
    """Geweke predictive-improvement test: does method's W_hat improve predictive
    log-likelihood over per-variable AR(1) baseline?
    """
    from scipy import stats as scstats
    N = Y.shape[1]
    y_prev, y_curr = _build_panel_lag_pairs_simple(Y, split_spec)
    M = y_prev.shape[0]

    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    df = int(support.sum())

    F_per_target = np.zeros(N)
    sigma2_base = np.zeros(N)
    sigma2_full = np.zeros(N)
    for i in range(N):
        y_target = y_curr[:, i]
        # Baseline: own lag + intercept only
        X_base = np.column_stack([y_prev[:, i], np.ones(M)])
        beta_base, _, _, _ = np.linalg.lstsq(X_base, y_target, rcond=None)
        s2_base = float(np.mean((y_target - X_base @ beta_base) ** 2))
        sigma2_base[i] = s2_base

        # Full: own lag + cross sources from support + intercept
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_full = np.column_stack(cols)
        if X_full.shape[1] >= M:
            sigma2_full[i] = s2_base
            F_per_target[i] = 0.0
            continue
        beta_full, _, _, _ = np.linalg.lstsq(X_full, y_target, rcond=None)
        s2_full = float(np.mean((y_target - X_full @ beta_full) ** 2))
        sigma2_full[i] = s2_full
        F_per_target[i] = float(np.log(s2_base / max(s2_full, 1e-20))) if s2_full > 0 else 0.0

    F_total = float(F_per_target.sum())
    chi2_stat = M * F_total
    if df > 0 and chi2_stat > 0:
        p_value = float(scstats.chi2.sf(chi2_stat, df=df))
    else:
        p_value = 1.0

    return {
        'F_total':         F_total,
        'chi2':            float(chi2_stat),
        'df':              df,
        'p_value':         p_value,
        'sigma2_base':     float(sigma2_base.mean()),
        'sigma2_full':     float(sigma2_full.mean()),
        'M':               int(M),
    }


def _implied_stationary_cov(A, Sigma_eps, max_iter=200, tol=1e-10):
    """Solve Σ = A Σ A^T + Σ_eps for stationary Σ. Returns None if unstable."""
    eigs = np.linalg.eigvals(A)
    if np.max(np.abs(eigs)) >= 1.0:
        return None
    try:
        from scipy.linalg import solve_discrete_lyapunov
        return solve_discrete_lyapunov(A, Sigma_eps)
    except ImportError:
        Q = Sigma_eps.copy()
        Ak = A.copy()
        for _ in range(max_iter):
            Q_new = Q + Ak @ Q @ Ak.T
            Ak_new = Ak @ Ak
            if np.linalg.norm(Q_new - Q) < tol * np.linalg.norm(Q):
                return Q_new
            Q, Ak = Q_new, Ak_new
        return Q


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    """OLS refit row-by-row of the AR(1) model with W_hat's support pattern.

    Returns (A_full, resid_full, Sigma_eps_full): the panel-level AR(1)
    coefficient matrix (own-lag on diagonal, cross-effects on support),
    one-step residuals, and residual covariance (positive-definite floored).
    """
    M, N = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N, N))
    resid_full = np.zeros((M, N))
    for i in range(N):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X = np.column_stack(cols)
        if X.shape[1] >= M:
            X = np.column_stack([y_prev[:, i], np.ones(M)])
            beta, _, _, _ = np.linalg.lstsq(X, y_target, rcond=None)
            A_full[i, i] = float(beta[0])
            resid_full[:, i] = y_target - X @ beta
            continue
        beta, _, _, _ = np.linalg.lstsq(X, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            A_full[i, j] = float(beta[1 + k])
        resid_full[:, i] = y_target - X @ beta
    Sigma_eps = (resid_full.T @ resid_full) / max(M - 1, 1)
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-12:
        Sigma_eps = Sigma_eps + (1e-10 - eigs.min()) * np.eye(N)
    return A_full, resid_full, Sigma_eps


def _block_fit_all(Sigma_emp_b, Sigma_imp_b, A_block_nnz, p_block, M):
    """Compute all five SEM-style fit indices for a single block, regardless
    of structural type. Returns:
      - SRMR, NFI: always computed when block is valid (no df problem)
      - RMSEA, CFI, TLI: computed when df > 0 (i.e., the block-level model
        is identified by covariance alone); NaN otherwise

    SRMR: standardized residual covariance (Hu & Bentler 1999). Robust to
    block density because it's purely residual-based.
    NFI: incremental fit (Bentler & Bonett 1980). Compares full to null
    (diagonal) model; doesn't depend on df-clamping.
    RMSEA, CFI, TLI: covariance-decomposition fit (Steiger 1990, Hu & Bentler
    1999). Conventional for reflective measurement models. Require df > 0.

    Args:
      Sigma_emp_b: (p_block, p_block) empirical covariance restricted to block.
      Sigma_imp_b: (p_block, p_block) implied covariance restricted to block.
      A_block_nnz: number of nonzero entries in within-block A sub-matrix.
      p_block: block size.
      M: total number of lag pairs.

    Returns dict with srmr, nfi, rmsea, cfi, tli, chi2, df, identified.
    """
    if p_block < 2 or Sigma_imp_b is None:
        return {'srmr': np.nan, 'nfi': np.nan,
                'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                'chi2': np.nan, 'df': 0, 'identified': False}

    # ---- SRMR (residual-based, no df dependence) ----
    se_diag = np.sqrt(np.diag(Sigma_emp_b) + 1e-20)
    diff = Sigma_emp_b - Sigma_imp_b
    standardized = diff / np.outer(se_diag, se_diag)
    n_distinct_block = p_block * (p_block + 1) // 2
    iu = np.tril_indices(p_block)
    srmr = float(np.sqrt(np.sum(standardized[iu] ** 2) / max(n_distinct_block, 1)))

    # ---- ML discrepancies for NFI, CFI, TLI, RMSEA ----
    try:
        sign_imp, ld_imp = np.linalg.slogdet(Sigma_imp_b)
        sign_emp, ld_emp = np.linalg.slogdet(Sigma_emp_b)
        if sign_imp <= 0 or sign_emp <= 0:
            return {'srmr': srmr, 'nfi': np.nan,
                    'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                    'chi2': np.nan, 'df': 0, 'identified': False}
        F_full = float(max(0.0, ld_imp + np.trace(Sigma_emp_b @ np.linalg.inv(Sigma_imp_b))
                            - ld_emp - p_block))
    except np.linalg.LinAlgError:
        return {'srmr': srmr, 'nfi': np.nan,
                'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                'chi2': np.nan, 'df': 0, 'identified': False}

    Sigma_null_b = np.diag(np.diag(Sigma_emp_b))
    try:
        sign_n, ld_n = np.linalg.slogdet(Sigma_null_b)
        F_null = float(max(0.0, ld_n + np.trace(Sigma_emp_b @ np.linalg.inv(Sigma_null_b))
                            - ld_emp - p_block))
    except np.linalg.LinAlgError:
        F_null = np.nan

    # NFI: incremental fit. Always computable from F_full and F_null;
    # not affected by df.
    if np.isfinite(F_null) and F_null > 0:
        nfi = float(max(0.0, min(1.0, (F_null - F_full) / F_null)))
    else:
        nfi = np.nan

    # df: n_distinct(p_block) - n_params(within-block A entries + p_block residuals)
    n_params_block = A_block_nnz + p_block
    df_block = n_distinct_block - n_params_block
    df_null_block = n_distinct_block - p_block

    chi2_full = M * F_full
    chi2_null = M * F_null if np.isfinite(F_null) else np.nan

    # RMSEA, CFI, TLI: only when df > 0 (model identified by covariance)
    if df_block > 0:
        rmsea = float(np.sqrt(max(0.0, (chi2_full / df_block - 1) / max(M - 1, 1))))
        if np.isfinite(chi2_null) and df_null_block > 0:
            num = max(0.0, chi2_full - df_block)
            den = max(num, chi2_null - df_null_block, 0.0)
            cfi = float(1 - num / den) if den > 0 else 1.0
            ratio_null = chi2_null / df_null_block
            ratio_full = chi2_full / df_block
            tli = float((ratio_null - ratio_full) / (ratio_null - 1)) if ratio_null != 1 else np.nan
        else:
            cfi = np.nan; tli = np.nan
        identified = True
    else:
        rmsea = np.nan; cfi = np.nan; tli = np.nan; identified = False

    return {
        'srmr': srmr, 'nfi': nfi,
        'rmsea': rmsea, 'cfi': cfi, 'tli': tli,
        'chi2': float(chi2_full), 'df': int(df_block),
        'identified': identified,
    }


def structure_aware_fit(W_hat, Y, split_spec, groups,
                         support_threshold=1e-10):
    """Compute all five SEM fit indices on every block, regardless of type.

    Each block (size >= 2) gets RMSEA, CFI, TLI, SRMR, NFI computed from
    the implied vs. empirical covariance restricted to that block. RMSEA /
    CFI / TLI are NaN when the block-level model is over-parameterized
    (df ≤ 0); SRMR and NFI are always computable.

    Args:
      W_hat: (N, N) method's edge-importance matrix.
      Y: (T_total, N) panel data.
      split_spec: list of unit lengths.
      groups: list of dicts, each with 'indices' (list[int]) and 'type'
              (str). Type is recorded for downstream summarization but does
              NOT determine which fit indices are computed — all are. Size-1
              groups (singletons) are skipped: no within-block covariance.

    Returns dict with:
      blocks:                list of per-block dicts (all 5 indices, type, indices)
      rmsea_mean, cfi_mean,  mean across blocks where the index is computable
      tli_mean, srmr_mean,
      nfi_mean:
      n_blocks:              total number of size>=2 blocks evaluated
      n_reflective_blocks,   counts by type (descriptive only)
      n_formative_blocks,
      n_independent_blocks:
      srmr_passes:           SRMR < 0.08 on all blocks (Hu & Bentler 1999)
      rmsea_passes:          RMSEA < 0.08 on all identifiable blocks
                              (NaN/unidentified blocks treated as N/A,
                               not as failures); False if no block is
                               RMSEA-identifiable
    """
    y_prev, y_curr = _build_panel_lag_pairs_simple(Y, split_spec)
    M = y_prev.shape[0]
    A_full, _, Sigma_eps = _ols_refit_panel(W_hat, y_prev, y_curr,
                                              support_threshold)
    Sigma_imp = _implied_stationary_cov(A_full, Sigma_eps)
    Yc = y_curr - y_curr.mean(axis=0)
    Sigma_emp = (Yc.T @ Yc) / max(M - 1, 1)

    block_results = []
    n_refl = 0; n_form = 0; n_indep = 0

    for g in groups:
        idx = list(g['indices'])
        p_block = len(idx)
        if p_block < 2:
            continue
        gtype = g.get('type', 'unknown')
        if gtype == 'reflective':
            n_refl += 1
        elif gtype == 'formative':
            n_form += 1
        elif gtype == 'independent':
            n_indep += 1

        Sigma_emp_b = Sigma_emp[np.ix_(idx, idx)]
        Sigma_imp_b = (Sigma_imp[np.ix_(idx, idx)]
                        if Sigma_imp is not None else None)
        A_block = A_full[np.ix_(idx, idx)]
        A_block_nnz = int(np.sum(np.abs(A_block) > 1e-12))

        r = _block_fit_all(Sigma_emp_b, Sigma_imp_b, A_block_nnz, p_block, M)
        r['block_indices'] = idx
        r['p_block'] = p_block
        r['gtype'] = gtype
        block_results.append(r)

    def _safe_mean(xs):
        xs = [x for x in xs if np.isfinite(x)]
        return float(np.mean(xs)) if xs else np.nan

    rmsea_mean = _safe_mean([r['rmsea'] for r in block_results])
    cfi_mean   = _safe_mean([r['cfi']   for r in block_results])
    tli_mean   = _safe_mean([r['tli']   for r in block_results])
    srmr_mean  = _safe_mean([r['srmr']  for r in block_results])
    nfi_mean   = _safe_mean([r['nfi']   for r in block_results])

    # SRMR passes: every block has SRMR < 0.08 (Hu & Bentler 1999 threshold)
    srmr_passes = (
        len(block_results) > 0
        and all(np.isfinite(r['srmr']) and r['srmr'] < 0.08 for r in block_results)
    )
    # RMSEA passes: every block that's RMSEA-identifiable has RMSEA < 0.08;
    # at least one block must be identifiable (else N/A → False).
    identifiable = [r for r in block_results
                    if r['identified'] and np.isfinite(r['rmsea'])]
    rmsea_passes = (
        len(identifiable) > 0
        and all(r['rmsea'] < 0.08 for r in identifiable)
    )

    return {
        'blocks':                block_results,
        'rmsea_mean':            rmsea_mean,
        'cfi_mean':              cfi_mean,
        'tli_mean':              tli_mean,
        'srmr_mean':             srmr_mean,
        'nfi_mean':              nfi_mean,
        'n_blocks':              len(block_results),
        'n_reflective_blocks':   n_refl,
        'n_formative_blocks':    n_form,
        'n_independent_blocks':  n_indep,
        'n_rmsea_identifiable':  len(identifiable),
        'srmr_passes':           bool(srmr_passes),
        'rmsea_passes':          bool(rmsea_passes),
        'M':                     int(M),
    }




# §9. Main loop — for each (regime × seed × method): generate panel, run method,
# compute LSC, compute CRPS. Save to CSV after every cell.


REGIMES = {
    'R1':              generate_R1,
    "R1'":             generate_R1prime,
    'R2':              generate_R2,
    "R2'":             generate_R2prime,
    'R3':              generate_R3,
    "R3'":             generate_R3prime,
    # V-Dem-realistic regimes (single-mechanism + combined diagnostic).
    # See §7.5 for design rationale.
    'R1-hetero':       generate_R1_hetero,
    'R1-bounded':      generate_R1_bounded,
    'R1-mixed':        generate_R1_mixed,
    'R1-heterogen':    generate_R1_heterogen,
    'R1-vdemlike':     generate_R1_vdemlike,
    # Subdimension-contamination diagnostic (intentionally misspecified block;
    # tests whether secondary-factor contamination breaks rank-1 detection).
    'R1-contaminated': generate_R1_contaminated,
    # Phase 2 diagnostic: cross-sectionally rank-1 + dynamically multi-block
    # (motivated by V-Dem dynamic-clustering result; vdem_dynamic_diagnostics.py).
    'R1-dynsplit':    generate_R1_dynsplit,
}


# §8.5. DYNOTEARS per-regime $\lambda_a$ tuning
#
# DYNOTEARS' L1 penalty $\lambda_a$ controls W_hat density. For fair comparison with
# the other 4 methods, we tune $\lambda_a$ separately on each regime so that DYNOTEARS'
# nnz lands close to the *median* nnz across the 4 comparator methods on that regime.
#
# **Tuning protocol** (matches the V-Dem density-matched policy from §5):
#   1. For each regime, run the 4 comparator methods on **seed=0** only and record nnz.
#   2. Compute median nnz across these 4 methods. Call it $\mathrm{nnz}_{\mathrm{ref}}$.
#   3. Run DYNOTEARS at every $\lambda_a$ in the prespecified sweep
#      $\{0.05, 0.01, 0.005, 0.001, 0.0005\}$ on the same seed=0 panel.
#   4. Pick the $\lambda_a$ minimizing $|\mathrm{nnz}_{\mathrm{dynotears}} - \mathrm{nnz}_{\mathrm{ref}}|$.
#   5. Apply that $\lambda_a$ to all 5 seeds in the main loop.
#
# This is conceptually identical to the V-Dem tuning (where $\lambda_a=0.001$ was
# selected from the same sweep to match comparator density on long_16var). Tuning is
# a regime-property, not a per-seed property; using a single seed for tuning saves
# ~80% of tuning compute and avoids selection variance.
#
# Output: `DYNOTEARS_LAMBDA_PER_REGIME` dict (in-memory, used by run_method) and
# `OUT_DIR/dynotears_lambda_per_regime.json` (persisted for the appendix).


DYNOTEARS_LAMBDA_SWEEP = [0.05, 0.01, 0.005, 0.001, 0.0005]
DYNOTEARS_TUNING_SEED  = 0  # use seed=0 for tuning; main loop uses seeds 0..N_SEEDS-1
DYNOTEARS_TUNING_METHODS = ['linear_var_granger', 'pcmci', 'cmlp', 'navar']
DYNOTEARS_SELECTION_RULE = 'closest_absolute_nnz_to_comparator_median'

dynotears_lambda_path = os.path.join(OUT_DIR, 'dynotears_lambda_per_regime.json')

# Resume logic with two-layer compatibility check:
#   Layer 1 (schema check): if any of tuning_seed / sweep / comparator_methods /
#     selection_rule has changed in the notebook vs the loaded JSON, archive
#     the old JSON and retune from scratch.
#   Layer 2 (per-regime gap check): for any regime in REGIMES not in the loaded
#     lambda_per_regime, run tuning for those only and append.
DYNOTEARS_LAMBDA_PER_REGIME = {}
tuning_diagnostics = {}

if os.path.exists(dynotears_lambda_path):
    with open(dynotears_lambda_path) as f:
        loaded = json.load(f)
    schema_ok = (
        loaded.get('tuning_seed')        == DYNOTEARS_TUNING_SEED
        and loaded.get('sweep')          == DYNOTEARS_LAMBDA_SWEEP
        and loaded.get('comparator_methods') == DYNOTEARS_TUNING_METHODS
        and loaded.get('selection_rule') == DYNOTEARS_SELECTION_RULE
    )
    if not schema_ok:
        from datetime import datetime
        archive_suffix = f"__stale_{datetime.now():%Y%m%d_%H%M%S}.json"
        archive_path = dynotears_lambda_path.replace('.json', archive_suffix)
        os.rename(dynotears_lambda_path, archive_path)
        print(f"⚠ Schema mismatch in existing JSON. Archived to {archive_path}.")
        print(f"  Will re-tune all regimes from scratch.")
    else:
        DYNOTEARS_LAMBDA_PER_REGIME = dict(loaded.get('lambda_per_regime', {}))
        tuning_diagnostics = dict(loaded.get('diagnostics', {}))
        print(f"Loaded existing DYNOTEARS lambda map "
              f"({len(DYNOTEARS_LAMBDA_PER_REGIME)} regimes):")
        for r, lam in DYNOTEARS_LAMBDA_PER_REGIME.items():
            print(f"  {r}: lambda_a = {lam}")

# Compute set of regimes still needing tuning
missing_regimes = [r for r in REGIMES.keys() if r not in DYNOTEARS_LAMBDA_PER_REGIME]

if missing_regimes:
    print(f"\nTuning DYNOTEARS lambda_a for {len(missing_regimes)} new regime(s): "
          f"{missing_regimes}")
    print(f"  (seed={DYNOTEARS_TUNING_SEED}, sweep={DYNOTEARS_LAMBDA_SWEEP})")
    t_tune_start = time.time()

    for regime_name in missing_regimes:
        gen_fn = REGIMES[regime_name]
        print(f"\n--- {regime_name} ---")
        panel = gen_fn(seed=DYNOTEARS_TUNING_SEED)
        Y_t, split_spec_t, var_names_t = panel['Y'], panel['split_spec'], panel['var_names']

        # Step 1: comparator nnz on seed=0
        comparator_nnz = {}
        for m in DYNOTEARS_TUNING_METHODS:
            res = run_method(m, Y_t, split_spec_t, variable_names=var_names_t,
                             seed=DYNOTEARS_TUNING_SEED, regime_name=regime_name)
            if res['error'] is not None or res['W_hat'] is None:
                print(f"  comparator {m}: ERROR, treating nnz as 0")
                comparator_nnz[m] = 0
            else:
                comparator_nnz[m] = int(np.sum(np.abs(res['W_hat']) > 1e-10))
        nnz_ref = float(np.median(list(comparator_nnz.values())))
        print(f"  comparator nnz on seed=0: {comparator_nnz}, median = {nnz_ref:.1f}")

        # Step 2: DYNOTEARS at each lambda_a
        sweep_nnz = {}
        for lam in DYNOTEARS_LAMBDA_SWEEP:
            try:
                out = run_dynotears(Y_t, split_spec_t, variable_names=var_names_t,
                                    lambda_a=lam, seed=DYNOTEARS_TUNING_SEED)
                W_lam = out['W_hat']
                nnz_lam = int(np.sum(np.abs(W_lam) > 1e-10))
            except Exception as e:
                print(f"  lambda={lam}: FAILED ({type(e).__name__}: {str(e)[:80]})")
                nnz_lam = -1
            sweep_nnz[lam] = nnz_lam
            print(f"  lambda={lam}: nnz={nnz_lam}")

        # Step 3: pick closest-nnz lambda (skip failed runs with nnz=-1)
        valid = {lam: nnz for lam, nnz in sweep_nnz.items() if nnz >= 0}
        if not valid:
            raise RuntimeError(f"All DYNOTEARS fits failed on regime {regime_name}; "
                               f"cannot tune lambda_a.")
        chosen_lam = min(valid.keys(), key=lambda l: abs(valid[l] - nnz_ref))
        DYNOTEARS_LAMBDA_PER_REGIME[regime_name] = chosen_lam
        tuning_diagnostics[regime_name] = {
            'comparator_nnz':     comparator_nnz,
            'nnz_ref_median':     nnz_ref,
            'sweep_nnz':          sweep_nnz,
            'chosen_lambda_a':    chosen_lam,
            'chosen_nnz':         valid[chosen_lam],
            'abs_dist_to_median': abs(valid[chosen_lam] - nnz_ref),
        }
        print(f"  CHOSEN: lambda_a = {chosen_lam} (nnz={valid[chosen_lam]}, "
              f"distance to median = {abs(valid[chosen_lam] - nnz_ref):.1f})")

    # Persist merged tuning result + diagnostics for the appendix
    with open(dynotears_lambda_path, 'w') as f:
        json.dump({
            'lambda_per_regime':  DYNOTEARS_LAMBDA_PER_REGIME,
            'tuning_seed':        DYNOTEARS_TUNING_SEED,
            'sweep':              DYNOTEARS_LAMBDA_SWEEP,
            'comparator_methods': DYNOTEARS_TUNING_METHODS,
            'selection_rule':     DYNOTEARS_SELECTION_RULE,
            'diagnostics':        tuning_diagnostics,
        }, f, indent=2, default=str)
    print(f"\nDYNOTEARS lambda tuning complete in {time.time() - t_tune_start:.1f}s")
    print(f"Saved: {dynotears_lambda_path}")
    print(f"Final map: {DYNOTEARS_LAMBDA_PER_REGIME}")
else:
    print(f"\nAll {len(REGIMES)} regimes have cached DYNOTEARS lambda values; "
          f"no tuning needed.")

results_path = os.path.join(OUT_DIR, 'synthetic_results.csv')
WHATS_DIR = os.path.join(OUT_DIR, 'W_hats')
PANELS_DIR = os.path.join(OUT_DIR, 'panels')
os.makedirs(WHATS_DIR, exist_ok=True)
os.makedirs(PANELS_DIR, exist_ok=True)
all_results = []

# Resume support: reload existing results if present
if os.path.exists(results_path):
    df_prev = pd.read_csv(results_path)
    all_results = df_prev.to_dict('records')
    seen = set((r['regime'], r['seed'], r['method']) for r in all_results)
    print(f"Resuming from {results_path}: {len(all_results)} existing rows")
    print(f"NOTE: existing rows may not have saved W_hats (older runs); only new runs save W_hats.")
else:
    seen = set()
    print(f"Starting fresh. Output: {results_path}")
print(f"  W_hats will be saved to {WHATS_DIR}/")
print(f"  Panel snapshots will be saved to {PANELS_DIR}/")

print(f"\nWill run: {len(REGIMES)} regimes × {N_SEEDS} seeds × {len(METHODS_TO_RUN)} methods "
      f"= {len(REGIMES) * N_SEEDS * len(METHODS_TO_RUN)} cells")

t_start_all = time.time()
for regime_name, gen_fn in REGIMES.items():
    print(f"\n{'='*70}\nREGIME: {regime_name}\n{'='*70}")
    for seed in range(N_SEEDS):
        # Generate panel once per (regime, seed)
        panel = gen_fn(seed=seed)
        Y, split_spec, var_names = panel['Y'], panel['split_spec'], panel['var_names']
        groups_true = panel['groups_true']
        # Save panel for later re-analysis (Geweke, SEM-fit) without re-running DGP.
        # Filename normalization: R3' -> R3prime, R1-hetero -> R1_hetero (filesystem-safe).
        fs_regime = regime_name.replace("'", "prime").replace("-", "_")
        panel_path = os.path.join(PANELS_DIR, f"panel_{fs_regime}_seed{seed}.pkl")
        if not os.path.exists(panel_path):
            with open(panel_path, 'wb') as f:
                pickle.dump({
                    'regime': regime_name, 'seed': seed,
                    'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
                    'groups_true': groups_true,
                    'config_extras': panel.get('config_extras', {}),
                }, f)
        # Sanity check: cov-rank-1 on the true reflective group(s)
        for g in groups_true:
            if g['size'] >= 4:
                T_cov, _ = cov_rank1_signature(Y, g['indices'])
                print(f"  [seed={seed}] true group ({g['type']}, size={g['size']}): "
                      f"empirical T_cov = {T_cov:.3f}")
        # Run methods
        for method_name in METHODS_TO_RUN:
            cell_key = (regime_name, seed, method_name)
            if cell_key in seen:
                print(f"  [seed={seed}] {method_name}: cached, skipping")
                continue
            t0 = time.time()
            res_method = run_method(method_name, Y, split_spec,
                                    variable_names=var_names, seed=seed,
                                    regime_name=regime_name)
            if res_method['error']:
                print(f"  [seed={seed}] {method_name}: ERROR — {res_method['error']}")
                row = {
                    'regime': regime_name, 'seed': seed, 'method': method_name,
                    'delta_PR_diag': np.nan, 'delta_PR': np.nan,
                    'delta_RC': np.nan, 'delta_FC': np.nan, 'delta_LSC': np.nan,
                    'p_perm': np.nan, 'p_PR': np.nan, 'p_RC': np.nan,
                    'p_FC': np.nan, 'p_LSC': np.nan,
                    'PR_obs': np.nan, 'PR_null_mean': np.nan,
                    'RC_obs': np.nan, 'RC_null_mean': np.nan,
                    'LSC_obs': np.nan, 'LSC_null_mean': np.nan,
                    'crps': np.nan,
                    'geweke_F': np.nan, 'geweke_chi2': np.nan, 'geweke_df': 0,
                    'geweke_p': np.nan,
                    'fit_rmsea_mean':       np.nan,
                    'fit_cfi_mean':         np.nan,
                    'fit_tli_mean':         np.nan,
                    'fit_srmr_mean':        np.nan,
                    'fit_nfi_mean':         np.nan,
                    'n_blocks':             0,
                    'n_reflective_blocks':  0,
                    'n_formative_blocks':   0,
                    'n_independent_blocks': 0,
                    'n_rmsea_identifiable': 0,
                    'srmr_passes':          False,
                    'rmsea_passes':         False,
                    'nnz': 0,
                    'method_elapsed_s': res_method['elapsed'],
                    'error': res_method['error'],
                }
                all_results.append(row)
                pd.DataFrame(all_results).to_csv(results_path, index=False)
                continue
            W_hat = res_method['W_hat']
            # Save W_hat for downstream re-analysis (Geweke, SEM-fit).
            fs_regime = regime_name.replace("'", "prime").replace("-", "_")
            whats_path = os.path.join(WHATS_DIR,
                f"W_hat_{fs_regime}_seed{seed}_{method_name}.pkl")
            with open(whats_path, 'wb') as f:
                pickle.dump({'regime': regime_name, 'seed': seed,
                             'method': method_name, 'W_hat': W_hat,
                             'nnz': int(np.sum(np.abs(W_hat) > 1e-10))}, f)
            # LSC + null (returns delta_LSC, delta_PR_diag, delta_RC, delta_FC, ...)
            lsc_res = lsc_with_null(W_hat, groups_true, B=B_PERMS, seed=seed)
            # CRPS (panel-safe: residuals from within-unit pairs only,
            # evaluation averaged across last-H pairs of each unit)
            crps = compute_crps_for_W(W_hat, Y, split_spec, seed=seed)
            # Geweke predictive-improvement test
            try:
                gw = geweke_improvement(W_hat, Y, split_spec)
            except Exception as e:
                print(f"  Geweke failed: {type(e).__name__}: {str(e)[:100]}")
                gw = {'F_total': np.nan, 'chi2': np.nan, 'df': 0,
                      'p_value': np.nan, 'sigma2_base': np.nan, 'sigma2_full': np.nan}
            # Structure-aware fit indices: all five (RMSEA, CFI, TLI, SRMR, NFI)
            # computed per block. RMSEA/CFI/TLI are NaN when the block-level
            # model is over-parameterized; SRMR/NFI always return values.
            try:
                sa = structure_aware_fit(W_hat, Y, split_spec, groups_true)
            except Exception as e:
                print(f"  Structure-aware fit failed: {type(e).__name__}: {str(e)[:100]}")
                sa = {'rmsea_mean': np.nan, 'cfi_mean': np.nan, 'tli_mean': np.nan,
                      'srmr_mean': np.nan, 'nfi_mean': np.nan,
                      'n_blocks': 0, 'n_reflective_blocks': 0,
                      'n_formative_blocks': 0, 'n_independent_blocks': 0,
                      'n_rmsea_identifiable': 0,
                      'srmr_passes': False, 'rmsea_passes': False}
            row = {
                'regime': regime_name, 'seed': seed, 'method': method_name,
                # ===== Headline excess-over-null metrics =====
                'delta_PR_diag': lsc_res['delta_PR_diag'],   # PR averaged over all admissible groups
                'delta_PR':      lsc_res['delta_PR'],        # PR over reflective groups only
                'delta_RC':      lsc_res['delta_RC'],        # cross-block (Component B)
                'delta_FC':      lsc_res['delta_FC'],        # loading coherence (Component C)
                'delta_LSC':     lsc_res['delta_LSC'],       # full composite
                # ===== Empirical p-values =====
                'p_perm':        lsc_res['p_PR_diag'],
                'p_PR':          lsc_res['p_PR'],
                'p_RC':          lsc_res['p_RC'],
                'p_FC':          lsc_res['p_FC'],
                'p_LSC':         lsc_res['p_LSC'],
                # ===== Observed and null-mean values (descriptive) =====
                'PR_obs':        lsc_res['PR_diagnostic'],
                'PR_null_mean':  lsc_res['null_PR_diag_mean'],
                'RC_obs':        lsc_res['RC'],
                'RC_null_mean':  lsc_res['null_RC_mean'],
                'LSC_obs':       lsc_res['LSC'],
                'LSC_null_mean': lsc_res['null_LSC_mean'],
                # ===== Predictive: CRPS =====
                'crps':          crps,
                # ===== Predictive: Geweke =====
                'geweke_F':       gw.get('F_total', np.nan),
                'geweke_chi2':    gw.get('chi2', np.nan),
                'geweke_df':      gw.get('df', 0),
                'geweke_p':       gw.get('p_value', np.nan),
                # ===== Structural: all five fit indices, computed on every block =====
                'fit_rmsea_mean':       sa.get('rmsea_mean', np.nan),
                'fit_cfi_mean':         sa.get('cfi_mean', np.nan),
                'fit_tli_mean':         sa.get('tli_mean', np.nan),
                'fit_srmr_mean':        sa.get('srmr_mean', np.nan),
                'fit_nfi_mean':         sa.get('nfi_mean', np.nan),
                'n_blocks':             int(sa.get('n_blocks', 0)),
                'n_reflective_blocks':  int(sa.get('n_reflective_blocks', 0)),
                'n_formative_blocks':   int(sa.get('n_formative_blocks', 0)),
                'n_independent_blocks': int(sa.get('n_independent_blocks', 0)),
                'n_rmsea_identifiable': int(sa.get('n_rmsea_identifiable', 0)),
                'srmr_passes':          bool(sa.get('srmr_passes', False)),
                'rmsea_passes':         bool(sa.get('rmsea_passes', False)),
                # ===== Bookkeeping =====
                'nnz':           int(np.sum(np.abs(W_hat) > 1e-10)),
                'method_elapsed_s': res_method['elapsed'],
                'error':         None,
            }
            all_results.append(row)
            pd.DataFrame(all_results).to_csv(results_path, index=False)
            # Print headline. Special-case nnz=0: method correctly identified
            # no significant edges, so ΔPR / CRPS are undefined by design.
            if row['nnz'] == 0:
                print(f"  [seed={seed}] {method_name}: 0 edges (correctly "
                      f"silent; ΔPR/CRPS undefined), time={res_method['elapsed']:.1f}s")
            else:
                # Print headline based on regime: delta_PR for single-group regimes,
                # delta_LSC for multi-reflective-group regimes (R2)
                n_refl = sum(1 for g in groups_true if g['type'] == 'reflective')
                if n_refl >= 2 and np.isfinite(row['delta_LSC']):
                    head_metric = f"ΔLSC={row['delta_LSC']:+.3f} (p={row['p_LSC']:.3f})"
                    head_extra = f" ΔRC={row['delta_RC']:+.3f}"
                else:
                    head_metric = f"ΔPR={row['delta_PR_diag']:+.3f} (p={row['p_perm']:.3f})"
                    head_extra = ""
                # All five fit indices on every block; print whichever is finite.
                # RMSEA/CFI/TLI may be NaN when block is over-parameterized;
                # SRMR/NFI are always computable.
                rmsea_str = f"RMSEA={row['fit_rmsea_mean']:.3f}" if np.isfinite(row['fit_rmsea_mean']) else "RMSEA=NA"
                cfi_str   = f"CFI={row['fit_cfi_mean']:.3f}"   if np.isfinite(row['fit_cfi_mean'])   else "CFI=NA"
                srmr_str  = f"SRMR={row['fit_srmr_mean']:.3f}" if np.isfinite(row['fit_srmr_mean']) else "SRMR=NA"
                nfi_str   = f"NFI={row['fit_nfi_mean']:.3f}"   if np.isfinite(row['fit_nfi_mean'])   else "NFI=NA"
                fit_str = f"{srmr_str} {nfi_str} {rmsea_str} {cfi_str}"
                # Two pass flags: SRMR-based (always evaluable) and RMSEA-based
                # (only when at least one block is identifiable)
                srmr_mark  = "✓" if row['srmr_passes']  else "✗"
                rmsea_mark = "✓" if row['rmsea_passes'] else ("·" if row['n_rmsea_identifiable'] == 0 else "✗")
                print(f"  [seed={seed}] {method_name}: {head_metric}{head_extra}, "
                      f"CRPS={row['crps']:.3f}, "
                      f"Geweke F={row['geweke_F']:+.3f} (p={row['geweke_p']:.3f}), "
                      f"fit SRMR[{srmr_mark}]/RMSEA[{rmsea_mark}] {fit_str}, "
                      f"nnz={row['nnz']}, time={res_method['elapsed']:.1f}s")

print(f"\nTotal elapsed: {(time.time()-t_start_all)/60:.1f} min")

df = pd.DataFrame(all_results)
print(f"\nSaved {len(df)} rows to {results_path}")
print(df.head(20).to_string(index=False))


# §10. Aggregate across seeds and visualize


agg = df.groupby(['regime', 'method']).agg(
    delta_PR_diag_mean=('delta_PR_diag', 'mean'),
    delta_PR_diag_sd=('delta_PR_diag', 'std'),
    delta_LSC_mean=('delta_LSC', 'mean'),
    delta_LSC_sd=('delta_LSC', 'std'),
    delta_RC_mean=('delta_RC', 'mean'),
    delta_RC_sd=('delta_RC', 'std'),
    p_perm_mean=('p_perm', 'mean'),
    p_LSC_mean=('p_LSC', 'mean'),
    p_RC_mean=('p_RC', 'mean'),
    crps_mean=('crps', 'mean'),
    crps_sd=('crps', 'std'),
    n_seeds_ok=('error', lambda x: int(np.sum(pd.isna(x)))),
).reset_index()

print("=== Aggregated results (mean across seeds) ===")
print(agg.to_string(index=False))

agg_path = os.path.join(OUT_DIR, 'synthetic_results_aggregated.csv')
agg.to_csv(agg_path, index=False)
print(f"\nSaved -> {agg_path}")


# 2x2 figure: one panel per regime.
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
regimes_order = ['R1', "R1'", 'R3', 'R2', "R2'", "R3'"]
# (regime → (y_metric_col, y_metric_sd_col, y_label, p_col))
regime_y = {
    'R1':   ('delta_PR_diag_mean', 'delta_PR_diag_sd', 'delta_PR_diag', 'p_perm_mean'),
    "R1'":  ('delta_PR_diag_mean', 'delta_PR_diag_sd', 'delta_PR_diag', 'p_perm_mean'),
    'R2':   ('delta_LSC_mean',     'delta_LSC_sd',     'delta_LSC',     'p_LSC_mean'),
    "R2'":  ('delta_LSC_mean',     'delta_LSC_sd',     'delta_LSC',     'p_LSC_mean'),
    'R3':   ('delta_PR_diag_mean', 'delta_PR_diag_sd', 'delta_PR_diag', 'p_perm_mean'),
    "R3'":  ('delta_PR_diag_mean', 'delta_PR_diag_sd', 'delta_PR_diag', 'p_perm_mean'),
}
regime_titles = {
    'R1':   "R1: 1-factor reflective + nonlinear (tanh) latent dynamics\n"
            "y = ΔPR (Component A; rank-1 W̃* should fire)",
    "R1'":  "R1': 1-factor reflective + cubic latent dynamics\n"
            "(form-robustness for R1; same metric, different nonlinearity)",
    'R2':   "R2: 2-factor reflective + additive-tanh cross-coupling\n"
            "y = ΔLSC (Components A + B; tests Theorem 4 cross-block)",
    "R2'":  "R2': 2-factor reflective + multiplicative cross-coupling\n"
            "(form-robustness for R2; tests if NAVAR's edge is form-specific)",
    'R3':   "R3: covariance-formative (banded innovations, no shared latent)\n"
            "y = ΔPR (should NOT fire)",
    "R3'":  "R3': independent indicators (sanity check)\n"
            "y = ΔPR (should NOT fire)",
}
method_colors = {
    'linear_var_granger': '#1f77b4',
    'pcmci':              '#ff7f0e',
    'cmlp':               '#2ca02c',
    'navar':              '#d62728',
    'trivial_sparse':     '#7f7f7f',
}
for ax, regime in zip(axes.flat, regimes_order):
    sub = agg[agg['regime'] == regime]
    if regime not in regime_y:
        ax.set_visible(False)
        continue
    y_col, y_sd, y_label, p_col = regime_y[regime]
    silent_methods = []   # methods that produced empty W and got NaN downstream
    for _, r in sub.iterrows():
        # Detect "method is silent": NaN in the y-axis metric. PCMCI on R3
        # often falls here — it correctly finds no significant edges, so
        # W_hat=0 and CRPS / ΔPR are undefined. We record this as a
        # substantive result ("method correctly identified no structure")
        # rather than as missing data.
        if pd.isna(r['crps_mean']) or pd.isna(r[y_col]):
            silent_methods.append(r['method'])
            continue
        marker = 'D' if r['method'] == 'trivial_sparse' else 'o'
        ax.errorbar(r['crps_mean'], r[y_col],
                    xerr=r['crps_sd'], yerr=r[y_sd],
                    fmt=marker, color=method_colors.get(r['method'], 'black'),
                    markersize=11, capsize=4, alpha=0.85)
        # Annotate with method name + p-value
        p_val = r.get(p_col, np.nan)
        sig = '*' if (np.isfinite(p_val) and p_val < 0.05) else ''
        label = f"{r['method']}{sig}"
        ax.annotate(label, (r['crps_mean'], r[y_col]),
                    xytext=(7, 5), textcoords='offset points', fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--', alpha=0.6)
    ax.set_xlabel('CRPS (lower = better predictive accuracy)')
    ax.set_ylabel(rf"$\Delta\mathrm{{{y_label.replace('_', r'\_')}}}$ (higher = better)")
    ax.set_title(regime_titles[regime], fontsize=10)
    ax.grid(True, alpha=0.3)
    # Annotate silent methods in lower-left corner of the panel
    if silent_methods:
        msg = "Found 0 significant edges:\n  " + "\n  ".join(silent_methods)
        ax.text(0.02, 0.02, msg, transform=ax.transAxes, fontsize=8,
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow',
                          edgecolor='gray', alpha=0.8))
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'fig_synthetic_validation.png')
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"\nSaved figure -> {fig_path}")


print("\n" + "="*70)
print("LSC SYNTHETIC VALIDATION COMPLETE")
print("="*70)
print(f"Output dir: {OUT_DIR}")
print(f"  synthetic_results.csv             — every cell with ΔPR, p, CRPS")
print(f"  synthetic_results_aggregated.csv  — mean ± sd across seeds")
print(f"  fig_synthetic_validation.png      — 2x2 scatter of CRPS vs ΔPR")

### Cell §1.2 — Synthetic OLS VAR diagnostic (Layer-1 baseline)

**Paper sections:** §4, Appendix O.

**Produces:** Layer-1 substrate-diagnostic verdict for each synthetic regime — does the simplest linear estimator (OLS VAR) pass the framework's binary criterion (ΔPR > 0 AND p < 0.05)? If yes, the regime cannot discriminate methods because the rank-1 reflective signal is dominant enough that any reasonable lag-1 estimator picks it up. This is the "too easy" check for the V-Dem-realistic synthetic regimes.

**Procedure (per regime, seed=0):** generate panel via the regime's generator → build OLS VAR coefficient matrix (panel-safe) → run the same ΔPR + row-permutation-null pipeline used in the main cell at B=100 → compare to median discovery-method ΔPR from §1.1's `synthetic_results.csv`.

**Inputs:** Reads regime panels generated by Cell §1.1.

**Outputs:** Per-regime OLS VAR ΔPR + verdict CSV.

**Runtime:** ~1 minute on Colab Pro, G4 GPU.

In [1]:
# §1.2 — OLS VAR Table 1 row

# Unified protocol (matches the rest of Table 1's comparator-method numbers):
#   • Row-permuting null
#   • 5 seeds per regime
#   • B = 1000 permutations per seed
#   • component_PR_unrestricted: average rank-1 fit quality over groups of size >= 3
#
# IMPORTANT: R3 is run at rho_innov=0.7 (not the default 0.5).
#   The default rho_innov=0.5 produces marginal rank-1 signal that fires only at
#   1-2 of 5 seeds. To clearly demonstrate the formative-covariance failure mode
#   that motivates the (dPR, T_cov) conjunction, we strengthen R3's banded
#   innovation correlation to rho_innov=0.7. T_cov on R3 at rho_innov=0.7
#   remains in [0.62, 0.66] across seeds, still below tau=0.70, so the conjunction
#   correctly fails. The parameter choice is documented in the paper alongside
#   the original rho_innov=0.5 numbers as a sensitivity check.
#
# Output: lsc_synthetic_validation_outputs/ols_var_table1.csv
#   one row per (regime, seed), plus an aggregated summary printout.

from google.colab import drive
drive.mount('/content/drive')

import os, sys, time
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
SRC_DIR   = os.path.join(DRIVE_DIR, 'src')
OUT_DIR   = os.path.join(DRIVE_DIR, 'lsc_synthetic_validation_outputs')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")


# §1. Load regime generators from lsc_synthetic_validation.py
#
# We exec just the generator-definition block from the source file.
# This keeps generators in sync with the main notebook automatically and
# avoids the import side-effects of the full module.

U_UNITS, T_PER_UNIT = 139, 75


def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def _make_heteroskedastic_sigma(N, regime_seed_int, mean_sigma=0.3, ratio=5.0):
    rng = np.random.default_rng(regime_seed_int)
    log_sigma = rng.standard_normal(N)
    log_sigma = log_sigma - log_sigma.mean()
    spread = log_sigma.max() - log_sigma.min()
    if spread > 0:
        log_sigma = log_sigma * (np.log(ratio) / spread)
    return mean_sigma * np.exp(log_sigma)


def _make_per_unit_lambda(Lam_pop, U, regime_seed_int, sigma_lam=0.10):
    rng = np.random.default_rng(regime_seed_int)
    log_dev = rng.standard_normal((U, len(Lam_pop))) * sigma_lam
    return Lam_pop[None, :] * np.exp(log_dev)


# Pull the rest of the generators from the canonical source.
notebook_path = os.path.join(SRC_DIR, 'lsc_synthetic_validation.py')
fallback_path = '/content/drive/MyDrive/NeurIPS2026_1296/lsc_synthetic_validation.py'
for p in [notebook_path, fallback_path]:
    if os.path.exists(p):
        notebook_path = p
        break
else:
    raise FileNotFoundError(
        f"Cannot find lsc_synthetic_validation.py at {notebook_path} or {fallback_path}"
    )

print(f"Loading regime generators from: {notebook_path}")
with open(notebook_path) as f:
    nb_src = f.read()

# Exec generator section (def generate_R1 through end of section §7.5)
start = nb_src.index('def generate_R1(')
end = nb_src.index('# §8. Convenience: CRPS wrapper')
gen_code = nb_src[start:end]

from typing import List  # noqa: F401  (used by exec'd code)
exec(gen_code, globals())
loaded = [n for n in dir() if n.startswith('generate_')]
print(f"Loaded {len(loaded)} generators: {loaded}")


# §2. OLS VAR estimator (panel-safe), masked-diagonal rank-1 fit, row-permuting null

def _build_lag_pairs(Y, split_spec):
    """Build (Y_t, Y_{t-1}) panel-safe lag pairs."""
    Y = np.asarray(Y, dtype=float)
    Yt_list, Ylag_list = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_list.append(unit[1:])
            Ylag_list.append(unit[:-1])
    return np.vstack(Yt_list), np.vstack(Ylag_list)


def fit_ols_var_panel(Y, split_spec):
    """Panel OLS VAR(1): A solves min_A ||Y_t - A Y_{t-1}||_F^2.

    Returns A of shape (N, N) in target-on-rows convention with diagonal zeroed.
    Panel-safe: lag pairs do not cross unit boundaries.
    """
    Yt, Ylag = _build_lag_pairs(Y, split_spec)
    XtX = Ylag.T @ Ylag
    XtY = Ylag.T @ Yt
    try:
        A_T = np.linalg.solve(XtX, XtY)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(XtX) @ XtY
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


ALS_MAX_ITERS = 50
ALS_TOL = 1e-7


def offdiag_rank1_fit(W_block, max_iters=ALS_MAX_ITERS, tol=ALS_TOL, seed=0):
    """Masked-diagonal rank-1 ALS fit. Returns R = 1 - SS_resid / SS_total
    on the off-diagonal entries of W_block."""
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy()
    np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(seed)
        u = rng.standard_normal(m)
        v = rng.standard_normal(m)

    prev_res = np.inf
    residual = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v
        den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new
        den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def component_PR_unrestricted(W, groups, min_size=3):
    """Average rank-1 fit quality over groups of size >= min_size.

    Matches the convention used in lsc_synthetic_validation.py §1.1.
    On single-block regimes (R1, R3, etc.) this reduces to a single rank-1 fit;
    on multi-block regimes (R2, R2') this averages over the blocks.
    """
    Rs = []
    for g in groups:
        if g['size'] < min_size:
            continue
        idx = g['indices']
        W_block = W[np.ix_(idx, idx)].copy()
        np.fill_diagonal(W_block, 0.0)
        R = offdiag_rank1_fit(W_block)
        if not np.isnan(R):
            Rs.append(R)
    return float(np.mean(Rs)) if Rs else np.nan


def row_permute(W, rng):
    """Row-permuting null: for each row, shuffle off-diagonal column positions."""
    N = W.shape[0]
    W_perm = np.zeros_like(W)
    for i in range(N):
        offdiag_cols = [j for j in range(N) if j != i]
        permuted_cols = rng.permutation(offdiag_cols)
        for src_j, tgt_j in zip(offdiag_cols, permuted_cols):
            W_perm[i, tgt_j] = W[i, src_j]
    return W_perm


def compute_delta_pr(W, groups, B, seed):
    """Returns (delta_PR, p_perm, PR_obs, null_mean, null_sd)."""
    PR_obs = component_PR_unrestricted(W, groups)
    if np.isnan(PR_obs):
        return np.nan, np.nan, np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    null_PRs = []
    for _ in range(B):
        W_b = row_permute(W, rng)
        null_PRs.append(component_PR_unrestricted(W_b, groups))
    null = np.array([x for x in null_PRs if np.isfinite(x)])
    if len(null) == 0:
        return np.nan, np.nan, PR_obs, np.nan, np.nan
    mu = float(np.mean(null))
    sd = float(np.std(null))
    delta = PR_obs - mu
    p_perm = (1 + np.sum(null >= PR_obs)) / (1 + len(null))
    return float(delta), float(p_perm), float(PR_obs), mu, sd


# §3. Sweep configuration
#
# 13 regimes × 5 seeds × B = 1000 row-permuting null
# R3 uses rho_innov=0.7 (see header docstring); all others use defaults.

B_PERMS = 1000
N_SEEDS = 5

# Map regime name -> (generator, kwargs).
# The kwargs let us inject rho_innov=0.7 only for R3, leaving everything else default.
REGIMES = [
    # 6 baseline regimes (Table 1)
    ('R1',              generate_R1,              {}),
    ("R1'",             generate_R1prime,         {}),
    ('R2',              generate_R2,              {}),
    ("R2'",             generate_R2prime,         {}),
    ('R3',              generate_R3,              {'rho_innov': 0.7}),  # strengthened
    ("R3'",             generate_R3prime,         {}),
    # 7 V-Dem-realistic regimes (Appendix B)
    ('R1-hetero',       generate_R1_hetero,       {}),
    ('R1-bounded',      generate_R1_bounded,      {}),
    ('R1-mixed',        generate_R1_mixed,        {}),
    ('R1-heterogen',    generate_R1_heterogen,    {}),
    ('R1-vdemlike',     generate_R1_vdemlike,     {}),
    ('R1-contaminated', generate_R1_contaminated, {}),
    ('R1-dynsplit',     generate_R1_dynsplit,     {}),
]

print(f"\nRunning OLS VAR Table 1 sweep:")
print(f"  {len(REGIMES)} regimes × {N_SEEDS} seeds × B = {B_PERMS}")
print(f"  Null: row-permuting (matches §1.1 protocol)")
print(f"  Note: R3 uses rho_innov=0.7 (paper R3 spec, supplements default rho_innov=0.5)")
print()


# §4. Sweep

t_total = time.time()
rows = []

for regime_name, gen_fn, gen_kwargs in REGIMES:
    t_regime = time.time()
    deltas = []
    pvals = []
    for seed in range(N_SEEDS):
        panel = gen_fn(seed=seed, **gen_kwargs)
        Y = panel['Y']
        split_spec = panel['split_spec']
        groups_true = panel['groups_true']

        A_ols = fit_ols_var_panel(Y, split_spec)
        nnz = int(np.sum(np.abs(A_ols) > 1e-10))

        delta, p_perm, PR_obs, null_mu, null_sd = compute_delta_pr(
            A_ols, groups_true, B=B_PERMS, seed=42 + seed,
        )

        rows.append({
            'regime': regime_name,
            'method': 'ols_var',
            'seed': seed,
            'rho_innov': gen_kwargs.get('rho_innov', np.nan),  # NaN unless R3
            'PR_obs': PR_obs,
            'null_PR_mean': null_mu,
            'null_PR_sd': null_sd,
            'delta_PR_diag': delta,
            'p_perm': p_perm,
            'passes': bool(np.isfinite(delta) and delta > 0
                           and np.isfinite(p_perm) and p_perm < 0.05),
            'A_nnz': nnz,
            'B_perms': B_PERMS,
        })
        deltas.append(delta)
        pvals.append(p_perm)

    mean_d = float(np.mean(deltas))
    std_d  = float(np.std(deltas, ddof=1))
    n_pass = int(np.sum([p < 0.05 for p in pvals]))
    print(f"  {regime_name:18s} | dPR = {mean_d:+.4f} ± {std_d:.4f} | "
          f"pass = {n_pass}/{N_SEEDS} | {time.time() - t_regime:.1f}s")

print(f"\nTotal sweep time: {time.time() - t_total:.0f}s")


# §5. Save and summarize

df = pd.DataFrame(rows)

os.makedirs(OUT_DIR, exist_ok=True)
out_csv = os.path.join(OUT_DIR, 'ols_var_table1.csv')
df.to_csv(out_csv, index=False)
print(f"\nSaved per-seed: {out_csv}")

# Aggregate to mean ± std for table-ready output
agg = (df.groupby('regime')
         .agg(mean_dPR=('delta_PR_diag', 'mean'),
              std_dPR =('delta_PR_diag', 'std'),
              mean_PR_obs=('PR_obs', 'mean'),
              n_pass  =('passes', 'sum'),
              n_seeds =('seed', 'count'))
         .reset_index())

# Preserve regime ordering from the REGIMES list
ordering = {r[0]: i for i, r in enumerate(REGIMES)}
agg['_o'] = agg['regime'].map(ordering)
agg = agg.sort_values('_o').drop(columns='_o').reset_index(drop=True)

out_agg = os.path.join(OUT_DIR, 'ols_var_table1_aggregated.csv')
agg.to_csv(out_agg, index=False)
print(f"Saved aggregated: {out_agg}")


# §6. Print Table-1-ready output

print()
print("=" * 80)
print("TABLE 1 OLS VAR ROW (paper-ready)")
print("=" * 80)
print(f"{'Regime':18s} | {'mean dPR':>10s} | {'± std':>7s} | {'pass':>5s} | "
      f"{'PR_obs':>8s} | LaTeX")
print("-" * 80)
for _, r in agg.iterrows():
    latex = f"$+{r['mean_dPR']:.3f}_{{({r['n_pass']:d}/{r['n_seeds']:d})}}^{{\\pm{r['std_dPR']:.3f}}}$" \
            if r['mean_dPR'] >= 0 else \
            f"${r['mean_dPR']:.3f}_{{({r['n_pass']:d}/{r['n_seeds']:d})}}^{{\\pm{r['std_dPR']:.3f}}}$"
    print(f"{r['regime']:18s} | {r['mean_dPR']:>+10.4f} | {r['std_dPR']:>7.4f} | "
          f"{r['n_pass']:>2d}/{r['n_seeds']:d} | {r['mean_PR_obs']:>8.4f} | {latex}")

print()
print("Note: R3 is run at rho_innov=0.7 to robustly demonstrate the formative-")
print("covariance failure mode for Claim B. T_cov on R3 at rho_innov=0.7 remains")
print("below tau=0.70 (verified separately), so the conjunction correctly fails.")
print()
print("Done.")

Mounted at /content/drive
DRIVE_DIR = /content/drive/MyDrive/NeurIPS2026_1296
OUT_DIR   = /content/drive/MyDrive/NeurIPS2026_1296/lsc_synthetic_validation_outputs
Loading regime generators from: /content/drive/MyDrive/NeurIPS2026_1296/src/lsc_synthetic_validation.py
Loaded 13 generators: ['generate_R1', 'generate_R1_bounded', 'generate_R1_contaminated', 'generate_R1_dynsplit', 'generate_R1_hetero', 'generate_R1_heterogen', 'generate_R1_mixed', 'generate_R1_vdemlike', 'generate_R1prime', 'generate_R2', 'generate_R2prime', 'generate_R3', 'generate_R3prime']

Running OLS VAR Table 1 sweep:
  13 regimes × 5 seeds × B = 1000
  Null: row-permuting (matches §1.1 protocol)
  Note: R3 uses rho_innov=0.7 (paper R3 spec, supplements default rho_innov=0.5)

  R1                 | dPR = +0.0774 ± 0.0153 | pass = 5/5 | 0.9s
  R1'                | dPR = +0.0742 ± 0.0143 | pass = 5/5 | 0.8s
  R2                 | dPR = +0.3379 ± 0.0067 | pass = 5/5 | 2.4s
  R2'                | dPR = +0.4240 ± 0.0034 

###Cell §1.3 — R3 T_cov boundary diagnostic at rho_innov=0.7

This cell complements §1.2 (OLS VAR Table 1 row) by computing T_cov on R3
panels at the canonical rho_innov=0.7. §1.2 produces the headline dPR claim
(R3 OLS VAR dPR = +0.164 +/- 0.035 across 5 seeds); this cell produces the
T_cov range that locates R3 with respect to the conjunction threshold tau=0.7.

**Purpose:** At canonical R3 generator parameters (N=12, decay=1.0, rho_innov=0.7), the T_cov range is [0.684, 0.739] across 5 seeds (mean 0.7124). R3 straddles tau=0.7: the conjunction fails on 2 of 5 seeds at tau=0.7 and on all 5 seeds at tau >= 0.75.  This is the boundary-case framing used in §3 of the paper.

**Output:** r3_rho07_outputs/
-  r3_rho07_per_seed.csv  -- one row per seed
-  r3_rho07_summary.csv   -- 1-row summary (mean +/- std, p-range, T_cov range)

**Runtime:** ~ 1 minute on CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'r3_rho07_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")


# §1. Configuration

U_UNITS       = 139
T_PER_UNIT    = 75
N_SEEDS       = 5
B_PERMS       = 1000
ALS_MAX_ITERS = 50
ALS_TOL       = 1e-7

# R3 generator parameters (matches §1.1 / §1.2)
RHO_INNOV     = 0.7
DECAY         = 1.0
N_INDICATORS  = 12

print(f"R3 at rho_innov={RHO_INNOV}, decay={DECAY}, N={N_INDICATORS}")
print(f"  {N_SEEDS} seeds x B={B_PERMS} permutations per seed")
print(f"  Panel: U={U_UNITS} units, T={T_PER_UNIT} per unit")


# §2. R3 generator (verbatim from §1.1, cell 3)

def generate_R3(seed, U=U_UNITS, T=T_PER_UNIT,
                rho_innov=RHO_INNOV, decay=DECAY, N=N_INDICATORS):
    """R3: covariance-formative. Independent AR(1) with banded innovation
    correlation. Verbatim copy of the generator in §1.1."""
    rng = np.random.default_rng(seed)
    rhos = rng.uniform(0.7, 0.95, size=N)
    Sigma_eps = np.eye(N)
    if rho_innov > 0:
        for i in range(N):
            for j in range(N):
                if i != j:
                    Sigma_eps[i, j] = rho_innov ** (decay * abs(i - j))
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-9:
        Sigma_eps = 0.99 * Sigma_eps + 0.01 * np.eye(N)
    L = np.linalg.cholesky(Sigma_eps)
    unit_paths = []
    for u in range(U):
        y = np.zeros(N); path = np.zeros((T, N))
        for t in range(T):
            eps = L @ rng.standard_normal(N)
            y = rhos * y + eps
            path[t] = y
        unit_paths.append(path)
    Y = np.vstack(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    split_spec = [T] * U
    return Y, split_spec, list(range(N))


# §3. Panel OLS VAR + masked rank-1 ALS + row-permuting null
#     (verbatim from §1.2)

def _build_lag_pairs(Y, split_spec):
    Yt_list, Ylag_list = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_list.append(unit[1:])
            Ylag_list.append(unit[:-1])
    return np.vstack(Yt_list), np.vstack(Ylag_list)


def fit_ols_var_panel(Y, split_spec):
    """Panel OLS VAR(1), diagonal zeroed (matches §1.2 / paper convention)."""
    Yt, Ylag = _build_lag_pairs(Y, split_spec)
    XtX = Ylag.T @ Ylag
    XtY = Ylag.T @ Yt
    try:
        A_T = np.linalg.solve(XtX, XtY)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(XtX) @ XtY
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=ALS_MAX_ITERS, tol=ALS_TOL):
    """Masked-diagonal rank-1 ALS fit. Returns 1 - SS_resid / SS_total
    on the off-diagonal cells of W_block."""
    W = W_block.astype(float).copy()
    m = W.shape[0]
    if m < 2: return np.nan
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy()
    np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    residual = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v
        den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new
        den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def component_PR_unrestricted(W, groups, min_size=3):
    """Average rank-1 fit quality over groups of size >= min_size."""
    Rs = []
    for g in groups:
        if g['size'] < min_size: continue
        idx = g['indices']
        W_block = W[np.ix_(idx, idx)].copy()
        np.fill_diagonal(W_block, 0.0)
        R = offdiag_rank1_fit(W_block)
        if not np.isnan(R):
            Rs.append(R)
    return float(np.mean(Rs)) if Rs else np.nan


def cov_rank1_signature(panel_data, indices):
    """T_cov: rank-1 fit on the indicator correlation matrix.
    Verbatim from §1.1 cov_rank1_signature."""
    if len(indices) < 4:
        return np.nan
    sub = panel_data[:, indices]
    sub_std = (sub - sub.mean(0)) / (sub.std(0) + 1e-12)
    corr = np.corrcoef(sub_std, rowvar=False)
    R = offdiag_rank1_fit(corr.copy())
    return R if np.isfinite(R) else np.nan


def row_permute(W, rng):
    N = W.shape[0]
    W_perm = np.zeros_like(W)
    for i in range(N):
        offdiag_cols = [j for j in range(N) if j != i]
        permuted_cols = rng.permutation(offdiag_cols)
        for src_j, tgt_j in zip(offdiag_cols, permuted_cols):
            W_perm[i, tgt_j] = W[i, src_j]
    return W_perm


def compute_delta_pr(W, groups, B, seed):
    """Returns (delta, p_perm, PR_obs, null_mean, null_sd)."""
    PR_obs = component_PR_unrestricted(W, groups)
    if np.isnan(PR_obs):
        return np.nan, np.nan, np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    null_PRs = []
    for _ in range(B):
        W_b = row_permute(W, rng)
        null_PRs.append(component_PR_unrestricted(W_b, groups))
    null = np.array([x for x in null_PRs if np.isfinite(x)])
    if len(null) == 0:
        return np.nan, np.nan, PR_obs, np.nan, np.nan
    mu = float(np.mean(null))
    sd = float(np.std(null))
    delta = PR_obs - mu
    p_perm = (1 + np.sum(null >= PR_obs)) / (1 + len(null))
    return float(delta), float(p_perm), float(PR_obs), mu, sd


# §4. Sweep across N_SEEDS

print()
print("=" * 70)
print(f"R3 at rho_innov={RHO_INNOV}: dPR + T_cov per seed")
print("=" * 70)

t_start = time.time()
rows = []
for seed in range(N_SEEDS):
    t_seed = time.time()
    Y, split_spec, idx = generate_R3(seed=seed)
    groups = [{'indices': idx, 'names': [f'R3_v{i}' for i in idx],
               'size': len(idx), 'T': 0.0, 'F': 1.0}]

    T_cov = cov_rank1_signature(Y, idx)
    A = fit_ols_var_panel(Y, split_spec)
    nnz = int(np.sum(np.abs(A) > 1e-10))

    delta, p_perm, PR_obs, mu, sd = compute_delta_pr(
        A, groups, B=B_PERMS, seed=42 + seed,
    )
    passes = bool(np.isfinite(delta) and delta > 0
                  and np.isfinite(p_perm) and p_perm < 0.05)
    rows.append({
        'regime': 'R3',
        'method': 'ols_var',
        'rho_innov': RHO_INNOV,
        'seed': seed,
        'T_cov': T_cov,
        'PR_obs': PR_obs,
        'null_PR_mean': mu,
        'null_PR_sd': sd,
        'delta_PR_diag': delta,
        'p_perm': p_perm,
        'passes': passes,
        'A_nnz': nnz,
        'B_perms': B_PERMS,
    })
    print(f"  seed {seed}: T_cov={T_cov:.4f} | PR_obs={PR_obs:.4f} | "
          f"dPR={delta:+.4f} | p={p_perm:.4f} | "
          f"{'PASS' if passes else 'fail'} | {time.time()-t_seed:.1f}s")

print(f"\nTotal: {time.time()-t_start:.1f}s")


# §5. Save and summarize

per_seed = pd.DataFrame(rows)
per_seed_path = os.path.join(OUT_DIR, 'r3_rho07_per_seed.csv')
per_seed.to_csv(per_seed_path, index=False)
print(f"\nSaved per-seed: {per_seed_path}")

summary = pd.DataFrame([{
    'regime': 'R3',
    'method': 'ols_var',
    'rho_innov': RHO_INNOV,
    'n_seeds': len(per_seed),
    'mean_dPR': per_seed['delta_PR_diag'].mean(),
    'std_dPR': per_seed['delta_PR_diag'].std(ddof=1),
    'min_p_perm': per_seed['p_perm'].min(),
    'max_p_perm': per_seed['p_perm'].max(),
    'n_pass': int(per_seed['passes'].sum()),
    'min_T_cov': per_seed['T_cov'].min(),
    'max_T_cov': per_seed['T_cov'].max(),
    'mean_T_cov': per_seed['T_cov'].mean(),
    'mean_PR_obs': per_seed['PR_obs'].mean(),
}])
summary_path = os.path.join(OUT_DIR, 'r3_rho07_summary.csv')
summary.to_csv(summary_path, index=False)
print(f"Saved summary:  {summary_path}")


# §6. Print paper-ready output

print()
print("=" * 70)
print("Paper-ready summary (cite source: r3_rho07_outputs/)")
print("=" * 70)
s = summary.iloc[0]
print(f"  dPR (mean +/- std):  {s['mean_dPR']:+.4f} +/- {s['std_dPR']:.4f}")
print(f"  p-values:            [{s['min_p_perm']:.4f}, {s['max_p_perm']:.4f}]")
print(f"  Pass count:          {s['n_pass']}/{s['n_seeds']}")
print(f"  T_cov range:         [{s['min_T_cov']:.4f}, {s['max_T_cov']:.4f}]")
print(f"  T_cov mean:          {s['mean_T_cov']:.4f}")
print()
print("Conjunction at tau=0.7:")
n_above = int((per_seed['T_cov'] > 0.70).sum())
print(f"  T_cov > 0.7: {n_above}/{N_SEEDS} seeds")
print(f"  Layer 1 statistical pass (dPR>0, p<0.05): "
      f"{int(per_seed['passes'].sum())}/{N_SEEDS} seeds")
print(f"  Conjunction PASS (both): "
      f"{int(((per_seed['T_cov'] > 0.70) & per_seed['passes']).sum())}/{N_SEEDS}")
print(f"  At tau=0.75: T_cov > 0.75 holds for "
      f"{int((per_seed['T_cov'] > 0.75).sum())}/{N_SEEDS} seeds")

### Cell §1.4 — Synthetic aggregation block (smoothing): exploratory

**Paper sections:** Not in paper.

This cell tests whether temporal smoothing reproduces the V-Dem aggregation signature (T_cov preserved, ΔPR attenuated, β preserved). Results from this and the next two cells (§1.4, §1.5) were used during paper development to investigate the V-Dem failure mechanism but are not cited or referenced in the final paper. Retained for reproducibility of the development process.

**Inputs:** None (self-generates R1-vdemlike panels).

**Outputs:** Aggregation CSV under `synthetic_aggregation_outputs/`.

**Runtime:** ~1 minute on Colab Pro, G4 GPU.

In [ ]:
# Block 16: Synthetic aggregation experiment
# Tests the falsifiability of the continuous-diagnostic framing by
# applying a stylized aggregation pipeline to R1-vdemlike and tracing
# the (T_cov, ΔPR, z, β) signature as a function of aggregation severity.

from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR = os.path.join(DRIVE_DIR, 'synthetic_aggregation_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# §1. Configuration

# Panel dimensions matching V-Dem matched panel
U_UNITS = 89
T_PER_UNIT = 55
N_BLOCK = 12  # indicators in the reflective block (R1-vdemlike adds 3 singletons)

# Smoothing-window sweep
W_VALUES = [0, 1, 2, 3, 5, 7]

# Number of seeds per condition
N_SEEDS = 10

# Null draws per seed (reduced from 1000 to manage compute; calibration confirmed at B=1000)
B_PERMS = 300

# Rater-simulation parameters for EXP-2
N_RATERS = 5
SIGMA_RATER = 0.5

# β-clip for parametric null
BETA_CLIP = 0.99
ALPHA = 0.05

print(f"Panel: U={U_UNITS}, T={T_PER_UNIT}, N_block={N_BLOCK}")
print(f"Smoothing windows: {W_VALUES}")
print(f"Seeds per condition: {N_SEEDS}, null B={B_PERMS}")
print(f"Total OLS VAR fits: {2 * len(W_VALUES) * N_SEEDS * (1 + B_PERMS)}")

# §2. Pipeline utilities (matching era_subsets / Block 14)

def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        A[i, :] = beta[:N]
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    bc = np.clip(betas, -beta_clip, beta_clip)
    stat_var = sigmas**2 / (1 - bc**2 + 1e-12)
    stat_sd = np.sqrt(np.maximum(stat_var, 1e-12))
    x = rng.standard_normal((U, N)) * stat_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = bc[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_metrics(Y, split_spec, ref_indices, B, seed):
    """Compute T_cov, PR_obs, ΔPR, p, z, median β on the given panel."""
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)]
    PR_obs = offdiag_rank1_fit(W_block)

    # T_cov: cov-rank-1 ALS fit of indicator correlation
    Y_block = Y[:, ref_indices]
    C_obs = np.corrcoef(Y_block.T)
    T_cov = offdiag_rank1_fit(C_obs)

    # Persistence
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    median_beta = float(np.median(betas[ref_indices]))

    # Parametric null
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    U_, T_ = len(split_spec), split_spec[0]
    for b in range(B):
        Y_sim = simulate_ar1_panel(betas, sigmas, U_, T_, rng, beta_clip=BETA_CLIP)
        ss_sim = [T_] * U_
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim = A_sim[np.ix_(ref_indices, ref_indices)]
        null_PRs[b] = offdiag_rank1_fit(W_sim)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    z = float((PR_obs - mu) / sd) if sd > 0 else float('nan')
    return {
        'T_cov': float(T_cov),
        'PR_obs': float(PR_obs),
        'null_mean': mu, 'null_std': sd,
        'delta_PR': float(delta),
        'p_parametric': float(p),
        'z_score': z,
        'median_beta': median_beta,
        'verdict': 'PASS' if p < ALPHA else 'FAIL',
    }


# §3. R1-vdemlike generator (verbatim from lsc_synthetic_validation.py)

def _make_heteroskedastic_sigma(N_block, regime_seed_int, mean_sigma=0.3, ratio=5.0):
    rng = np.random.default_rng(regime_seed_int)
    sigma_min = mean_sigma * 2 / (1 + ratio)
    sigma_max = ratio * sigma_min
    sigmas = rng.uniform(sigma_min, sigma_max, size=N_block)
    return sigmas

def _make_per_unit_lambda(Lam_pop, U, regime_seed_int, sigma_lam=0.10):
    rng = np.random.default_rng(regime_seed_int)
    log_dev = rng.normal(0, sigma_lam, size=(U, len(Lam_pop)))
    return Lam_pop[None, :] * np.exp(log_dev)

def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def generate_R1_vdemlike(seed, U=U_UNITS, T=T_PER_UNIT,
                         rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                         mean_sigma=0.3, sigma_ratio=5.0,
                         sigma_lam=0.10, burn_in=50,
                         n_singletons=3, rho_singleton=0.7,
                         weak_loading=0.2, sigma_singleton=0.5,
                         standardize=True):
    rng = np.random.default_rng(seed)
    N_block = 12
    N = N_block + n_singletons
    Lam_pop = rng.uniform(0.6, 0.9, size=N_block)
    sigma_eps = _make_heteroskedastic_sigma(N_block, regime_seed_int=2026005,
                                            mean_sigma=mean_sigma, ratio=sigma_ratio)
    Lam_u = _make_per_unit_lambda(Lam_pop, U, regime_seed_int=2026006,
                                  sigma_lam=sigma_lam)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        s_state = rng.normal(0, sigma_singleton / np.sqrt(1 - rho_singleton**2),
                             size=n_singletons)
        path = np.zeros((T, N))
        Lam_this_unit = Lam_u[u]
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            z_block = Lam_this_unit * eta + rng.normal(0, sigma_eps)
            path[t, :N_block] = np.tanh(z_block)
            s_state = (rho_singleton * s_state
                       + weak_loading * eta
                       + rng.normal(0, sigma_singleton, size=n_singletons))
            path[t, N_block:] = np.tanh(s_state)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    if standardize:
        Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, split_spec, N_block


# §4. Aggregation primitives

def temporal_smooth(Y, split_spec, w):
    """Apply moving-average smoothing of (half-)width w within each unit's
    time series. w=0 returns Y unchanged. w=1 averages each value with one
    neighbor on each side (3-point MA). Edge handling: shrink window at
    edges (1-sided averaging at t=0 and t=T-1)."""
    if w == 0:
        return Y.copy()
    Y_smooth = np.zeros_like(Y)
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        T = unit.shape[0]
        for t in range(T):
            t0 = max(0, t - w)
            t1 = min(T, t + w + 1)
            Y_smooth[cursor + t] = unit[t0:t1].mean(axis=0)
        cursor += n_u
    return Y_smooth


def rater_simulate_aggregate(Y, n_raters, sigma_rater, rng):
    """Simulate K noisy rater views per indicator value, then average back
    to a single estimate. Net effect: adds noise of variance σ²/K to each
    cell. Variance reduction by averaging is implicit; no structural change."""
    noise = rng.normal(0, sigma_rater / np.sqrt(n_raters), size=Y.shape)
    return Y + noise


# §5. Run experiments

def run_experiment(experiment_name, smoothing_widths, with_raters,
                    n_seeds, U, T, N_block, B):
    """Run an aggregation-severity sweep.

    For each smoothing window w in smoothing_widths:
      - Generate n_seeds R1-vdemlike panels
      - Optionally simulate raters + aggregate (with_raters=True)
      - Apply temporal smoothing of width w
      - Re-standardize per indicator (matches the framework's intake)
      - Compute (T_cov, PR_obs, ΔPR, z, p, β) on the 12-indicator block
    Returns list of result dicts.
    """
    print(f"\n{'='*70}\n{experiment_name}\n{'='*70}")
    results = []
    ref_indices = np.arange(N_block)  # eval the rank-1 metric on the block, ignoring singletons
    for w in smoothing_widths:
        print(f"\n--- Smoothing width w={w} ---")
        per_seed = []
        for seed in range(n_seeds):
            Y_raw, split_spec, _ = generate_R1_vdemlike(
                seed=seed, U=U, T=T, standardize=False
            )
            # Apply aggregation pipeline
            rng_pipe = np.random.default_rng(10000 + seed)
            if with_raters:
                Y_pipe = rater_simulate_aggregate(Y_raw, N_RATERS, SIGMA_RATER, rng_pipe)
            else:
                Y_pipe = Y_raw.copy()
            Y_pipe = temporal_smooth(Y_pipe, split_spec, w)
            # Re-standardize per indicator (framework intake convention)
            Y_pipe = (Y_pipe - Y_pipe.mean(0)) / (Y_pipe.std(0) + 1e-12)
            # Compute metrics
            t0 = time.time()
            m = compute_metrics(Y_pipe, split_spec, ref_indices,
                                 B=B, seed=20000 + seed)
            elapsed = time.time() - t0
            per_seed.append(m)
            print(f"  seed {seed:>2d}: T_cov={m['T_cov']:.4f}  ΔPR={m['delta_PR']:+.4f}  "
                  f"z={m['z_score']:+.2f}  p={m['p_parametric']:.4f}  β={m['median_beta']:.3f}  "
                  f"({elapsed:.1f}s)")
        # Aggregate across seeds
        agg = {
            'experiment': experiment_name,
            'smoothing_w': w,
            'n_seeds': len(per_seed),
            'T_cov_mean': np.mean([r['T_cov'] for r in per_seed]),
            'T_cov_std':  np.std([r['T_cov'] for r in per_seed]),
            'PR_obs_mean': np.mean([r['PR_obs'] for r in per_seed]),
            'delta_PR_mean': np.mean([r['delta_PR'] for r in per_seed]),
            'delta_PR_std':  np.std([r['delta_PR'] for r in per_seed]),
            'z_mean': np.mean([r['z_score'] for r in per_seed]),
            'z_std':  np.std([r['z_score'] for r in per_seed]),
            'p_mean': np.mean([r['p_parametric'] for r in per_seed]),
            'beta_mean': np.mean([r['median_beta'] for r in per_seed]),
            'beta_std':  np.std([r['median_beta'] for r in per_seed]),
            'pass_count': sum(r['verdict'] == 'PASS' for r in per_seed),
            'per_seed': per_seed,
        }
        print(f"  AGG: T_cov={agg['T_cov_mean']:.4f}±{agg['T_cov_std']:.4f}  "
              f"ΔPR={agg['delta_PR_mean']:+.4f}±{agg['delta_PR_std']:.4f}  "
              f"z={agg['z_mean']:+.2f}  β={agg['beta_mean']:.3f}  "
              f"PASS {agg['pass_count']}/{n_seeds}")
        results.append(agg)
    return results


# Run EXP-1: smoothing only
exp1 = run_experiment(
    experiment_name='EXP-1: smoothing only',
    smoothing_widths=W_VALUES,
    with_raters=False,
    n_seeds=N_SEEDS,
    U=U_UNITS, T=T_PER_UNIT, N_block=N_BLOCK,
    B=B_PERMS,
)

# Run EXP-2: full pipeline (raters + aggregation + smoothing)
exp2 = run_experiment(
    experiment_name='EXP-2: full pipeline (raters + smoothing)',
    smoothing_widths=W_VALUES,
    with_raters=True,
    n_seeds=N_SEEDS,
    U=U_UNITS, T=T_PER_UNIT, N_block=N_BLOCK,
    B=B_PERMS,
)

# §6. Summary tables and falsifiability check

def print_summary(results, label):
    print(f"\n{label}")
    print(f"{'w':>3s}  {'T_cov':>8s}  {'ΔPR':>8s}  {'z':>6s}  {'β':>6s}  {'PASS':>6s}  {'ΔPR/ΔPR_w0':>12s}  {'z/z_w0':>8s}")
    print("-" * 70)
    delta_PR_w0 = results[0]['delta_PR_mean']
    z_w0 = results[0]['z_mean']
    for r in results:
        ratio_dpr = r['delta_PR_mean'] / delta_PR_w0 if delta_PR_w0 > 0 else float('nan')
        ratio_z = r['z_mean'] / z_w0 if z_w0 > 0 else float('nan')
        print(f"{r['smoothing_w']:>3d}  {r['T_cov_mean']:>8.4f}  "
              f"{r['delta_PR_mean']:>+8.4f}  {r['z_mean']:>+6.2f}  "
              f"{r['beta_mean']:>6.3f}  {r['pass_count']:>3d}/{r['n_seeds']:>2d}  "
              f"{ratio_dpr:>12.3f}  {ratio_z:>8.3f}")

print_summary(exp1, "EXP-1: smoothing only — aggregation severity sweep")
print_summary(exp2, "EXP-2: full pipeline (raters + smoothing) — aggregation severity sweep")

# §7. Falsifiability verdict

print("\n" + "=" * 70)
print("FALSIFIABILITY CHECK")
print("=" * 70)

print("""
V-Dem matched-panel observed pattern (target):
  Indicators (raw):     T_cov=0.999, ΔPR=+0.233, z=+5.43, β=0.981
  Indices (aggregated): T_cov=0.987, ΔPR=+0.160, z=+2.50, β=0.982
  Ratios (indices/indicators):
    T_cov ratio:  0.99 (preserved)
    ΔPR ratio:    0.69 (~30% attenuation)
    z ratio:      0.46 (~50% attenuation)
    β ratio:      1.00 (preserved)
""")

def check_pattern(results, label):
    """Test whether the experiment reproduces the V-Dem pattern."""
    base = results[0]
    print(f"\n{label}:")
    print(f"  Best matches to V-Dem pattern (ΔPR ratio ≈ 0.69, z ratio ≈ 0.46):")
    for r in results[1:]:
        ratio_dpr = r['delta_PR_mean'] / base['delta_PR_mean']
        ratio_z = r['z_mean'] / base['z_mean']
        ratio_tcov = r['T_cov_mean'] / base['T_cov_mean']
        ratio_beta = r['beta_mean'] / base['beta_mean']
        print(f"    w={r['smoothing_w']:>2d}: ΔPR ratio={ratio_dpr:.3f}, z ratio={ratio_z:.3f}, "
              f"T_cov ratio={ratio_tcov:.3f}, β ratio={ratio_beta:.3f}")

    # Falsifiability checks
    print(f"\n  FALSIFIABILITY:")

    # Check 1: ΔPR decreases monotonically with w
    dprs = [r['delta_PR_mean'] for r in results]
    monotone_dpr = all(dprs[i] >= dprs[i+1] for i in range(len(dprs)-1))
    print(f"    [1] ΔPR monotonically decreasing with w: "
          f"{'PASS' if monotone_dpr else 'FAIL'}")
    if not monotone_dpr:
        violations = [(results[i]['smoothing_w'], dprs[i], dprs[i+1])
                      for i in range(len(dprs)-1) if dprs[i] < dprs[i+1]]
        print(f"        violations: {violations}")

    # Check 2: T_cov roughly preserved (drop < 2x ΔPR drop)
    tcov_drop = abs(results[-1]['T_cov_mean'] - results[0]['T_cov_mean'])
    dpr_drop = abs(results[-1]['delta_PR_mean'] - results[0]['delta_PR_mean'])
    tcov_preserved = (tcov_drop < 2 * dpr_drop)
    print(f"    [2] T_cov drop ({tcov_drop:.4f}) < 2× ΔPR drop ({dpr_drop:.4f}): "
          f"{'PASS' if tcov_preserved else 'FAIL'}")

    # Check 3: β roughly preserved (change < 0.05)
    beta_change = abs(results[-1]['beta_mean'] - results[0]['beta_mean'])
    beta_preserved = (beta_change < 0.05)
    print(f"    [3] β change ({beta_change:.3f}) < 0.05: "
          f"{'PASS' if beta_preserved else 'FAIL'}")

    # Check 4: z attenuates faster than ΔPR
    z_ratio_at_max_w = results[-1]['z_mean'] / base['z_mean'] if base['z_mean'] > 0 else float('nan')
    dpr_ratio_at_max_w = results[-1]['delta_PR_mean'] / base['delta_PR_mean'] if base['delta_PR_mean'] > 0 else float('nan')
    z_faster = z_ratio_at_max_w < dpr_ratio_at_max_w
    print(f"    [4] z ratio at max w ({z_ratio_at_max_w:.3f}) < ΔPR ratio ({dpr_ratio_at_max_w:.3f}): "
          f"{'PASS' if z_faster else 'FAIL'}")

    # Reproducibility of V-Dem pattern: any w in [0.5, 0.85] for ΔPR ratio?
    matching_w = [r['smoothing_w'] for r in results
                  if 0.5 <= r['delta_PR_mean']/base['delta_PR_mean'] <= 0.85]
    print(f"    [5] Some w produces ΔPR ratio in V-Dem range [0.50, 0.85]: "
          f"{'PASS' if matching_w else 'FAIL'} (matching w: {matching_w})")

    return {
        'monotone_dpr': monotone_dpr,
        'tcov_preserved': tcov_preserved,
        'beta_preserved': beta_preserved,
        'z_faster': z_faster,
        'reproduces_vdem_range': bool(matching_w),
        'matching_w_values': matching_w,
    }

verdict_exp1 = check_pattern(exp1, "EXP-1 verdict")
verdict_exp2 = check_pattern(exp2, "EXP-2 verdict")

print("""
INTERPRETATION:
  - All 5 checks PASS in EXP-1: smoothing alone reproduces the V-Dem
    aggregation signature. Continuous-diagnostic framing well-supported.
  - All 5 checks PASS in EXP-2: full pipeline works too. Smoothing is
    the dominant primitive; rater simulation adds nothing new.
  - Most checks PASS, but some borderline: framing partially supported,
    discuss the failures.
  - Multiple checks FAIL: framing not supported by the synthetic test.
    Need to revise.
""")

# Save
output = {
    'config': {
        'U': U_UNITS, 'T': T_PER_UNIT, 'N_block': N_BLOCK,
        'smoothing_widths': W_VALUES,
        'n_seeds': N_SEEDS, 'B_perms': B_PERMS,
        'rater_K': N_RATERS, 'sigma_rater': SIGMA_RATER,
    },
    'exp1_smoothing_only': exp1,
    'exp2_full_pipeline': exp2,
    'falsifiability_verdict_exp1': verdict_exp1,
    'falsifiability_verdict_exp2': verdict_exp2,
    'vdem_target_pattern': {
        'T_cov_ratio': 0.99,
        'delta_PR_ratio': 0.69,
        'z_ratio': 0.46,
        'beta_ratio': 1.00,
    },
}

# Strip per-seed details before json.dump (they're verbose)
out_for_json = json.loads(json.dumps(output, default=str))
out_path = os.path.join(OUT_DIR, 'synthetic_aggregation_results.json')
with open(out_path, 'w') as f:
    json.dump(out_for_json, f, indent=2)
print(f"\nResults saved to {out_path}")

### Cell §1.5 — Synthetic aggregation block (shrinkage primitives): exploratory

**Paper sections:** Not in paper.

Tests three shrinkage primitives (within-country, cross-country, hierarchical FE+trend) as candidate aggregation mechanisms for the V-Dem signature. All five primitives across this cell and §1.3 produced ~7-10% ΔPR attenuation, much weaker than V-Dem's ~30%, with side-effects on β that V-Dem does not show.

Tests within-country shrinkage, cross-country shrinkage, and hierarchical (FE + trend) shrinkage against the V-Dem matched-panel attenuation signature. Linear, information-preserving primitives are insufficient. Retained for reproducibility of the development process; not cited in the paper.

**Inputs:** None.

**Outputs:** Aggregation CSV under `synthetic_aggregation_v2_outputs/`.

**Runtime:** ~1 minutes on Colab Pro, T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR = os.path.join(DRIVE_DIR, 'synthetic_aggregation_v2_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# §1. Configuration

U_UNITS = 89
T_PER_UNIT = 55
N_BLOCK = 12

LAMBDA_VALUES = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
N_SEEDS = 10
B_PERMS = 300
BETA_CLIP = 0.99
ALPHA = 0.05

# V-Dem matched-panel target ratios
VDEM_TARGET = {
    'T_cov_ratio': 0.99,
    'delta_PR_ratio': 0.69,
    'z_ratio': 0.46,
    'beta_ratio': 1.00,
}

print(f"Panel: U={U_UNITS}, T={T_PER_UNIT}, N_block={N_BLOCK}")
print(f"λ sweep: {LAMBDA_VALUES}")
print(f"Seeds per condition: {N_SEEDS}, null B={B_PERMS}")
print(f"V-Dem target: ΔPR ratio ≈ 0.69, z ratio ≈ 0.46, T_cov & β preserved")
print(f"Total OLS VAR fits: {3 * len(LAMBDA_VALUES) * N_SEEDS * (1 + B_PERMS)}")


# §2. Pipeline utilities (same as Block 16)

def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        A[i, :] = beta[:N]
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    bc = np.clip(betas, -beta_clip, beta_clip)
    stat_var = sigmas**2 / (1 - bc**2 + 1e-12)
    stat_sd = np.sqrt(np.maximum(stat_var, 1e-12))
    x = rng.standard_normal((U, N)) * stat_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = bc[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_metrics(Y, split_spec, ref_indices, B, seed):
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)]
    PR_obs = offdiag_rank1_fit(W_block)
    Y_block = Y[:, ref_indices]
    C_obs = np.corrcoef(Y_block.T)
    T_cov = offdiag_rank1_fit(C_obs)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    median_beta = float(np.median(betas[ref_indices]))
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    U_, T_ = len(split_spec), split_spec[0]
    for b in range(B):
        Y_sim = simulate_ar1_panel(betas, sigmas, U_, T_, rng, beta_clip=BETA_CLIP)
        ss_sim = [T_] * U_
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim = A_sim[np.ix_(ref_indices, ref_indices)]
        null_PRs[b] = offdiag_rank1_fit(W_sim)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    z = float((PR_obs - mu) / sd) if sd > 0 else float('nan')
    return {
        'T_cov': float(T_cov),
        'PR_obs': float(PR_obs),
        'null_mean': mu, 'null_std': sd,
        'delta_PR': float(delta),
        'p_parametric': float(p),
        'z_score': z,
        'median_beta': median_beta,
        'verdict': 'PASS' if p < ALPHA else 'FAIL',
    }


# §3. R1-vdemlike generator (verbatim, same as Block 16)

def _make_heteroskedastic_sigma(N_block, regime_seed_int, mean_sigma=0.3, ratio=5.0):
    rng = np.random.default_rng(regime_seed_int)
    sigma_min = mean_sigma * 2 / (1 + ratio)
    sigma_max = ratio * sigma_min
    return rng.uniform(sigma_min, sigma_max, size=N_block)

def _make_per_unit_lambda(Lam_pop, U, regime_seed_int, sigma_lam=0.10):
    rng = np.random.default_rng(regime_seed_int)
    log_dev = rng.normal(0, sigma_lam, size=(U, len(Lam_pop)))
    return Lam_pop[None, :] * np.exp(log_dev)

def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def generate_R1_vdemlike(seed, U=U_UNITS, T=T_PER_UNIT,
                         rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                         mean_sigma=0.3, sigma_ratio=5.0,
                         sigma_lam=0.10, burn_in=50,
                         n_singletons=3, rho_singleton=0.7,
                         weak_loading=0.2, sigma_singleton=0.5,
                         standardize=True):
    rng = np.random.default_rng(seed)
    N_block = 12
    N = N_block + n_singletons
    Lam_pop = rng.uniform(0.6, 0.9, size=N_block)
    sigma_eps = _make_heteroskedastic_sigma(N_block, regime_seed_int=2026005,
                                            mean_sigma=mean_sigma, ratio=sigma_ratio)
    Lam_u = _make_per_unit_lambda(Lam_pop, U, regime_seed_int=2026006,
                                  sigma_lam=sigma_lam)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        s_state = rng.normal(0, sigma_singleton / np.sqrt(1 - rho_singleton**2),
                             size=n_singletons)
        path = np.zeros((T, N))
        Lam_this_unit = Lam_u[u]
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            z_block = Lam_this_unit * eta + rng.normal(0, sigma_eps)
            path[t, :N_block] = np.tanh(z_block)
            s_state = (rho_singleton * s_state
                       + weak_loading * eta
                       + rng.normal(0, sigma_singleton, size=n_singletons))
            path[t, N_block:] = np.tanh(s_state)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    if standardize:
        Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, split_spec, N_block


# §4. Three shrinkage primitives

def shrink_within_country(Y, split_spec, lam):
    """y'_{u,t,i} = (1-λ) y_{u,t,i} + λ y_bar_{u,·,i}.
    Shrinks each year toward the country's mean over all years."""
    if lam == 0.0:
        return Y.copy()
    Y_out = Y.copy()
    cursor = 0
    for n_u in split_spec:
        unit_slice = slice(cursor, cursor + n_u)
        country_mean = Y[unit_slice].mean(axis=0, keepdims=True)
        Y_out[unit_slice] = (1 - lam) * Y[unit_slice] + lam * country_mean
        cursor += n_u
    return Y_out


def shrink_cross_country(Y, split_spec, lam):
    """y'_{u,t,i} = (1-λ) y_{u,t,i} + λ y_bar_{·,·,i}.
    Shrinks every country-year toward the global indicator mean."""
    if lam == 0.0:
        return Y.copy()
    global_mean = Y.mean(axis=0, keepdims=True)
    return (1 - lam) * Y + lam * global_mean


def shrink_hierarchical(Y, split_spec, lam):
    """Fit y_{u,t,i} = α_{u,i} + β_i · t, then shrink toward fitted values:
    y'_{u,t,i} = (1-λ) y_{u,t,i} + λ · ŷ_{u,t,i}.
    Country fixed effects + global linear trend per indicator."""
    if lam == 0.0:
        return Y.copy()
    N = Y.shape[1]
    # Build per-row time index and per-row country index
    cursor = 0
    t_per_row = np.zeros(Y.shape[0])
    country_idx = np.zeros(Y.shape[0], dtype=int)
    for u, n_u in enumerate(split_spec):
        t_per_row[cursor:cursor + n_u] = np.arange(n_u)
        country_idx[cursor:cursor + n_u] = u
        cursor += n_u
    n_countries = len(split_spec)

    # Per-indicator fit: country FE + global linear t
    Y_hat = np.zeros_like(Y)
    for i in range(N):
        # Design: [country dummies (no intercept), t]
        # Equivalent: y = α_u + β·t  =>  for each obs r, y_r = α_{country_idx[r]} + β·t_per_row[r]
        # Solve via per-country demeaning + OLS for β:
        country_means_y = np.array([Y[country_idx == u, i].mean() for u in range(n_countries)])
        country_means_t = np.array([t_per_row[country_idx == u].mean() for u in range(n_countries)])
        y_dem = Y[:, i] - country_means_y[country_idx]
        t_dem = t_per_row - country_means_t[country_idx]
        denom = np.sum(t_dem ** 2)
        if denom > 1e-12:
            beta_t = np.sum(t_dem * y_dem) / denom
        else:
            beta_t = 0.0
        # Country-specific intercept absorbs the rest:
        alpha_u = country_means_y - beta_t * country_means_t
        Y_hat[:, i] = alpha_u[country_idx] + beta_t * t_per_row

    return (1 - lam) * Y + lam * Y_hat


# §5. Run experiments

def run_test(test_name, shrink_fn, lambda_values, n_seeds, U, T, N_block, B):
    print(f"\n{'='*70}\n{test_name}\n{'='*70}")
    results = []
    ref_indices = np.arange(N_block)
    for lam in lambda_values:
        print(f"\n--- λ={lam:.2f} ---")
        per_seed = []
        for seed in range(n_seeds):
            Y_raw, split_spec, _ = generate_R1_vdemlike(
                seed=seed, U=U, T=T, standardize=False
            )
            # Apply shrinkage primitive
            Y_shrunk = shrink_fn(Y_raw, split_spec, lam)
            # Re-standardize per indicator (framework intake convention)
            Y_shrunk = (Y_shrunk - Y_shrunk.mean(0)) / (Y_shrunk.std(0) + 1e-12)
            # Compute metrics
            t0 = time.time()
            m = compute_metrics(Y_shrunk, split_spec, ref_indices,
                                 B=B, seed=20000 + seed)
            elapsed = time.time() - t0
            per_seed.append(m)
            print(f"  seed {seed:>2d}: T_cov={m['T_cov']:.4f}  ΔPR={m['delta_PR']:+.4f}  "
                  f"z={m['z_score']:+.2f}  β={m['median_beta']:.3f}  ({elapsed:.1f}s)")
        agg = {
            'test_name': test_name,
            'lambda': lam,
            'n_seeds': len(per_seed),
            'T_cov_mean': float(np.mean([r['T_cov'] for r in per_seed])),
            'T_cov_std':  float(np.std([r['T_cov'] for r in per_seed])),
            'PR_obs_mean': float(np.mean([r['PR_obs'] for r in per_seed])),
            'delta_PR_mean': float(np.mean([r['delta_PR'] for r in per_seed])),
            'delta_PR_std':  float(np.std([r['delta_PR'] for r in per_seed])),
            'z_mean': float(np.mean([r['z_score'] for r in per_seed])),
            'z_std':  float(np.std([r['z_score'] for r in per_seed])),
            'p_mean': float(np.mean([r['p_parametric'] for r in per_seed])),
            'beta_mean': float(np.mean([r['median_beta'] for r in per_seed])),
            'beta_std':  float(np.std([r['median_beta'] for r in per_seed])),
            'pass_count': int(sum(r['verdict'] == 'PASS' for r in per_seed)),
            'per_seed': per_seed,
        }
        print(f"  AGG: T_cov={agg['T_cov_mean']:.4f}  ΔPR={agg['delta_PR_mean']:+.4f}  "
              f"z={agg['z_mean']:+.2f}  β={agg['beta_mean']:.3f}  "
              f"PASS {agg['pass_count']}/{n_seeds}")
        results.append(agg)
    return results


test_a = run_test('Test A: within-country shrinkage', shrink_within_country,
                   LAMBDA_VALUES, N_SEEDS, U_UNITS, T_PER_UNIT, N_BLOCK, B_PERMS)
test_b = run_test('Test B: cross-country shrinkage', shrink_cross_country,
                   LAMBDA_VALUES, N_SEEDS, U_UNITS, T_PER_UNIT, N_BLOCK, B_PERMS)
test_c = run_test('Test C: hierarchical (country FE + trend) shrinkage',
                   shrink_hierarchical,
                   LAMBDA_VALUES, N_SEEDS, U_UNITS, T_PER_UNIT, N_BLOCK, B_PERMS)

# §6. Summary tables

def print_summary(results, label):
    print(f"\n{label}")
    print(f"{'λ':>5s}  {'T_cov':>8s}  {'ΔPR':>9s}  {'z':>7s}  {'β':>6s}  {'PASS':>6s}  "
          f"{'ΔPR/λ0':>8s}  {'z/λ0':>7s}  {'T_cov/λ0':>9s}  {'β/λ0':>7s}")
    print("-" * 95)
    base = results[0]
    for r in results:
        ratio_dpr = r['delta_PR_mean'] / base['delta_PR_mean'] if base['delta_PR_mean'] != 0 else float('nan')
        ratio_z = r['z_mean'] / base['z_mean'] if base['z_mean'] != 0 else float('nan')
        ratio_tcov = r['T_cov_mean'] / base['T_cov_mean'] if base['T_cov_mean'] != 0 else float('nan')
        ratio_beta = r['beta_mean'] / base['beta_mean'] if base['beta_mean'] != 0 else float('nan')
        print(f"{r['lambda']:>5.2f}  {r['T_cov_mean']:>8.4f}  "
              f"{r['delta_PR_mean']:>+9.4f}  {r['z_mean']:>+7.2f}  "
              f"{r['beta_mean']:>6.3f}  {r['pass_count']:>3d}/{r['n_seeds']:>2d}  "
              f"{ratio_dpr:>8.3f}  {ratio_z:>7.3f}  {ratio_tcov:>9.4f}  {ratio_beta:>7.3f}")

print_summary(test_a, "Test A — Within-country shrinkage")
print_summary(test_b, "Test B — Cross-country shrinkage")
print_summary(test_c, "Test C — Hierarchical shrinkage")


# §7. Check against V-Dem signature

print("\n" + "=" * 70)
print("V-DEM TARGET vs SYNTHETIC SHRINKAGE TESTS")
print("=" * 70)
print(f"V-Dem target: T_cov ratio≈0.99, ΔPR ratio≈0.69, z ratio≈0.46, β ratio≈1.00")
print()


def match_score(results, target):
    """For each λ, compute distance between (T_cov, ΔPR, z, β) ratios and V-Dem target.
    Returns the λ with smallest L2 distance and the per-component ratio at that λ."""
    base = results[0]
    if base['delta_PR_mean'] <= 0 or base['z_mean'] <= 0 or base['T_cov_mean'] <= 0 or base['beta_mean'] <= 0:
        return None
    best = None
    for r in results[1:]:
        rdpr = r['delta_PR_mean'] / base['delta_PR_mean']
        rz = r['z_mean'] / base['z_mean']
        rtc = r['T_cov_mean'] / base['T_cov_mean']
        rb = r['beta_mean'] / base['beta_mean']
        # Distance in 4-dim ratio space (weighted: ΔPR & z most important)
        d = ((rdpr - target['delta_PR_ratio'])**2 +
             (rz - target['z_ratio'])**2 +
             0.5 * (rtc - target['T_cov_ratio'])**2 +
             0.5 * (rb - target['beta_ratio'])**2)
        if best is None or d < best['distance']:
            best = {'lambda': r['lambda'],
                    'delta_PR_ratio': rdpr, 'z_ratio': rz,
                    'T_cov_ratio': rtc, 'beta_ratio': rb,
                    'distance': d,
                    'pass_count': r['pass_count']}
    return best


def evaluate_test(results, label):
    best = match_score(results, VDEM_TARGET)
    if best is None:
        print(f"\n{label}: COULD NOT EVALUATE (degenerate base case)")
        return {'label': label, 'matches_vdem': False, 'best': None}
    print(f"\n{label}")
    print(f"  Best-matching λ: {best['lambda']:.2f}")
    print(f"    ΔPR ratio:   {best['delta_PR_ratio']:.3f}  (target {VDEM_TARGET['delta_PR_ratio']:.2f})")
    print(f"    z ratio:     {best['z_ratio']:.3f}  (target {VDEM_TARGET['z_ratio']:.2f})")
    print(f"    T_cov ratio: {best['T_cov_ratio']:.4f} (target {VDEM_TARGET['T_cov_ratio']:.2f})")
    print(f"    β ratio:     {best['beta_ratio']:.3f}  (target {VDEM_TARGET['beta_ratio']:.2f})")
    print(f"    L2 distance: {best['distance']:.4f}")
    print(f"    PASS: {best['pass_count']}/{N_SEEDS} (V-Dem indices on matched panel: PASS)")

    # Tolerances: each ratio within ±0.10 of target
    in_range = lambda x, tgt, tol=0.10: abs(x - tgt) <= tol
    matches_dpr   = in_range(best['delta_PR_ratio'], VDEM_TARGET['delta_PR_ratio'])
    matches_z     = in_range(best['z_ratio'], VDEM_TARGET['z_ratio'])
    matches_tcov  = in_range(best['T_cov_ratio'], VDEM_TARGET['T_cov_ratio'], tol=0.05)
    matches_beta  = in_range(best['beta_ratio'], VDEM_TARGET['beta_ratio'], tol=0.05)
    n_matches = sum([matches_dpr, matches_z, matches_tcov, matches_beta])

    print(f"  Matches V-Dem signature on individual axes:")
    print(f"    ΔPR ratio:   {'✓' if matches_dpr else '✗'}")
    print(f"    z ratio:     {'✓' if matches_z else '✗'}")
    print(f"    T_cov ratio: {'✓' if matches_tcov else '✗'}")
    print(f"    β ratio:     {'✓' if matches_beta else '✗'}")
    print(f"  Total axes matching: {n_matches}/4")

    matches_overall = (n_matches >= 3 and matches_dpr and matches_z)
    print(f"  Overall verdict: "
          f"{'MATCHES V-Dem signature' if matches_overall else 'does NOT match V-Dem signature'}")
    return {
        'label': label,
        'best_lambda': best['lambda'],
        'best_ratios': {'ΔPR': best['delta_PR_ratio'], 'z': best['z_ratio'],
                        'T_cov': best['T_cov_ratio'], 'β': best['beta_ratio']},
        'l2_distance': best['distance'],
        'axes_matching': n_matches,
        'matches_dpr': matches_dpr,
        'matches_z': matches_z,
        'matches_tcov': matches_tcov,
        'matches_beta': matches_beta,
        'matches_overall': matches_overall,
    }


verdict_a = evaluate_test(test_a, "Test A — Within-country shrinkage")
verdict_b = evaluate_test(test_b, "Test B — Cross-country shrinkage")
verdict_c = evaluate_test(test_c, "Test C — Hierarchical (FE + trend) shrinkage")

print("\n" + "=" * 70)
print("OVERALL VERDICT")
print("=" * 70)
matchers = [v for v in [verdict_a, verdict_b, verdict_c] if v.get('matches_overall')]
if len(matchers) == 1:
    v = matchers[0]
    print(f"Scenario 1/2/3: ONE primitive matches → {v['label']}.")
    print(f"  V-Dem-like signature reproduced at λ={v['best_lambda']:.2f}.")
    print(f"  Mechanism candidate identified.")
elif len(matchers) > 1:
    print(f"Scenario 4: MULTIPLE primitives match ({[v['label'] for v in matchers]}).")
    print(f"  V-Dem signature is non-specific. Cannot uniquely identify mechanism.")
else:
    print("Scenario 5: NONE of the three primitives reproduces the V-Dem signature.")
    print("  Standard linear shrinkage primitives are insufficient.")
    print("  Hypothesis space narrows toward NONLINEAR mechanisms")
    print("  (nonlinear IRT, thresholding/discretization, uncertainty-weighted aggregation).")
    print("  This is a stronger negative result than Block 16 alone.")

# Save
output = {
    'config': {
        'U': U_UNITS, 'T': T_PER_UNIT, 'N_block': N_BLOCK,
        'lambda_values': LAMBDA_VALUES,
        'n_seeds': N_SEEDS, 'B_perms': B_PERMS,
    },
    'vdem_target': VDEM_TARGET,
    'test_a_within_country': test_a,
    'test_b_cross_country': test_b,
    'test_c_hierarchical': test_c,
    'verdict_a': verdict_a,
    'verdict_b': verdict_b,
    'verdict_c': verdict_c,
}
out_path = os.path.join(OUT_DIR, 'synthetic_aggregation_v2_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

### Cell §1.6 — Synthetic aggregation block (discretization): exploratory

**Paper sections:** Not in paper.

Tests a nonlinear, information-destroying primitive (discretization to K-level uniform bins, mimicking ordinal expert coding). Discretization quantizes within-country temporal differences (the lag-1 signal that ΔPR detects) but cross-sectional differences between countries are typically larger than the discretization quantum, so cross-sectional T_cov survives. Pattern matches V-Dem qualitatively. Retained for reproducibility; not cited in the paper.

**Inputs:** None.

**Outputs:** Aggregation CSV under `synthetic_aggregation_v3_outputs/`.

**Runtime:** ~1 minute on Colab Pro, G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR = os.path.join(DRIVE_DIR, 'synthetic_aggregation_v3_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# §1. Configuration

U_UNITS = 89
T_PER_UNIT = 55
N_BLOCK = 12

K_VALUES = [3, 4, 5, 7, 10, 20]
N_SEEDS = 10
B_PERMS = 300
BETA_CLIP = 0.99
ALPHA = 0.05

VDEM_TARGET = {
    'T_cov_ratio': 0.99,
    'delta_PR_ratio': 0.69,
    'z_ratio': 0.46,
    'beta_ratio': 1.00,
}

print(f"Panel: U={U_UNITS}, T={T_PER_UNIT}, N_block={N_BLOCK}")
print(f"K (discretization levels) sweep: {K_VALUES}")
print(f"Seeds per condition: {N_SEEDS}, null B={B_PERMS}")
print(f"V-Dem target: ΔPR ratio≈0.69, z ratio≈0.46, T_cov & β preserved")
print(f"Total OLS VAR fits: {len(K_VALUES) * N_SEEDS * (1 + B_PERMS)}")


# §2. Pipeline utilities (same as Blocks 16/17)

def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        A[i, :] = beta[:N]
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    bc = np.clip(betas, -beta_clip, beta_clip)
    stat_var = sigmas**2 / (1 - bc**2 + 1e-12)
    stat_sd = np.sqrt(np.maximum(stat_var, 1e-12))
    x = rng.standard_normal((U, N)) * stat_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = bc[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_metrics(Y, split_spec, ref_indices, B, seed):
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)]
    PR_obs = offdiag_rank1_fit(W_block)
    Y_block = Y[:, ref_indices]
    C_obs = np.corrcoef(Y_block.T)
    T_cov = offdiag_rank1_fit(C_obs)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    median_beta = float(np.median(betas[ref_indices]))
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    U_, T_ = len(split_spec), split_spec[0]
    for b in range(B):
        Y_sim = simulate_ar1_panel(betas, sigmas, U_, T_, rng, beta_clip=BETA_CLIP)
        ss_sim = [T_] * U_
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim = A_sim[np.ix_(ref_indices, ref_indices)]
        null_PRs[b] = offdiag_rank1_fit(W_sim)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    z = float((PR_obs - mu) / sd) if sd > 0 else float('nan')
    return {
        'T_cov': float(T_cov),
        'PR_obs': float(PR_obs),
        'null_mean': mu, 'null_std': sd,
        'delta_PR': float(delta),
        'p_parametric': float(p),
        'z_score': z,
        'median_beta': median_beta,
        'verdict': 'PASS' if p < ALPHA else 'FAIL',
    }


# §3. R1-vdemlike generator (verbatim from Block 16/17)

def _make_heteroskedastic_sigma(N_block, regime_seed_int, mean_sigma=0.3, ratio=5.0):
    rng = np.random.default_rng(regime_seed_int)
    sigma_min = mean_sigma * 2 / (1 + ratio)
    sigma_max = ratio * sigma_min
    return rng.uniform(sigma_min, sigma_max, size=N_block)

def _make_per_unit_lambda(Lam_pop, U, regime_seed_int, sigma_lam=0.10):
    rng = np.random.default_rng(regime_seed_int)
    log_dev = rng.normal(0, sigma_lam, size=(U, len(Lam_pop)))
    return Lam_pop[None, :] * np.exp(log_dev)

def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def generate_R1_vdemlike(seed, U=U_UNITS, T=T_PER_UNIT,
                         rho=0.95, sigma_zeta=0.6, tanh_scale=2.0,
                         mean_sigma=0.3, sigma_ratio=5.0,
                         sigma_lam=0.10, burn_in=50,
                         n_singletons=3, rho_singleton=0.7,
                         weak_loading=0.2, sigma_singleton=0.5,
                         standardize=True):
    rng = np.random.default_rng(seed)
    N_block = 12
    N = N_block + n_singletons
    Lam_pop = rng.uniform(0.6, 0.9, size=N_block)
    sigma_eps = _make_heteroskedastic_sigma(N_block, regime_seed_int=2026005,
                                            mean_sigma=mean_sigma, ratio=sigma_ratio)
    Lam_u = _make_per_unit_lambda(Lam_pop, U, regime_seed_int=2026006,
                                  sigma_lam=sigma_lam)
    unit_paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        s_state = rng.normal(0, sigma_singleton / np.sqrt(1 - rho_singleton**2),
                             size=n_singletons)
        path = np.zeros((T, N))
        Lam_this_unit = Lam_u[u]
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            z_block = Lam_this_unit * eta + rng.normal(0, sigma_eps)
            path[t, :N_block] = np.tanh(z_block)
            s_state = (rho_singleton * s_state
                       + weak_loading * eta
                       + rng.normal(0, sigma_singleton, size=n_singletons))
            path[t, N_block:] = np.tanh(s_state)
        unit_paths.append(path)
    Y, split_spec = _stack_units(unit_paths)
    if standardize:
        Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, split_spec, N_block

# §4. Discretization primitive

def discretize_to_K_levels(Y, K):
    """Round each indicator's values to one of K uniformly-spaced bins
    on the indicator's empirical [min, max] range.

    K=3:  very coarse (3 levels, e.g., low/mid/high)
    K=4:  matches typical V-Dem rater coding (0-3 ordinal scale)
    K=5:  matches typical V-Dem rater coding (0-4 ordinal scale)
    K=20: essentially continuous (minimal information loss)

    The bin midpoints are returned (not bin indices) so that
    standardization downstream behaves naturally.
    """
    Y_out = np.zeros_like(Y)
    N = Y.shape[1]
    for i in range(N):
        yi = Y[:, i]
        ymin, ymax = yi.min(), yi.max()
        if ymax - ymin < 1e-12:
            Y_out[:, i] = yi
            continue
        # K bins of equal width on [ymin, ymax]
        # Bin midpoints: ymin + (k + 0.5) * width for k=0..K-1
        width = (ymax - ymin) / K
        # Assign each value to its bin (clipping the upper edge)
        bin_idx = np.clip(np.floor((yi - ymin) / width).astype(int), 0, K - 1)
        midpoints = ymin + (bin_idx + 0.5) * width
        Y_out[:, i] = midpoints
    return Y_out


# §5. Run discretization sweep

def run_discretization_sweep(K_values, n_seeds, U, T, N_block, B):
    print(f"\n{'='*70}\nTest D: Discretization\n{'='*70}")
    results = []
    ref_indices = np.arange(N_block)
    for K in K_values:
        print(f"\n--- K={K} bins ---")
        per_seed = []
        for seed in range(n_seeds):
            Y_raw, split_spec, _ = generate_R1_vdemlike(
                seed=seed, U=U, T=T, standardize=False
            )
            Y_disc = discretize_to_K_levels(Y_raw, K)
            # Re-standardize per indicator (framework intake convention)
            Y_disc = (Y_disc - Y_disc.mean(0)) / (Y_disc.std(0) + 1e-12)
            t0 = time.time()
            m = compute_metrics(Y_disc, split_spec, ref_indices,
                                 B=B, seed=20000 + seed)
            elapsed = time.time() - t0
            per_seed.append(m)
            print(f"  seed {seed:>2d}: T_cov={m['T_cov']:.4f}  ΔPR={m['delta_PR']:+.4f}  "
                  f"z={m['z_score']:+.2f}  β={m['median_beta']:.3f}  ({elapsed:.1f}s)")
        agg = {
            'test_name': 'discretization',
            'K': K,
            'n_seeds': len(per_seed),
            'T_cov_mean': float(np.mean([r['T_cov'] for r in per_seed])),
            'T_cov_std':  float(np.std([r['T_cov'] for r in per_seed])),
            'PR_obs_mean': float(np.mean([r['PR_obs'] for r in per_seed])),
            'delta_PR_mean': float(np.mean([r['delta_PR'] for r in per_seed])),
            'delta_PR_std':  float(np.std([r['delta_PR'] for r in per_seed])),
            'z_mean': float(np.mean([r['z_score'] for r in per_seed])),
            'z_std':  float(np.std([r['z_score'] for r in per_seed])),
            'p_mean': float(np.mean([r['p_parametric'] for r in per_seed])),
            'beta_mean': float(np.mean([r['median_beta'] for r in per_seed])),
            'beta_std':  float(np.std([r['median_beta'] for r in per_seed])),
            'pass_count': int(sum(r['verdict'] == 'PASS' for r in per_seed)),
            'per_seed': per_seed,
        }
        print(f"  AGG: T_cov={agg['T_cov_mean']:.4f}  ΔPR={agg['delta_PR_mean']:+.4f}  "
              f"z={agg['z_mean']:+.2f}  β={agg['beta_mean']:.3f}  "
              f"PASS {agg['pass_count']}/{n_seeds}")
        results.append(agg)
    return results


# K=20 is the closest-to-continuous baseline (essentially no information loss).
# We use it as the reference for ratio computations because:
#   - K=∞ would be the true baseline but would require an extra run
#   - K=20 has negligible attenuation (verifiable post-hoc)
# Run K=20 first, then sweep down.
results = run_discretization_sweep(K_values=K_VALUES,
                                    n_seeds=N_SEEDS,
                                    U=U_UNITS, T=T_PER_UNIT, N_block=N_BLOCK,
                                    B=B_PERMS)

# §6. Summary

# Find the K=20 row as the reference (least-discretized condition)
ref_idx = next(i for i, r in enumerate(results) if r['K'] == 20)
ref = results[ref_idx]

print("\n" + "=" * 100)
print(f"DISCRETIZATION SWEEP — using K=20 as reference (≈continuous baseline)")
print("=" * 100)
print(f"{'K':>3s}  {'T_cov':>8s}  {'ΔPR':>9s}  {'z':>7s}  {'β':>6s}  {'PASS':>6s}  "
      f"{'ΔPR/K20':>8s}  {'z/K20':>7s}  {'T_cov/K20':>10s}  {'β/K20':>7s}")
print("-" * 100)
for r in results:
    rdpr = r['delta_PR_mean'] / ref['delta_PR_mean'] if ref['delta_PR_mean'] != 0 else float('nan')
    rz = r['z_mean'] / ref['z_mean'] if ref['z_mean'] != 0 else float('nan')
    rtc = r['T_cov_mean'] / ref['T_cov_mean'] if ref['T_cov_mean'] != 0 else float('nan')
    rb = r['beta_mean'] / ref['beta_mean'] if ref['beta_mean'] != 0 else float('nan')
    print(f"{r['K']:>3d}  {r['T_cov_mean']:>8.4f}  "
          f"{r['delta_PR_mean']:>+9.4f}  {r['z_mean']:>+7.2f}  "
          f"{r['beta_mean']:>6.3f}  {r['pass_count']:>3d}/{r['n_seeds']:>2d}  "
          f"{rdpr:>8.3f}  {rz:>7.3f}  {rtc:>10.4f}  {rb:>7.3f}")


# §7. Falsifiability

print("\n" + "=" * 70)
print("V-DEM TARGET vs DISCRETIZATION SWEEP")
print("=" * 70)
print(f"V-Dem target: T_cov ratio≈0.99, ΔPR ratio≈0.69, z ratio≈0.46, β ratio≈1.00")
print()


def evaluate_test(results, ref, target):
    """Find the K with smallest L2 distance to V-Dem target ratios.
    Match criteria (pre-committed):
      ΔPR ratio in [0.55, 0.80]
      z ratio in [0.35, 0.60]
      T_cov ratio > 0.95
      β ratio in [0.90, 1.10]
    Match overall: ≥3 of 4 axes match AND both ΔPR and z match.
    """
    if ref['delta_PR_mean'] <= 0 or ref['z_mean'] <= 0:
        return None
    best = None
    for r in results:
        if r['K'] == ref['K']:
            continue
        rdpr = r['delta_PR_mean'] / ref['delta_PR_mean']
        rz = r['z_mean'] / ref['z_mean']
        rtc = r['T_cov_mean'] / ref['T_cov_mean']
        rb = r['beta_mean'] / ref['beta_mean']
        d = ((rdpr - target['delta_PR_ratio'])**2 +
             (rz - target['z_ratio'])**2 +
             0.5 * (rtc - target['T_cov_ratio'])**2 +
             0.5 * (rb - target['beta_ratio'])**2)
        if best is None or d < best['distance']:
            best = {'K': r['K'],
                    'delta_PR_ratio': rdpr, 'z_ratio': rz,
                    'T_cov_ratio': rtc, 'beta_ratio': rb,
                    'distance': d, 'pass_count': r['pass_count']}
    if best is None:
        return None

    # Pre-committed match criteria
    matches_dpr = (0.55 <= best['delta_PR_ratio'] <= 0.80)
    matches_z = (0.35 <= best['z_ratio'] <= 0.60)
    matches_tcov = (best['T_cov_ratio'] > 0.95)
    matches_beta = (0.90 <= best['beta_ratio'] <= 1.10)
    n_matches = sum([matches_dpr, matches_z, matches_tcov, matches_beta])

    matches_overall = (n_matches >= 3 and matches_dpr and matches_z)

    print(f"Best-matching K: {best['K']}")
    print(f"  ΔPR ratio:   {best['delta_PR_ratio']:.3f}  (target {target['delta_PR_ratio']:.2f}; "
          f"in-range [0.55, 0.80]: {'✓' if matches_dpr else '✗'})")
    print(f"  z ratio:     {best['z_ratio']:.3f}  (target {target['z_ratio']:.2f}; "
          f"in-range [0.35, 0.60]: {'✓' if matches_z else '✗'})")
    print(f"  T_cov ratio: {best['T_cov_ratio']:.4f} (target {target['T_cov_ratio']:.2f}; "
          f">0.95: {'✓' if matches_tcov else '✗'})")
    print(f"  β ratio:     {best['beta_ratio']:.3f}  (target {target['beta_ratio']:.2f}; "
          f"in-range [0.90, 1.10]: {'✓' if matches_beta else '✗'})")
    print(f"  L2 distance: {best['distance']:.4f}")
    print(f"  Total axes matching: {n_matches}/4")
    print(f"  Overall verdict: "
          f"{'MATCHES V-Dem signature' if matches_overall else 'does NOT match V-Dem signature'}")

    return {
        'best_K': best['K'],
        'best_ratios': {'ΔPR': best['delta_PR_ratio'], 'z': best['z_ratio'],
                        'T_cov': best['T_cov_ratio'], 'β': best['beta_ratio']},
        'l2_distance': best['distance'],
        'axes_matching': n_matches,
        'matches_dpr': matches_dpr,
        'matches_z': matches_z,
        'matches_tcov': matches_tcov,
        'matches_beta': matches_beta,
        'matches_overall': matches_overall,
    }


verdict = evaluate_test(results, ref, VDEM_TARGET)

# §8. Pre-committed protocol decision

print("\n" + "=" * 70)
print("PRE-COMMITTED PROTOCOL DECISION")
print("=" * 70)

if verdict is not None and verdict['matches_overall']:
    print(f"""
Discretization MATCHES V-Dem signature at K={verdict['best_K']}.

Mechanism candidate identified: nonlinear quantization of indicator values
reproduces the V-Dem matched-panel attenuation pattern (T_cov preserved,
ΔPR attenuated by ~30%, β preserved).

Block 19 (composition test) NOT NEEDED. Discretization alone is sufficient
to reproduce the signature, so testing composition would be redundant.

Pre-committed next step: write up §6.3 with discretization as the
identified candidate mechanism. Substantively this fits with V-Dem's
data-generation pipeline: experts code indicators on ordinal scales
(typically K=4 or K=5), then the IRT model aggregates these ordinal
codings. The ordinal coding step is the source of the structural
attenuation our framework detects.
""")
else:
    print(f"""
Discretization does NOT match V-Dem signature.

Best K-value is K={verdict['best_K'] if verdict else 'N/A'}, but
{verdict['axes_matching']}/4 axes match (need ≥3 with ΔPR and z both in range).

This eliminates discretization as a single-primitive candidate. Combined
with the linear primitives ruled out in Blocks 16-17, we have now
falsified six candidate primitives:

  - Temporal smoothing (Block 16)
  - Rater-noise + smoothing (Block 16)
  - Within-country shrinkage (Block 17)
  - Cross-country shrinkage (Block 17)
  - Hierarchical FE+trend shrinkage (Block 17)
  - Discretization (Block 18)

Pre-committed next step: run Block 19 testing COMPOSITION of three
primitives (temporal smoothing + within-country shrinkage + discretization
applied in sequence). This tests whether the V-Dem effect is compositional
rather than from any single primitive. After Block 19, stop and write up
regardless of outcome.
""")

# Save results
output = {
    'config': {
        'U': U_UNITS, 'T': T_PER_UNIT, 'N_block': N_BLOCK,
        'K_values': K_VALUES,
        'n_seeds': N_SEEDS, 'B_perms': B_PERMS,
        'reference_K': 20,
    },
    'vdem_target': VDEM_TARGET,
    'discretization_results': results,
    'verdict': verdict,
}
out_path = os.path.join(OUT_DIR, 'synthetic_aggregation_v3_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

## Section 2 — §3.2 inference + Appendix K (calibration)

Calibration evidence for the row-permuting null and the parametric AR(1) null on synthetic null DGPs. These cells produce the Type-I rate calibration tables and the persistence-zone evidence cited in §3.2 and detailed in Appendix K. The application of the calibrated null to V-Dem itself is in §4.11.

### Cell §2.1 — Literal-null DGP test

**Paper sections:** §3.2, Appendix K.

Tests whether the row-permuting null delivers Type I control by simulating from a literally null DGP (white noise) and an independent-AR(1)-null DGP and checking ΔPR's null distribution under OLS VAR. Three conditions on synthetic panels (U=139, T=75, N=12): (1) white-noise null — the true ΔPR null DGP, expected Type I = 5%; (2) independent AR(1) at β=0.95 — tests whether self-persistence alone induces apparent rank-1 structure; (3) R1 positive control — expected Type I = 100%.

**Inputs:** None.

**Outputs:** `literal_null_results.csv`, `literal_null_summary.csv`, calibration figure.

**Runtime:** ~1 minute on Colab Pro, G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'literal_null_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")


# §1. Configuration

# Panel dimensions matching V-Dem long_16var (so the test is in the regime
# where the V-Dem analysis lives).
U_UNITS, T_PER_UNIT = 139, 75
N_INDICATORS        = 12       # match R1's 12-indicator block

# Test parameters
N_PANELS_PER_COND   = 100      # number of independent panel draws per condition
B_PERMS             = 1000     # row-permutation null draws per panel
ALPHA               = 0.05     # significance threshold for Type I rate

# AR(1) persistence for Condition 2 — match V-Dem's empirical median (~0.99
# from Phase 1 diagnostic), but use 0.95 for stability and to match R1.
AR1_BETA            = 0.95

# Positive control (R1) parameters — match the original generate_R1
R1_RHO              = 0.95     # latent persistence
R1_SIGMA_ZETA       = 0.6
R1_TANH_SCALE       = 2.0
R1_LOAD_LOW         = 0.7
R1_LOAD_HIGH        = 0.9
R1_NOISE_SD         = 0.3
R1_BURN_IN          = 50

print(f"Panel dims: U={U_UNITS}, T={T_PER_UNIT}, N={N_INDICATORS}")
print(f"Panels per condition: {N_PANELS_PER_COND}")
print(f"Permutations per panel: {B_PERMS}")
print(f"AR(1) β for Condition 2: {AR1_BETA}")


# §2. DGPs for the three conditions

def gen_white_noise(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    """Condition 1: pure i.i.d. Gaussian noise. No structure of any kind."""
    rng = np.random.default_rng(seed)
    Y = rng.standard_normal(size=(U * T, N))
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def gen_independent_ar1(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS,
                       beta=AR1_BETA):
    """Condition 2: each indicator independently AR(1) with β=0.95.
    No cross-variable dependence, no latent factor."""
    rng = np.random.default_rng(seed)
    paths = []
    for u in range(U):
        # Burn in each indicator independently
        x = np.zeros(N)
        for _ in range(50):
            x = beta * x + rng.standard_normal(N)
        unit = np.zeros((T, N))
        for t in range(T):
            x = beta * x + rng.standard_normal(N)
            unit[t] = x
        paths.append(unit)
    Y = np.vstack(paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def gen_R1(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    """Condition 3: positive control — R1 single-factor reflective."""
    rng = np.random.default_rng(seed)
    Lam = rng.uniform(R1_LOAD_LOW, R1_LOAD_HIGH, size=N)
    paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(R1_BURN_IN):
            eta = R1_RHO * np.tanh(R1_TANH_SCALE * eta) + rng.normal(0, R1_SIGMA_ZETA)
        unit = np.zeros((T, N))
        for t in range(T):
            eta = R1_RHO * np.tanh(R1_TANH_SCALE * eta) + rng.normal(0, R1_SIGMA_ZETA)
            unit[t] = Lam * eta + rng.normal(0, R1_NOISE_SD, size=N)
        paths.append(unit)
    Y = np.vstack(paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U

# §3. OLS VAR estimator and ΔPR computation
# Same code as in synthetic_olsvar_diagnostic.py, included inline for
# self-containment.

def fit_ols_var(Y, split_spec):
    """Fit pooled OLS VAR. Returns coefficient matrix A (target on rows),
    diagonal zeroed."""
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:])
            Yl_l.append(unit[:-1])
    Yt  = np.vstack(Yt_l)
    Ylag = np.vstack(Yl_l)
    XtX = Ylag.T @ Ylag
    XtY = Ylag.T @ Yt
    try:
        A_T = np.linalg.solve(XtX, XtY)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(XtX) @ XtY
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A

def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    """Off-diagonal rank-one fit via SVD initialization + ALS refinement.
    Returns PR ∈ [0, 1]."""
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v
        den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new
        den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    R = max(0.0, min(1.0, 1.0 - residual / tss))
    return R


def compute_delta_pr(W, ref_indices, B=B_PERMS, seed=0):
    """ΔPR with row-permuting null. Returns (delta, p_perm, PR_obs, null_mean, null_sd)."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)

    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    N = W.shape[0]
    for b in range(B):
        # Row-permuting null on the FULL W (matches main pipeline)
        W_perm = np.zeros_like(W)
        for i in range(N):
            offdiag_cols = np.array([j for j in range(N) if j != i])
            permuted_cols = rng.permutation(offdiag_cols)
            for src_j, tgt_j in zip(offdiag_cols, permuted_cols):
                W_perm[i, tgt_j] = W[i, src_j]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)

    valid = np.isfinite(null_PRs)
    if not valid.any():
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    null_PRs = null_PRs[valid]
    mu = float(np.mean(null_PRs))
    sd = float(np.std(null_PRs))
    delta = PR_obs - mu
    p_perm = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p_perm), float(PR_obs), mu, sd)

# §4. Run the three conditions

ref_indices = list(range(N_INDICATORS))  # full block as one reflective group

CONDITIONS = {
    'C1_white_noise':       gen_white_noise,
    'C2_independent_ar1':   gen_independent_ar1,
    'C3_R1_positive_ctrl':  gen_R1,
}

results_rows = []
t_start_total = time.time()

for cond_name, gen_fn in CONDITIONS.items():
    print(f"\n{'='*70}")
    print(f"CONDITION: {cond_name}")
    print(f"{'='*70}")
    t_cond = time.time()
    for seed in range(N_PANELS_PER_COND):
        Y, split_spec = gen_fn(seed=seed)
        A = fit_ols_var(Y, split_spec)
        # Each panel uses a different permutation seed for independence
        delta, p_perm, PR_obs, null_mu, null_sd = compute_delta_pr(
            A, ref_indices, B=B_PERMS, seed=seed + 10000)
        results_rows.append({
            'condition':    cond_name,
            'panel_seed':   seed,
            'delta_PR':     delta,
            'p_perm':       p_perm,
            'PR_obs':       PR_obs,
            'null_mean':    null_mu,
            'null_sd':      null_sd,
            'rejects_05':   bool(np.isfinite(delta) and delta > 0
                                 and np.isfinite(p_perm) and p_perm < ALPHA),
        })
        if (seed + 1) % 25 == 0:
            elapsed = time.time() - t_cond
            print(f"  seed {seed+1}/{N_PANELS_PER_COND} | elapsed {elapsed:.1f}s")
    elapsed = time.time() - t_cond
    print(f"  CONDITION TIME: {elapsed:.1f}s")

print(f"\nTOTAL TIME: {time.time() - t_start_total:.1f}s")

df_results = pd.DataFrame(results_rows)
csv_path = os.path.join(OUT_DIR, 'literal_null_results.csv')
df_results.to_csv(csv_path, index=False)
print(f"\nSaved per-panel results: {csv_path}")

# §5. Summary statistics and Type I error

print(f"\n{'='*70}")
print(f"SUMMARY ACROSS {N_PANELS_PER_COND} PANELS PER CONDITION")
print(f"{'='*70}")
print(f"{'Condition':24s} | {'mean ΔPR':>10s} | {'sd ΔPR':>8s} | "
      f"{'mean p':>8s} | {'reject@.05':>10s} | {'95% CI':>16s}")
print("-" * 90)

summary_rows = []
for cond_name in CONDITIONS:
    sub = df_results[df_results['condition'] == cond_name]
    n = len(sub)
    mean_delta = sub['delta_PR'].mean()
    sd_delta   = sub['delta_PR'].std()
    mean_p     = sub['p_perm'].mean()
    n_rej      = sub['rejects_05'].sum()
    rej_rate   = n_rej / n
    # Wilson 95% CI for binomial proportion
    z = 1.96
    p_hat = rej_rate
    denom = 1 + z**2/n
    center = (p_hat + z**2/(2*n)) / denom
    halfw = z * np.sqrt((p_hat * (1 - p_hat) / n) + (z**2 / (4*n**2))) / denom
    ci_low  = max(0.0, center - halfw)
    ci_high = min(1.0, center + halfw)
    print(f"{cond_name:24s} | {mean_delta:+10.4f} | {sd_delta:8.4f} | "
          f"{mean_p:8.3f} | {n_rej:3d}/{n:3d} ({rej_rate*100:4.1f}%) | "
          f"[{ci_low*100:4.1f}%, {ci_high*100:4.1f}%]")
    summary_rows.append({
        'condition':       cond_name,
        'n_panels':        n,
        'mean_delta_PR':   mean_delta,
        'sd_delta_PR':     sd_delta,
        'mean_p_perm':     mean_p,
        'n_reject_05':     n_rej,
        'reject_rate_05':  rej_rate,
        'ci95_low':        ci_low,
        'ci95_high':       ci_high,
    })

df_summary = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUT_DIR, 'literal_null_summary.csv')
df_summary.to_csv(summary_csv, index=False)
print(f"\nSaved summary: {summary_csv}")

# §6. Diagnostic plots

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, cond_name in enumerate(CONDITIONS):
    sub = df_results[df_results['condition'] == cond_name]

    # Top row: ΔPR histogram
    axes[0, col].hist(sub['delta_PR'].values, bins=20, edgecolor='black', alpha=0.7)
    axes[0, col].axvline(0, color='red', linestyle='--', label='ΔPR = 0')
    axes[0, col].set_title(f'{cond_name}: ΔPR distribution')
    axes[0, col].set_xlabel('ΔPR')
    axes[0, col].set_ylabel('panels')
    axes[0, col].legend(fontsize=8)

    # Bottom row: p-value histogram
    axes[1, col].hist(sub['p_perm'].values, bins=20, edgecolor='black', alpha=0.7)
    axes[1, col].axvline(0.05, color='red', linestyle='--', label='α = 0.05')
    axes[1, col].set_title(f'{cond_name}: p_perm distribution')
    axes[1, col].set_xlabel('p_perm')
    axes[1, col].set_ylabel('panels')
    axes[1, col].legend(fontsize=8)

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'literal_null_diagnostic.png')
plt.savefig(fig_path, dpi=120)
print(f"\nSaved diagnostic figure: {fig_path}")
plt.show()

# §7. Interpretation key
#
# **If C1 (white noise) reject rate is in [3%, 8%]**: row-permuting null is
# correctly calibrated under the simplest null. ΔPR is well-behaved on
# truly-null data.
#
# **If C1 reject rate is >> 5% (say > 10%)**: the row-permuting null is too
# lax for OLS VAR's dense output. ΔPR fires "by accident" on noise, and the
# framework's claim that it "silences on regimes without reflective structure"
# is wrong. This would explain the R3/R3' OLS VAR passes.
#
# **If C1 is fine but C2 (independent AR(1)) reject rate is >> 5%**: the
# null is well-calibrated under cross-sectional independence but not under
# temporal dependence + cross-variable dependence (which OLS VAR introduces
# via finite-sample correlations). This would explain why OLS VAR passes on
# R3 (banded innovation correlation) and borderline on R3' (independent
# AR(1)).
#
# **If C3 (R1 positive control) reject rate is near 100%**: the metric fires
# correctly when reflective structure is present. Combined with C1/C2 results,
# this tells us whether the null is the problem or whether ΔPR is genuinely
# detecting structure.


print("\n" + "="*70)
print("Block 1 sub-step 1 complete.")
print("="*70)
print("Outputs:")
print(f"  {csv_path}")
print(f"  {summary_csv}")
print(f"  {fig_path}")
print()
print("Send these three files (or just the printed summary table + figure) for")
print("interpretation before proceeding to Block 2.")

### Cell §2.2 — Null calibration (FAST smoke-test variant)

**Paper sections:** Not in paper directly; supports Appendix K calibration.

Reduced-dimension smoke test of the null-calibration methodology before committing to the full run in Cell §2.3. Same four sections (tight-CI calibration, β sweep, β-correlation, parametric-null calibration) at 50-200 panels × B=300 (vs 200-1000 × B=1000 for the full run). Used to validate that all four sections produce sensible numbers.

**Inputs:** None.

**Outputs:** `null_calibration_v2_FAST_summary.csv`, diagnostic figure.

**Runtime:** ~13 minutes on Colab Pro, G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kstest, spearmanr, binomtest

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'null_calibration_v2_FAST_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"OUT_DIR = {OUT_DIR}")

# §0. Configuration


U_UNITS, T_PER_UNIT = 139, 75
N_INDICATORS        = 12
ALPHA               = 0.05

# Section A: 5x reduction in panels, 3.3x reduction in B
SEC_A_N_PANELS  = 200
SEC_A_B_PERMS   = 300

# Section B: 4x reduction in panels per β, 3.3x reduction in B
SEC_B_BETAS     = (0.50, 0.70, 0.85, 0.95, 0.99)
SEC_B_N_PANELS  = 50
SEC_B_B_PERMS   = 300

# Section D: 4x reduction in panels, 3.3x reduction in B
SEC_D_N_PANELS  = 50
SEC_D_B_PERMS   = 300

# R1 positive control parameters (only used if needed; not run in fast variant)
R1_RHO         = 0.95
R1_SIGMA_ZETA  = 0.6
R1_TANH_SCALE  = 2.0
R1_LOAD_LOW    = 0.7
R1_LOAD_HIGH   = 0.9
R1_NOISE_SD    = 0.3
R1_BURN_IN     = 50

print(f"FAST CONFIG:")
print(f"  Section A: {SEC_A_N_PANELS} panels × {SEC_A_B_PERMS} perms × 2 conds")
print(f"  Section B: {SEC_B_N_PANELS} panels × {SEC_B_B_PERMS} perms × {len(SEC_B_BETAS)} βs")
print(f"  Section D: {SEC_D_N_PANELS} panels × {SEC_D_B_PERMS} parametric sims × 2 conds")
print(f"\nThis is a smoke test. If results look right, run null_calibration_v2.py for tight CIs.")

# §1. Core utilities

def gen_white_noise(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    rng = np.random.default_rng(seed)
    Y = rng.standard_normal(size=(U * T, N))
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def gen_independent_ar1(seed, beta, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    """Vectorized: initial state from stationary distribution (no burn-in needed),
    vectorized recursion across units. ~6.7× faster than per-unit loops."""
    rng = np.random.default_rng(seed)
    stationary_sd = 1.0 / np.sqrt(1 - beta**2 + 1e-12) if abs(beta) < 1 else 1.0
    x = rng.standard_normal((U, N)) * stationary_sd
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = beta * x + eps[:, t, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def fit_ols_var(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v; den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new; den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null. ~3.4× faster than the per-row Python loop."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    # Pre-compute off-diagonal indices and values once
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]   # (N, N-1)
    null_PRs = np.zeros(B)
    for b in range(B):
        # Independent per-row permutation via argsort of random keys
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu, sd = float(null_PRs.mean()), float(null_PRs.std())
    delta = PR_obs - mu
    p_perm = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p_perm), float(PR_obs), mu, sd)


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0
    yt_list, yl_list = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            yt_list.append(unit[1:]); yl_list.append(unit[:-1])
    yt = np.vstack(yt_list); yl = np.vstack(yl_list)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b
        sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    """Vectorized: initial state from per-indicator stationary distribution
    (no burn-in needed), then vectorized AR(1) recursion across units.
    ~6.7× faster than the per-unit/per-step loop."""
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed):
    """Parametric AR(1) null."""
    A = fit_ols_var(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)

    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(
            betas, sigmas, U_UNITS, T_PER_UNIT, rng)
        A_sim = fit_ols_var(Y_sim, split_spec)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu, sd = float(null_PRs.mean()), float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def wilson_ci(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2))) / denom
    return max(0.0, center - half), min(1.0, center + half)


ref_indices = list(range(N_INDICATORS))
print("Utilities defined.")

# §2. Section A — Tighter CI on C1 and C2 (200 panels each)

print("=" * 70)
print(f"SECTION A — calibration (n={SEC_A_N_PANELS} per condition, B={SEC_A_B_PERMS})")
print("=" * 70)

sec_a_rows = []
t_start = time.time()

for cond_name, gen_fn in [
    ('C1_white_noise',     lambda s: gen_white_noise(s)),
    ('C2_independent_ar1', lambda s: gen_independent_ar1(s, beta=0.95)),
]:
    print(f"\n--- {cond_name} ---")
    t_cond = time.time()
    for seed in range(SEC_A_N_PANELS):
        Y, ss = gen_fn(seed)
        A = fit_ols_var(Y, ss)
        d, p, pr_obs, mu, sd = compute_delta_pr_rowperm(
            A, ref_indices, B=SEC_A_B_PERMS, seed=seed + 100000)
        sec_a_rows.append({
            'section': 'A', 'condition': cond_name, 'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
        if (seed + 1) % 50 == 0:
            elapsed = time.time() - t_cond
            rate = (seed + 1) / elapsed
            remaining = (SEC_A_N_PANELS - seed - 1) / rate
            print(f"  seed {seed+1}/{SEC_A_N_PANELS} | "
                  f"elapsed {elapsed/60:.1f}min | ETA {remaining/60:.1f}min")
    print(f"  cond time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection A total: {(time.time()-t_start)/60:.1f} min")
df_a = pd.DataFrame(sec_a_rows)
df_a.to_csv(os.path.join(OUT_DIR, 'sec_A_panels.csv'), index=False)



print(f"\n{'Condition':24s} | {'mean ΔPR':>10s} | {'reject@.05':>12s} | {'95% CI':>16s}")
print("-" * 75)
for cond_name in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_a[df_a['condition'] == cond_name]
    n = len(sub); n_rej = sub['rejects_05'].sum(); rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    print(f"{cond_name:24s} | {sub['delta_PR'].mean():+10.4f} | "
          f"{n_rej:3d}/{n:3d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%]")



# §3. Section B — β sweep


print("\n" + "=" * 70)
print(f"SECTION B — β sweep ({SEC_B_N_PANELS} panels per β, {len(SEC_B_BETAS)} βs)")
print("=" * 70)

sec_b_rows = []
t_start = time.time()

for beta in SEC_B_BETAS:
    cond_name = f'C2_ar1_beta{beta:.2f}'
    print(f"\n--- {cond_name} (β = {beta}) ---")
    t_cond = time.time()
    for seed in range(SEC_B_N_PANELS):
        Y, ss = gen_independent_ar1(seed, beta=beta)
        A = fit_ols_var(Y, ss)
        d, p, pr_obs, mu, sd = compute_delta_pr_rowperm(
            A, ref_indices, B=SEC_B_B_PERMS, seed=seed + 200000 + int(beta*1000))
        sec_b_rows.append({
            'section': 'B', 'condition': cond_name, 'beta': beta,
            'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
    print(f"  β={beta} time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection B total: {(time.time()-t_start)/60:.1f} min")
df_b = pd.DataFrame(sec_b_rows)
df_b.to_csv(os.path.join(OUT_DIR, 'sec_B_panels.csv'), index=False)



print(f"\n{'β':>6s} | {'mean ΔPR':>10s} | {'reject@.05':>12s} | {'95% CI':>16s}")
print("-" * 60)
for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]
    n = len(sub); n_rej = sub['rejects_05'].sum(); rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    print(f"{beta:>6.2f} | {sub['delta_PR'].mean():+10.4f} | "
          f"{n_rej:3d}/{n:3d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%]")



# §4. Section C — β-correlation test


print("\n" + "=" * 70)
print("SECTION C — β-correlation test")
print("=" * 70)

beta_vals = df_b['beta'].values
delta_vals = df_b['delta_PR'].values
rho, p_spearman = spearmanr(beta_vals, delta_vals)
print(f"\nSpearman ρ(β, ΔPR) = {rho:+.4f}  (p = {p_spearman:.4g})")
print(f"  Interpretation: positive ρ means ΔPR increases monotonically with β.")

print(f"\n{'β':>6s} | {'mean ΔPR':>10s} | {'95% bootstrap CI':>22s}")
print("-" * 50)
rng = np.random.default_rng(99999)
for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]['delta_PR'].values
    boot_means = np.array([sub[rng.integers(0, len(sub), len(sub))].mean()
                           for _ in range(2000)])
    ci_lo, ci_hi = np.quantile(boot_means, [0.025, 0.975])
    print(f"{beta:>6.2f} | {sub.mean():+10.4f} | [{ci_lo:+.4f}, {ci_hi:+.4f}]")

df_c = pd.DataFrame([{
    'spearman_rho': rho, 'spearman_p': p_spearman,
    'n_total_panels': len(beta_vals),
}])
df_c.to_csv(os.path.join(OUT_DIR, 'sec_C_correlation.csv'), index=False)



# §5. Section D — Parametric AR(1) null calibration


print("\n" + "=" * 70)
print(f"SECTION D — parametric null ({SEC_D_N_PANELS} panels × B={SEC_D_B_PERMS})")
print("=" * 70)

sec_d_rows = []
t_start = time.time()

for cond_name, gen_fn in [
    ('C1_white_noise',     lambda s: gen_white_noise(s)),
    ('C2_independent_ar1', lambda s: gen_independent_ar1(s, beta=0.95)),
]:
    print(f"\n--- {cond_name} (parametric null) ---")
    t_cond = time.time()
    for seed in range(SEC_D_N_PANELS):
        Y, ss = gen_fn(seed)
        d, p, pr_obs, mu, sd = compute_delta_pr_parametric(
            Y, ss, ref_indices, B=SEC_D_B_PERMS, seed=seed + 300000)
        sec_d_rows.append({
            'section': 'D', 'condition': cond_name, 'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
        if (seed + 1) % 10 == 0:
            elapsed = time.time() - t_cond
            rate = (seed + 1) / elapsed
            remaining = (SEC_D_N_PANELS - seed - 1) / rate
            print(f"  seed {seed+1}/{SEC_D_N_PANELS} | "
                  f"elapsed {elapsed/60:.1f}min | ETA {remaining/60:.1f}min")
    print(f"  cond time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection D total: {(time.time()-t_start)/60:.1f} min")
df_d = pd.DataFrame(sec_d_rows)
df_d.to_csv(os.path.join(OUT_DIR, 'sec_D_panels.csv'), index=False)



print(f"\n{'Condition':24s} | {'mean ΔPR':>10s} | {'reject@.05':>12s} | {'95% CI':>16s}")
print("-" * 75)
for cond_name in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_d[df_d['condition'] == cond_name]
    n = len(sub); n_rej = sub['rejects_05'].sum(); rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    print(f"{cond_name:24s} | {sub['delta_PR'].mean():+10.4f} | "
          f"{n_rej:3d}/{n:3d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%]")



# §6. Diagnostic figure


fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# (0,0): Section A C1 ΔPR histogram
ax = axes[0, 0]
sub = df_a[df_a['condition'] == 'C1_white_noise']['delta_PR']
ax.hist(sub.values, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--')
ax.set_title(f'C1 white noise: ΔPR (n={len(sub)})')
ax.set_xlabel('ΔPR'); ax.set_ylabel('panels')

# (0,1): Section A C2 ΔPR histogram
ax = axes[0, 1]
sub = df_a[df_a['condition'] == 'C2_independent_ar1']['delta_PR']
ax.hist(sub.values, bins=30, edgecolor='black', alpha=0.7, color='coral')
ax.axvline(0, color='red', linestyle='--')
ax.set_title(f'C2 ind. AR(1) β=0.95: ΔPR (n={len(sub)})')
ax.set_xlabel('ΔPR'); ax.set_ylabel('panels')

# (0,2): β-sweep reject rate
ax = axes[0, 2]
betas_arr = np.array(SEC_B_BETAS)
rates = []; los = []; his = []
for beta in betas_arr:
    sub = df_b[df_b['beta'] == beta]
    n_rej = sub['rejects_05'].sum(); n = len(sub)
    rates.append(n_rej / n)
    lo, hi = wilson_ci(n_rej, n)
    los.append(lo); his.append(hi)
rates = np.array(rates); los = np.array(los); his = np.array(his)
ax.errorbar(betas_arr, rates*100,
            yerr=[(rates-los)*100, (his-rates)*100],
            fmt='o-', capsize=5, color='darkblue', linewidth=2)
ax.axhline(5, color='red', linestyle='--', label='nominal 5%')
ax.set_xlabel('AR(1) coefficient β')
ax.set_ylabel('Reject rate at α=0.05 (%)')
ax.set_title('β sweep: false positive rate vs persistence')
ax.legend(); ax.grid(alpha=0.3)

# (1,0): β vs mean ΔPR
ax = axes[1, 0]
mean_dpr = []; sd_dpr = []
for beta in betas_arr:
    sub = df_b[df_b['beta'] == beta]['delta_PR']
    mean_dpr.append(sub.mean()); sd_dpr.append(sub.std() / np.sqrt(len(sub)))
mean_dpr = np.array(mean_dpr); sd_dpr = np.array(sd_dpr)
ax.errorbar(betas_arr, mean_dpr, yerr=1.96*sd_dpr, fmt='s-',
            capsize=5, color='darkgreen', linewidth=2)
ax.axhline(0, color='red', linestyle='--', label='ΔPR = 0')
ax.set_xlabel('AR(1) coefficient β')
ax.set_ylabel('Mean ΔPR (95% CI)')
ax.set_title(f'β vs ΔPR (Spearman ρ = {rho:+.3f})')
ax.legend(); ax.grid(alpha=0.3)

# (1,1): Section D — null procedure comparison
ax = axes[1, 1]
labels, vals, los_d, his_d = [], [], [], []
for cond_name, lbl in [('C1_white_noise', 'C1'),
                        ('C2_independent_ar1', 'C2 β=0.95')]:
    for sec, df_sec, sec_lbl in [('A', df_a, 'row-perm'), ('D', df_d, 'parametric')]:
        sub = df_sec[df_sec['condition'] == cond_name]
        n_rej = sub['rejects_05'].sum(); n = len(sub)
        labels.append(f'{lbl}\n{sec_lbl}')
        vals.append(n_rej / n)
        lo, hi = wilson_ci(n_rej, n)
        los_d.append(lo); his_d.append(hi)
vals = np.array(vals); los_d = np.array(los_d); his_d = np.array(his_d)
x = np.arange(len(labels))
colors = ['steelblue', 'navy', 'coral', 'darkred']
ax.bar(x, vals*100, yerr=[(vals-los_d)*100, (his_d-vals)*100],
       capsize=5, color=colors, edgecolor='black')
ax.axhline(5, color='red', linestyle='--', label='nominal 5%')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Reject rate (%)')
ax.set_title('Null procedure comparison')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# (1,2): p-value distributions for C2 under both nulls
ax = axes[1, 2]
sub_a = df_a[df_a['condition'] == 'C2_independent_ar1']['p_perm']
sub_d = df_d[df_d['condition'] == 'C2_independent_ar1']['p_perm']
bins = np.linspace(0, 1, 21)
ax.hist(sub_a.values, bins=bins, alpha=0.5, color='coral',
        edgecolor='black', label=f'row-perm (n={len(sub_a)})', density=True)
ax.hist(sub_d.values, bins=bins, alpha=0.5, color='darkred',
        edgecolor='black', label=f'parametric (n={len(sub_d)})', density=True)
ax.axhline(1, color='black', linestyle=':', label='uniform')
ax.axvline(0.05, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('p-value'); ax.set_ylabel('density')
ax.set_title('C2 p-values under each null')
ax.legend(fontsize=8)

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'null_calibration_v2_FAST_diagnostic.png')
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f"\nSaved figure: {fig_path}")



# §7. Cross-section summary


print("\n" + "=" * 70)
print("FAST-VARIANT FINAL SUMMARY")
print("=" * 70)

summary_rows = []

for cond in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_a[df_a['condition'] == cond]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'A', 'null': 'row-permuting', 'condition': cond,
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
    })

for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'B', 'null': 'row-permuting',
        'condition': f'C2_ar1_beta{beta:.2f}',
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
        'beta': beta,
    })

for cond in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_d[df_d['condition'] == cond]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'D', 'null': 'parametric_AR1', 'condition': cond,
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(OUT_DIR, 'null_calibration_v2_FAST_summary.csv'),
                  index=False)

print(f"\n{'Section':>8s} | {'Null':16s} | {'Condition':24s} | "
      f"{'Reject rate':>14s} | {'95% CI':>16s}")
print("-" * 95)
for r in summary_rows:
    print(f"{r['section']:>8s} | {r['null']:16s} | {r['condition']:24s} | "
          f"{r['reject_rate']*100:5.1f}% ({r['n_reject']}/{r['n_panels']}) | "
          f"[{r['ci95_low']*100:5.1f}%, {r['ci95_high']*100:5.1f}%]")

print(f"\nSpearman ρ(β, ΔPR): {rho:+.4f}, p = {p_spearman:.4g}")

# Sanity-check criteria
print(f"\n--- Sanity-check criteria for full-run go/no-go ---")
c1_a_rate = df_summary[(df_summary.section=='A')&(df_summary.condition=='C1_white_noise')].iloc[0]['reject_rate']
c2_a_rate = df_summary[(df_summary.section=='A')&(df_summary.condition=='C2_independent_ar1')].iloc[0]['reject_rate']
c1_d_rate = df_summary[(df_summary.section=='D')&(df_summary.condition=='C1_white_noise')].iloc[0]['reject_rate']
c2_d_rate = df_summary[(df_summary.section=='D')&(df_summary.condition=='C2_independent_ar1')].iloc[0]['reject_rate']

checks = [
    ("(1) C1 row-perm ≈ 2-7%",          0.01 <= c1_a_rate <= 0.10, f"got {c1_a_rate*100:.1f}%"),
    ("(2) C2 row-perm ≈ 8-15%",         0.05 <= c2_a_rate <= 0.20, f"got {c2_a_rate*100:.1f}%"),
    ("(3) Spearman ρ > 0",              rho > 0,                    f"got ρ = {rho:+.3f}"),
    ("(4) C1 parametric ≈ 0-12%",       0.0 <= c1_d_rate <= 0.15,  f"got {c1_d_rate*100:.1f}%"),
    ("(5) C2 parametric ≈ 0-12%",       0.0 <= c2_d_rate <= 0.15,  f"got {c2_d_rate*100:.1f}%"),
    ("(6) Reject rate increases with β",
     all(df_summary[(df_summary.section=='B')].sort_values('beta')['reject_rate'].diff().dropna() >= -0.05),
     "monotone-with-tolerance check"),
]
all_ok = True
for name, ok, info in checks:
    flag = '✓ OK' if ok else '✗ FAIL'
    if not ok: all_ok = False
    print(f"  {flag} {name}: {info}")

print()
if all_ok:
    print("All checks passed. Safe to run the full null_calibration_v2.py.")
else:
    print("One or more checks failed. Inspect outputs before running full version.")

### Cell §2.3 — Null calibration full sweep across persistence

**Paper sections:** §3.2, Appendix K.

Produces Table 8 in Appendix K (row-permuting null Type-I rates by persistence) and the β=0.99 → 11.5% finding cited in §3.2. Four sections: (A) tight 95% CI calibration on white noise and AR(1) at 1000 panels × B=1000; (B) β sweep across {0.50, 0.70, 0.85, 0.95, 0.99} at 200 panels each; (C) β-correlation test (Spearman ρ between β and ΔPR); (D) parametric AR(1) null validation on the same conditions.

**Inputs:** None.

**Outputs:** `null_calibration_v2_summary.csv`, `sec_A/B/C/D_panels.csv`, diagnostic figure.

**Runtime:** ~20 minutes on Colab Pro, G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kstest, spearmanr, binomtest

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'null_calibration_v2_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"OUT_DIR = {OUT_DIR}")


# §0. Configuration


U_UNITS, T_PER_UNIT = 139, 75
N_INDICATORS        = 12
ALPHA               = 0.05

# Section A: tight CI
SEC_A_N_PANELS  = 1000
SEC_A_B_PERMS   = 1000

# Section B: β sweep
SEC_B_BETAS     = (0.50, 0.70, 0.85, 0.95, 0.99)
SEC_B_N_PANELS  = 200
SEC_B_B_PERMS   = 1000

# Section D: parametric null calibration
SEC_D_N_PANELS  = 200
SEC_D_B_PERMS   = 1000   # number of parametric simulations per panel

# R1 positive control parameters (for sanity checks)
R1_RHO         = 0.95
R1_SIGMA_ZETA  = 0.6
R1_TANH_SCALE  = 2.0
R1_LOAD_LOW    = 0.7
R1_LOAD_HIGH   = 0.9
R1_NOISE_SD    = 0.3
R1_BURN_IN     = 50

print(f"Section A: {SEC_A_N_PANELS} panels × {SEC_A_B_PERMS} perms × 2 conds")
print(f"Section B: {SEC_B_N_PANELS} panels × {SEC_B_B_PERMS} perms × {len(SEC_B_BETAS)} βs")
print(f"Section D: {SEC_D_N_PANELS} panels × {SEC_D_B_PERMS} parametric sims × 2 conds")



# §1. Core utilities (DGPs, OLS VAR, ΔPR, parametric null)


def gen_white_noise(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    rng = np.random.default_rng(seed)
    Y = rng.standard_normal(size=(U * T, N))
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def gen_independent_ar1(seed, beta, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    """Vectorized: initial state from stationary distribution (no burn-in needed),
    then vectorized AR(1) recursion across units. ~6.7× faster than per-unit loops."""
    rng = np.random.default_rng(seed)
    stationary_sd = 1.0 / np.sqrt(1 - beta**2 + 1e-12) if abs(beta) < 1 else 1.0
    x = rng.standard_normal((U, N)) * stationary_sd
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = beta * x + eps[:, t, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def gen_R1(seed, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS):
    """Positive control: R1 single-factor reflective."""
    rng = np.random.default_rng(seed)
    Lam = rng.uniform(R1_LOAD_LOW, R1_LOAD_HIGH, size=N)
    paths = []
    for u in range(U):
        eta = 0.0
        for _ in range(R1_BURN_IN):
            eta = R1_RHO * np.tanh(R1_TANH_SCALE * eta) + rng.normal(0, R1_SIGMA_ZETA)
        unit = np.zeros((T, N))
        for t in range(T):
            eta = R1_RHO * np.tanh(R1_TANH_SCALE * eta) + rng.normal(0, R1_SIGMA_ZETA)
            unit[t] = Lam * eta + rng.normal(0, R1_NOISE_SD, size=N)
        paths.append(unit)
    Y = np.vstack(paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y, [T] * U


def fit_ols_var(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v; den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new; den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null. ~3.4× faster than the per-row Python loop."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    # Pre-compute off-diagonal indices and values once
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]   # (N, N-1)
    null_PRs = np.zeros(B)
    for b in range(B):
        # Independent per-row permutation via argsort of random keys
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu, sd = float(null_PRs.mean()), float(null_PRs.std())
    delta = PR_obs - mu
    p_perm = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p_perm), float(PR_obs), mu, sd)


# Parametric AR(1) null
def fit_per_indicator_ar1(Y, split_spec):
    """Fit y_{i,t} = β_i y_{i,t-1} + ε_{i,t} per indicator, panel-pooled."""
    cursor = 0
    yt_list, yl_list = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            yt_list.append(unit[1:]); yl_list.append(unit[:-1])
    yt = np.vstack(yt_list); yl = np.vstack(yl_list)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b
        sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed):
    """Parametric AR(1) null: fit per-indicator AR(1) on the panel, simulate
    B independent panels matching those persistences, use the resulting
    ΔPR distribution as the null. PR_obs is computed on the original panel's
    OLS VAR fit; null is the distribution of OLS VAR PR fits on simulated
    AR(1) panels.
    """
    A = fit_ols_var(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)

    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)

    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(
            betas, sigmas, U_UNITS, T_PER_UNIT, rng)
        A_sim = fit_ols_var(Y_sim, split_spec)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu, sd = float(null_PRs.mean()), float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def wilson_ci(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p*(1-p)/n) + (z**2/(4*n**2))) / denom
    return max(0.0, center - half), min(1.0, center + half)


ref_indices = list(range(N_INDICATORS))
print("Utilities defined.")



# §2. Section A — Tight CI on C1 and C2 (1000 panels each)


print("=" * 70)
print("SECTION A — Tight calibration on C1 (white noise) and C2 (AR(1) β=0.95)")
print("=" * 70)

sec_a_rows = []
t_start = time.time()

for cond_name, gen_fn in [
    ('C1_white_noise',     lambda s: gen_white_noise(s)),
    ('C2_independent_ar1', lambda s: gen_independent_ar1(s, beta=0.95)),
]:
    print(f"\n--- {cond_name} ---")
    t_cond = time.time()
    for seed in range(SEC_A_N_PANELS):
        Y, ss = gen_fn(seed)
        A = fit_ols_var(Y, ss)
        d, p, pr_obs, mu, sd = compute_delta_pr_rowperm(
            A, ref_indices, B=SEC_A_B_PERMS, seed=seed + 100000)
        sec_a_rows.append({
            'section': 'A', 'condition': cond_name, 'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
        if (seed + 1) % 100 == 0:
            elapsed = time.time() - t_cond
            rate = (seed + 1) / elapsed
            remaining = (SEC_A_N_PANELS - seed - 1) / rate
            print(f"  seed {seed+1}/{SEC_A_N_PANELS} | "
                  f"elapsed {elapsed/60:.1f}min | ETA {remaining/60:.1f}min")
    print(f"  Section A condition time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection A total time: {(time.time()-t_start)/60:.1f} min")

df_a = pd.DataFrame(sec_a_rows)
df_a.to_csv(os.path.join(OUT_DIR, 'sec_A_panels.csv'), index=False)



# Section A summary
print("\n--- Section A summary ---")
print(f"{'Condition':24s} | {'mean ΔPR':>10s} | {'sd ΔPR':>8s} | "
      f"{'reject@.05':>12s} | {'95% CI':>16s}")
print("-" * 80)
for cond_name in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_a[df_a['condition'] == cond_name]
    n = len(sub)
    n_rej = sub['rejects_05'].sum()
    rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    print(f"{cond_name:24s} | {sub['delta_PR'].mean():+10.4f} | "
          f"{sub['delta_PR'].std():8.4f} | "
          f"{n_rej:4d}/{n:4d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%]")
    test = binomtest(n_rej, n, p=0.05, alternative='two-sided')
    print(f"  → Binomial test for H0: rate = 0.05  →  p = {test.pvalue:.4f}")



# §3. Section B — β sweep on independent AR(1)


print("\n" + "=" * 70)
print("SECTION B — Independent AR(1) β sweep")
print("=" * 70)

sec_b_rows = []
t_start = time.time()

for beta in SEC_B_BETAS:
    cond_name = f'C2_ar1_beta{beta:.2f}'
    print(f"\n--- {cond_name} (β = {beta}) ---")
    t_cond = time.time()
    for seed in range(SEC_B_N_PANELS):
        Y, ss = gen_independent_ar1(seed, beta=beta)
        A = fit_ols_var(Y, ss)
        d, p, pr_obs, mu, sd = compute_delta_pr_rowperm(
            A, ref_indices, B=SEC_B_B_PERMS, seed=seed + 200000 + int(beta*1000))
        sec_b_rows.append({
            'section': 'B', 'condition': cond_name, 'beta': beta,
            'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
        if (seed + 1) % 50 == 0:
            elapsed = time.time() - t_cond
            print(f"  seed {seed+1}/{SEC_B_N_PANELS} | elapsed {elapsed/60:.1f}min")
    print(f"  β={beta} time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection B total time: {(time.time()-t_start)/60:.1f} min")

df_b = pd.DataFrame(sec_b_rows)
df_b.to_csv(os.path.join(OUT_DIR, 'sec_B_panels.csv'), index=False)



# Section B summary table
print("\n--- Section B summary: false positive rate vs persistence ---")
print(f"{'β':>6s} | {'mean ΔPR':>10s} | {'reject@.05':>12s} | {'95% CI':>16s}")
print("-" * 60)
for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]
    n = len(sub)
    n_rej = sub['rejects_05'].sum()
    rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    print(f"{beta:>6.2f} | {sub['delta_PR'].mean():+10.4f} | "
          f"{n_rej:4d}/{n:4d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%]")



# §4. Section C — β-correlation test
# Does ΔPR scale monotonically with β? If so, we have a predictive
# characterization of the inflation, not just an empirical observation.


print("\n" + "=" * 70)
print("SECTION C — β-correlation test")
print("=" * 70)

# Pool all β-sweep panels
beta_vals = df_b['beta'].values
delta_vals = df_b['delta_PR'].values

# Spearman correlation between β and ΔPR
rho, p_spearman = spearmanr(beta_vals, delta_vals)
print(f"\nSpearman ρ(β, ΔPR) = {rho:+.4f}  (p = {p_spearman:.4g})")
print(f"  Interpretation: positive ρ means ΔPR increases monotonically with β.")
print(f"  Null hypothesis: no monotonic relationship.")

# Per-β bootstrap CIs on mean ΔPR
print(f"\nPer-β mean ΔPR with bootstrap 95% CIs:")
print(f"{'β':>6s} | {'mean':>10s} | {'95% CI':>22s}")
print("-" * 50)
rng = np.random.default_rng(99999)
for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]['delta_PR'].values
    boot_means = np.array([sub[rng.integers(0, len(sub), len(sub))].mean()
                           for _ in range(2000)])
    ci_lo, ci_hi = np.quantile(boot_means, [0.025, 0.975])
    print(f"{beta:>6.2f} | {sub.mean():+10.4f} | [{ci_lo:+.4f}, {ci_hi:+.4f}]")

# Save Section C summary
df_c = pd.DataFrame([{
    'spearman_rho': rho, 'spearman_p': p_spearman,
    'n_total_panels': len(beta_vals),
}])
df_c.to_csv(os.path.join(OUT_DIR, 'sec_C_correlation.csv'), index=False)



# §5. Section D — Parametric AR(1) null calibration
# Validate the corrected null procedure on C1 and C2. Should yield ~5%
# rejection on both if the parametric null is correctly calibrated.


print("\n" + "=" * 70)
print("SECTION D — Parametric AR(1) null calibration")
print("=" * 70)

sec_d_rows = []
t_start = time.time()

for cond_name, gen_fn in [
    ('C1_white_noise',     lambda s: gen_white_noise(s)),
    ('C2_independent_ar1', lambda s: gen_independent_ar1(s, beta=0.95)),
]:
    print(f"\n--- {cond_name} (parametric null) ---")
    t_cond = time.time()
    for seed in range(SEC_D_N_PANELS):
        Y, ss = gen_fn(seed)
        d, p, pr_obs, mu, sd = compute_delta_pr_parametric(
            Y, ss, ref_indices, B=SEC_D_B_PERMS, seed=seed + 300000)
        sec_d_rows.append({
            'section': 'D', 'condition': cond_name, 'panel_seed': seed,
            'delta_PR': d, 'p_perm': p, 'PR_obs': pr_obs,
            'null_mean': mu, 'null_sd': sd,
            'rejects_05': bool(np.isfinite(d) and d > 0
                                and np.isfinite(p) and p < ALPHA),
        })
        if (seed + 1) % 25 == 0:
            elapsed = time.time() - t_cond
            rate = (seed + 1) / elapsed
            remaining = (SEC_D_N_PANELS - seed - 1) / rate
            print(f"  seed {seed+1}/{SEC_D_N_PANELS} | "
                  f"elapsed {elapsed/60:.1f}min | ETA {remaining/60:.1f}min")
    print(f"  Section D condition time: {(time.time()-t_cond)/60:.1f} min")

print(f"\nSection D total time: {(time.time()-t_start)/60:.1f} min")

df_d = pd.DataFrame(sec_d_rows)
df_d.to_csv(os.path.join(OUT_DIR, 'sec_D_panels.csv'), index=False)



# Section D summary
print("\n--- Section D summary: parametric null calibration ---")
print(f"{'Condition':24s} | {'mean ΔPR':>10s} | {'reject@.05':>12s} | "
      f"{'95% CI':>16s} | {'binom p':>10s}")
print("-" * 90)
for cond_name in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_d[df_d['condition'] == cond_name]
    n = len(sub)
    n_rej = sub['rejects_05'].sum()
    rate = n_rej / n
    lo, hi = wilson_ci(n_rej, n)
    test = binomtest(n_rej, n, p=0.05, alternative='two-sided')
    print(f"{cond_name:24s} | {sub['delta_PR'].mean():+10.4f} | "
          f"{n_rej:4d}/{n:4d} ({rate*100:5.1f}%) | "
          f"[{lo*100:5.1f}%, {hi*100:5.1f}%] | {test.pvalue:>10.4f}")



# §6. Diagnostic figures


fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# (0,0): Section A C1 ΔPR histogram + zero line
ax = axes[0, 0]
sub = df_a[df_a['condition'] == 'C1_white_noise']['delta_PR']
ax.hist(sub.values, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--')
ax.set_title(f'C1 white noise: ΔPR (n={len(sub)})\n'
             f'reject@.05: {sub[df_a["condition"]=="C1_white_noise"].apply(lambda x: x>0).sum()}/{len(sub)} ({100*(sub>0).mean():.1f}%)')
ax.set_xlabel('ΔPR'); ax.set_ylabel('panels')

# (0,1): Section A C2 ΔPR histogram
ax = axes[0, 1]
sub = df_a[df_a['condition'] == 'C2_independent_ar1']['delta_PR']
ax.hist(sub.values, bins=40, edgecolor='black', alpha=0.7, color='coral')
ax.axvline(0, color='red', linestyle='--')
ax.set_title(f'C2 ind. AR(1) β=0.95: ΔPR (n={len(sub)})')
ax.set_xlabel('ΔPR'); ax.set_ylabel('panels')

# (0,2): β-sweep reject rate
ax = axes[0, 2]
betas_arr = np.array(SEC_B_BETAS)
rates = []; los = []; his = []
for beta in betas_arr:
    sub = df_b[df_b['beta'] == beta]
    n_rej = sub['rejects_05'].sum(); n = len(sub)
    rates.append(n_rej / n)
    lo, hi = wilson_ci(n_rej, n)
    los.append(lo); his.append(hi)
rates = np.array(rates); los = np.array(los); his = np.array(his)
ax.errorbar(betas_arr, rates*100,
            yerr=[(rates-los)*100, (his-rates)*100],
            fmt='o-', capsize=5, color='darkblue', linewidth=2)
ax.axhline(5, color='red', linestyle='--', label='nominal 5%')
ax.set_xlabel('AR(1) coefficient β')
ax.set_ylabel('Reject rate at α=0.05 (%)')
ax.set_title('β sweep: false positive rate vs persistence')
ax.legend(); ax.grid(alpha=0.3)

# (1,0): β vs mean ΔPR (Section C)
ax = axes[1, 0]
mean_dpr = []; sd_dpr = []
for beta in betas_arr:
    sub = df_b[df_b['beta'] == beta]['delta_PR']
    mean_dpr.append(sub.mean()); sd_dpr.append(sub.std() / np.sqrt(len(sub)))
mean_dpr = np.array(mean_dpr); sd_dpr = np.array(sd_dpr)
ax.errorbar(betas_arr, mean_dpr, yerr=1.96*sd_dpr, fmt='s-',
            capsize=5, color='darkgreen', linewidth=2)
ax.axhline(0, color='red', linestyle='--', label='ΔPR = 0')
ax.set_xlabel('AR(1) coefficient β')
ax.set_ylabel('Mean ΔPR (95% CI)')
ax.set_title(f'β vs ΔPR (Spearman ρ = {rho:+.3f})')
ax.legend(); ax.grid(alpha=0.3)

# (1,1): Section D — parametric null calibration check
ax = axes[1, 1]
labels, vals, los_d, his_d = [], [], [], []
for cond_name, lbl in [('C1_white_noise', 'C1'),
                        ('C2_independent_ar1', 'C2 β=0.95')]:
    for sec, df_sec, sec_lbl in [('A', df_a, 'row-perm'), ('D', df_d, 'parametric')]:
        sub = df_sec[df_sec['condition'] == cond_name]
        n_rej = sub['rejects_05'].sum(); n = len(sub)
        labels.append(f'{lbl}\n{sec_lbl}')
        vals.append(n_rej / n)
        lo, hi = wilson_ci(n_rej, n)
        los_d.append(lo); his_d.append(hi)
vals = np.array(vals); los_d = np.array(los_d); his_d = np.array(his_d)
x = np.arange(len(labels))
colors = ['steelblue', 'navy', 'coral', 'darkred']
ax.bar(x, vals*100, yerr=[(vals-los_d)*100, (his_d-vals)*100],
       capsize=5, color=colors, edgecolor='black')
ax.axhline(5, color='red', linestyle='--', label='nominal 5%')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Reject rate (%)')
ax.set_title('Null procedure comparison')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# (1,2): p-value histogram for C2 under both nulls
ax = axes[1, 2]
sub_a = df_a[df_a['condition'] == 'C2_independent_ar1']['p_perm']
sub_d = df_d[df_d['condition'] == 'C2_independent_ar1']['p_perm']
bins = np.linspace(0, 1, 21)
ax.hist(sub_a.values, bins=bins, alpha=0.5, color='coral',
        edgecolor='black', label=f'row-perm null (n={len(sub_a)})', density=True)
ax.hist(sub_d.values, bins=bins, alpha=0.5, color='darkred',
        edgecolor='black', label=f'parametric null (n={len(sub_d)})', density=True)
ax.axhline(1, color='black', linestyle=':', label='uniform')
ax.axvline(0.05, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('p-value'); ax.set_ylabel('density')
ax.set_title('C2 p-value distribution under each null')
ax.legend(fontsize=8)

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'null_calibration_v2_diagnostic.png')
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f"\nSaved figure: {fig_path}")



# §7. Cross-section summary table


print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

summary_rows = []

# Section A (row-perm null)
for cond in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_a[df_a['condition'] == cond]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'A', 'null': 'row-permuting', 'condition': cond,
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
    })

# Section B (β sweep)
for beta in SEC_B_BETAS:
    sub = df_b[df_b['beta'] == beta]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'B', 'null': 'row-permuting',
        'condition': f'C2_ar1_beta{beta:.2f}',
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
        'beta': beta,
    })

# Section D (parametric null)
for cond in ('C1_white_noise', 'C2_independent_ar1'):
    sub = df_d[df_d['condition'] == cond]
    n = len(sub); n_rej = sub['rejects_05'].sum()
    lo, hi = wilson_ci(n_rej, n)
    summary_rows.append({
        'section': 'D', 'null': 'parametric_AR1', 'condition': cond,
        'n_panels': n, 'n_reject': n_rej, 'reject_rate': n_rej/n,
        'ci95_low': lo, 'ci95_high': hi,
        'mean_delta_PR': sub['delta_PR'].mean(),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(OUT_DIR, 'null_calibration_v2_summary.csv'), index=False)

# Print compact table
print(f"\n{'Section':>8s} | {'Null':16s} | {'Condition':24s} | "
      f"{'Reject rate':>12s} | {'95% CI':>16s}")
print("-" * 95)
for r in summary_rows:
    print(f"{r['section']:>8s} | {r['null']:16s} | {r['condition']:24s} | "
          f"{r['reject_rate']*100:5.1f}% ({r['n_reject']}/{r['n_panels']}) | "
          f"[{r['ci95_low']*100:5.1f}%, {r['ci95_high']*100:5.1f}%]")

print(f"\nSpearman ρ(β, ΔPR) across β-sweep: {rho:+.4f}, p = {p_spearman:.4g}")
print(f"\nSaved summary: {os.path.join(OUT_DIR, 'null_calibration_v2_summary.csv')}")
print(f"Saved figure : {fig_path}")
print(f"\nNext: parametric_null_applications.py — apply the corrected null to V-Dem,")
print(f"      R3, R3', and R1-dynsplit. ~5 min runtime.")

## Section 3 — Synthetic block analyses

Two parameter sweeps on the synthetic regimes that produce specific tables in Appendix O. Both cells are pure-synthetic; no V-Dem dependencies. The V-Dem-specific block analyses (signed-vs-unsigned ablation, era subsets, adversary main sweep, parametric null on V-Dem) are in Section 4.

### Cell §3.1 — σ_λ sweep on R1-heterogen

**Paper sections:** §4, Appendix O (parameter sweep).

Sweeps R1-heterogen ΔPR across loading-dispersion σ_λ ∈ [0, 1.5]. Establishes the heterogeneous-loading robustness reported in §4 (a single σ_λ value is insufficient to break ΔPR's pass on R1-heterogen).

R1-vdemlike does not test the framework's behavior under heterogeneous measurement (different units having different latent → indicator mappings). The existing R1 has homogeneous loadings: a single Lam vector shared across all units. To test heterogeneity, sweep σ_lam (per-unit loading dispersion) across multiple values.

DGP (R1-heterogen):
  - Population mean loading: λ_i ~ U(0.6, 0.9), drawn once per regime.
  - Per-unit loading:        λ_{u,i} = λ_i + σ_lam · z_{u,i},
                              z_{u,i} ~ N(0, 1).
  - Latent dynamics:         η_{u,t+1} = ρ tanh(s η_{u,t}) + ζ
                              (same as R1: ρ=0.95, s=2.0, σ_ζ=0.6).
  - Measurement:             y_{u,t,i} = λ_{u,i} · η_{u,t}
                                       + idiosyncratic_sd · ε_{u,t,i}.

 - σ_lam = 0.0 reduces to the original R1 (sanity check).
 - σ_lam → 0.25 produces strongly heterogeneous loadings; expected to
  degrade ΔPR's significance because the population reduced-form W
  matrix is no longer rank-one when each unit's measurement model
  differs.

Sweep:
  - σ_lam ∈ {0.0, 0.05, 0.10, 0.15, 0.20, 0.25}  (6 values)
  - n_seeds = 10 per σ_lam value
  - Method: OLS VAR (headline)
  - Null: parametric AR(1) at B=1000 (the principled null per Block 1)
  - Total: 60 runs × ~30s = ~30 minutes

**Inputs:**
  - lsc_synthetic_validation.py with generate_R1 (we'll re-implement
    inline to add σ_lam, since the existing R1 hard-codes homogeneous
    loadings).

**Outputs:** `sigma_lam_sweep_aggregate.csv`, incremental CSV.

**Runtime:** ~1 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'sigma_lam_sweep_block_7_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


SIGMA_LAM_GRID = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]
N_SEEDS = 10
B_PERMS = 1000
ALPHA = 0.05

# DGP defaults (matched to R1)
U_UNITS = 89          # match V-Dem panel
T_PER_UNIT = 75       # match V-Dem panel
N_INDICATORS = 12     # match R1
RHO = 0.95
SIGMA_ZETA = 0.6
TANH_SCALE = 2.0
IDIOSYNCRATIC_SD = 0.3
BURN_IN = 50

# Loading population mean is U(0.6, 0.9), same as R1
LAM_LOW, LAM_HIGH = 0.6, 0.9

print(f"Sweep: σ_lam ∈ {SIGMA_LAM_GRID}")
print(f"  n_seeds={N_SEEDS}, B_perms={B_PERMS}")
print(f"  Panel: U={U_UNITS}, T={T_PER_UNIT}, N={N_INDICATORS}")



# §2. R1-heterogen DGP


def generate_R1_heterogen(seed, sigma_lam, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS,
                           rho=RHO, sigma_zeta=SIGMA_ZETA, tanh_scale=TANH_SCALE,
                           idiosyncratic_sd=IDIOSYNCRATIC_SD, burn_in=BURN_IN,
                           lam_low=LAM_LOW, lam_high=LAM_HIGH):
    """R1-heterogen: 1-factor reflective with PER-UNIT loading dispersion.

    σ_lam controls how much each unit's loadings deviate from the
    population mean λ_i. σ_lam=0 reduces exactly to R1.
    """
    rng = np.random.default_rng(seed)
    Lam_pop = rng.uniform(lam_low, lam_high, size=N)        # population mean λ_i
    # Per-unit deviations: z_{u,i} ~ N(0,1). λ_{u,i} = λ_i + σ_lam · z_{u,i}.
    Z_unit = rng.standard_normal(size=(U, N))               # (U, N)
    Lam_unit = Lam_pop[None, :] + sigma_lam * Z_unit        # (U, N)

    unit_paths = []
    for u in range(U):
        lam_u = Lam_unit[u]                                 # this unit's loadings
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            path[t] = lam_u * eta + rng.normal(0, idiosyncratic_sd, size=N)
        unit_paths.append(path)

    Y = np.vstack(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    split_spec = [T] * U
    indices = list(range(N))

    return {
        'Y': Y, 'split_spec': split_spec,
        'Lam_pop': Lam_pop, 'Lam_unit': Lam_unit,
        'sigma_lam': sigma_lam, 'seed': seed,
        'reflective_indices': indices,
    }



# §3. Core utilities (same as Blocks 1, 5)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed, U_resample, T_resample):
    """Parametric AR(1) null. Returns (ΔPR, p, PR_obs, null_mu, null_sd)."""
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


print("Utilities defined.")



# §4. Sweep loop (incremental save)


print("\n" + "=" * 70)
print(f"Sweep: σ_lam × seed (parametric null, B={B_PERMS})")
print("=" * 70)

incremental_csv = os.path.join(OUT_DIR, 'sigma_lam_sweep_incremental.csv')

# Resume support
completed = set()
all_results = []
if os.path.exists(incremental_csv):
    df_prev = pd.read_csv(incremental_csv)
    for _, r in df_prev.iterrows():
        completed.add((float(r['sigma_lam']), int(r['seed'])))
        all_results.append(r.to_dict())
    print(f"Resumed: {len(completed)} runs already done")
else:
    print(f"Starting fresh: {incremental_csv}")

t_start = time.time()
total_runs = len(SIGMA_LAM_GRID) * N_SEEDS
done_count = len(completed)

for sigma_lam in SIGMA_LAM_GRID:
    for seed_idx in range(N_SEEDS):
        seed = 1000 + seed_idx
        key = (sigma_lam, seed)
        if key in completed:
            continue

        run_t0 = time.time()
        # Generate panel under R1-heterogen with this σ_lam + seed
        regime = generate_R1_heterogen(seed=seed, sigma_lam=sigma_lam)
        Y = regime['Y']; ss = regime['split_spec']; ref_idx = regime['reflective_indices']

        # Verify Lam_unit dispersion is roughly σ_lam (sanity check)
        lam_unit_std = float(regime['Lam_unit'].std(axis=0).mean())  # mean across indicators of per-unit std

        # Compute ΔPR + p under parametric null
        delta, p, pr_obs, null_mu, null_sd = compute_delta_pr_parametric(
            Y, ss, ref_idx, B=B_PERMS, seed=seed * 7 + 3,
            U_resample=U_UNITS, T_resample=T_PER_UNIT)

        passes = bool(np.isfinite(delta) and delta > 0 and p < ALPHA)
        elapsed = time.time() - run_t0

        row = {
            'sigma_lam': sigma_lam, 'seed': seed,
            'PR_obs': pr_obs,
            'parametric_delta': delta, 'parametric_p': p,
            'parametric_null_mean': null_mu, 'parametric_null_sd': null_sd,
            'passes': passes,
            'realized_lam_unit_std': lam_unit_std,
            'elapsed_sec': elapsed,
        }
        all_results.append(row)
        done_count += 1

        # Incremental save
        pd.DataFrame(all_results).to_csv(incremental_csv, index=False)

        total_elapsed = (time.time() - t_start) / 60
        print(f"  σ_lam={sigma_lam:.2f} seed={seed}: "
              f"PR_obs={pr_obs:.4f}, ΔPR={delta:+.4f}, p={p:.4f}, "
              f"pass={passes} [{elapsed:.1f}s, {done_count}/{total_runs}, "
              f"{total_elapsed:.1f}min total]")

print(f"\nSweep complete: {time.time()-t_start:.0f}s = {(time.time()-t_start)/60:.1f}min")



# §5. Per-σ_lam aggregate


print("\n" + "=" * 70)
print(f"AGGREGATE BY σ_lam (n_seeds={N_SEEDS})")
print("=" * 70)

df = pd.DataFrame(all_results)

print(f"\n{'σ_lam':>7s} {'mean(λ_unit_std)':>16s} | "
      f"{'PR_obs':>8s} {'ΔPR':>8s} {'p (median)':>12s} | "
      f"{'pass-rate':>10s} {'p<α':>5s}")
print("-" * 90)

agg_rows = []
for sl in SIGMA_LAM_GRID:
    sub = df[df['sigma_lam'] == sl]
    if len(sub) == 0:
        continue
    pr_obs_med = sub['PR_obs'].median()
    delta_med = sub['parametric_delta'].median()
    p_med = sub['parametric_p'].median()
    pass_rate = sub['passes'].mean()
    n_pass = int(sub['passes'].sum())
    lam_std_med = sub['realized_lam_unit_std'].median()

    print(f"{sl:>7.2f} {lam_std_med:>16.3f} | "
          f"{pr_obs_med:>8.4f} {delta_med:>+8.4f} {p_med:>12.4f} | "
          f"{pass_rate:>10.2f} {n_pass:>3d}/{len(sub)}")

    agg_rows.append({
        'sigma_lam': sl, 'n_seeds': len(sub),
        'PR_obs_median': float(pr_obs_med),
        'PR_obs_p25': float(sub['PR_obs'].quantile(0.25)),
        'PR_obs_p75': float(sub['PR_obs'].quantile(0.75)),
        'delta_median': float(delta_med),
        'delta_p25': float(sub['parametric_delta'].quantile(0.25)),
        'delta_p75': float(sub['parametric_delta'].quantile(0.75)),
        'p_median': float(p_med),
        'p_p25': float(sub['parametric_p'].quantile(0.25)),
        'p_p75': float(sub['parametric_p'].quantile(0.75)),
        'pass_rate': float(pass_rate),
        'realized_lam_unit_std_median': float(lam_std_med),
    })

df_agg = pd.DataFrame(agg_rows)
agg_csv = os.path.join(OUT_DIR, 'sigma_lam_sweep_aggregate.csv')
df_agg.to_csv(agg_csv, index=False)
print(f"\nSaved aggregate: {agg_csv}")



# §6. Interpretation guide


print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

# Find smallest σ_lam where pass_rate drops below 50%
crossover_sl = None
for sl in SIGMA_LAM_GRID:
    sub = df[df['sigma_lam'] == sl]
    if len(sub) == 0: continue
    if sub['passes'].mean() < 0.5:
        crossover_sl = sl
        break

if crossover_sl is None:
    print(f"\nFraming wins: ΔPR maintains pass-rate ≥ 50% across all σ_lam in "
          f"the sweep ({SIGMA_LAM_GRID}). The framework is robust to per-unit "
          f"loading heterogeneity at the levels tested.")
else:
    # Find the σ_lam where it first drops
    print(f"\nGraceful degradation: ΔPR pass-rate drops below 50% at "
          f"σ_lam = {crossover_sl:.2f}. Below this, the framework correctly "
          f"identifies reflective structure; above, the framework correctly "
          f"flags that loading heterogeneity has broken population rank-1 "
          f"reduced-form structure.")

# Median p-value trajectory
p_med_at_zero = df[df['sigma_lam']==0.0]['parametric_p'].median() if (df['sigma_lam']==0.0).any() else None
p_med_at_max = df[df['sigma_lam']==SIGMA_LAM_GRID[-1]]['parametric_p'].median() if (df['sigma_lam']==SIGMA_LAM_GRID[-1]).any() else None
if p_med_at_zero is not None and p_med_at_max is not None:
    print(f"\nMedian p-value trajectory: {p_med_at_zero:.4f} (σ_lam=0) → "
          f"{p_med_at_max:.4f} (σ_lam={SIGMA_LAM_GRID[-1]:.2f})")

print(f"\nFor the paper:")
print(f"  - Plot per-σ_lam median p-value with IQR band; show α=0.05 line")
print(f"  - Plot per-σ_lam median PR_obs to illustrate the underlying signal degradation")
print(f"  - σ_lam swept; framework behavior characterized across heterogeneity range")

print(f"\nDone.")

### Cell §3.2 — R1-dynsplit correlation sweep at V-Dem-matched conditions

**Paper sections:** §4, Appendix O (parameter sweep).

Sweeps R1-dynsplit cross-block correlation r ∈ [0, 1] at the V-Dem-matched 7/6 partition with σ_ε up to V-Dem's empirical noise level. Tests whether the cross-sectionally-rank-1-but-dynamically-multi-block diagnostic in §4 transfers from the paper-default (6/6 partition, low noise) to V-Dem-matched conditions.

This block sweeps idiosyncratic noise σ_ε from the paper default (0.30)
through V-Dem-matched (0.60) and beyond, at the 7/6 partition matching
V-Dem's empirical dynamic split. As σ_ε grows, mean indicator correlation
drops:
-  σ_ε=0.30 → mean corr ≈ 0.74 (paper default)
-  σ_ε=0.45 → mean corr ≈ 0.62 (close to V-Dem size-13 block)
-  σ_ε=0.60 → mean corr ≈ 0.53 (V-Dem-matched, size-16 panel)
-  σ_ε=0.75 → mean corr ≈ 0.45 (beyond V-Dem)
-  σ_ε=1.00 → mean corr ≈ 0.32 (stress test)

Plus the original 6/6 paper baseline as anchor, six configs total.
Run OLS VAR + both nulls (row-perm and parametric AR(1)) at B=1000,
n=10 seeds per config. 60 runs total, ~30 minutes.

The question: does R1-dynsplit's borderline pass survive at V-Dem-matched
correlation? Or does the synthetic regime fail to pass at 7/6 + low
correlation, closing the gap to V-Dem's actual failure?


**Inputs:** src/lsc_synthetic_validation.py with generate_R1_dynsplit
  (we re-implement the DGP inline to vary 7/6 partition + σ_ε)

**Outputs:** `r1_dynsplit_sweep_aggregate.csv`, incremental CSV.

**Runtime:** ~2 minutes on Colab Pro, G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'r1_dynsplit_sweep_block_3_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


# Six configs: baseline (paper, 6/6) plus five at 7/6 with varying σ_ε
# Each config gets a name for tracking; mean corr targets from quick local sim.
CONFIGS = [
    # name, n_block_A, n_block_B, idiosyncratic_sd, target_mean_corr_note
    ('baseline_6_6',       6, 6, 0.30, 'paper version, mean corr ~0.74'),
    ('split_7_6_tight',    7, 6, 0.30, 'concern-F partition fix only, ~0.74'),
    ('split_7_6_mid',      7, 6, 0.45, 'closer to V-Dem size-13 ~0.62'),
    ('split_7_6_vdem',     7, 6, 0.60, 'V-Dem-matched mean corr ~0.53'),
    ('split_7_6_loose',    7, 6, 0.75, 'beyond V-Dem, ~0.45'),
    ('split_7_6_extreme',  7, 6, 1.00, 'stress test, ~0.32'),
]

N_SEEDS = 10
B_PERMS = 1000
ALPHA = 0.05

# DGP fixed parameters (match generate_R1_dynsplit defaults)
U_UNITS = 89
T_PER_UNIT = 75
RHO = 0.95
SIGMA_ZETA = 0.6
TANH_SCALE = 2.0
BURN_IN = 50
R_LATENT = 0.85
LOAD_LOW, LOAD_HIGH = 0.7, 0.9

print(f"Configs: {len(CONFIGS)}")
for name, nA, nB, sd, note in CONFIGS:
    print(f"  {name}: blocks=({nA},{nB}), σ_ε={sd}, {note}")
print(f"Seeds per config: {N_SEEDS}")
print(f"B_perms: {B_PERMS}")



# §2. R1-dynsplit DGP (matches src/lsc_synthetic_validation.py)


def generate_R1_dynsplit(seed, U=U_UNITS, T=T_PER_UNIT,
                          rho=RHO, sigma_zeta=SIGMA_ZETA, tanh_scale=TANH_SCALE,
                          idiosyncratic_sd=0.30, burn_in=BURN_IN,
                          r_latent=R_LATENT,
                          n_block_A=6, n_block_B=6,
                          loading_low=LOAD_LOW, loading_high=LOAD_HIGH):
    """R1-dynsplit: two-latent panel with correlated innovations and
    independent dynamics. Sub-block A loads on η_A, sub-block B on η_B."""
    rng = np.random.default_rng(seed)
    N = n_block_A + n_block_B

    Lam_A = rng.uniform(loading_low, loading_high, size=n_block_A)
    Lam_B = rng.uniform(loading_low, loading_high, size=n_block_B)

    cov_zeta = sigma_zeta**2 * np.array([[1.0, r_latent], [r_latent, 1.0]])
    L_zeta = np.linalg.cholesky(cov_zeta)

    unit_paths = []
    for u in range(U):
        eta_A, eta_B = 0.0, 0.0
        for _ in range(burn_in):
            zeta = L_zeta @ rng.standard_normal(2)
            eta_A = rho * np.tanh(tanh_scale * eta_A) + zeta[0]
            eta_B = rho * np.tanh(tanh_scale * eta_B) + zeta[1]
        path = np.zeros((T, N))
        for t in range(T):
            zeta = L_zeta @ rng.standard_normal(2)
            eta_A = rho * np.tanh(tanh_scale * eta_A) + zeta[0]
            eta_B = rho * np.tanh(tanh_scale * eta_B) + zeta[1]
            path[t, :n_block_A] = (Lam_A * eta_A
                                    + rng.normal(0, idiosyncratic_sd, n_block_A))
            path[t, n_block_A:] = (Lam_B * eta_B
                                    + rng.normal(0, idiosyncratic_sd, n_block_B))
        unit_paths.append(path)

    Y = np.vstack(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    split_spec = [T] * U

    return {
        'Y': Y, 'split_spec': split_spec,
        'reflective_indices': list(range(N)),
        'n_block_A': n_block_A, 'n_block_B': n_block_B,
    }



# §3. Core utilities (same as Block 7, Block 5, Block 1)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]
    null_PRs = np.zeros(B)
    for b in range(B):
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed, U_resample, T_resample):
    """Parametric AR(1) null."""
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def compute_indicator_correlation_summary(Y, n_block_A, n_block_B):
    """Compute mean off-diag, within-block, cross-block correlations.
    Returns a dict for sanity-checking that we hit the target correlation."""
    C = np.corrcoef(Y, rowvar=False)
    N = C.shape[0]
    idx_A = np.arange(n_block_A)
    idx_B = np.arange(n_block_A, n_block_A + n_block_B)
    return {
        'mean_corr_all_offdiag': float(C[~np.eye(N, dtype=bool)].mean()),
        'mean_corr_within_A': float(C[np.ix_(idx_A, idx_A)][~np.eye(n_block_A, dtype=bool)].mean()),
        'mean_corr_within_B': float(C[np.ix_(idx_B, idx_B)][~np.eye(n_block_B, dtype=bool)].mean()),
        'mean_corr_cross_AB': float(C[np.ix_(idx_A, idx_B)].mean()),
    }


print("Utilities defined.")

# §4. Sweep loop


print("\n" + "=" * 70)
print(f"R1-dynsplit sweep: 6 configs × {N_SEEDS} seeds × 2 nulls (B={B_PERMS})")
print("=" * 70)

incremental_csv = os.path.join(OUT_DIR, 'r1_dynsplit_sweep_incremental.csv')

# Resume support
completed = set()
all_results = []
if os.path.exists(incremental_csv):
    df_prev = pd.read_csv(incremental_csv)
    for _, r in df_prev.iterrows():
        completed.add((r['config_name'], int(r['seed'])))
        all_results.append(r.to_dict())
    print(f"Resumed: {len(completed)} runs already done")
else:
    print(f"Starting fresh: {incremental_csv}")

t_start = time.time()
total_runs = len(CONFIGS) * N_SEEDS
done_count = len(completed)

for cfg_name, n_block_A, n_block_B, idio_sd, note in CONFIGS:
    for seed_idx in range(N_SEEDS):
        seed = 2000 + seed_idx
        key = (cfg_name, seed)
        if key in completed:
            continue

        run_t0 = time.time()
        regime = generate_R1_dynsplit(
            seed=seed,
            n_block_A=n_block_A, n_block_B=n_block_B,
            idiosyncratic_sd=idio_sd,
        )
        Y = regime['Y']; ss = regime['split_spec']
        ref_idx = regime['reflective_indices']

        corr_summary = compute_indicator_correlation_summary(
            Y, n_block_A, n_block_B)

        # Row-perm null on signed A
        A_obs = fit_ols_var_signed(Y, ss)
        d_rp, p_rp, pr_rp, mu_rp, sd_rp = compute_delta_pr_rowperm(
            A_obs, ref_idx, B=B_PERMS, seed=seed * 13 + 7)

        # Parametric null
        d_pa, p_pa, pr_pa, mu_pa, sd_pa = compute_delta_pr_parametric(
            Y, ss, ref_idx, B=B_PERMS, seed=seed * 17 + 11,
            U_resample=U_UNITS, T_resample=T_PER_UNIT)

        passes_rp = bool(np.isfinite(d_rp) and d_rp > 0 and p_rp < ALPHA)
        passes_pa = bool(np.isfinite(d_pa) and d_pa > 0 and p_pa < ALPHA)
        elapsed = time.time() - run_t0

        row = {
            'config_name': cfg_name, 'seed': seed,
            'n_block_A': n_block_A, 'n_block_B': n_block_B,
            'idiosyncratic_sd': idio_sd,
            'PR_obs': pr_rp,                    # same as pr_pa by construction
            'mean_corr_all_offdiag': corr_summary['mean_corr_all_offdiag'],
            'mean_corr_within_A': corr_summary['mean_corr_within_A'],
            'mean_corr_within_B': corr_summary['mean_corr_within_B'],
            'mean_corr_cross_AB': corr_summary['mean_corr_cross_AB'],
            'rowperm_delta': d_rp, 'rowperm_p': p_rp,
            'rowperm_null_mean': mu_rp, 'rowperm_null_sd': sd_rp,
            'parametric_delta': d_pa, 'parametric_p': p_pa,
            'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
            'rowperm_passes': passes_rp,
            'parametric_passes': passes_pa,
            'elapsed_sec': elapsed,
        }
        all_results.append(row)
        done_count += 1

        # Sanity check: PR_obs match between nulls
        if abs(pr_rp - pr_pa) > 1e-9:
            print(f"  WARNING: PR_obs mismatch row-perm={pr_rp:.6f} vs param={pr_pa:.6f} "
                  f"in {cfg_name} seed={seed}")

        # Incremental save
        pd.DataFrame(all_results).to_csv(incremental_csv, index=False)

        total_elapsed = (time.time() - t_start) / 60
        print(f"  {cfg_name} seed={seed}: corr={corr_summary['mean_corr_all_offdiag']:.3f}, "
              f"PR_obs={pr_rp:.4f} | rp ΔPR={d_rp:+.4f} p={p_rp:.4f} {'✓' if passes_rp else '✗'} | "
              f"pa ΔPR={d_pa:+.4f} p={p_pa:.4f} {'✓' if passes_pa else '✗'} "
              f"[{elapsed:.1f}s, {done_count}/{total_runs}, {total_elapsed:.1f}min]")

print(f"\nSweep complete: {time.time()-t_start:.0f}s = {(time.time()-t_start)/60:.1f}min")



# §5. Per-config aggregate


print("\n" + "=" * 70)
print(f"AGGREGATE BY CONFIG (n_seeds={N_SEEDS})")
print("=" * 70)

df = pd.DataFrame(all_results)

print(f"\n{'config':22s} {'parts':>5s} {'σ_ε':>4s} {'mean_corr':>9s} | "
      f"{'PR_obs':>7s} | {'row-perm ΔPR/p':>17s} {'rp_pass':>7s} | "
      f"{'param ΔPR/p':>15s} {'pa_pass':>7s}")
print("-" * 120)

agg_rows = []
for cfg_name, n_block_A, n_block_B, idio_sd, note in CONFIGS:
    sub = df[df['config_name'] == cfg_name]
    if len(sub) == 0:
        continue
    pr_med = sub['PR_obs'].median()
    corr_med = sub['mean_corr_all_offdiag'].median()
    rp_d_med = sub['rowperm_delta'].median()
    rp_p_med = sub['rowperm_p'].median()
    pa_d_med = sub['parametric_delta'].median()
    pa_p_med = sub['parametric_p'].median()
    rp_pass_rate = sub['rowperm_passes'].mean()
    pa_pass_rate = sub['parametric_passes'].mean()
    rp_n_pass = int(sub['rowperm_passes'].sum())
    pa_n_pass = int(sub['parametric_passes'].sum())

    print(f"{cfg_name:22s} {n_block_A}/{n_block_B:<3d} {idio_sd:>4.2f} {corr_med:>9.3f} | "
          f"{pr_med:>7.4f} | "
          f"{rp_d_med:>+7.4f}/{rp_p_med:>5.4f}  {rp_n_pass:>2d}/{len(sub):<2d} | "
          f"{pa_d_med:>+7.4f}/{pa_p_med:>5.4f}  {pa_n_pass:>2d}/{len(sub):<2d}")

    agg_rows.append({
        'config_name': cfg_name,
        'n_block_A': n_block_A, 'n_block_B': n_block_B,
        'idiosyncratic_sd': idio_sd,
        'note': note,
        'n_seeds': len(sub),
        'PR_obs_median': float(pr_med),
        'mean_corr_median': float(corr_med),
        'rowperm_delta_median': float(rp_d_med),
        'rowperm_p_median': float(rp_p_med),
        'rowperm_p25': float(sub['rowperm_p'].quantile(0.25)),
        'rowperm_p75': float(sub['rowperm_p'].quantile(0.75)),
        'rowperm_pass_rate': float(rp_pass_rate),
        'parametric_delta_median': float(pa_d_med),
        'parametric_p_median': float(pa_p_med),
        'parametric_p25': float(sub['parametric_p'].quantile(0.25)),
        'parametric_p75': float(sub['parametric_p'].quantile(0.75)),
        'parametric_pass_rate': float(pa_pass_rate),
    })

df_agg = pd.DataFrame(agg_rows)
agg_csv = os.path.join(OUT_DIR, 'r1_dynsplit_sweep_aggregate.csv')
df_agg.to_csv(agg_csv, index=False)
print(f"\nSaved aggregate: {agg_csv}")



# §6. Interpretation


print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

# V-Dem reference for context
print(f"\nV-Dem reference values (from prior pipeline):")
print(f"  V-Dem all-16 mean off-diag corr: 0.54")
print(f"  V-Dem size-13 reflective block:  ~0.60 (estimated)")
print(f"  V-Dem T_cov:                     0.987")
print(f"  V-Dem ΔPR (OLS VAR):             +0.024 row-perm p=0.27, +0.045 param p=0.27 — FAILS BOTH")

# Find configs that match V-Dem correlation
print(f"\nConfigs ranked by closeness to V-Dem mean correlation (0.54):")
df_agg_sorted = df_agg.copy()
df_agg_sorted['vdem_dist'] = (df_agg_sorted['mean_corr_median'] - 0.54).abs()
df_agg_sorted = df_agg_sorted.sort_values('vdem_dist')
for _, r in df_agg_sorted.iterrows():
    print(f"  {r['config_name']:22s} corr={r['mean_corr_median']:.3f} (Δ={r['vdem_dist']:+.3f}) | "
          f"row-perm pass {int(r['rowperm_pass_rate']*r['n_seeds'])}/{int(r['n_seeds'])}, "
          f"param pass {int(r['parametric_pass_rate']*r['n_seeds'])}/{int(r['n_seeds'])}")

# Where does pass-rate drop below 50%?
print(f"\nPass-rate by config (sorted by σ_ε):")
df_agg_by_sd = df_agg.sort_values('idiosyncratic_sd')
for _, r in df_agg_by_sd.iterrows():
    rp_rate = r['rowperm_pass_rate']
    pa_rate = r['parametric_pass_rate']
    print(f"  σ_ε={r['idiosyncratic_sd']:.2f} ({r['config_name']:22s}): "
          f"row-perm {rp_rate:.0%}, param {pa_rate:.0%}")

# Substantive interpretation
print(f"\nSubstantive readings:")
baseline = df_agg[df_agg['config_name'] == 'baseline_6_6'].iloc[0]
vdem_match = df_agg[df_agg['config_name'] == 'split_7_6_vdem'].iloc[0]
print(f"  baseline_6_6 (paper):       row-perm pass-rate {baseline['rowperm_pass_rate']:.0%}, "
      f"param pass-rate {baseline['parametric_pass_rate']:.0%}")
print(f"  split_7_6_vdem (V-Dem-mc):  row-perm pass-rate {vdem_match['rowperm_pass_rate']:.0%}, "
      f"param pass-rate {vdem_match['parametric_pass_rate']:.0%}")

if vdem_match['parametric_pass_rate'] < 0.5 and baseline['rowperm_pass_rate'] >= 0.5:
    print(f"\n  ⇒ V-Dem-matched correlation breaks R1-dynsplit's pass under parametric null.")
    print(f"    The paper's borderline pass at default correlation does NOT survive at")
    print(f"    V-Dem-matched correlation. This closes the gap to V-Dem's actual failure.")
elif vdem_match['parametric_pass_rate'] >= 0.5:
    print(f"\n  ⇒ R1-dynsplit's parametric pass is robust to V-Dem-matched correlation.")
    print(f"    Cross-sectional/dynamic mismatch alone is NOT sufficient to reproduce V-Dem's failure.")
    print(f"    The paper's existing claim is well-supported by this fuller sweep.")
else:
    print(f"\n  ⇒ Mixed result, see per-config table above.")

print(f"\nDone.")

### Cell §3.3 — R1-dynsplit Sweep Verification

No re-run is required. The canonical sweep data is already in
 `r1_dynsplit_sweep_block_3_outputs/r1_dynsplit_sweep_aggregate.csv`.

This cell:
- 1. Re-reads the canonical aggregate file to confirm current values
- 2. Prints the canonical median dPR values that the paper table SHOULD report
- 3. Generates the LaTeX table rows

**Runtime:** <1 minute (just reads existing CSV).


In [4]:
from google.colab import drive
drive.mount('/content/drive')


import os
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
AGG_CSV   = os.path.join(DRIVE_DIR, 'r1_dynsplit_sweep_block_3_outputs',
                                     'r1_dynsplit_sweep_aggregate.csv')

df = pd.read_csv(AGG_CSV)
print("Canonical dynsplit sweep aggregate (existing file):")
print(df[['config_name', 'idiosyncratic_sd', 'mean_corr_median',
          'parametric_delta_median', 'parametric_pass_rate', 'rowperm_pass_rate']].to_string(index=False))
print()

# Generate corrected LaTeX rows
print("="*70)
print("LaTeX rows for tab:dynsplit-sweep:")
print("="*70)

# Order matches paper
config_order = ['baseline_6_6', 'split_7_6_tight', 'split_7_6_mid',
                'split_7_6_vdem', 'split_7_6_loose', 'split_7_6_extreme']
display_names = {
    'baseline_6_6':      r'baseline\_6\_6 (paper)  ',
    'split_7_6_tight':   r'split\_7\_6\_tight      ',
    'split_7_6_mid':     r'split\_7\_6\_mid        ',
    'split_7_6_vdem':    r'split\_7\_6\_vdem       ',
    'split_7_6_loose':   r'split\_7\_6\_loose      ',
    'split_7_6_extreme': r'split\_7\_6\_extreme    ',
}
partition_str = {'baseline_6_6': '6/6', 'split_7_6_tight': '7/6', 'split_7_6_mid': '7/6',
                 'split_7_6_vdem': '7/6', 'split_7_6_loose': '7/6', 'split_7_6_extreme': '7/6'}

for config in config_order:
    row = df[df['config_name']==config].iloc[0]
    sigma = row['idiosyncratic_sd']
    corr = row['mean_corr_median']
    dPR = row['parametric_delta_median']
    pass_rate = row['parametric_pass_rate']
    n_pass = int(round(pass_rate * row['n_seeds']))
    n_total = int(row['n_seeds'])

    if config == 'split_7_6_mid':
        # Bolded row in paper
        print(rf"\textbf{{{display_names[config].strip()}}} & \textbf{{{partition_str[config]}}} & "
              rf"$\mathbf{{{sigma:.2f}}}$ & $\mathbf{{{corr:.2f}}}$ & "
              rf"$\mathbf{{+{dPR:.3f}}}$ & $\mathbf{{{n_pass}/{n_total}}}$ \\")
    else:
        print(rf"{display_names[config]} & {partition_str[config]} & "
              rf"${sigma:.2f}$ & ${corr:.2f}$ & "
              rf"$+{dPR:.3f}$ & ${n_pass}/{n_total}$ \\")

print()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Canonical dynsplit sweep aggregate (existing file):
      config_name  idiosyncratic_sd  mean_corr_median  parametric_delta_median  parametric_pass_rate  rowperm_pass_rate
     baseline_6_6              0.30          0.740949                 0.206797                   1.0                0.3
  split_7_6_tight              0.30          0.745977                 0.233974                   1.0                0.5
    split_7_6_mid              0.45          0.643998                 0.241818                   1.0                0.0
   split_7_6_vdem              0.60          0.540360                 0.279917                   1.0                0.0
  split_7_6_loose              0.75          0.446788                 0.328036                   1.0                0.0
split_7_6_extreme              1.00          0.325447                 0.403836                   1.

### Cell §3.4 —Parametric AR(1) β-sweep calibration

Generates a table for Appendix B, which is the parametric AR(1) null calibration table.
 - **Section A** (white noise + independent_ar1, n=1000): preserved from existing file
 - **Section B** (row-permuting β-sweep, n=200 each): restored from canonical values
 - **Section D** (parametric β-sweep): n=200 at β ∈ {0.50, 0.70, 0.85},
   **n=500 at β ∈ {0.95, 0.99}** (boundary disambiguation per Decision 1 evaluation)
 - Existing β=0.50/0.70/0.85 rows are preserved if present (deterministic seeds); β=0.95/0.99 are re-run at n=500 regardless to get tight CIs at the unit-root.

Protocol:
 - Generate independent AR(1) panels with fixed β across all indicators
 - Apply parametric AR(1) null (per-indicator-fitted β/σ, B_PERMS=1000)
 - Reject at p < α=0.05; report rate
 - 200 panels per β value

 Runtime estimate: ~1 hour on CPU for full β-sweep (5 β values × 200 panels × B=1000 under parametric null at U=139, T=75). Parametric null is the slow component (per-iteration OLS VAR fit on a U·T × N panel).

**Runtime:**  ~150 minutes on CPU when run fresh (no cache); GPU does not provide an advantage:
   - β=0.50, 0.70, 0.85 at n=200: ~33 min combined
   - β=0.95 at n=500: ~55 min
  - β=0.99 at n=500: ~55 min

This cell will overwrite the summary CSV with the final canonical state.




In [1]:
from google.colab import drive
drive.mount('/content/drive')


import os, time
import numpy as np
import pandas as pd
from scipy import stats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'null_calibration_v2_outputs')
SUMMARY_CSV   = os.path.join(OUT_DIR, 'null_calibration_v2_summary.csv')
PER_PANEL_CSV = os.path.join(OUT_DIR, 'sec_D_param_betasweep_per_panel.csv')

# --- Configuration ---
N_BOUNDARY        = 500    # for β ∈ {0.95, 0.99} (unit-root boundary disambiguation)
N_INTERIOR        = 200    # for β ∈ {0.50, 0.70, 0.85}
B_PERMS           = 1000
ALPHA             = 0.05
U_UNITS           = 139    # V-Dem long_16var dimensions
T_PER_UNIT        = 75
N_INDICATORS      = 13
SIGMA_INNOV       = 1.0
INTERIOR_BETAS    = [0.50, 0.70, 0.85]
BOUNDARY_BETAS    = [0.95, 0.99]



# Canonical Section B (row-permuting β-sweep) — hardcoded restoration values from prior run.

SECTION_B_ROWS = [
    # (condition, beta, n_reject, reject_rate, ci95_low, ci95_high, mean_dPR)
    ('C2_ar1_beta0.50', 0.50,  8, 0.040, 0.020405387150656,  0.076932938294198, -0.001290915556259),
    ('C2_ar1_beta0.70', 0.70, 10, 0.050, 0.027382349028644,  0.089579056297843,  0.000172499789017),
    ('C2_ar1_beta0.85', 0.85, 14, 0.070, 0.042151782922918,  0.114055782166837,  0.003627005321704),
    ('C2_ar1_beta0.95', 0.95, 18, 0.090, 0.057686978714533,  0.137766746138489,  0.007462484849753),
    ('C2_ar1_beta0.99', 0.99, 23, 0.115, 0.077863172218243,  0.166648252338863,  0.012872273493799),
]



# Canonical helpers (idioms from parametric_null_applications_v2.py)


def fit_ols_var(Y, split_spec):
    """Pooled OLS VAR fit, diagonal zeroed."""
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    """Masked rank-1 ALS PR_obs computation."""
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v; den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new; den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def fit_per_indicator_ar1(Y, split_spec):
    """Per-indicator AR(1) fit for parametric null."""
    cursor = 0
    yt_list, yl_list = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_list.append(unit[1:]); yl_list.append(unit[:-1])
    yt = np.vstack(yt_list); yl = np.vstack(yl_list)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    """Vectorized parametric AR(1) panel simulator."""
    N = len(betas)
    sv = sigmas**2 / (1 - betas**2 + 1e-12)
    sv = np.where(np.abs(betas) < 1, sv, sigmas**2)
    sd = np.sqrt(sv)
    x = rng.standard_normal((U, N)) * sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    return (Y - Y.mean(0)) / (Y.std(0) + 1e-12)


def simulate_independent_ar1_panel(beta_const, sigma_const, U, T, N, rng):
    return simulate_ar1_panel_from_params(np.full(N, beta_const), np.full(N, sigma_const), U, T, rng)


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed):
    """Parametric AR(1) null applied to a panel."""
    A = fit_ols_var(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    U = len(split_spec); T = split_spec[0]
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U, T, rng)
        ss_sim = [T] * U
        A_sim = fit_ols_var(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs)
    delta = PR_obs - float(null_PRs.mean())
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs))


def run_param_betasweep(beta, n_panels, panels_existing):
    """Run the parametric β-sweep at given β with n_panels.

    If panels_existing has n_panels rows for this β, return cached results
    (deterministic seeds). Otherwise, run from scratch.
    """
    cached = panels_existing[
        (panels_existing['beta'] == beta) & (panels_existing['panel_idx'] < n_panels)
    ] if panels_existing is not None else pd.DataFrame()
    if len(cached) >= n_panels:
        print(f"  β={beta:.2f} (n={n_panels}): using {len(cached)} cached panels")
        per_panel = cached.head(n_panels).to_dict('records')
        rejects = sum(1 for r in per_panel if r['rejected'])
        deltas = [r['delta_PR'] for r in per_panel if np.isfinite(r['delta_PR'])]
        return per_panel, rejects, deltas

    # Run from scratch
    print(f"  β={beta:.2f} (n={n_panels}): running from scratch...", flush=True)
    t0 = time.time()
    per_panel = []; rejects = 0; deltas = []
    for panel_idx in range(n_panels):
        master_seed = 100000 + int(beta*100)*1000 + panel_idx
        rng_panel = np.random.default_rng(master_seed)
        Y = simulate_independent_ar1_panel(beta, SIGMA_INNOV, U_UNITS, T_PER_UNIT,
                                            N_INDICATORS, rng_panel)
        split_spec = [T_PER_UNIT] * U_UNITS
        ref = list(range(N_INDICATORS))
        delta, p, pr = compute_delta_pr_parametric(Y, split_spec, ref, B_PERMS, master_seed + 7)
        if np.isfinite(p) and p < ALPHA:
            rejects += 1
        if np.isfinite(delta):
            deltas.append(delta)
        per_panel.append({
            'beta': beta, 'panel_idx': panel_idx, 'master_seed': master_seed,
            'PR_obs': pr, 'delta_PR': delta, 'p_value': p,
            'rejected': bool(np.isfinite(p) and p < ALPHA),
        })
        if (panel_idx + 1) % 50 == 0:
            elapsed = time.time() - t0
            print(f"    {panel_idx+1}/{n_panels}  ({elapsed:.0f}s, {rejects} rejects)", flush=True)
    print(f"  → β={beta:.2f}: {rejects}/{n_panels} rejects, elapsed={time.time()-t0:.0f}s")
    return per_panel, rejects, deltas



# Step 1: Read existing per-panel CSV (cache for already-run β values)

print("=" * 70)
print("Step 1: Check for cached per-panel results")
print("=" * 70)
panels_existing = None
if os.path.exists(PER_PANEL_CSV):
    panels_existing = pd.read_csv(PER_PANEL_CSV)
    print(f"Found existing per-panel file: {PER_PANEL_CSV}")
    print(f"  Total rows: {len(panels_existing)}")
    print(f"  Per-β counts: {panels_existing.groupby('beta').size().to_dict()}")
else:
    print(f"No existing per-panel file. Will run all β from scratch.")


# Step 2: Run all 5 β values (cached if possible, fresh if needed)

print()
print("=" * 70)
print(f"Step 2: Parametric β-sweep")
print(f"  Interior βs (n={N_INTERIOR}): {INTERIOR_BETAS}")
print(f"  Boundary βs (n={N_BOUNDARY}): {BOUNDARY_BETAS}")
print("=" * 70)

new_section_D_rows = []
all_per_panel = []
overall_t0 = time.time()

# Interior βs at n=200 (use cached if available)
for beta in INTERIOR_BETAS:
    per_panel, rejects, deltas = run_param_betasweep(beta, N_INTERIOR, panels_existing)
    all_per_panel.extend(per_panel)
    rate = rejects / N_INTERIOR
    ci = stats.binomtest(rejects, N_INTERIOR).proportion_ci(method='wilson')
    new_section_D_rows.append({
        'section': 'D', 'null': 'parametric_AR1',
        'condition': f'C2_ar1_beta{beta:.2f}',
        'n_panels': N_INTERIOR, 'n_reject': rejects, 'reject_rate': rate,
        'ci95_low': float(ci.low), 'ci95_high': float(ci.high),
        'mean_delta_PR': float(np.mean(deltas)) if deltas else np.nan,
        'beta': beta,
    })

# Boundary βs at n=500 (always re-run from scratch since cached is n=200)
# Force fresh run for boundary betas by setting panels_existing to empty for them
for beta in BOUNDARY_BETAS:
    cached_for_beta = panels_existing[
        (panels_existing['beta'] == beta) & (panels_existing['panel_idx'] < N_BOUNDARY)
    ] if panels_existing is not None else pd.DataFrame()
    if len(cached_for_beta) >= N_BOUNDARY:
        # Already n=500 cached
        per_panel, rejects, deltas = run_param_betasweep(beta, N_BOUNDARY, panels_existing)
    else:
        # Need to run additional panels (or all of them)
        per_panel, rejects, deltas = run_param_betasweep(beta, N_BOUNDARY, panels_existing)
    all_per_panel.extend(per_panel)
    rate = rejects / N_BOUNDARY
    ci = stats.binomtest(rejects, N_BOUNDARY).proportion_ci(method='wilson')
    new_section_D_rows.append({
        'section': 'D', 'null': 'parametric_AR1',
        'condition': f'C2_ar1_beta{beta:.2f}',
        'n_panels': N_BOUNDARY, 'n_reject': rejects, 'reject_rate': rate,
        'ci95_low': float(ci.low), 'ci95_high': float(ci.high),
        'mean_delta_PR': float(np.mean(deltas)) if deltas else np.nan,
        'beta': beta,
    })

print(f"\nTotal sweep elapsed: {(time.time()-overall_t0)/60:.1f} min")



# Step 3: Save per-panel CSV

df_per_panel = pd.DataFrame(all_per_panel)
df_per_panel.to_csv(PER_PANEL_CSV, index=False)
print(f"\nWrote per-panel CSV: {PER_PANEL_CSV} ({len(df_per_panel)} rows)")



# Step 4: Build final summary CSV (Section A preserved + Section B restored
#                                    + Section D fresh)

print()
print("=" * 70)
print("Step 4: Assemble final summary CSV")
print("=" * 70)

# Read existing summary (for Section A preservation)
df_existing_summary = pd.read_csv(SUMMARY_CSV) if os.path.exists(SUMMARY_CSV) else pd.DataFrame()

# Section A: keep from existing
section_A = df_existing_summary[df_existing_summary['section'] == 'A'] if len(df_existing_summary) > 0 else pd.DataFrame()
print(f"Section A: preserved {len(section_A)} rows from existing")

# Section B: rebuild from hardcoded canonical values
section_B_rows = []
for cond, beta, n_rej, rate, ci_lo, ci_hi, mean_dpr in SECTION_B_ROWS:
    section_B_rows.append({
        'section': 'B', 'null': 'row-permuting',
        'condition': cond, 'n_panels': 200, 'n_reject': n_rej, 'reject_rate': rate,
        'ci95_low': ci_lo, 'ci95_high': ci_hi,
        'mean_delta_PR': mean_dpr, 'beta': beta,
    })
section_B = pd.DataFrame(section_B_rows)
print(f"Section B: restored {len(section_B)} rows from canonical values")

# Section D non-β rows: keep from existing (white_noise + independent_ar1)
section_D_baseline = df_existing_summary[
    (df_existing_summary['section'] == 'D') &
    (df_existing_summary['condition'].isin(['C1_white_noise', 'C2_independent_ar1']))
] if len(df_existing_summary) > 0 else pd.DataFrame()
print(f"Section D baseline: preserved {len(section_D_baseline)} rows (white_noise + independent_ar1)")

# Section D β-sweep rows: from this run
section_D_betasweep = pd.DataFrame(new_section_D_rows)
print(f"Section D β-sweep: {len(section_D_betasweep)} new rows")

# Concatenate and save
df_final = pd.concat([section_A, section_B, section_D_baseline, section_D_betasweep],
                     ignore_index=True)
df_final.to_csv(SUMMARY_CSV, index=False)
print(f"\nWrote {SUMMARY_CSV}: {len(df_final)} total rows")
print()
print("Final state by (section, null):")
print(df_final.groupby(['section', 'null']).size().to_string())
print()
print("Section D parametric_AR1 β-sweep:")
sub = df_final[(df_final['null']=='parametric_AR1') & (df_final['beta'].notna())].sort_values('beta')
print(sub[['condition','n_panels','n_reject','reject_rate','ci95_low','ci95_high','beta']].to_string(index=False))

Mounted at /content/drive
Step 1: Check for cached per-panel results
No existing per-panel file. Will run all β from scratch.

Step 2: Parametric β-sweep
  Interior βs (n=200): [0.5, 0.7, 0.85]
  Boundary βs (n=500): [0.95, 0.99]
  β=0.50 (n=200): running from scratch...
    50/200  (158s, 5 rejects)
    100/200  (317s, 5 rejects)
    150/200  (480s, 6 rejects)
    200/200  (644s, 8 rejects)
  → β=0.50: 8/200 rejects, elapsed=644s
  β=0.70 (n=200): running from scratch...
    50/200  (163s, 1 rejects)
    100/200  (325s, 4 rejects)
    150/200  (487s, 6 rejects)
    200/200  (648s, 11 rejects)
  → β=0.70: 11/200 rejects, elapsed=648s
  β=0.85 (n=200): running from scratch...
    50/200  (161s, 0 rejects)
    100/200  (323s, 3 rejects)
    150/200  (484s, 7 rejects)
    200/200  (646s, 9 rejects)
  → β=0.85: 9/200 rejects, elapsed=646s
  β=0.95 (n=500): running from scratch...
    50/500  (162s, 6 rejects)
    100/500  (323s, 8 rejects)
    150/500  (480s, 14 rejects)
    200/500  (638s

## Section 4 — §5 V-Dem-16 (long_16var) application

V-Dem-16 is the worked failure-and-repair case in §5. Layer 1 (substrate diagnostic via OLS VAR) fails at α=0.05; three of five comparator methods at Layer 2 produce W_hat that passes ΔPR; Layer 3 (post-EDA repair) flips the verdict to PASS after dropping `Elected_officials`.

- Cell §4.1 is the canonical 5-method random-baseline sweep (CRPS + rb-beats); it produces `method_outputs.pkl` consumed by every downstream cell.

- Cells §4.2 and §4.3 are superseded DYNOTEARS bring-up cells from paper development; anyone running this script top-to-bottom can skip them — they are pure DYNOTEARS infrastructure and their results are subsumed by §4.1.

- Cell §4.4 is the LSC diagnostic on the §4.1 method outputs and identifies V-Dem-16's reflective groups, writing `groups_identified.json` consumed by every V-Dem-dependent cell that follows.

- Cell §4.5 is a superseded verification cell that re-runs the LSC + multi-criteria analysis after merging DYNOTEARS in via the historical bring-up pathway; anyone running this script top to bottom can skip it. Cells §4.6–§4.10 are multi-criteria evaluation, era diagnostics, SRMR robustness, and the era-decomposed verdict-flip producing Table 4.

- Cells §4.11–§4.15 are V-Dem-dependent block analyses (parametric null on V-Dem, signed/unsigned ablation, era subsets, adversary main sweep) that load `groups_identified.json` produced by §4.4.

### Cell §4.1 — V-Dem-16 random-baseline sweep (5 methods × 30 baselines, CANONICAL)

**Paper sections:** §5, Appendix P.

The canonical 5-method sweep on V-Dem-16. Fits all 5 methods (linear_var_granger, pcmci, cmlp, navar, dynotears), generates 30 random baselines per method (10 each from iid_density_matched, perm_rows, perm_full), scores all under both unconstrained and constrained CRPS protocols, and computes empirical p-values for the rb-beats criterion. Produces Table 3 in the paper (V-Dem-16 row).

The CRPS protocol is panel-safe: residual pool built from within-unit (t-1, t) pairs only (no cross-country leakage); evaluation window is per-unit (last H within-unit pairs of each unit, averaged across units), not the last H rows of the stacked panel.

**Inputs:** V-Dem panel. Optional `methods_pkl_path_resume` for warm-start.

**Outputs:** `sweep_raw.csv` (all method × baseline × protocol scores), `verdict.csv` (rb-beats p-values), `verdict.txt` (human-readable), `fig_baseline_crps.png`, `method_outputs.pkl` (5-method pickle).

**Self-contained:** Yes — redefines all setup functions inline.

**Runtime:** ~11 minutes on Colab Pro G4 GPU.

In [ ]:
# 1. Setup


from google.colab import drive
drive.mount('/content/drive')

import os, sys, pickle

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
SRC_DIR   = os.path.join(DRIVE_DIR, 'src')
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR_BASE = os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs')
os.makedirs(OUT_DIR_BASE, exist_ok=True)

sys.path = [p for p in sys.path if p != '/mnt/user-data/uploads']
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Verify NAVAR codebase + DYNOTEARS module + V-Dem CSVs reachable
print("Source files in SRC_DIR:")
for f in ['NAVAR.py', 'dataloader.py', 'train_NAVAR.py', 'edge_ablation.py',
          'dynotears_panel.py']:
    p = os.path.join(SRC_DIR, f)
    print(f"  {f}: {'EXISTS' if os.path.exists(p) else 'MISSING'} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} B)")

print("\nV-Dem data files:")
for f in ['my_data_16.csv', 'my_data_75_years.csv']:
    p = os.path.join(DATA_DIR, f)
    print(f"  {f}: {'EXISTS' if os.path.exists(p) else 'MISSING'} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} B)")

print(f"\nDRIVE_DIR    = {DRIVE_DIR}")
print(f"OUT_DIR_BASE = {OUT_DIR_BASE}")



!pip install -q tigramite

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device={DEVICE}")



# 2. Run configuration


import numpy as np

CONFIG_NAME = "long_16var"   # or "medium_16var"

N_REPS_PER_FAMILY = 10

METHODS_TO_RUN = ["linear_var_granger", "pcmci", "cmlp", "navar", "dynotears"]

H_HORIZON = 20
B_BOOT    = 300
GRID_SIZE = 50
RIDGE     = 1e-4
BANDWIDTH = 0.2
EPSILON   = 0.01
SEED      = 0

# Fix 4: deterministic per-method RNG seeds. hash(str) is NOT reproducible
# across Python sessions because of PYTHONHASHSEED randomization.
METHOD_RNG_OFFSETS = {
    "linear_var_granger": 11,
    "pcmci":               23,
    "cmlp":                37,
    "navar":               41,
    "dynotears":           53,
}

OUT_DIR = os.path.join(OUT_DIR_BASE, CONFIG_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"CONFIG_NAME = {CONFIG_NAME}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"Methods: {METHODS_TO_RUN}")
print(f"N_REPS_PER_FAMILY = {N_REPS_PER_FAMILY}")
print(f"3 baseline families x {N_REPS_PER_FAMILY} reps = {3*N_REPS_PER_FAMILY} baselines per method")
print(f"Total tvNAR fits: {len(METHODS_TO_RUN)} methods x (1 + {3*N_REPS_PER_FAMILY} baselines) "
      f"x 2 protocols = {len(METHODS_TO_RUN)*(1+3*N_REPS_PER_FAMILY)*2} fits")


# 3. Panel spec and loader


from dataclasses import dataclass
from typing import List, Optional
import pandas as pd

MIN_SERIES_LENGTH = 9
VAL_YEARS         = 5


@dataclass
class PanelSpec:
    name: str
    filename: str
    variables: List[str]


_VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]


PANEL_SPECS = {
    "medium_16var": PanelSpec(name="medium_16var", filename="my_data_16.csv",
                              variables=_VDEM_VARIABLES),
    "long_16var":   PanelSpec(name="long_16var",   filename="my_data_75_years.csv",
                              variables=_VDEM_VARIABLES),
}


def _load_csv(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")


def load_vdem_panel(config_name, data_dir=DATA_DIR):
    if config_name not in PANEL_SPECS:
        raise ValueError(f"Unknown config: {config_name}")
    spec = PANEL_SPECS[config_name]
    csv_path = os.path.join(data_dir, spec.filename)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"{csv_path} not found.")

    df_raw = _load_csv(csv_path)
    print(f"Loaded {spec.filename}: {df_raw.shape}")

    if "country_id" not in df_raw.columns or "year" not in df_raw.columns:
        raise ValueError("Expected 'country_id' and 'year' columns.")

    keep_cols = ["country_id", "year"] + spec.variables
    if "country_name" in df_raw.columns:
        keep_cols.append("country_name")
    df_panel = (df_raw[keep_cols]
                .dropna(subset=spec.variables)
                .sort_values(["country_id", "year"])
                .reset_index(drop=True))

    lengths = df_panel.groupby("country_id")["year"].count()
    eligible = lengths[lengths >= MIN_SERIES_LENGTH].index
    df_panel = df_panel[df_panel["country_id"].isin(eligible)]

    train_parts = []
    for cid, g in df_panel.groupby("country_id", sort=False):
        g = g.sort_values("year").reset_index(drop=True)
        if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
            train_parts.append(g.iloc[:-VAL_YEARS])
        else:
            train_parts.append(g)

    train_df = pd.concat(train_parts, ignore_index=True)
    split_spec = (train_df.groupby("country_id", sort=False)["year"]
                  .count().tolist())
    Y_train = train_df[spec.variables].to_numpy(float)
    var_names = spec.variables

    print(f"  {config_name}: U={len(split_spec)} units, "
          f"mean T_per_unit={np.mean(split_spec):.1f}, "
          f"range [{min(split_spec)}, {max(split_spec)}], "
          f"N={Y_train.shape[1]}")
    return Y_train, split_spec, var_names


Y_train, split_spec, var_names = load_vdem_panel(CONFIG_NAME)
N = Y_train.shape[1]
U = len(split_spec)

mu_train = Y_train.mean(axis=0)
sd_train = Y_train.std(axis=0)
sd_train = np.where(sd_train < 1e-12, 1.0, sd_train)
Y_norm = (Y_train - mu_train) / sd_train
print(f"\nY_norm: shape={Y_norm.shape}")



# 4. tvNAR fitters (unconstrained + constrained)


from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union
import numpy as np

ArrayLike = Union[np.ndarray]


def _as_2d_float(X, name):
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError(f"{name} must be 2D, got shape {X.shape}")
    return X.astype(float, copy=False)


def _split_lengths_to_slices(lengths):
    if any(l <= 1 for l in lengths):
        raise ValueError("All series lengths must be >= 2.")
    slices = []
    start = 0
    for L in lengths:
        slices.append(slice(start, start + L))
        start += L
    return slices


def _infer_split_lengths(T, split_timeseries):
    if split_timeseries is None or split_timeseries is False:
        return None
    if isinstance(split_timeseries, bool):
        raise ValueError("split_timeseries=True is ambiguous.")
    if isinstance(split_timeseries, (int, np.integer)):
        L = int(split_timeseries)
        if L <= 1 or T % L != 0:
            raise ValueError(f"T={T} not divisible by split_timeseries={L}.")
        return [L] * (T // L)
    lengths = [int(x) for x in list(split_timeseries)]
    if sum(lengths) != T:
        raise ValueError(f"sum={sum(lengths)} != T={T}.")
    return lengths


@dataclass
class PanelLagPairs:
    y_prev: np.ndarray   # (M, N)
    y_curr: np.ndarray   # (M, N)
    tau: np.ndarray      # (M,) within-unit normalized time
    series_id: np.ndarray  # (M,) integer unit id


def build_panel_lag_pairs(Y, split_timeseries=None, tau_mode="within_series"):
    """
    Build (t-1, t) lag pairs WITHIN each unit. Never crosses unit boundaries.

    The pair (Y[t-1], Y[t]) is included only if both timepoints belong to the
    same unit. This is the panel-safe primitive used everywhere downstream.
    """
    Y = _as_2d_float(Y, "Y")
    T, N = Y.shape
    lengths = _infer_split_lengths(T, split_timeseries)

    if lengths is None:
        # Single series
        y_prev, y_curr = Y[:-1], Y[1:]
        if tau_mode in ("within_series", "global_index"):
            tau = np.arange(1, T) / (T - 1)
        else:
            raise ValueError("tau_mode must be 'within_series' or 'global_index'.")
        return PanelLagPairs(y_prev=y_prev, y_curr=y_curr, tau=tau,
                             series_id=np.zeros(T - 1, dtype=int))

    slices = _split_lengths_to_slices(lengths)
    y_prev_list, y_curr_list, tau_list, sid_list = [], [], [], []
    for sid, sl in enumerate(slices):
        Ys = Y[sl]
        L = Ys.shape[0]
        y_prev_list.append(Ys[:-1])
        y_curr_list.append(Ys[1:])
        if tau_mode == "within_series":
            tau_list.append(np.arange(1, L) / (L - 1))
        elif tau_mode == "global_index":
            tau_list.append(np.arange(sl.start + 1, sl.stop) / (T - 1))
        else:
            raise ValueError("tau_mode must be 'within_series' or 'global_index'.")
        sid_list.append(np.full(L - 1, sid, dtype=int))

    return PanelLagPairs(
        y_prev=np.vstack(y_prev_list),
        y_curr=np.vstack(y_curr_list),
        tau=np.concatenate(tau_list),
        series_id=np.concatenate(sid_list),
    )


def gaussian_kernel(u): return np.exp(-0.5 * (u ** 2))
def epanechnikov_kernel(u):
    w = np.zeros_like(u, dtype=float)
    m = np.abs(u) <= 1.0
    w[m] = 0.75 * (1.0 - u[m] ** 2)
    return w


def _get_kernel(name):
    if name.lower() in ("gaussian", "normal"): return gaussian_kernel
    if name.lower() in ("epanechnikov", "epa"): return epanechnikov_kernel
    raise ValueError("kernel must be 'gaussian' or 'epanechnikov'.")


def fit_tvNAR_lambda_paths(Y, W, *, split_timeseries=None, tau_grid=None,
                            grid_size=50, bandwidth=0.15, kernel="gaussian",
                            ridge=1e-4, tau_mode="within_series", center=False):
    Y = _as_2d_float(Y, "Y"); W = _as_2d_float(W, "W")
    T, N = Y.shape
    if W.shape != (N, N):
        raise ValueError(f"W must be ({N},{N}), got {W.shape}")

    if center:
        Y = Y - Y.mean(axis=0, keepdims=True)

    pairs = build_panel_lag_pairs(Y, split_timeseries=split_timeseries, tau_mode=tau_mode)
    y_prev, y_curr, tau = pairs.y_prev, pairs.y_curr, pairs.tau
    M = y_prev.shape[0]
    if M == 0:
        raise ValueError("No usable (t-1,t) pairs.")

    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, int(grid_size))
    else:
        tau_grid = np.asarray(tau_grid, dtype=float).ravel()

    if not (0 < bandwidth <= 1.0):
        raise ValueError("bandwidth must be in (0, 1].")

    K = _get_kernel(kernel)
    WpI = W + np.eye(N, dtype=float)
    lambdas = np.zeros((tau_grid.size, N), dtype=float)

    for g, tau0 in enumerate(tau_grid):
        u = (tau - tau0) / bandwidth
        w = K(u)
        if w.sum() <= 1e-12:
            w = np.ones_like(w)
        A = np.zeros((N, N), dtype=float)
        b = np.zeros(N, dtype=float)
        for m in range(M):
            wm = w[m]
            if wm <= 0:
                continue
            X = WpI * y_prev[m][None, :]
            y = y_curr[m]
            A += wm * (X.T @ X)
            b += wm * (X.T @ y)
        A += ridge * np.eye(N, dtype=float)
        try:
            lambdas[g] = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            lambdas[g] = np.linalg.lstsq(A, b, rcond=None)[0]

    return {
        "tau_grid": tau_grid, "lambda": lambdas, "W_plus_I": WpI,
        "meta": {"bandwidth": float(bandwidth), "kernel": kernel,
                 "ridge": float(ridge), "tau_mode": tau_mode,
                 "center": bool(center), "n_pairs": int(M)},
    }



def _project_lambda_row(lam_row, epsilon):
    m = float(np.max(np.abs(lam_row)))
    boundary = 1.0 - epsilon
    if m > boundary:
        return lam_row * (boundary / m), True
    return lam_row, False


def project_lambda_stable(tv_result, epsilon=0.01):
    if "lambda" not in tv_result:
        raise KeyError("tv_result must contain 'lambda' key.")
    lam_raw = np.asarray(tv_result["lambda"], dtype=float)
    if lam_raw.ndim != 2:
        raise ValueError(f"lambda must be 2D, got {lam_raw.shape}")
    if not 0.0 <= epsilon < 1.0:
        raise ValueError(f"epsilon in [0,1); got {epsilon}")
    lam_proj = np.empty_like(lam_raw)
    n_projected = 0
    for g in range(lam_raw.shape[0]):
        lam_proj[g], was = _project_lambda_row(lam_raw[g], epsilon)
        if was:
            n_projected += 1
    result = dict(tv_result)
    result["lambda"] = lam_proj
    result["lambda_raw"] = lam_raw
    result["n_projected"] = int(n_projected)
    result["n_grid_rows"] = int(lam_raw.shape[0])
    result["max_abs_lam_raw"] = float(np.max(np.abs(lam_raw)))
    result["max_abs_lam_projected"] = float(np.max(np.abs(lam_proj)))
    result["projection_epsilon"] = float(epsilon)
    if "meta" in result and isinstance(result["meta"], dict):
        result["meta"] = dict(result["meta"])
        result["meta"]["projected"] = True
        result["meta"]["projection_epsilon"] = float(epsilon)
    return result


def fit_tvNAR_lambda_paths_constrained(Y, W, *, constrain=True, epsilon=0.01,
                                        fit_fn=None, **kwargs):
    if fit_fn is None:
        fit_fn = fit_tvNAR_lambda_paths
    tv_raw = fit_fn(Y=Y, W=W, **kwargs)
    if constrain:
        tv_out = project_lambda_stable(tv_raw, epsilon=epsilon)
    else:
        lam = np.asarray(tv_raw["lambda"], dtype=float)
        max_per_row = np.max(np.abs(lam), axis=1)
        tv_out = dict(tv_raw)
        tv_out["lambda_raw"] = lam.copy()
        tv_out["n_projected"] = 0
        tv_out["n_grid_rows"] = int(lam.shape[0])
        tv_out["max_abs_lam_raw"] = float(np.max(max_per_row))
        tv_out["max_abs_lam_projected"] = float(np.max(max_per_row))
        tv_out["projection_epsilon"] = float(epsilon)
        if "meta" in tv_out and isinstance(tv_out["meta"], dict):
            tv_out["meta"] = dict(tv_out["meta"])
            tv_out["meta"]["projected"] = False
            tv_out["meta"]["projection_epsilon"] = float(epsilon)
    tv_out["constrain_requested"] = bool(constrain)
    return tv_out



# 5. CRPS scorer (panel-safe — Fixes 1 and 2)


from typing import Dict, Optional, Tuple, Any, List


def crps_1d_from_samples(samples_1d, y_true):
    x = np.sort(samples_1d.astype(float))
    Bn = x.size
    if Bn == 0:
        raise ValueError("CRPS requires at least one sample")
    term1 = float(np.mean(np.abs(x - y_true)))
    i = np.arange(1, Bn + 1, dtype=float)
    sum_abs = 2.0 * float(np.sum((2.0 * i - Bn - 1.0) * x))
    term2 = 0.5 * (sum_abs / (Bn * Bn))
    return term1 - term2


def mean_crps(y_true_vec, samples_BN):
    y_true_vec = np.asarray(y_true_vec, dtype=float)
    samples_BN = np.asarray(samples_BN, dtype=float)
    if samples_BN.ndim != 2 or samples_BN.shape[1] != y_true_vec.size:
        raise ValueError("samples_BN shape mismatch")
    return float(np.mean([
        crps_1d_from_samples(samples_BN[:, j], y_true_vec[j])
        for j in range(y_true_vec.size)
    ]))


def tvnar_mean_one_step(W, lam_vec, y_prev):
    N = W.shape[0]
    return ((W + np.eye(N)) * y_prev[None, :]) @ lam_vec


def score_W_against_panel(W, Y, split_spec, *, H=10, B=500, bandwidth=0.2,
                           ridge=1e-4, grid_size=30, kernel="gaussian",
                           lambda_policy="last", constrain=False, epsilon=0.01,
                           rng=None, fit_fn=None):

    if rng is None:
        rng = np.random.default_rng(0)
    if fit_fn is None:
        fit_fn = fit_tvNAR_lambda_paths

    T_panel, N = Y.shape
    if W.shape != (N, N):
        return {"status": f"skip: W shape {W.shape} != ({N}, {N})",
                "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
                "nnz": int((np.abs(W) > 1e-12).sum()),
                "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
                "n_projected": 0, "constrain_requested": bool(constrain),
                "protocol": "constrained" if constrain else "unconstrained",
                "epsilon": float(epsilon), "n_eval_pairs": 0}
    if np.allclose(W, 0.0):
        return {"status": "skip: all-zero W",
                "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
                "nnz": 0, "max_abs_lam_raw": np.nan,
                "max_abs_lam_projected": np.nan, "n_projected": 0,
                "constrain_requested": bool(constrain),
                "protocol": "constrained" if constrain else "unconstrained",
                "epsilon": float(epsilon), "n_eval_pairs": 0}

    # Fit tvNAR (the fit itself already uses panel-safe pairs internally)
    try:
        tv = fit_tvNAR_lambda_paths_constrained(
            Y=Y, W=W, constrain=constrain, epsilon=epsilon, fit_fn=fit_fn,
            split_timeseries=split_spec, grid_size=grid_size,
            bandwidth=bandwidth, kernel=kernel, ridge=ridge,
            tau_mode="within_series", center=False,
        )
    except Exception as e:
        return {"status": f"fit_fail: {e}",
                "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
                "nnz": int((np.abs(W) > 1e-12).sum()),
                "max_abs_lam_raw": np.nan, "max_abs_lam_projected": np.nan,
                "n_projected": 0, "constrain_requested": bool(constrain),
                "protocol": "constrained" if constrain else "unconstrained",
                "epsilon": float(epsilon), "n_eval_pairs": 0}

    lam_grid = np.asarray(tv["lambda"], dtype=float)
    if lambda_policy == "last":
        lam_use = lam_grid[-1]
    elif lambda_policy == "mean":
        lam_use = lam_grid.mean(axis=0)
    else:
        raise ValueError(f"Unknown lambda_policy '{lambda_policy}'")

    # ---- Panel-safe residual pool ----
    # Build (t-1, t) pairs WITHIN each unit; never cross unit boundaries.
    pairs = build_panel_lag_pairs(Y, split_timeseries=split_spec,
                                   tau_mode="within_series")
    M_pairs = pairs.y_prev.shape[0]
    resid_pool = np.zeros((M_pairs, N))
    for m in range(M_pairs):
        resid_pool[m] = pairs.y_curr[m] - tvnar_mean_one_step(
            W, lam_use, pairs.y_prev[m])

    # ---- Per-unit evaluation, then average ----
    # For each unit, identify its within-unit pairs and score CRPS on the
    # LAST H of them (or fewer if the unit is shorter than H+1).
    crps_per_unit = []
    sq_err_accum = 0.0
    abs_err_accum = 0.0
    n_eval_total = 0
    preds_all = []
    truths_all = []

    unit_ids = np.unique(pairs.series_id)
    for u in unit_ids:
        mask = pairs.series_id == u
        if mask.sum() == 0:
            continue
        # Last H pairs of this unit
        idx_unit = np.where(mask)[0]
        idx_eval = idx_unit[-H:] if len(idx_unit) >= H else idx_unit

        y_prev_u = pairs.y_prev[idx_eval]   # (H_u, N)
        y_curr_u = pairs.y_curr[idx_eval]   # (H_u, N)

        crps_unit = []
        for k in range(len(idx_eval)):
            mean_pred = tvnar_mean_one_step(W, lam_use, y_prev_u[k])
            # Bootstrap from the panel-safe residual pool
            boot_idx = rng.integers(0, resid_pool.shape[0], size=B)
            samples = mean_pred[None, :] + resid_pool[boot_idx]
            crps_unit.append(mean_crps(y_curr_u[k], samples))
            preds_all.append(mean_pred)
            truths_all.append(y_curr_u[k])
            sq_err_accum += float(np.mean((y_curr_u[k] - mean_pred) ** 2))
            abs_err_accum += float(np.mean(np.abs(y_curr_u[k] - mean_pred)))
            n_eval_total += 1
        crps_per_unit.append(np.mean(crps_unit))

    if n_eval_total == 0:
        return {"status": "skip: no eval pairs",
                "CRPS": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan,
                "nnz": int((np.abs(W) > 1e-12).sum()),
                "max_abs_lam_raw": float(tv["max_abs_lam_raw"]),
                "max_abs_lam_projected": float(tv["max_abs_lam_projected"]),
                "n_projected": int(tv["n_projected"]),
                "constrain_requested": bool(tv["constrain_requested"]),
                "protocol": "constrained" if constrain else "unconstrained",
                "epsilon": float(epsilon), "n_eval_pairs": 0}

    # CRPS averaged across units (each unit contributes equally)
    crps_panel = float(np.mean(crps_per_unit))
    mse_panel = sq_err_accum / n_eval_total
    mae_panel = abs_err_accum / n_eval_total

    return {
        "status":                "ok",
        "CRPS":                  crps_panel,
        "RMSE":                  float(np.sqrt(mse_panel)),
        "MAE":                   float(mae_panel),
        "MSE":                   float(mse_panel),
        "nnz":                   int((np.abs(W) > 1e-12).sum()),
        "max_abs_lam_raw":       float(tv["max_abs_lam_raw"]),
        "max_abs_lam_projected": float(tv["max_abs_lam_projected"]),
        "n_projected":           int(tv["n_projected"]),
        "constrain_requested":   bool(tv["constrain_requested"]),
        "protocol":              "constrained" if constrain else "unconstrained",
        "epsilon":               float(epsilon),
        "n_eval_pairs":          int(n_eval_total),
        "n_units_evaluated":     int(len(crps_per_unit)),
    }



# 6. Method runners
#
# (Method runners are unchanged; only the §5.3 evaluation infrastructure was fixed.)


# ### 6.1 Linear VAR Granger


import time
from scipy import stats


def _unstack_panel(Y, split_spec):
    units, cursor = [], 0
    for T_u in split_spec:
        units.append(Y[cursor:cursor + T_u])
        cursor += T_u
    return units


def fit_linear_var_granger(Y, split_spec, *, alpha=0.05, threshold_policy="p_value"):
    N = Y.shape[1]
    units = _unstack_panel(Y, split_spec)
    X_rows, Y_rows = [], []
    for unit in units:
        if len(unit) < 2:
            continue
        X_rows.append(unit[:-1]); Y_rows.append(unit[1:])
    if not X_rows:
        raise ValueError("No valid pairs.")
    X = np.concatenate(X_rows, axis=0)
    Yt = np.concatenate(Y_rows, axis=0)
    n_obs = X.shape[0]
    X_design = np.concatenate([X, np.ones((n_obs, 1))], axis=1)
    A_raw = np.zeros((N, N)); F_stats = np.zeros((N, N))
    p_values = np.ones((N, N)); intercepts = np.zeros(N)
    XtX_inv = np.linalg.pinv(X_design.T @ X_design)
    for i in range(N):
        y_i = Yt[:, i]
        beta = XtX_inv @ X_design.T @ y_i
        A_raw[i, :] = beta[:N]; intercepts[i] = beta[N]
        resid = y_i - X_design @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * stats.t.sf(np.abs(t_stats), dof)
        F_stats[i, :] = np.abs(t_stats[:N]); p_values[i, :] = p_vals[:N]
    if threshold_policy == "p_value":
        A_thresh = A_raw.copy()
        A_thresh[p_values >= alpha] = 0.0
    elif threshold_policy == "none":
        A_thresh = A_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(A_thresh, 0.0)
    # Unsigned-magnitude convention (consistent with other methods)
    A_thresh = np.abs(A_thresh)
    return {"A_raw": A_raw, "A_thresh": A_thresh, "F_stats": F_stats,
            "p_values": p_values, "intercepts": intercepts,
            "n_obs": n_obs, "dof": n_obs - (N + 1)}


def run_linear_var_granger(Y_train, split_spec, *, variable_names=None,
                            alpha=0.05, seed=0):
    t0 = time.time()
    out = fit_linear_var_granger(Y_train, split_spec, alpha=alpha)
    elapsed = time.time() - t0
    N = Y_train.shape[1]
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    return {
        "method": "linear_var_granger",
        "W_hat": out["A_thresh"], "W_hat_raw": out["A_raw"],
        "scores": out["F_stats"], "p_values": out["p_values"],
        "variable_names": list(variable_names),
        "method_config": {"alpha": alpha, "threshold_policy": "p_value",
                          "include_intercept": True,
                          "n_obs": int(out["n_obs"]), "dof": int(out["dof"])},
        "training_time_sec": elapsed, "seed": seed,
    }



# ### 6.2 PCMCI


def _build_unit_boundary_mask(split_spec, max_lag):
    T_total = sum(split_spec)
    mask = np.ones(T_total, dtype=bool)
    cursor = 0
    for T_u in split_spec:
        mask[cursor:cursor + max_lag] = False
        cursor += T_u
    return mask


def fit_pcmci(Y, split_spec, *, tau_max=1, pc_alpha=0.2, alpha_level=0.01,
               verbosity=0):
    from tigramite.pcmci import PCMCI
    from tigramite.independence_tests.parcorr import ParCorr
    from tigramite import data_processing as pp

    N = Y.shape[1]
    boundary_mask = _build_unit_boundary_mask(split_spec, max_lag=tau_max)
    mask_2d = np.zeros_like(Y, dtype=int)
    mask_2d[~boundary_mask, :] = 1
    dataframe = pp.DataFrame(Y, mask=mask_2d)
    parcorr = ParCorr(mask_type="y")
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr,
                  verbosity=verbosity)
    results = pcmci.run_pcmci(tau_max=tau_max, pc_alpha=pc_alpha,
                              alpha_level=alpha_level)
    val = results["val_matrix"]; p_mat = results["p_matrix"]
    if tau_max < 1:
        raise ValueError("tau_max must be >= 1")
    val_abs = np.abs(val[:, :, 1:tau_max + 1])
    W_raw_transposed = val_abs.max(axis=-1)
    W_raw = W_raw_transposed.T
    p_min_transposed = p_mat[:, :, 1:tau_max + 1].min(axis=-1)
    W_p = p_min_transposed.T
    return {"W_raw": W_raw, "W_p": W_p, "val_matrix": val, "p_matrix": p_mat}


def run_pcmci(Y_train, split_spec, *, variable_names=None, tau_max=1,
               pc_alpha=0.2, alpha_level=0.01,
               threshold_policy="p_value", threshold_value=0.01, seed=0):
    t0 = time.time()
    out = fit_pcmci(Y_train, split_spec, tau_max=tau_max,
                     pc_alpha=pc_alpha, alpha_level=alpha_level)
    elapsed = time.time() - t0
    N = Y_train.shape[1]
    W_raw, W_p = out["W_raw"], out["W_p"]
    scores = W_raw.copy()
    if threshold_policy == "p_value":
        W_hat = np.where(W_p < threshold_value, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    return {
        "method": "pcmci", "W_hat": W_hat, "W_hat_raw": W_raw,
        "scores": scores, "p_values": W_p,
        "variable_names": list(variable_names),
        "method_config": {"tau_max": tau_max, "pc_alpha": pc_alpha,
                          "alpha_level": alpha_level,
                          "threshold_policy": threshold_policy,
                          "threshold_value": threshold_value,
                          "cond_ind_test": "ParCorr"},
        "training_time_sec": elapsed, "seed": seed,
    }



# ### 6.3 cMLP


def _build_lagged_data(Y, split_spec, lag):
    N = Y.shape[1]
    X_rows, y_rows = [], []
    cursor = 0
    for T_u in split_spec:
        unit = Y[cursor:cursor + T_u]; cursor += T_u
        if T_u <= lag:
            continue
        for t in range(lag, T_u):
            X_rows.append(unit[t - lag:t])
            y_rows.append(unit[t])
    if not X_rows:
        raise ValueError(f"No samples with lag={lag}.")
    return np.stack(X_rows, axis=0), np.stack(y_rows, axis=0)


def fit_cmlp(Y, split_spec, *, lag=8, hidden_dim=10, n_epochs=500, lr=1e-3,
              lambda_group=0.05, penalty="hierarchical", device="cpu",
              seed=0, verbose=False):
    import torch
    import torch.nn as nn
    N = Y.shape[1]
    X, yt = _build_lagged_data(Y, split_spec, lag)
    n_samples = X.shape[0]
    X_flat = X.reshape(n_samples, lag * N)
    X_t = torch.tensor(X_flat, dtype=torch.float32, device=device)
    Y_t = torch.tensor(yt, dtype=torch.float32, device=device)

    class ComponentMLP(nn.Module):
        def __init__(self, in_dim, hidden_dim):
            super().__init__()
            self.linear1 = nn.Linear(in_dim, hidden_dim)
            self.linear2 = nn.Linear(hidden_dim, 1)
        def forward(self, x):
            return self.linear2(torch.relu(self.linear1(x))).squeeze(-1)

    def group_indices(j):
        return [lag_idx * N + j for lag_idx in range(lag)]

    torch.manual_seed(seed); np.random.seed(seed)
    W_raw = np.zeros((N, N))
    losses_per_target = []
    for i in range(N):
        model = ComponentMLP(lag * N, hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        last_loss = None
        for epoch in range(n_epochs):
            optimizer.zero_grad()
            pred = model(X_t)
            mse = ((pred - Y_t[:, i]) ** 2).mean()
            W_input = model.linear1.weight
            if penalty == "group_lasso":
                pen = torch.stack([torch.norm(W_input[:, group_indices(j)], p="fro")
                                    for j in range(N)]).sum()
            elif penalty == "hierarchical":
                hier_norms = []
                for j in range(N):
                    idx = group_indices(j)
                    for k in range(lag):
                        sub_idx = idx[k:]
                        if len(sub_idx) == 0:
                            continue
                        hier_norms.append(torch.norm(W_input[:, sub_idx], p="fro"))
                pen = torch.stack(hier_norms).sum()
            else:
                raise ValueError(f"Unknown penalty: {penalty}")
            loss = mse + lambda_group * pen
            loss.backward(); optimizer.step()
            last_loss = float(loss.item())
        losses_per_target.append(last_loss)
        with torch.no_grad():
            W_in = model.linear1.weight.cpu().numpy()
            for j in range(N):
                idx = group_indices(j)
                W_raw[i, j] = float(np.linalg.norm(W_in[:, idx], ord="fro"))
    return {"W_raw": W_raw, "n_samples": n_samples,
            "losses_per_target": losses_per_target}


def run_cmlp(Y_train, split_spec, *, variable_names=None, lag=8,
              hidden_dim=10, n_epochs=500, lr=1e-3, lambda_group=0.1,
              penalty="hierarchical", threshold_policy="relative",
              threshold_value=0.01, device="cpu", seed=0):
    t0 = time.time()
    out = fit_cmlp(Y_train, split_spec, lag=lag, hidden_dim=hidden_dim,
                    n_epochs=n_epochs, lr=lr, lambda_group=lambda_group,
                    penalty=penalty, device=device, seed=seed)
    elapsed = time.time() - t0
    N = Y_train.shape[1]
    W_raw = out["W_raw"]; scores = W_raw.copy()
    if threshold_policy == "relative":
        W_offdiag = W_raw.copy(); np.fill_diagonal(W_offdiag, 0.0)
        cutoff = threshold_value * np.max(W_offdiag)
        W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    return {
        "method": "cmlp", "W_hat": W_hat, "W_hat_raw": W_raw, "scores": scores,
        "variable_names": list(variable_names),
        "method_config": {"lag": lag, "hidden_dim": hidden_dim,
                          "n_epochs": n_epochs, "lr": lr,
                          "lambda_group": lambda_group, "penalty": penalty,
                          "threshold_policy": threshold_policy,
                          "threshold_value": threshold_value,
                          "n_samples": int(out["n_samples"])},
        "training_time_sec": elapsed, "seed": seed,
    }



# ### 6.4 NAVAR


def _to_split_arg(split_spec):
    if len(set(split_spec)) == 1:
        return int(split_spec[0])
    return list(split_spec)


def _apply_dm_ablation(model, data, variable_names, split_spec, scores_W,
                        *, dm_alpha, maxlags, val_proportion, verbose):
    from edge_ablation import edge_ablation_dm_all_pairs
    pvals_df, dm_df, diff_df = edge_ablation_dm_all_pairs(
        model=model, data=data, variable_names=list(variable_names),
        maxlags=maxlags, split_timeseries=split_spec,
        val_proportion=val_proportion, normalize_for_eval=True,
        batch_size=512, include_diagonal=False, verbose=verbose,
    )
    pvals_W = pvals_df.to_numpy().T
    keep_mask = (pvals_W < dm_alpha) & np.isfinite(pvals_W)
    W_hat = np.where(keep_mask, scores_W, 0.0)
    if verbose:
        n_kept = int(keep_mask.sum()); n_total = int((pvals_W < 2.0).sum())
        print(f"  [DM] {n_kept}/{n_total} edges significant at alpha={dm_alpha}")
    return W_hat, pvals_W


def run_navar(Y_train, split_spec, *, variable_names=None, maxlags=8,
               hidden_nodes=32, hidden_layers=1, dropout=0.0, n_epochs=500,
               lr=3e-4, batch_size=300, lambda1=0.15, weight_decay=0.0,
               threshold_policy="dm_significance", threshold_value=0.1,
               use_dm_ablation=True, dm_alpha=0.05, dm_val_proportion=0.10,
               seed=0, verbose=False):
    import torch
    try:
        from train_NAVAR import train_NAVAR
    except ImportError as e:
        src_dir = globals().get("SRC_DIR", None)
        if src_dir is not None and os.path.isdir(src_dir) and src_dir not in sys.path:
            sys.path.insert(0, src_dir)
        from train_NAVAR import train_NAVAR

    N = Y_train.shape[1]
    if variable_names is None:
        variable_names = [f"var_{i}" for i in range(N)]
    torch.manual_seed(seed); np.random.seed(seed)

    t0 = time.time()
    out = train_NAVAR(
        Y_train, maxlags=maxlags, hidden_nodes=hidden_nodes,
        hidden_layers=hidden_layers, dropout=dropout, epochs=n_epochs,
        learning_rate=lr, batch_size=batch_size, lambda1=lambda1,
        weight_decay=weight_decay, split_timeseries=_to_split_arg(split_spec),
        val_proportion=0.0, normalize=False,
        check_every=10**9 if not verbose else max(1, n_epochs // 10),
    )
    if isinstance(out, tuple):
        scores_src_tgt = out[0]
        navar_model = out[3] if len(out) >= 4 else None
    else:
        scores_src_tgt = out; navar_model = None
    elapsed = time.time() - t0

    scores = np.asarray(scores_src_tgt).T.copy()
    scores = np.abs(scores)
    W_raw = scores.copy(); dm_pvals = None

    if threshold_policy == "relative":
        W_offdiag = W_raw.copy(); np.fill_diagonal(W_offdiag, 0.0)
        cutoff = threshold_value * np.max(W_offdiag)
        W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "absolute":
        W_hat = np.where(W_raw >= threshold_value, W_raw, 0.0)
    elif threshold_policy == "top_k":
        k = int(threshold_value)
        flat = W_raw.copy(); np.fill_diagonal(flat, -np.inf)
        if k >= flat.size:
            W_hat = W_raw.copy()
        else:
            cutoff = np.partition(flat.flatten(), -k)[-k]
            W_hat = np.where(W_raw >= cutoff, W_raw, 0.0)
    elif threshold_policy == "dm_significance":
        if not use_dm_ablation:
            raise ValueError("dm_significance requires use_dm_ablation=True")
        if navar_model is None:
            raise RuntimeError("DM ablation requires train_NAVAR to return model.")
        W_hat, dm_pvals = _apply_dm_ablation(
            model=navar_model, data=Y_train,
            variable_names=variable_names, split_spec=split_spec,
            scores_W=W_raw, dm_alpha=dm_alpha, maxlags=maxlags,
            val_proportion=dm_val_proportion, verbose=verbose,
        )
    elif threshold_policy == "none":
        W_hat = W_raw.copy()
    else:
        raise ValueError(f"Unknown threshold_policy: {threshold_policy}")
    np.fill_diagonal(W_hat, 0.0)

    return {
        "method": "navar", "W_hat": W_hat, "W_hat_raw": W_raw,
        "scores": scores, "dm_pvals": dm_pvals,
        "variable_names": list(variable_names),
        "method_config": {"maxlags": maxlags, "hidden_nodes": hidden_nodes,
                          "hidden_layers": hidden_layers, "dropout": dropout,
                          "n_epochs": n_epochs, "lr": lr,
                          "batch_size": batch_size, "lambda1": lambda1,
                          "weight_decay": weight_decay,
                          "threshold_policy": threshold_policy,
                          "threshold_value": threshold_value,
                          "use_dm_ablation": use_dm_ablation,
                          "dm_alpha": dm_alpha if use_dm_ablation else None},
        "training_time_sec": elapsed, "seed": seed,
    }



# ### 6.5 DYNOTEARS (vendored from causalnex; src/dynotears_panel.py)
#
# Imports the standalone module that vendors causalnex's DYNOTEARS optimizer
# (Apache 2.0). The default lambda_a=0.001 is the V-Dem density-matched
# tuning, selected from a prespecified sweep over {0.05, 0.01, 0.005, 0.001,
# 0.0005} to place DYNOTEARS in the non-degenerate density range of the
# comparator methods. See appendix for the sweep table and lambda_a-sensitivity
# robustness check.


def run_dynotears_wrapper(Y_train, split_spec, *, variable_names=None, seed=0,
                           p=1, lambda_w=1.0, lambda_a=0.001,
                           max_iter=100, h_tol=1e-8, w_threshold=0.001):
    """Thin pass-through to dynotears_panel.run_dynotears."""
    try:
        from dynotears_panel import run_dynotears
    except ImportError as e:
        src_dir = globals().get("SRC_DIR", None)
        if src_dir is not None and os.path.isdir(src_dir) and src_dir not in sys.path:
            sys.path.insert(0, src_dir)
        from dynotears_panel import run_dynotears

    return run_dynotears(
        Y_train, split_spec,
        variable_names=variable_names,
        p=p, lambda_w=lambda_w, lambda_a=lambda_a,
        max_iter=max_iter, h_tol=h_tol, w_threshold=w_threshold,
        seed=seed,
    )



# 7. Random-baseline generators
#
# Three families per method, each preserving different structural properties.
# Note (Fix 5): under our convention W[i, j] = source-j -> target-i, row i
# of W contains incoming-edge weights TO target i. So permuting row i
# preserves the per-target marginal; it does NOT preserve "outgoing influence
# per source" (that would require permuting columns).


def _off_diag_indices(N_):
    ii, jj = np.meshgrid(np.arange(N_), np.arange(N_), indexing="ij")
    mask = (ii != jj)
    return ii[mask], jj[mask]


def _weight_bag(W_in):
    Ww = W_in.copy(); np.fill_diagonal(Ww, 0.0)
    vals = np.abs(Ww[Ww != 0.0])
    if vals.size == 0:
        raise ValueError("W has no off-diagonal nonzero entries.")
    return vals


def random_iid_density_matched(W_in, n_nnz, rng):
    """
    Place n_nnz nonzeros uniformly off-diagonal; weights drawn from |W_in| bag.
    Sign convention: matches the input — if W_in is unsigned (no negatives),
    baseline is unsigned; if W_in has signs, baseline samples sign uniformly.
    """
    N_ = W_in.shape[0]
    ii_all, jj_all = _off_diag_indices(N_)
    if n_nnz > ii_all.size:
        raise ValueError(f"n_nnz={n_nnz} > n_cells={ii_all.size}")
    bag = _weight_bag(W_in)
    chosen = rng.choice(ii_all.size, size=n_nnz, replace=False)
    W_out = np.zeros_like(W_in)
    weights = rng.choice(bag, size=n_nnz, replace=True)
    # Sign convention: if input W has no negatives, keep baseline unsigned;
    # otherwise sample signs uniformly.
    if (W_in < 0).any():
        signs = rng.choice([-1.0, 1.0], size=n_nnz)
        W_out[ii_all[chosen], jj_all[chosen]] = weights * signs
    else:
        W_out[ii_all[chosen], jj_all[chosen]] = weights
    return W_out


def random_perm_rows(W_in, rng):
    """
    Independently permute the off-diagonal entries within each row.

    Under W[i, j] = source j -> target i convention, row i is the vector of
    incoming-edge weights TO target i. Permuting row i preserves the
    per-target incoming-weight marginal but destroys the source identity
    (which source variable each weight belonged to).
    """
    N_ = W_in.shape[0]
    W_out = np.zeros_like(W_in)
    for i in range(N_):
        row = W_in[i].copy(); row[i] = 0.0
        others = np.r_[:i, i+1:N_]
        perm = rng.permutation(others)
        W_out[i, perm] = row[others]
    return W_out


def random_perm_full(W_in, rng):
    """Shuffle all off-diagonal entries among each other."""
    N_ = W_in.shape[0]
    ii_all, jj_all = _off_diag_indices(N_)
    vals = W_in[ii_all, jj_all]
    perm = rng.permutation(vals.size)
    W_out = np.zeros_like(W_in)
    W_out[ii_all, jj_all] = vals[perm]
    return W_out


# Sanity check
_W_test = np.array([[0.0, 0.5, 0.3], [0.0, 0.0, 0.7], [0.2, 0.0, 0.0]])
_rng = np.random.default_rng(0)
print("Sanity check (3x3, all-positive W):")
_iid = random_iid_density_matched(_W_test, 4, _rng)
print(f"  iid: nnz={int((_iid != 0).sum())}, any_neg={(_iid < 0).any()}")
_pr = random_perm_rows(_W_test, _rng)
print(f"  perm_rows: nnz={int((_pr != 0).sum())}, any_neg={(_pr < 0).any()}")
_pf = random_perm_full(_W_test, _rng)
print(f"  perm_full: nnz={int((_pf != 0).sum())}, any_neg={(_pf < 0).any()}")



# 8. Score-one-W helper


def score_one_W(W_label, W_matrix, *, method_label, family_label, rep_idx,
                Y, split_spec, seed):
    row = {
        "config": CONFIG_NAME, "method": method_label, "family": family_label,
        "rep": rep_idx, "W_label": W_label,
        "nnz_W": int((np.abs(W_matrix) > 1e-12).sum()),
    }
    for protocol in ["unconstrained", "constrained"]:
        constrain = (protocol == "constrained")
        try:
            res = score_W_against_panel(
                W=W_matrix, Y=Y, split_spec=split_spec,
                H=H_HORIZON, B=B_BOOT, grid_size=GRID_SIZE,
                constrain=constrain, epsilon=EPSILON,
                bandwidth=BANDWIDTH, ridge=RIDGE,
                rng=np.random.default_rng(seed + rep_idx + 1),
            )
            row[f"CRPS_{protocol}"]    = float(res["CRPS"])
            row[f"max_abs_lam_{protocol}"]  = float(res["max_abs_lam_raw"])
            row[f"n_eval_pairs_{protocol}"] = int(res.get("n_eval_pairs", 0))
            row[f"n_units_eval_{protocol}"] = int(res.get("n_units_evaluated", 0))
            if constrain:
                row["max_abs_lam_projected"] = float(res.get("max_abs_lam_projected", np.nan))
                row["n_projected"] = int(res.get("n_projected", 0))
        except Exception as e:
            row[f"CRPS_{protocol}"]    = np.nan
            row[f"max_abs_lam_{protocol}"]  = np.nan
            row[f"err_{protocol}"]     = f"{type(e).__name__}: {e}"
    return row


print("Score helper ready.")



# 9. Main sweep — fit each method, generate baselines, score everything
#
# Saves incrementally so a Colab disconnect is recoverable. At the end,
# saves method_outputs.pkl with full per-method fit-output dicts.


import pandas as pd

METHODS = {
    "linear_var_granger": run_linear_var_granger,
    "pcmci":              run_pcmci,
    "cmlp":               run_cmlp,
    "navar":              run_navar,
    "dynotears":          run_dynotears_wrapper,
}

results_path = os.path.join(OUT_DIR, "sweep_raw.csv")
methods_pkl_path_resume = os.path.join(OUT_DIR, "method_outputs.pkl")
rows = []

if os.path.exists(results_path):
    df_prev = pd.read_csv(results_path)
    rows = df_prev.to_dict("records")
    print(f"Resuming: loaded {len(rows)} existing rows from {results_path}")
    seen_method_W = set((r["method"], r["family"], r["rep"]) for r in rows)
else:
    seen_method_W = set()

# Two caches: W_hat_by_method (just W, used for baseline generation) and
# method_outputs_full (full fit-output dicts, saved to method_outputs.pkl).
W_hat_by_method = {}
method_outputs_full = {}     # FIX: persist full per-method output for downstream notebooks

# Resume: load existing method_outputs.pkl so previously-fit methods don't
# get dropped from the saved pickle when adding a new method.
if os.path.exists(methods_pkl_path_resume):
    try:
        with open(methods_pkl_path_resume, "rb") as fh:
            prev_pkl = pickle.load(fh)
        prev_methods = prev_pkl.get('method_outputs', {})
        for m_prev, sub in prev_methods.items():
            if isinstance(sub, dict) and 'W_hat' in sub:
                method_outputs_full[m_prev] = sub
                W_hat_by_method[m_prev] = np.asarray(sub['W_hat'])
        if method_outputs_full:
            print(f"Resuming: loaded prior method_outputs for "
                  f"{list(method_outputs_full.keys())} from {methods_pkl_path_resume}")
    except Exception as e:
        print(f"WARN: could not load prior method_outputs.pkl ({e}); will refit if scored rows exist")

for method_name in METHODS_TO_RUN:
    if method_name not in METHODS:
        print(f"Unknown method '{method_name}', skipping")
        continue

    print(f"\n{'='*68}")
    print(f"  METHOD: {method_name}")
    print(f"{'='*68}")

    method_key = (method_name, "method", -1)

    # If we have prior fit cached AND the method-W was already scored, skip refit
    already_fit = (method_name in method_outputs_full
                   and method_outputs_full[method_name].get('W_hat') is not None)
    already_scored = method_key in seen_method_W

    if already_fit and already_scored:
        # Hydrate W_hat from cache; baselines for this method will still run if not yet scored
        print(f"[fit] Skipping refit for {method_name} (cached). "
              f"nnz={int((np.abs(method_outputs_full[method_name]['W_hat']) > 1e-12).sum())}")
        W_hat = method_outputs_full[method_name]['W_hat']
        W_hat_by_method[method_name] = W_hat
        fit_time = float(method_outputs_full[method_name].get('training_time_sec', 0.0))
    else:
        print(f"[fit] Running {method_name}...")
        t_fit_start = time.time()
        try:
            kwargs = dict(variable_names=var_names, seed=SEED)
            if method_name == "cmlp":
                kwargs["device"] = DEVICE
            elif method_name == "navar":
                kwargs["verbose"] = False
            out = METHODS[method_name](Y_norm, split_spec, **kwargs)
            fit_time = time.time() - t_fit_start
            W_hat = out["W_hat"]
            W_hat_by_method[method_name] = W_hat
            method_outputs_full[method_name] = out  # FIX: persist full dict
            nnz_method = int((np.abs(W_hat) > 1e-12).sum())
            print(f"[fit] Done in {fit_time:.1f}s. W_hat: nnz={nnz_method}, "
                  f"max|W|={np.abs(W_hat).max():.4f}, "
                  f"any_neg={(W_hat<0).any()}")
        except Exception as e:
            print(f"[fit] FAILED: {type(e).__name__}: {e}")
            rows.append({
                "config": CONFIG_NAME, "method": method_name,
                "family": "method", "rep": -1, "W_label": method_name,
                "nnz_W": 0,
                "CRPS_unconstrained": np.nan, "CRPS_constrained": np.nan,
                "max_abs_lam_unconstrained": np.nan, "max_abs_lam_constrained": np.nan,
                "max_abs_lam_projected": np.nan, "n_projected": 0,
                "err_fit": f"{type(e).__name__}: {e}",
                "training_time_sec": time.time() - t_fit_start,
            })
            pd.DataFrame(rows).to_csv(results_path, index=False)
            continue

    if method_key not in seen_method_W:
        print(f"[score] Scoring {method_name} W_hat under both protocols...")
        t = time.time()
        row = score_one_W(
            W_label=method_name, W_matrix=W_hat,
            method_label=method_name, family_label="method", rep_idx=-1,
            Y=Y_norm, split_spec=split_spec, seed=SEED,
        )
        row["training_time_sec"] = fit_time
        rows.append(row)
        print(f"[score] CRPS_unc={row.get('CRPS_unconstrained', float('nan')):.4f}, "
              f"CRPS_con={row.get('CRPS_constrained', float('nan')):.4f}, "
              f"n_eval={row.get('n_eval_pairs_unconstrained', 'NA')}, "
              f"max|lam|_unc={row.get('max_abs_lam_unconstrained', float('nan')):.3f} "
              f"({time.time()-t:.1f}s)")
        pd.DataFrame(rows).to_csv(results_path, index=False)
    else:
        print(f"[score] Skipping {method_name} W_hat scoring (already done)")

    # FIX 4: deterministic per-method baseline RNG (not hash())
    rng = np.random.default_rng(SEED + 100 + METHOD_RNG_OFFSETS.get(method_name, 0))
    nnz_for_iid = int((np.abs(W_hat) > 1e-12).sum())

    for family_name, gen_fn in [
        ("iid_density_matched", lambda: random_iid_density_matched(W_hat, nnz_for_iid, rng)),
        ("perm_rows",            lambda: random_perm_rows(W_hat, rng)),
        ("perm_full",            lambda: random_perm_full(W_hat, rng)),
    ]:
        print(f"\n  [baselines] family={family_name}")
        for rep in range(N_REPS_PER_FAMILY):
            key = (method_name, family_name, rep)
            if key in seen_method_W:
                continue
            W_base = gen_fn()
            t = time.time()
            row = score_one_W(
                W_label=f"{method_name}__{family_name}__r{rep:02d}",
                W_matrix=W_base, method_label=method_name,
                family_label=family_name, rep_idx=rep,
                Y=Y_norm, split_spec=split_spec, seed=SEED,
            )
            rows.append(row)
            elapsed = time.time() - t
            print(f"    rep {rep:2d}: CRPS_unc={row.get('CRPS_unconstrained', float('nan')):.4f}, "
                  f"CRPS_con={row.get('CRPS_constrained', float('nan')):.4f}  "
                  f"({elapsed:.1f}s)")
            pd.DataFrame(rows).to_csv(results_path, index=False)

print(f"\n{'='*68}")
print(f"Done. {len(rows)} total rows saved to {results_path}")
df = pd.DataFrame(rows)
print(f"Rows by family:")
print(df.groupby(["method", "family"]).size())

# ===== Save full per-method fit outputs to method_outputs.pkl =====
# Permanent fix: this is what multi_criteria_evaluation.py reads downstream.
output_dict = {
    'config_name':    CONFIG_NAME,
    'method_outputs': method_outputs_full,
    'panel_summary': {
        'N':                int(N),
        'U':                int(U),
        'T_per_unit_mean':  float(np.mean(split_spec)),
        'variable_names':   list(var_names),
    },
}
methods_pkl_path = os.path.join(OUT_DIR, "method_outputs.pkl")
with open(methods_pkl_path, "wb") as f:
    pickle.dump(output_dict, f)
print(f"\nSaved method outputs -> {methods_pkl_path}")
print(f"  Methods saved: {list(method_outputs_full.keys())}")
for m, sub in method_outputs_full.items():
    W = sub.get('W_hat')
    cfg = sub.get('method_config', {})
    if W is not None:
        print(f"    {m}: W shape={W.shape}, nnz={int((np.abs(W) > 1e-10).sum())}, "
              f"threshold_policy={cfg.get('threshold_policy', 'unknown')}")



# 10. Verdicts: empirical p-values (Fix 3: plus-one correction)


df = pd.read_csv(results_path)

verdict_rows = []
for method_name in df["method"].unique():
    df_m = df[df["method"] == method_name]
    df_method_W = df_m[df_m["family"] == "method"]
    if df_method_W.empty:
        continue
    for protocol in ["unconstrained", "constrained"]:
        crps_method = df_method_W[f"CRPS_{protocol}"].iloc[0]
        if not np.isfinite(crps_method):
            continue
        for family_name in ["iid_density_matched", "perm_rows", "perm_full"]:
            df_b = df_m[df_m["family"] == family_name]
            crps_base = df_b[f"CRPS_{protocol}"].dropna().to_numpy()
            if crps_base.size == 0:
                continue
            # FIX 3: plus-one correction. p = (1 + #beats)/(1 + B)
            n_beats = int((crps_base <= crps_method).sum())
            p_emp = (1 + n_beats) / (1 + crps_base.size)
            verdict_rows.append({
                "method": method_name, "protocol": protocol,
                "baseline_family": family_name,
                "n_baselines": int(crps_base.size),
                "n_baselines_beat_method": n_beats,
                "CRPS_method": float(crps_method),
                "CRPS_baseline_mean": float(crps_base.mean()),
                "CRPS_baseline_min":  float(crps_base.min()),
                "CRPS_baseline_max":  float(crps_base.max()),
                "p_emp": float(p_emp),
                "method_beats": (p_emp <= 0.10),
            })

df_verdict = pd.DataFrame(verdict_rows)
df_verdict = df_verdict.sort_values(["method", "protocol", "baseline_family"])
verdict_path = os.path.join(OUT_DIR, "verdict.csv")
df_verdict.to_csv(verdict_path, index=False)

print("Verdict table (Fix 3: p_emp = (1 + #beats)/(1 + B)):")
print(df_verdict.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nSaved -> {verdict_path}")



verdict_text_path = os.path.join(OUT_DIR, "verdict.txt")
with open(verdict_text_path, "w") as fh:
    fh.write(f"=== §5.3 Random-Baseline Verdict — {CONFIG_NAME} ===\n\n")
    fh.write("Fixes applied: panel-safe CRPS, per-unit eval, plus-one p,\n")
    fh.write("              deterministic per-method seed, sign-aware iid baseline.\n\n")
    for protocol in ["unconstrained", "constrained"]:
        fh.write(f"\n--- {protocol.upper()} PROTOCOL ---\n")
        df_p = df_verdict[df_verdict["protocol"] == protocol]
        n_pairs = len(df_p)
        n_beats = int(df_p["method_beats"].sum())
        fh.write(f"\n  {n_beats} of {n_pairs} (method, baseline-family) pairs: "
                 f"method beats baselines (p_emp <= 0.10)\n\n")
        for _, r in df_p.iterrows():
            verdict_str = "BEATS" if r["method_beats"] else "FAILS"
            fh.write(f"  {r['method']:22s} vs {r['baseline_family']:22s}: "
                     f"p_emp={r['p_emp']:.3f}  [{verdict_str}]  "
                     f"(method CRPS={r['CRPS_method']:.4f}, "
                     f"baseline mean={r['CRPS_baseline_mean']:.4f})\n")
    fh.write(f"\n\n--- PER-METHOD AGGREGATE (across both protocols + 3 families = 6 pairs) ---\n\n")
    for method_name in df_verdict["method"].unique():
        df_m = df_verdict[df_verdict["method"] == method_name]
        n = len(df_m); n_beats = int(df_m["method_beats"].sum())
        if n_beats == n:
            label = "STRONGLY BEATS baselines"
        elif n_beats >= n * 2 / 3:
            label = "MOSTLY BEATS baselines"
        elif n_beats >= n / 3:
            label = "MIXED — partial discrimination"
        else:
            label = "FAILS — deployability != validity"
        fh.write(f"  {method_name:22s}: {n_beats}/{n} pairs beat -> {label}\n")

with open(verdict_text_path, "r") as fh:
    print(fh.read())
print(f"Saved -> {verdict_text_path}")



# 11. Figure


import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=False)
method_list = METHODS_TO_RUN
family_order = ["method", "iid_density_matched", "perm_rows", "perm_full"]
family_colors = {
    "method": "#1f77b4", "iid_density_matched": "#d62728",
    "perm_rows": "#ff7f0e", "perm_full": "#9467bd",
}
family_labels = {
    "method": "Method W_hat", "iid_density_matched": "iid density-matched",
    "perm_rows": "perm_rows", "perm_full": "perm_full",
}

for ax, protocol in zip(axes, ["unconstrained", "constrained"]):
    crps_col = f"CRPS_{protocol}"
    n_methods = len(method_list)
    bar_width = 0.18
    x = np.arange(n_methods)
    for k, family in enumerate(family_order):
        means, sds = [], []
        for method in method_list:
            sub = df[(df["method"] == method) & (df["family"] == family)]
            crps_vals = sub[crps_col].dropna().to_numpy()
            if crps_vals.size == 0:
                means.append(np.nan); sds.append(0.0)
            else:
                means.append(float(crps_vals.mean()))
                sds.append(float(crps_vals.std(ddof=1)) if crps_vals.size > 1 else 0.0)
        offset = (k - (len(family_order) - 1) / 2) * bar_width
        ax.bar(x + offset, means, width=bar_width, yerr=sds, capsize=3,
               alpha=0.85, color=family_colors[family],
               label=family_labels[family])
    ax.set_xticks(x)
    ax.set_xticklabels(method_list, rotation=20, ha="right")
    ax.set_ylabel(f"CRPS ({protocol})")
    ax.set_title(f"{protocol.title()} protocol")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    if protocol == "unconstrained":
        ax.legend(loc="best", fontsize=9)

plt.suptitle(f"§5.3 Method W_hat vs random baselines  ({CONFIG_NAME}) — corrected\n"
             f"Lower CRPS = better; method bars should be lower than baseline bars if "
             f"the method captures structure beyond density",
             fontsize=11, y=1.02)
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "fig_baseline_crps.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {fig_path}")

### Cell §4.2 — DYNOTEARS standalone test on V-Dem (vendored implementation): superseded

**Paper section:** §5 (infrastructure).

Verification of the vendored DYNOTEARS implementation. Vendors `_learn_dynamic_structure` and `_reshape_wa` from causalnex (Apache 2.0; https://github.com/mckinsey/causalnex/blob/develop/causalnex/structure/dynotears.py) plus a panel-aware data-prep helper. We do not vendor `StructureModel`, `DynamicDataTransformer`, or pandas wrappers; the function returns numpy arrays directly. Vendored to bypass causalnex's Python <3.11 cap (Colab is 3.11+).

**Reference:** Pamfil et al. (2020), "DYNOTEARS: Structure Learning from Time-Series Data", AISTATS, https://arxiv.org/abs/2002.00498

**Source:** https://github.com/mckinsey/causalnex/blob/develop/causalnex/structure/dynotears.py

**License:** Apache 2.0 (file-level header preserved below)

**Inputs:** V-Dem panel `data/my_data_75_years.csv`.

**Outputs:** DYNOTEARS test pickle `dynotears_vdem_test_v2.pkl`.

**Runtime:** ~2 minutes on Colab Pro G4 GPU.

**Note: This cell is superseded.** The canonical 5-method sweep is now §4.1. This cell remains for historical and infrastructure-verification purposes (it documents the bring-up path used during paper development). The numerical outputs here match those produced by §4.1.

In [ ]:
# §0. Setup


import os, sys, pickle, time, warnings
from typing import List, Tuple

import numpy as np
import pandas as pd
import scipy.linalg as slin
import scipy.optimize as sopt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'dynotears_test_outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")



# §1. Vendored DYNOTEARS optimizer (Apache 2.0)
#
# Source file header from causalnex's dynotears.py:
#
#   Copyright 2019-2020 QuantumBlack Visual Analytics Limited
#   Licensed under the Apache License, Version 2.0
#
# We vendor `_reshape_wa` and `_learn_dynamic_structure` verbatim from
# https://github.com/mckinsey/causalnex/blob/develop/causalnex/structure/dynotears.py
# to support Python 3.11 (causalnex itself requires Python <3.11).


def _reshape_wa(
    wa_vec: np.ndarray, d_vars: int, p_orders: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Helper: transform adjacency vector to matrix form (vendored verbatim)."""
    w_tilde = wa_vec.reshape([2 * (p_orders + 1) * d_vars, d_vars])
    w_plus = w_tilde[:d_vars, :]
    w_minus = w_tilde[d_vars : 2 * d_vars, :]
    w_mat = w_plus - w_minus
    a_plus = (
        w_tilde[2 * d_vars :]
        .reshape(2 * p_orders, d_vars**2)[::2]
        .reshape(d_vars * p_orders, d_vars)
    )
    a_minus = (
        w_tilde[2 * d_vars :]
        .reshape(2 * p_orders, d_vars**2)[1::2]
        .reshape(d_vars * p_orders, d_vars)
    )
    a_mat = a_plus - a_minus
    return w_mat, a_mat


def _learn_dynamic_structure(
    X: np.ndarray,
    Xlags: np.ndarray,
    bnds: List[Tuple[float, float]],
    lambda_w: float = 0.1,
    lambda_a: float = 0.1,
    max_iter: int = 100,
    h_tol: float = 1e-8,
) -> Tuple[np.ndarray, np.ndarray]:
    """DYNOTEARS optimizer (vendored from causalnex). Returns (W, A)."""
    if X.size == 0:
        raise ValueError("Input data X is empty, cannot learn any structure")
    if Xlags.size == 0:
        raise ValueError("Input data Xlags is empty, cannot learn any structure")
    if X.shape[0] != Xlags.shape[0]:
        raise ValueError("Input data X and Xlags must have the same number of rows")
    if Xlags.shape[1] % X.shape[1] != 0:
        raise ValueError(
            "Number of columns of Xlags must be a multiple of number of columns of X"
        )

    n, d_vars = X.shape
    p_orders = Xlags.shape[1] // d_vars

    def _h(wa_vec: np.ndarray) -> float:
        _w_mat, _ = _reshape_wa(wa_vec, d_vars, p_orders)
        return np.trace(slin.expm(_w_mat * _w_mat)) - d_vars

    def _func(wa_vec: np.ndarray) -> float:
        _w_mat, _a_mat = _reshape_wa(wa_vec, d_vars, p_orders)
        loss = (
            0.5
            / n
            * np.square(
                np.linalg.norm(
                    X.dot(np.eye(d_vars, d_vars) - _w_mat) - Xlags.dot(_a_mat), "fro"
                )
            )
        )
        _h_value = _h(wa_vec)
        l1_penalty = lambda_w * (wa_vec[: 2 * d_vars**2].sum()) + lambda_a * (
            wa_vec[2 * d_vars**2 :].sum()
        )
        return loss + 0.5 * rho * _h_value * _h_value + alpha * _h_value + l1_penalty

    def _grad(wa_vec: np.ndarray) -> np.ndarray:
        _w_mat, _a_mat = _reshape_wa(wa_vec, d_vars, p_orders)
        e_mat = slin.expm(_w_mat * _w_mat)
        loss_grad_w = (
            -1.0
            / n
            * (X.T.dot(X.dot(np.eye(d_vars, d_vars) - _w_mat) - Xlags.dot(_a_mat)))
        )
        obj_grad_w = (
            loss_grad_w
            + (rho * (np.trace(e_mat) - d_vars) + alpha) * e_mat.T * _w_mat * 2
        )
        obj_grad_a = (
            -1.0
            / n
            * (Xlags.T.dot(X.dot(np.eye(d_vars, d_vars) - _w_mat) - Xlags.dot(_a_mat)))
        )
        grad_vec_w = np.append(
            obj_grad_w, -obj_grad_w, axis=0
        ).flatten() + lambda_w * np.ones(2 * d_vars**2)
        grad_vec_a = obj_grad_a.reshape(p_orders, d_vars**2)
        grad_vec_a = np.hstack(
            (grad_vec_a, -grad_vec_a)
        ).flatten() + lambda_a * np.ones(2 * p_orders * d_vars**2)
        return np.append(grad_vec_w, grad_vec_a, axis=0)

    # Initialise matrix, weights and constraints
    wa_est = np.zeros(2 * (p_orders + 1) * d_vars**2)
    wa_new = np.zeros(2 * (p_orders + 1) * d_vars**2)
    rho, alpha, h_value, h_new = 1.0, 0.0, np.inf, np.inf

    for n_iter in range(max_iter):
        while (rho < 1e20) and (h_new > 0.25 * h_value or h_new == np.inf):
            wa_new = sopt.minimize(
                _func, wa_est, method="L-BFGS-B", jac=_grad, bounds=bnds
            ).x
            h_new = _h(wa_new)
            if h_new > 0.25 * h_value:
                rho *= 10

        wa_est = wa_new
        h_value = h_new
        alpha += rho * h_value
        if h_value <= h_tol:
            break
        if h_value > h_tol and n_iter == max_iter - 1:
            warnings.warn("Failed to converge. Consider increasing max_iter.")
    return _reshape_wa(wa_est, d_vars, p_orders)


def _make_bounds(d_vars: int, p_orders: int) -> List[Tuple[float, float]]:
    """Box constraints: zero diagonal of W, non-negative w_plus/w_minus/a_plus/a_minus."""
    bnds_w = 2 * [
        (0, 0) if i == j else (0, None)
        for i in range(d_vars)
        for j in range(d_vars)
    ]
    bnds_a = []
    for k in range(1, p_orders + 1):
        bnds_a.extend(
            2 * [(0, None) for i in range(d_vars) for j in range(d_vars)]
        )
    return bnds_w + bnds_a



# §2. Panel-aware data preparation
#
# Build (X, Xlags) from a panel (Y, split_spec) without crossing unit boundaries.
# For p=1: X = unit[1:], Xlags = unit[:-1], stacked across units.


def build_X_Xlags_panel(Y, split_spec, p):
    """Build DYNOTEARS-format (X, Xlags) from a panel.

    Args:
      Y           : (T_total, d) stacked panel
      split_spec  : list of per-unit lengths
      p           : number of past lags

    Returns:
      X     : (n, d) "current" timepoints
      Xlags : (n, d*p) lag-stacked, [shift(X, 1) | shift(X, 2) | ... | shift(X, p)]

    No (t-1, t) pair crosses a unit boundary.
    """
    Y = np.asarray(Y, dtype=float)
    T_total, d = Y.shape
    if sum(split_spec) != T_total:
        raise ValueError(
            f"split_spec sum ({sum(split_spec)}) != T_total ({T_total})"
        )

    X_rows, Xlags_rows = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u <= p:
            continue
        X_rows.append(unit[p:])
        # For each lag k=1..p, take unit[p-k : n_u-k]
        lag_stack = np.hstack(
            [unit[p - k : n_u - k] for k in range(1, p + 1)]
        )
        Xlags_rows.append(lag_stack)
    if not X_rows:
        raise ValueError("No usable lag pairs in panel")
    X = np.vstack(X_rows)
    Xlags = np.vstack(Xlags_rows)
    return X, Xlags



# §3. Wrapper: fit DYNOTEARS on a panel, return W in our convention
#
# Our convention: W_hat[target, source] = magnitude of (source@t-1 → target@t).
# DYNOTEARS returns:
#   W_intra : (d, d) contemporaneous DAG (we discard — we model only lag-1)
#   A_inter : (d*p, d) stacked lag matrices; A_inter[i, j] = i@(t-1) → j@t for p=1
#
# For p=1, A_inter is (d, d) with source-on-rows. Our convention is
# target-on-rows. So W_hat = |A_inter.T|.


def fit_dynotears_panel(
    Y, split_spec, *,
    p=1, lambda_w=1.0, lambda_a=0.05,
    max_iter=100, h_tol=1e-8, w_threshold=0.01,
):
    """Fit DYNOTEARS on a panel, return W_hat in our (target-on-row) convention.

    Hyperparameters:
      p          = 1     : lag-1 dynamics (matches our other methods)
      lambda_w   = 1.0   : strong L1 on contemporaneous W (we discard it)
      lambda_a   = 0.05  : L1 on lagged A (DYNOTEARS-paper default)
      max_iter   = 100   : outer-loop dual ascent iterations
      h_tol      = 1e-8  : acyclicity tolerance
      w_threshold= 0.01  : post-fit edge magnitude threshold
    """
    X, Xlags = build_X_Xlags_panel(Y, split_spec, p)
    d_vars = X.shape[1]
    bnds = _make_bounds(d_vars, p)

    t0 = time.time()
    W_intra, A_inter = _learn_dynamic_structure(
        X, Xlags, bnds, lambda_w, lambda_a, max_iter, h_tol,
    )
    elapsed = time.time() - t0

    # Threshold edges by magnitude (Apache-2.0 line, kept verbatim)
    W_intra[np.abs(W_intra) < w_threshold] = 0
    A_inter[np.abs(A_inter) < w_threshold] = 0

    # For p=1: A_inter is (d, d) with source-on-rows. Our convention:
    # W_hat[target, source]. So transpose.
    if p == 1:
        A_lag1 = A_inter  # already (d, d)
    else:
        # General p: A_inter is (d*p, d); first d rows are lag-1 sub-block
        A_lag1 = A_inter[:d_vars, :]
    W_hat = np.abs(A_lag1.T)  # unsigned-magnitude convention
    np.fill_diagonal(W_hat, 0.0)

    return {
        'W_hat':       W_hat,
        'W_intra_raw': W_intra,
        'A_inter_raw': A_inter,
        'training_time_sec': elapsed,
        'n_pairs':     int(X.shape[0]),
        'd_vars':      int(d_vars),
        'p_orders':    int(p),
    }



# §4. Load V-Dem long_16var


VDEM_DATA_PATH = os.path.join(DATA_DIR, 'my_data_75_years.csv')
VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]
MIN_SERIES_LENGTH = 9
VAL_YEARS = 5

try:
    df_raw = pd.read_csv(VDEM_DATA_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(VDEM_DATA_PATH, encoding='latin1')

keep_cols = ['country_id', 'year'] + VDEM_VARIABLES
df_panel = (df_raw[keep_cols]
            .dropna(subset=VDEM_VARIABLES)
            .sort_values(['country_id', 'year'])
            .reset_index(drop=True))
lengths = df_panel.groupby('country_id')['year'].count()
df_panel = df_panel[df_panel['country_id'].isin(
    lengths[lengths >= MIN_SERIES_LENGTH].index)]

train_parts = []
for cid, g in df_panel.groupby('country_id', sort=False):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
        train_parts.append(g.iloc[:-VAL_YEARS])
    else:
        train_parts.append(g)
train_df = pd.concat(train_parts, ignore_index=True)
split_spec = train_df.groupby('country_id', sort=False)['year'].count().tolist()

Y_train = train_df[VDEM_VARIABLES].to_numpy(float)
mu = Y_train.mean(0)
sd = Y_train.std(0)
sd = np.where(sd < 1e-12, 1.0, sd)
Y_norm = (Y_train - mu) / sd
N = Y_norm.shape[1]
U = len(split_spec)
print(f"V-Dem panel: U={U}, T_total={Y_norm.shape[0]}, N={N}")
print(f"  T_per_unit: mean={np.mean(split_spec):.1f}, range=[{min(split_spec)}, {max(split_spec)}]")



# §5. Run DYNOTEARS on V-Dem


print("\nFitting DYNOTEARS on V-Dem long_16var (this may take 5-15 min)...")
out = fit_dynotears_panel(
    Y_norm, split_spec,
    p=1, lambda_w=1.0, lambda_a=0.05,
    max_iter=100, h_tol=1e-8, w_threshold=0.01,
)

W_dyno = out['W_hat']
nnz = int((np.abs(W_dyno) > 1e-10).sum())
print(f"\nDYNOTEARS fit:")
print(f"  Training time:    {out['training_time_sec']:.1f}s")
print(f"  n_pairs (panel):  {out['n_pairs']}")
print(f"  W_hat shape:      {W_dyno.shape}")
print(f"  nnz:              {nnz} / {N*N - N}  ({100 * nnz / (N*N - N):.1f}% off-diagonal)")
print(f"  max|W|:           {np.abs(W_dyno).max():.4f}")
nnz_vals = W_dyno[W_dyno != 0]
print(f"  mean|W| (nnz):    {np.abs(nnz_vals).mean() if nnz > 0 else 0:.4f}")
print(f"  any negative?     {(W_dyno < 0).any()}")



# §6. Sanity check: max|eigenvalue| of OLS-refit A
#
# Mirrors what multi_criteria_evaluation does. Other methods on V-Dem all
# produced max|eig|≈0.99 (near unit-root). DYNOTEARS should be in the same
# ballpark for it to be a sensible comparator.


def _build_panel_lag_pairs_simple(Y, split_spec):
    yp, yc = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            yp.append(unit[:-1]); yc.append(unit[1:])
        cursor += n_u
    return np.vstack(yp), np.vstack(yc)


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    M, N_ = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N_, N_))
    for i in range(N_):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_des = np.column_stack(cols)
        if X_des.shape[1] >= M:
            X_des = np.column_stack([y_prev[:, i], np.ones(M)])
        beta, _, _, _ = np.linalg.lstsq(X_des, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            if 1 + k < len(beta) - 1:
                A_full[i, j] = float(beta[1 + k])
    return A_full


y_prev, y_curr = _build_panel_lag_pairs_simple(Y_norm, split_spec)
A_full = _ols_refit_panel(W_dyno, y_prev, y_curr)
eigs = np.linalg.eigvals(A_full)
max_abs_eig = float(np.max(np.abs(eigs)))
n_above_095 = int(np.sum(np.abs(eigs) > 0.95))
print(f"\nOLS-refit A from DYNOTEARS support:")
print(f"  max|eig|       = {max_abs_eig:.4f}")
print(f"  n_eigs > 0.95  = {n_above_095} / {N}")
print(f"  stationary?    = {'YES' if max_abs_eig < 1.0 else 'NO (boundary)'}")
print(f"\n  Reference (other methods on V-Dem): 0.99-1.00, all near unit-root")



# §7. Save test pickle


result = {
    'method': 'dynotears',
    'W_hat':  W_dyno,
    'W_hat_raw': W_dyno.copy(),
    'variable_names': VDEM_VARIABLES,
    'method_config': {
        'p':           1,
        'lambda_w':    1.0,
        'lambda_a':    0.05,
        'max_iter':    100,
        'h_tol':       1e-8,
        'w_threshold': 0.01,
        'threshold_policy': 'absolute_post_fit',
    },
    'training_time_sec': out['training_time_sec'],
    'seed': 0,
}

test_pkl_path = os.path.join(OUT_DIR, 'dynotears_vdem_test.pkl')
with open(test_pkl_path, 'wb') as f:
    pickle.dump(result, f)
print(f"\nSaved -> {test_pkl_path}")



# What "reasonable" looks like
#
# - **nnz**: should be > 0 and not saturated. Range 20-150 plausible on V-Dem
#   long_16var (compare lvg=38, pcmci=114, navar=148, cmlp=228).
# - **Training time**: 5-15 min on Colab CPU is typical for V-Dem-scale.
#   If hangs >30 min, the augmented Lagrangian is struggling.
# - **max|eig| of OLS-refit A**: should be 0.99-1.00, like other methods.
# - **No negative entries**: we apply unsigned magnitude convention.
#
# If those look right, we proceed to integrate DYNOTEARS into the four
# downstream notebooks (random_baselines, lsc_diagnostic, synthetic, multi_criteria).

print("\n" + "=" * 60)
print("DONE. Send the output above for inspection.")
print("=" * 60)

### Cell §4.3 — DYNOTEARS λ_a sweep on V-Dem: superseded

**Paper section:** §5 (infrastructure).

Sweeps DYNOTEARS' L1 sparsity penalty λ_a ∈ {0.05, 0.01, 0.005, 0.001, 0.0005} to find the regime that produces non-trivial W_hat (target nnz roughly 30-200, matching the density of other methods on V-Dem). λ_a = 0.05 produces only one surviving edge; λ_a = 0.001 produces a sensible-density W. Selected λ_a values per regime are cached in `outputs/configs/dynotears_lambda_per_regime.json`.

**Inputs:** V-Dem panel.

**Outputs:** DYNOTEARS sweep pickle.

**Runtime:** ~20 minutes on Colab Pro G4 GPU.

**Note: This cell is superseded.** The canonical 5-method sweep is now §4.1. This cell remains for historical and infrastructure-verification purposes (it documents the bring-up path used during paper development); anyone running the notebook top-to-bottom can skip it without affecting the canonical results. The numerical outputs here match those produced by §4.1.

In [ ]:
import os, sys, pickle, time, warnings
from typing import List, Tuple

import numpy as np
import pandas as pd
import scipy.linalg as slin
import scipy.optimize as sopt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'dynotears_test_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# Vendored DYNOTEARS optimizer (same as before)


def _reshape_wa(wa_vec, d_vars, p_orders):
    w_tilde = wa_vec.reshape([2 * (p_orders + 1) * d_vars, d_vars])
    w_plus = w_tilde[:d_vars, :]
    w_minus = w_tilde[d_vars : 2 * d_vars, :]
    w_mat = w_plus - w_minus
    a_plus = (
        w_tilde[2 * d_vars :]
        .reshape(2 * p_orders, d_vars**2)[::2]
        .reshape(d_vars * p_orders, d_vars)
    )
    a_minus = (
        w_tilde[2 * d_vars :]
        .reshape(2 * p_orders, d_vars**2)[1::2]
        .reshape(d_vars * p_orders, d_vars)
    )
    a_mat = a_plus - a_minus
    return w_mat, a_mat


def _learn_dynamic_structure(X, Xlags, bnds, lambda_w=0.1, lambda_a=0.1,
                              max_iter=100, h_tol=1e-8):
    if X.size == 0 or Xlags.size == 0:
        raise ValueError("Empty data")
    n, d_vars = X.shape
    p_orders = Xlags.shape[1] // d_vars

    def _h(wa_vec):
        _w_mat, _ = _reshape_wa(wa_vec, d_vars, p_orders)
        return np.trace(slin.expm(_w_mat * _w_mat)) - d_vars

    def _func(wa_vec):
        _w_mat, _a_mat = _reshape_wa(wa_vec, d_vars, p_orders)
        loss = 0.5 / n * np.square(np.linalg.norm(
            X.dot(np.eye(d_vars) - _w_mat) - Xlags.dot(_a_mat), "fro"))
        _h_value = _h(wa_vec)
        l1_penalty = lambda_w * (wa_vec[:2*d_vars**2].sum()) + lambda_a * (wa_vec[2*d_vars**2:].sum())
        return loss + 0.5 * rho * _h_value * _h_value + alpha * _h_value + l1_penalty

    def _grad(wa_vec):
        _w_mat, _a_mat = _reshape_wa(wa_vec, d_vars, p_orders)
        e_mat = slin.expm(_w_mat * _w_mat)
        loss_grad_w = -1.0 / n * (X.T.dot(X.dot(np.eye(d_vars) - _w_mat) - Xlags.dot(_a_mat)))
        obj_grad_w = loss_grad_w + (rho * (np.trace(e_mat) - d_vars) + alpha) * e_mat.T * _w_mat * 2
        obj_grad_a = -1.0 / n * (Xlags.T.dot(X.dot(np.eye(d_vars) - _w_mat) - Xlags.dot(_a_mat)))
        grad_vec_w = np.append(obj_grad_w, -obj_grad_w, axis=0).flatten() + lambda_w * np.ones(2 * d_vars**2)
        grad_vec_a = obj_grad_a.reshape(p_orders, d_vars**2)
        grad_vec_a = np.hstack((grad_vec_a, -grad_vec_a)).flatten() + lambda_a * np.ones(2 * p_orders * d_vars**2)
        return np.append(grad_vec_w, grad_vec_a, axis=0)

    wa_est = np.zeros(2 * (p_orders + 1) * d_vars**2)
    wa_new = np.zeros(2 * (p_orders + 1) * d_vars**2)
    rho, alpha, h_value, h_new = 1.0, 0.0, np.inf, np.inf
    n_iter_done = 0
    for n_iter in range(max_iter):
        n_iter_done = n_iter + 1
        while (rho < 1e20) and (h_new > 0.25 * h_value or h_new == np.inf):
            wa_new = sopt.minimize(_func, wa_est, method="L-BFGS-B", jac=_grad, bounds=bnds).x
            h_new = _h(wa_new)
            if h_new > 0.25 * h_value:
                rho *= 10
        wa_est = wa_new
        h_value = h_new
        alpha += rho * h_value
        if h_value <= h_tol:
            break
    return _reshape_wa(wa_est, d_vars, p_orders), n_iter_done


def _make_bounds(d_vars, p_orders):
    bnds_w = 2 * [(0, 0) if i == j else (0, None)
                  for i in range(d_vars) for j in range(d_vars)]
    bnds_a = []
    for k in range(1, p_orders + 1):
        bnds_a.extend(2 * [(0, None) for i in range(d_vars) for j in range(d_vars)])
    return bnds_w + bnds_a


def build_X_Xlags_panel(Y, split_spec, p):
    Y = np.asarray(Y, dtype=float)
    T_total, d = Y.shape
    X_rows, Xlags_rows = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u <= p:
            continue
        X_rows.append(unit[p:])
        lag_stack = np.hstack([unit[p - k : n_u - k] for k in range(1, p + 1)])
        Xlags_rows.append(lag_stack)
    return np.vstack(X_rows), np.vstack(Xlags_rows)



# Load V-Dem (same as before)


VDEM_DATA_PATH = os.path.join(DATA_DIR, 'my_data_75_years.csv')
VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]
MIN_SERIES_LENGTH = 9
VAL_YEARS = 5

try:
    df_raw = pd.read_csv(VDEM_DATA_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(VDEM_DATA_PATH, encoding='latin1')

keep_cols = ['country_id', 'year'] + VDEM_VARIABLES
df_panel = (df_raw[keep_cols].dropna(subset=VDEM_VARIABLES)
            .sort_values(['country_id', 'year']).reset_index(drop=True))
lengths = df_panel.groupby('country_id')['year'].count()
df_panel = df_panel[df_panel['country_id'].isin(lengths[lengths >= MIN_SERIES_LENGTH].index)]
train_parts = []
for cid, g in df_panel.groupby('country_id', sort=False):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
        train_parts.append(g.iloc[:-VAL_YEARS])
    else:
        train_parts.append(g)
train_df = pd.concat(train_parts, ignore_index=True)
split_spec = train_df.groupby('country_id', sort=False)['year'].count().tolist()
Y_train = train_df[VDEM_VARIABLES].to_numpy(float)
mu, sd = Y_train.mean(0), Y_train.std(0)
sd = np.where(sd < 1e-12, 1.0, sd)
Y_norm = (Y_train - mu) / sd
N = Y_norm.shape[1]
print(f"V-Dem panel: U={len(split_spec)}, T_total={Y_norm.shape[0]}, N={N}")

X, Xlags = build_X_Xlags_panel(Y_norm, split_spec, p=1)
print(f"DYNOTEARS data: X={X.shape}, Xlags={Xlags.shape}")



# Sweep lambda_a

print("\n" + "=" * 75)
print(f"  {'lambda_a':>10s}  {'time(s)':>8s}  {'iters':>6s}  {'nnz':>5s}  "
      f"{'max|A|':>9s}  {'mean|A|':>9s}  {'frac>w_thr':>11s}")
print("=" * 75)

lambda_w_fixed = 1.0   # contemporaneous L1 — keep large to suppress
max_iter = 100
h_tol = 1e-8
w_threshold_test = 0.001  # use a much smaller threshold for the sweep

results_sweep = []

for lambda_a_test in [0.05, 0.01, 0.005, 0.001, 0.0005]:
    bnds = _make_bounds(N, 1)
    t0 = time.time()
    (W_intra, A_inter), n_iters = _learn_dynamic_structure(
        X, Xlags, bnds,
        lambda_w=lambda_w_fixed, lambda_a=lambda_a_test,
        max_iter=max_iter, h_tol=h_tol,
    )
    elapsed = time.time() - t0

    A_lag1 = A_inter  # for p=1, A_inter is (d, d)
    # Pre-threshold stats
    A_pre_thresh = A_lag1.copy()
    np.fill_diagonal(A_pre_thresh, 0.0)
    pre_max = float(np.abs(A_pre_thresh).max())
    pre_nnz_above_001 = int((np.abs(A_pre_thresh) > 0.01).sum())
    pre_nnz_above_0001 = int((np.abs(A_pre_thresh) > 0.001).sum())
    pre_nnz_pos = int((np.abs(A_pre_thresh) > 1e-10).sum())
    if pre_nnz_pos > 0:
        pre_mean = float(np.abs(A_pre_thresh[A_pre_thresh != 0]).mean())
    else:
        pre_mean = 0.0

    # Apply threshold
    A_thresh = A_lag1.copy()
    A_thresh[np.abs(A_thresh) < w_threshold_test] = 0
    np.fill_diagonal(A_thresh, 0.0)
    nnz = int((np.abs(A_thresh) > 1e-10).sum())

    print(f"  {lambda_a_test:>10.4f}  {elapsed:>8.1f}  {n_iters:>6d}  "
          f"{pre_nnz_pos:>5d}  {pre_max:>9.4f}  {pre_mean:>9.4f}  "
          f"{pre_nnz_above_001:>4d}/{pre_nnz_above_0001:<4d}")

    results_sweep.append({
        'lambda_a':       lambda_a_test,
        'A_lag1':         A_lag1.copy(),
        'pre_thresh_nnz': pre_nnz_pos,
        'pre_max':        pre_max,
        'pre_mean':       pre_mean,
        'nnz_at_001':     pre_nnz_above_001,
        'nnz_at_0001':    pre_nnz_above_0001,
        'time':           elapsed,
        'iters':          n_iters,
    })

print("=" * 75)
print(f"  Format: 'frac>w_thr' = nnz>0.01 / nnz>0.001 (where 0.01 was the original threshold)")
print(f"  Reference: lvg=38, pcmci=114, navar=148, cmlp=228 on V-Dem")



# Pick the best lambda_a and produce final W_hat

print("\n" + "=" * 70)
print("PICK BEST lambda_a")
print("=" * 70)

# Pick the lambda_a that gives nnz at threshold 0.001 in [30, 200]
target_min, target_max = 30, 200
best = None
for r in results_sweep:
    if target_min <= r['nnz_at_0001'] <= target_max:
        if best is None or abs(r['nnz_at_0001'] - 100) < abs(best['nnz_at_0001'] - 100):
            best = r

if best is None:
    # Fallback: pick the one closest to 100 nnz at threshold 0.001
    best = min(results_sweep, key=lambda r: abs(r['nnz_at_0001'] - 100))
    print(f"WARN: no lambda_a in target range [{target_min}, {target_max}]; "
          f"picking closest to 100 (lambda_a={best['lambda_a']}, nnz={best['nnz_at_0001']})")
else:
    print(f"Selected lambda_a={best['lambda_a']} (nnz={best['nnz_at_0001']} at threshold=0.001)")

# Build final W_hat with the chosen lambda_a and 0.001 threshold
A_lag1 = best['A_lag1']
A_thresh = A_lag1.copy()
A_thresh[np.abs(A_thresh) < 0.001] = 0
W_dyno = np.abs(A_thresh.T)  # transpose to our convention: target on rows
np.fill_diagonal(W_dyno, 0.0)

nnz = int((np.abs(W_dyno) > 1e-10).sum())
print(f"\nFinal W_hat:")
print(f"  shape:    {W_dyno.shape}")
print(f"  nnz:      {nnz} / {N*N - N}  ({100 * nnz / (N*N - N):.1f}% off-diagonal)")
print(f"  max|W|:   {np.abs(W_dyno).max():.4f}")
print(f"  any neg?  {(W_dyno < 0).any()}")



# Sanity check: max|eigenvalue| of OLS-refit A


def _build_panel_lag_pairs_simple(Y, split_spec):
    yp, yc = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            yp.append(unit[:-1]); yc.append(unit[1:])
        cursor += n_u
    return np.vstack(yp), np.vstack(yc)


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    M, N_ = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N_, N_))
    for i in range(N_):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_des = np.column_stack(cols)
        if X_des.shape[1] >= M:
            X_des = np.column_stack([y_prev[:, i], np.ones(M)])
        beta, _, _, _ = np.linalg.lstsq(X_des, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            if 1 + k < len(beta) - 1:
                A_full[i, j] = float(beta[1 + k])
    return A_full


y_prev, y_curr = _build_panel_lag_pairs_simple(Y_norm, split_spec)
A_full = _ols_refit_panel(W_dyno, y_prev, y_curr)
eigs = np.linalg.eigvals(A_full)
max_abs_eig = float(np.max(np.abs(eigs)))
n_above_095 = int(np.sum(np.abs(eigs) > 0.95))
print(f"\nOLS-refit A (post-DYNOTEARS support):")
print(f"  max|eig|       = {max_abs_eig:.4f}")
print(f"  n_eigs > 0.95  = {n_above_095} / {N}")
print(f"  Reference: other methods on V-Dem produced max|eig| in [0.9938, 0.9969]")



# Save final pickle


result = {
    'method': 'dynotears',
    'W_hat':  W_dyno,
    'W_hat_raw': W_dyno.copy(),
    'variable_names': VDEM_VARIABLES,
    'method_config': {
        'p':           1,
        'lambda_w':    lambda_w_fixed,
        'lambda_a':    best['lambda_a'],
        'max_iter':    max_iter,
        'h_tol':       h_tol,
        'w_threshold': 0.001,
        'threshold_policy': 'absolute_post_fit',
        'sweep_results': [
            {k: (v if k != 'A_lag1' else None) for k, v in r.items()}
            for r in results_sweep
        ],
    },
    'training_time_sec': best['time'],
    'seed': 0,
}

test_pkl_path = os.path.join(OUT_DIR, 'dynotears_vdem_test_v2.pkl')
with open(test_pkl_path, 'wb') as f:
    pickle.dump(result, f)
print(f"\nSaved -> {test_pkl_path}")

print("\n" + "=" * 60)
print("DONE. Send the lambda_a sweep table and the final W_hat stats.")
print("=" * 60)

### Cell §4.4 — LSC diagnostic on V-Dem-16 method outputs (5 methods, CANONICAL)

**Paper sections:** §5, Appendix P.

Computes ΔPR with both row-permuting and parametric AR(1) nulls on the 5-method `method_outputs.pkl` produced by §4.4. Generates the V-Dem-16 row of Table 3 and the per-method ΔPR + p-values for both nulls.

Computes the Latent Structure Consistency (LSC) metric on causal-discovery
outputs (W_hat) from 4 methods on V-Dem (medium_16var or long_16var).
LSC is derived from measurement theory: it tests whether W_hat's reflective
within-blocks have approximately rank-1 outer-product structure, as
predicted by Theorem 1 of the LSC appendix.

Key questions this notebook answers:
1. Does LSC discriminate the 4 methods on V-Dem where one-step CRPS does not?
2. Do any methods score significantly above the row-permuting null?
3. Are the LSC scores stable to threshold choice in group classification?

**Inputs:** `method_outputs.pkl` from §4.4; V-Dem panel for parametric-null simulation.

**Outputs:** `lsc_scores.csv`, `groups_identified.json`, `verdict.txt`, sensitivity CSV, ΔPR figure.

**Runtime:** ~11 minutes on Colab Pro G4 GPU.

In [ ]:
# 1. Setup


# Mount Drive, set paths.
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, pickle
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR_BASE = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs')
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# Method-outputs pickles live under p2_outputs/method_outputs/ (phase 2 evaluation run).
# The §5.3 random-baselines run (random_baselines_4method.py) also saves them under a
# different folder; we check both, with phase 2 first.
def find_method_outputs(config_name):
    candidates = [
        os.path.join(DRIVE_DIR, f'p2_outputs/method_outputs/method_outputs_{config_name}.pkl'),
        os.path.join(DRIVE_DIR, f'method_outputs_{config_name}.pkl'),
        os.path.join(DRIVE_DIR, f'random_baselines_4method_outputs/{config_name}/method_outputs_{config_name}.pkl'),
        os.path.join(DRIVE_DIR, f'random_baselines_4method_outputs/{config_name}/method_outputs.pkl'),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find method_outputs for {config_name}. Tried:\n  " + "\n  ".join(candidates))

print(f"DRIVE_DIR    = {DRIVE_DIR}")
print(f"DATA_DIR     = {DATA_DIR}")
print(f"OUT_DIR_BASE = {OUT_DIR_BASE}")

# Verify V-Dem CSVs reachable
print("\nV-Dem data files:")
for f in ['my_data_16.csv', 'my_data_75_years.csv']:
    p = os.path.join(DATA_DIR, f)
    print(f"  {f}: {'EXISTS' if os.path.exists(p) else 'MISSING'} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} B)")



# 2. Run configuration


# --- Choose which V-Dem panel to run ---
CONFIG_NAME = "long_16var"   # or "medium_16var"

# --- LSC parameters ---
B_PERMS                = 100      # number of row-permuting null samples
ALS_MAX_ITERS          = 50
ALS_TOL                = 1e-7
Z_CUTOFF               = 2.0      # tetrad declared "vanishing" if |tau/SD| < Z_CUTOFF.
                                  # NOT a Type I error level — see tetrad_signature() docstring.
                                  # Larger values relax the criterion to "approximate reflectivity."
TETRAD_TYPE_THRESHOLD  = 0.7      # T(C) > 0.7 => reflective; F(C) > 0.7 => formative
WEIGHT_R               = 1/3      # default weight on within-reflective coherence
WEIGHT_C               = 1/3      # default weight on cross-reflective coherence
WEIGHT_F               = 1/3      # default weight on formative coherence

# --- Method selection (use whatever's present in pickle) ---
METHODS_TO_RUN = ["linear_var_granger", "pcmci", "cmlp", "navar", "dynotears"]

# --- Reproducibility ---
SEED = 42

OUT_DIR = os.path.join(OUT_DIR_BASE, CONFIG_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"CONFIG       = {CONFIG_NAME}")
print(f"B_PERMS      = {B_PERMS}")
print(f"OUT_DIR      = {OUT_DIR}")



# 3. Load method outputs and V-Dem panel
#
# We load both: (a) method outputs pickle (W_hat from each of 4 methods), (b) the V-Dem panel itself for tetrad computation.


mo_path = find_method_outputs(CONFIG_NAME)
print(f"Loading method outputs from: {mo_path}")
with open(mo_path, 'rb') as f:
    mo_data = pickle.load(f)

print(f"\nKeys in pickle: {list(mo_data.keys())}")

method_outputs = mo_data['method_outputs']
variable_names = mo_data['panel_summary']['variable_names']
N = len(variable_names)
print(f"N variables: {N}")
print(f"Variables: {variable_names}")


# Identify which methods have valid W_hat (filter out errored runs)
methods_valid = []
for method in METHODS_TO_RUN:
    if method not in method_outputs:
        print(f"  {method}: NOT IN PICKLE — skipping")
        continue
    out = method_outputs[method]
    if 'error' in out:
        print(f"  {method}: ERROR (\"{out['error'][:80]}...\") — skipping")
        continue
    if 'W_hat' not in out:
        print(f"  {method}: no W_hat — skipping")
        continue
    methods_valid.append(method)
    W_hat = out['W_hat']
    nnz = int(np.sum(np.abs(W_hat) > 1e-10))
    max_abs = float(np.max(np.abs(W_hat)))
    print(f"  {method}: W_hat shape={W_hat.shape}, nnz={nnz}, max|W|={max_abs:.4f}")

print(f"\n{len(methods_valid)} methods with valid output: {methods_valid}")
assert len(methods_valid) >= 1, "No valid method outputs found"



# Load V-Dem panel for tetrad computation (need empirical covariance)
import pandas as pd

# Choose the matching CSV for this config
csv_map = {
    "medium_16var": "my_data_16.csv",
    "long_16var":   "my_data_75_years.csv",
}
csv_path = os.path.join(DATA_DIR, csv_map[CONFIG_NAME])
print(f"Loading V-Dem panel: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Panel shape: {df.shape}")
print(f"Columns: {list(df.columns)[:8]}...")

# Stack the 16 indicator columns across all (country, year) observations to compute Cov.
# We use the same 16 variables as in the method outputs pickle.
panel_matrix = df[variable_names].dropna().values  # shape (T*U, N)
print(f"Stacked panel matrix shape: {panel_matrix.shape}")
T_total, N_check = panel_matrix.shape
assert N_check == N, f"Variable count mismatch: panel {N_check} vs method outputs {N}"



# 4. Off-diagonal rank-one ALS solver
#
# **Component A** of LSC requires the off-diagonal rank-one explained-variance ratio:
#
# $$ R_{kk} := 1 - \frac{\min_{u, v} \sum_{i \neq j \in G_k} (\widetilde{W}_{ij} - u_i v_j)^2}{\sum_{i \neq j \in G_k} \widetilde{W}_{ij}^2}. $$
#
# Standard SVD on the diagonal-zeroed matrix would NOT give this — see Remark in the appendix
# (a rank-1 outer product with zeroed diagonal is generally full rank).
#
# We solve via alternating least squares:
#   - $u_i \leftarrow (\sum_{j \neq i} v_j W_{ij}) / (\sum_{j \neq i} v_j^2)$
#   - $v_j \leftarrow (\sum_{i \neq j} u_i W_{ij}) / (\sum_{i \neq j} u_i^2)$
#
# Initialize $u, v$ from the SVD of the diagonal-zeroed matrix (biased start, but converges fast).


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7, return_uv=False, seed=0):
    """Fit u, v minimizing sum_{i != j} (W_ij - u_i v_j)^2. Return explained-variance ratio.

    Parameters
    ----------
    W_block : (m, m) ndarray. Diagonal is set to 0 internally; only off-diagonal cells matter.
    max_iters : ALS iteration cap.
    tol : convergence tolerance on relative residual decrease.
    return_uv : if True, return (R, u, v) instead of just R.
    seed : random seed for fallback initialization.

    Returns
    -------
    R : float in [0, 1]. R=1 means off-diagonal cells exactly equal u_i v_j; R=0 means no rank-1 structure.
        If the off-diagonal sum-of-squares is 0, returns NaN (zero block, excluded from averaging).
    """
    W = W_block.astype(float).copy()
    m = W.shape[0]
    assert W.shape == (m, m)

    # Mask: 1 off-diagonal, 0 on diagonal
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return (np.nan, None, None) if return_uv else np.nan

    # Initialize u, v from SVD of diagonal-zeroed W (biased but reasonable starting point)
    W_zeroed = W.copy()
    np.fill_diagonal(W_zeroed, 0.0)
    try:
        U, S, Vt = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U[:, 0] * np.sqrt(S[0])
        v = Vt[0, :] * np.sqrt(S[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(seed)
        u = rng.standard_normal(m)
        v = rng.standard_normal(m)

    prev_res = np.inf
    for it in range(max_iters):
        # Update u given v: for each i, u_i = sum_{j != i} v_j W_ij / sum_{j != i} v_j^2
        # Vectorized: numerator[i] = sum_j (mask[i,j] * v_j * W_ij)
        #             denominator[i] = sum_j (mask[i,j] * v_j^2)
        num_u = (W * mask) @ v  # (m,)
        v2 = v * v
        den_u = mask @ v2  # (m,)
        # Guard zero denominators
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)

        # Update v given u_new
        num_v = (W * mask).T @ u_new
        u2 = u_new * u_new
        den_v = mask.T @ u2
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)

        u, v = u_new, v_new

        # Check residual on off-diagonal
        rank1 = np.outer(u, v)
        residual = float(np.sum(((W - rank1)[mask]) ** 2))

        if prev_res < np.inf:
            rel_decrease = abs(prev_res - residual) / max(prev_res, 1e-12)
            if rel_decrease < tol:
                break
        prev_res = residual

    R = max(0.0, 1.0 - residual / tss)
    R = min(R, 1.0)  # numerical safety
    if return_uv:
        return R, u, v
    return R



# Quick sanity check on a synthetic rank-1 example
def _test_als():
    rng = np.random.default_rng(0)
    m = 8
    u_true = rng.standard_normal(m)
    v_true = rng.standard_normal(m)
    W_clean = np.outer(u_true, v_true)
    np.fill_diagonal(W_clean, 0.0)
    R = offdiag_rank1_fit(W_clean)
    print(f"Synthetic rank-1 (no noise): R = {R:.6f}  [expect ~1.0]")
    assert R > 0.99, f"ALS failed on clean rank-1 case: R={R}"

    # Add noise
    W_noisy = W_clean + 0.1 * rng.standard_normal((m, m))
    np.fill_diagonal(W_noisy, 0.0)
    R = offdiag_rank1_fit(W_noisy)
    print(f"Synthetic rank-1 (10% noise): R = {R:.4f}  [expect 0.9-0.99]")

    # Random matrix (no rank-1 structure)
    W_rand = rng.standard_normal((m, m))
    np.fill_diagonal(W_rand, 0.0)
    R = offdiag_rank1_fit(W_rand)
    print(f"Random matrix:                R = {R:.4f}  [expect ~1/m = 0.125]")

_test_als()



# 5. Group identification: reflective signatures
#
# We test whether a candidate group of indicators is consistent with 1-factor reflective
# measurement. Two complementary tests:
#
# **Primary: Covariance-rank-1 signature.** Under 1-factor reflective measurement with linear
# independent measurement errors, the off-diagonal cells of the indicator correlation matrix
# satisfy $\rho_{ij} = \lambda_i \lambda_j$, i.e., they fit a rank-1 outer product. We measure
# this directly via the same off-diagonal ALS solver used for Component A of LSC. This gives
# $T(C) \in [0, 1]$ — the explained-variance ratio. High $T(C)$ ⟺ 1-factor reflective.
#
# **Secondary: Tetrad-magnitude signature** (Bollen-Ting 1993). A tetrad
# $\tau_{abcd} = \mathrm{Cov}(a,c)\mathrm{Cov}(b,d) - \mathrm{Cov}(a,d)\mathrm{Cov}(b,c)$ vanishes
# in the population under 1-factor reflective measurement. We compute the fraction of 4-subsets
# with $|\hat\tau / \widehat{SD}| < z_{cutoff}$ as $T_{tetrad}(C)$. Vanishing tetrads are NECESSARY
# but not SUFFICIENT for 1-factor reflectivity (some multi-factor configurations also produce
# them); the cov-rank-1 signature is the more direct test.
#
# Both signatures are reported per group; classification (reflective/formative/unclassified) uses
# the cov-rank-1 signature as primary.


def sample_tetrad(cov, a, b, c, d):
    """tau_{abcd} = Cov(a,c) Cov(b,d) - Cov(a,d) Cov(b,c)"""
    return cov[a, c] * cov[b, d] - cov[a, d] * cov[b, c]


def cov_rank1_signature(panel_data, indices):
    """Test 1-factor reflective structure DIRECTLY via off-diagonal rank-1 fit on the indicator
    correlation matrix.

    Under 1-factor reflective measurement (linear, independent measurement errors), the population
    correlation matrix on the indicators of group C has off-diagonal entries
        Sigma_{ij} = lambda_i * lambda_j     (for i != j in C)
    after standardization, so the off-diagonal cells are rank-one. We fit a rank-one model to
    these off-diagonal cells using the same ALS solver as Component A of LSC, and return the
    explained-variance ratio R^cov_C in [0, 1].

    This is a stronger characterization of 1-factor reflective measurement than vanishing tetrads:
    tetrad vanishing is necessary but not sufficient (multi-factor models can also produce
    vanishing tetrads in particular subsets). Rank-1 covariance is essentially the defining
    property of 1-factor reflective measurement.

    Returns
    -------
    T_C : float in [0, 1] — fraction of off-diagonal correlation variance explained by rank-1 fit.
          High (>0.9) => strongly consistent with 1-factor reflective.
          Low (<0.5)  => clearly NOT 1-factor reflective.
    F_C : 1 - T_C
    """
    if len(indices) < 4:
        return np.nan, np.nan
    sub = panel_data[:, indices]
    sub_std = (sub - sub.mean(0)) / (sub.std(0) + 1e-12)
    corr = np.corrcoef(sub_std, rowvar=False)
    # Use the same off-diagonal ALS we use for Component A
    R = offdiag_rank1_fit(corr.copy(), max_iters=ALS_MAX_ITERS, tol=ALS_TOL)
    if np.isnan(R):
        return np.nan, np.nan
    return float(R), float(1.0 - R)


def tetrad_signature(panel_data, indices, z_cutoff=2.0, n_bootstrap=200, seed=0):
    """Compute T(C), F(C) for a candidate group C with |C| >= 4 via the standardized tetrad
    magnitude test.

    A tetrad is declared "vanishing" if |tau_hat / SD(tau_hat)| < z_cutoff, where SD is estimated
    by panel-bootstrap. The z_cutoff parameter is NOT a Type-I error level; it's a tolerance for
    "approximate vanishing." Larger z_cutoff is more permissive.

    NOTE: vanishing tetrads are NECESSARY for 1-factor reflective measurement but not SUFFICIENT
    (multi-factor models with the right correlation structure can also produce vanishing tetrads).
    This test is therefore best used alongside cov_rank1_signature() which directly checks rank-1.
    """
    from itertools import combinations
    if len(indices) < 4:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    T_total = panel_data.shape[0]
    sub = panel_data[:, indices]
    cov_full = np.cov(sub, rowvar=False, bias=True)
    quartets = list(combinations(range(len(indices)), 4))
    tau_pts = np.array([sample_tetrad(cov_full, a, b, c, d) for (a, b, c, d) in quartets])

    tau_boots = np.zeros((n_bootstrap, len(quartets)))
    for b in range(n_bootstrap):
        idx_resample = rng.integers(0, T_total, T_total)
        sub_b = sub[idx_resample]
        cov_b = np.cov(sub_b, rowvar=False, bias=True)
        for q, (a_, b_, c_, d_) in enumerate(quartets):
            tau_boots[b, q] = sample_tetrad(cov_b, a_, b_, c_, d_)

    sds = np.std(tau_boots, axis=0, ddof=1)
    sds = np.maximum(sds, 1e-12)
    z_scores = np.abs(tau_pts) / sds

    n_vanish = int(np.sum(z_scores < z_cutoff))
    T_C = n_vanish / len(quartets)
    F_C = 1.0 - T_C
    return T_C, F_C



# Group identification: agglomerative clustering on |correlation|, with the cluster count chosen
# to maximize the size-weighted reflective signature. The threshold for labeling groups
# (reflective/formative/unclassified) is APPLIED AFTER selection — it does not affect which
# clustering is chosen, only how its groups are labeled. This decouples the two decisions and
# avoids the failure mode where no clustering reaches the threshold and selection collapses.

def identify_groups(panel_data, variable_names, n_clusters_try=(1, 2, 3, 4),
                    refl_threshold=0.7, form_threshold=0.7, min_group_size=4,
                    z_cutoff=2.0, n_bootstrap=200, seed=0, verbose=True):
    """Data-driven group identification.

    The primary signature T(C) is the off-diagonal rank-1 explained-variance ratio of the
    indicator correlation matrix on group C (cov_rank1_signature). Under 1-factor reflective
    measurement with linear independent measurement errors, this T(C) approaches 1.

    The secondary signature T_tetrad(C) is the standardized-magnitude tetrad test
    (tetrad_signature with the given z_cutoff). Tetrad vanishing is necessary but not
    sufficient for 1-factor reflectivity; cov_rank1 is the more direct test.

    Selection criterion: among candidate cluster counts in ``n_clusters_try``, pick the one
    whose admissible groups (size >= ``min_group_size``) have the highest size-weighted
    average T(C) (rank-1 cov score). Singletons/pairs/triples contribute zero to the score,
    so over-fragmented partitions are penalized automatically.

    Labeling: each group is labeled 'reflective' if T(C) > ``refl_threshold``,
    'formative' if F(C) > ``form_threshold``, else 'unclassified'.

    Returns
    -------
    groups : list of dicts (selected partition, with type labels)
    n_clusters_used : the cluster count chosen
    diagnostics : dict containing per-clustering details for ALL n_clusters tried
    """
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform

    panel_std = (panel_data - panel_data.mean(0)) / (panel_data.std(0) + 1e-12)
    corr = np.corrcoef(panel_std, rowvar=False)
    dist = 1.0 - np.abs(corr)
    dist = (dist + dist.T) / 2
    np.fill_diagonal(dist, 0.0)
    Z = linkage(squareform(dist, checks=False), method='average')

    diagnostics = {}
    candidate_partitions = {}

    for n_c in n_clusters_try:
        labels = fcluster(Z, t=n_c, criterion='maxclust')
        groups_this = []
        for cid in np.unique(labels):
            indices = [i for i, lab in enumerate(labels) if lab == cid]
            size = len(indices)
            if size < min_group_size:
                groups_this.append({
                    'indices': indices,
                    'names': [variable_names[i] for i in indices],
                    'type': 'too_small',
                    'T': np.nan,
                    'F': np.nan,
                    'T_tetrad': np.nan,
                    'F_tetrad': np.nan,
                    'size': size,
                })
                continue
            # Primary: cov-rank-1 signature
            T_C, F_C = cov_rank1_signature(panel_std, indices)
            # Secondary: tetrad-magnitude signature (kept for comparison)
            T_tet, F_tet = tetrad_signature(panel_std, indices, z_cutoff=z_cutoff,
                                            n_bootstrap=n_bootstrap, seed=seed)
            if T_C > refl_threshold:
                gtype = 'reflective'
            elif F_C > form_threshold:
                gtype = 'formative'
            else:
                gtype = 'unclassified'
            groups_this.append({
                'indices': indices,
                'names': [variable_names[i] for i in indices],
                'type': gtype,
                'T': float(T_C),
                'F': float(F_C),
                'T_tetrad': float(T_tet),
                'F_tetrad': float(F_tet),
                'size': size,
            })

        # Selection score: size-weighted average T(C) over admissible groups (cov-rank-1).
        admissible = [g for g in groups_this if g['size'] >= min_group_size]
        if admissible:
            num = sum(g['size'] * g['T'] for g in admissible if not np.isnan(g['T']))
            den = sum(g['size'] for g in admissible if not np.isnan(g['T']))
            score = num / den if den > 0 else 0.0
        else:
            score = 0.0

        candidate_partitions[n_c] = {'groups': groups_this, 'score': score}
        diagnostics[n_c] = {
            'n_admissible_groups': len(admissible),
            'admissible_T_values': [g['T'] for g in admissible],
            'admissible_T_tetrad_values': [g['T_tetrad'] for g in admissible],
            'admissible_sizes': [g['size'] for g in admissible],
            'weighted_T_score': score,
        }

        if verbose:
            type_summary = ", ".join(
                f"{g['type']}({g['size']},T={g['T']:.2f},Ttet={g['T_tetrad']:.2f})"
                if not np.isnan(g['T']) else f"{g['type']}({g['size']})"
                for g in groups_this
            )
            print(f"  n_clusters={n_c}: weighted_T_score={score:.4f} | groups: {type_summary}")

    # Pick best by weighted T-score (with parsimony tie-break: prefer smaller n_clusters)
    best_n = None
    best_score = -np.inf
    for n_c in n_clusters_try:
        s = candidate_partitions[n_c]['score']
        if s > best_score + 1e-9:
            best_score = s
            best_n = n_c

    best_groups = candidate_partitions[best_n]['groups']
    best_groups = [g for g in best_groups if g['size'] >= min_group_size]

    if verbose:
        print(f"\n  Selected: n_clusters={best_n} with weighted_T_score={best_score:.4f}")

    return best_groups, best_n, diagnostics



# Run group identification on the V-Dem panel
print("=" * 70)
print("GROUP IDENTIFICATION via tetrad signatures + agglomerative clustering")
print("=" * 70)
print(f"Trying n_clusters in {(1, 2, 3, 4)}")
print(f"  z_cutoff = {Z_CUTOFF}  (tetrad declared vanishing if |tau/SD| < z_cutoff)")
print(f"  refl_threshold T(C) > {TETRAD_TYPE_THRESHOLD}, form_threshold F(C) > {TETRAD_TYPE_THRESHOLD}")
print(f"  Selection criterion: max size-weighted average T(C) over admissible groups")
print()

groups, n_clusters_used, group_diagnostics = identify_groups(
    panel_matrix, variable_names,
    n_clusters_try=(1, 2, 3, 4),
    refl_threshold=TETRAD_TYPE_THRESHOLD,
    form_threshold=TETRAD_TYPE_THRESHOLD,
    min_group_size=4,
    z_cutoff=Z_CUTOFF,
    n_bootstrap=200,
    seed=SEED,
    verbose=True,
)

print(f"\nFinal group structure ({len(groups)} groups, n_clusters={n_clusters_used}):")
for k, g in enumerate(groups):
    print(f"  Group {k+1} [{g['type']}] size={g['size']}, "
          f"T_cov={g['T']:.3f}, T_tetrad={g['T_tetrad']:.3f}")
    print(f"    Variables: {g['names']}")

# Save group structure with full diagnostics over candidate clusterings
groups_path = os.path.join(OUT_DIR, 'groups_identified.json')
with open(groups_path, 'w') as f:
    json.dump({
        'config_name': CONFIG_NAME,
        'n_clusters_selected': int(n_clusters_used),
        'groups': groups,
        'variable_names': variable_names,
        'z_cutoff': Z_CUTOFF,
        'type_threshold': TETRAD_TYPE_THRESHOLD,
        'candidate_clustering_diagnostics': group_diagnostics,
    }, f, indent=2, default=str)
print(f"\nGroups saved to: {groups_path}")



# 6. LSC components and combined metric
#
# **Component A** ($\mathcal{P}_R$): average $R_{kk}$ over reflective groups with size ≥ 3 and nonzero off-diagonal block.
#
# **Component B** ($\mathcal{R}$): average $R_{l \to k} = \sigma_1^2(\widetilde{E}_{l \to k}) / \|\widetilde{E}_{l \to k}\|_F^2$
# over reflective-to-reflective cross-blocks with size ≥ 2x2 and nonzero block.
#
# **Component C** ($\mathcal{F}$): average $1 - R_{l \to k}$ over formative-source cross-blocks (heuristic).
#
# Cross-block convention: rows = source group, columns = target group, $\widetilde{E}_{l \to k}[i, j] := \widetilde{W}_{ji}$
# for $i \in G_l$, $j \in G_k$.


def cross_block_matrix(W, group_l_indices, group_k_indices):
    """Build E_{l -> k}[i, j] = W[j, i] for i in G_l (source), j in G_k (target).

    Returns matrix of shape (|G_l|, |G_k|).
    """
    # E[a, b] = W[group_k[b], group_l[a]]
    return W[np.ix_(group_k_indices, group_l_indices)].T


def component_PR(W, groups, min_size=3, restrict_to_reflective=True, return_per_group=False):
    """Within-group rank-1 coherence (off-diagonal ALS fit).

    Parameters
    ----------
    restrict_to_reflective : if True, average only over groups labeled 'reflective'
                             (this is the LSC metric proper as defined in the appendix).
                             If False, average over every group of size >= min_size with a
                             nonzero off-diagonal block, ignoring labels. This is the
                             "PR diagnostic" — useful when the data don't classify any group
                             as strictly reflective but we still want to inspect rank-1 structure.

    Returns
    -------
    PR : float or NaN
    per_group : list of dicts (if return_per_group=True), one per scored group
    """
    if restrict_to_reflective:
        eligible = [g for g in groups if g['type'] == 'reflective' and g['size'] >= min_size]
    else:
        eligible = [g for g in groups if g['size'] >= min_size]

    if not eligible:
        return (np.nan, []) if return_per_group else np.nan

    Rs = []
    per_group = []
    for g in eligible:
        idx = g['indices']
        W_block = W[np.ix_(idx, idx)].copy()
        # Zero diagonal (W_ii := 0 by tvNAR convention)
        np.fill_diagonal(W_block, 0.0)
        R = offdiag_rank1_fit(W_block, max_iters=ALS_MAX_ITERS, tol=ALS_TOL)
        if not np.isnan(R):
            Rs.append(R)
        per_group.append({
            'name': g['names'][:3] + ['...'] if len(g['names']) > 3 else g['names'],
            'size': g['size'], 'type': g['type'], 'T': g['T'], 'R_kk': R,
        })

    if not Rs:
        return (np.nan, per_group) if return_per_group else np.nan

    PR = float(np.mean(Rs))
    return (PR, per_group) if return_per_group else PR


def component_RC(W, groups, min_size=2, return_per_pair=False):
    """Cross-reflective-group rank-1 coherence.

    Returns average R_{l -> k} = sigma_1^2 / ||E||_F^2 over reflective-reflective cross-block pairs.
    """
    refl = [g for g in groups if g['type'] == 'reflective' and g['size'] >= min_size]
    if len(refl) < 2:
        return (np.nan, []) if return_per_pair else np.nan

    Rs = []
    per_pair = []
    for i, g_l in enumerate(refl):
        for j, g_k in enumerate(refl):
            if i == j:
                continue
            E = cross_block_matrix(W, g_l['indices'], g_k['indices'])
            fro2 = float(np.sum(E ** 2))
            if fro2 <= 0.0:
                continue
            try:
                s = np.linalg.svd(E, compute_uv=False)
                R = float(s[0] ** 2 / fro2)
            except np.linalg.LinAlgError:
                continue
            Rs.append(R)
            per_pair.append({'source': g_l['names'][:2] + ['...'] if len(g_l['names']) > 2 else g_l['names'],
                             'target': g_k['names'][:2] + ['...'] if len(g_k['names']) > 2 else g_k['names'],
                             'R': R})
    if not Rs:
        return (np.nan, per_pair) if return_per_pair else np.nan
    return (float(np.mean(Rs)), per_pair) if return_per_pair else float(np.mean(Rs))


def component_FC(W, groups, min_size=2, return_per_pair=False):
    """Formative-source heterogeneity: 1 - R_{l -> k} for l formative."""
    form = [g for g in groups if g['type'] == 'formative' and g['size'] >= min_size]
    if not form:
        return (np.nan, []) if return_per_pair else np.nan
    other = [g for g in groups if g['size'] >= min_size]
    if len(other) < 2:
        return (np.nan, []) if return_per_pair else np.nan

    Rs = []
    per_pair = []
    for g_l in form:
        for g_k in other:
            if g_l is g_k:
                continue
            E = cross_block_matrix(W, g_l['indices'], g_k['indices'])
            fro2 = float(np.sum(E ** 2))
            if fro2 <= 0.0:
                continue
            try:
                s = np.linalg.svd(E, compute_uv=False)
                R = float(s[0] ** 2 / fro2)
            except np.linalg.LinAlgError:
                continue
            FH = 1.0 - R
            Rs.append(FH)
            per_pair.append({'source': g_l['names'], 'target': g_k['names'], 'FH': FH})
    if not Rs:
        return (np.nan, per_pair) if return_per_pair else np.nan
    return (float(np.mean(Rs)), per_pair) if return_per_pair else float(np.mean(Rs))


def compute_LSC(W, groups, w_R=WEIGHT_R, w_C=WEIGHT_C, w_F=WEIGHT_F, return_components=False):
    """Combined LSC score with weight renormalization for absent components."""
    PR = component_PR(W, groups)
    RC = component_RC(W, groups)
    FC = component_FC(W, groups)

    weights = []
    values = []
    if not np.isnan(PR):
        weights.append(w_R); values.append(PR)
    if not np.isnan(RC):
        weights.append(w_C); values.append(RC)
    if not np.isnan(FC):
        weights.append(w_F); values.append(FC)

    if not weights:
        return (np.nan, {'PR': PR, 'RC': RC, 'FC': FC, 'weights_used': None}) if return_components else np.nan

    # Renormalize so weights sum to 1
    weights = np.array(weights) / sum(weights)
    LSC = float(np.dot(weights, values))

    if return_components:
        return LSC, {'PR': PR, 'RC': RC, 'FC': FC, 'weights_used': weights.tolist()}
    return LSC



# 7. Row-permuting null
#
# For each row $i$ independently, shuffle the off-diagonal column positions. This preserves the multiset of
# incoming-edge magnitudes per target (since row $i$ contains incoming edges to target $i$ under the
# convention $W_{ij}$ = source $j$ → target $i$) while randomizing which sources contribute to which targets.


def row_permute(W, rng):
    """Row-permuting null: for each row, shuffle off-diagonal column positions independently."""
    N = W.shape[0]
    W_perm = np.zeros_like(W)
    for i in range(N):
        # Off-diagonal positions for this row
        offdiag_cols = [j for j in range(N) if j != i]
        permuted_cols = rng.permutation(offdiag_cols)
        for src_j, tgt_j in zip(offdiag_cols, permuted_cols):
            W_perm[i, tgt_j] = W[i, src_j]
        # Diagonal stays 0
    return W_perm


def lsc_with_null(W, groups, B=B_PERMS, seed=SEED):
    """Compute LSC + PR_diagnostic and their row-permuting null distributions.

    PR_diagnostic differs from PR (LSC component A) in one respect: PR averages R_kk only over
    groups labeled 'reflective', while PR_diagnostic averages R_kk over EVERY group of size >= 3
    regardless of label. PR_diagnostic is therefore always defined when at least one group has
    size >= 3, while LSC may be NaN when no group passes the reflective threshold.
    """
    # LSC proper (uses group labels)
    LSC_obs, comps_obs = compute_LSC(W, groups, return_components=True)

    # PR_diagnostic (unconditional, always computed when a group of size >= 3 exists)
    PR_diag, _ = component_PR(W, groups, restrict_to_reflective=False, return_per_group=True)

    rng = np.random.default_rng(seed)
    null_LSCs = []
    null_PRs = []
    null_RCs = []
    null_FCs = []
    null_PR_diags = []
    for b in range(B):
        W_b = row_permute(W, rng)
        LSC_b, comps_b = compute_LSC(W_b, groups, return_components=True)
        PR_diag_b = component_PR(W_b, groups, restrict_to_reflective=False)
        null_LSCs.append(LSC_b)
        null_PRs.append(comps_b['PR'])
        null_RCs.append(comps_b['RC'])
        null_FCs.append(comps_b['FC'])
        null_PR_diags.append(PR_diag_b)

    null_LSCs = np.array(null_LSCs)
    null_PR_diags = np.array(null_PR_diags)

    def _z_p(obs, null_arr):
        finite = null_arr[np.isfinite(null_arr)]
        if len(finite) == 0 or np.isnan(obs):
            return np.nan, np.nan, np.nan, np.nan
        mu = float(np.mean(finite)); sd = float(np.std(finite))
        z = (obs - mu) / max(sd, 1e-12)
        # right-tailed empirical p
        p = (1 + np.sum(finite >= obs)) / (1 + len(finite))
        return mu, sd, float(z), float(p)

    lsc_mu, lsc_sd, z_lsc, p_lsc = _z_p(LSC_obs, null_LSCs)
    prd_mu, prd_sd, z_prd, p_prd = _z_p(PR_diag, null_PR_diags)

    # Excess metrics (the headline numbers): observed minus null mean.
    # Sparsity-driven floors get subtracted out, so different methods become directly comparable.
    delta_LSC = (LSC_obs - lsc_mu) if (np.isfinite(LSC_obs) and np.isfinite(lsc_mu)) else np.nan
    delta_PR_diag = (PR_diag - prd_mu) if (np.isfinite(PR_diag) and np.isfinite(prd_mu)) else np.nan

    return {
        'LSC': LSC_obs,
        'PR': comps_obs['PR'],
        'RC': comps_obs['RC'],
        'FC': comps_obs['FC'],
        'PR_diagnostic': PR_diag,
        'delta_LSC': float(delta_LSC) if np.isfinite(delta_LSC) else np.nan,
        'delta_PR_diag': float(delta_PR_diag) if np.isfinite(delta_PR_diag) else np.nan,
        'null_LSC_mean': lsc_mu, 'null_LSC_sd': lsc_sd,
        'z_LSC': z_lsc, 'p_LSC': p_lsc,
        'null_PR_diag_mean': prd_mu, 'null_PR_diag_sd': prd_sd,
        'z_PR_diag': z_prd, 'p_PR_diag': p_prd,
        'null_LSCs': null_LSCs,
        'null_PRs': np.array(null_PRs),
        'null_RCs': np.array(null_RCs),
        'null_FCs': np.array(null_FCs),
        'null_PR_diags': null_PR_diags,
    }



# 8. Run on each method's W_hat


print("=" * 70)
print(f"COMPUTING LSC for {len(methods_valid)} methods on {CONFIG_NAME}")
print(f"  Group structure: {len(groups)} groups (n_clusters={n_clusters_used})")
print(f"  Null permutations: B={B_PERMS}")
print("=" * 70)

results = {}
t_start_total = time.time()

for method in methods_valid:
    print(f"\n--- {method} ---")
    t_start = time.time()
    W_hat = method_outputs[method]['W_hat']

    # Quick descriptive stats
    nnz = int(np.sum(np.abs(W_hat) > 1e-10))
    max_abs = float(np.max(np.abs(W_hat)))
    print(f"  W_hat: nnz={nnz}, max|W|={max_abs:.4f}")

    res = lsc_with_null(W_hat, groups, B=B_PERMS, seed=SEED)
    results[method] = res

    elapsed = time.time() - t_start

    def _fmt(x):
        return f"{x:.4f}" if isinstance(x, (int, float)) and not np.isnan(x) else "NaN (no eligible groups)"
    def _fmt_p(x):
        return f"{x:.4f}" if isinstance(x, (int, float)) and not np.isnan(x) else "NaN"

    # HEADLINE: Delta_PR (excess over null) + empirical p-value.
    # Both metrics control for sparsity-driven inflation in raw PR.
    print(f"  --- HEADLINE: excess over row-permuting null ---")
    print(f"  Delta_PR_diag = {_fmt(res['delta_PR_diag'])}   "
          f"(observed {_fmt(res['PR_diagnostic'])}  -  null mean {_fmt(res['null_PR_diag_mean'])})")
    print(f"  p_perm        = {_fmt_p(res['p_PR_diag'])}   (right-tailed; #(null >= obs)+1) / (B+1))")
    if not np.isnan(res['delta_LSC']):
        print(f"  Delta_LSC     = {_fmt(res['delta_LSC'])}   "
              f"(LSC proper; equal to Delta_PR_diag when only one group is reflective)")

    # SECONDARY: raw PR and Gaussian z-score (descriptive, biased by sparsity / non-Gaussian null).
    print(f"  --- secondary (raw, descriptive) ---")
    print(f"  PR observed  = {_fmt(res['PR'])}        [sparsity-inflated; compare to null mean]")
    print(f"  null sd      = {_fmt(res['null_PR_diag_sd'])}")
    print(f"  z (Gaussian) = {res['z_PR_diag']:.2f}    [biased low when null is non-Gaussian]")
    print(f"  elapsed: {elapsed:.1f}s")

print(f"\nTotal time: {time.time() - t_start_total:.1f}s")



# 9. Compare to CRPS rankings
#
# Load §5.3 scored CSVs and compute CRPS per method, then compare the LSC ranking to the CRPS ranking.
# The hypothesis is that LSC discriminates methods at a wider scale than one-step CRPS.


# §9 loads CRPS scores from the phase 2 scored_results folder (or fallback paths).
# Scored CSV columns (per phase 2 run): config, protocol, variant, CRPS, ...
crps_csv_candidates = [
    os.path.join(DRIVE_DIR, f'p2_outputs/scored_results/scored_{CONFIG_NAME}.csv'),
    os.path.join(DRIVE_DIR, f'scored_{CONFIG_NAME}.csv'),
    os.path.join(DRIVE_DIR, f'random_baselines_4method_outputs/{CONFIG_NAME}/scored_{CONFIG_NAME}.csv'),
]
crps_path = next((p for p in crps_csv_candidates if os.path.exists(p)), None)

crps_per_method = {}
if crps_path is not None:
    print(f"Loading CRPS scores from: {crps_path}")
    df_scored = pd.read_csv(crps_path)
    print(f"  shape: {df_scored.shape}, columns: {list(df_scored.columns)}")
    # Phase 2 layout: rows are (protocol, variant) pairs. We want CRPS for each method
    # under the unconstrained protocol (matching how methods were originally evaluated).
    # `variant` is e.g. 'linear_var_granger', 'pcmci', 'cmlp', 'navar', or 'baseline_<k>'.
    # Try alternative column-name conventions defensively.
    method_col = 'variant' if 'variant' in df_scored.columns else (
                 'method' if 'method' in df_scored.columns else None)
    crps_col = 'CRPS' if 'CRPS' in df_scored.columns else (
               'crps' if 'crps' in df_scored.columns else (
               'crps_unconstrained' if 'crps_unconstrained' in df_scored.columns else None))
    if method_col is None or crps_col is None:
        print(f"  Could not parse CRPS columns (method_col={method_col}, crps_col={crps_col})")
    else:
        # Filter to unconstrained protocol if available
        if 'protocol' in df_scored.columns and 'unconstrained' in df_scored['protocol'].unique():
            df_use = df_scored[df_scored['protocol'] == 'unconstrained']
        else:
            df_use = df_scored
        for method in methods_valid:
            rows = df_use[df_use[method_col] == method]
            if len(rows) > 0:
                crps_per_method[method] = float(rows[crps_col].mean())
        print(f"  CRPS per method (unconstrained): {crps_per_method}")
else:
    print("No phase 2 scored_results CSV found at the expected paths. Skipping CRPS comparison.")
    print("  Tried:")
    for p in crps_csv_candidates:
        print(f"    {p}")



# 10. Save results


# Per-method CSV.
# Column order: headline metrics first (Delta_PR_diag and p_perm), then secondary descriptive
# metrics (raw PR, null mean/sd, z-score), then LSC-proper, then CRPS for cross-comparison.
rows_out = []
for method in methods_valid:
    r = results[method]
    rows_out.append({
        'method': method,
        # HEADLINE
        'Delta_PR_diag':       r['delta_PR_diag'],
        'p_perm':              r['p_PR_diag'],
        # SECONDARY (descriptive, biased by sparsity / non-Gaussianity)
        'PR_diagnostic_obs':   r['PR_diagnostic'],
        'null_PR_diag_mean':   r['null_PR_diag_mean'],
        'null_PR_diag_sd':     r['null_PR_diag_sd'],
        'z_gaussian':          r['z_PR_diag'],
        # LSC PROPER (= Delta_PR_diag when only one reflective group, but kept as separate column)
        'Delta_LSC':           r['delta_LSC'],
        'p_perm_LSC':          r['p_LSC'],
        'LSC_obs':             r['LSC'],
        'null_LSC_mean':       r['null_LSC_mean'],
        'null_LSC_sd':         r['null_LSC_sd'],
        'z_LSC_gaussian':      r['z_LSC'],
        # COMPONENTS
        'PR':                  r['PR'],
        'RC':                  r['RC'] if r['RC'] is not None else np.nan,
        'FC':                  r['FC'] if r['FC'] is not None else np.nan,
        # EXTERNAL COMPARISON
        'crps':                crps_per_method.get(method, np.nan),
    })
df_out = pd.DataFrame(rows_out)
# Sort by headline metric (descending Delta_PR_diag = best methods first)
df_out_sorted = df_out.sort_values('Delta_PR_diag', ascending=False).reset_index(drop=True)
csv_path = os.path.join(OUT_DIR, 'lsc_scores.csv')
df_out_sorted.to_csv(csv_path, index=False)
print(f"Saved LSC scores: {csv_path}")
print(df_out_sorted.to_string(index=False))



# Save raw null distributions for later analysis
null_path = os.path.join(OUT_DIR, 'null_distribution.npz')
np.savez(
    null_path,
    **{f"{m}_null_LSCs": results[m]['null_LSCs'] for m in methods_valid},
    **{f"{m}_null_PRs": results[m]['null_PRs'] for m in methods_valid},
    **{f"{m}_null_PR_diags": results[m]['null_PR_diags'] for m in methods_valid},
)
print(f"Saved null distributions: {null_path}")



# Verdict text
verdict_path = os.path.join(OUT_DIR, 'verdict.txt')
with open(verdict_path, 'w') as fh:
    fh.write(f"LSC Diagnostic — {CONFIG_NAME}\n")
    fh.write("=" * 70 + "\n\n")
    fh.write(f"Group identification:\n")
    fh.write(f"  Primary signature: cov-rank-1 (off-diagonal ALS fit on indicator correlation matrix)\n")
    fh.write(f"  Secondary signature: tetrad-magnitude (z_cutoff = {Z_CUTOFF})\n")
    fh.write(f"  Type threshold = {TETRAD_TYPE_THRESHOLD}  (T_cov > thresh => reflective; F > thresh => formative)\n\n")
    fh.write(f"Selected partition: {len(groups)} groups (n_clusters={n_clusters_used})\n")
    for k, g in enumerate(groups):
        fh.write(f"  Group {k+1} [{g['type']}] size={g['size']}, "
                 f"T_cov={g['T']:.3f}, T_tetrad={g['T_tetrad']:.3f}\n")
        fh.write(f"    Variables: {g['names']}\n")
    fh.write("\n")
    fh.write("HEADLINE: methods ranked by Delta_PR_diag (excess over row-permuting null):\n\n")
    # Concise headline view (first 4 columns only)
    head_cols = ['method', 'Delta_PR_diag', 'p_perm', 'PR_diagnostic_obs',
                 'null_PR_diag_mean', 'crps']
    fh.write(df_out_sorted[head_cols].to_string(index=False) + "\n\n")
    fh.write("Full metrics (including secondary descriptive columns):\n\n")
    fh.write(df_out_sorted.to_string(index=False) + "\n\n")
    fh.write("Interpretation:\n")
    fh.write("  - Delta_PR_diag = PR_observed - mean(PR_null). This is the headline metric.\n")
    fh.write("    It corrects for sparsity-driven inflation in raw PR: a method with very sparse\n")
    fh.write("    W_hat will have high observed PR AND high null mean PR, so Delta is small.\n")
    fh.write("    A method that genuinely encodes measurement-theoretic structure has high Delta.\n")
    fh.write("  - p_perm is the empirical right-tailed permutation p-value: (#null >= obs + 1)/(B+1).\n")
    fh.write("    p_perm < 0.05 means the observed PR significantly exceeds the row-permuting null.\n")
    fh.write("  - z_gaussian is shown for description but is biased low when the null is bounded\n")
    fh.write("    or non-Gaussian. Prefer p_perm for inference.\n")
    fh.write("  - The n_clusters sensitivity sweep (sensitivity_sweep.csv) tests robustness of\n")
    fh.write("    method discrimination across grouping granularities.\n")
print(f"Saved verdict: {verdict_path}")



# 11. Visualization


import matplotlib.pyplot as plt

# Methods sorted by headline metric (Delta_PR_diag), descending
methods_sorted = sorted(methods_valid,
                        key=lambda m: -results[m]['delta_PR_diag']
                        if np.isfinite(results[m]['delta_PR_diag']) else 0)

delta_vals = [results[m]['delta_PR_diag'] for m in methods_sorted]
p_vals     = [results[m]['p_PR_diag'] for m in methods_sorted]
obs_vals   = [results[m]['PR_diagnostic'] for m in methods_sorted]
null_means = [results[m]['null_PR_diag_mean'] for m in methods_sorted]
null_sds   = [results[m]['null_PR_diag_sd'] for m in methods_sorted]
null_dists = [results[m]['null_PR_diags'] for m in methods_sorted]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
x = np.arange(len(methods_sorted))

# ============= Panel 1 (HEADLINE): Delta_PR_diag with p-values =============
ax = axes[0]
colors = ['#2ca02c' if p < 0.05 else '#888888' for p in p_vals]
bars = ax.bar(x, delta_vals, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(methods_sorted, rotation=20)
ax.set_ylabel(r'$\Delta\mathrm{PR}_{\mathrm{diag}}$ = observed $-$ null mean')
ax.set_title(f'Headline: excess over row-permuting null  ({CONFIG_NAME})\n'
             f'Bar color: green = $p_{{\\mathrm{{perm}}}} < 0.05$, gray = not significant')
# Annotate p-values on bars
for xi, dv, pv in zip(x, delta_vals, p_vals):
    p_str = f"p={pv:.3f}" if np.isfinite(pv) else "p=NaN"
    y_text = (dv + 0.01) if dv >= 0 else (dv - 0.025)
    ax.text(xi, y_text, p_str, ha='center', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# ============= Panel 2: observed vs null distribution =============
ax = axes[1]
# Plot null as box-and-whisker (shows non-Gaussian shape better than mean ± sd)
bp = ax.boxplot(null_dists, positions=x, widths=0.55, showfliers=True,
                patch_artist=True,
                boxprops=dict(facecolor='lightgray', alpha=0.7),
                medianprops=dict(color='black'))
# Overlay observed values as red diamonds
ax.scatter(x, obs_vals, marker='D', s=100, color='red', zorder=10,
           label='Observed PR (the bar)')
ax.set_xticks(x)
ax.set_xticklabels(methods_sorted, rotation=20)
ax.set_ylabel(r'$\mathrm{PR}_{\mathrm{diag}}$')
ax.set_title(f'Observed vs row-permuting null distribution\n'
             f'(box: null IQR + whiskers; diamond: observed; B={B_PERMS})')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(True, axis='y', alpha=0.3)

# ============= Panel 3: Delta_PR_diag vs CRPS scatter =============
ax = axes[2]
if crps_per_method:
    crps_vals = [crps_per_method.get(m, np.nan) for m in methods_sorted]
    finite = [(m, d, c) for m, d, c in zip(methods_sorted, delta_vals, crps_vals)
              if np.isfinite(d) and np.isfinite(c)]
    if finite:
        ms, ds, cs = zip(*finite)
        ax.scatter(cs, ds, s=140, alpha=0.7, color='steelblue', edgecolor='black')
        for m, d, c in finite:
            ax.annotate(m, (c, d), xytext=(7, 5), textcoords='offset points', fontsize=9)
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
        ax.set_xlabel('CRPS (lower = better predictive accuracy)')
        ax.set_ylabel(r'$\Delta\mathrm{PR}_{\mathrm{diag}}$ (higher = better latent structure)')
        ax.set_title('Deployability vs validity\n'
                     '(no diagonal trend = predictive accuracy and structure recovery are different objectives)')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No finite (Delta, CRPS) pairs',
                ha='center', va='center', transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, 'CRPS scores not available',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Delta_PR_diag vs CRPS (unavailable)')

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'fig_lsc_vs_crps.png')
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved figure: {fig_path}")



# 12. Diagnostic: per-group $R_{kk}$ details
#
# For interpretation: which groups did each method score well on, and which poorly?


print("Per-group R_kk for each method (unconditional, ignores reflective/formative labels):")
print("=" * 70)
for method in methods_valid:
    W_hat = method_outputs[method]['W_hat']
    PR_val, per_group = component_PR(W_hat, groups, restrict_to_reflective=False, return_per_group=True)
    PR_str = "NaN" if (isinstance(PR_val, float) and np.isnan(PR_val)) else f"{PR_val:.4f}"
    print(f"\n{method}: PR_diagnostic = {PR_str}")
    for pg in per_group:
        size = pg['size']
        R = pg['R_kk']
        R_str = "NaN" if (R is None or (isinstance(R, float) and np.isnan(R))) else f"{R:.4f}"
        T_val = pg.get('T', float('nan'))
        T_str = "NaN" if (isinstance(T_val, float) and np.isnan(T_val)) else f"{T_val:.3f}"
        print(f"  group size={size} [{pg['type']}, T_cov={T_str}]: R_kk = {R_str}")



# 13. Sensitivity sweep: n_clusters × method
#
# We hold (z_cutoff, threshold) fixed at their default values and vary only the partition
# granularity n_clusters ∈ {1, 2, 3, 4}. For each n_clusters we force the agglomerative
# clustering to produce that exact number of clusters (no auto-selection within the sweep
# cell), then re-compute Delta_PR_diag + p_perm with B=100 null permutations per method.
#
# This directly tests whether method discrimination depends on grouping granularity —
# i.e., whether ranking the four causal-discovery methods is robust to how we partition
# the 16 V-Dem indicators into reflective sub-groups.


N_CLUSTERS_SWEEP = [1, 2, 3, 4]

print("=" * 70)
print(f"SENSITIVITY SWEEP: n_clusters in {N_CLUSTERS_SWEEP}")
print(f"  z_cutoff fixed at {Z_CUTOFF}, threshold fixed at {TETRAD_TYPE_THRESHOLD}")
print(f"  Each cell runs B={B_PERMS} null permutations per method")
print(f"  Total: {len(N_CLUSTERS_SWEEP) * len(methods_valid)} method-fits with B={B_PERMS} each")
print("=" * 70)

# Helper: force a given n_clusters by calling identify_groups with n_clusters_try=(n_c,).
# This bypasses the auto-selection inside identify_groups so we get exactly the partition
# at that granularity.

sens_rows = []
sweep_t_start = time.time()

for n_c in N_CLUSTERS_SWEEP:
    cell_t_start = time.time()
    print(f"\n[n_clusters={n_c}]")
    g_alt, _, _ = identify_groups(
        panel_matrix, variable_names,
        n_clusters_try=(n_c,),  # force exactly this n_clusters
        refl_threshold=TETRAD_TYPE_THRESHOLD,
        form_threshold=TETRAD_TYPE_THRESHOLD,
        min_group_size=4,
        z_cutoff=Z_CUTOFF,
        n_bootstrap=200,
        seed=SEED,
        verbose=False,
    )
    n_admissible = len(g_alt)
    n_refl = sum(1 for g in g_alt if g['type'] == 'reflective')
    n_form = sum(1 for g in g_alt if g['type'] == 'formative')
    n_unk  = sum(1 for g in g_alt if g['type'] == 'unclassified')
    type_summary = ", ".join(f"{g['type']}({g['size']},T_cov={g['T']:.2f})"
                             for g in g_alt) if g_alt else "(no admissible groups)"
    print(f"  admissible groups: {type_summary}")

    if n_admissible == 0:
        print(f"  No admissible groups (all sub-clusters smaller than min_group_size=4). Skipping.")
        continue

    for m in methods_valid:
        W_hat = method_outputs[m]['W_hat']
        res_alt = lsc_with_null(W_hat, g_alt, B=B_PERMS, seed=SEED)
        sens_rows.append({
            'n_clusters':         n_c,
            'n_admissible_groups':n_admissible,
            'n_reflective':       n_refl,
            'n_formative':        n_form,
            'n_unclassified':     n_unk,
            'method':             m,
            # HEADLINE metrics
            'Delta_PR_diag':      res_alt['delta_PR_diag'],
            'p_perm':             res_alt['p_PR_diag'],
            # Secondary
            'PR_diagnostic_obs':  res_alt['PR_diagnostic'],
            'null_PR_diag_mean':  res_alt['null_PR_diag_mean'],
            'null_PR_diag_sd':    res_alt['null_PR_diag_sd'],
            'z_gaussian':         res_alt['z_PR_diag'],
            # LSC proper (= Delta_PR_diag when only one reflective group)
            'Delta_LSC':          res_alt['delta_LSC'],
            'p_perm_LSC':         res_alt['p_LSC'],
        })
    print(f"  cell elapsed: {time.time() - cell_t_start:.1f}s")

print(f"\nTotal sweep time: {time.time() - sweep_t_start:.1f}s")

df_sens = pd.DataFrame(sens_rows)
sens_path = os.path.join(OUT_DIR, 'sensitivity_sweep.csv')
df_sens.to_csv(sens_path, index=False)

# Pretty pivots: HEADLINE metric and p-value, indexed by n_clusters x method.
if len(df_sens) > 0:
    print("\nDelta_PR_diag (HEADLINE) by n_clusters x method:")
    piv_d = df_sens.pivot_table(index='n_clusters', columns='method',
                                 values='Delta_PR_diag', aggfunc='first')
    print(piv_d.round(4).to_string())

    print("\np_perm (empirical permutation p-value) by n_clusters x method:")
    piv_p = df_sens.pivot_table(index='n_clusters', columns='method',
                                 values='p_perm', aggfunc='first')
    print(piv_p.round(4).to_string())

    # Stability check: do method rankings agree across n_clusters?
    print("\nMethod ranking by Delta_PR_diag at each n_clusters (best -> worst):")
    for n_c in sorted(df_sens['n_clusters'].unique()):
        sub = df_sens[df_sens['n_clusters'] == n_c].sort_values('Delta_PR_diag', ascending=False)
        order = " > ".join(f"{r.method}({r.Delta_PR_diag:+.3f})" for r in sub.itertuples())
        print(f"  n_clusters={n_c}: {order}")

print(f"\nSaved sweep: {sens_path}")



# 14. Done
#
# Outputs written to:
# - `OUT_DIR/lsc_scores.csv`         — per-method headline (Delta_PR_diag + p_perm) and secondaries
# - `OUT_DIR/groups_identified.json` — selected partition + diagnostics for all candidate clusterings
# - `OUT_DIR/null_distribution.npz`  — B=100 null draws per method
# - `OUT_DIR/verdict.txt`            — human-readable summary
# - `OUT_DIR/fig_lsc_vs_crps.png`    — 3-panel: Delta bars, null boxplots, Delta vs CRPS
# - `OUT_DIR/sensitivity_sweep.csv`  — n_clusters × method sweep with B=100 nulls per cell
#
# **What to look for:**
# 1. Headline: which methods have Delta_PR_diag > 0 with p_perm < 0.05? Those are the methods
#    whose W_hat encodes target-relevant source structure beyond row-marginal sparsity artifacts.
# 2. Method ranking by Delta_PR_diag should differ from the raw-PR ranking — sparsity-heavy
#    methods (e.g., cmlp with very few nonzero entries) get inflated raw PR but smaller Delta.
# 3. Sensitivity sweep: do rankings hold across n_clusters? If yes, the diagnostic is robust to
#    partition granularity. If they flip, that's worth flagging in the paper.


print("\n" + "=" * 70)
print("LSC DIAGNOSTIC COMPLETE")
print("=" * 70)
print(f"Config: {CONFIG_NAME}")
print(f"Methods scored: {len(methods_valid)}")
print(f"Output directory: {OUT_DIR}")

### Cell §4.5 — DYNOTEARS verification: merge into pipeline + re-run LSC + multi-criteria: superseded

**Paper section:** §5 (infrastructure).

Merges the DYNOTEARS-fitted W_hat from §4.3 into the existing 4-method `method_outputs.pkl`, then re-runs LSC diagnostic and multi-criteria evaluation on the resulting 5-method pickle. This was the historical bring-up path before §4.1 (the canonical 5-method sweep) was created; it produces the same 5-method results.

**Inputs:** DYNOTEARS pickle from §4.3; original 4-method `method_outputs.pkl`.

**Outputs:** `MERGED_METHODS_PKL` (5-method pickle), `lsc_path` CSV, `summary_path` CSV.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

**Note: This cell is superseded.** The canonical 5-method sweep is now §4.1. This cell remains for historical and infrastructure-verification purposes (it documents the bring-up path used during paper development); anyone running the notebook top-to-bottom can skip it without affecting the canonical results. The numerical outputs here match those produced by §4.1.

In [ ]:
# §0. Setup


import os, sys, pickle, json, time, shutil
from typing import List
import numpy as np
import pandas as pd
import warnings

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

CONFIG_NAME       = 'long_16var'
RB_DIR            = os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs', CONFIG_NAME)
DYNO_TEST_DIR     = os.path.join(DRIVE_DIR, 'dynotears_test_outputs')
LSC_DIAG_DIR      = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', CONFIG_NAME)
MC_OUT_DIR        = os.path.join(DRIVE_DIR, 'multi_criteria_evaluation_outputs')
VERIFY_OUT_DIR    = os.path.join(DRIVE_DIR, 'dynotears_verify_outputs')
os.makedirs(VERIFY_OUT_DIR, exist_ok=True)

# Paths to the existing pickles
ORIG_METHODS_PKL    = os.path.join(RB_DIR, 'method_outputs.pkl')
DYNO_V2_PKL         = os.path.join(DYNO_TEST_DIR, 'dynotears_vdem_test_v2.pkl')

# Output: a new merged pickle with 5 methods
MERGED_METHODS_PKL  = os.path.join(VERIFY_OUT_DIR, 'method_outputs_5method.pkl')

# Verify all inputs exist
print("Input file check:")
for p in [ORIG_METHODS_PKL, DYNO_V2_PKL]:
    print(f"  {p}: {'EXISTS' if os.path.exists(p) else 'MISSING'}")
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required input missing: {p}")
print()



# §1. Merge: load both pickles, combine into 5-method dict


with open(ORIG_METHODS_PKL, 'rb') as f:
    orig = pickle.load(f)
print(f"Original pickle ({CONFIG_NAME}):")
print(f"  Top-level keys: {list(orig.keys())}")
print(f"  Methods: {list(orig['method_outputs'].keys())}")
for m, sub in orig['method_outputs'].items():
    W = sub.get('W_hat')
    if W is not None:
        nnz = int((np.abs(W) > 1e-10).sum())
        print(f"    {m:24s}: W shape={W.shape}, nnz={nnz}")

print()

with open(DYNO_V2_PKL, 'rb') as f:
    dyno = pickle.load(f)
print(f"DYNOTEARS v2 pickle:")
print(f"  Keys: {list(dyno.keys())}")
W_dyno = dyno['W_hat']
nnz_dyno = int((np.abs(W_dyno) > 1e-10).sum())
print(f"  W shape={W_dyno.shape}, nnz={nnz_dyno}, max|W|={np.abs(W_dyno).max():.4f}")

# Build merged method_outputs dict
merged_method_outputs = dict(orig['method_outputs'])  # copy
merged_method_outputs['dynotears'] = {
    'method': 'dynotears',
    'W_hat':  W_dyno,
    'W_hat_raw': dyno.get('W_hat_raw', W_dyno).copy(),
    'variable_names': dyno.get('variable_names', []),
    'method_config':  dyno.get('method_config', {}),
    'training_time_sec': dyno.get('training_time_sec', 0.0),
    'seed': dyno.get('seed', 0),
}

merged = {
    'config_name':    CONFIG_NAME,
    'method_outputs': merged_method_outputs,
    'panel_summary':  orig.get('panel_summary', {}),
}

with open(MERGED_METHODS_PKL, 'wb') as f:
    pickle.dump(merged, f)
print(f"\n5-method merged pickle saved -> {MERGED_METHODS_PKL}")
print(f"Methods now: {list(merged['method_outputs'].keys())}")



# §2. Load V-Dem panel (same loader as other notebooks)


DATA_DIR = os.path.join(DRIVE_DIR, 'data')
VDEM_DATA_PATH = os.path.join(DATA_DIR, 'my_data_75_years.csv')
VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]
MIN_SERIES_LENGTH = 9; VAL_YEARS = 5

try:
    df_raw = pd.read_csv(VDEM_DATA_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(VDEM_DATA_PATH, encoding='latin1')

keep_cols = ['country_id', 'year'] + VDEM_VARIABLES
df_panel = (df_raw[keep_cols].dropna(subset=VDEM_VARIABLES)
            .sort_values(['country_id', 'year']).reset_index(drop=True))
lengths = df_panel.groupby('country_id')['year'].count()
df_panel = df_panel[df_panel['country_id'].isin(
    lengths[lengths >= MIN_SERIES_LENGTH].index)]
train_parts = []
for cid, g in df_panel.groupby('country_id', sort=False):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
        train_parts.append(g.iloc[:-VAL_YEARS])
    else:
        train_parts.append(g)
train_df = pd.concat(train_parts, ignore_index=True)
vdem_split_spec = train_df.groupby('country_id', sort=False)['year'].count().tolist()
Y_train = train_df[VDEM_VARIABLES].to_numpy(float)
mu, sd = Y_train.mean(0), Y_train.std(0)
sd = np.where(sd < 1e-12, 1.0, sd)
vdem_Y_norm = (Y_train - mu) / sd
N_VDEM = vdem_Y_norm.shape[1]
print(f"V-Dem panel: U={len(vdem_split_spec)}, T_total={vdem_Y_norm.shape[0]}, N={N_VDEM}")



# §3. ΔPR computation: replay LSC diagnostic for DYNOTEARS only


# Load the existing groups
groups_path = os.path.join(LSC_DIAG_DIR, 'groups_identified.json')
with open(groups_path) as f:
    groups_obj = json.load(f)
groups = groups_obj['groups'] if isinstance(groups_obj, dict) else groups_obj
print(f"Groups (from existing diagnostic): "
      f"{[(g.get('type'), len(g.get('indices', []))) for g in groups]}")



# PR computation: off-diagonal ALS rank-one fit on within-block sub-matrix.
# This is the same computation lsc_diagnostic_vdem uses; we vendor a minimal
# version here so we don't need to import the whole notebook.

def _pr_within_block(W, indices, max_iters=50, tol=1e-7):
    """PR(W | block): off-diagonal rank-one ALS fit, normalized by Frob-squared."""
    p = len(indices)
    if p < 2:
        return np.nan
    sub = W[np.ix_(indices, indices)].copy()
    # Off-diagonal mask
    mask = ~np.eye(p, dtype=bool)
    s_off = sub[mask]
    s_off_norm2 = float(np.sum(s_off ** 2))
    if s_off_norm2 < 1e-20:
        return 0.0
    # ALS: minimize sum_ij (sub_ij - u_i v_j)^2 over off-diagonal entries
    rng = np.random.default_rng(0)
    u = rng.standard_normal(p)
    v = rng.standard_normal(p)
    prev_err = np.inf
    for _ in range(max_iters):
        # Update u_i: u_i = sum_j (sub_ij * v_j) / sum_j v_j^2 over j != i
        for i in range(p):
            mask_j = np.arange(p) != i
            num = np.sum(sub[i, mask_j] * v[mask_j])
            den = np.sum(v[mask_j] ** 2) + 1e-20
            u[i] = num / den
        # Update v_j similarly
        for j in range(p):
            mask_i = np.arange(p) != j
            num = np.sum(sub[mask_i, j] * u[mask_i])
            den = np.sum(u[mask_i] ** 2) + 1e-20
            v[j] = num / den
        # Convergence check
        approx = np.outer(u, v)
        approx[~mask] = 0
        sub_off = sub.copy(); sub_off[~mask] = 0
        err = float(np.sum((sub_off - approx) ** 2))
        if abs(prev_err - err) < tol * (1 + prev_err):
            break
        prev_err = err
    return 1.0 - err / s_off_norm2


def _pr_total(W, groups):
    """Average PR across reflective groups (size>=2)."""
    refl = [g for g in groups if g.get('type') == 'reflective' and len(g.get('indices', [])) >= 2]
    if not refl:
        return np.nan
    vals = [_pr_within_block(W, g['indices']) for g in refl]
    return float(np.mean(vals))


def _row_permute(W, rng):
    """Row-permuting null: for each row, permute off-diagonal entries."""
    N_ = W.shape[0]
    Wp = np.zeros_like(W)
    for i in range(N_):
        row = W[i].copy()
        row[i] = 0.0
        others = np.r_[:i, i+1:N_]
        perm = rng.permutation(others)
        Wp[i, perm] = row[others]
    return Wp



# Compute ΔPR for all 5 methods using the same procedure
B_PERMS = 100
print(f"Computing PR (observed and {B_PERMS}-perm null) for all 5 methods...")
print()

lsc_results = []
for method_name, sub in merged['method_outputs'].items():
    W = sub.get('W_hat')
    if W is None:
        continue
    t0 = time.time()
    pr_obs = _pr_total(W, groups)
    rng = np.random.default_rng(42 + hash(method_name) % 1000)
    null_pr = []
    for b in range(B_PERMS):
        Wp = _row_permute(W, rng)
        null_pr.append(_pr_total(Wp, groups))
    null_pr = np.asarray(null_pr)
    null_mean = float(np.mean(null_pr))
    null_sd = float(np.std(null_pr, ddof=1))
    delta_pr = pr_obs - null_mean
    # Plus-one permutation p-value
    p_perm = float((1 + np.sum(null_pr >= pr_obs)) / (1 + B_PERMS))
    elapsed = time.time() - t0
    nnz = int((np.abs(W) > 1e-10).sum())
    lsc_results.append({
        'method':            method_name,
        'nnz':               nnz,
        'PR_obs':            pr_obs,
        'null_PR_mean':      null_mean,
        'null_PR_sd':        null_sd,
        'Delta_PR_diag':     delta_pr,
        'p_perm':            p_perm,
        'time_sec':          elapsed,
    })
    print(f"  {method_name:24s}: nnz={nnz:4d}  PR_obs={pr_obs:.4f}  "
          f"null_mean={null_mean:.4f}  ΔPR={delta_pr:+.4f}  p_perm={p_perm:.4f}  "
          f"({elapsed:.1f}s)")

lsc_df = pd.DataFrame(lsc_results)
lsc_path = os.path.join(VERIFY_OUT_DIR, 'lsc_5method_verification.csv')
lsc_df.to_csv(lsc_path, index=False)
print(f"\nSaved -> {lsc_path}")



# §4. Geweke + VAR-implied covariance fit for DYNOTEARS
#
# Vendor the relevant computations from multi_criteria_evaluation.py.


def _build_panel_lag_pairs_simple(Y, split_spec):
    yp, yc = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            yp.append(unit[:-1]); yc.append(unit[1:])
        cursor += n_u
    return np.vstack(yp), np.vstack(yc)


def geweke_improvement(W_hat, Y, split_spec, support_threshold=1e-10):
    from scipy import stats as scstats
    N_ = Y.shape[1]
    y_prev, y_curr = _build_panel_lag_pairs_simple(Y, split_spec)
    M = y_prev.shape[0]
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    df = int(support.sum())
    F_per_target = np.zeros(N_)
    n_skipped = 0
    for i in range(N_):
        y_target = y_curr[:, i]
        X_base = np.column_stack([y_prev[:, i], np.ones(M)])
        beta_base, _, _, _ = np.linalg.lstsq(X_base, y_target, rcond=None)
        s2_base = float(np.mean((y_target - X_base @ beta_base) ** 2))
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_full = np.column_stack(cols)
        if X_full.shape[1] >= M:
            n_skipped += 1
            continue
        beta_full, _, _, _ = np.linalg.lstsq(X_full, y_target, rcond=None)
        s2_full = float(np.mean((y_target - X_full @ beta_full) ** 2))
        F_per_target[i] = float(np.log(s2_base / max(s2_full, 1e-20))) if s2_full > 0 else 0.0
    F_total = float(F_per_target.sum())
    chi2_stat = M * F_total
    p_value = float(scstats.chi2.sf(chi2_stat, df=df)) if (df > 0 and chi2_stat > 0) else 1.0
    return {'F_total': F_total, 'chi2': chi2_stat, 'df': df,
            'p_value': p_value, 'n_skipped': n_skipped, 'M': M}


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    M, N_ = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N_, N_))
    resid_full = np.zeros((M, N_))
    for i in range(N_):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_des = np.column_stack(cols)
        if X_des.shape[1] >= M:
            X_des = np.column_stack([y_prev[:, i], np.ones(M)])
            beta, _, _, _ = np.linalg.lstsq(X_des, y_target, rcond=None)
            A_full[i, i] = float(beta[0])
            resid_full[:, i] = y_target - X_des @ beta
            continue
        beta, _, _, _ = np.linalg.lstsq(X_des, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            if 1 + k < len(beta) - 1:
                A_full[i, j] = float(beta[1 + k])
        resid_full[:, i] = y_target - X_des @ beta
    Sigma_eps = (resid_full.T @ resid_full) / max(M - 1, 1)
    eigs_e = np.linalg.eigvalsh(Sigma_eps)
    if eigs_e.min() < 1e-12:
        Sigma_eps = Sigma_eps + (1e-10 - eigs_e.min()) * np.eye(N_)
    return A_full, resid_full, Sigma_eps


def _implied_stationary_cov(A, Sigma_eps):
    eigs = np.linalg.eigvals(A)
    if np.max(np.abs(eigs)) >= 1.0:
        return None
    from scipy.linalg import solve_discrete_lyapunov
    return solve_discrete_lyapunov(A, Sigma_eps)


def _srmr_block(Sigma_emp_b, Sigma_imp_b, p_block):
    if p_block < 2 or Sigma_imp_b is None:
        return np.nan
    se_diag = np.sqrt(np.diag(Sigma_emp_b) + 1e-20)
    diff = Sigma_emp_b - Sigma_imp_b
    standardized = diff / np.outer(se_diag, se_diag)
    n_distinct = p_block * (p_block + 1) // 2
    iu = np.tril_indices(p_block)
    return float(np.sqrt(np.sum(standardized[iu] ** 2) / max(n_distinct, 1)))



print("\nComputing Geweke F + VAR-implied SRMR for all 5 methods...")
print()

y_prev, y_curr = _build_panel_lag_pairs_simple(vdem_Y_norm, vdem_split_spec)
M_pairs = y_prev.shape[0]
Yc = y_curr - y_curr.mean(axis=0)
Sigma_emp = (Yc.T @ Yc) / max(M_pairs - 1, 1)

extra_results = []
for method_name, sub in merged['method_outputs'].items():
    W = sub.get('W_hat')
    if W is None:
        continue
    # Geweke F
    gw = geweke_improvement(W, vdem_Y_norm, vdem_split_spec)
    # VAR-implied SRMR (per block, average)
    A_full, _, Sigma_eps = _ols_refit_panel(W, y_prev, y_curr)
    Sigma_imp = _implied_stationary_cov(A_full, Sigma_eps)
    block_srmrs = []
    for g in groups:
        idx = g.get('indices', [])
        if len(idx) < 2:
            continue
        Sigma_emp_b = Sigma_emp[np.ix_(idx, idx)]
        Sigma_imp_b = Sigma_imp[np.ix_(idx, idx)] if Sigma_imp is not None else None
        srmr = _srmr_block(Sigma_emp_b, Sigma_imp_b, len(idx))
        if np.isfinite(srmr):
            block_srmrs.append(srmr)
    srmr_mean = float(np.mean(block_srmrs)) if block_srmrs else np.nan
    eigs_A = np.linalg.eigvals(A_full)
    max_eig = float(np.max(np.abs(eigs_A)))
    extra_results.append({
        'method':           method_name,
        'geweke_F':         gw['F_total'],
        'geweke_p_postsel': gw['p_value'],
        'geweke_n_skipped': gw['n_skipped'],
        'fit_srmr':         srmr_mean,
        'srmr_passes':      bool(srmr_mean < 0.08) if np.isfinite(srmr_mean) else False,
        'max_abs_eig_A':    max_eig,
    })
    print(f"  {method_name:24s}: Geweke F={gw['F_total']:+.4f} (p={gw['p_value']:.2g}, "
          f"skipped={gw['n_skipped']:2d})  SRMR={srmr_mean:.4f}  max|eig|={max_eig:.4f}")

extra_df = pd.DataFrame(extra_results)



# §5. Combined 5-method summary


combined = lsc_df.merge(extra_df, on='method', how='left')

# Add lsc_passes binary
combined['lsc_passes'] = (combined['Delta_PR_diag'] > 0) & (combined['p_perm'] < 0.05)
# Geweke pass (descriptive, post-selection-biased — see paper §3.1)
combined['geweke_passes_descriptive'] = combined['geweke_p_postsel'] < 0.05

print("\n" + "=" * 85)
print("5-METHOD VERIFICATION SUMMARY (V-Dem long_16var, no random-baseline test yet)")
print("=" * 85)
display_cols = ['method', 'nnz', 'Delta_PR_diag', 'p_perm', 'lsc_passes',
                'geweke_F', 'geweke_passes_descriptive', 'fit_srmr', 'srmr_passes',
                'max_abs_eig_A']
print(combined[display_cols].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

summary_path = os.path.join(VERIFY_OUT_DIR, 'verification_5method_summary.csv')
combined.to_csv(summary_path, index=False)
print(f"\nSaved -> {summary_path}")



# §6. Verdict


print("\n" + "=" * 60)
print("VERIFICATION COMPLETE")
print("=" * 60)
print("Send the table above. If DYNOTEARS row looks sensible (finite ΔPR,")
print("max|eig| near 0.99, SRMR similar magnitude to others), we proceed")
print("to full pipeline integration.")

### Cell §4.6 — Regenerate §5 figure with DYNOTEARS

**Paper section:** §5 (figure regeneration).

Re-renders `fig_baseline_crps.png` with DYNOTEARS bars added. Cosmetic update to include the 5th method. The original figure was produced by lsc_diagnostic_vdem_long.py, which loads CRPS
from `scored_<config>.csv` (a phase-2 file that doesn't include dynotears). This
standalone script merges 5-method CRPS from `random_baselines_4method_outputs/
{config}/sweep_raw.csv` instead, then re-draws the same 3-panel figure.

**Outputs:** DRIVE_DIR/lsc_diagnostic_outputs/long_16var/fig_lsc_vs_crps.png  (overwritten)

**Inputs:** `lsc_path` from §4.4, `sweep_path` from §4.1.

**Outputs:** Updated figure.

**Runtime:** Seconds.

In [ ]:
#Regenerate the figure with dynotears


from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DRIVE_DIR   = '/content/drive/MyDrive/NeurIPS2026_1296'
CONFIG_NAME = 'long_16var'
LSC_DIR     = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', CONFIG_NAME)
RB_DIR      = os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs', CONFIG_NAME)
B_PERMS     = 100  # for caption only

print(f"LSC_DIR = {LSC_DIR}")
print(f"RB_DIR  = {RB_DIR}")


# 1. Load the LSC diagnostic outputs (ΔPR + null distributions)


lsc_path = os.path.join(LSC_DIR, 'lsc_scores.csv')
df_lsc = pd.read_csv(lsc_path)
print(f"Loaded {lsc_path}: {df_lsc.shape}")
print(df_lsc[['method', 'Delta_PR_diag', 'p_perm', 'PR_diagnostic_obs',
              'null_PR_diag_mean']].to_string(index=False))

null_path = os.path.join(LSC_DIR, 'null_distribution.npz')
npz = np.load(null_path)
print(f"\nLoaded {null_path}")
print(f"  keys: {list(npz.keys())[:6]} ...")

# Build per-method null PR_diag arrays
null_pr_diags = {}
for m in df_lsc['method']:
    key = f"{m}_null_PR_diags"
    if key in npz:
        null_pr_diags[m] = npz[key]
    else:
        null_pr_diags[m] = np.array([])
        print(f"  WARNING: no null draws for {m} (key {key} missing)")


# 2. Load 5-method CRPS from sweep_raw.csv (rep == -1 rows)


sweep_path = os.path.join(RB_DIR, 'sweep_raw.csv')
df_sweep = pd.read_csv(sweep_path)
print(f"Loaded {sweep_path}: {df_sweep.shape}")

# Method-level rows have rep == -1 and family == 'method'
df_method = df_sweep[(df_sweep['rep'] == -1)].copy()
print(f"\nMethod-level rows ({len(df_method)}):")
print(df_method[['method', 'CRPS_unconstrained', 'CRPS_constrained',
                 'nnz_W']].to_string(index=False))

# Use unconstrained CRPS (consistent with verdict.txt headline reporting)
crps_per_method = dict(zip(df_method['method'], df_method['CRPS_unconstrained']))
print(f"\ncrps_per_method (unconstrained):")
for m, c in crps_per_method.items():
    print(f"  {m}: {c:.4f}")


# 3. Build the 3-panel figure
#
# Panel 1 (left): ΔPR_diag bars with p-values, green if p<0.05.
# Panel 2 (middle): observed PR vs null distribution boxplots.
# Panel 3 (right): ΔPR_diag vs CRPS scatter with all 5 methods.


# Sort methods by ΔPR_diag descending (best first)
df_sorted = df_lsc.sort_values('Delta_PR_diag', ascending=False).reset_index(drop=True)
methods_sorted = df_sorted['method'].tolist()
delta_vals     = df_sorted['Delta_PR_diag'].tolist()
p_vals         = df_sorted['p_perm'].tolist()
obs_vals       = df_sorted['PR_diagnostic_obs'].tolist()
null_means     = df_sorted['null_PR_diag_mean'].tolist()
null_dists     = [null_pr_diags[m] for m in methods_sorted]
crps_vals      = [crps_per_method.get(m, np.nan) for m in methods_sorted]

print(f"\nSorted by Delta_PR_diag (descending): {methods_sorted}")
print(f"CRPS for each: {[round(c, 4) if np.isfinite(c) else None for c in crps_vals]}")


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
x = np.arange(len(methods_sorted))

# ============= Panel 1 (HEADLINE): ΔPR_diag with p-values =============
ax = axes[0]
colors = ['#2ca02c' if (np.isfinite(p) and p < 0.05) else '#888888' for p in p_vals]
ax.bar(x, delta_vals, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(methods_sorted, rotation=20, ha='right')
ax.set_ylabel(r'$\Delta\mathrm{PR}_{\mathrm{diag}}$ = observed $-$ null mean')
ax.set_title(
    f'Headline: excess over row-permuting null  ({CONFIG_NAME})\n'
    r'Bar color: green = $p_{\mathrm{perm}} < 0.05$, gray = not significant',
    fontsize=11,
)
# Annotate p-values on bars.
# Positive bars: label sits just above the bar top (above zero).
# Negative bars: label sits ABOVE the bar (between bar top and zero), so that
# rotated x-axis tick labels don't overlap it.
for xi, dv, pv in zip(x, delta_vals, p_vals):
    p_str = f"p={pv:.3f}" if np.isfinite(pv) else "p=NaN"
    if dv >= 0:
        y_text = dv + 0.012
        va = 'bottom'
    else:
        # Place label between bar top (= dv, negative) and 0, biased toward 0
        y_text = max(dv + 0.012, -0.012)
        va = 'bottom'
    ax.text(xi, y_text, p_str, ha='center', va=va, fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# ============= Panel 2: observed vs null distribution =============
ax = axes[1]
# Filter to methods with non-empty null draws
positions_with_data = [i for i, nd in enumerate(null_dists) if len(nd) > 0]
dists_with_data     = [null_dists[i] for i in positions_with_data]
if dists_with_data:
    ax.boxplot(
        dists_with_data,
        positions=[x[i] for i in positions_with_data],
        widths=0.55, showfliers=True, patch_artist=True,
        boxprops=dict(facecolor='lightgray', alpha=0.7),
        medianprops=dict(color='black'),
    )
ax.scatter(x, obs_vals, marker='D', s=100, color='red', zorder=10,
           label='Observed PR (the bar)')
ax.set_xticks(x)
ax.set_xticklabels(methods_sorted, rotation=20, ha='right')
ax.set_ylabel(r'$\mathrm{PR}_{\mathrm{diag}}$')
ax.set_title(
    'Observed vs row-permuting null distribution\n'
    f'(box: null IQR + whiskers; diamond: observed; B={B_PERMS})',
    fontsize=11,
)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(True, axis='y', alpha=0.3)

# ============= Panel 3: ΔPR_diag vs CRPS scatter =============
ax = axes[2]
finite = [(m, d, c) for m, d, c in zip(methods_sorted, delta_vals, crps_vals)
          if np.isfinite(d) and np.isfinite(c)]
if finite:
    ms, ds, cs = zip(*finite)
    ax.scatter(cs, ds, s=140, alpha=0.7, color='steelblue', edgecolor='black')

    # Smart label placement: when two methods are close together (e.g. lvg and
    # dynotears on V-Dem with near-identical CRPS and ΔPR), put the second one's
    # label below-right rather than above-right so they don't collide.
    label_offsets = {}  # method -> (dx, dy) in points
    default_offset = (7, 5)

    pts = list(zip(ms, cs, ds))
    used_below = set()
    eps_x = 0.002   # CRPS proximity threshold
    eps_y = 0.020   # ΔPR proximity threshold
    for i, (m_i, c_i, d_i) in enumerate(pts):
        if m_i in label_offsets:
            continue
        # Find nearby points
        for j, (m_j, c_j, d_j) in enumerate(pts):
            if i >= j:
                continue
            if abs(c_i - c_j) < eps_x and abs(d_i - d_j) < eps_y:
                # Two collide. Put first label above-right, second below-right.
                label_offsets[m_i] = (7, 5)
                label_offsets[m_j] = (7, -12)
                used_below.add(m_j)
        if m_i not in label_offsets:
            label_offsets[m_i] = default_offset

    for m, d, c in finite:
        dx, dy = label_offsets.get(m, default_offset)
        ax.annotate(m, (c, d), xytext=(dx, dy), textcoords='offset points', fontsize=9)

    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_xlabel('CRPS (lower = better predictive accuracy)')
    ax.set_ylabel(r'$\Delta\mathrm{PR}_{\mathrm{diag}}$ (higher = better latent structure)')
    ax.set_title(
        'Deployability vs validity\n'
        '(no diagonal trend = predictive accuracy and structure recovery are different objectives)',
        fontsize=11,
    )
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No finite (Delta, CRPS) pairs',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
fig_path = os.path.join(LSC_DIR, 'fig_lsc_vs_crps.png')
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"\nSaved figure: {fig_path}")

### Cell §4.7 — Multi-criteria evaluation on V-Dem-16 (Geweke + fit indices)

**Paper sections:** §5, Appendix P.

Computes the Geweke predictive-improvement test and SEM-style fit indices (SRMR, RMSEA) for the 5 V-Dem methods. Completes the multi-criteria summary table for V-Dem-16 (the other criteria — CRPS, rb-beats are produced by §4.1, ΔPR is produced by §4.4).


 **The five criteria:**
   1. **CRPS** — predictive accuracy (loaded from §5.3 sweep results).
   2. **CRPS-vs-random-baseline** — does method beat density-matched random
      W on CRPS? (Loaded from §5.3 verdict.)
   3. **LSC ΔPR / ΔLSC + permutation null** — latent-structure recovery
      (loaded from prior diagnostic).
   4. **Geweke predictive improvement** — does method's W_hat improve
      predictive log-likelihood over per-variable AR(1) baseline?
      (See caveat in §1: the chi-squared p-value is anti-conservative
      because the support is data-driven; we report it as descriptive.)
   5. **VAR-implied covariance fit** — RMSEA, CFI, TLI, SRMR, NFI computed
      from the implied stationary covariance under the W-induced VAR(1).
      Note: this is NOT a CFA reflective-measurement-model fit; we use the
      formulas analogously, applied to a dynamic-VAR-implied covariance.
      (See caveat in §2: block-level df is approximate.)

**Output structure:** 5 criteria reported; 4 of them have binary pass conditions
 (CRPS-vs-RB, LSC, Geweke, fit); CRPS itself is reported as a descriptive
 measurement. Pass score is out of 4.

 **Methodological note**: V-Dem block types come from cov-rank-1 group
 identification (the same procedure used in §3 of the paper). All five
 fit indices are computed on every block of size >= 2, regardless of type;
 block type is reported alongside but does not gate which indices fire.

**Setup:** relies on
   - `$DRIVE_DIR/method_outputs_long_16var.pkl` (V-Dem method outputs)
     OR `$DRIVE_DIR/random_baselines_4method_outputs/long_16var/method_outputs.pkl`
   - `$DRIVE_DIR/random_baselines_4method_outputs/long_16var/sweep_raw.csv`
     and `verdict.csv` (CRPS + random-baseline verdict)
   - `$DRIVE_DIR/data/my_data_75_years.csv` (V-Dem panel)
   - `$DRIVE_DIR/lsc_diagnostic_vdem_outputs/groups_identified.json`
     (cov-rank-1 group identification; both wrapper and bare-list formats handled)

**Inputs:** V-Dem panel; `method_outputs.pkl` from §4.1.

**Runtime:** ~10-15 minutes on Colab Pro G4 GPU.

In [ ]:
# §0. Colab setup


import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

VDEM_RANDBASE_DIR = os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs', 'long_16var')
VDEM_DIAG_DIR     = os.path.join(DRIVE_DIR, 'lsc_diagnostic_vdem_outputs')
# Actual lsc_diagnostic_vdem.py output dir (config-subdirectoried)
VDEM_LSC_DIAG_DIR = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
OUT_DIR           = os.path.join(DRIVE_DIR, 'multi_criteria_evaluation_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"DRIVE_DIR  = {DRIVE_DIR}")
print(f"OUT_DIR    = {OUT_DIR}")


import numpy as np
import pandas as pd
import pickle
import json
from typing import Any, Dict, List, Optional, Tuple

# Threshold for declaring rb_passes = True. The verdict.txt file uses
# 4/6 ("MOSTLY BEATS") and 6/6 ("STRONGLY BEATS"); 3/6 is "MIXED".
# Default here: pass requires at least RB_PASS_FRACTION of pairs to beat.
RB_PASS_FRACTION = 0.5   # >= 3/6 = MIXED-or-better. Bump to 0.67 for stricter.


# §1. Geweke predictive-improvement test
#
# Refit method's support by OLS, compare residual variance to per-variable
# AR(1) baseline, return the log-ratio F = log(σ²_base / σ²_full) and an
# asymptotic χ² statistic.
#
# **CAVEAT (read before interpreting p-values):**
# The chi-squared statistic is computed as M * F_total with df = support.sum(),
# but the support set itself is data-driven (selected by W_hat's nonzeros,
# which were derived from the same data). Treating this as a clean χ²(df)
# test overstates significance — it does not account for selection.
#
# We report the p-value because it is informative as a comparative
# descriptive (a smaller "p" indicates a larger effect relative to df), but
# downstream `geweke_passes` should be read as "the descriptive predictive-
# improvement statistic exceeds the chi-squared 0.05 quantile under the
# (post-selection-biased) test." It is NOT a calibrated significance test.


def _build_panel_lag_pairs_simple(Y, split_spec):
    y_prev_list, y_curr_list = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            y_prev_list.append(unit[:-1])
            y_curr_list.append(unit[1:])
        cursor += n_u
    if not y_prev_list:
        raise ValueError("No valid (t-1, t) pairs.")
    return np.vstack(y_prev_list), np.vstack(y_curr_list)


def geweke_improvement(W_hat, Y, split_spec, support_threshold=1e-10):
    """Predictive-improvement test of W_hat's support over per-variable AR(1).

    Returns dict with:
      F_total          : log-variance reduction summed across targets
      chi2             : M * F_total (asymptotic χ² test statistic)
      df               : support.sum() — number of cross-effects in the model
      p_value          : χ² p-value at this df. **POST-SELECTION-BIASED.**
                          Treat as descriptive, not as a calibrated test.
      sigma2_base/full : mean residual variance per target under each model
      M                : number of (t-1, t) pairs used
      n_skipped        : number of targets where the full model was over-
                          parameterized (X.shape[1] >= M); F set to 0 for
                          these. Reported for transparency.
    """
    from scipy import stats as scstats
    N = Y.shape[1]
    y_prev, y_curr = _build_panel_lag_pairs_simple(Y, split_spec)
    M = y_prev.shape[0]

    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    df = int(support.sum())

    F_per_target = np.zeros(N)
    sigma2_base = np.zeros(N)
    sigma2_full = np.zeros(N)
    n_skipped = 0   # X3: count over-parameterized targets for transparency

    for i in range(N):
        y_target = y_curr[:, i]
        X_base = np.column_stack([y_prev[:, i], np.ones(M)])
        beta_base, _, _, _ = np.linalg.lstsq(X_base, y_target, rcond=None)
        s2_base = float(np.mean((y_target - X_base @ beta_base) ** 2))
        sigma2_base[i] = s2_base

        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X_full = np.column_stack(cols)
        if X_full.shape[1] >= M:
            # Over-parameterized: cannot fit full model, no improvement possible
            sigma2_full[i] = s2_base
            F_per_target[i] = 0.0
            n_skipped += 1
            continue
        beta_full, _, _, _ = np.linalg.lstsq(X_full, y_target, rcond=None)
        s2_full = float(np.mean((y_target - X_full @ beta_full) ** 2))
        sigma2_full[i] = s2_full
        F_per_target[i] = float(np.log(s2_base / max(s2_full, 1e-20))) if s2_full > 0 else 0.0

    F_total = float(F_per_target.sum())
    chi2_stat = M * F_total
    if df > 0 and chi2_stat > 0:
        p_value = float(scstats.chi2.sf(chi2_stat, df=df))
    else:
        p_value = 1.0

    return {
        'F_total':         F_total,
        'chi2':            float(chi2_stat),
        'df':              df,
        'p_value':         p_value,        # POST-SELECTION-BIASED, descriptive only
        'sigma2_base':     float(sigma2_base.mean()),
        'sigma2_full':     float(sigma2_full.mean()),
        'M':               int(M),
        'n_skipped':       n_skipped,      # X3: targets where full model was over-parameterized
    }


# §2. VAR-implied covariance fit indices
#
# Per-block fit indices computed from the empirical within-block covariance
# vs. the implied stationary covariance under the W-induced VAR(1). All five
# indices (RMSEA, CFI, TLI, SRMR, NFI) are computed on every block of size
# >= 2; block type ('reflective', 'formative', 'independent') is recorded
# but does not gate which indices fire.
#
# **NAMING CAVEAT (M4):** These are NOT classical SEM RMSEA/CFI/TLI/SRMR/NFI.
# Classical SEM fit indices are computed against a CFA reflective measurement
# model (latent factor → indicators). We compute the formulas analogously,
# applied to the implied STATIONARY COVARIANCE under a fitted dynamic VAR.
# The math is parallel; the underlying generative model is different.
#
# **DEGREES-OF-FREEDOM CAVEAT (M5):** The block-level df we use is
# `n_distinct - (A_block_nnz + p_block)`, which counts only the within-block
# A entries and the p_block residual variances. It does NOT count cross-
# block A entries that influence the within-block stationary covariance
# through the Lyapunov equation. The resulting RMSEA/CFI/TLI df is
# approximate; treat the indices as comparative diagnostics, not as
# calibrated test statistics. SRMR and NFI do not depend on df and are
# always interpretable.
#
# Conventions:
#  - SRMR (Hu & Bentler 1999): standardized residual covariance. df-free.
#  - NFI (Bentler & Bonett 1980): incremental fit; (F_null - F_full)/F_null. df-free.
#  - RMSEA, CFI, TLI: require df > 0. NaN when block is over-parameterized.
#
# Aggregation: arithmetic mean across blocks where the index is finite.


def _implied_stationary_cov(A, Sigma_eps, max_iter=200, tol=1e-10):
    eigs = np.linalg.eigvals(A)
    if np.max(np.abs(eigs)) >= 1.0:
        return None
    try:
        from scipy.linalg import solve_discrete_lyapunov
        return solve_discrete_lyapunov(A, Sigma_eps)
    except ImportError:
        Q = Sigma_eps.copy()
        Ak = A.copy()
        for _ in range(max_iter):
            Q_new = Q + Ak @ Q @ Ak.T
            Ak_new = Ak @ Ak
            if np.linalg.norm(Q_new - Q) < tol * np.linalg.norm(Q):
                return Q_new
            Q, Ak = Q_new, Ak_new
        return Q


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    M, N = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N, N))
    resid_full = np.zeros((M, N))
    for i in range(N):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X = np.column_stack(cols)
        if X.shape[1] >= M:
            X = np.column_stack([y_prev[:, i], np.ones(M)])
            beta, _, _, _ = np.linalg.lstsq(X, y_target, rcond=None)
            A_full[i, i] = float(beta[0])
            resid_full[:, i] = y_target - X @ beta
            continue
        beta, _, _, _ = np.linalg.lstsq(X, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            A_full[i, j] = float(beta[1 + k])
        resid_full[:, i] = y_target - X @ beta
    Sigma_eps = (resid_full.T @ resid_full) / max(M - 1, 1)
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-12:
        Sigma_eps = Sigma_eps + (1e-10 - eigs.min()) * np.eye(N)
    return A_full, resid_full, Sigma_eps


def _block_fit_all(Sigma_emp_b, Sigma_imp_b, A_block_nnz, p_block, M):
    """Compute all five VAR-implied covariance fit indices on a single block.

    Block type does NOT determine which indices fire — all are computed.
    SRMR / NFI: always returned when block is valid (no df dependence).
    RMSEA / CFI / TLI: returned when df_block > 0 (block-level model is
                       identified by covariance alone); NaN otherwise.

    df_block = n_distinct(p_block) - (A_block_nnz + p_block).
    APPROXIMATE: does not count cross-block A entries that influence the
    within-block stationary covariance through the Lyapunov equation.
    See §2 docstring.

    Index families (with caveat on naming — see §2 header):
      - SRMR (Hu & Bentler 1999 formula): residual-based; bounded in [0,∞).
      - NFI (Bentler & Bonett 1980 formula): incremental fit; bounded in [0,1].
      - RMSEA / CFI / TLI: covariance-decomposition fit; require df > 0.
    """
    if p_block < 2 or Sigma_imp_b is None:
        return {'srmr': np.nan, 'nfi': np.nan,
                'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                'chi2': np.nan, 'df': 0, 'identified': False}

    # SRMR — residual-based, no df dependence
    se_diag = np.sqrt(np.diag(Sigma_emp_b) + 1e-20)
    diff = Sigma_emp_b - Sigma_imp_b
    standardized = diff / np.outer(se_diag, se_diag)
    n_distinct_block = p_block * (p_block + 1) // 2
    iu = np.tril_indices(p_block)
    srmr = float(np.sqrt(np.sum(standardized[iu] ** 2) / max(n_distinct_block, 1)))

    # ML discrepancies for NFI / CFI / TLI / RMSEA
    try:
        sign_imp, ld_imp = np.linalg.slogdet(Sigma_imp_b)
        sign_emp, ld_emp = np.linalg.slogdet(Sigma_emp_b)
        if sign_imp <= 0 or sign_emp <= 0:
            return {'srmr': srmr, 'nfi': np.nan,
                    'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                    'chi2': np.nan, 'df': 0, 'identified': False}
        F_full = float(max(0.0, ld_imp + np.trace(Sigma_emp_b @ np.linalg.inv(Sigma_imp_b))
                            - ld_emp - p_block))
    except np.linalg.LinAlgError:
        return {'srmr': srmr, 'nfi': np.nan,
                'rmsea': np.nan, 'cfi': np.nan, 'tli': np.nan,
                'chi2': np.nan, 'df': 0, 'identified': False}

    Sigma_null_b = np.diag(np.diag(Sigma_emp_b))
    try:
        sign_n, ld_n = np.linalg.slogdet(Sigma_null_b)
        F_null = float(max(0.0, ld_n + np.trace(Sigma_emp_b @ np.linalg.inv(Sigma_null_b))
                            - ld_emp - p_block))
    except np.linalg.LinAlgError:
        F_null = np.nan

    if np.isfinite(F_null) and F_null > 0:
        nfi = float(max(0.0, min(1.0, (F_null - F_full) / F_null)))
    else:
        nfi = np.nan

    n_params_block = A_block_nnz + p_block
    df_block = n_distinct_block - n_params_block
    df_null_block = n_distinct_block - p_block

    chi2_full = M * F_full
    chi2_null = M * F_null if np.isfinite(F_null) else np.nan

    if df_block > 0:
        rmsea = float(np.sqrt(max(0.0, (chi2_full / df_block - 1) / max(M - 1, 1))))
        if np.isfinite(chi2_null) and df_null_block > 0:
            num = max(0.0, chi2_full - df_block)
            den = max(num, chi2_null - df_null_block, 0.0)
            cfi = float(1 - num / den) if den > 0 else 1.0
            ratio_null = chi2_null / df_null_block
            ratio_full = chi2_full / df_block
            tli = float((ratio_null - ratio_full) / (ratio_null - 1)) if ratio_null != 1 else np.nan
        else:
            cfi = np.nan; tli = np.nan
        identified = True
    else:
        rmsea = np.nan; cfi = np.nan; tli = np.nan; identified = False

    return {
        'srmr': srmr, 'nfi': nfi,
        'rmsea': rmsea, 'cfi': cfi, 'tli': tli,
        'chi2': float(chi2_full), 'df': int(df_block),
        'identified': identified,
    }


def var_implied_covariance_fit(W_hat, Y, split_spec, groups,
                                support_threshold=1e-10):
    """VAR-implied covariance fit indices computed per block.

    Renamed from `structure_aware_fit` (M4) — the previous name suggested
    SEM-style CFA fit, which this is not (see §2 docstring). All five
    indices (RMSEA, CFI, TLI, SRMR, NFI) are computed on every block of
    size >= 2 regardless of block type.

    Returns dict with per-block results, aggregated mean indices, type
    counts (descriptive only), and two pass flags:
      srmr_passes : SRMR < 0.08 on every block (always evaluable)
      rmsea_passes: RMSEA < 0.08 on every RMSEA-identifiable block;
                    False if no block is identifiable
    """
    y_prev, y_curr = _build_panel_lag_pairs_simple(Y, split_spec)
    M = y_prev.shape[0]
    A_full, _, Sigma_eps = _ols_refit_panel(W_hat, y_prev, y_curr,
                                              support_threshold)
    Sigma_imp = _implied_stationary_cov(A_full, Sigma_eps)
    Yc = y_curr - y_curr.mean(axis=0)
    Sigma_emp = (Yc.T @ Yc) / max(M - 1, 1)

    block_results = []
    n_refl = 0; n_form = 0; n_indep = 0

    for g in groups:
        idx = list(g['indices'])
        p_block = len(idx)
        if p_block < 2:
            continue
        gtype = g.get('type', 'unknown')
        if gtype == 'reflective':
            n_refl += 1
        elif gtype == 'formative':
            n_form += 1
        elif gtype == 'independent':
            n_indep += 1

        Sigma_emp_b = Sigma_emp[np.ix_(idx, idx)]
        Sigma_imp_b = (Sigma_imp[np.ix_(idx, idx)]
                        if Sigma_imp is not None else None)
        A_block = A_full[np.ix_(idx, idx)]
        A_block_nnz = int(np.sum(np.abs(A_block) > 1e-12))

        r = _block_fit_all(Sigma_emp_b, Sigma_imp_b, A_block_nnz, p_block, M)
        r['block_indices'] = idx
        r['p_block'] = p_block
        r['gtype'] = gtype
        block_results.append(r)

    def _safe_mean(xs):
        xs = [x for x in xs if np.isfinite(x)]
        return float(np.mean(xs)) if xs else np.nan

    rmsea_mean = _safe_mean([r['rmsea'] for r in block_results])
    cfi_mean   = _safe_mean([r['cfi']   for r in block_results])
    tli_mean   = _safe_mean([r['tli']   for r in block_results])
    srmr_mean  = _safe_mean([r['srmr']  for r in block_results])
    nfi_mean   = _safe_mean([r['nfi']   for r in block_results])

    srmr_passes = (
        len(block_results) > 0
        and all(np.isfinite(r['srmr']) and r['srmr'] < 0.08 for r in block_results)
    )
    identifiable = [r for r in block_results
                    if r['identified'] and np.isfinite(r['rmsea'])]
    rmsea_passes = (
        len(identifiable) > 0
        and all(r['rmsea'] < 0.08 for r in identifiable)
    )

    return {
        'blocks':                block_results,
        'rmsea_mean':            rmsea_mean,
        'cfi_mean':              cfi_mean,
        'tli_mean':              tli_mean,
        'srmr_mean':             srmr_mean,
        'nfi_mean':              nfi_mean,
        'n_blocks':              len(block_results),
        'n_reflective_blocks':   n_refl,
        'n_formative_blocks':    n_form,
        'n_independent_blocks':  n_indep,
        'n_rmsea_identifiable':  len(identifiable),
        'srmr_passes':           bool(srmr_passes),
        'rmsea_passes':          bool(rmsea_passes),
        'M':                     int(M),
    }


# Backward-compatibility alias: code elsewhere may still call structure_aware_fit
structure_aware_fit = var_implied_covariance_fit


# §3. Load V-Dem panel and method outputs


def _find_vdem_method_outputs():
    candidates = [
        os.path.join(VDEM_RANDBASE_DIR, 'method_outputs.pkl'),
        os.path.join(DRIVE_DIR, 'method_outputs_long_16var.pkl'),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


vdem_path = _find_vdem_method_outputs()
if vdem_path is None:
    print("V-Dem method_outputs not found. Searched:")
    for p in [os.path.join(VDEM_RANDBASE_DIR, 'method_outputs.pkl'),
              os.path.join(DRIVE_DIR, 'method_outputs_long_16var.pkl')]:
        print(f"  {p}: {'EXISTS' if os.path.exists(p) else 'MISSING'}")
    raise FileNotFoundError("Need V-Dem method outputs to proceed.")
print(f"Loading V-Dem method outputs from: {vdem_path}")
with open(vdem_path, 'rb') as f:
    vdem_out = pickle.load(f)


VDEM_DATA_PATH = os.path.join(DRIVE_DIR, 'data', 'my_data_75_years.csv')
VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]
MIN_SERIES_LENGTH = 9
VAL_YEARS = 5

if not os.path.exists(VDEM_DATA_PATH):
    raise FileNotFoundError(f"V-Dem data not found at {VDEM_DATA_PATH}")

try:
    df_raw = pd.read_csv(VDEM_DATA_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(VDEM_DATA_PATH, encoding='latin1')

keep_cols = ['country_id', 'year'] + VDEM_VARIABLES
df_panel = (df_raw[keep_cols]
            .dropna(subset=VDEM_VARIABLES)
            .sort_values(['country_id', 'year'])
            .reset_index(drop=True))
lengths = df_panel.groupby('country_id')['year'].count()
eligible = lengths[lengths >= MIN_SERIES_LENGTH].index
df_panel = df_panel[df_panel['country_id'].isin(eligible)]
train_parts = []
for cid, g in df_panel.groupby('country_id', sort=False):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
        train_parts.append(g.iloc[:-VAL_YEARS])
    else:
        train_parts.append(g)
train_df = pd.concat(train_parts, ignore_index=True)
vdem_split_spec = (train_df.groupby('country_id', sort=False)['year']
                    .count().tolist())
Y_train = train_df[VDEM_VARIABLES].to_numpy(float)
mu = Y_train.mean(0); sd = Y_train.std(0)
sd = np.where(sd < 1e-12, 1.0, sd)
vdem_Y_norm = (Y_train - mu) / sd
N_VDEM = vdem_Y_norm.shape[1]
print(f"V-Dem panel: U={len(vdem_split_spec)}, T_total={vdem_Y_norm.shape[0]}, N={N_VDEM}")


# §4. Group identification on V-Dem
#
# Loads cov-rank-1 group identification. Handles BOTH file formats (M2):
#  - bare list:    [{...}, {...}, ...]
#  - wrapper dict: {"groups": [...], "variable_names": [...], ...}


def _load_or_recompute_groups(N, var_names):
    candidates = [
        os.path.join(VDEM_LSC_DIAG_DIR, 'groups_identified.json'),
        os.path.join(VDEM_DIAG_DIR, 'groups_identified.json'),
        os.path.join(DRIVE_DIR, 'groups_identified.json'),
    ]
    for p in candidates:
        if os.path.exists(p):
            print(f"Loading V-Dem groups from: {p}")
            with open(p) as f:
                obj = json.load(f)
            # M2: handle both wrapper-dict and bare-list formats
            if isinstance(obj, dict) and 'groups' in obj:
                groups = obj['groups']
                print(f"  (Loaded from wrapper dict; top-level keys: {list(obj.keys())})")
            elif isinstance(obj, list):
                groups = obj
                print(f"  (Loaded as bare list)")
            else:
                print(f"  WARN: unexpected JSON structure (type={type(obj).__name__}); "
                      f"falling back to single reflective block")
                continue

            # Reconstruct 'indices' from 'names' if missing
            for g in groups:
                if 'indices' not in g and 'names' in g:
                    g['indices'] = [var_names.index(n) for n in g['names']
                                     if n in var_names]
            groups = [g for g in groups
                      if all(0 <= i < N for i in g.get('indices', []))]
            print(f"  Loaded {len(groups)} groups: "
                  + ", ".join(f"{g.get('type','?')}({len(g.get('indices',[]))})" for g in groups))
            return groups
    print("No V-Dem groups_identified.json found; falling back to single reflective block.")
    return [{'indices': list(range(N)), 'type': 'reflective',
             'names': var_names}]


vdem_groups = _load_or_recompute_groups(N_VDEM, VDEM_VARIABLES)


# §5. Compute Geweke + VAR-implied covariance fit per V-Dem method


vdem_rows = []
method_outputs = vdem_out['method_outputs']
for method_name, method_data in method_outputs.items():
    if not isinstance(method_data, dict) or 'W_hat' not in method_data:
        print(f"  V-Dem {method_name}: no W_hat (errored?), skipping")
        continue
    W_hat = method_data['W_hat']

    try:
        gw = geweke_improvement(W_hat, vdem_Y_norm, vdem_split_spec)
    except Exception as e:
        print(f"  Geweke failed on V-Dem {method_name}: {e}")
        gw = {'F_total': np.nan, 'chi2': np.nan, 'df': 0, 'p_value': np.nan,
              'sigma2_base': np.nan, 'sigma2_full': np.nan, 'n_skipped': 0}

    try:
        sa = var_implied_covariance_fit(W_hat, vdem_Y_norm,
                                          vdem_split_spec, vdem_groups)
    except Exception as e:
        print(f"  VAR-implied covariance fit failed on V-Dem {method_name}: {e}")
        sa = {'rmsea_mean': np.nan, 'cfi_mean': np.nan, 'tli_mean': np.nan,
              'srmr_mean': np.nan, 'nfi_mean': np.nan,
              'n_blocks': 0, 'n_reflective_blocks': 0,
              'n_formative_blocks': 0, 'n_independent_blocks': 0,
              'n_rmsea_identifiable': 0,
              'srmr_passes': False, 'rmsea_passes': False}

    row = {
        'config': 'long_16var', 'method': method_name,
        'nnz': int(np.sum(np.abs(W_hat) > 1e-10)),
        # Geweke (M3: p-value renamed to make caveat explicit)
        'geweke_F_total':              gw.get('F_total', np.nan),
        'geweke_chi2':                 gw.get('chi2', np.nan),
        'geweke_df':                   gw.get('df', 0),
        'geweke_p_postsel_biased':     gw.get('p_value', np.nan),  # M3: explicit caveat in name
        'geweke_n_skipped':            gw.get('n_skipped', 0),     # X3
        'geweke_significant_descrip':  float(gw.get('p_value', 1.0) < 0.05) if np.isfinite(gw.get('p_value', np.nan)) else 0.0,
        # VAR-implied covariance fit (M4: renamed in docstrings; columns kept for backward compat)
        'fit_rmsea_mean':       sa.get('rmsea_mean', np.nan),
        'fit_cfi_mean':         sa.get('cfi_mean', np.nan),
        'fit_tli_mean':         sa.get('tli_mean', np.nan),
        'fit_srmr_mean':        sa.get('srmr_mean', np.nan),
        'fit_nfi_mean':         sa.get('nfi_mean', np.nan),
        'n_blocks':             int(sa.get('n_blocks', 0)),
        'n_reflective_blocks':  int(sa.get('n_reflective_blocks', 0)),
        'n_formative_blocks':   int(sa.get('n_formative_blocks', 0)),
        'n_independent_blocks': int(sa.get('n_independent_blocks', 0)),
        'n_rmsea_identifiable': int(sa.get('n_rmsea_identifiable', 0)),
        'srmr_passes':          bool(sa.get('srmr_passes', False)),
        'rmsea_passes':         bool(sa.get('rmsea_passes', False)),
    }
    vdem_rows.append(row)
    rmsea_str = f"RMSEA={row['fit_rmsea_mean']:.3f}" if np.isfinite(row['fit_rmsea_mean']) else "RMSEA=NA"
    cfi_str   = f"CFI={row['fit_cfi_mean']:.3f}"   if np.isfinite(row['fit_cfi_mean'])   else "CFI=NA"
    srmr_str  = f"SRMR={row['fit_srmr_mean']:.3f}" if np.isfinite(row['fit_srmr_mean']) else "SRMR=NA"
    nfi_str   = f"NFI={row['fit_nfi_mean']:.3f}"   if np.isfinite(row['fit_nfi_mean'])   else "NFI=NA"
    srmr_mark  = "PASS" if row['srmr_passes']  else "FAIL"
    rmsea_mark = "PASS" if row['rmsea_passes'] else ("NA" if row['n_rmsea_identifiable'] == 0 else "FAIL")
    skip_note = f" (skipped={row['geweke_n_skipped']})" if row['geweke_n_skipped'] > 0 else ""
    print(f"  V-Dem {method_name:24s}: nnz={row['nnz']:4d}  "
          f"Geweke F={row['geweke_F_total']:+.4f} (p~{row['geweke_p_postsel_biased']:.2g}){skip_note}  "
          f"fit SRMR[{srmr_mark}]/RMSEA[{rmsea_mark}] {srmr_str} {nfi_str} {rmsea_str} {cfi_str}")

if vdem_rows:
    vdem_df = pd.DataFrame(vdem_rows)
    vdem_path_out = os.path.join(OUT_DIR, 'multi_criteria_vdem.csv')
    vdem_df.to_csv(vdem_path_out, index=False)
    print(f"\nSaved V-Dem multi-criteria -> {vdem_path_out}")
else:
    raise RuntimeError("No V-Dem rows produced.")


# §6. Build the integrated 5-criterion summary


def _load_csv_if_exists(path):
    return pd.read_csv(path) if os.path.exists(path) else None


sweep_path   = os.path.join(VDEM_RANDBASE_DIR, 'sweep_raw.csv')
verdict_path = os.path.join(VDEM_RANDBASE_DIR, 'verdict.csv')
lsc_path_candidates = [
    os.path.join(VDEM_LSC_DIAG_DIR, 'lsc_scores.csv'),
    os.path.join(VDEM_DIAG_DIR, 'lsc_scores.csv'),
    os.path.join(DRIVE_DIR, 'lsc_scores.csv'),
]

sweep   = _load_csv_if_exists(sweep_path)
verdict = _load_csv_if_exists(verdict_path)
lsc_csv = None
lsc_csv_path = None
for p in lsc_path_candidates:
    if os.path.exists(p):
        lsc_csv = pd.read_csv(p)
        lsc_csv_path = p
        break
if lsc_csv is not None:
    print(f"Loaded LSC scores from: {lsc_csv_path}")
    print(f"  Columns: {list(lsc_csv.columns)}")
else:
    print("WARN: no lsc_scores.csv found; LSC criterion will be NaN")

crps_lookup = {}
if sweep is not None and 'method' in sweep.columns:
    for m in vdem_df['method'].unique():
        sub = sweep[(sweep.method == m) & (sweep.get('family', 'method') == 'method')]
        if not sub.empty:
            for col in ['CRPS_unconstrained', 'CRPS', 'crps']:
                if col in sub.columns:
                    crps_lookup[m] = float(sub.iloc[0][col])
                    break
print(f"CRPS lookup: {crps_lookup}")

rb_verdict_lookup = {}
if verdict is not None and 'method' in verdict.columns:
    for m in vdem_df['method'].unique():
        sub = verdict[verdict.method == m]
        if not sub.empty:
            n_total = len(sub)
            if 'p_emp' in sub.columns:
                n_beats = int((sub['p_emp'] <= 0.10).sum())
            elif 'beats' in sub.columns:
                n_beats = int(sub['beats'].sum())
            else:
                n_beats = 0
            rb_verdict_lookup[m] = (n_beats, n_total)
print(f"RB verdict lookup: {rb_verdict_lookup}")

# M1: LSC column lookup with fallback for both naming conventions.
# - New convention (delta_PR_diag, p_perm): direct read.
# - Old convention (PR_diagnostic, null_PR_diag_mean, p_PR_diag): construct
#   delta as (PR_diagnostic - null_PR_diag_mean), use p_PR_diag as p-value.
lsc_lookup = {}
if lsc_csv is not None and 'method' in lsc_csv.columns:
    for m in vdem_df['method'].unique():
        sub = lsc_csv[lsc_csv.method == m]
        if sub.empty:
            continue
        row_d = sub.iloc[0].to_dict()

        # delta_PR_diag (Component A excess over null)
        # Lookup tries lowercase first (newer convention), then capital-D
        # (lsc_diagnostic_vdem.py output), then constructs from PR_diagnostic.
        if 'delta_PR_diag' in row_d and np.isfinite(row_d.get('delta_PR_diag', np.nan)):
            dPR = float(row_d['delta_PR_diag'])
        elif 'Delta_PR_diag' in row_d and np.isfinite(row_d.get('Delta_PR_diag', np.nan)):
            dPR = float(row_d['Delta_PR_diag'])
        elif ('PR_diagnostic_obs' in row_d and 'null_PR_diag_mean' in row_d
              and np.isfinite(row_d['PR_diagnostic_obs']) and np.isfinite(row_d['null_PR_diag_mean'])):
            dPR = float(row_d['PR_diagnostic_obs'] - row_d['null_PR_diag_mean'])
        elif ('PR_diagnostic' in row_d and 'null_PR_diag_mean' in row_d
              and np.isfinite(row_d['PR_diagnostic']) and np.isfinite(row_d['null_PR_diag_mean'])):
            dPR = float(row_d['PR_diagnostic'] - row_d['null_PR_diag_mean'])
        else:
            dPR = np.nan

        # p-value for ΔPR
        if 'p_perm' in row_d and np.isfinite(row_d.get('p_perm', np.nan)):
            p_dPR = float(row_d['p_perm'])
        elif 'p_PR_diag' in row_d and np.isfinite(row_d.get('p_PR_diag', np.nan)):
            p_dPR = float(row_d['p_PR_diag'])
        else:
            p_dPR = np.nan

        # delta_LSC (composite excess over null)
        if 'delta_LSC' in row_d and np.isfinite(row_d.get('delta_LSC', np.nan)):
            dLSC = float(row_d['delta_LSC'])
        elif 'Delta_LSC' in row_d and np.isfinite(row_d.get('Delta_LSC', np.nan)):
            dLSC = float(row_d['Delta_LSC'])
        elif ('LSC_obs' in row_d and 'null_LSC_mean' in row_d
              and np.isfinite(row_d['LSC_obs']) and np.isfinite(row_d['null_LSC_mean'])):
            dLSC = float(row_d['LSC_obs'] - row_d['null_LSC_mean'])
        elif ('LSC' in row_d and 'null_LSC_mean' in row_d
              and np.isfinite(row_d['LSC']) and np.isfinite(row_d['null_LSC_mean'])):
            dLSC = float(row_d['LSC'] - row_d['null_LSC_mean'])
        else:
            dLSC = np.nan

        # p-value for ΔLSC
        if 'p_LSC' in row_d and np.isfinite(row_d.get('p_LSC', np.nan)):
            p_dLSC = float(row_d['p_LSC'])
        elif 'p_perm_LSC' in row_d and np.isfinite(row_d.get('p_perm_LSC', np.nan)):
            p_dLSC = float(row_d['p_perm_LSC'])
        else:
            p_dLSC = np.nan

        lsc_lookup[m] = {
            'delta_PR_diag': dPR,
            'p_perm':        p_dPR,
            'delta_LSC':     dLSC,
            'p_LSC':         p_dLSC,
        }
print(f"LSC lookup: {lsc_lookup}")


summary_rows = []
for _, r in vdem_df.iterrows():
    m = r['method']
    crps_val = crps_lookup.get(m, np.nan)
    rb = rb_verdict_lookup.get(m, (0, 0))

    # X4: parametrize rb_passes threshold via RB_PASS_FRACTION
    rb_passes = bool(rb[1] > 0 and rb[0] >= rb[1] * RB_PASS_FRACTION)

    lsc_d = lsc_lookup.get(m, {})
    lsc_dPR = lsc_d.get('delta_PR_diag', np.nan)
    lsc_p   = lsc_d.get('p_perm', np.nan)
    lsc_passes = bool(np.isfinite(lsc_dPR) and lsc_dPR > 0
                       and np.isfinite(lsc_p) and lsc_p < 0.05)

    # M3: geweke_passes is a DESCRIPTIVE indicator using a post-selection-
    # biased p-value. It is NOT a calibrated significance test. The summary
    # table reports it because it's informative as a comparative metric;
    # the column name makes the caveat explicit.
    g_p = r['geweke_p_postsel_biased']
    geweke_passes_descrip = bool(np.isfinite(g_p) and g_p < 0.05)

    srmr_passes  = bool(r['srmr_passes'])
    rmsea_passes = bool(r['rmsea_passes'])
    fit_passes = srmr_passes  # SRMR is the always-evaluable structural-fit pass

    summary_rows.append({
        'method':                       m,
        'CRPS':                         crps_val,
        'rb_beats_pairs':               f"{rb[0]}/{rb[1]}" if rb[1] > 0 else "n/a",
        'rb_threshold_fraction':        RB_PASS_FRACTION,
        'rb_passes':                    rb_passes,
        'lsc_delta_PR':                 lsc_dPR,
        'lsc_p':                        lsc_p,
        'lsc_passes':                   lsc_passes,
        'geweke_F':                     r['geweke_F_total'],
        'geweke_p_postsel_biased':      g_p,
        'geweke_passes_descriptive':    geweke_passes_descrip,
        'fit_rmsea':                    r['fit_rmsea_mean'],
        'fit_cfi':                      r['fit_cfi_mean'],
        'fit_tli':                      r['fit_tli_mean'],
        'fit_srmr':                     r['fit_srmr_mean'],
        'fit_nfi':                      r['fit_nfi_mean'],
        'n_refl_blocks':                r['n_reflective_blocks'],
        'n_form_blocks':                r['n_formative_blocks'],
        'n_indep_blocks':               r['n_independent_blocks'],
        'srmr_passes':                  srmr_passes,
        'rmsea_passes':                 rmsea_passes,
        'fit_passes':                   fit_passes,
        'n_passed':                     int(rb_passes) + int(lsc_passes) + int(geweke_passes_descrip) + int(fit_passes),
        'nnz':                          r['nnz'],
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, 'vdem_5criteria_summary.csv')
summary_df.to_csv(summary_path, index=False)

print("\n=== V-Dem 5-Criterion Summary ===")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nSaved -> {summary_path}")

print("\n=== How many criteria does each method pass? (out of 4 testable; CRPS reported descriptively) ===")
print(f"  CRPS:    descriptive (lower = better)")
print(f"  CRPS-RB: pass if >= {RB_PASS_FRACTION:.0%} of (protocol x family) pairs beat baselines")
print(f"  LSC:     pass if delta_PR > 0 and p < 0.05")
print(f"  Geweke:  DESCRIPTIVE pass (post-selection-biased p < 0.05)")
print(f"  Fit:     pass if SRMR < 0.08 on every block (always evaluable)")
print(f"           [+RMSEA] indicates RMSEA also passes on identifiable blocks")
for _, r in summary_df.iterrows():
    passed = []
    if r['rb_passes']:                  passed.append("CRPS-RB")
    if r['lsc_passes']:                 passed.append("LSC")
    if r['geweke_passes_descriptive']:  passed.append("Geweke*")
    if r['fit_passes']:                 passed.append("Fit")
    rmsea_aux = " [+RMSEA]" if r['rmsea_passes'] else ""
    print(f"  {r['method']:24s}: {len(passed)}/4 -> {', '.join(passed) if passed else '(none)'}{rmsea_aux}")
print("\n  * Geweke pass uses post-selection-biased p; treat as descriptive.")


print("\n" + "=" * 70)
print("V-DEM MULTI-CRITERIA EVALUATION COMPLETE")
print("=" * 70)
print(f"Output dir: {OUT_DIR}")
print("  multi_criteria_vdem.csv      - per V-Dem method (Geweke + VAR-implied fit)")
print("  vdem_5criteria_summary.csv   - V-Dem 5-criterion table")

### Cell §4.8 — SRMR robustness check on V-Dem-16

**Paper sections:** §5, Appendix P.

Diagnostic: is SRMR's failure on V-Dem-16 robust or fragile? Computes (A) eigenvalues of A_full per method (near-unit-root check), (B) SRMR sensitivity to support_threshold, (C) empirical Σ_emp's off-diagonal magnitude. Reads the existing pickle (no method refits).

**Inputs:** V-Dem-16 panel; `method_outputs.pkl`.

**Outputs:** Diagnostic prints.

**Runtime:** Seconds.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
except ImportError:
    DRIVE_DIR = '/tmp/fake_drive'

VDEM_RANDBASE_DIR = os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs', 'long_16var')
VDEM_LSC_DIAG_DIR = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')

# Load V-Dem panel
VDEM_DATA_PATH = os.path.join(DRIVE_DIR, 'data', 'my_data_75_years.csv')
VDEM_VARIABLES = [
    "Freedom_of_expression", "Freedom_of_association", "Suffrage",
    "Clean_elections", "Elected_officials", "Individual_liberty",
    "Judicial_constraints", "Legislative_constraints",
    "Civil_participation", "Direct_vote", "Local_government",
    "Regional_government", "Deliberative", "Equal_access",
    "Equal_distribution", "Equal_protection",
]
MIN_SERIES_LENGTH = 9; VAL_YEARS = 5

try:
    df_raw = pd.read_csv(VDEM_DATA_PATH)
except UnicodeDecodeError:
    df_raw = pd.read_csv(VDEM_DATA_PATH, encoding='latin1')
keep_cols = ['country_id', 'year'] + VDEM_VARIABLES
df_panel = (df_raw[keep_cols].dropna(subset=VDEM_VARIABLES)
            .sort_values(['country_id', 'year']).reset_index(drop=True))
lengths = df_panel.groupby('country_id')['year'].count()
df_panel = df_panel[df_panel['country_id'].isin(
    lengths[lengths >= MIN_SERIES_LENGTH].index)]
train_parts = []
for cid, g in df_panel.groupby('country_id', sort=False):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) >= MIN_SERIES_LENGTH + VAL_YEARS:
        train_parts.append(g.iloc[:-VAL_YEARS])
    else:
        train_parts.append(g)
train_df = pd.concat(train_parts, ignore_index=True)
split_spec = train_df.groupby('country_id', sort=False)['year'].count().tolist()
Y_train = train_df[VDEM_VARIABLES].to_numpy(float)
mu = Y_train.mean(0); sd = Y_train.std(0)
sd = np.where(sd < 1e-12, 1.0, sd)
Y_norm = (Y_train - mu) / sd
N = Y_norm.shape[1]
print(f"V-Dem panel: U={len(split_spec)}, T_total={Y_norm.shape[0]}, N={N}")

# Load pickle
pkl_path = os.path.join(VDEM_RANDBASE_DIR, 'method_outputs.pkl')
with open(pkl_path, 'rb') as f:
    vdem_out = pickle.load(f)
method_outputs = vdem_out['method_outputs']
print(f"Loaded {list(method_outputs.keys())}")

# Load groups
groups_path = os.path.join(VDEM_LSC_DIAG_DIR, 'groups_identified.json')
with open(groups_path) as f:
    groups_obj = json.load(f)
groups = groups_obj['groups']
print(f"Groups: {[(g['type'], len(g['indices'])) for g in groups]}")


# Helpers (copied from multi_criteria_evaluation.py for self-containment)
def _build_panel_lag_pairs_simple(Y, split_spec):
    yp, yc = [], []
    cursor = 0
    for n_u in split_spec:
        if n_u >= 2:
            unit = Y[cursor:cursor + n_u]
            yp.append(unit[:-1]); yc.append(unit[1:])
        cursor += n_u
    return np.vstack(yp), np.vstack(yc)


def _ols_refit_panel(W_hat, y_prev, y_curr, support_threshold=1e-10):
    M, N = y_curr.shape
    support = np.abs(W_hat) > support_threshold
    np.fill_diagonal(support, False)
    A_full = np.zeros((N, N))
    resid_full = np.zeros((M, N))
    for i in range(N):
        y_target = y_curr[:, i]
        sources = list(np.where(support[i])[0])
        cols = [y_prev[:, i]] + [y_prev[:, j] for j in sources] + [np.ones(M)]
        X = np.column_stack(cols)
        if X.shape[1] >= M:
            X = np.column_stack([y_prev[:, i], np.ones(M)])
        beta, _, _, _ = np.linalg.lstsq(X, y_target, rcond=None)
        A_full[i, i] = float(beta[0])
        for k, j in enumerate(sources):
            if 1 + k < len(beta) - 1:
                A_full[i, j] = float(beta[1 + k])
        resid_full[:, i] = y_target - X @ beta
    Sigma_eps = (resid_full.T @ resid_full) / max(M - 1, 1)
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-12:
        Sigma_eps = Sigma_eps + (1e-10 - eigs.min()) * np.eye(N)
    return A_full, resid_full, Sigma_eps


def _implied_stationary_cov(A, Sigma_eps):
    eigs = np.linalg.eigvals(A)
    if np.max(np.abs(eigs)) >= 1.0:
        return None
    from scipy.linalg import solve_discrete_lyapunov
    return solve_discrete_lyapunov(A, Sigma_eps)


def _srmr_block(Sigma_emp_b, Sigma_imp_b, p_block):
    if p_block < 2 or Sigma_imp_b is None:
        return np.nan
    se_diag = np.sqrt(np.diag(Sigma_emp_b) + 1e-20)
    diff = Sigma_emp_b - Sigma_imp_b
    standardized = diff / np.outer(se_diag, se_diag)
    n_distinct = p_block * (p_block + 1) // 2
    iu = np.tril_indices(p_block)
    return float(np.sqrt(np.sum(standardized[iu] ** 2) / max(n_distinct, 1)))



# Check A: Eigenvalues of A_full per method
#
# If max|eigenvalue(A)| > 0.95, the implied covariance is sensitive to small
# A perturbations. If max|eigenvalue| >= 1.0, Sigma_imp is undefined and SRMR
# returns NaN.


print("=" * 70)
print("CHECK A: max|eigenvalue(A)| per method")
print("=" * 70)
y_prev, y_curr = _build_panel_lag_pairs_simple(Y_norm, split_spec)
for m, sub in method_outputs.items():
    W = sub.get('W_hat')
    if W is None:
        continue
    A, _, Se = _ols_refit_panel(W, y_prev, y_curr)
    eigs = np.linalg.eigvals(A)
    max_abs = float(np.max(np.abs(eigs)))
    n_eigs_close_to_1 = int(np.sum(np.abs(eigs) > 0.95))
    print(f"  {m:24s}: max|eig|={max_abs:.4f}, "
          f"n_eigs_above_0.95={n_eigs_close_to_1}, "
          f"stationary={'YES' if max_abs < 1.0 else 'NO'}")
print()


# Check B: SRMR sensitivity to support_threshold
#
# Default is 1e-10. Try {1e-10, 1e-8, 1e-6, 1e-4, 1e-3} to see whether SRMR
# is stable or moves with how strictly we count edges.


print("=" * 70)
print("CHECK B: SRMR vs support_threshold (block 1, the size-13 reflective)")
print("=" * 70)
block_idx = groups[0]['indices']
p_block = len(block_idx)
M = y_prev.shape[0]
Yc = y_curr - y_curr.mean(axis=0)
Sigma_emp = (Yc.T @ Yc) / max(M - 1, 1)
Sigma_emp_b = Sigma_emp[np.ix_(block_idx, block_idx)]

print(f"\n{'Method':24s} {'thresh=1e-10':>13s} {'thresh=1e-8':>13s} "
      f"{'thresh=1e-6':>13s} {'thresh=1e-4':>13s} {'thresh=1e-3':>13s}")
for m, sub in method_outputs.items():
    W = sub.get('W_hat')
    if W is None:
        continue
    srmr_vals = []
    for thresh in [1e-10, 1e-8, 1e-6, 1e-4, 1e-3]:
        nnz_at_thresh = int(np.sum(np.abs(W) > thresh))
        A, _, Se = _ols_refit_panel(W, y_prev, y_curr, support_threshold=thresh)
        Si = _implied_stationary_cov(A, Se)
        if Si is None:
            srmr_vals.append((np.nan, nnz_at_thresh))
            continue
        Si_b = Si[np.ix_(block_idx, block_idx)]
        srmr_vals.append((_srmr_block(Sigma_emp_b, Si_b, p_block), nnz_at_thresh))
    line = f"{m:24s}"
    for v, n in srmr_vals:
        line += f"  {v:.3f} (nnz={n:3d})"
    print(line)
print()


# Check C: Empirical covariance baseline
#
# What is the typical off-diagonal magnitude in Sigma_emp on the size-13
# reflective block? If the off-diagonals are very large (close to 1), SRMR<0.08
# may be infeasible: residual misfit on a near-collinear empirical covariance
# can't easily be < 8% of the diagonal magnitudes.


print("=" * 70)
print("CHECK C: V-Dem empirical Sigma_emp characteristics on size-13 block")
print("=" * 70)
diag_b = np.diag(Sigma_emp_b)
offdiag_mask = ~np.eye(p_block, dtype=bool)
offdiag_b = Sigma_emp_b[offdiag_mask]
# Standardized off-diagonal (correlations)
se_diag = np.sqrt(diag_b)
corr_b = Sigma_emp_b / np.outer(se_diag, se_diag)
corr_offdiag = corr_b[offdiag_mask]

print(f"  Block size: p={p_block}")
print(f"  Diagonal entries (variances): mean={diag_b.mean():.4f}, "
      f"min={diag_b.min():.4f}, max={diag_b.max():.4f}")
print(f"  Off-diagonal covariances:    mean={offdiag_b.mean():.4f}, "
      f"abs_mean={np.abs(offdiag_b).mean():.4f}, max={np.abs(offdiag_b).max():.4f}")
print(f"  Off-diagonal CORRELATIONS:   mean={corr_offdiag.mean():.4f}, "
      f"abs_mean={np.abs(corr_offdiag).mean():.4f}, max={np.abs(corr_offdiag).max():.4f}")

# How small would residual covariance need to be for SRMR<0.08?
# SRMR^2 = (1/n_distinct) * sum (residual_corr_ij)^2
# For SRMR=0.08, average |residual_corr| ~= 0.08 (assuming uniform)
# i.e., residual cov ij / sqrt(s_ii*s_jj) = 0.08
# So we need residuals covering all but ~92% of empirical correlation
n_distinct = p_block * (p_block + 1) // 2
target_srmr = 0.08
target_avg_resid = target_srmr  # if uniform
print(f"\n  For SRMR < 0.08 on block of size 13:")
print(f"    Average standardized residual |s_ij - sigma_imp_ij|/sqrt(s_ii*s_jj) "
      f"must be < {target_avg_resid:.4f}")
print(f"    But empirical |corr| averages {np.abs(corr_offdiag).mean():.4f}")
print(f"    => W-implied covariance would need to capture "
      f"{(1 - target_avg_resid/np.abs(corr_offdiag).mean())*100:.1f}% of "
      f"average correlation magnitude")


# Verdict
#
# - If Check A shows max|eig|<0.95 for all methods: A is well within stationary
#   region; Sigma_imp computation is stable. ✓
# - If Check B shows SRMR varies <0.02 across thresholds: SRMR is robust to
#   how we count edges. ✓ The 0.08 cutoff failure is a real signal.
# - If Check C shows empirical correlations are very high (e.g., >0.7 average):
#   SRMR<0.08 may be near-infeasible by construction. Then the framework
#   should report SRMR descriptively rather than as a binary pass.


print("\n\n" + "=" * 70)
print("ROBUSTNESS CHECK COMPLETE")
print("=" * 70)
print("Send the output above and we'll decide whether the SRMR-fail finding")
print("is substantive or whether to drop binary fit-pass from the framework.")

### Cell §4.9 — Era diagnostic on V-Dem-16 (descriptive comparison)

**Paper section:** §5.

Descriptive comparison of V-Dem-16 pre-1970 vs post-1970 subsets: country-level coverage, per-indicator AR(1) β, sd, range, indicator correlations. Used to identify which indicators have substantially different statistical properties pre/post-1970, motivating the §4.10 era-decomposed test.

**Inputs:** V-Dem panel.

**Outputs:** Era diagnostic CSV.
  1. Country-level coverage: how many countries have which years; flag
     suspicious patterns (e.g., countries with constant values pre-1970
     suggesting imputation).
  2. Per-indicator descriptive stats by era: mean, sd, range, AR(1) β,
     median absolute year-to-year change.
  3. Pairwise indicator correlations within each era and the difference
     matrix.
  4. T_cov (rank-1 fit of indicator correlation) for each era.
  5. PR_obs and ΔPR diagnostic for each era's OLS VAR fit.

**Runtime:** ~1-2 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR = os.path.join(DRIVE_DIR, 'data')
OUT_DIR = os.path.join(DRIVE_DIR, 'era_diagnostic_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

INDICES_CSV = os.path.join(DATA_DIR, 'my_data_75_years.csv')
ERA_CUTOFF = 1970



# §1. Load and split


df = pd.read_csv(INDICES_CSV)
df = df.rename(columns={'country_id': 'country_name'})
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)

INDEX_COLS = [c for c in df.columns if c not in ['country_name', 'year']]
print(f"long_16var: {df.shape}, {df['country_name'].nunique()} countries, "
      f"years {df['year'].min()}-{df['year'].max()}")
print(f"Indices: {INDEX_COLS}")
print(f"Era cutoff: {ERA_CUTOFF}")

df_pre = df[df['year'] < ERA_CUTOFF].reset_index(drop=True)
df_post = df[df['year'] >= ERA_CUTOFF].reset_index(drop=True)
print(f"\nPre-{ERA_CUTOFF}: {df_pre.shape}, "
      f"{df_pre['country_name'].nunique()} countries, "
      f"years {df_pre['year'].min()}-{df_pre['year'].max()}")
print(f"Post-{ERA_CUTOFF}: {df_post.shape}, "
      f"{df_post['country_name'].nunique()} countries, "
      f"years {df_post['year'].min()}-{df_post['year'].max()}")



# §2. Country-level coverage and imputation flags


print("\n" + "=" * 70)
print("COUNTRY-LEVEL COVERAGE — pre-1970 subset")
print("=" * 70)

# For each country, count: years pre-1970, years post-1970, mean variability
# pre-1970 vs post-1970 (proxy for "is this real data or imputed?")
country_diagnostics = []
for country in sorted(df['country_name'].unique()):
    cdf_pre = df_pre[df_pre['country_name'] == country]
    cdf_post = df_post[df_post['country_name'] == country]
    n_pre = len(cdf_pre)
    n_post = len(cdf_post)

    # For each indicator, compute the year-to-year std within this country
    # within each era. Constant-value series will have std≈0.
    pre_yty_stds = []
    post_yty_stds = []
    for col in INDEX_COLS:
        if n_pre >= 2:
            pre_yty_stds.append(cdf_pre[col].diff().abs().mean())
        if n_post >= 2:
            post_yty_stds.append(cdf_post[col].diff().abs().mean())

    # Number of indicators that are constant within era for this country
    n_const_pre = sum(1 for s in pre_yty_stds if s < 1e-6) if pre_yty_stds else 0
    n_const_post = sum(1 for s in post_yty_stds if s < 1e-6) if post_yty_stds else 0

    country_diagnostics.append({
        'country': country,
        'n_years_pre': n_pre,
        'n_years_post': n_post,
        'mean_yty_change_pre': np.mean(pre_yty_stds) if pre_yty_stds else np.nan,
        'mean_yty_change_post': np.mean(post_yty_stds) if post_yty_stds else np.nan,
        'n_constant_indicators_pre': n_const_pre,
        'n_constant_indicators_post': n_const_post,
    })

cov_df = pd.DataFrame(country_diagnostics)

# How many countries have *any* pre-1970 data?
print(f"Countries with any pre-{ERA_CUTOFF} data: {(cov_df['n_years_pre'] > 0).sum()}")
print(f"Countries with no pre-{ERA_CUTOFF} data: {(cov_df['n_years_pre'] == 0).sum()}")

# Pre-1970 panel year distribution
print(f"\nPre-{ERA_CUTOFF} years-per-country distribution:")
print(cov_df['n_years_pre'].describe())

# Flag countries with high # of constant indicators in pre-1970 (likely imputed)
print(f"\nCountries with ≥10 (of 16) constant indicators pre-{ERA_CUTOFF} "
      f"(likely imputed/colonial-status data):")
suspicious = cov_df[cov_df['n_constant_indicators_pre'] >= 10].sort_values(
    'n_constant_indicators_pre', ascending=False)
print(suspicious[['country', 'n_years_pre', 'n_constant_indicators_pre',
                   'mean_yty_change_pre']].to_string(index=False))
print(f"\n{len(suspicious)} countries flagged as having mostly-constant "
      f"pre-{ERA_CUTOFF} data")

# Compare year-to-year variability between eras
print(f"\nMean year-to-year absolute change (averaged over indicators) per country:")
print(f"  Pre-{ERA_CUTOFF}:  median={cov_df['mean_yty_change_pre'].median():.5f}, "
      f"mean={cov_df['mean_yty_change_pre'].mean():.5f}")
print(f"  Post-{ERA_CUTOFF}: median={cov_df['mean_yty_change_post'].median():.5f}, "
      f"mean={cov_df['mean_yty_change_post'].mean():.5f}")
print(f"  Ratio (post/pre): "
      f"{cov_df['mean_yty_change_post'].median() / cov_df['mean_yty_change_pre'].median():.3f}")



# §3. Per-indicator descriptive stats by era


print("\n" + "=" * 90)
print("PER-INDICATOR STATS BY ERA")
print("=" * 90)
print(f"{'indicator':25s}  "
      f"{'pre_mean':>9s} {'post_mean':>9s} {'pre_sd':>7s} {'post_sd':>7s}  "
      f"{'pre_β':>7s} {'post_β':>7s}  {'pre_yty':>8s} {'post_yty':>8s}")
print("-" * 90)

ind_stats = {}
for col in INDEX_COLS:
    pre_vals = df_pre[col]
    post_vals = df_post[col]

    # AR(1) β pooled across countries (within-country lagged regression)
    def pooled_ar1(df_sub, col):
        bs = []
        for cn, sub in df_sub.groupby('country_name'):
            s = sub.sort_values('year')[col].values
            if len(s) >= 5 and s.std() > 1e-8:
                x = s[:-1]; y = s[1:]
                xc = x - x.mean(); yc = y - y.mean()
                denom = (xc**2).sum()
                if denom > 1e-12:
                    bs.append((xc * yc).sum() / denom)
        return np.median(bs) if bs else np.nan

    pre_beta = pooled_ar1(df_pre, col)
    post_beta = pooled_ar1(df_post, col)

    # Mean year-to-year absolute change pooled across countries
    def pooled_yty(df_sub, col):
        ds = []
        for cn, sub in df_sub.groupby('country_name'):
            s = sub.sort_values('year')[col].values
            if len(s) >= 2:
                ds.append(np.abs(np.diff(s)).mean())
        return np.mean(ds) if ds else np.nan

    pre_yty = pooled_yty(df_pre, col)
    post_yty = pooled_yty(df_post, col)

    print(f"{col:25s}  {pre_vals.mean():>9.3f} {post_vals.mean():>9.3f} "
          f"{pre_vals.std():>7.3f} {post_vals.std():>7.3f}  "
          f"{pre_beta:>7.3f} {post_beta:>7.3f}  "
          f"{pre_yty:>8.4f} {post_yty:>8.4f}")

    ind_stats[col] = {
        'pre_mean': pre_vals.mean(), 'post_mean': post_vals.mean(),
        'pre_sd': pre_vals.std(), 'post_sd': post_vals.std(),
        'pre_beta': pre_beta, 'post_beta': post_beta,
        'pre_yty': pre_yty, 'post_yty': post_yty,
    }



# §4. Cross-indicator correlation matrices and difference


print("\n" + "=" * 70)
print("CROSS-INDICATOR CORRELATIONS BY ERA")
print("=" * 70)

corr_pre = df_pre[INDEX_COLS].corr()
corr_post = df_post[INDEX_COLS].corr()
corr_diff = corr_post - corr_pre

print(f"\nPre-{ERA_CUTOFF} correlation matrix (rounded):")
print(corr_pre.round(3).to_string())

print(f"\nPost-{ERA_CUTOFF} correlation matrix (rounded):")
print(corr_post.round(3).to_string())

print(f"\nDifference (post - pre), rounded:")
print(corr_diff.round(3).to_string())

# Summary statistics on the difference
upper_tri = np.triu(np.ones(len(INDEX_COLS), dtype=bool), k=1)
diff_vals = corr_diff.values[upper_tri]
print(f"\nUpper-triangular correlation differences (post - pre):")
print(f"  Mean:   {diff_vals.mean():+.4f}")
print(f"  Median: {np.median(diff_vals):+.4f}")
print(f"  Std:    {diff_vals.std():.4f}")
print(f"  Min:    {diff_vals.min():+.4f}")
print(f"  Max:    {diff_vals.max():+.4f}")

# Largest absolute differences
print(f"\nLargest absolute correlation shifts (post-pre):")
shifts = []
for i, c1 in enumerate(INDEX_COLS):
    for j, c2 in enumerate(INDEX_COLS):
        if i < j:
            shifts.append((c1, c2, corr_diff.loc[c1, c2]))
shifts_df = pd.DataFrame(shifts, columns=['indicator_1', 'indicator_2', 'shift'])
shifts_df['abs_shift'] = shifts_df['shift'].abs()
print(shifts_df.sort_values('abs_shift', ascending=False).head(10).to_string(index=False))



# §5. T_cov and rank-one structure by era


def offdiag_rank1_fit(C, max_iters=50, tol=1e-7):
    W = C.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


# Use the size-13 cov-rank-1 block (drop Suffrage, Direct_vote, Regional_government)
SIZE13 = [c for c in INDEX_COLS
          if c not in ['Suffrage', 'Direct_vote', 'Regional_government']]

print("\n" + "=" * 70)
print("T_cov COMPARISON BY ERA")
print("=" * 70)
T_cov_pre_full16 = offdiag_rank1_fit(corr_pre.values)
T_cov_post_full16 = offdiag_rank1_fit(corr_post.values)
T_cov_pre_size13 = offdiag_rank1_fit(df_pre[SIZE13].corr().values)
T_cov_post_size13 = offdiag_rank1_fit(df_post[SIZE13].corr().values)
T_cov_full_full16 = offdiag_rank1_fit(df[INDEX_COLS].corr().values)
T_cov_full_size13 = offdiag_rank1_fit(df[SIZE13].corr().values)

print(f"T_cov on all 16 indices:")
print(f"  full panel (1950-2024): {T_cov_full_full16:.4f}")
print(f"  pre-{ERA_CUTOFF}:           {T_cov_pre_full16:.4f}")
print(f"  post-{ERA_CUTOFF}:          {T_cov_post_full16:.4f}")
print(f"\nT_cov on size-13 block (cov-rank-1 from headline analysis):")
print(f"  full panel (1950-2024): {T_cov_full_size13:.4f}")
print(f"  pre-{ERA_CUTOFF}:           {T_cov_pre_size13:.4f}")
print(f"  post-{ERA_CUTOFF}:          {T_cov_post_size13:.4f}")



# §6. Pre-1970 panel composition: who's actually in the panel?


print("\n" + "=" * 70)
print(f"PRE-{ERA_CUTOFF} PANEL COMPOSITION")
print("=" * 70)

# By region/era of independence (rough):
# This is a substantive characterization to see who pre-1970 includes.
# We don't have a region column, so just show the 89 countries with their
# pre-1970 vs post-1970 year counts to spot the pattern.

print(f"\nAll 89 countries with year coverage and constant-indicator counts:")
print(f"{'country':30s}  {'n_pre':>5s}  {'n_post':>6s}  {'const_pre':>9s}  {'const_post':>10s}  {'yty_pre':>8s}  {'yty_post':>9s}")
print("-" * 90)
for r in cov_df.sort_values(['n_constant_indicators_pre', 'country'],
                             ascending=[False, True]).itertuples():
    print(f"{r.country:30s}  {r.n_years_pre:>5d}  {r.n_years_post:>6d}  "
          f"{r.n_constant_indicators_pre:>9d}  {r.n_constant_indicators_post:>10d}  "
          f"{r.mean_yty_change_pre:>8.4f}  {r.mean_yty_change_post:>9.4f}")

# Save raw stats
output = {
    'config': {
        'era_cutoff': ERA_CUTOFF,
        'index_cols': INDEX_COLS,
        'size13_cols': SIZE13,
    },
    'panel_dimensions': {
        'pre': {'n_rows': len(df_pre), 'n_countries': df_pre['country_name'].nunique(),
                'year_range': [int(df_pre['year'].min()), int(df_pre['year'].max())]},
        'post': {'n_rows': len(df_post), 'n_countries': df_post['country_name'].nunique(),
                 'year_range': [int(df_post['year'].min()), int(df_post['year'].max())]},
    },
    'T_cov': {
        'full_16indices': {'pre': T_cov_pre_full16, 'post': T_cov_post_full16, 'full': T_cov_full_full16},
        'size13_block': {'pre': T_cov_pre_size13, 'post': T_cov_post_size13, 'full': T_cov_full_size13},
    },
    'indicator_stats': ind_stats,
    'corr_diff_stats': {
        'mean': float(diff_vals.mean()), 'median': float(np.median(diff_vals)),
        'std': float(diff_vals.std()),
        'min': float(diff_vals.min()), 'max': float(diff_vals.max()),
    },
    'top_corr_shifts': shifts_df.sort_values('abs_shift', ascending=False).head(10).to_dict('records'),
}
import json
with open(os.path.join(OUT_DIR, 'era_diagnostic_results.json'), 'w') as f:
    json.dump(output, f, indent=2, default=str)
cov_df.to_csv(os.path.join(OUT_DIR, 'country_coverage.csv'), index=False)
corr_pre.to_csv(os.path.join(OUT_DIR, 'corr_pre.csv'))
corr_post.to_csv(os.path.join(OUT_DIR, 'corr_post.csv'))
corr_diff.to_csv(os.path.join(OUT_DIR, 'corr_diff.csv'))

print(f"\nResults saved to {OUT_DIR}")

### Cell §4.10 — Era-decomposed ΔPR (regime-shift hypothesis test)

**Paper sections:** §5, Appendix P (Table 4).

Direct test of the regime-shift hypothesis: computes ΔPR separately on pre-1991 and post-1991 V-Dem-16 panels, plus the cross-era residual structure. Produces the size-13 → size-12 verdict-flip table (Table 4) when `Elected_officials` is dropped.

Hypothesis: the long_16var failure on the full 1950-2024 panel reflects pooling two distinct statistical regimes. The framework's ΔPR detects
that the pooled VAR coefficient matrix is a mixture and can't be
cleanly rank-one.

Two tests:

 - Test 1 — Era-decomposed ΔPR on size-13 block.
    Run ΔPR separately on pre-1970 (T=20) and post-1970 (T=55), each
    on the same 89 countries, on the size-13 block (already excludes
    Suffrage). Compare to full panel (T=75) where ΔPR fails.

 - Test 2 — Drop regime-shifters from full panel.
    On full 1950-2024 panel, run ΔPR on the size-13 block minus
    Elected_officials and Direct_vote (the indicators with the
    biggest regime shifts in σ or β). Test whether this fixes the
    failure.

Pipeline matches all prior blocks: OLS VAR + parametric AR(1) null at
B=1000, β-clip at 0.99.

**Inputs:** V-Dem panel.

**Outputs:** `era_decomposed_results.json`.

**Runtime:** ~5 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR = os.path.join(DRIVE_DIR, 'data')
OUT_DIR = os.path.join(DRIVE_DIR, 'era_decomposed_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

INDICES_CSV = os.path.join(DATA_DIR, 'my_data_75_years.csv')

B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99



# §1. Pipeline utilities (verbatim from prior blocks)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        A[i, :] = beta[:N]
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    bc = np.clip(betas, -beta_clip, beta_clip)
    stat_var = sigmas**2 / (1 - bc**2 + 1e-12)
    stat_sd = np.sqrt(np.maximum(stat_var, 1e-12))
    x = rng.standard_normal((U, N)) * stat_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = bc[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_metrics(Y, split_spec, ref_indices, B, seed):
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)]
    PR_obs = offdiag_rank1_fit(W_block)
    Y_block = Y[:, ref_indices]
    C_obs = np.corrcoef(Y_block.T)
    T_cov = offdiag_rank1_fit(C_obs)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    median_beta = float(np.median(betas[ref_indices]))
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    U_, T_ = len(split_spec), split_spec[0]
    for b in range(B):
        Y_sim = simulate_ar1_panel(betas, sigmas, U_, T_, rng, beta_clip=BETA_CLIP)
        ss_sim = [T_] * U_
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim = A_sim[np.ix_(ref_indices, ref_indices)]
        null_PRs[b] = offdiag_rank1_fit(W_sim)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    z = float((PR_obs - mu) / sd) if sd > 0 else float('nan')
    return {
        'T_cov': float(T_cov), 'PR_obs': float(PR_obs),
        'null_mean': mu, 'null_std': sd,
        'delta_PR': float(delta), 'p_parametric': float(p),
        'z_score': z, 'median_beta': median_beta,
        'verdict': 'PASS' if p < ALPHA else 'FAIL',
    }


def run_block(df, indicator_cols, block_label, B, seed=42):
    df_block = df[['country_name', 'year'] + indicator_cols].copy()
    for c in indicator_cols:
        df_block[c] = (df_block[c] - df_block[c].mean()) / df_block[c].std()
    countries = df_block['country_name'].unique().tolist()
    Y_l, ss = [], []
    for cn in countries:
        sub = df_block[df_block['country_name'] == cn].sort_values('year')
        Y_l.append(sub[indicator_cols].values)
        ss.append(len(sub))
    Y = np.vstack(Y_l)
    U, T = len(countries), ss[0]
    N = len(indicator_cols)
    print(f"\n=== {block_label} ===")
    print(f"  Y shape: {Y.shape}, U={U}, T={T}, N={N}")

    ref_indices = np.arange(N)
    t0 = time.time()
    m = compute_metrics(Y, ss, ref_indices, B=B, seed=seed)
    elapsed = time.time() - t0

    print(f"  Elapsed: {elapsed:.1f}s")
    print(f"  T_cov  = {m['T_cov']:.4f}")
    print(f"  PR_obs = {m['PR_obs']:.4f}")
    print(f"  ΔPR    = {m['delta_PR']:+.4f}")
    print(f"  p      = {m['p_parametric']:.4f}")
    print(f"  z      = {m['z_score']:+.2f}")
    print(f"  med β  = {m['median_beta']:.4f}")
    print(f"  verdict = {m['verdict']}")

    return {
        'block_label': block_label, 'indicator_cols': indicator_cols,
        'U': U, 'T': T, 'N': N, **m, 'elapsed_s': elapsed,
    }



# §2. Load long_16var


df = pd.read_csv(INDICES_CSV).rename(columns={'country_id': 'country_name'})
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)
print(f"long_16var: {df.shape}")

INDEX_COLS_16 = [c for c in df.columns if c not in ['country_name', 'year']]

# Size-13 block (excludes Suffrage, Direct_vote, Regional_government — the
# original cov-rank-1 singletons)
SIZE13 = [c for c in INDEX_COLS_16
          if c not in ['Suffrage', 'Direct_vote', 'Regional_government']]
print(f"Size-13 block: {SIZE13}")



# §3. Test 1 — Era-decomposed ΔPR on size-13 block


print("\n" + "=" * 70)
print("TEST 1: Era-decomposed ΔPR on size-13 block")
print("=" * 70)
print("Hypothesis: pooled 1950-2024 fails because it mixes two regimes.")
print("Each era separately should have a cleaner rank-one signal than pooled.")
print("(Both eras share 89 countries; size-13 already excludes Suffrage.)")

results_test1 = []

# Pre-1970 era
df_pre = df[df['year'] < 1970].reset_index(drop=True)
results_test1.append(run_block(
    df_pre, SIZE13, block_label='size13_pre_1970 (T=20)', B=B_PERMS
))

# Post-1970 era (already known from apples-to-apples)
df_post = df[df['year'] >= 1970].reset_index(drop=True)
results_test1.append(run_block(
    df_post, SIZE13, block_label='size13_post_1970 (T=55)', B=B_PERMS
))

# Full panel for reference
results_test1.append(run_block(
    df, SIZE13, block_label='size13_full_1950_2024 (T=75)', B=B_PERMS
))



# §4. Test 2 — Drop regime-shifters from full panel
#
# The era diagnostic flagged Elected_officials and Direct_vote as having
# substantial pre/post-1970 regime shifts:
#   Elected_officials: sd 0.47→0.36, β 0.85→0.81
#   Direct_vote:       β 0.64→0.92 (huge persistence shift)
# Direct_vote is already excluded from the size-13 block. Dropping
# Elected_officials too gives a size-12 block.
#
# We also drop Suffrage explicitly (it's already out of size-13, but for
# completeness in case we want to run on size-15 too).


print("\n" + "=" * 70)
print("TEST 2: Drop regime-shifters from full panel")
print("=" * 70)

# Test 2a: full 16 indices on full panel — establishes baseline FAIL
results_test2 = []
results_test2.append(run_block(
    df, INDEX_COLS_16, block_label='all16_full_panel (T=75)', B=B_PERMS
))

# Test 2b: size-13 minus Elected_officials (drops the σ-shifter)
SIZE12_NO_ELECT = [c for c in SIZE13 if c != 'Elected_officials']
results_test2.append(run_block(
    df, SIZE12_NO_ELECT,
    block_label='size12_no_elected_officials (T=75)', B=B_PERMS
))

# Test 2c: size-13 (the headline Suffrage-Direct_vote-Regional_government drop)
# This is our baseline already in test 1, but we re-print for table clarity:
results_test2.append({
    **results_test1[2],  # full-panel size-13
    'block_label': 'size13_full_panel (T=75) [from Test 1]',
})



# §5. Summary


print("\n" + "=" * 100)
print("RESULTS SUMMARY")
print("=" * 100)
print(f"{'block':45s} {'N':>3s} {'U':>4s} {'T':>3s} {'T_cov':>7s} {'PR_obs':>7s} "
      f"{'ΔPR':>8s} {'p':>7s} {'z':>7s} {'verdict':>8s}")
print("-" * 100)
for r in results_test1 + results_test2[:2]:  # don't double-print size-13
    print(f"{r['block_label']:45s} {r['N']:>3d} {r['U']:>4d} {r['T']:>3d} "
          f"{r['T_cov']:>7.4f} {r['PR_obs']:>7.4f} "
          f"{r['delta_PR']:>+8.4f} {r['p_parametric']:>7.4f} "
          f"{r['z_score']:>+7.2f} {r['verdict']:>8s}")

print("\nFor reference (matched-panel, prior runs):")
print(f"  size13 matched 89×55:           ΔPR=+0.160, p=0.015, z=+2.50, PASS")
print(f"  V-Dem indicators top13 matched: ΔPR=+0.233, p=0.001, z=+5.43, PASS")

print("\n" + "=" * 70)
print("INTERPRETATION GUIDE")
print("=" * 70)
print("""
Test 1 — Era-decomposed ΔPR:
  Question: does each era pass on its own, or is each era also weak?
    - If pre and post both PASS individually but full-panel FAILs: pooling
      is the problem. Regime-shift hypothesis confirmed.
    - If pre fails too: pre-1970 era has its own problems beyond pooling.
    - If post passes very strongly (much higher z than full): consistent
      with the matched-panel result. Pooling pre weakens it.

  Note: pre-1970 has T=20, which is borderline for OLS VAR with N=13.
  Lower power expected purely from sample size. Compare ΔPR magnitude,
  not just verdict.

Test 2 — Drop regime-shifters:
  Question: does removing Elected_officials (the σ-shifter) fix the
  full-panel failure?
    - If size12_no_elected_officials PASSes on full panel: we've
      identified Elected_officials as a key contributor to the failure.
    - If still FAILs: the regime shift is more diffuse, not driven
      by one or two indicators alone.
""")

# Save
output = {
    'config': {'B_PERMS': B_PERMS, 'BETA_CLIP': BETA_CLIP, 'ALPHA': ALPHA,
               'size13_cols': SIZE13},
    'test1_era_decomposed': results_test1,
    'test2_drop_regime_shifters': results_test2,
}
out_path = os.path.join(OUT_DIR, 'era_decomposed_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

### Cell §4.11 — Parametric null applied to V-Dem and synthetic regimes

**Paper sections:** §3.2, Appendix K (Table 9).

Applies the parametric AR(1) null and the row-permuting null side-by-side to four panels: V-Dem long_16var (the headline result), R3 (covariance-formative with banded innovation correlation), R3' (independent indicators), and R1 (positive control). Provides the parametric-null p-values cited throughout the paper.

**Inputs:** V-Dem panel `data/my_data_75_years.csv`, V-Dem groups `lsc_diagnostic_outputs/long_16var/groups_identified.json`.

**Outputs:** `parametric_null_applications.csv`.

**Runtime:** ~5 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
OUT_DIR   = os.path.join(DRIVE_DIR, 'parametric_null_applications_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Synthetic panel scale (matches V-Dem long_16var dimensions)
U_UNITS    = 139
T_PER_UNIT = 75


# §1. Core utilities (vectorized)


B_PERMS = 1000
ALPHA = 0.05


def fit_ols_var(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v; den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new; den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0
    yt_list, yl_list = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]
        cursor += n_u
        if n_u >= 2:
            yt_list.append(unit[1:]); yl_list.append(unit[:-1])
    yt = np.vstack(yt_list); yl = np.vstack(yl_list)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b
        sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    """Vectorized parametric AR(1) panel simulator."""
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]
    null_PRs = np.zeros(B)
    for b in range(B):
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed,
                                 U_resample, T_resample):
    """Parametric AR(1) null applied to a real panel."""
    A = fit_ols_var(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, None, None)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(
            betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim = fit_ols_var(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, betas, sigmas)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd, betas, sigmas)


print("Utilities defined.")



# §2. Inlined synthetic DGPs (R3, R3') — copied from lsc_synthetic_validation.py


def _stack_units(unit_paths):
    Y = np.vstack(unit_paths)
    split_spec = [p.shape[0] for p in unit_paths]
    return Y, split_spec


def generate_R3(seed, U=U_UNITS, T=T_PER_UNIT, rho_innov=0.5, decay=1.0):
    """R3: covariance-formative. Independent AR(1) per indicator with banded
    innovation correlation. N=12 indicators, declared as one reflective block
    (deliberately misclassified — the framework should silence on it).

    Inlined from lsc_synthetic_validation.py.
    """
    rng = np.random.default_rng(seed)
    N = 12
    rhos = rng.uniform(0.7, 0.95, size=N)
    Sigma_eps = np.eye(N)
    if rho_innov > 0:
        for i in range(N):
            for j in range(N):
                if i != j:
                    Sigma_eps[i, j] = rho_innov ** (decay * abs(i - j))
    eigs = np.linalg.eigvalsh(Sigma_eps)
    if eigs.min() < 1e-9:
        Sigma_eps = 0.99 * Sigma_eps + 0.01 * np.eye(N)
    L = np.linalg.cholesky(Sigma_eps)
    unit_paths = []
    for u in range(U):
        y = np.zeros(N)
        path = np.zeros((T, N))
        for t in range(T):
            eps = L @ rng.standard_normal(N)
            y = rhos * y + eps
            path[t] = y
        unit_paths.append(path)
    Y = np.vstack(unit_paths)
    split_spec = [T] * U
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    var_names = [f'R3_v{i}' for i in range(N)]
    return {
        'Y': Y, 'split_spec': split_spec, 'var_names': var_names,
        'ref_indices': list(range(N)),
        'regime': 'R3',
        'meta': {'rho_innov': rho_innov, 'decay': decay, 'rhos': rhos.tolist()},
    }


def generate_R3prime(seed, U=U_UNITS, T=T_PER_UNIT):
    """R3': Independent AR(1) per indicator (rho_innov=0, no cross-variable
    coupling). Sanity-check baseline. Inlined from lsc_synthetic_validation.py.
    """
    out = generate_R3(seed, U, T, rho_innov=0.0)
    out['regime'] = "R3'"
    out['var_names'] = [f"R3prime_v{i}" for i in range(out['Y'].shape[1])]
    return out



# §3. Build the three target panels


print("\n" + "=" * 70)
print("Loading target panels")
print("=" * 70)

# (1) V-Dem long_16var
csv_path = os.path.join(DATA_DIR, 'my_data_75_years.csv')
groups_path = os.path.join(LSC_DIR, 'groups_identified.json')

df_vdem = pd.read_csv(csv_path)
with open(groups_path) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
ref_block = groups_data['groups'][0]
vdem_ref_indices = ref_block['indices']

Y_vdem = df_vdem[all_var_names].values
mask = ~np.isnan(Y_vdem).any(axis=1)
Y_vdem = Y_vdem[mask]
country_col = df_vdem['country_id'][mask]
split_spec_vdem = country_col.value_counts(sort=False).reindex(
    country_col.unique()).values.tolist()
Y_vdem = (Y_vdem - Y_vdem.mean(0)) / (Y_vdem.std(0) + 1e-12)
T_vdem = split_spec_vdem[0]
U_vdem = len(split_spec_vdem)
N_vdem = Y_vdem.shape[1]

print(f"V-Dem: shape={Y_vdem.shape}, U={U_vdem}, T={T_vdem}, N={N_vdem}")
print(f"  Reflective block size: {len(vdem_ref_indices)} (indices {vdem_ref_indices})")

# (2-3) Synthetic regimes — using inlined DGPs above
syn_specs = []

print("\nGenerating R3 (seed 0)...")
out_r3 = generate_R3(seed=0)
syn_specs.append({
    'name': 'R3',
    'Y': out_r3['Y'],
    'split_spec': out_r3['split_spec'],
    'ref_indices': out_r3['ref_indices'],
})
print(f"  R3: shape={out_r3['Y'].shape}, ref_block_size={len(out_r3['ref_indices'])}")

print("\nGenerating R3' (seed 0)...")
out_r3p = generate_R3prime(seed=0)
syn_specs.append({
    'name': "R3'",
    'Y': out_r3p['Y'],
    'split_spec': out_r3p['split_spec'],
    'ref_indices': out_r3p['ref_indices'],
})
print(f"  R3': shape={out_r3p['Y'].shape}, ref_block_size={len(out_r3p['ref_indices'])}")



# §4. Run both nulls on each panel


print("\n" + "=" * 70)
print("Running BOTH nulls on each panel")
print("=" * 70)

results = []

# Panel 1: V-Dem
print(f"\n--- V-Dem long_16var ---")
A_vdem = fit_ols_var(Y_vdem, split_spec_vdem)
t0 = time.time()
d_rp, p_rp, pr_obs, mu_rp, sd_rp = compute_delta_pr_rowperm(
    A_vdem, vdem_ref_indices, B=B_PERMS, seed=11111)
print(f"  Row-perm null    : ΔPR={d_rp:+.4f}, p={p_rp:.4f}, "
      f"PR_obs={pr_obs:.4f}, null_mu={mu_rp:.4f}, null_sd={sd_rp:.4f} "
      f"({time.time()-t0:.1f}s)")
t0 = time.time()
d_pa, p_pa, _, mu_pa, sd_pa, betas, sigmas = compute_delta_pr_parametric(
    Y_vdem, split_spec_vdem, vdem_ref_indices,
    B=B_PERMS, seed=22222, U_resample=U_vdem, T_resample=T_vdem)
print(f"  Parametric AR(1) : ΔPR={d_pa:+.4f}, p={p_pa:.4f}, "
      f"null_mu={mu_pa:.4f}, null_sd={sd_pa:.4f} "
      f"({time.time()-t0:.1f}s)")
print(f"  Fitted β range: [{betas.min():.3f}, {betas.max():.3f}], "
      f"mean={betas.mean():.3f}")
results.append({
    'panel': 'V-Dem long_16var',
    'block_size': len(vdem_ref_indices),
    'PR_obs': pr_obs,
    'rowperm_delta': d_rp, 'rowperm_p': p_rp,
    'rowperm_null_mean': mu_rp, 'rowperm_null_sd': sd_rp,
    'parametric_delta': d_pa, 'parametric_p': p_pa,
    'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
    'rowperm_passes': bool(d_rp > 0 and p_rp < ALPHA),
    'parametric_passes': bool(d_pa > 0 and p_pa < ALPHA),
    'fitted_beta_min': float(betas.min()),
    'fitted_beta_max': float(betas.max()),
    'fitted_beta_mean': float(betas.mean()),
})

# Panels 2-3: synthetic regimes
for spec in syn_specs:
    name = spec['name']
    Y = spec['Y']; ss = spec['split_spec']; ref_idx = spec['ref_indices']
    U = len(ss); T = ss[0]
    print(f"\n--- {name} ---")
    A = fit_ols_var(Y, ss)
    t0 = time.time()
    d_rp, p_rp, pr_obs, mu_rp, sd_rp = compute_delta_pr_rowperm(
        A, ref_idx, B=B_PERMS, seed=33333 + hash(name) % 10000)
    print(f"  Row-perm null    : ΔPR={d_rp:+.4f}, p={p_rp:.4f}, "
          f"PR_obs={pr_obs:.4f}, null_mu={mu_rp:.4f}, null_sd={sd_rp:.4f} "
          f"({time.time()-t0:.1f}s)")
    t0 = time.time()
    d_pa, p_pa, _, mu_pa, sd_pa, betas, sigmas = compute_delta_pr_parametric(
        Y, ss, ref_idx, B=B_PERMS, seed=44444 + hash(name) % 10000,
        U_resample=U, T_resample=T)
    print(f"  Parametric AR(1) : ΔPR={d_pa:+.4f}, p={p_pa:.4f}, "
          f"null_mu={mu_pa:.4f}, null_sd={sd_pa:.4f} "
          f"({time.time()-t0:.1f}s)")
    print(f"  Fitted β range: [{betas.min():.3f}, {betas.max():.3f}], "
          f"mean={betas.mean():.3f}")
    results.append({
        'panel': name,
        'block_size': len(ref_idx),
        'PR_obs': pr_obs,
        'rowperm_delta': d_rp, 'rowperm_p': p_rp,
        'rowperm_null_mean': mu_rp, 'rowperm_null_sd': sd_rp,
        'parametric_delta': d_pa, 'parametric_p': p_pa,
        'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
        'rowperm_passes': bool(d_rp > 0 and p_rp < ALPHA),
        'parametric_passes': bool(d_pa > 0 and p_pa < ALPHA),
        'fitted_beta_min': float(betas.min()),
        'fitted_beta_max': float(betas.max()),
        'fitted_beta_mean': float(betas.mean()),
    })

df_results = pd.DataFrame(results)
csv_path_out = os.path.join(OUT_DIR, 'parametric_null_applications.csv')
df_results.to_csv(csv_path_out, index=False)
print(f"\nSaved results: {csv_path_out}")



# §5. Final comparison table


print("\n" + "=" * 70)
print("FINAL COMPARISON: row-permutation vs parametric AR(1) null")
print("=" * 70)

print(f"\n{'Panel':22s} | {'PR_obs':>7s} | "
      f"{'row-perm Δ':>10s} {'p':>7s} {'pass':>5s} | "
      f"{'param Δ':>10s} {'p':>7s} {'pass':>5s}")
print("-" * 95)
for r in results:
    rp_pass = '✓' if r['rowperm_passes'] else '✗'
    pa_pass = '✓' if r['parametric_passes'] else '✗'
    print(f"{r['panel']:22s} | {r['PR_obs']:>7.4f} | "
          f"{r['rowperm_delta']:>+10.4f} {r['rowperm_p']:>7.4f} {rp_pass:>5s} | "
          f"{r['parametric_delta']:>+10.4f} {r['parametric_p']:>7.4f} {pa_pass:>5s}")

# Interpretation
print("\n--- Interpretation ---")
for r in results:
    delta_change = r['parametric_delta'] - r['rowperm_delta']
    print(f"\n{r['panel']}:")
    print(f"  Null mean shifted from {r['rowperm_null_mean']:.4f} (row-perm) "
          f"to {r['parametric_null_mean']:.4f} (parametric); "
          f"ΔPR shifted by {delta_change:+.4f}.")
    if r['rowperm_passes'] and not r['parametric_passes']:
        print(f"  → Original ΔPR pass DOES NOT survive corrected null. "
              f"Original pass was likely null inflation.")
    elif not r['rowperm_passes'] and r['parametric_passes']:
        print(f"  → Original ΔPR fail FLIPS to pass under corrected null. "
              f"(Unusual; check fitted AR(1) parameters.)")
    elif r['rowperm_passes'] and r['parametric_passes']:
        print(f"  → ΔPR passes BOTH nulls. Robust signal.")
    else:
        print(f"  → ΔPR fails BOTH nulls. Robust non-detection.")

### Cell §4.12 — Signed-vs-unsigned ΔPR ablation

**Paper section:** Appendix M.

The framework reports ΔPR on unsigned magnitudes |W| as the primary convention because two of the five methods (cMLP, NAVAR) produce intrinsically non-negative outputs (Frobenius norm, time-averaged absolute contributions), and unifying the convention enables direct cross-method comparison on the same metric. The rank-1 prediction is invariant to per-row sign flips: if W̃* = uvᵀ then |W̃*| = |u||v|ᵀ is also rank-1. Signed and unsigned ΔPR should therefore agree up to Monte-Carlo noise.

This cell verifies that empirically. Re-evaluates ΔPR for the two pipeline methods that natively produce signed coefficients (lvg = OLS VAR + Granger thresholding; dynotears) under both signing conventions and under both null procedures. cMLP and NAVAR are excluded (non-negative by construction). PCMCI is excluded because the cached pipeline retains only `|val_matrix|`.

Method A: refit lvg from cached panels (cheap, ~5ms/panel; done inline). Method B: load DYNOTEARS from cached pickles at `lsc_synthetic_validation_outputs/W_hats/W_hat_{fs_regime}_seed{seed}_dynotears.pkl` (saved by Cell §1.1 in native signed form; unsigned counterpart is np.abs).

Per-block computation: ΔPR on each block of size ≥ 4 separately. R2/R2' have two blocks of 8 → two rows per (panel, seed, method).

**Result on V-Dem-16 (Appendix M):** signed PR = 0.368, unsigned PR = 0.374, difference < 0.01 — well within Monte-Carlo noise. Substantive verdicts unchanged.

**Inputs:** Cached panels from Cell §1.1 (`panels/`) and DYNOTEARS pickles (`W_hats/`).

**Outputs:** `signed_delta_pr_aggregated.csv` (long format: regime × seed × method × block × signing × null).

**Runtime:** ~35 minutes on Colab Pro G4 GPU. Bottleneck is the parametric AR(1) null at B=1000 per panel.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json, pickle
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
SYN_DIR   = os.path.join(DRIVE_DIR, 'lsc_synthetic_validation_outputs')
PANELS_DIR = os.path.join(SYN_DIR, 'panels')
WHATS_DIR  = os.path.join(SYN_DIR, 'W_hats')
OUT_DIR    = os.path.join(DRIVE_DIR, 'signed_delta_pr_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


B_PERMS = 1000
ALPHA   = 0.05
MIN_BLOCK_SIZE = 4
LVG_ALPHA = 0.05      # significance threshold for the per-coefficient t-test
SEEDS = [0, 1, 2, 3, 4]

# Synthetic regime list (matches lsc_synthetic_validation.py REGIMES dict order)
SYN_REGIMES = ['R1', "R1'", 'R2', "R2'", 'R3', "R3'",
               'R1-hetero', 'R1-bounded', 'R1-mixed',
               'R1-heterogen', 'R1-vdemlike']
# Note: R1-contaminated and R1-dynsplit aren't in the standard list per the
# `REGIMES` dict in lsc_synthetic_validation.py.

print(f"OUT_DIR = {OUT_DIR}")
print(f"PANELS_DIR = {PANELS_DIR}  (exists: {os.path.isdir(PANELS_DIR)})")
print(f"WHATS_DIR = {WHATS_DIR}  (exists: {os.path.isdir(WHATS_DIR)})")
print(f"B_PERMS = {B_PERMS}, LVG_ALPHA = {LVG_ALPHA}, SEEDS = {SEEDS}")



# §2. Core utilities (vectorized)


def fit_ols_var_signed(Y, split_spec):
    """Pooled OLS VAR(1), returns full SIGNED coefficient matrix.
    Diagonal is left non-zeroed to allow t-test computations downstream.
    """
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]
    N = Yt.shape[1]
    # Append intercept column
    X = np.column_stack([Ylag, np.ones(n_obs)])  # (n_obs, N+1)
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N))
    p_values = np.ones((N, N))
    for i in range(N):
        y_target = Yt[:, i]
        beta = XtX_inv @ X.T @ y_target  # (N+1,)
        resid = y_target - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]
            continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]   # keep signed coefficient values
        p_values[i, :] = p_vals[:N]
    return A, p_values


def fit_lvg_signed(Y, split_spec, alpha=LVG_ALPHA):
    """Signed lvg: pooled VAR(1) + per-coefficient t-test, threshold at p<alpha,
    SIGN PRESERVED. Diagonal is set to zero (no self-loops in W).

    This matches the convention from lsc_synthetic_validation.py's
    fit_linear_var_granger but skips the np.abs() step.
    """
    A_raw, p_values = fit_ols_var_signed(Y, split_spec)
    A_thresh = A_raw.copy()
    A_thresh[p_values >= alpha] = 0.0
    np.fill_diagonal(A_thresh, 0.0)
    return A_thresh   # signed


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    """Rank-one ALS fit on off-diagonal entries. Returns explained-variance ratio."""
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    W_zeroed = W.copy(); np.fill_diagonal(W_zeroed, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(W_zeroed, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    prev_res = np.inf
    for _ in range(max_iters):
        num_u = (W * mask) @ v; den_u = mask @ (v * v)
        u_new = np.where(den_u > 1e-12, num_u / np.maximum(den_u, 1e-12), 0.0)
        num_v = (W * mask).T @ u_new; den_v = mask.T @ (u_new * u_new)
        v_new = np.where(den_v > 1e-12, num_v / np.maximum(den_v, 1e-12), 0.0)
        u, v = u_new, v_new
        residual = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if prev_res < np.inf and abs(prev_res - residual) / max(prev_res, 1e-12) < tol:
            break
        prev_res = residual
    return max(0.0, min(1.0, 1.0 - residual / tss))


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null: per-row, permute off-diagonal column
    positions. ΔPR is PR_obs minus null mean.

    NOTE: works correctly on signed W. Permuting positions preserves the
    multiset of values per row (including signs), so signed rank-one
    structure is destroyed without changing the marginal distribution.
    """
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]
    null_PRs = np.zeros(B)
    for b in range(B):
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def fit_per_indicator_ar1(Y, split_spec):
    """Per-indicator AR(1): fit y_{i,t} = β_i y_{i,t-1} + ε on within-unit pairs."""
    cursor = 0
    yt_list, yl_list = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_list.append(unit[1:]); yl_list.append(unit[:-1])
    yt = np.vstack(yt_list); yl = np.vstack(yl_list)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    """Vectorized parametric AR(1) panel simulator from per-indicator (β,σ)."""
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed,
                                 method_fn, U_resample, T_resample):
    """Parametric AR(1) null: fit per-indicator AR(1), simulate B panels matching
    those persistences, apply method_fn to each simulated panel, score ΔPR.

    method_fn(Y_sim, split_spec) -> signed coefficient matrix (full N×N).

    IMPORTANT: PR_obs and the null PRs are computed using the same method_fn,
    so the null distribution is calibrated to whatever transformation of the
    panel that method represents. For lvg we pass fit_lvg_signed; for the
    "use existing W as observed but simulate null from refit on simulated panel"
    pattern, we pass the same fit_lvg_signed.
    """
    W_obs = method_fn(Y, split_spec)
    W_block = W_obs[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, None, None)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(
            betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        W_sim = method_fn(Y_sim, ss_sim)
        W_sim_block = W_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, betas, sigmas)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd, betas, sigmas)


def compute_delta_pr_parametric_for_W(W_obs, Y, split_spec, ref_indices,
                                       B, seed, method_fn,
                                       U_resample, T_resample):
    W_block = W_obs[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, None, None)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(
            betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        W_sim = method_fn(Y_sim, ss_sim)
        W_sim_block = W_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan, betas, sigmas)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd, betas, sigmas)


print("Utilities defined.")



# §3. Load V-Dem panel + cached synthetic panels + cached dynotears W_hats


# V-Dem (single panel)
print("Loading V-Dem...")
df_vdem = pd.read_csv(os.path.join(DATA_DIR, 'my_data_75_years.csv'))
with open(os.path.join(LSC_DIR, 'groups_identified.json')) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
vdem_groups = groups_data['groups']

Y_vdem_full = df_vdem[all_var_names].values
mask = ~np.isnan(Y_vdem_full).any(axis=1)
Y_vdem = Y_vdem_full[mask]
country_col = df_vdem['country_id'][mask]
split_spec_vdem = country_col.value_counts(sort=False).reindex(
    country_col.unique()).values.tolist()
Y_vdem = (Y_vdem - Y_vdem.mean(0)) / (Y_vdem.std(0) + 1e-12)
T_vdem = split_spec_vdem[0]
U_vdem = len(split_spec_vdem)
N_vdem = Y_vdem.shape[1]
print(f"  V-Dem: shape={Y_vdem.shape}, U={U_vdem}, T={T_vdem}, N={N_vdem}")
print(f"  V-Dem groups: {len(vdem_groups)} blocks, sizes={[len(g['indices']) for g in vdem_groups]}")

# Synthetic panels (cached)
syn_panels = {}   # (regime, seed) -> dict with Y, split_spec, var_names, groups_true
print(f"\nLoading cached synthetic panels...")
for regime_name in SYN_REGIMES:
    fs_regime = regime_name.replace("'", "prime").replace("-", "_")
    for seed in SEEDS:
        path = os.path.join(PANELS_DIR, f"panel_{fs_regime}_seed{seed}.pkl")
        if not os.path.exists(path):
            print(f"  MISSING: {path}")
            continue
        with open(path, 'rb') as f:
            syn_panels[(regime_name, seed)] = pickle.load(f)
print(f"  Loaded {len(syn_panels)} synthetic (regime, seed) panels.")

# Cached dynotears W_hats
dynotears_cached = {}   # (regime, seed) -> W_hat (signed, native)
print(f"\nLoading cached dynotears W_hats...")
for regime_name in SYN_REGIMES:
    fs_regime = regime_name.replace("'", "prime").replace("-", "_")
    for seed in SEEDS:
        path = os.path.join(WHATS_DIR, f"W_hat_{fs_regime}_seed{seed}_dynotears.pkl")
        if not os.path.exists(path):
            continue
        with open(path, 'rb') as f:
            d = pickle.load(f)
        dynotears_cached[(regime_name, seed)] = d['W_hat']
print(f"  Loaded {len(dynotears_cached)} dynotears matrices.")


# §4. Per-panel evaluation
#
# For each (panel, method) pair: extract signed W, compute |W|, run ΔPR per
# block under both nulls × both signing conventions.


def panels_iter():
    """Yield (label, Y, split_spec, blocks) tuples for V-Dem and synthetic panels.

    Each `blocks` entry is a list of dicts: {'block_idx': int, 'indices': list[int],
    'block_size': int, 'block_type': str}. We keep blocks of size >= MIN_BLOCK_SIZE.
    """
    # V-Dem (single panel, multiple blocks)
    vdem_blocks = []
    for bi, g in enumerate(vdem_groups):
        if len(g['indices']) >= MIN_BLOCK_SIZE:
            vdem_blocks.append({
                'block_idx': bi,
                'indices':   list(g['indices']),
                'block_size': len(g['indices']),
                'block_type': g.get('type', 'unknown'),
            })
    yield ('V-Dem', None, Y_vdem, split_spec_vdem, vdem_blocks)

    # Synthetic (multiple panels, possibly multiple blocks each)
    for (regime_name, seed), panel_dict in syn_panels.items():
        groups_true = panel_dict['groups_true']
        blocks = []
        for bi, g in enumerate(groups_true):
            idx = list(g['indices'])
            if len(idx) >= MIN_BLOCK_SIZE:
                blocks.append({
                    'block_idx':  bi,
                    'indices':    idx,
                    'block_size': len(idx),
                    'block_type': g.get('type', 'unknown'),
                })
        if blocks:
            yield (regime_name, seed, panel_dict['Y'],
                   panel_dict['split_spec'], blocks)


def eval_one_method_one_panel(label, seed, Y, split_spec, blocks,
                               method_name, W_signed_obs):
    rows = []
    W_unsigned_obs = np.abs(W_signed_obs)
    U_panel = len(split_spec)
    T_panel = split_spec[0]
    base_seed = (hash(method_name + label + str(seed)) & 0xFFFFFFFF)

    # The parametric null requires the ability to apply method_fn to simulated
    # panels. For lvg this is fit_lvg_signed (cheap). For dynotears, refitting
    # is prohibitively expensive (~1 min/fit), so we use a proxy: simulate
    # panels and compute the null PR distribution from OLS VAR signed (no
    # threshold) on the simulated panels.
    if method_name == 'lvg_signed':
        method_fn_signed   = lambda Y_, ss_: fit_lvg_signed(Y_, ss_)
        method_fn_unsigned = lambda Y_, ss_: np.abs(fit_lvg_signed(Y_, ss_))
    elif method_name == 'dynotears':
        method_fn_signed   = lambda Y_, ss_: fit_ols_var_signed(Y_, ss_)[0]
        method_fn_unsigned = lambda Y_, ss_: np.abs(fit_ols_var_signed(Y_, ss_)[0])
    else:
        raise ValueError(f"Unknown method_name: {method_name}")

    for blk in blocks:
        ref = blk['indices']
        # ===== ROW-PERM NULL =====
        # Signed convention
        d_rs, p_rs, pr_s, mu_rs, sd_rs = compute_delta_pr_rowperm(
            W_signed_obs, ref, B_PERMS, base_seed + 11)
        # Unsigned convention
        d_ru, p_ru, pr_u, mu_ru, sd_ru = compute_delta_pr_rowperm(
            W_unsigned_obs, ref, B_PERMS, base_seed + 12)

        # ===== PARAMETRIC AR(1) NULL =====
        # Signed convention: PR_obs is from W_signed_obs; null is from method_fn_signed
        # applied to simulated panels.
        d_ps, p_ps, _, mu_ps, sd_ps, _, _ = compute_delta_pr_parametric_for_W(
            W_signed_obs, Y, split_spec, ref, B_PERMS, base_seed + 21,
            method_fn_signed, U_panel, T_panel)
        # Unsigned convention
        d_pu, p_pu, _, mu_pu, sd_pu, _, _ = compute_delta_pr_parametric_for_W(
            W_unsigned_obs, Y, split_spec, ref, B_PERMS, base_seed + 22,
            method_fn_unsigned, U_panel, T_panel)

        for signing, null_name, delta, p, pr_obs, null_mu, null_sd in [
            ('signed',   'rowperm',    d_rs, p_rs, pr_s, mu_rs, sd_rs),
            ('unsigned', 'rowperm',    d_ru, p_ru, pr_u, mu_ru, sd_ru),
            ('signed',   'parametric', d_ps, p_ps, pr_s, mu_ps, sd_ps),
            ('unsigned', 'parametric', d_pu, p_pu, pr_u, mu_pu, sd_pu),
        ]:
            rows.append({
                'panel':              label,
                'seed':               seed if seed is not None else -1,
                'method':             method_name,
                'block_idx':          blk['block_idx'],
                'block_size':         blk['block_size'],
                'block_type':         blk['block_type'],
                'signing':            signing,
                'null':               null_name,
                'PR_obs':             pr_obs,
                'delta_PR':           delta,
                'p_value':            p,
                'null_mean':          null_mu,
                'null_sd':            null_sd,
                'passes_05':          bool(np.isfinite(delta) and delta > 0
                                            and np.isfinite(p) and p < ALPHA),
            })
    return rows



# §5. Run the evaluation


all_rows = []
t_start = time.time()
n_panels = sum(1 for _ in panels_iter())
panel_count = 0

for label, seed, Y, split_spec, blocks in panels_iter():
    panel_count += 1
    panel_label = f"{label}" + (f"_seed{seed}" if seed is not None else "")
    elapsed_min = (time.time() - t_start) / 60
    print(f"\n[{panel_count}/{n_panels}] {panel_label}: blocks={len(blocks)} "
          f"(sizes={[b['block_size'] for b in blocks]}) "
          f"[elapsed: {elapsed_min:.1f} min]")

    # ===== Method 1: lvg signed (refit) =====
    t_lvg = time.time()
    W_lvg_signed = fit_lvg_signed(Y, split_spec)
    nnz_signed = int((W_lvg_signed != 0).sum())
    print(f"  lvg refit: {time.time()-t_lvg:.2f}s, signed nnz={nnz_signed}")
    if nnz_signed >= 2:
        rows_lvg = eval_one_method_one_panel(
            label, seed, Y, split_spec, blocks, 'lvg_signed', W_lvg_signed)
        all_rows.extend(rows_lvg)
    else:
        print(f"  lvg: too few nonzero entries; skipping.")

    # ===== Method 2: dynotears (cached) =====
    if label == 'V-Dem':
        print(f"  dynotears: V-Dem matrix not in synthetic cache; skipping.")
    else:
        key = (label, seed)
        if key not in dynotears_cached:
            print(f"  dynotears: cached W_hat not found for {key}; skipping.")
        else:
            W_dyn_signed = dynotears_cached[key]
            nnz_dyn = int((W_dyn_signed != 0).sum())
            if nnz_dyn >= 2:
                rows_dyn = eval_one_method_one_panel(
                    label, seed, Y, split_spec, blocks, 'dynotears', W_dyn_signed)
                all_rows.extend(rows_dyn)
                print(f"  dynotears: nnz={nnz_dyn}")
            else:
                print(f"  dynotears: too few nonzero entries; skipping.")

    # Save incrementally so partial results survive a Colab disconnect
    pd.DataFrame(all_rows).to_csv(os.path.join(OUT_DIR, 'signed_delta_pr_results.csv'),
                                  index=False)

elapsed_total = (time.time() - t_start) / 60
print(f"\n\nDone in {elapsed_total:.1f} min. Total rows: {len(all_rows)}")



# §6. Summary tables


df = pd.DataFrame(all_rows)
print(f"\nTotal rows: {len(df)}")

# Aggregate across seeds: for each (panel, method, block_idx, signing, null),
# average delta_PR and p_value across seeds.
def panel_root(label):
    """Strip _seed suffix to get the panel name."""
    if label == 'V-Dem':
        return 'V-Dem'
    return label   # synthetic panel labels are already regime names

df['panel_root'] = df['panel']
agg = df.groupby(['panel', 'method', 'block_idx', 'signing', 'null']).agg(
    delta_PR_mean=('delta_PR', 'mean'),
    delta_PR_sd=('delta_PR', 'std'),
    p_value_mean=('p_value', 'mean'),
    PR_obs_mean=('PR_obs', 'mean'),
    null_mean_mean=('null_mean', 'mean'),
    n_seeds=('delta_PR', 'count'),
    passes_05_count=('passes_05', 'sum'),
).reset_index()

# Print V-Dem detailed summary
print("\n" + "="*85)
print("V-DEM RESULTS (single panel; one row per block × signing × null)")
print("="*85)
vdem_sub = df[df['panel'] == 'V-Dem'].sort_values(
    ['method', 'block_idx', 'signing', 'null'])
print(f"\n{'Method':12s} {'Block':6s} {'Signing':10s} {'Null':12s} "
      f"{'PR_obs':>7s} {'ΔPR':>9s} {'p':>7s} {'pass':>5s}")
print("-" * 75)
for _, r in vdem_sub.iterrows():
    pass_mark = '✓' if r['passes_05'] else '✗'
    print(f"{r['method']:12s} {r['block_idx']:6d} {r['signing']:10s} {r['null']:12s} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+9.4f} {r['p_value']:>7.4f} {pass_mark:>5s}")

# Print core synthetic regimes (R1, R2, R3, R3') aggregated across seeds
print("\n" + "="*95)
print("CORE SYNTHETIC REGIMES (aggregated across 5 seeds; mean ΔPR shown)")
print("="*95)
core_regimes = ['R1', 'R2', 'R3', "R3'"]
core_sub = agg[agg['panel'].isin(core_regimes)].sort_values(
    ['panel', 'method', 'block_idx', 'signing', 'null'])
print(f"\n{'Regime':10s} {'Method':12s} {'Block':6s} {'Signing':10s} {'Null':12s} "
      f"{'ΔPR mean':>10s} {'p mean':>9s} {'passes':>10s}")
print("-" * 95)
for _, r in core_sub.iterrows():
    pass_str = f"{r['passes_05_count']}/{r['n_seeds']}"
    print(f"{r['panel']:10s} {r['method']:12s} {r['block_idx']:6d} "
          f"{r['signing']:10s} {r['null']:12s} "
          f"{r['delta_PR_mean']:>+10.4f} {r['p_value_mean']:>9.4f} {pass_str:>10s}")

agg.to_csv(os.path.join(OUT_DIR, 'signed_delta_pr_aggregated.csv'), index=False)
print(f"\nSaved aggregated to {os.path.join(OUT_DIR, 'signed_delta_pr_aggregated.csv')}")


# §7. Headline pivot: signed vs unsigned for the four key cells
#
# For V-Dem and the four core synthetic regimes (R1, R2, R3, R3'), report
# unsigned-vs-signed ΔPR side by side under each null.


print("\n" + "="*100)
print("HEADLINE PIVOT: signed vs unsigned ΔPR (parametric null only, lvg method)")
print("="*100)

pivot_panels = ['V-Dem'] + core_regimes
pivot_rows = []
for panel_name in pivot_panels:
    sub = agg[(agg['panel'] == panel_name)
              & (agg['method'] == 'lvg_signed')
              & (agg['null'] == 'parametric')]
    for block_idx in sorted(sub['block_idx'].unique()):
        bsub = sub[sub['block_idx'] == block_idx]
        signed = bsub[bsub['signing'] == 'signed']
        unsigned = bsub[bsub['signing'] == 'unsigned']
        if signed.empty or unsigned.empty:
            continue
        s = signed.iloc[0]; u = unsigned.iloc[0]
        pivot_rows.append({
            'Panel':           panel_name,
            'Block':           int(block_idx),
            'PR_obs (signed)':   float(s['PR_obs_mean']),
            'PR_obs (unsigned)': float(u['PR_obs_mean']),
            'ΔPR (signed)':    float(s['delta_PR_mean']),
            'ΔPR (unsigned)':  float(u['delta_PR_mean']),
            'p (signed)':      float(s['p_value_mean']),
            'p (unsigned)':    float(u['p_value_mean']),
        })
pivot_df = pd.DataFrame(pivot_rows)
print("\n" + pivot_df.to_string(index=False))
pivot_df.to_csv(os.path.join(OUT_DIR, 'signed_vs_unsigned_pivot.csv'), index=False)

print(f"\n\nDone. Outputs:")
print(f"  {os.path.join(OUT_DIR, 'signed_delta_pr_results.csv')} (full long-format)")
print(f"  {os.path.join(OUT_DIR, 'signed_delta_pr_aggregated.csv')} (mean across seeds)")
print(f"  {os.path.join(OUT_DIR, 'signed_vs_unsigned_pivot.csv')} (headline 5-panel pivot)")

### Cell §4.13 — V-Dem-16 era subsets (pre-1991 / post-1991)

**Paper sections:** §5, Appendix P.

Stratifies V-Dem long_16var into pre-1991 (1950-1990, T=41 × 89 countries) and post-1991 (1991-2024, T=34 × 89 countries) eras and reruns the headline ΔPR analysis within each. Tests whether V-Dem's failure on the full panel is a cross-era averaging artifact rather than a per-era robust finding.

Era split at 1991 (Cold War's end, USSR dissolution; standard periodization in democratization literature). Methods: OLS VAR (the headline failure) + PCMCI (the headline pass). Re-standardizes per era. Block: size-13 cov-rank-1 reflective block, same as headline.

**Inputs:** V-Dem panel `data/my_data_75_years.csv`, V-Dem groups `lsc_diagnostic_outputs/long_16var/groups_identified.json`.

**Outputs:** `era_subsets_results.csv`, `era_subsets_olsvar.csv`.

**Runtime:** ~30 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
OUT_DIR   = os.path.join(DRIVE_DIR, 'era_subsets_block_5_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05
ERA_BOUNDARY = 1991       # pre-1991 < 1991, post-1991 >= 1991
PCMCI_TAU_MAX = 1
PCMCI_PC_ALPHA = 0.2
PCMCI_ALPHA_LEVEL = 0.01

print(f"B_PERMS={B_PERMS}, era boundary at {ERA_BOUNDARY}")



# §2. Core utilities (same as Blocks 1-2)


def fit_ols_var_signed(Y, split_spec):
    """Pooled OLS VAR(1), returns full SIGNED coefficient matrix and per-coef p-values."""
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N)); p_values = np.ones((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        resid = y - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]
            continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]
        p_values[i, :] = p_vals[:N]
    np.fill_diagonal(A, 0.0)
    return A, p_values


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_rowperm(W, ref_indices, B, seed):
    """Vectorized row-permuting null. Works on signed or unsigned W."""
    W_block = W[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    N_ = W.shape[0]
    off_idx = np.array([[j for j in range(N_) if j != i] for i in range(N_)])
    off_vals = W[np.arange(N_)[:, None], off_idx]
    null_PRs = np.zeros(B)
    for b in range(B):
        rand_keys = rng.random((N_, N_-1))
        perm_idx = np.argsort(rand_keys, axis=1)
        off_vals_perm = np.take_along_axis(off_vals, perm_idx, axis=1)
        W_perm = np.zeros_like(W)
        for i in range(N_):
            W_perm[i, off_idx[i]] = off_vals_perm[i]
        W_perm_block = W_perm[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_perm_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


def compute_delta_pr_parametric_olsvar(Y, split_spec, ref_indices, B, seed,
                                        U_resample, T_resample):
    """Parametric AR(1) null where the method is OLS VAR (refit on each
    simulated panel). Used for both: (a) the OLS VAR analysis directly,
    and (b) as a proxy null for pcmci (where refitting pcmci 1000 times
    is prohibitive)."""
    A, _ = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


print("Utilities defined.")



# §3. PCMCI runner (uses tigramite)


def run_pcmci(Y, split_spec, tau_max=PCMCI_TAU_MAX,
              pc_alpha=PCMCI_PC_ALPHA, alpha_level=PCMCI_ALPHA_LEVEL):
    """Returns the p<alpha_level thresholded |val_matrix| at lag>=1.
    W[target, source] convention. Diagonal zeroed. Unsigned magnitudes
    (matches the original pipeline; signed-vs-unsigned is handled in a
    separate ablation, not here)."""
    try:
        from tigramite.pcmci import PCMCI
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite import data_processing as pp
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tigramite"])
        from tigramite.pcmci import PCMCI
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite import data_processing as pp

    N = Y.shape[1]
    # Build unit-boundary mask: don't use first tau_max obs of each unit as targets
    mask_2d = np.zeros_like(Y, dtype=int)
    cursor = 0
    for n_u in split_spec:
        mask_2d[cursor:cursor + tau_max, :] = 1
        cursor += n_u

    dataframe = pp.DataFrame(Y, mask=mask_2d)
    parcorr = ParCorr(mask_type="y")
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr, verbosity=0)
    results = pcmci.run_pcmci(tau_max=tau_max, pc_alpha=pc_alpha, alpha_level=alpha_level)

    val = results["val_matrix"]
    p_mat = results["p_matrix"]
    val_abs = np.abs(val[:, :, 1:tau_max + 1])
    W_raw_T = val_abs.max(axis=-1)
    W_raw = W_raw_T.T   # (target, source)
    W_p_T = p_mat[:, :, 1:tau_max + 1].min(axis=-1)
    W_p = W_p_T.T

    W_hat = np.where(W_p < alpha_level, W_raw, 0.0)
    np.fill_diagonal(W_hat, 0.0)
    return W_hat



# §4. Load V-Dem panel and split into eras


print("\n" + "=" * 70)
print("Loading V-Dem and splitting by era")
print("=" * 70)

df_vdem = pd.read_csv(os.path.join(DATA_DIR, 'my_data_75_years.csv'))
with open(os.path.join(LSC_DIR, 'groups_identified.json')) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
ref_block = groups_data['groups'][0]
vdem_ref_indices = ref_block['indices']

print(f"V-Dem panel: {len(df_vdem)} rows, year range {df_vdem['year'].min()}-{df_vdem['year'].max()}")
print(f"Reflective block size: {len(vdem_ref_indices)}")

# Drop rows with NaN in any of the indicator columns (none expected for the
# size-13 block, but defensive)
mask = ~df_vdem[all_var_names].isna().any(axis=1)
df_vdem = df_vdem[mask].reset_index(drop=True)

# Era split
era_specs = [
    ('pre-1991', df_vdem['year'] < ERA_BOUNDARY),
    ('post-1991', df_vdem['year'] >= ERA_BOUNDARY),
]

era_panels = {}
for era_name, era_mask in era_specs:
    df_era = df_vdem[era_mask].copy()
    # Sort to country-major order (country_id, year). Critical: the row
    # order determines how Y is split into per-country segments downstream.
    df_era = df_era.sort_values(['country_id', 'year']).reset_index(drop=True)

    # (redacted)-chapter-8 pattern: derive split_spec directly from the sorted
    # dataframe via groupby(sort=False), which preserves first-occurrence
    # order of country_id. This is guaranteed to match the row order of
    # Y_era because both come from df_era (already sorted).
    split_spec_era = (
        df_era.groupby('country_id', sort=False)['year']
        .count()
        .tolist()
    )
    n_countries = len(split_spec_era)
    T_era_min = min(split_spec_era)
    T_era_max = max(split_spec_era)

    Y_era = df_era[all_var_names].values
    # Re-standardize within era. Each era's panel gets its own mean/SD.
    Y_era = (Y_era - Y_era.mean(0)) / (Y_era.std(0) + 1e-12)


    assert sum(split_spec_era) == Y_era.shape[0], (
        f"{era_name}: split_spec sum {sum(split_spec_era)} != n_obs "
        f"{Y_era.shape[0]} — alignment broken")
    if T_era_min != T_era_max:
        raise ValueError(
            f"{era_name}: per-country year counts vary "
            f"(min={T_era_min}, max={T_era_max}). The era split should "
            f"produce a balanced sub-panel since the source data is "
            f"balanced 1950-2024. Check the data and the era boundary."
        )
    T_era = T_era_max  # uniform after the check above

    era_panels[era_name] = {
        'Y': Y_era,
        'split_spec': split_spec_era,
        'U': n_countries,
        'T': T_era,
        'year_range': (int(df_era['year'].min()), int(df_era['year'].max())),
    }
    print(f"\n{era_name}: shape={Y_era.shape}, U={n_countries}, T={T_era}, "
          f"years={df_era['year'].min()}-{df_era['year'].max()}")



# §5. OLS VAR ΔPR per era × 2 nulls


print("\n" + "=" * 70)
print("OLS VAR ΔPR per era × 2 nulls (B=1000)")
print("=" * 70)

results_olsvar = []
for era_name, panel in era_panels.items():
    Y = panel['Y']; ss = panel['split_spec']; U = panel['U']; T = panel['T']
    print(f"\n--- {era_name} (U={U}, T={T}) ---")

    A, p_vals = fit_ols_var_signed(Y, ss)
    np.fill_diagonal(A, 0.0)   # already done in fit_ols_var_signed but defensive

    # Row-perm null on signed A
    t0 = time.time()
    d_rp, p_rp, pr_rp, mu_rp, sd_rp = compute_delta_pr_rowperm(
        A, vdem_ref_indices, B=B_PERMS, seed=11111 + hash(era_name) % 10000)
    print(f"  OLS VAR row-perm null   : ΔPR={d_rp:+.4f}, p={p_rp:.4f}, "
          f"PR_obs={pr_rp:.4f} ({time.time()-t0:.1f}s)")

    # Parametric null on signed A (refit OLS VAR on each simulated panel)
    t0 = time.time()
    d_pa, p_pa, pr_pa, mu_pa, sd_pa = compute_delta_pr_parametric_olsvar(
        Y, ss, vdem_ref_indices, B=B_PERMS, seed=22222 + hash(era_name) % 10000,
        U_resample=U, T_resample=T)
    print(f"  OLS VAR parametric null : ΔPR={d_pa:+.4f}, p={p_pa:.4f}, "
          f"PR_obs={pr_pa:.4f} ({time.time()-t0:.1f}s)")

    # Sanity: PR_obs MUST match between row-perm and parametric (same W).

    if abs(pr_rp - pr_pa) > 1e-9:
        raise RuntimeError(
            f"PR_obs mismatch between row-perm ({pr_rp:.6f}) and parametric "
            f"({pr_pa:.6f}) for {era_name} OLS VAR. Both nulls operate on "
            f"the same observed W, so PR_obs must be identical. Check the "
            f"signing convention is consistent across both null procedures."
        )

    # Fitted AR(1) parameters (for context)
    betas, sigmas = fit_per_indicator_ar1(Y, ss)
    print(f"  Per-indicator AR(1) β: range [{betas.min():.3f}, {betas.max():.3f}], "
          f"mean={betas.mean():.3f}")

    results_olsvar.append({
        'era': era_name, 'method': 'OLS VAR',
        'U': U, 'T': T,
        'PR_obs': pr_rp,
        'rowperm_delta': d_rp, 'rowperm_p': p_rp,
        'rowperm_null_mean': mu_rp, 'rowperm_null_sd': sd_rp,
        'parametric_delta': d_pa, 'parametric_p': p_pa,
        'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
        'rowperm_passes': bool(d_rp > 0 and p_rp < ALPHA),
        'parametric_passes': bool(d_pa > 0 and p_pa < ALPHA),
        'beta_min': float(betas.min()), 'beta_max': float(betas.max()),
        'beta_mean': float(betas.mean()),
    })

# Save partial results in case pcmci stage takes long / fails
pd.DataFrame(results_olsvar).to_csv(
    os.path.join(OUT_DIR, 'era_subsets_olsvar.csv'), index=False)
print(f"\nSaved OLS VAR results: {os.path.join(OUT_DIR, 'era_subsets_olsvar.csv')}")



# §6. PCMCI ΔPR per era
#
# - Row-perm null: uses pcmci's own coefficient distribution (fast).

print("\n" + "=" * 70)
print("PCMCI ΔPR per era × 2 nulls")
print("=" * 70)

results_pcmci = []
for era_name, panel in era_panels.items():
    Y = panel['Y']; ss = panel['split_spec']; U = panel['U']; T = panel['T']
    print(f"\n--- {era_name} (U={U}, T={T}) ---")

    # Fit pcmci on observed era panel
    t0 = time.time()
    print(f"  Fitting pcmci on observed panel...")
    W_pcmci = run_pcmci(Y, ss)
    nnz = int((W_pcmci > 0).sum())
    print(f"  pcmci fit elapsed: {time.time()-t0:.1f}s, nnz={nnz}")

    if nnz < 2:
        print(f"  pcmci returned <2 nonzero entries; skipping ΔPR analysis.")
        results_pcmci.append({
            'era': era_name, 'method': 'pcmci',
            'U': U, 'T': T,
            'PR_obs': np.nan,
            'rowperm_delta': np.nan, 'rowperm_p': np.nan,
            'parametric_delta': np.nan, 'parametric_p': np.nan,
            'rowperm_passes': False, 'parametric_passes': False,
            'note': f'pcmci fit returned only {nnz} nonzero entries',
        })
        continue

    # Row-perm null on pcmci's W
    t0 = time.time()
    d_rp, p_rp, pr_obs, mu_rp, sd_rp = compute_delta_pr_rowperm(
        W_pcmci, vdem_ref_indices, B=B_PERMS, seed=33333 + hash(era_name) % 10000)
    print(f"  pcmci row-perm null     : ΔPR={d_rp:+.4f}, p={p_rp:.4f}, "
          f"PR_obs={pr_obs:.4f} ({time.time()-t0:.1f}s)")

    # Parametric null: refit pcmci on each simulated panel.
    t0 = time.time()
    PR_obs_pcmci = offdiag_rank1_fit(W_pcmci[np.ix_(vdem_ref_indices, vdem_ref_indices)])
    betas, sigmas = fit_per_indicator_ar1(Y, ss)
    rng_p = np.random.default_rng(44444 + hash(era_name) % 10000)
    null_PRs = np.zeros(B_PERMS)
    n_failures = 0
    for b in range(B_PERMS):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U, T, rng_p)
        ss_sim = [T] * U
        try:
            W_sim = run_pcmci(Y_sim, ss_sim)
            W_sim_block = W_sim[np.ix_(vdem_ref_indices, vdem_ref_indices)].copy()
            null_PRs[b] = offdiag_rank1_fit(W_sim_block)
        except Exception as e:
            null_PRs[b] = np.nan
            n_failures += 1
        # Progress indicator every 100 sims
        if (b + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (b + 1) / elapsed
            eta = (B_PERMS - b - 1) / max(rate, 1e-6)
            print(f"    pcmci parametric: {b+1}/{B_PERMS} sims ({elapsed:.0f}s, "
                  f"~{eta:.0f}s remaining, {n_failures} failures)")
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if len(null_PRs) < 100:
        print(f"  WARNING: only {len(null_PRs)}/{B_PERMS} pcmci null fits succeeded")
    mu_pa = float(null_PRs.mean()); sd_pa = float(null_PRs.std())
    d_pa = PR_obs_pcmci - mu_pa
    p_pa = (1 + np.sum(null_PRs >= PR_obs_pcmci)) / (1 + len(null_PRs))
    print(f"  pcmci parametric null   : ΔPR={d_pa:+.4f}, p={p_pa:.4f}, "
          f"PR_obs={PR_obs_pcmci:.4f}, null_mu={mu_pa:.4f}, null_sd={sd_pa:.4f} "
          f"({time.time()-t0:.1f}s, {len(null_PRs)}/{B_PERMS} valid)")

    results_pcmci.append({
        'era': era_name, 'method': 'pcmci',
        'U': U, 'T': T,
        'PR_obs': PR_obs_pcmci,
        'rowperm_delta': d_rp, 'rowperm_p': p_rp,
        'rowperm_null_mean': mu_rp, 'rowperm_null_sd': sd_rp,
        'parametric_delta': d_pa, 'parametric_p': p_pa,
        'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
        'rowperm_passes': bool(d_rp > 0 and p_rp < ALPHA),
        'parametric_passes': bool(d_pa > 0 and p_pa < ALPHA),
        'n_pcmci_null_valid': int(len(null_PRs)),
        'n_pcmci_null_failures': int(n_failures),
    })

# Save full results
all_results = results_olsvar + results_pcmci
df_results = pd.DataFrame(all_results)
df_results.to_csv(os.path.join(OUT_DIR, 'era_subsets_results.csv'), index=False)
print(f"\nSaved combined results: {os.path.join(OUT_DIR, 'era_subsets_results.csv')}")


# §7. Headline comparison table


print("\n" + "=" * 70)
print("HEADLINE: ΔPR per era × method × null")
print("=" * 70)

print(f"\n{'Era':12s} {'Method':10s} {'PR_obs':>7s} | "
      f"{'row-perm Δ':>10s} {'p':>7s} {'pass':>5s} | "
      f"{'param Δ':>10s} {'p':>7s} {'pass':>5s}")
print("-" * 92)
for r in all_results:
    if pd.isna(r.get('PR_obs', np.nan)):
        print(f"{r['era']:12s} {r['method']:10s} {'N/A':>7s} | (failed/skipped)")
        continue
    rp_pass = '✓' if r['rowperm_passes'] else '✗'
    pa_pass = '✓' if r['parametric_passes'] else '✗'
    print(f"{r['era']:12s} {r['method']:10s} {r['PR_obs']:>7.4f} | "
          f"{r['rowperm_delta']:>+10.4f} {r['rowperm_p']:>7.4f} {rp_pass:>5s} | "
          f"{r['parametric_delta']:>+10.4f} {r['parametric_p']:>7.4f} {pa_pass:>5s}")



# §8. Comparison to full-panel results

print("\n" + "=" * 70)
print("COMPARISON to full-panel results (1950-2024)")
print("=" * 70)

print(f"\nFull-panel reference values (from Block 1 + earlier pipeline):")
print(f"  OLS VAR full V-Dem: PR_obs=0.440, row-perm Δ=+0.024 p=0.272, parametric Δ=+0.045 p=0.266 — FAILS both")
print(f"  pcmci   full V-Dem: ΔPR=+0.263, p=0.010 (under earlier null procedure) — PASSES")

print(f"\nInterpretation guidance:")
print(f"  - If OLS VAR fails in BOTH eras: full-panel failure is robust; not an averaging artifact.")
print(f"  - If OLS VAR passes in one era but fails in the other: substantive era effect.")
print(f"  - If pcmci passes in BOTH eras: the strongest method's success is era-robust.")
print(f"  - If pcmci passes in one era but fails in the other: pcmci's success is era-specific.")

print(f"\n\nDone.")

### Cell §4.14 — Era subsets PCMCI follow-up: exploratory

**Paper sections:** Not in paper.

Diagnoses the ~20% sim dropout in Cell §3.2's PCMCI parametric null. Re-runs the PCMCI parametric null at B=300 per era with full per-sim logging of (nnz, PR, status). Used during paper development to characterize the dropout distribution; the headline PCMCI era result reported in the paper uses the B=1000 run from Cell §3.2.

**Inputs:** Same as §3.2.

**Outputs:** `pcmci_diagnostic.csv`.

**Runtime:** ~10 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
OUT_DIR   = os.path.join(DRIVE_DIR, 'era_subsets_block_5_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


B_DIAG = 300              # diagnostic B (vs 1000 in Block 5)
ERA_BOUNDARY = 1991
PCMCI_TAU_MAX = 1
PCMCI_PC_ALPHA = 0.2
PCMCI_ALPHA_LEVEL = 0.01

print(f"B_DIAG={B_DIAG}, era boundary at {ERA_BOUNDARY}")



# §2. Utilities (subset of Block 5's, only what we need)


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def run_pcmci(Y, split_spec, tau_max=PCMCI_TAU_MAX,
              pc_alpha=PCMCI_PC_ALPHA, alpha_level=PCMCI_ALPHA_LEVEL):
    """Same as Block 5's runner."""
    try:
        from tigramite.pcmci import PCMCI
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite import data_processing as pp
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tigramite"])
        from tigramite.pcmci import PCMCI
        from tigramite.independence_tests.parcorr import ParCorr
        from tigramite import data_processing as pp

    N = Y.shape[1]
    mask_2d = np.zeros_like(Y, dtype=int)
    cursor = 0
    for n_u in split_spec:
        mask_2d[cursor:cursor + tau_max, :] = 1
        cursor += n_u

    dataframe = pp.DataFrame(Y, mask=mask_2d)
    parcorr = ParCorr(mask_type="y")
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr, verbosity=0)
    results = pcmci.run_pcmci(tau_max=tau_max, pc_alpha=pc_alpha, alpha_level=alpha_level)
    val = results["val_matrix"]
    p_mat = results["p_matrix"]
    val_abs = np.abs(val[:, :, 1:tau_max + 1])
    W_raw_T = val_abs.max(axis=-1)
    W_raw = W_raw_T.T
    W_p_T = p_mat[:, :, 1:tau_max + 1].min(axis=-1)
    W_p = W_p_T.T
    W_hat = np.where(W_p < alpha_level, W_raw, 0.0)
    np.fill_diagonal(W_hat, 0.0)
    return W_hat


print("Utilities defined.")



# §3. Load V-Dem and split into eras (same as Block 5)


print("\n" + "=" * 70)
print("Loading V-Dem and splitting by era")
print("=" * 70)

df_vdem = pd.read_csv(os.path.join(DATA_DIR, 'my_data_75_years.csv'))
with open(os.path.join(LSC_DIR, 'groups_identified.json')) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
ref_block = groups_data['groups'][0]
vdem_ref_indices = ref_block['indices']

mask = ~df_vdem[all_var_names].isna().any(axis=1)
df_vdem = df_vdem[mask].reset_index(drop=True)

era_specs = [
    ('pre-1991', df_vdem['year'] < ERA_BOUNDARY),
    ('post-1991', df_vdem['year'] >= ERA_BOUNDARY),
]

era_panels = {}
for era_name, era_mask in era_specs:
    df_era = df_vdem[era_mask].sort_values(['country_id', 'year']).reset_index(drop=True)
    split_spec_era = (df_era.groupby('country_id', sort=False)['year']
                            .count().tolist())
    n_countries = len(split_spec_era)
    T_era = max(split_spec_era)
    Y_era = df_era[all_var_names].values
    Y_era = (Y_era - Y_era.mean(0)) / (Y_era.std(0) + 1e-12)
    era_panels[era_name] = {'Y': Y_era, 'split_spec': split_spec_era,
                            'U': n_countries, 'T': T_era}
    print(f"  {era_name}: shape={Y_era.shape}, U={n_countries}, T={T_era}")



# §4. Per-era diagnostic


print("\n" + "=" * 70)
print(f"PCMCI null sim diagnostic (B={B_DIAG} per era, full per-sim logging)")
print("=" * 70)

diagnostic_rows = []

for era_name, panel in era_panels.items():
    Y = panel['Y']; ss = panel['split_spec']; U = panel['U']; T = panel['T']
    print(f"\n--- {era_name} (U={U}, T={T}) ---")

    # Reference: nnz of pcmci on real V-Dem panel
    W_real = run_pcmci(Y, ss)
    nnz_real = int((W_real != 0).sum())
    pr_real = offdiag_rank1_fit(W_real[np.ix_(vdem_ref_indices, vdem_ref_indices)])
    print(f"  Real V-Dem panel: nnz={nnz_real}, PR_obs={pr_real:.4f}")

    # Refit per-indicator AR(1) (use same betas/sigmas as Block 5's null)
    betas, sigmas = fit_per_indicator_ar1(Y, ss)
    rng = np.random.default_rng(44444 + hash(era_name) % 10000)

    t0 = time.time()
    for b in range(B_DIAG):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U, T, rng)
        ss_sim = [T] * U
        try:
            W_sim = run_pcmci(Y_sim, ss_sim)
            nnz_sim = int((W_sim != 0).sum())
            W_block = W_sim[np.ix_(vdem_ref_indices, vdem_ref_indices)].copy()
            nnz_block = int((W_block != 0).sum())
            pr_sim = offdiag_rank1_fit(W_block)
            status = 'ok' if np.isfinite(pr_sim) else 'nan_pr'
        except Exception as e:
            nnz_sim = -1; nnz_block = -1; pr_sim = np.nan
            status = f'exception:{type(e).__name__}'

        diagnostic_rows.append({
            'era': era_name, 'sim_idx': b,
            'nnz_full': nnz_sim, 'nnz_block': nnz_block,
            'PR_sim': pr_sim, 'status': status,
        })

        if (b + 1) % 50 == 0:
            elapsed = time.time() - t0
            print(f"    {b+1}/{B_DIAG} sims done ({elapsed:.0f}s)")

    print(f"  Total elapsed: {time.time()-t0:.0f}s")

df_diag = pd.DataFrame(diagnostic_rows)
csv_path = os.path.join(OUT_DIR, 'pcmci_diagnostic.csv')
df_diag.to_csv(csv_path, index=False)
print(f"\nSaved per-sim diagnostic: {csv_path}")



# §5. Diagnostic summary


print("\n" + "=" * 70)
print("DIAGNOSTIC SUMMARY")
print("=" * 70)

for era_name in ['pre-1991', 'post-1991']:
    sub = df_diag[df_diag['era'] == era_name]
    n_total = len(sub)

    # Status breakdown
    print(f"\n--- {era_name} ---")
    print(f"  Total sims: {n_total}")
    status_counts = sub['status'].value_counts()
    for s, c in status_counts.items():
        print(f"    status={s}: {c} ({c/n_total*100:.1f}%)")

    # nnz_full distribution
    valid = sub[sub['nnz_full'] >= 0]
    if len(valid) > 0:
        nnz_full = valid['nnz_full']
        print(f"  pcmci full-W nnz across all sims: "
              f"min={nnz_full.min()}, median={int(nnz_full.median())}, "
              f"mean={nnz_full.mean():.1f}, max={nnz_full.max()}")
        n_zero = (nnz_full == 0).sum()
        n_lt5 = (nnz_full < 5).sum()
        print(f"    sims with nnz_full == 0: {n_zero} ({n_zero/n_total*100:.1f}%)")
        print(f"    sims with nnz_full < 5:  {n_lt5} ({n_lt5/n_total*100:.1f}%)")

    # nnz_block distribution (for the reflective block specifically)
    valid_block = sub[sub['nnz_block'] >= 0]
    if len(valid_block) > 0:
        nnz_block = valid_block['nnz_block']
        print(f"  pcmci block-W nnz (reflective block, max possible {len(vdem_ref_indices)*(len(vdem_ref_indices)-1)}={len(vdem_ref_indices)*(len(vdem_ref_indices)-1)}):")
        print(f"    min={nnz_block.min()}, median={int(nnz_block.median())}, "
              f"mean={nnz_block.mean():.1f}, max={nnz_block.max()}")
        n_zero_blk = (nnz_block == 0).sum()
        n_lt2_blk = (nnz_block < 2).sum()
        print(f"    sims with nnz_block == 0: {n_zero_blk} ({n_zero_blk/n_total*100:.1f}%)")
        print(f"    sims with nnz_block < 2:  {n_lt2_blk} ({n_lt2_blk/n_total*100:.1f}%)")

    # PR distribution
    valid_pr = sub[sub['status'] == 'ok']
    print(f"  Valid sims (status=ok): {len(valid_pr)}/{n_total} ({len(valid_pr)/n_total*100:.1f}%)")
    if len(valid_pr) > 0:
        prs = valid_pr['PR_sim']
        print(f"    PR mean={prs.mean():.4f}, sd={prs.std():.4f}, "
              f"min={prs.min():.4f}, max={prs.max():.4f}")

    # Cross-tab: status vs nnz_block
    print(f"\n  Status vs nnz_block:")
    for s in sub['status'].unique():
        ss = sub[sub['status'] == s]
        if len(ss) == 0: continue
        if 'nnz_block' in ss.columns and (ss['nnz_block'] >= 0).any():
            valid_ss = ss[ss['nnz_block'] >= 0]
            if len(valid_ss) > 0:
                print(f"    status={s}: nnz_block range [{valid_ss['nnz_block'].min()}, "
                      f"{valid_ss['nnz_block'].max()}], median={int(valid_ss['nnz_block'].median())}")



# §6. Imputed-PR analysis: what if we set PR=0 for empty-W sims?


print("\n" + "=" * 70)
print("IMPUTED-PR ANALYSIS")
print("=" * 70)
print("If failures are all 'empty W -> NaN PR' from offdiag_rank1_fit, then")
print("treating empty-W sims as PR=0 (pcmci silenced, no rank-one structure")
print("detected) is a defensible imputation. Compute alternative ΔPR + p")
print("under that imputation.")

# Real V-Dem PR_obs from Block 5 results (re-using here)
PR_OBS_VDEM = {'pre-1991': 0.6940, 'post-1991': 0.4994}

for era_name in ['pre-1991', 'post-1991']:
    sub = df_diag[df_diag['era'] == era_name]
    pr_obs = PR_OBS_VDEM[era_name]

    print(f"\n--- {era_name} (PR_obs={pr_obs:.4f}) ---")

    # Method 1: drop NaN sims (current Block 5 behavior)
    valid_pr = sub[sub['status'] == 'ok']['PR_sim']
    if len(valid_pr) > 0:
        delta_drop = pr_obs - valid_pr.mean()
        p_drop = (1 + (valid_pr >= pr_obs).sum()) / (1 + len(valid_pr))
        print(f"  drop-NaN (current): mu={valid_pr.mean():.4f}, sd={valid_pr.std():.4f}, "
              f"ΔPR={delta_drop:+.4f}, p={p_drop:.4f}, n_valid={len(valid_pr)}")

    # Method 2: impute PR=0 for nan_pr status
    pr_imputed = sub.apply(
        lambda r: r['PR_sim'] if r['status'] == 'ok'
                  else (0.0 if r['status'] == 'nan_pr' else np.nan),
        axis=1
    )
    pr_imputed = pr_imputed.dropna()  # only drop true exceptions
    if len(pr_imputed) > 0:
        delta_imp = pr_obs - pr_imputed.mean()
        p_imp = (1 + (pr_imputed >= pr_obs).sum()) / (1 + len(pr_imputed))
        print(f"  impute PR=0 :       mu={pr_imputed.mean():.4f}, "
              f"sd={pr_imputed.std():.4f}, "
              f"ΔPR={delta_imp:+.4f}, p={p_imp:.4f}, n_used={len(pr_imputed)}")

    # Method 3: impute pr_imputed but keep at least 2 nonzero block entries
    # (i.e., empty W -> PR=0 only when block has < 2 entries; otherwise keep
    # original PR even if NaN... actually NaN means rank-1 ALS couldn't fit,
    # which only happens when block sum-of-squares is zero, which only happens
    # when block has 0 or 1 nonzero entries)

print(f"\nDone. CSV saved to {csv_path}")

### Cell §4.15 — Rank-1-regularization adversary main sweep

**Paper sections:** §6.4, Appendix P (main adversary sweep).

Defensive demonstration: the multi-criteria framework catches a method
that games ΔPR via rank-one regularization, because such gaming
necessarily costs predictive accuracy (CRPS / rb-beats).


Calibrated rank-1 adversary (`olsvar_rank1_regularized`): minimizes prediction error subject to a rank-1 penalty on the within-block coefficient matrix, with λ_eff = λ · trace(XᵀX)/N (panel-size-normalized). Sweeps λ ∈ {0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0}. Tests whether an adversary can game ΔPR by artificially biasing the coefficient matrix toward rank-1 structure.

The defense-Pareto follow-up using the corrected tvNAR-based CRPS protocol is in §7.1 of this notebook.

**Inputs:**
- V-Dem panel: /content/drive/MyDrive/NeurIPS2026_1296/data/my_data_75_years.csv
-  V-Dem groups: /content/drive/MyDrive/NeurIPS2026_1296/lsc_diagnostic_outputs/long_16var/groups_identified.json

**Outputs:** `adversary_sweep_aggregate.csv`, `all_evaluations.csv`.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [1]:
# # Block 8: Rank-1 Regularized OLS VAR Adversary


from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
OUT_DIR   = os.path.join(DRIVE_DIR, 'adversary_rank1_block_8_outputs')
os.makedirs(OUT_DIR, exist_ok=True)



# §1. Configuration


# λ values: based on local pre-test, λ in [0, 0.01, 0.05, 0.1, 0.25, 0.5,
# 1, 2, 5, 20] gives full coverage from OLS VAR (PR_obs ≈ 0.4) to
# nearly-perfect rank-one (PR_obs ≈ 1.0). λ is normalized by trace(XtX)/N
# inside the fit so it has interpretation independent of panel size.
LAMBDA_GRID = [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0]
B_PERMS = 1000
B_RB = 200              # rb-beats: number of random-matrix draws (slightly higher than B=100 for tighter rb-pass calibration)
ALPHA = 0.05

# Adversary fit parameters
ADV_MAX_ITERS = 200
ADV_TOL = 1e-7

print(f"λ grid: {LAMBDA_GRID}")
print(f"Substrate: V-Dem only (R3' dropped — see header docstring for rationale)")



# §3. Core utilities (shared)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    # np.fill_diagonal(A, 0.0)
    return A, Yt, Ylag


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


print("Utilities defined.")



# §4. The adversary: rank-1-regularized OLS VAR
#
# Solve:
#     min_A,u,v  ||Y_t - A Y_{t-1}||²_F  +  λ · ||A[ref,ref] - u v^T||²_F
#
# Off-diagonal of A[ref,ref] only (we keep A_diag at zero, matching OLS
# VAR convention). Joint optimization via alternating updates:
#   Step 1: with u,v fixed, A has closed-form solution (ridge-like with
#           shifted target on the reflective block).
#   Step 2: with A fixed, u,v are the rank-1 ALS update on A[ref,ref].
# Iterate to convergence.


def fit_olsvar_rank1_regularized(Y, split_spec, ref_indices, lam,
                                  max_iters=ADV_MAX_ITERS, tol=ADV_TOL,
                                  verbose=False):
    """Adversary: OLS VAR with rank-1 regularization on the reflective block.

    The objective is:
        min_A,u,v ||Y_t - A Y_{t-1}||²_F + λ_eff · ||A[ref,ref] - u v^T||²_F

    where λ_eff = λ * trace(XtX)/N is panel-size-normalized so λ has
    interpretation independent of n_obs. Without this normalization, the
    relative weight of the regularizer vs the prediction loss depends
    on panel size in a way that obscures λ's interpretation.

    Returns: A (full coefficient matrix), u, v (rank-1 factors of
    A[ref,ref]), n_iters, final_loss.
    """
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs, N = Yt.shape

    XtX = Ylag.T @ Ylag        # (N, N)
    XtY = Ylag.T @ Yt          # (N, N)

    ref = np.array(ref_indices, dtype=int)
    n_ref = len(ref)
    off_mask = ~np.eye(n_ref, dtype=bool)

    # Normalize λ by trace(XtX)/N for panel-size-invariant interpretation
    XtX_scale = float(np.trace(XtX) / N)
    lam_eff = lam * XtX_scale

    # Init: ordinary OLS VAR
    try:
        A_T = np.linalg.solve(XtX, XtY)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(XtX) @ XtY
    A = A_T.T
    np.fill_diagonal(A, 0.0)

    if lam == 0:
        # Pure OLS VAR; recover u, v as rank-1 ALS fit of A[ref,ref] for reporting
        Aref = A[np.ix_(ref, ref)].copy()
        np.fill_diagonal(Aref, 0.0)
        try:
            U_, S_, Vt_ = np.linalg.svd(Aref, full_matrices=False)
            u_init = U_[:, 0] * np.sqrt(S_[0])
            v_init = Vt_[0, :] * np.sqrt(S_[0])
        except np.linalg.LinAlgError:
            u_init = np.zeros(n_ref); v_init = np.zeros(n_ref)
        return A, u_init, v_init, 0, float('nan')

    # Init u, v from initial A[ref,ref]
    Aref = A[np.ix_(ref, ref)].copy()
    np.fill_diagonal(Aref, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Aref, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        u = np.zeros(n_ref); v = np.zeros(n_ref)

    prev_loss = np.inf
    for it in range(max_iters):
        # ---- Step 1: update A given (u, v) ----
        new_A = np.zeros_like(A)
        for j in range(N):
            if j not in ref:
                # No regularizer for non-reflective rows
                try:
                    a_j = np.linalg.solve(XtX, XtY[:, j])
                except np.linalg.LinAlgError:
                    a_j = np.linalg.pinv(XtX) @ XtY[:, j]
            else:
                # j is in ref; find position of j in ref
                j_pos = np.where(ref == j)[0][0]
                target_full = np.zeros(N)
                reg_diag = np.zeros(N)
                for k in range(n_ref):
                    if k == j_pos: continue
                    target_full[ref[k]] = u[j_pos] * v[k]
                    reg_diag[ref[k]] = lam_eff
                M = XtX + np.diag(reg_diag)
                b = XtY[:, j] + reg_diag * target_full
                try:
                    a_j = np.linalg.solve(M, b)
                except np.linalg.LinAlgError:
                    a_j = np.linalg.pinv(M) @ b
            new_A[j, :] = a_j
        np.fill_diagonal(new_A, 0.0)

        # ---- Step 2: update u, v given A ----
        Aref = new_A[np.ix_(ref, ref)].copy()
        np.fill_diagonal(Aref, 0.0)
        for _ in range(20):
            nu_ = (Aref * off_mask) @ v
            du_ = off_mask @ (v * v)
            u_new = np.where(du_ > 1e-12, nu_ / np.maximum(du_, 1e-12), 0.0)
            nv_ = (Aref * off_mask).T @ u_new
            dv_ = off_mask.T @ (u_new * u_new)
            v_new = np.where(dv_ > 1e-12, nv_ / np.maximum(dv_, 1e-12), 0.0)
            u, v = u_new, v_new

        # Total loss (using lam_eff in the objective)
        resid = Yt - Ylag @ new_A.T
        pred_loss = float(np.sum(resid ** 2))
        Aref_post = new_A[np.ix_(ref, ref)].copy()
        np.fill_diagonal(Aref_post, 0.0)
        reg_loss_off = float(np.sum(((Aref_post - np.outer(u, v)) * off_mask) ** 2))
        total_loss = pred_loss + lam_eff * reg_loss_off

        A = new_A
        if abs(prev_loss - total_loss) / max(abs(prev_loss), 1.0) < tol:
            if verbose:
                print(f"    converged at iter {it}, loss={total_loss:.2f}, lam_eff={lam_eff:.2f}")
            return A, u, v, it + 1, total_loss
        prev_loss = total_loss

    if verbose:
        print(f"    max_iters reached ({max_iters}), loss={total_loss:.2f}, lam_eff={lam_eff:.2f}")
    return A, u, v, max_iters, total_loss



# §5. Predictive evaluation: CRPS and rb-beats
#
# CRPS: per-indicator one-step-ahead predictive score, averaged over
# all (unit, time) pairs and indicators. We use the Gaussian approximation
# CRPS(F, y) = σ * (1/√π - 2 φ((y-μ)/σ) - (y-μ)/σ * (2 Φ((y-μ)/σ) - 1))
# where (μ, σ) come from the residual distribution.
#
# rb-beats: random-matrix baseline. Generate B_RB random A matrices with
# the same density as the candidate A, compute their CRPS, and check
# whether candidate's CRPS is lower than the median of the random-matrix
# CRPS distribution. The candidate "beats random" if it does.


def predict_one_step(Y, split_spec, A):
    """One-step-ahead predictions: ŷ_{u,t} = A · y_{u,t-1}.
    Returns Yt (targets) and Yhat (predictions), aligned."""
    Yt_l, Yhat_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:])
            Yhat_l.append(unit[:-1] @ A.T)
    Yt = np.vstack(Yt_l); Yhat = np.vstack(Yhat_l)
    return Yt, Yhat


def crps_gaussian(y_true, y_pred, sigma):
    """Closed-form CRPS for Gaussian predictive distribution.
    sigma is per-indicator predictive std (computed from training residuals)."""
    z = (y_true - y_pred) / np.maximum(sigma, 1e-9)
    pdf = scstats.norm.pdf(z); cdf = scstats.norm.cdf(z)
    crps = sigma * (z * (2 * cdf - 1) + 2 * pdf - 1.0 / np.sqrt(np.pi))
    return crps  # same shape as y_true


def compute_crps(Y, split_spec, A):
    """Average CRPS over all (unit, time, indicator) entries."""
    Yt, Yhat = predict_one_step(Y, split_spec, A)
    resid = Yt - Yhat
    # Per-indicator predictive sigma (use residual std from this fit)
    sigma_per_indicator = resid.std(axis=0, ddof=1)  # (N,)
    sigma_full = np.tile(sigma_per_indicator, (Yt.shape[0], 1))
    crps = crps_gaussian(Yt, Yhat, sigma_full)
    return float(crps.mean())


def density_matched_random_A(A_template, rng):
    """Generate random A with same nonzero-density and entry-magnitude
    distribution as A_template. Diagonal zeroed."""
    N = A_template.shape[0]
    A_off = A_template.copy()
    np.fill_diagonal(A_off, 0)
    # Permute the off-diagonal entries
    off_idx = np.where(~np.eye(N, dtype=bool))
    off_vals = A_off[off_idx]
    perm = rng.permutation(len(off_vals))
    A_rand = np.zeros_like(A_template)
    A_rand[off_idx] = off_vals[perm]
    return A_rand


def rb_beats(Y, split_spec, A_candidate, B=B_RB, seed=0):
    """Returns (rb_pass, candidate_crps, random_crps_median).
    Candidate "beats random" if its CRPS is lower than the median of B
    density-matched random-matrix CRPS draws."""
    cand_crps = compute_crps(Y, split_spec, A_candidate)
    rng = np.random.default_rng(seed)
    rand_crps_l = []
    for _ in range(B):
        A_r = density_matched_random_A(A_candidate, rng)
        rand_crps_l.append(compute_crps(Y, split_spec, A_r))
    rand_med = float(np.median(rand_crps_l))
    return (cand_crps < rand_med, cand_crps, rand_med)



# §6. ΔPR under parametric null (same as Block 1, 5, 7, 3)


def compute_delta_pr_parametric_with_A(A_obs, Y, split_spec, ref_indices,
                                        B, seed, U_resample, T_resample):
    """Variant: use a pre-computed A_obs as the observed coefficient
    matrix (rather than re-fitting OLS VAR). The null distribution is
    still OLS VAR refits on AR(1) sims — this is the Block 2 dynotears
    convention applied to our adversary, which is appropriate because
    the adversary is meant to be evaluated by the same null procedure
    used for honest methods."""
    W_block = A_obs[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim, _, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)



# §7. Substrate prep: V-Dem


print("\n" + "=" * 70)
print("Preparing substrate (V-Dem)")
print("=" * 70)

substrates = []

# V-Dem substrate (only one)
df_vdem = pd.read_csv(os.path.join(DATA_DIR, 'my_data_75_years.csv'))
with open(os.path.join(LSC_DIR, 'groups_identified.json')) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
ref_block = groups_data['groups'][0]
vdem_ref_indices = ref_block['indices']

mask = ~df_vdem[all_var_names].isna().any(axis=1)
df_vdem = df_vdem[mask].sort_values(['country_id', 'year']).reset_index(drop=True)
split_spec_vdem = (df_vdem.groupby('country_id', sort=False)['year']
                          .count().tolist())
Y_vdem = df_vdem[all_var_names].values
Y_vdem = (Y_vdem - Y_vdem.mean(0)) / (Y_vdem.std(0) + 1e-12)
T_vdem = max(split_spec_vdem); U_vdem = len(split_spec_vdem)

substrates.append({
    'name': 'V-Dem',
    'kind': 'V-Dem',
    'seed': 0,
    'Y': Y_vdem,
    'split_spec': split_spec_vdem,
    'ref_indices': vdem_ref_indices,
    'U': U_vdem, 'T': T_vdem,
})
print(f"  V-Dem: shape={Y_vdem.shape}, ref_block={len(vdem_ref_indices)}, "
      f"U={U_vdem}, T={T_vdem}")



# §8. Run sweep: each substrate × each λ
#
# For each (substrate, λ): fit adversary, compute ΔPR + p, CRPS, rb-beats.
# Resume support per (substrate, λ).


print("\n" + "=" * 70)
print(f"Sweep: {len(substrates)} substrates × {len(LAMBDA_GRID)} λ values "
      f"(B_perms={B_PERMS}, B_RB={B_RB})")
print("=" * 70)

incremental_csv = os.path.join(OUT_DIR, 'adversary_sweep_incremental.csv')
all_results = []
completed = set()
if os.path.exists(incremental_csv):
    df_prev = pd.read_csv(incremental_csv)
    for _, r in df_prev.iterrows():
        completed.add((r['substrate'], float(r['lambda'])))
        all_results.append(r.to_dict())
    print(f"Resumed: {len(completed)} runs already done")

t_start = time.time()
total_runs = len(substrates) * len(LAMBDA_GRID)
done_count = len(completed)

for sub in substrates:
    Y = sub['Y']; ss = sub['split_spec']; ref = sub['ref_indices']
    U = sub['U']; T = sub['T']

    for lam in LAMBDA_GRID:
        if (sub['name'], lam) in completed:
            continue

        run_t0 = time.time()

        # Fit adversary
        A_adv, u_adv, v_adv, n_iters, fit_loss = fit_olsvar_rank1_regularized(
            Y, ss, ref, lam, max_iters=ADV_MAX_ITERS, tol=ADV_TOL)

        # ΔPR + parametric null
        d_pa, p_pa, pr_obs, mu_pa, sd_pa = compute_delta_pr_parametric_with_A(
            A_adv, Y, ss, ref, B=B_PERMS, seed=int(7919 + lam * 1000),
            U_resample=U, T_resample=T)

        # CRPS
        crps_adv = compute_crps(Y, ss, A_adv)

        # rb-beats
        rb_pass, cand_crps_check, rand_crps_med = rb_beats(
            Y, ss, A_adv, B=B_RB, seed=int(31 + lam * 100))

        passes_dpr = bool(np.isfinite(d_pa) and d_pa > 0 and p_pa < ALPHA)
        elapsed = time.time() - run_t0

        row = {
            'substrate': sub['name'], 'kind': sub['kind'], 'seed': sub['seed'],
            'lambda': lam,
            'PR_obs': pr_obs,
            'parametric_delta': d_pa, 'parametric_p': p_pa,
            'parametric_null_mean': mu_pa, 'parametric_null_sd': sd_pa,
            'passes_dpr': passes_dpr,
            'crps': crps_adv,
            'crps_random_median': rand_crps_med,
            'rb_pass': bool(rb_pass),
            'fit_n_iters': n_iters, 'fit_loss': fit_loss,
        }
        all_results.append(row)
        done_count += 1

        pd.DataFrame(all_results).to_csv(incremental_csv, index=False)
        total_elapsed = (time.time() - t_start) / 60
        print(f"  {sub['name']:20s} λ={lam:>5.2f}: "
              f"PR_obs={pr_obs:.4f} ΔPR={d_pa:+.4f} p={p_pa:.4f} {'✓' if passes_dpr else '✗'} | "
              f"CRPS={crps_adv:.4f} (rand_med={rand_crps_med:.4f}) rb={'✓' if rb_pass else '✗'} | "
              f"iters={n_iters} [{elapsed:.1f}s, {done_count}/{total_runs}, "
              f"{total_elapsed:.1f}min]")

print(f"\nSweep complete: {(time.time()-t_start)/60:.1f}min")



# §9. Aggregate by λ


print("\n" + "=" * 70)
print("AGGREGATE: per-λ results on V-Dem")
print("=" * 70)

df = pd.DataFrame(all_results)

print(f"\n{'λ':>6s} | {'PR_obs':>7s} {'ΔPR':>8s} {'p':>7s} {'dpr':>4s} | "
      f"{'CRPS':>8s} {'rand_med':>9s} {'rb':>3s} | iters")
print("-" * 80)

agg_rows = []
for lam in LAMBDA_GRID:
    sub = df[df['lambda'] == lam]
    if len(sub) == 0:
        continue
    r = sub.iloc[0]
    dpr_marker = '✓' if r['passes_dpr'] else '✗'
    rb_marker = '✓' if r['rb_pass'] else '✗'
    print(f"{lam:>6.2f} | "
          f"{r['PR_obs']:>7.4f} {r['parametric_delta']:>+8.4f} {r['parametric_p']:>7.4f}  {dpr_marker:>3s} | "
          f"{r['crps']:>8.4f} {r['crps_random_median']:>9.4f}   {rb_marker:>2s} | "
          f"{r['fit_n_iters']:>5d}")

    agg_rows.append({
        'lambda': lam,
        'PR_obs': float(r['PR_obs']),
        'parametric_delta': float(r['parametric_delta']),
        'parametric_p': float(r['parametric_p']),
        'passes_dpr': bool(r['passes_dpr']),
        'crps': float(r['crps']),
        'crps_random_median': float(r['crps_random_median']),
        'rb_pass': bool(r['rb_pass']),
        'crps_excess_vs_random': float(r['crps'] - r['crps_random_median']),
        'crps_excess_vs_olsvar': float(r['crps'] - df[df['lambda'] == 0.0]['crps'].iloc[0])
                                 if (df['lambda'] == 0.0).any() else float('nan'),
    })

df_agg = pd.DataFrame(agg_rows)
agg_csv = os.path.join(OUT_DIR, 'adversary_sweep_aggregate.csv')
df_agg.to_csv(agg_csv, index=False)
print(f"\nSaved aggregate: {agg_csv}")



# §10. Defense interpretation
#
# The framework defends successfully if there is no λ where BOTH ΔPR
# passes AND rb-beats passes. We also report the CRPS excess over
# the OLS VAR baseline (λ=0) as a continuous measure of
# predictive cost paid for ΔPR-gaming.


print("\n" + "=" * 70)
print("DEFENSE INTERPRETATION")
print("=" * 70)

baseline = df_agg[df_agg['lambda'] == 0.0]
if len(baseline) > 0:
    br = baseline.iloc[0]
    print(f"\nBaseline (λ=0, honest OLS VAR):")
    print(f"  ΔPR: PR_obs={br['PR_obs']:.4f}, p={br['parametric_p']:.4f} {'✓' if br['passes_dpr'] else '✗'}")
    print(f"  CRPS: {br['crps']:.4f}, rand_med={br['crps_random_median']:.4f}, rb-pass={'✓' if br['rb_pass'] else '✗'}")

# Where does the adversary first game ΔPR?
games_dpr = df_agg[df_agg['passes_dpr']]
if len(games_dpr) == 0:
    print(f"\n⇒ Adversary fails to game ΔPR at any λ tested. The framework's ΔPR criterion")
    print(f"  is robust to rank-one regularization on V-Dem.")
else:
    min_lam = games_dpr['lambda'].min()
    print(f"\nAdversary games ΔPR starting at λ={min_lam:.2f}.")

    # Among λ where adversary games ΔPR, does rb-beats also pass?
    both_pass = games_dpr[games_dpr['rb_pass']]
    if len(both_pass) == 0:
        print(f"⇒ DEFENSE HOLDS: the adversary passes ΔPR but FAILS rb-beats at every λ "
              f"≥ {min_lam:.2f}. The multi-criteria framework catches the adversary on rb-beats.")
    else:
        # Defense gap
        print(f"⚠ DEFENSE GAP: adversary passes BOTH ΔPR and rb-beats at λ ∈ "
              f"{both_pass['lambda'].tolist()}.")
        # But also check CRPS excess
        for _, r in both_pass.iterrows():
            print(f"    λ={r['lambda']:>5.2f}: PR_obs={r['PR_obs']:.3f}, p={r['parametric_p']:.4f}, "
                  f"CRPS={r['crps']:.4f} (baseline={br['crps']:.4f}, "
                  f"excess_vs_olsvar={r['crps_excess_vs_olsvar']:+.4f})")
        # If CRPS excess is meaningful even though rb-pass holds, the defense is partial:
        # an adversary's CRPS is worse than honest OLS VAR but still beats random.
        max_excess = both_pass['crps_excess_vs_olsvar'].max()
        if max_excess > 0.001:
            print(f"\n  Note: CRPS excess over honest OLS VAR baseline is up to {max_excess:.4f}.")
            print(f"  Even where rb-beats passes, the adversary's predictive performance is")
            print(f"  measurably worse than honest OLS VAR. A more careful framework that")
            print(f"  compares against the honest baseline (rather than against random) would")
            print(f"  catch the adversary; the current rb-beats criterion is permissive but")
            print(f"  not the only line of defense.")

# Show full Pareto picture
print(f"\nFull Pareto picture (PR_obs vs CRPS excess vs random baseline):")
print(f"{'λ':>6s} | {'PR_obs':>7s} {'p':>7s} | {'CRPS':>8s} {'excs_rnd':>8s} {'rb':>3s}")
for _, r in df_agg.iterrows():
    print(f"{r['lambda']:>6.2f} | {r['PR_obs']:>7.4f} {r['parametric_p']:>7.4f} | "
          f"{r['crps']:>8.4f} {r['crps_excess_vs_random']:>+8.4f}  "
          f"{'✓' if r['rb_pass'] else '✗':>2s}")

print(f"\nDone.")

Mounted at /content/drive
λ grid: [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0]
Substrate: V-Dem only (R3' dropped — see header docstring for rationale)
Utilities defined.

Preparing substrate (V-Dem)
  V-Dem: shape=(6675, 16), ref_block=13, U=89, T=75

Sweep: 1 substrates × 10 λ values (B_perms=1000, B_RB=200)
  V-Dem                λ= 0.00: PR_obs=0.4399 ΔPR=+0.0450 p=0.2697 ✗ | CRPS=0.5566 (rand_med=0.5608) rb=✓ | iters=0 [3.2s, 1/10, 0.1min]
  V-Dem                λ= 0.01: PR_obs=0.4821 ΔPR=+0.0880 p=0.1289 ✗ | CRPS=0.5568 (rand_med=0.5606) rb=✓ | iters=5 [3.2s, 2/10, 0.1min]
  V-Dem                λ= 0.05: PR_obs=0.5948 ΔPR=+0.2024 p=0.0080 ✓ | CRPS=0.5574 (rand_med=0.5611) rb=✓ | iters=12 [3.2s, 3/10, 0.2min]
  V-Dem                λ= 0.10: PR_obs=0.6789 ΔPR=+0.2856 p=0.0010 ✓ | CRPS=0.5579 (rand_med=0.5612) rb=✓ | iters=19 [3.2s, 4/10, 0.2min]
  V-Dem                λ= 0.25: PR_obs=0.8065 ΔPR=+0.4129 p=0.0010 ✓ | CRPS=0.5587 (rand_med=0.5617) rb=✓ | iters=38 [3.2s, 5/10

## Section 5 — §5 WB-WPP and V-Dem-60 application

The WB-WPP demographic panel and V-Dem-60 raw indicator panel use the principled EDA → cov-rank-1 → multi-criteria pipeline (§5.6 and §5.7). Cells §5.1-§5.5 are exploratory viability checks used during paper development; the canonical results are in §5.6 and §5.7.

### Cell §5.1 — WB-WPP viability check: exploratory

**Paper sections:** Not in paper directly; superseded by §5.6.

Initial viability assessment of the WB-WPP demographic panel. Tests whether OLS VAR fits cleanly, whether the parametric AR(1) null runs without explosive simulations (median age has β > 1, urban share β ≈ 0.99), and whether candidate block configurations admit reflective structure.

**Inputs:** WB-WPP panel `data/my_data_WB.csv`.

**Outputs:** Viability results pickle.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
# # Block 13 viability: WB-WPP as second real panel
#
# 173 countries × 56 years (1970-2025), 16 indicators after dedup,
# zero missingness, perfectly balanced. Two candidate development-factor
# blocks tested under OLS VAR + parametric AR(1) null with β-clip at 0.99.


from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'wb_wpp_viability_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Path to the WB-WPP file. Adjust if you placed it elsewhere on Drive.
WB_WPP_CSV = os.path.join(DATA_DIR, 'my_data_WB.csv')



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99   # cap simulated AR(1) β to avoid explosive paths

print(f"B_PERMS={B_PERMS}, BETA_CLIP={BETA_CLIP}")



# §2. Core utilities (copied from era_subsets_block_5.py, with β-clip
# added to the AR(1) null simulation)


def fit_ols_var_signed(Y, split_spec):
    """Pooled OLS VAR(1), returns full SIGNED coefficient matrix and per-coef p-values."""
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N)); p_values = np.ones((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        resid = y - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]; continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]
        p_values[i, :] = p_vals[:N]
    np.fill_diagonal(A, 0.0)
    return A, p_values


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    """β-CLIP added: clamp |β| to beta_clip to avoid explosive sims when
    fitted β >= 1 (e.g., median age, which trends).

    Returns standardized panel of shape (U*T, N).
    """
    N = len(betas)
    betas_clipped = np.clip(betas, -beta_clip, beta_clip)
    stationary_var = sigmas**2 / (1 - betas_clipped**2 + 1e-12)
    stationary_sd = np.sqrt(np.maximum(stationary_var, 1e-12))
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas_clipped[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric_olsvar(Y, split_spec, ref_indices, B, seed,
                                       U_resample, T_resample, beta_clip=BETA_CLIP,
                                       return_diagnostics=False):
    """Parametric AR(1) null with β-clip. Returns (ΔPR, p, PR_obs, μ_null, σ_null)
    plus optional diagnostics dict."""
    A, _ = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    sim_max_abs = []  # diagnostic: largest |y| across the simulated panels
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample,
                                                rng, beta_clip=beta_clip)
        if return_diagnostics:
            sim_max_abs.append(float(np.max(np.abs(Y_sim))))
        ss_sim = [T_resample] * U_resample
        A_sim, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    result = (float(delta), float(p), float(PR_obs), mu, sd)
    if return_diagnostics:
        return result + ({'betas_fit': betas.tolist(),
                          'betas_clipped': np.clip(betas, -beta_clip, beta_clip).tolist(),
                          'sigmas_fit': sigmas.tolist(),
                          'sim_max_abs_q': np.percentile(sim_max_abs, [50, 90, 99]).tolist(),
                          'null_PRs_distribution': null_PRs.tolist()},)
    return result


print("Utilities defined.")



# §3. Load WB-WPP panel and assemble candidate blocks


df = pd.read_csv(WB_WPP_CSV)
print(f"Raw shape: {df.shape}")
print(f"Countries: {df['country_name'].nunique()}, "
      f"year range: {df['year'].min()}-{df['year'].max()}")

# Sanity: balanced panel with no missingness
n_rows_per_country = df.groupby('country_name').size()
assert n_rows_per_country.min() == n_rows_per_country.max() == 56, \
    f"Panel not balanced: rows per country range {n_rows_per_country.min()}-{n_rows_per_country.max()}"
assert df.isna().sum().sum() == 0, "Missingness present"
print(f"Balanced panel, no missingness ✓")

# Sort by country then year (the order fit_ols_var_signed expects)
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)

# Define the two candidate blocks
BLOCK_MOD_6 = [
    'wb_life_expectancy', 'wb_urban_pop_share', 'wpp_median_age_jul',
    'kofipgidj', 'kofpogidf', 'kofpogidj'
]
BLOCK_MOD_9_BASE = BLOCK_MOD_6 + [
    'wb_pop_growth', 'wb_urban_pop_growth', 'wpp_nat_change_rate'
]
SIGN_FLIP = ['wb_pop_growth', 'wb_urban_pop_growth', 'wpp_nat_change_rate']

print(f"\nBlock mod_6 (N={len(BLOCK_MOD_6)}): {BLOCK_MOD_6}")
print(f"Block mod_9_signed (N={len(BLOCK_MOD_9_BASE)}): {BLOCK_MOD_9_BASE}")
print(f"  with sign-flipped: {SIGN_FLIP}")



# §4. Per-block analysis runner


def run_block_analysis(df, indicator_cols, sign_flip_cols, block_label, B, seed):
    """Run OLS VAR + parametric AR(1) null on the specified block.

    Sign-flips columns in sign_flip_cols (multiplied by -1) BEFORE
    standardization, so all loadings on the latent factor align positively.
    """
    df_block = df[['country_name', 'year'] + indicator_cols].copy()

    # Sign-flip
    for c in sign_flip_cols:
        if c in indicator_cols:
            df_block[c] = -df_block[c]

    # Standardize per-indicator (panel-level)
    for c in indicator_cols:
        df_block[c] = (df_block[c] - df_block[c].mean()) / df_block[c].std()

    # Build Y and split_spec
    countries = df_block['country_name'].unique().tolist()
    Y_l, ss = [], []
    for cn in countries:
        sub = df_block[df_block['country_name'] == cn].sort_values('year')
        Y_l.append(sub[indicator_cols].values)
        ss.append(len(sub))
    Y = np.vstack(Y_l)
    U, T = len(countries), ss[0]  # balanced
    N = len(indicator_cols)
    print(f"\n=== Block: {block_label} ===")
    print(f"  Y shape: {Y.shape}, U={U}, T={T}, N={N}")

    # T_cov diagnostic on the indicator correlation
    C = np.corrcoef(Y.T)
    T_cov = offdiag_rank1_fit(C)
    print(f"  T_cov (off-diagonal rank-1 fit of indicator correlation) = {T_cov:.4f}")

    # ref_indices = all indicators in the block (single block, no exclusion)
    ref_indices = np.arange(N)

    print(f"  Running parametric AR(1) null with B={B}, β-clip={BETA_CLIP}...")
    t0 = time.time()
    delta, p, pr_obs, mu, sd, diag = compute_delta_pr_parametric_olsvar(
        Y, ss, ref_indices, B, seed, U_resample=U, T_resample=T,
        beta_clip=BETA_CLIP, return_diagnostics=True
    )
    elapsed = time.time() - t0
    print(f"  Elapsed: {elapsed:.1f}s")

    # Report
    print(f"\n  PR_obs        = {pr_obs:.4f}")
    print(f"  null mean     = {mu:.4f}")
    print(f"  null std      = {sd:.4f}")
    print(f"  ΔPR           = {delta:+.4f}")
    print(f"  p (parametric)= {p:.4f}  ({'PASS' if p < ALPHA else 'FAIL'} at α={ALPHA})")

    print(f"\n  β diagnostics (fit on observed panel):")
    for c, b_fit, b_clip in zip(indicator_cols, diag['betas_fit'], diag['betas_clipped']):
        flag = '  ← CLIPPED' if abs(b_fit) > BETA_CLIP else ''
        print(f"    {c:30s}  β={b_fit:+.4f}{flag}")

    print(f"\n  Simulated panel |y| diagnostics (median / 90th / 99th percentile across {B} sims):")
    print(f"    {diag['sim_max_abs_q'][0]:.2f} / {diag['sim_max_abs_q'][1]:.2f} / {diag['sim_max_abs_q'][2]:.2f}")
    print(f"    (compare to observed standardized panel max |y|: {np.abs(Y).max():.2f})")
    print(f"    If 99th-percentile sim |y| >> 100, β-clip may not be sufficient.")

    return {
        'block_label': block_label,
        'indicator_cols': indicator_cols,
        'sign_flip_cols': sign_flip_cols,
        'U': U, 'T': T, 'N': N,
        'T_cov': T_cov,
        'PR_obs': pr_obs,
        'null_mean': mu, 'null_std': sd,
        'delta_PR': delta,
        'p_parametric': p,
        'verdict_at_0.05': 'PASS' if p < ALPHA else 'FAIL',
        'betas_fit': diag['betas_fit'],
        'betas_clipped_at': BETA_CLIP,
        'sigmas_fit': diag['sigmas_fit'],
        'sim_max_abs_q50_q90_q99': diag['sim_max_abs_q'],
        'observed_panel_max_abs': float(np.abs(Y).max()),
        'elapsed_s': elapsed,
    }



# §5. Run both blocks


results = []

# Block mod_6 — the cleanest interpretable development-factor block
res_mod6 = run_block_analysis(
    df, BLOCK_MOD_6, sign_flip_cols=[],
    block_label='mod_6', B=B_PERMS, seed=42
)
results.append(res_mod6)

# Block mod_9_signed — adds the sign-flipped demographic-transition rates
res_mod9 = run_block_analysis(
    df, BLOCK_MOD_9_BASE, sign_flip_cols=SIGN_FLIP,
    block_label='mod_9_signed', B=B_PERMS, seed=42
)
results.append(res_mod9)



# §6. Summary and decision


print("\n" + "="*70)
print("WB-WPP VIABILITY SUMMARY")
print("="*70)
print(f"{'block':18s} {'N':>3s} {'T_cov':>7s} {'PR_obs':>7s} {'ΔPR':>8s} {'p_param':>9s} {'verdict':>8s}")
for r in results:
    print(f"{r['block_label']:18s} {r['N']:>3d} {r['T_cov']:>7.4f} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+8.4f} "
          f"{r['p_parametric']:>9.4f} {r['verdict_at_0.05']:>8s}")

print("\nFor V-Dem comparison:")
print("  long_16var (size-13 block): T_cov=0.987, PR_obs=0.440, ΔPR=+0.045, "
      "p_param=0.266, verdict=FAIL (this is the headline failure)")

print("\nDecision rule:")
print("  - If both blocks PASS → V-Dem failure looks idiosyncratic to V-Dem.")
print("    Headline weakens; but the synthetic-real divergence becomes a")
print("    V-Dem-specific finding. Substrate-specificity is itself worth")
print("    reporting (and aligns with the §6.3 IRT-aggregation hypothesis).")
print("  - If both blocks FAIL → headline strengthens substantially.")
print("    Two real panels, both with high T_cov, both fail OLS VAR ΔPR.")
print("    Synthetic-real divergence is general, not V-Dem-specific.")
print("  - Mixed (one PASS one FAIL) → study which features differ between")
print("    blocks and learn from the contrast.")

# Save results
out_path = os.path.join(OUT_DIR, 'wb_wpp_viability_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {out_path}")

### Cell §5.2 — WB-WPP viability v2 (outlier diagnostic): exploratory

**Paper sections:** Not in paper directly; superseded by §5.6.

Outlier diagnostic and robustness rerun for WB-WPP. The first viability run flagged an outlier in `wb_urban_pop_growth` (raw range [-142.88, +43.75], max |z| ≈ 45). This cell identifies the offending observation and reruns on the cleaned panel. The 14 indicators dropped during EDA cleaning that this cell identifies are documented in `outputs/eda_verification/`.

**Inputs:** WB-WPP CSV.

**Outputs:** Viability v2 results JSON.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# Paths (match v1)
DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'wb_wpp_viability_outputs')
os.makedirs(OUT_DIR, exist_ok=True)
WB_WPP_CSV = os.path.join(DATA_DIR, 'my_data_WB.csv')

B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99



# §1. Locate the outlier


df = pd.read_csv(WB_WPP_CSV)
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)

print("=" * 70)
print("OUTLIER DIAGNOSTIC — per-indicator max |z| and identifying observations")
print("=" * 70)

# Candidates from v1's blocks plus rates that might bite later
candidates = [
    'wb_life_expectancy', 'wb_urban_pop_share', 'wpp_median_age_jul',
    'kofipgidj', 'kofpogidf', 'kofpogidj',
    'wb_pop_growth', 'wb_urban_pop_growth', 'wpp_nat_change_rate',
    'wpp_net_migration_rate', 'wpp_net_migrants',
]

print(f"{'indicator':30s} {'min':>10s} {'max':>10s} {'mean':>10s} {'std':>10s} "
      f"{'max|z|':>7s} {'extreme observation':>40s}")
print("-" * 130)
for c in candidates:
    if c not in df.columns:
        continue
    s = df[c]
    z = (s - s.mean()) / s.std()
    idx = z.abs().idxmax()
    extreme_row = df.loc[idx]
    print(f"{c:30s} {s.min():>10.2f} {s.max():>10.2f} {s.mean():>10.2f} "
          f"{s.std():>10.2f} {z.abs().max():>7.2f}  "
          f"{extreme_row['country_name'][:25]:>25s} ({int(extreme_row['year'])}) "
          f"raw={s[idx]:>+8.2f}")



# §2. Define the three robustness blocks


BLOCK_MOD_8_SIGNED = [
    'wb_life_expectancy', 'wb_urban_pop_share', 'wpp_median_age_jul',
    'kofipgidj', 'kofpogidf', 'kofpogidj',
    'wb_pop_growth', 'wpp_nat_change_rate',
]
SIGN_FLIP_MOD_8 = ['wb_pop_growth', 'wpp_nat_change_rate']

BLOCK_MOD_5_WBWPP_CLEAN = [
    'wb_life_expectancy', 'wb_urban_pop_share', 'wpp_median_age_jul',
    'wb_pop_growth', 'wpp_nat_change_rate',
]
SIGN_FLIP_MOD_5 = ['wb_pop_growth', 'wpp_nat_change_rate']

BLOCK_MOD_4_LEVELS_KOF = [
    'wb_life_expectancy', 'wb_urban_pop_share', 'wpp_median_age_jul', 'kofipgidj',
]
SIGN_FLIP_MOD_4 = []  # no flips — all positive loadings

NEW_BLOCKS = [
    ('mod_8_signed', BLOCK_MOD_8_SIGNED, SIGN_FLIP_MOD_8),
    ('mod_5_wbwpp_clean', BLOCK_MOD_5_WBWPP_CLEAN, SIGN_FLIP_MOD_5),
    ('mod_4_levels_kof', BLOCK_MOD_4_LEVELS_KOF, SIGN_FLIP_MOD_4),
]



# §3. Run the three new blocks


new_results = []
for label, cols, flips in NEW_BLOCKS:
    res = run_block_analysis(
        df, cols, sign_flip_cols=flips,
        block_label=label, B=B_PERMS, seed=42
    )
    new_results.append(res)



# §4. Combined summary (v1 + v2 results)


# Load v1 results if saved, else use the values from the v1 run output:
v1_results_path = os.path.join(OUT_DIR, 'wb_wpp_viability_results.json')
if os.path.exists(v1_results_path):
    with open(v1_results_path) as f:
        v1_results = json.load(f)
else:
    # Hard-coded from v1 run if file missing
    v1_results = [
        {'block_label': 'mod_6', 'N': 6, 'T_cov': 0.9701, 'PR_obs': 0.5686,
         'delta_PR': -0.0124, 'p_parametric': 0.5485, 'verdict_at_0.05': 'FAIL',
         'observed_panel_max_abs': 5.07},
        {'block_label': 'mod_9_signed', 'N': 9, 'T_cov': 0.9538, 'PR_obs': 0.8096,
         'delta_PR': +0.1694, 'p_parametric': 0.0220, 'verdict_at_0.05': 'PASS',
         'observed_panel_max_abs': 45.78},
    ]

all_results = v1_results + new_results

print("\n" + "=" * 100)
print("WB-WPP VIABILITY — full results (v1 + outlier-diagnostic v2)")
print("=" * 100)
print(f"{'block':22s} {'N':>3s} {'T_cov':>7s} {'PR_obs':>7s} {'ΔPR':>8s} "
      f"{'p_param':>9s} {'max|y|_obs':>11s} {'verdict':>8s}")
print("-" * 100)
for r in all_results:
    print(f"{r['block_label']:22s} {r['N']:>3d} {r['T_cov']:>7.4f} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+8.4f} "
          f"{r['p_parametric']:>9.4f} "
          f"{r.get('observed_panel_max_abs', float('nan')):>11.2f} "
          f"{r['verdict_at_0.05']:>8s}")

print("\nFor V-Dem comparison:")
print("  long_16var (size-13 block): T_cov=0.987, PR_obs=0.440, ΔPR=+0.045, "
      "p_param=0.266, verdict=FAIL")

print("\n" + "=" * 100)
print("INTERPRETATION GUIDE")
print("=" * 100)
print("""
Outlier hypothesis check:
  mod_9_signed obs max |y| = 45.78 — outlier-contaminated.
  mod_8_signed = mod_9 minus wb_urban_pop_growth.
    - If mod_8 still PASS at p<0.05: mod_9 verdict survives outlier removal.
      Headline: WB-WPP shows OLS VAR can pass on real latent-factor panels.
    - If mod_8 FAIL or borderline: mod_9 PASS was outlier-driven.
      We need to discount the mod_9 result.

KOF-vs-N hypothesis check:
  mod_5_wbwpp_clean (N=5, no KOF, no urban_growth) vs mod_6 (N=6, with KOF):
    - mod_5 PASS, mod_6 FAIL: KOF inclusion was the issue (KOF has
      different temporal dynamics than the WB/WPP indicators).
    - Both FAIL: small-N underpower is the issue.
    - mod_5 FAIL, mod_6 PASS: implausible given v1.

mod_4_levels_kof (T_cov=0.987 like V-Dem):
  Smallest block. Tests whether matching V-Dem's T_cov exactly produces
  V-Dem-like behavior. If FAIL with low p: weak evidence that high-T_cov
  cross-section is associated with OLS VAR ΔPR failure (consistent with
  V-Dem and with mod_6 FAIL). If PASS: weakens that association.
""")

# Save merged results
with open(os.path.join(OUT_DIR, 'wb_wpp_viability_results_v2.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"Merged results saved to {os.path.join(OUT_DIR, 'wb_wpp_viability_results_v2.json')}")

### Cell §5.3 — V-Dem-60 indicators viability check: exploratory

**Paper sections:** Not in paper directly; superseded by §5.6.

Initial viability check of the V-Dem 70-indicator panel (the `my_data_VDem_Ind.csv` raw indicators that get reduced to the V-Dem-60 panel after EDA cleaning). Tests whether indicator-level V-Dem admits a reflective block at all, before running the principled pipeline in §5.6.


Three blocks tested:

  vdem_ind_top13: top 13 indicators by |PC1 loading| from the 60-indicator
                  panel-correlation eigendecomposition. N=13 matches
                  long_16var's size-13 reflective block exactly. Direct
                  apples-to-apples comparison.

  vdem_ind_civil_lib_19: the 19 civil-liberties indicators (v2cl*).
                  Substantively coherent family, larger N → more power.
                  Tests whether a domain-defined block also passes.

  vdem_ind_electoral_7: the 7 electoral indicators (v2el*, v2ps*).
                  Smaller substantively coherent family. Tests
                  consistency across families.

Pipeline matches era_subsets_block_5 / WB-WPP viability:
  - OLS VAR signed coefficient matrix
  - Off-diagonal rank-1 ALS for PR_obs
  - Parametric AR(1) null at B=1000 with β-clip at 0.99


**Inputs:** V-Dem indicators CSV `data/my_data_VDem_Ind.csv`.

**Outputs:** Viability results JSON.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'vdem_indicators_viability_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

VDEM_IND_CSV = os.path.join(DATA_DIR, 'my_data_VDem_Ind.csv')



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99   # cap simulated AR(1) β to avoid explosive paths

print(f"B_PERMS={B_PERMS}, BETA_CLIP={BETA_CLIP}")



# §2. Core utilities (verbatim copy from era_subsets_block_5,
# with β-clip added — same as WB-WPP viability v1)


def fit_ols_var_signed(Y, split_spec):
    """Pooled OLS VAR(1), returns full SIGNED coefficient matrix and per-coef p-values."""
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N)); p_values = np.ones((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        resid = y - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]; continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]
        p_values[i, :] = p_vals[:N]
    np.fill_diagonal(A, 0.0)
    return A, p_values


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    """β-CLIP: clamp |β| to beta_clip to avoid explosive sims."""
    N = len(betas)
    betas_clipped = np.clip(betas, -beta_clip, beta_clip)
    stationary_var = sigmas**2 / (1 - betas_clipped**2 + 1e-12)
    stationary_sd = np.sqrt(np.maximum(stationary_var, 1e-12))
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas_clipped[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric_olsvar(Y, split_spec, ref_indices, B, seed,
                                       U_resample, T_resample, beta_clip=BETA_CLIP,
                                       return_diagnostics=False):
    """Parametric AR(1) null with β-clip. Returns (ΔPR, p, PR_obs, μ, σ) plus optional diagnostics."""
    A, _ = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    sim_max_abs = []
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample,
                                                rng, beta_clip=beta_clip)
        if return_diagnostics:
            sim_max_abs.append(float(np.max(np.abs(Y_sim))))
        ss_sim = [T_resample] * U_resample
        A_sim, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    result = (float(delta), float(p), float(PR_obs), mu, sd)
    if return_diagnostics:
        return result + ({'betas_fit': betas.tolist(),
                          'betas_clipped': np.clip(betas, -beta_clip, beta_clip).tolist(),
                          'sigmas_fit': sigmas.tolist(),
                          'sim_max_abs_q': np.percentile(sim_max_abs, [50, 90, 99]).tolist(),
                          'null_PRs_distribution': null_PRs.tolist()},)
    return result


def run_block_analysis(df, indicator_cols, sign_flip_cols, block_label, B, seed):
    """Run OLS VAR + parametric AR(1) null on the specified block."""
    df_block = df[['country_name', 'year'] + indicator_cols].copy()
    for c in sign_flip_cols:
        if c in indicator_cols:
            df_block[c] = -df_block[c]
    for c in indicator_cols:
        df_block[c] = (df_block[c] - df_block[c].mean()) / df_block[c].std()
    countries = df_block['country_name'].unique().tolist()
    Y_l, ss = [], []
    for cn in countries:
        sub = df_block[df_block['country_name'] == cn].sort_values('year')
        Y_l.append(sub[indicator_cols].values)
        ss.append(len(sub))
    Y = np.vstack(Y_l)
    U, T = len(countries), ss[0]
    N = len(indicator_cols)
    print(f"\n=== Block: {block_label} ===")
    print(f"  Y shape: {Y.shape}, U={U}, T={T}, N={N}")

    # T_cov diagnostic
    C = np.corrcoef(Y.T)
    T_cov = offdiag_rank1_fit(C)
    print(f"  T_cov (off-diagonal rank-1 fit of indicator correlation) = {T_cov:.4f}")

    ref_indices = np.arange(N)
    print(f"  Running parametric AR(1) null with B={B}, β-clip={BETA_CLIP}...")
    t0 = time.time()
    delta, p, pr_obs, mu, sd, diag = compute_delta_pr_parametric_olsvar(
        Y, ss, ref_indices, B, seed, U_resample=U, T_resample=T,
        beta_clip=BETA_CLIP, return_diagnostics=True
    )
    elapsed = time.time() - t0
    print(f"  Elapsed: {elapsed:.1f}s")

    print(f"\n  PR_obs        = {pr_obs:.4f}")
    print(f"  null mean     = {mu:.4f}")
    print(f"  null std      = {sd:.4f}")
    print(f"  ΔPR           = {delta:+.4f}")
    print(f"  p (parametric)= {p:.4f}  ({'PASS' if p < ALPHA else 'FAIL'} at α={ALPHA})")

    print(f"\n  β diagnostics:")
    n_clipped = sum(1 for b in diag['betas_fit'] if abs(b) > BETA_CLIP)
    print(f"    median β = {np.median(diag['betas_fit']):.4f}")
    print(f"    β range  = [{min(diag['betas_fit']):.4f}, {max(diag['betas_fit']):.4f}]")
    print(f"    n_clipped at |β|>{BETA_CLIP}: {n_clipped}/{len(diag['betas_fit'])}")
    print(f"    long_16var comparison: median β=0.987, range=[0.928, 0.991], all clipped")

    print(f"\n  Simulated panel |y| diagnostics (median / 90th / 99th):")
    print(f"    {diag['sim_max_abs_q'][0]:.2f} / {diag['sim_max_abs_q'][1]:.2f} / {diag['sim_max_abs_q'][2]:.2f}")
    print(f"    (observed standardized panel max |y|: {np.abs(Y).max():.2f})")

    return {
        'block_label': block_label,
        'indicator_cols': indicator_cols,
        'sign_flip_cols': sign_flip_cols,
        'U': U, 'T': T, 'N': N,
        'T_cov': T_cov,
        'PR_obs': pr_obs,
        'null_mean': mu, 'null_std': sd,
        'delta_PR': delta,
        'p_parametric': p,
        'verdict_at_0.05': 'PASS' if p < ALPHA else 'FAIL',
        'betas_fit': diag['betas_fit'],
        'betas_median': float(np.median(diag['betas_fit'])),
        'sigmas_fit': diag['sigmas_fit'],
        'sim_max_abs_q50_q90_q99': diag['sim_max_abs_q'],
        'observed_panel_max_abs': float(np.abs(Y).max()),
        'elapsed_s': elapsed,
    }


print("Utilities defined.")



# §3. Load V-Dem indicator panel and select cleaned indicator pool


df = pd.read_csv(VDEM_IND_CSV)
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)
print(f"Raw shape: {df.shape}")
print(f"Countries: {df['country_name'].nunique()}, year range: {df['year'].min()}-{df['year'].max()}")

# Sanity check
n_per_country = df.groupby('country_name').size()
assert n_per_country.min() == n_per_country.max(), "Panel not balanced"
assert df.iloc[:, 3:].isna().sum().sum() == 0, "Missingness present"
print(f"Balanced: {n_per_country.iloc[0]} years per country, no missingness")

# Build cleaned indicator pool: drop direct-democracy family (extreme outliers)
# and the two composite indices that mix in IRT-aggregated content
all_ind_cols = [c for c in df.columns if c not in ['country_id', 'country_name', 'year']]
DD_FAMILY = [c for c in all_ind_cols if c.startswith('v2dd')]
COMPOSITE_INDICES = ['v2x_elecoff', 'v2xdd_dd']
DROP = DD_FAMILY + COMPOSITE_INDICES
ind_pool = [c for c in all_ind_cols if c not in DROP]
print(f"\nDropped: v2dd* ({len(DD_FAMILY)} indicators) + composite indices ({len(COMPOSITE_INDICES)})")
print(f"Indicator pool for analysis: {len(ind_pool)} raw V-Dem indicators")



# §4. Identify the top-13 by |PC1 loading|


X = df[ind_pool]
Xz = (X - X.mean()) / X.std()
C_pool = Xz.corr().values

eigvals_full, eigvecs = np.linalg.eigh(C_pool)
order = np.argsort(eigvals_full)[::-1]
eigvecs = eigvecs[:, order]
eigvals_sorted = eigvals_full[order]
print(f"Top eigenvalue (PC1): {eigvals_sorted[0]:.3f} ({eigvals_sorted[0]/len(ind_pool):.1%} of variance)")
print(f"PC1/PC2 ratio: {eigvals_sorted[0]/eigvals_sorted[1]:.2f}")

loadings_abs = np.abs(eigvecs[:, 0])
order_pc1 = np.argsort(loadings_abs)[::-1]
TOP13 = [ind_pool[i] for i in order_pc1[:13]]
print(f"\nTop 13 by |PC1 loading|:")
for c in TOP13:
    print(f"  {c:25s}  loading={eigvecs[ind_pool.index(c), 0]:+.4f}")



# §5. Define the three blocks


BLOCK_TOP13 = TOP13
BLOCK_CIVIL_LIB = [c for c in ind_pool if c.startswith('v2cl')]   # 19 indicators
BLOCK_ELECTORAL = [c for c in ind_pool if c.startswith(('v2el', 'v2ps'))]  # 7 indicators

BLOCKS = [
    ('vdem_ind_top13',           BLOCK_TOP13,      []),
    ('vdem_ind_civil_lib_19',    BLOCK_CIVIL_LIB,  []),
    ('vdem_ind_electoral_7',     BLOCK_ELECTORAL,  []),
]

print(f"\nBlock 1 (vdem_ind_top13): N={len(BLOCK_TOP13)} top-loading indicators")
print(f"Block 2 (vdem_ind_civil_lib_19): N={len(BLOCK_CIVIL_LIB)} v2cl* indicators")
print(f"  {BLOCK_CIVIL_LIB}")
print(f"Block 3 (vdem_ind_electoral_7): N={len(BLOCK_ELECTORAL)} v2el*/v2ps* indicators")
print(f"  {BLOCK_ELECTORAL}")



# §6. Run the three blocks


results = []
for label, cols, flips in BLOCKS:
    res = run_block_analysis(
        df, cols, sign_flip_cols=flips,
        block_label=label, B=B_PERMS, seed=42
    )
    results.append(res)



# §7. Summary


print("\n" + "=" * 100)
print("V-DEM INDICATOR-LEVEL VIABILITY — full results")
print("=" * 100)
print(f"{'block':25s} {'N':>3s} {'T_cov':>7s} {'PR_obs':>7s} {'ΔPR':>8s} "
      f"{'p_param':>9s} {'med_β':>7s} {'max|y|':>7s} {'verdict':>8s}")
print("-" * 100)
for r in results:
    print(f"{r['block_label']:25s} {r['N']:>3d} {r['T_cov']:>7.4f} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+8.4f} "
          f"{r['p_parametric']:>9.4f} {r['betas_median']:>7.3f} "
          f"{r['observed_panel_max_abs']:>7.2f} "
          f"{r['verdict_at_0.05']:>8s}")

print("\nFor V-Dem long_16var (INDEX-level) headline comparison:")
print(f"  long_16var (size-13): T_cov=0.987, PR_obs=0.440, ΔPR=+0.045, p_param=0.27, "
      "median β=0.987, FAIL")

print(f"\nFor WB-WPP comparison:")
print(f"  mod_5_wbwpp_clean (N=5): T_cov=0.971, PR_obs=0.958, ΔPR=+0.131, p_param=0.035, "
      "median β=0.978, PASS")
print(f"  mod_8_signed (N=8):       T_cov=0.964, PR_obs=0.947, ΔPR=+0.250, p_param=0.001, PASS")

print("\n" + "=" * 100)
print("INTERPRETATION GUIDE")
print("=" * 100)
print("""
KEY TEST: §6.3 IRT-aggregation hypothesis.

If indicator-V-Dem PASSES (especially at the top13 block, the apples-to-apples
comparison to long_16var size-13):
  → Strong direct evidence that V-Dem long_16var's failure is an aggregation
    artifact, not a property of V-Dem's substantive political dynamics.
  → Hypothesis (c) confirmed by direct comparison, not just by elimination.
  → Headline restructures: "Framework correctly diagnoses V-Dem's
    long_16var failure as an aggregation artifact." Much stronger story.

If indicator-V-Dem FAILS:
  → Aggregation is not the (sole) mechanism. Failure is robust to
    aggregation choice. We're back to mixture-of-regimes / nonstationarity
    as candidate mechanisms.
  → Headline stays roughly as drafted, mixture test becomes critical.

If MIXED (some blocks pass, some fail):
  → Diagnostic. Which families pass which fail tells us something
    substantive about V-Dem's structure.

Note on β:
  Indicator-level β should be lower than long_16var's near-unit-root range
  (median β ≈ 0.93 vs 0.987). Lower persistence is itself a candidate
  mechanism for the difference: less near-unit-root behavior → less
  finite-sample VAR coefficient distortion → cleaner rank-one signal.
""")

# Save results
with open(os.path.join(OUT_DIR, 'vdem_indicators_viability_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {os.path.join(OUT_DIR, 'vdem_indicators_viability_results.json')}")

### Cell §5.4 — V-Dem-60 indicators viability v2 (Track A robustness): exploratory

**Paper sections:** Not in paper directly; superseded by §5.6.

Track A robustness rerun for V-Dem-60 viability. Tests whether the top13 / civil_lib_19 / electoral_7 sub-block findings from §5.3 are robust to alternative preprocessing decisions (drop-near-constant policies, alternative indicator standardizations).

**Inputs:** V-Dem indicators CSV.

**Outputs:** Viability v2 results JSON.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'vdem_indicators_viability_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

VDEM_IND_CSV = os.path.join(DATA_DIR, 'my_data_VDem_Ind.csv')



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99
NEAR_CONST_THRESHOLD = 0.50   # drop indicators that are constant for ≥ this fraction of countries

print(f"B_PERMS={B_PERMS}, BETA_CLIP={BETA_CLIP}")
print(f"Near-constant threshold: drop indicator if constant for ≥{NEAR_CONST_THRESHOLD:.0%} of countries")



# §2. Core utilities (same as v1, condensed)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N)); p_values = np.ones((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        resid = y - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]; continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]
        p_values[i, :] = p_vals[:N]
    np.fill_diagonal(A, 0.0)
    return A, p_values


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    betas_clipped = np.clip(betas, -beta_clip, beta_clip)
    stationary_var = sigmas**2 / (1 - betas_clipped**2 + 1e-12)
    stationary_sd = np.sqrt(np.maximum(stationary_var, 1e-12))
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas_clipped[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric_olsvar(Y, split_spec, ref_indices, B, seed,
                                       U_resample, T_resample, beta_clip=BETA_CLIP,
                                       return_diagnostics=False):
    A, _ = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    sim_max_abs = []
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample,
                                                rng, beta_clip=beta_clip)
        if return_diagnostics:
            sim_max_abs.append(float(np.max(np.abs(Y_sim))))
        ss_sim = [T_resample] * U_resample
        A_sim, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    result = (float(delta), float(p), float(PR_obs), mu, sd)
    if return_diagnostics:
        return result + ({'betas_fit': betas.tolist(),
                          'betas_clipped': np.clip(betas, -beta_clip, beta_clip).tolist(),
                          'sigmas_fit': sigmas.tolist(),
                          'sim_max_abs_q': np.percentile(sim_max_abs, [50, 90, 99]).tolist(),
                          'null_PRs_distribution': null_PRs.tolist()},)
    return result


def run_block_analysis(df, indicator_cols, sign_flip_cols, block_label, B, seed):
    df_block = df[['country_name', 'year'] + indicator_cols].copy()
    for c in sign_flip_cols:
        if c in indicator_cols:
            df_block[c] = -df_block[c]
    for c in indicator_cols:
        df_block[c] = (df_block[c] - df_block[c].mean()) / df_block[c].std()
    countries = df_block['country_name'].unique().tolist()
    Y_l, ss = [], []
    for cn in countries:
        sub = df_block[df_block['country_name'] == cn].sort_values('year')
        Y_l.append(sub[indicator_cols].values)
        ss.append(len(sub))
    Y = np.vstack(Y_l)
    U, T = len(countries), ss[0]
    N = len(indicator_cols)
    print(f"\n=== Block: {block_label} ===")
    print(f"  Y shape: {Y.shape}, U={U}, T={T}, N={N}")
    print(f"  indicators: {indicator_cols}")

    C = np.corrcoef(Y.T)
    T_cov = offdiag_rank1_fit(C)
    print(f"  T_cov = {T_cov:.4f}")

    ref_indices = np.arange(N)
    print(f"  Running parametric AR(1) null with B={B}, β-clip={BETA_CLIP}...")
    t0 = time.time()
    delta, p, pr_obs, mu, sd, diag = compute_delta_pr_parametric_olsvar(
        Y, ss, ref_indices, B, seed, U_resample=U, T_resample=T,
        beta_clip=BETA_CLIP, return_diagnostics=True
    )
    elapsed = time.time() - t0
    print(f"  Elapsed: {elapsed:.1f}s")

    print(f"  PR_obs={pr_obs:.4f}  μ_null={mu:.4f}  σ_null={sd:.4f}  "
          f"ΔPR={delta:+.4f}  p={p:.4f}  z={(pr_obs-mu)/sd:+.2f}  "
          f"{'PASS' if p < ALPHA else 'FAIL'}")

    n_clipped = sum(1 for b in diag['betas_fit'] if abs(b) > BETA_CLIP)
    print(f"  β: median={np.median(diag['betas_fit']):.4f}, "
          f"range=[{min(diag['betas_fit']):.4f},{max(diag['betas_fit']):.4f}], "
          f"n_clipped={n_clipped}")
    print(f"  obs max|y|={np.abs(Y).max():.2f}, sim max|y| q50/q90/q99="
          f"{diag['sim_max_abs_q'][0]:.2f}/{diag['sim_max_abs_q'][1]:.2f}/{diag['sim_max_abs_q'][2]:.2f}")

    return {
        'block_label': block_label,
        'indicator_cols': indicator_cols,
        'sign_flip_cols': sign_flip_cols,
        'U': U, 'T': T, 'N': N,
        'T_cov': T_cov,
        'PR_obs': pr_obs,
        'null_mean': mu, 'null_std': sd,
        'delta_PR': delta,
        'p_parametric': p,
        'z_score': (pr_obs - mu) / sd if sd > 0 else float('nan'),
        'verdict_at_0.05': 'PASS' if p < ALPHA else 'FAIL',
        'betas_fit': diag['betas_fit'],
        'betas_median': float(np.median(diag['betas_fit'])),
        'sigmas_fit': diag['sigmas_fit'],
        'sim_max_abs_q50_q90_q99': diag['sim_max_abs_q'],
        'observed_panel_max_abs': float(np.abs(Y).max()),
        'elapsed_s': elapsed,
    }


print("Utilities defined.")



# §3. Load V-Dem indicator panel + initial pool


df = pd.read_csv(VDEM_IND_CSV)
df = df.sort_values(['country_name', 'year']).reset_index(drop=True)
print(f"Raw shape: {df.shape}")

# Initial pool (same as v1): drop v2dd* outliers + composite indices
all_ind_cols = [c for c in df.columns if c not in ['country_id', 'country_name', 'year']]
DD_FAMILY = [c for c in all_ind_cols if c.startswith('v2dd')]
COMPOSITE_INDICES = ['v2x_elecoff', 'v2xdd_dd']
ind_pool_initial = [c for c in all_ind_cols if c not in DD_FAMILY + COMPOSITE_INDICES]
print(f"Initial pool (after dropping v2dd* + composite): {len(ind_pool_initial)} indicators")



# §4. Diagnostic: identify near-constant indicators


print("\n=== Near-constant indicator diagnostic ===")
print(f"Threshold: indicator is 'near-constant for a country' if std < 1e-6 within that country")
print(f"Drop indicator if it's near-constant for ≥{NEAR_CONST_THRESHOLD:.0%} of countries")

near_constant_diagnostics = []
for c in ind_pool_initial:
    by_country_std = df.groupby('country_name')[c].std()
    n_constant = (by_country_std < 1e-6).sum()
    n_total = len(by_country_std)
    frac_constant = n_constant / n_total
    if frac_constant > 0.10:  # report any indicator constant in >10% of countries
        # Also compute global frac of country-years at the modal value
        modal_val = df[c].mode().iloc[0]
        frac_at_mode = (df[c] == modal_val).mean()
        near_constant_diagnostics.append({
            'indicator': c,
            'n_countries_constant': int(n_constant),
            'frac_countries_constant': float(frac_constant),
            'modal_value': float(modal_val),
            'frac_at_modal': float(frac_at_mode),
        })

near_constant_df = pd.DataFrame(near_constant_diagnostics).sort_values(
    'frac_countries_constant', ascending=False)
print(f"\nIndicators near-constant for >10% of countries:")
print(near_constant_df.to_string(index=False))

DROP_NEAR_CONST = near_constant_df[
    near_constant_df['frac_countries_constant'] >= NEAR_CONST_THRESHOLD
]['indicator'].tolist()
print(f"\nDROPPING {len(DROP_NEAR_CONST)} near-constant indicators "
      f"(constant for ≥{NEAR_CONST_THRESHOLD:.0%} of countries):")
for c in DROP_NEAR_CONST:
    print(f"  {c}")

ind_pool = [c for c in ind_pool_initial if c not in DROP_NEAR_CONST]
print(f"\nFinal cleaned indicator pool: {len(ind_pool)} indicators")



# §5. Build candidate blocks


# Re-compute PC1 loadings on the cleaned pool
X = df[ind_pool]
Xz = (X - X.mean()) / X.std()
C_pool = Xz.corr().values
eigvals_full, eigvecs = np.linalg.eigh(C_pool)
order = np.argsort(eigvals_full)[::-1]
eigvecs = eigvecs[:, order]
loadings_abs = np.abs(eigvecs[:, 0])
order_pc1 = np.argsort(loadings_abs)[::-1]
TOP13_BY_PC1 = [ind_pool[i] for i in order_pc1[:13]]
print(f"Top 13 by |PC1 loading| from CLEANED pool: {TOP13_BY_PC1}")
print(f"  (compared to v1's top13 — should match since v1 already excluded v2dd* + composite)")

# Top 13 by within-country variance — sensitivity check on selection method
print("\nComputing top 13 by within-country variance...")
within_vars = []
for c in ind_pool:
    within_var = df.groupby('country_name')[c].var().mean()
    within_vars.append((c, within_var))
within_vars_df = pd.DataFrame(within_vars, columns=['indicator', 'within_var']).sort_values(
    'within_var', ascending=False)
TOP13_BY_WITHIN_VAR = within_vars_df.head(13)['indicator'].tolist()
print(f"Top 13 by within-country variance: {TOP13_BY_WITHIN_VAR}")

# Substantive families (filtered to cleaned pool)
ELECTORAL_FULL = [c for c in ind_pool if c.startswith(('v2el', 'v2ps'))]
ELECTORAL_NO_SUFFRAGE = [c for c in ELECTORAL_FULL if c != 'v2elsuffrage']
CIVIL_LIB = [c for c in ind_pool if c.startswith('v2cl')]
JUDICIAL = [c for c in ind_pool if c.startswith('v2ju')]
CIVIL_SOCIETY = [c for c in ind_pool if c.startswith('v2cs')]
MEDIA = [c for c in ind_pool if c.startswith('v2me')]

print(f"\nFamily blocks (filtered to cleaned pool):")
print(f"  electoral_full: N={len(ELECTORAL_FULL)}: {ELECTORAL_FULL}")
print(f"  electoral_no_suffrage: N={len(ELECTORAL_NO_SUFFRAGE)}: {ELECTORAL_NO_SUFFRAGE}")
print(f"  civil_lib: N={len(CIVIL_LIB)}: {CIVIL_LIB}")
print(f"  judicial: N={len(JUDICIAL)}: {JUDICIAL}")
print(f"  civil_society: N={len(CIVIL_SOCIETY)}: {CIVIL_SOCIETY}")
print(f"  media: N={len(MEDIA)}: {MEDIA}")



# §6. Define the 7 blocks for analysis


BLOCKS = [
    ('vdem_ind_top13_v2_pc1',          TOP13_BY_PC1,            []),
    ('vdem_ind_top13_within_var',      TOP13_BY_WITHIN_VAR,     []),
    ('vdem_ind_civil_lib_19',          CIVIL_LIB,                []),
    ('vdem_ind_electoral_no_suffrage', ELECTORAL_NO_SUFFRAGE,    []),
    ('vdem_ind_judicial_4',            JUDICIAL,                 []),
    ('vdem_ind_civil_society_5',       CIVIL_SOCIETY,            []),
    ('vdem_ind_media_6',               MEDIA,                    []),
]

print(f"\nBlocks to run: {len(BLOCKS)}")
for label, cols, _ in BLOCKS:
    print(f"  {label}: N={len(cols)}")



# §7. Run all 7 blocks


results = []
for label, cols, flips in BLOCKS:
    res = run_block_analysis(df, cols, sign_flip_cols=flips,
                              block_label=label, B=B_PERMS, seed=42)
    results.append(res)



# §8. Summary


print("\n" + "=" * 110)
print("V-DEM INDICATOR-LEVEL — Track A robustness summary")
print("=" * 110)
print(f"{'block':35s} {'N':>3s} {'T_cov':>7s} {'PR_obs':>7s} {'ΔPR':>8s} "
      f"{'p_param':>9s} {'z':>7s} {'med_β':>7s} {'verdict':>8s}")
print("-" * 110)
for r in results:
    print(f"{r['block_label']:35s} {r['N']:>3d} {r['T_cov']:>7.4f} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+8.4f} "
          f"{r['p_parametric']:>9.4f} {r['z_score']:>+7.2f} "
          f"{r['betas_median']:>7.3f} "
          f"{r['verdict_at_0.05']:>8s}")

# v1 comparison row(s) for context
print("\nFor v1 comparison (same dataset, before near-constant cleaning):")
print(f"  vdem_ind_top13 (v1):                N=13, ΔPR=+0.461, p=0.001, z=+11.86, PASS")
print(f"  vdem_ind_civil_lib_19 (v1):         N=19, ΔPR=+0.252, p=0.001, z=+10.17, PASS")
print(f"  vdem_ind_electoral_7 (v1):           N=7, ΔPR=+0.055, p=0.249, z=+0.68,  FAIL")

# Cross-substrate comparison
print("\n=== CROSS-SUBSTRATE COMPARISON ===")
print(f"  V-Dem long_16var (INDICES, headline): N=13, T_cov=0.987, PR_obs=0.440, ΔPR=+0.045, p=0.27,  FAIL")
print(f"  V-Dem indicators top13_pc1 (this run): see above — should also PASS")
print(f"  WB-WPP mod_5_wbwpp_clean:             N=5,  T_cov=0.971, PR_obs=0.958, ΔPR=+0.131, p=0.035, PASS")
print(f"  WB-WPP mod_8_signed:                  N=8,  T_cov=0.964, PR_obs=0.947, ΔPR=+0.250, p=0.001, PASS")

print("\n" + "=" * 110)
print("INTERPRETATION GUIDE")
print("=" * 110)
print("""
Robustness checks:
  1. top13_pc1 (cleaned): should match v1's top13 result (z ≈ +12, strong PASS).
     If verdicts differ from v1, near-constant cleanup changed the picture —
     likely insignificantly, but flag if PR_obs or ΔPR shifts noticeably.
  2. top13_within_var: alternative selection method. If both top13 variants
     PASS, the headline isn't an artifact of how we pick the top 13.

Family-level coverage (for §5 of the paper):
  3. civil_lib_19: large family, expected PASS (matches v1).
  4. electoral_no_suffrage: DIAGNOSTIC. v1's electoral_7 FAILed with ΔPR=+0.055.
     If electoral_no_suffrage PASSes, the FAIL was suffrage-driven — a clean
     diagnosis. If still FAIL, electoral indicators have something genuinely
     different (e.g., institutional discrete jumps).
  5. judicial_4: small but T_cov=1.00. May be underpowered (similar concern
     to mod_4_levels_kof). Watch for borderline result.
  6. civil_society_5: T_cov ≈ 1.00, similar size to mod_5_wbwpp_clean which
     PASSed strongly.
  7. media_6: T_cov ≈ 1.00, expect PASS.

Aggregate verdict for the paper:
  If 6/7 or 7/7 blocks PASS → indicator-level V-Dem PASSes ΔPR robustly
  across substantive families. The aggregation hypothesis (§6.3-c) is
  strongly confirmed. Headline restructures around aggregation as the
  identified mechanism.

  If electoral still FAILs even without suffrage → suffrage isn't the
  full story; electoral indicators have intrinsically different dynamics
  worth discussing. Doesn't contradict the headline (which rests on
  top13 + family coverage), but adds substantive nuance.
""")

with open(os.path.join(OUT_DIR, 'vdem_indicators_viability_results_v2.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to vdem_indicators_viability_results_v2.json")

### Cell §5.5 — V-Dem apples-to-apples matched-panel construction: exploratory

**Paper sections:** Not in paper directly; superseded by §5.6.

Within-V-Dem contrast on the matched 89-country × 55-year support, comparing the V-Dem-16 indices block to the V-Dem top-13 indicators block. Used during development to check that aggregation alone (not country-year coverage) drives the index-vs-indicator divergence on V-Dem.

**Inputs:** V-Dem indicators + V-Dem indices CSVs.
-  V-Dem indices: /content/drive/MyDrive/NeurIPS2026_1296/data/my_data_75_years.csv
-  V-Dem indicators: /content/drive/MyDrive/NeurIPS2026_1296/data/my_data_VDem_Ind.csv

**Outputs:** Apples-to-apples results JSON.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'vdem_apples_to_apples_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

INDICES_CSV    = os.path.join(DATA_DIR, 'my_data_75_years.csv')
INDICATORS_CSV = os.path.join(DATA_DIR, 'my_data_VDem_Ind.csv')



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05
BETA_CLIP = 0.99
NEAR_CONST_THRESHOLD = 0.50

print(f"B_PERMS={B_PERMS}, BETA_CLIP={BETA_CLIP}")



# §2. Core utilities (same as Block 14)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs = Yt.shape[0]; N = Yt.shape[1]
    X = np.column_stack([Ylag, np.ones(n_obs)])
    try:
        XtX_inv = np.linalg.inv(X.T @ X)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(X.T @ X)
    A = np.zeros((N, N)); p_values = np.ones((N, N))
    for i in range(N):
        y = Yt[:, i]
        beta = XtX_inv @ X.T @ y
        resid = y - X @ beta
        dof = n_obs - (N + 1)
        if dof <= 0:
            A[i, :] = beta[:N]; continue
        sigma2 = float(resid @ resid) / dof
        cov_beta = sigma2 * XtX_inv
        se = np.sqrt(np.clip(np.diag(cov_beta), 1e-12, None))
        t_stats = beta / se
        p_vals = 2.0 * scstats.t.sf(np.abs(t_stats), dof)
        A[i, :] = beta[:N]
        p_values[i, :] = p_vals[:N]
    np.fill_diagonal(A, 0.0)
    return A, p_values


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng, beta_clip=BETA_CLIP):
    N = len(betas)
    betas_clipped = np.clip(betas, -beta_clip, beta_clip)
    stationary_var = sigmas**2 / (1 - betas_clipped**2 + 1e-12)
    stationary_sd = np.sqrt(np.maximum(stationary_var, 1e-12))
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas_clipped[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric_olsvar(Y, split_spec, ref_indices, B, seed,
                                       U_resample, T_resample, beta_clip=BETA_CLIP):
    A, _ = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return None
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample,
                                                rng, beta_clip=beta_clip)
        ss_sim = [T_resample] * U_resample
        A_sim, _ = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    valid = np.isfinite(null_PRs); null_PRs = null_PRs[valid]
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return {
        'PR_obs': float(PR_obs),
        'null_mean': mu, 'null_std': sd,
        'delta_PR': float(delta),
        'p_parametric': float(p),
        'z_score': float((PR_obs - mu) / sd) if sd > 0 else float('nan'),
        'verdict_at_0.05': 'PASS' if p < ALPHA else 'FAIL',
        'betas_fit': betas.tolist(),
        'betas_median': float(np.median(betas)),
    }


print("Utilities defined.")



# §3. Load both files and construct matched panels


df_indices_raw = pd.read_csv(INDICES_CSV)
# The indices file uses 'country_id' as the country-name column (string)
df_indices_raw = df_indices_raw.rename(columns={'country_id': 'country_name'})

df_indicators_raw = pd.read_csv(INDICATORS_CSV)

print(f"Indices file (long_16var): {df_indices_raw.shape}")
print(f"  Countries: {df_indices_raw['country_name'].nunique()}, "
      f"years: {df_indices_raw['year'].min()}-{df_indices_raw['year'].max()}")
print(f"Indicators file: {df_indicators_raw.shape}")
print(f"  Countries: {df_indicators_raw['country_name'].nunique()}, "
      f"years: {df_indicators_raw['year'].min()}-{df_indicators_raw['year'].max()}")

# Match: countries that appear in BOTH files; years that appear in BOTH year ranges
common_countries = sorted(
    set(df_indices_raw['country_name'].unique()) &
    set(df_indicators_raw['country_name'].unique())
)
common_years = sorted(
    set(df_indices_raw['year'].unique()) &
    set(df_indicators_raw['year'].unique())
)
print(f"\nMatched panel:")
print(f"  Countries (intersection): {len(common_countries)}")
print(f"  Years (intersection): {len(common_years)} from {min(common_years)} to {max(common_years)}")

# Subset both files to matched panel
df_indices = df_indices_raw[
    df_indices_raw['country_name'].isin(common_countries) &
    df_indices_raw['year'].isin(common_years)
].copy().sort_values(['country_name', 'year']).reset_index(drop=True)
df_indicators = df_indicators_raw[
    df_indicators_raw['country_name'].isin(common_countries) &
    df_indicators_raw['year'].isin(common_years)
].copy().sort_values(['country_name', 'year']).reset_index(drop=True)

# Confirm balanced
assert df_indices.groupby('country_name').size().nunique() == 1, "indices not balanced"
assert df_indicators.groupby('country_name').size().nunique() == 1, "indicators not balanced"
n_per_c = df_indices.groupby('country_name').size().iloc[0]
print(f"  Rows per country: {n_per_c}")
print(f"  Total rows: {len(df_indices)} (indices) and {len(df_indicators)} (indicators)")
assert len(df_indices) == len(df_indicators), "row count mismatch"



# §4. Build INDICES block (long_16var size-13 cov-rank-1)
#
# The headline V-Dem analysis identifies a single reflective block of size
# 13 from the 16 indices via the cov-rank-1 procedure. Three indices are
# excluded as singletons (low loadings, high idiosyncratic variance).
# The standard exclusion set in the original analysis is:
#   - Suffrage (near-constant — most countries have universal suffrage)
#   - Direct_vote (rare event in most countries)
#   - Regional_government (limited cross-country variation)


# All 16 V-Dem indices in long_16var
ALL_16_INDICES = [
    'Freedom_of_expression', 'Freedom_of_association', 'Suffrage',
    'Clean_elections', 'Elected_officials', 'Individual_liberty',
    'Judicial_constraints', 'Legislative_constraints',
    'Civil_participation', 'Direct_vote', 'Local_government',
    'Regional_government', 'Deliberative', 'Equal_access',
    'Equal_distribution', 'Equal_protection',
]

# The size-13 cov-rank-1 block from the headline analysis
SIZE13_INDICES = [c for c in ALL_16_INDICES
                  if c not in ['Suffrage', 'Direct_vote', 'Regional_government']]
print(f"Size-13 indices block: N={len(SIZE13_INDICES)}")
print(f"  {SIZE13_INDICES}")

# Diagnostic: T_cov on the 16-index correlation, then on the size-13 subset
X16 = df_indices[ALL_16_INDICES]
X13 = df_indices[SIZE13_INDICES]
C16 = ((X16 - X16.mean()) / X16.std()).corr().values
C13 = ((X13 - X13.mean()) / X13.std()).corr().values
print(f"\nT_cov on all 16 indices:  {offdiag_rank1_fit(C16):.4f}")
print(f"T_cov on size-13 subset:  {offdiag_rank1_fit(C13):.4f}")
print(f"  (headline analysis reported 0.987 on the full 75-year panel)")



# §5. Build INDICATORS block (top-13 by |PC1 loading| on matched pool)


# Build the cleaned indicator pool (same procedure as Block 14 v2)
all_ind_cols = [c for c in df_indicators.columns
                if c not in ['country_id', 'country_name', 'year']]
DD_FAMILY = [c for c in all_ind_cols if c.startswith('v2dd')]
COMPOSITE_INDICES = ['v2x_elecoff', 'v2xdd_dd']
ind_pool_initial = [c for c in all_ind_cols
                    if c not in DD_FAMILY + COMPOSITE_INDICES]

# Identify near-constant indicators (constant for >= NEAR_CONST_THRESHOLD of countries)
DROP_NEAR_CONST = []
for c in ind_pool_initial:
    by_country_std = df_indicators.groupby('country_name')[c].std()
    frac_const = (by_country_std < 1e-6).mean()
    if frac_const >= NEAR_CONST_THRESHOLD:
        DROP_NEAR_CONST.append(c)
print(f"Near-constant indicators dropped (constant for ≥{NEAR_CONST_THRESHOLD:.0%} of countries): "
      f"{len(DROP_NEAR_CONST)}")
for c in DROP_NEAR_CONST:
    print(f"  {c}")

ind_pool = [c for c in ind_pool_initial if c not in DROP_NEAR_CONST]
print(f"Final cleaned indicator pool: {len(ind_pool)} indicators")

# PC1 loadings on the matched panel
X_pool = df_indicators[ind_pool]
Xz_pool = (X_pool - X_pool.mean()) / X_pool.std()
C_pool = Xz_pool.corr().values
eigvals_full, eigvecs = np.linalg.eigh(C_pool)
order = np.argsort(eigvals_full)[::-1]
eigvecs = eigvecs[:, order]
loadings_abs = np.abs(eigvecs[:, 0])
order_pc1 = np.argsort(loadings_abs)[::-1]
TOP13_INDICATORS = [ind_pool[i] for i in order_pc1[:13]]
print(f"\nTop 13 indicators by |PC1 loading| on matched panel: N={len(TOP13_INDICATORS)}")
print(f"  {TOP13_INDICATORS}")

# T_cov diagnostic
X13ind = df_indicators[TOP13_INDICATORS]
C13ind = ((X13ind - X13ind.mean()) / X13ind.std()).corr().values
print(f"T_cov on top-13 indicators: {offdiag_rank1_fit(C13ind):.4f}")



# §6. Run both blocks


def run_block(df, indicator_cols, block_label, B, seed):
    df_block = df[['country_name', 'year'] + indicator_cols].copy()
    for c in indicator_cols:
        df_block[c] = (df_block[c] - df_block[c].mean()) / df_block[c].std()
    countries = df_block['country_name'].unique().tolist()
    Y_l, ss = [], []
    for cn in countries:
        sub = df_block[df_block['country_name'] == cn].sort_values('year')
        Y_l.append(sub[indicator_cols].values)
        ss.append(len(sub))
    Y = np.vstack(Y_l)
    U, T = len(countries), ss[0]
    N = len(indicator_cols)
    print(f"\n=== Block: {block_label} ===")
    print(f"  Y shape: {Y.shape}, U={U}, T={T}, N={N}")

    C = np.corrcoef(Y.T)
    T_cov = offdiag_rank1_fit(C)
    print(f"  T_cov = {T_cov:.4f}")

    ref_indices = np.arange(N)
    print(f"  Running parametric AR(1) null with B={B}, β-clip={BETA_CLIP}...")
    t0 = time.time()
    res = compute_delta_pr_parametric_olsvar(
        Y, ss, ref_indices, B, seed, U_resample=U, T_resample=T,
        beta_clip=BETA_CLIP
    )
    elapsed = time.time() - t0

    print(f"  Elapsed: {elapsed:.1f}s")
    print(f"  PR_obs = {res['PR_obs']:.4f}")
    print(f"  ΔPR    = {res['delta_PR']:+.4f}")
    print(f"  p      = {res['p_parametric']:.4f}")
    print(f"  z      = {res['z_score']:+.2f}")
    print(f"  median β = {res['betas_median']:.4f}")
    print(f"  verdict = {res['verdict_at_0.05']}")

    return {
        'block_label': block_label,
        'indicator_cols': indicator_cols,
        'U': U, 'T': T, 'N': N,
        'T_cov': T_cov,
        **res,
        'elapsed_s': elapsed,
    }


print("=" * 80)
print("V-DEM APPLES-TO-APPLES — matched 89-country × 55-year panel")
print("=" * 80)

result_indices = run_block(
    df_indices, SIZE13_INDICES,
    block_label='vdem_indices_size13_matched',
    B=B_PERMS, seed=42
)
result_indicators = run_block(
    df_indicators, TOP13_INDICATORS,
    block_label='vdem_indicators_top13pc1_matched',
    B=B_PERMS, seed=42
)



# §7. Summary


print("\n" + "=" * 100)
print("V-DEM APPLES-TO-APPLES SUMMARY")
print("=" * 100)
print(f"All on matched panel: 89 countries × 55 years (1970-2024), 4895 country-years")
print(f"{'block':40s} {'N':>3s} {'T_cov':>7s} {'PR_obs':>7s} {'ΔPR':>8s} "
      f"{'p_param':>9s} {'z':>7s} {'med_β':>7s} {'verdict':>8s}")
print("-" * 100)
for r in [result_indices, result_indicators]:
    print(f"{r['block_label']:40s} {r['N']:>3d} {r['T_cov']:>7.4f} "
          f"{r['PR_obs']:>7.4f} {r['delta_PR']:>+8.4f} "
          f"{r['p_parametric']:>9.4f} {r['z_score']:>+7.2f} "
          f"{r['betas_median']:>7.3f} "
          f"{r['verdict_at_0.05']:>8s}")

print("\nFor reference (NON-matched, original panel sizes):")
print(f"  long_16var (89c × 75y, full headline):           "
      f"PR_obs=0.440, ΔPR=+0.045, p=0.27,  FAIL")
print(f"  V-Dem indicators top13_pc1 (152c × 56y):         "
      f"PR_obs=0.762, ΔPR=+0.461, p=0.001, PASS")

print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)
verdict_indices_match = (result_indices['verdict_at_0.05'] == 'FAIL')
verdict_indicators_match = (result_indicators['verdict_at_0.05'] == 'PASS')
if verdict_indices_match and verdict_indicators_match:
    print("""
Both verdicts match the non-matched analyses. The within-V-Dem contrast
is fully apples-to-apples: same 89 countries, same 55-year period, only
the aggregation step differs. Indices FAIL, indicators PASS.

This closes the country-set/year-range alternative explanations:
  - It is NOT that indices come from a smaller country set with different
    political dynamics. Same country set as indicators on the matched panel.
  - It is NOT that indices use a different time period with different
    structural-break density. Same 55-year period.
  - It IS the aggregation step. The IRT-aggregation pipeline that
    converts raw expert codings into V-Dem indices disrupts the rank-one
    reduced-form structure that latent-factor measurement theory predicts.
""")
elif verdict_indices_match and not verdict_indicators_match:
    print("""
Indices verdict matches (FAIL) but indicators verdict differs from the
non-matched analysis. The indicator PASS may be sensitive to country/year
set. Investigate further — possibly the 89-country subset has different
indicator-level dynamics than the full 152-country set.
""")
elif not verdict_indices_match:
    print("""
Indices verdict on matched panel differs from headline. The indices FAIL
may have been period- or country-specific. Need to investigate which
countries/years drive the change in verdict.
""")

# Save
output = {
    'matched_panel_specs': {
        'n_countries': len(common_countries),
        'n_years': len(common_years),
        'year_range': [int(min(common_years)), int(max(common_years))],
        'common_countries': common_countries,
        'common_years': [int(y) for y in common_years],
    },
    'indices_result': result_indices,
    'indicators_result': result_indicators,
    'top13_indicators_chosen': TOP13_INDICATORS,
    'size13_indices_chosen': SIZE13_INDICES,
    'near_const_dropped': DROP_NEAR_CONST,
}
out_path = os.path.join(OUT_DIR, 'vdem_apples_to_apples_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

### Cell §5.6 — Multi-criteria pipeline (CANONICAL): EDA → cov-rank-1 → 5-method evaluation

**Paper sections:** §5, Appendix P.

The canonical principled pipeline used for both WB-WPP and V-Dem-60 substrates. Replaces the hardcoded-substrates approach of §5.1-§5.5 with: load CSV → EDA cleaning (Conditions 1-4 from Appendix Q) → cov-rank-1 partition into reflective blocks → for each block, run the 5-method multi-criteria evaluation. Produces the WB-WPP and V-Dem-60 rows of Table 3.

Substrates here are *derived* from the data, not specified in advance. For each panel:
  1. Load CSV
  2. Apply EDA cleaning (drop indicators flagged by Conditions 1-4 of Appendix C):
       - Condition 1: regime-shifted (joint stat+practical)
       - Condition 2: saturated (>30% countries flat)
       - Condition 3b: outlier-drop (max|z| > 2× expected)
       - Condition 4: one of each collinear pair (|r| > 0.99)
  3. Run cov-rank-1 to partition cleaned indicators into reflective blocks
  4. For each block of size >= 2, run the 5-method multi-criteria pipeline
     (CRPS, rb-beats, ΔPR with both nulls, SRMR, RMSEA)

**Outputs per panel:**
 - multi_criteria_outputs/{panel_name}/
 -   eda_drops.json           — list of dropped indicators with reasons
 -   covrank1_blocks.json     — block assignments + T_cov per block
 -   sweep_raw.csv            — per (block, method, W_label, protocol) row
 -   method_outputs.pkl       — per-block, per-method full fit dicts
  -  multi_criteria_summary.csv — assembled per-block per-method table

**Configurable panels (set PANEL_TO_RUN below):**
 - 'long_16var'        : V-Dem indices    (already long-since processed; here for completeness)
 - 'vdem_indicators'   : V-Dem 70-indicator panel
 - 'wb_wpp'            : WB-WPP demographic panel
 - 'all'               : all three sequentially

**Inputs:** `csv_path` for the panel of interest. Configurable via `PANEL_TO_RUN ∈ {'long_16var', 'vdem_indicators', 'wb_wpp', 'all'}`.

**Self-contained:** Yes.

**Runtime:** ~7 hours (if PANELS_TO_RUN='all') on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, pickle, time, json
import numpy as np
import pandas as pd

DRIVE_DIR    = '/content/drive/MyDrive/NeurIPS2026_1296'
SRC_DIR      = os.path.join(DRIVE_DIR, 'src')
DATA_DIR     = os.path.join(DRIVE_DIR, 'data')
OUT_DIR_BASE = os.path.join(DRIVE_DIR, 'multi_criteria_outputs')
os.makedirs(OUT_DIR_BASE, exist_ok=True)


# Configuration


# Which panel to run. Pick one of:
#   'long_16var'      — V-Dem indices (16 → block of 13)
#   'vdem_indicators' — V-Dem indicators (70 → cleaned + block-identified)
#   'wb_wpp'          — WB-WPP (21 → cleaned + block-identified)
#   'all'             — all three sequentially (smallest first)
PANEL_TO_RUN = 'vdem_indicators'

# rb-beats baselines per family. With plus-one correction, min p_emp = 1/(N+1).
# At N=50, min p_emp = 0.020 (achieves α=0.05 with margin).
N_REPS_PER_FAMILY = 50

METHODS_TO_RUN = ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears']
SEED = 0
METHOD_RNG_OFFSETS = {
    'linear_var_granger': 0, 'pcmci': 1, 'cmlp':  2, 'navar': 3, 'dynotears': 4,
}

# CRPS pipeline constants (match the source script for cross-substrate
# comparability; re-asserted again after exec since the source clobbers them)
H_HORIZON, B_BOOT, GRID_SIZE, EPSILON, BANDWIDTH, RIDGE = 20, 300, 50, 0.01, 0.2, 1e-4

# Cov-rank-1 threshold (paper convention)
COV_RANK1_THRESHOLD = 0.7

# Parametric AR(1) null bootstrap reps (for ΔPR p-values)
NULL_B = 1000



# §1. Panel specifications (just CSV + ID + year; no variable lists)


# NOTE: We use NEW_PANEL_SPECS (not PANEL_SPECS) to avoid being overwritten by
# the exec'd source script, which defines its own PANEL_SPECS dict.
NEW_PANEL_SPECS = {
    'long_16var': {
        'csv_filename':    'my_data_75_years.csv',
        'id_col':           'country_id',
        'year_col':         'year',
        'variables':        [   # V-Dem indices (16) — fixed, hand-curated panel
            "Freedom_of_expression", "Freedom_of_association", "Suffrage",
            "Clean_elections", "Elected_officials", "Individual_liberty",
            "Judicial_constraints", "Legislative_constraints",
            "Civil_participation", "Direct_vote", "Local_government",
            "Regional_government", "Deliberative", "Equal_access",
            "Equal_distribution", "Equal_protection",
        ],
        'min_series_length': 9,
        'val_years':        5,
    },
    'vdem_indicators': {
        'csv_filename':    'my_data_VDem_Ind.csv',
        'id_col':           'country_name',
        'year_col':         'year',
        'variables':        None,    # auto-detect: all numeric columns except id/year
        'min_series_length': 9,
        'val_years':        5,
    },
    'wb_wpp': {
        'csv_filename':    'my_data_WB.csv',
        'id_col':           'country_name',
        'year_col':         'year',
        'variables':        None,    # auto-detect
        'min_series_length': 9,
        'val_years':        5,
    },
}



# §2. Load random_baselines pipeline functions


PIPELINE_PATH = os.path.join(SRC_DIR, 'random_baselines_4method_corrected.py')
if not os.path.exists(PIPELINE_PATH):
    raise FileNotFoundError(
        f"{PIPELINE_PATH} not found. Upload random_baselines_4method_corrected.py "
        f"(with the lambda_group) to your src directory.")

with open(PIPELINE_PATH) as fh:
    pipeline_src = fh.read()

import re
# Cutoff: include all function definitions through §8 (score_one_W),
# stop before §9 (main sweep) which would actually run the V-Dem-only fit loop.
# Line 1167 = score_one_W def; line 1207 = §9 main sweep code cell.
LINES_TO_EXEC_END = 1207
pipeline_lines = pipeline_src.splitlines(keepends=True)
header_only = ''.join(pipeline_lines[:LINES_TO_EXEC_END])

# Strip the panel load that hard-codes V-Dem long_16var
header_only = re.sub(
    r"Y_train, split_spec, var_names = load_vdem_panel\(CONFIG_NAME\).*?Y_norm: shape=\{Y_norm\.shape\}\"\)",
    "# (panel load deferred)",
    header_only, flags=re.DOTALL,
)
# Comment out IPython magics (! commands) that break under exec
header_only = re.sub(
    r"^(\s*)(!.*)$",
    r"\1# \2",
    header_only,
    flags=re.MULTILINE,
)
# Comment out "from google.colab import drive" and drive.mount() lines
# (we already mounted at the top of this cell)
header_only = re.sub(
    r"^(\s*)(from google\.colab.*|drive\.mount\(.*\))$",
    r"\1# \2",
    header_only,
    flags=re.MULTILINE,
)

future_imports = []
def _capture_future(match):
    future_imports.append(match.group(0).rstrip())
    return ""    # remove from original location

header_only = re.sub(
    r"^\s*from __future__ import [^\n]*\n",
    _capture_future,
    header_only,
    flags=re.MULTILINE,
)
if future_imports:
    # Deduplicate while preserving order
    seen = set(); unique = []
    for f in future_imports:
        if f not in seen:
            seen.add(f); unique.append(f)
    header_only = "\n".join(unique) + "\n\n" + header_only
    print(f"Hoisted {len(unique)} __future__ import(s) to top of exec block")

exec(header_only, globals())
print("Pipeline functions loaded.")

# The exec'd source script defines its own values for several module-level
# globals (PANEL_SPECS, N_REPS_PER_FAMILY, OUT_DIR_BASE, H_HORIZON, B_BOOT,
# GRID_SIZE, BANDWIDTH, EPSILON, RIDGE, SEED, METHODS_TO_RUN, METHOD_RNG_OFFSETS,
# DRIVE_DIR, SRC_DIR, DATA_DIR). These overwrite the values we set at the top
# of this cell. Re-assert our values now to take precedence for downstream code.
DRIVE_DIR    = '/content/drive/MyDrive/NeurIPS2026_1296'
SRC_DIR      = os.path.join(DRIVE_DIR, 'src')
DATA_DIR     = os.path.join(DRIVE_DIR, 'data')
OUT_DIR_BASE = os.path.join(DRIVE_DIR, 'multi_criteria_outputs')
os.makedirs(OUT_DIR_BASE, exist_ok=True)

N_REPS_PER_FAMILY = 50
METHODS_TO_RUN = ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears']
SEED = 0
METHOD_RNG_OFFSETS = {
    'linear_var_granger': 0, 'pcmci': 1, 'cmlp':  2, 'navar': 3, 'dynotears': 4,
}
# Match the source script's CRPS pipeline values so results are comparable
# with the v2 paper's long_16var numbers
H_HORIZON, B_BOOT, GRID_SIZE, EPSILON, BANDWIDTH, RIDGE = 20, 300, 50, 0.01, 0.2, 1e-4

print(f"Re-asserted run config: OUT_DIR_BASE={OUT_DIR_BASE}, N_REPS={N_REPS_PER_FAMILY}, "
      f"H={H_HORIZON}, B={B_BOOT}")



# §3. Cov-rank-1 — try import, fall back to from-scratch


LSC_PATH = os.path.join(SRC_DIR, 'lsc_synthetic_validation.py')

try:
    if os.path.exists(LSC_PATH):
        with open(LSC_PATH) as fh:
            lsc_src = fh.read()
        exec(lsc_src, globals())
        # Try the most likely function names
        if 'cov_rank_1_groups' in globals():
            cov_rank_1 = cov_rank_1_groups
            print("Imported cov_rank_1_groups from lsc_synthetic_validation.py")
        elif 'identify_blocks_cov_rank_1' in globals():
            cov_rank_1 = identify_blocks_cov_rank_1
            print("Imported identify_blocks_cov_rank_1 from lsc_synthetic_validation.py")
        elif 'cov_rank_1' in globals():
            print("Imported cov_rank_1 from lsc_synthetic_validation.py")
        else:
            raise NameError("No cov-rank-1 function found in lsc_synthetic_validation.py")
    else:
        raise FileNotFoundError(LSC_PATH)
except (FileNotFoundError, NameError, SyntaxError) as e:
    print(f"  Could not import cov_rank_1 ({e}); using fallback implementation")

    def _fit_rank1_als(C_block, max_iter=200, tol=1e-8):
        """Alternating-least-squares rank-1 fit to a square matrix block.
        Returns (u, v, fit_score) where fit_score ∈ [0,1] is the fraction of
        ||C_block||_F^2 explained by u v^T.
        """
        C = np.asarray(C_block, dtype=float)
        n = C.shape[0]
        # Initialize from leading eigenvector of C C^T
        try:
            eigvals, eigvecs = np.linalg.eigh(C @ C.T)
            u = eigvecs[:, -1]
        except np.linalg.LinAlgError:
            u = np.ones(n) / np.sqrt(n)
        u = u / (np.linalg.norm(u) + 1e-12)
        prev_loss = np.inf
        for _ in range(max_iter):
            v = C.T @ u / (u @ u + 1e-12)
            u = C @ v / (v @ v + 1e-12)
            recon = np.outer(u, v)
            loss = np.linalg.norm(C - recon, 'fro') ** 2
            if abs(prev_loss - loss) < tol * (prev_loss + 1e-12):
                break
            prev_loss = loss
        total_var = np.linalg.norm(C, 'fro') ** 2
        fit_score = 1.0 - (loss / (total_var + 1e-12))
        return u, v, max(0.0, min(1.0, fit_score))

    def cov_rank_1(Y, threshold=COV_RANK1_THRESHOLD, max_block_search=None,
                    verbose=True):
        """Identify reflective blocks via cov-rank-1 procedure.

        Procedure (matches Appendix A.4 description):
          1. Compute empirical correlation matrix C of indicators (cross-sectional)
          2. Greedy/agglomerative block identification: start with each indicator
             as a singleton; merge greedily by largest off-diagonal correlation;
             at each candidate merge, check rank-1 ALS fit on the candidate
             block's correlation sub-matrix; accept merge iff fit_score >= threshold
          3. Return: list of blocks (each a list of indicator indices) + per-block
             T_cov (the rank-1 ALS fit score)

        Returns: dict {
          'blocks': [list of indicator-index lists, sorted descending by size],
          'block_T_cov': [rank-1 fit score per block],
          'singletons': [indicator indices not in any block of size >= 2],
          'block_indicator_names': [list of name lists] if Y has var_names,
        }
        """
        Y = np.asarray(Y, dtype=float)
        n_obs, N = Y.shape
        # Compute correlation matrix
        Y_c = (Y - Y.mean(axis=0)) / (Y.std(axis=0) + 1e-12)
        C = (Y_c.T @ Y_c) / max(n_obs - 1, 1)
        np.fill_diagonal(C, 1.0)

        # Greedy agglomerative: at each step, find candidate merge that maximizes
        # rank-1 fit on the resulting block, accept iff above threshold.
        active_blocks = [[i] for i in range(N)]
        block_fits = [1.0] * N   # singletons trivially rank-1
        improved = True
        round_idx = 0
        while improved and len(active_blocks) >= 2:
            improved = False
            best_merge = None
            best_fit = threshold
            for i in range(len(active_blocks)):
                for j in range(i + 1, len(active_blocks)):
                    candidate = active_blocks[i] + active_blocks[j]
                    if len(candidate) > N:
                        continue
                    sub = C[np.ix_(candidate, candidate)]
                    _, _, fit = _fit_rank1_als(sub)
                    if fit > best_fit:
                        best_fit = fit
                        best_merge = (i, j, candidate, fit)
            if best_merge is not None:
                i, j, candidate, fit = best_merge
                # Replace the two blocks with the merged one
                new_blocks = [b for k, b in enumerate(active_blocks) if k not in (i, j)]
                new_fits   = [f for k, f in enumerate(block_fits)    if k not in (i, j)]
                new_blocks.append(candidate)
                new_fits.append(fit)
                active_blocks = new_blocks
                block_fits = new_fits
                improved = True
                round_idx += 1
                if verbose and round_idx % 5 == 0:
                    sizes = sorted([len(b) for b in active_blocks], reverse=True)
                    print(f"  cov_rank_1 round {round_idx}: {len(active_blocks)} blocks, "
                          f"sizes (top 5): {sizes[:5]}")

        # Sort by size descending
        order = sorted(range(len(active_blocks)),
                       key=lambda k: -len(active_blocks[k]))
        blocks = [active_blocks[k] for k in order]
        block_fits_sorted = [block_fits[k] for k in order]

        non_singleton = [b for b in blocks if len(b) >= 2]
        non_singleton_fits = [block_fits_sorted[k] for k, b in enumerate(blocks) if len(b) >= 2]
        singletons = sum([b for b in blocks if len(b) < 2], [])
        return {
            'blocks':       non_singleton,
            'block_T_cov':  non_singleton_fits,
            'singletons':   singletons,
        }

    print("  Fallback cov_rank_1 ready.")



# §4. EDA cleaning


# Re-implement the relevant condition checks from eda_conditions_verification_v2.py
# (Conditions 1, 2, 3b, 4 — the "drop" conditions)

SUP_WALD_CV_ONE_PARAM = 8.85
TRIM_FRACTION         = 0.15
BETA_SHIFT_THRESHOLD  = 0.05
SIGMA_RATIO_THRESHOLD = 1.5
SATURATION_THRESHOLD  = 0.30
NEAR_CONSTANT_EPS     = 1e-6
OUTLIER_DROP_RATIO    = 2.0
COLLINEARITY_THRESHOLD = 0.99


def _expected_max_abs_z(N):
    if N < 2:
        return np.nan
    return np.sqrt(2 * np.log(N)) - (np.log(np.log(N)) + np.log(4 * np.pi)) / (2 * np.sqrt(2 * np.log(N)))


def _sup_wald_with_effect_size(values_by_country, trim_frac=0.15):
    all_y, all_ylag, all_t = [], [], []
    for vals in values_by_country.values():
        if len(vals) < 5:
            continue
        for i in range(1, len(vals)):
            all_y.append(vals[i]); all_ylag.append(vals[i-1]); all_t.append(i)
    if len(all_y) < 20:
        return None
    y, ylag, t = np.array(all_y), np.array(all_ylag), np.array(all_t)
    T_max = t.max()
    trim_lo = int(np.ceil(trim_frac * T_max))
    trim_hi = int(np.floor((1 - trim_frac) * T_max))

    def _ols(yy, xx):
        xc = xx - xx.mean(); yc = yy - yy.mean()
        denom = (xc**2).sum()
        if denom < 1e-12: return None
        b = (xc * yc).sum() / denom
        resid = yc - b * xc
        sigma2 = (resid**2).sum() / max(len(yy) - 2, 1)
        sigma = np.sqrt(sigma2) if sigma2 > 0 else None
        se = np.sqrt(sigma2 / denom) if denom > 0 else np.inf
        return b, se, sigma

    sup_w = -np.inf; best = None
    for tb in range(trim_lo, trim_hi + 1):
        m_pre = t <= tb; m_post = t > tb
        if m_pre.sum() < 5 or m_post.sum() < 5: continue
        r_pre = _ols(y[m_pre], ylag[m_pre]); r_post = _ols(y[m_post], ylag[m_post])
        if r_pre is None or r_post is None: continue
        b_pre, se_pre, s_pre = r_pre; b_post, se_post, s_post = r_post
        if not (np.isfinite(b_pre) and np.isfinite(b_post)): continue
        var_diff = se_pre**2 + se_post**2
        if var_diff < 1e-12: continue
        wald = (b_pre - b_post)**2 / var_diff
        if wald > sup_w:
            sup_w = wald
            best = {'b_pre': b_pre, 'b_post': b_post, 's_pre': s_pre, 's_post': s_post}
    if best is None: return None
    abs_db = abs(best['b_pre'] - best['b_post'])
    if best['s_pre'] is not None and best['s_post'] is not None:
        s_min = min(best['s_pre'], best['s_post']); s_max = max(best['s_pre'], best['s_post'])
        sigma_ratio = s_max / s_min if s_min > 1e-9 else np.inf
    else:
        sigma_ratio = np.nan
    return {'sup_wald': float(sup_w), 'abs_beta_shift': float(abs_db),
            'sigma_ratio': float(sigma_ratio)}


def eda_clean_indicators(df, indicators, country_col='country_name'):
    """Apply Conditions 1 (regime, joint), 2 (saturation), 3b (outlier drop),
    4 (collinearity) of Appendix C. Returns:
      kept (list), dropped (dict {indicator: reason})
    """
    n_total = len(df)
    expected_max_z = _expected_max_abs_z(n_total)

    dropped = {}
    keep = []

    # Conditions 1, 2, 3b: per-indicator
    for ind in indicators:
        if ind not in df.columns:
            dropped[ind] = "MISSING from CSV"
            continue
        # Condition 2: saturation
        sat_frac = float((df.groupby(country_col)[ind].std() < NEAR_CONSTANT_EPS).mean())
        if sat_frac > SATURATION_THRESHOLD:
            dropped[ind] = f"saturation ({sat_frac:.0%} flat)"
            continue
        # Condition 3b: outlier drop
        vals = df[ind].values
        z = (vals - vals.mean()) / (vals.std() + 1e-12)
        max_z = float(np.abs(z).max())
        z_ratio = max_z / expected_max_z if expected_max_z > 0 else np.nan
        if z_ratio > OUTLIER_DROP_RATIO:
            dropped[ind] = f"outlier (max|z|={max_z:.1f}, ratio={z_ratio:.2f})"
            continue
        # Condition 1: regime shift (joint stat+practical)
        vals_by_country = {}
        for c, sub in df.groupby(country_col):
            v = sub.sort_values('year')[ind].values
            if len(v) >= 5 and np.std(v) > 1e-8:
                vals_by_country[c] = v
        rs = _sup_wald_with_effect_size(vals_by_country)
        if rs is not None:
            stat_sig = rs['sup_wald'] > SUP_WALD_CV_ONE_PARAM
            practical = (rs['abs_beta_shift'] > BETA_SHIFT_THRESHOLD) or \
                        (np.isfinite(rs['sigma_ratio']) and rs['sigma_ratio'] > SIGMA_RATIO_THRESHOLD)
            if stat_sig and practical:
                dropped[ind] = (f"regime shift (sup-W={rs['sup_wald']:.1f}, "
                                f"|Δβ|={rs['abs_beta_shift']:.3f}, σr={rs['sigma_ratio']:.2f})")
                continue
        keep.append(ind)

    # Condition 4: collinearity. Drop one of each pair |r| > 0.99.
    # Strategy: keep the one with greater within-country variation.
    if len(keep) >= 2:
        corr = df[keep].corr().values
        idx_to_drop = set()
        for i in range(len(keep)):
            if i in idx_to_drop: continue
            for j in range(i+1, len(keep)):
                if j in idx_to_drop: continue
                if abs(corr[i, j]) > COLLINEARITY_THRESHOLD:
                    # Drop the one with smaller within-country variation
                    wv_i = df.groupby(country_col)[keep[i]].std().mean()
                    wv_j = df.groupby(country_col)[keep[j]].std().mean()
                    drop_idx = j if wv_j <= wv_i else i
                    idx_to_drop.add(drop_idx)
                    other_idx = i if drop_idx == j else j
                    dropped[keep[drop_idx]] = (f"collinear with {keep[other_idx]} "
                                                f"(r={corr[i,j]:+.3f})")
        keep = [k for idx, k in enumerate(keep) if idx not in idx_to_drop]

    return keep, dropped



# §5. Per-panel pipeline: load → EDA-clean → cov-rank-1 → multi-criteria per block


def load_and_eda_clean(panel_name, data_dir=DATA_DIR):
    """Load the panel CSV, apply EDA cleaning, return (df_panel_clean,
    kept_indicators, dropped_dict, panel_meta).

    Does NOT yet do the train/val split or standardization — that happens later
    inside the per-block pipeline since blocks have different countries.
    """
    spec = NEW_PANEL_SPECS[panel_name]
    csv_path = os.path.join(data_dir, spec['csv_filename'])
    if not os.path.exists(csv_path):
        raise FileNotFoundError(csv_path)
    df_raw = pd.read_csv(csv_path)
    print(f"Loaded {spec['csv_filename']}: {df_raw.shape}")

    id_col = spec['id_col']; year_col = spec['year_col']
    if spec['variables'] is not None:
        indicators = spec['variables']
    else:
        # Auto-detect: numeric columns excluding ID/year/country_text_id/COWcode
        exclude = {id_col, year_col, 'country_id', 'country_name', 'country_text_id',
                   'COWcode', 'iso3', 'country_code', 'ISO'}
        indicators = [c for c in df_raw.columns if c not in exclude
                       and df_raw[c].dtype != object]

    print(f"  Indicator pool: {len(indicators)}")

    df = df_raw[[id_col, year_col] + indicators].copy()
    df = df.dropna(subset=indicators).sort_values([id_col, year_col]).reset_index(drop=True)
    # Standardize country_name col for the EDA helper
    if id_col != 'country_name':
        df['country_name'] = df[id_col].astype(str)

    # Restrict to units with sufficient series length
    lengths = df.groupby(id_col)[year_col].count()
    eligible = lengths[lengths >= spec['min_series_length']].index
    df = df[df[id_col].isin(eligible)]

    # EDA cleaning
    print(f"  Running EDA cleaning on {len(indicators)} indicators...")
    kept, dropped = eda_clean_indicators(df, indicators, country_col='country_name')
    print(f"  EDA: kept {len(kept)}, dropped {len(dropped)}")
    if dropped:
        for ind, reason in list(dropped.items())[:10]:
            print(f"    DROP {ind}: {reason}")
        if len(dropped) > 10:
            print(f"    ... ({len(dropped) - 10} more)")

    panel_meta = {
        'panel_name':  panel_name,
        'id_col':       id_col, 'year_col': year_col,
        'csv_filename': spec['csv_filename'],
        'n_countries':  df[id_col].nunique(),
        'n_years_max':  int(df.groupby(id_col)[year_col].count().max()),
        'min_series_length': spec['min_series_length'],
        'val_years':    spec['val_years'],
    }

    return df, kept, dropped, panel_meta


def build_panel_for_block(df, block_indicators, panel_meta):
    """For a given block of indicators, build the (Y_train, split_spec)
    panel matrix using the panel_meta's train/val split."""
    id_col = panel_meta['id_col']; year_col = panel_meta['year_col']
    val_yrs = panel_meta['val_years']; min_len = panel_meta['min_series_length']

    # Restrict to rows with all block indicators non-null
    df_block = df[[id_col, year_col] + list(block_indicators)].dropna(subset=block_indicators)
    df_block = df_block.sort_values([id_col, year_col]).reset_index(drop=True)
    # Restrict to units with sufficient series length
    lengths = df_block.groupby(id_col)[year_col].count()
    eligible = lengths[lengths >= min_len].index
    df_block = df_block[df_block[id_col].isin(eligible)]
    # Train split (drop last val_yrs years per unit)
    train_parts = []
    for cid, g in df_block.groupby(id_col, sort=False):
        g = g.sort_values(year_col).reset_index(drop=True)
        if len(g) >= min_len + val_yrs:
            train_parts.append(g.iloc[:-val_yrs])
        else:
            train_parts.append(g)
    train_df = pd.concat(train_parts, ignore_index=True)
    split_spec = (train_df.groupby(id_col, sort=False)[year_col].count().tolist())
    Y_train = train_df[list(block_indicators)].to_numpy(float)
    return Y_train, split_spec, list(block_indicators)


def run_panel(panel_name):
    """Full pipeline for a panel: EDA-clean → cov-rank-1 → per-block multi-criteria."""
    print(f"\n{'='*78}")
    print(f"  PANEL: {panel_name}")
    print(f"{'='*78}\n")

    OUT_DIR = os.path.join(OUT_DIR_BASE, panel_name)
    os.makedirs(OUT_DIR, exist_ok=True)

    # 1. Load + EDA clean
    df, kept, dropped, panel_meta = load_and_eda_clean(panel_name)
    with open(os.path.join(OUT_DIR, 'eda_drops.json'), 'w') as fh:
        json.dump({'kept': kept, 'dropped': dropped, 'panel_meta': panel_meta}, fh, indent=2)

    if len(kept) < 2:
        print(f"  After EDA cleaning, only {len(kept)} indicators remain. Aborting.")
        return

    # 2. Cov-rank-1 on cleaned panel
    print(f"\n  Running cov-rank-1 on {len(kept)} cleaned indicators...")
    # Build a single panel-mean correlation matrix; cov-rank-1 takes Y matrix
    Y_for_covrank = df[kept].dropna().to_numpy(float)
    # Standardize for correlation
    Y_for_covrank = (Y_for_covrank - Y_for_covrank.mean(axis=0)) / (Y_for_covrank.std(axis=0) + 1e-12)
    cr1_result = cov_rank_1(Y_for_covrank, threshold=COV_RANK1_THRESHOLD, verbose=True)
    blocks = cr1_result['blocks']
    block_T_cov = cr1_result['block_T_cov']
    print(f"\n  cov-rank-1 found {len(blocks)} blocks of size >= 2:")
    for b_idx, (block, t) in enumerate(zip(blocks, block_T_cov)):
        names = [kept[i] for i in block]
        print(f"    Block {b_idx}: size {len(block)}, T_cov={t:.3f}")
        for n in names:
            print(f"      {n}")
    if cr1_result['singletons']:
        print(f"  Singletons (excluded from blocks):")
        for s in cr1_result['singletons']:
            print(f"      {kept[s]}")

    # Save block assignments
    blocks_out = {
        'panel_name':  panel_name,
        'cleaned_indicators': kept,
        'cov_rank1_threshold': COV_RANK1_THRESHOLD,
        'blocks':       [[kept[i] for i in b] for b in blocks],
        'block_T_cov':  [float(t) for t in block_T_cov],
        'singletons':   [kept[s] for s in cr1_result['singletons']],
    }
    with open(os.path.join(OUT_DIR, 'covrank1_blocks.json'), 'w') as fh:
        json.dump(blocks_out, fh, indent=2)

    # 3. For each block, run multi-criteria pipeline
    all_rows = []
    method_outputs_by_block = {}

    for b_idx, (block, t_cov) in enumerate(zip(blocks, block_T_cov)):
        block_indicator_names = [kept[i] for i in block]
        block_label = f"block{b_idx}_size{len(block)}"
        print(f"\n  {'-'*70}\n  BLOCK {b_idx} (size {len(block)}, T_cov={t_cov:.3f})")
        print(f"  Indicators: {block_indicator_names}\n  {'-'*70}\n")

        Y_train, split_spec, var_names_block = build_panel_for_block(
            df, block_indicator_names, panel_meta)

        if len(split_spec) < 5 or Y_train.shape[1] < 2:
            print(f"  Block too small after panel construction; skipping")
            continue

        # Standardize (training-only)
        mu = Y_train.mean(axis=0); sd = Y_train.std(axis=0)
        sd = np.where(sd < 1e-12, 1.0, sd)
        Y_norm = (Y_train - mu) / sd

        N = Y_norm.shape[1]; U = len(split_spec)
        print(f"  Block panel: U={U}, T_per_unit_mean={np.mean(split_spec):.1f}, N={N}")

        # --- Run 5 methods on this block ---
        METHODS = {
            "linear_var_granger": run_linear_var_granger,
            "pcmci":              run_pcmci,
            "cmlp":               run_cmlp,
            "navar":              run_navar,
            "dynotears":          run_dynotears_wrapper,
        }

        block_method_outputs = {}
        for method_name in METHODS_TO_RUN:
            print(f"\n  [{method_name}] fit + score on block {b_idx}...")
            t0 = time.time()
            try:
                kwargs = dict(variable_names=var_names_block, seed=SEED)
                if method_name == "cmlp":   kwargs["device"] = DEVICE
                elif method_name == "navar": kwargs["verbose"] = False
                out = METHODS[method_name](Y_norm, split_spec, **kwargs)
                block_method_outputs[method_name] = out
                W_hat = out["W_hat"]
                fit_time = time.time() - t0
                print(f"    fit done in {fit_time:.1f}s, nnz={int((np.abs(W_hat) > 1e-12).sum())}")
            except Exception as e:
                print(f"    FIT FAILED: {type(e).__name__}: {e}")
                continue

            # Score W_hat
            row = score_one_W(
                W_label=f"{method_name}__method", W_matrix=W_hat,
                method_label=method_name, family_label="method", rep_idx=-1,
                Y=Y_norm, split_spec=split_spec, seed=SEED,
            )
            row['block_idx'] = b_idx; row['block_size'] = len(block)
            row['T_cov'] = t_cov; row['panel'] = panel_name
            all_rows.append(row)

            # Random baselines
            rng = np.random.default_rng(SEED + 100 + METHOD_RNG_OFFSETS.get(method_name, 0))
            nnz = int((np.abs(W_hat) > 1e-12).sum())
            for family_name, gen_fn in [
                ("iid_density_matched", lambda: random_iid_density_matched(W_hat, nnz, rng)),
                ("perm_rows",            lambda: random_perm_rows(W_hat, rng)),
                ("perm_full",            lambda: random_perm_full(W_hat, rng)),
            ]:
                for rep in range(N_REPS_PER_FAMILY):
                    W_base = gen_fn()
                    base_row = score_one_W(
                        W_label=f"{method_name}__{family_name}__r{rep:02d}",
                        W_matrix=W_base, method_label=method_name,
                        family_label=family_name, rep_idx=rep,
                        Y=Y_norm, split_spec=split_spec, seed=SEED,
                    )
                    base_row['block_idx'] = b_idx; base_row['block_size'] = len(block)
                    base_row['T_cov'] = t_cov; base_row['panel'] = panel_name
                    all_rows.append(base_row)
            # incremental save
            pd.DataFrame(all_rows).to_csv(os.path.join(OUT_DIR, 'sweep_raw.csv'), index=False)

        method_outputs_by_block[b_idx] = block_method_outputs

    # Save per-block method_outputs
    with open(os.path.join(OUT_DIR, 'method_outputs.pkl'), 'wb') as fh:
        pickle.dump({
            'panel_name': panel_name,
            'method_outputs_by_block': method_outputs_by_block,
            'blocks':     blocks_out['blocks'],
            'block_T_cov': blocks_out['block_T_cov'],
            'panel_meta': panel_meta,
            'n_reps_per_family': N_REPS_PER_FAMILY,
        }, fh)

    print(f"\n  Saved sweep_raw.csv ({len(all_rows)} rows) and method_outputs.pkl to {OUT_DIR}")



# §6. Run the configured panel(s)


# Install tigramite if needed (one-time per Colab session)
try:
    import tigramite
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'tigramite'], check=True)


if PANEL_TO_RUN == 'all':
    for panel in ['wb_wpp', 'long_16var', 'vdem_indicators']:
        try:
            run_panel(panel)
        except Exception as e:
            print(f"\n[ERROR] Panel '{panel}' failed: {type(e).__name__}: {e}")
else:
    run_panel(PANEL_TO_RUN)

### Cell §5.7 — Compute ΔPR follow-up (parametric AR(1) null on principled-pipeline blocks)

**Paper sections:** §5, Appendix P.

Step 2 of the principled pipeline. Loads the `method_outputs.pkl` from §5.6 and computes ΔPR with the parametric AR(1) null at B=1000 for each block × method. Refits OLS VAR on each simulated panel; computes PR on its W_hat; collects the null distribution. Produces the parametric-null p-values in the WB-WPP and V-Dem-60 rows of Table 3.

**Inputs:** `method_outputs.pkl` from §5.6.

**Outputs:** `multi_criteria_summary.csv` (final per-panel per-block per-method table with ΔPR + p-values for both nulls).

**Runtime:** ~3 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pickle, json, time
import numpy as np
import pandas as pd

DRIVE_DIR    = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR     = os.path.join(DRIVE_DIR, 'data')
OUT_DIR_BASE = os.path.join(DRIVE_DIR, 'multi_criteria_outputs')

# Which panels to process. Each panel must have its method_outputs.pkl,
# sweep_raw.csv, and covrank1_blocks.json already produced by §5.6.
PANELS_TO_RUN = ['wb_wpp', 'long_16var', 'vdem_indicators']
# To run only a subset, e.g.:
# PANELS_TO_RUN = ['wb_wpp', 'vdem_indicators']

SEED       = 0
NULL_B     = 1000             # bootstrap reps for both nulls



# §1. Reload panel + cleaned indicators + reflective block


def load_panel_for_block(panel_name, block_indicators):
    """Load the panel CSV and return (Y_norm, split_spec) for the block.

    Matches §5.6's preprocessing exactly:
      - filter to units with min_series_length=9 + val_years=5 minimum
      - drop last 5 years per unit (val split)
      - per-indicator pooled standardization on the train portion
    """
    if panel_name == 'wb_wpp':
        csv_filename = 'my_data_WB.csv'
        id_col, year_col = 'country_name', 'year'
    elif panel_name == 'long_16var':
        csv_filename = 'my_data_75_years.csv'
        id_col, year_col = 'country_id', 'year'
    elif panel_name == 'vdem_indicators':
        csv_filename = 'my_data_VDem_Ind.csv'
        id_col, year_col = 'country_name', 'year'
    else:
        raise ValueError(f"Unknown panel: {panel_name}")

    df = pd.read_csv(os.path.join(DATA_DIR, csv_filename))
    keep = [id_col, year_col] + list(block_indicators)
    df = df[keep].dropna(subset=block_indicators).sort_values([id_col, year_col]).reset_index(drop=True)
    # Restrict to units with min_series_length=9 + val_years=5 minimum
    lengths = df.groupby(id_col)[year_col].count()
    eligible = lengths[lengths >= 9].index
    df = df[df[id_col].isin(eligible)]
    # Train split: drop last 5 years per unit
    train_parts = []
    for cid, g in df.groupby(id_col, sort=False):
        g = g.sort_values(year_col).reset_index(drop=True)
        if len(g) >= 9 + 5:
            train_parts.append(g.iloc[:-5])
        else:
            train_parts.append(g)
    train_df = pd.concat(train_parts, ignore_index=True)
    split_spec = (train_df.groupby(id_col, sort=False)[year_col].count().tolist())
    Y_train = train_df[list(block_indicators)].to_numpy(float)
    # Standardize
    mu = Y_train.mean(axis=0); sd = Y_train.std(axis=0)
    sd = np.where(sd < 1e-12, 1.0, sd)
    Y_norm = (Y_train - mu) / sd
    return Y_norm, split_spec



# §2. Helpers: PR, AR(1) fit, OLS VAR


def pr_metric(W):
    """Rank-one fit metric on a square matrix.
    PR(W) = 1 - ||W - σ_1 u_1 v_1^T||_F^2 / ||W||_F^2 ∈ [0, 1].

    For W with all zeros, returns 0.
    """
    W = np.asarray(W, dtype=float)
    fro_sq = (W**2).sum()
    if fro_sq < 1e-20:
        return 0.0
    try:
        U, S, Vt = np.linalg.svd(W, full_matrices=False)
        if S.size == 0:
            return 0.0
        rank1_recon = S[0] * np.outer(U[:, 0], Vt[0, :])
        resid_sq = ((W - rank1_recon) ** 2).sum()
        return float(1.0 - resid_sq / fro_sq)
    except np.linalg.LinAlgError:
        return 0.0


def fit_ar1_per_indicator(Y, split_spec):
    """Fit per-indicator AR(1) on within-unit lag pairs.
    Returns: betas (N,), sigmas (N,).
    """
    N = Y.shape[1]
    cursor = 0
    all_y = [[] for _ in range(N)]
    all_ylag = [[] for _ in range(N)]
    for T_u in split_spec:
        unit_data = Y[cursor:cursor + T_u]
        for t in range(1, T_u):
            for i in range(N):
                all_y[i].append(unit_data[t, i])
                all_ylag[i].append(unit_data[t-1, i])
        cursor += T_u

    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        y = np.asarray(all_y[i]); yl = np.asarray(all_ylag[i])
        ymean = y.mean(); ylmean = yl.mean()
        yc = y - ymean; ylc = yl - ylmean
        denom = (ylc**2).sum()
        if denom < 1e-12:
            betas[i] = 0.0; sigmas[i] = 0.0
            continue
        beta = (ylc * yc).sum() / denom
        intercept = ymean - beta * ylmean
        resid = y - (intercept + beta * yl)
        betas[i] = beta
        sigmas[i] = np.sqrt((resid**2).mean())
    return betas, sigmas


def simulate_ar1_panel(betas, sigmas, split_spec, Y_orig_first_obs, rng):
    """Simulate a panel from per-indicator AR(1) with the same split structure.

    Y_orig_first_obs: shape (n_units, N) — first observation of each unit;
      we re-use these as initial conditions to keep the simulated panel on the
      same scale as the original.
    """
    N = len(betas)
    cursor = 0
    Y_sim = np.zeros((sum(split_spec), N))
    for u_idx, T_u in enumerate(split_spec):
        Y_sim[cursor] = Y_orig_first_obs[u_idx]
        for t in range(1, T_u):
            innov = rng.normal(0, sigmas)
            Y_sim[cursor + t] = betas * Y_sim[cursor + t - 1] + innov
        cursor += T_u
    return Y_sim


def fit_ols_var(Y, split_spec):
    """Pooled OLS VAR with intercept. Returns coefficient matrix W (N, N)
    where W[i, j] = effect of lagged j on i (paper convention).
    """
    N = Y.shape[1]
    cursor = 0
    y_pairs = []
    yl_pairs = []
    for T_u in split_spec:
        unit = Y[cursor:cursor + T_u]
        for t in range(1, T_u):
            y_pairs.append(unit[t])
            yl_pairs.append(unit[t-1])
        cursor += T_u
    Yt = np.asarray(y_pairs)
    Yl = np.asarray(yl_pairs)
    Yl_aug = np.column_stack([Yl, np.ones(len(Yl))])
    try:
        W_full, *_ = np.linalg.lstsq(Yl_aug, Yt, rcond=None)
        W = W_full[:N, :].T
    except np.linalg.LinAlgError:
        W = np.zeros((N, N))
    return W



# §3. Null distributions


def parametric_ar1_null(Y, split_spec, B=1000, seed=0, verbose=False):
    """Bootstrap PR null from parametric AR(1) simulation."""
    rng = np.random.default_rng(seed)
    betas, sigmas = fit_ar1_per_indicator(Y, split_spec)
    Y_first = []
    cursor = 0
    for T_u in split_spec:
        Y_first.append(Y[cursor])
        cursor += T_u
    Y_first = np.asarray(Y_first)

    pr_null = np.zeros(B)
    t0 = time.time()
    for b in range(B):
        Y_sim = simulate_ar1_panel(betas, sigmas, split_spec, Y_first, rng)
        W_sim = fit_ols_var(Y_sim, split_spec)
        pr_null[b] = pr_metric(W_sim)
        if verbose and (b + 1) % 100 == 0:
            print(f"      parametric null draw {b+1}/{B} ({time.time()-t0:.1f}s elapsed)")
    return pr_null


def row_perm_null(W_hat, B=1000, seed=0):
    """Row-permuting null: permute off-diagonal entries within each row of W_hat,
    compute PR of permuted matrix.
    """
    W = np.asarray(W_hat, dtype=float)
    N = W.shape[0]
    rng = np.random.default_rng(seed + 1000)
    pr_null = np.zeros(B)
    for b in range(B):
        W_out = np.zeros_like(W)
        for i in range(N):
            others = np.array([j for j in range(N) if j != i])
            perm = rng.permutation(others)
            W_out[i, perm] = W[i, others]
        pr_null[b] = pr_metric(W_out)
    return pr_null


def compute_p_value(observed, null_dist):
    """One-tailed p-value: P(null >= observed). Plus-one correction."""
    n_at_or_above = int((null_dist >= observed).sum())
    return (1 + n_at_or_above) / (1 + len(null_dist))


# §4. Run for each panel in PANELS_TO_RUN


t_overall = time.time()

# Track per-panel results so we can summarize at the end
all_summaries = {}

for PANEL_NAME in PANELS_TO_RUN:
    print(f"\n\n{'#' * 75}")
    print(f"#  PANEL: {PANEL_NAME}")
    print(f"{'#' * 75}")
    t_panel = time.time()

    OUT_DIR = os.path.join(OUT_DIR_BASE, PANEL_NAME)
    method_outputs_path = os.path.join(OUT_DIR, 'method_outputs.pkl')
    sweep_raw_path = os.path.join(OUT_DIR, 'sweep_raw.csv')

    # Sanity check: required §5.6 outputs must exist
    if not os.path.exists(method_outputs_path):
        print(f"  SKIP — {method_outputs_path} not found. Run §5.6 for this panel first.")
        continue
    if not os.path.exists(sweep_raw_path):
        print(f"  SKIP — {sweep_raw_path} not found. Run §5.6 for this panel first.")
        continue

    # Load saved method_outputs
    with open(method_outputs_path, 'rb') as fh:
        saved = pickle.load(fh)

    panel_name = saved['panel_name']
    blocks = saved['blocks']
    block_T_cov = saved['block_T_cov']
    method_outputs_by_block = saved['method_outputs_by_block']
    print(f"  Loaded {panel_name}: {len(blocks)} block(s)")

    # Load sweep_raw for CRPS values
    sweep = pd.read_csv(sweep_raw_path)

    summary_rows = []
    for b_idx, (block_indicators, t_cov) in enumerate(zip(blocks, block_T_cov)):
        print(f"\n  {'='*70}")
        print(f"  Block {b_idx} (size {len(block_indicators)}, T_cov={t_cov:.3f})")
        print(f"    Indicators: {block_indicators}")
        print(f"  {'='*70}")

        # Reload panel for this block (same procedure as §5.6 pipeline)
        Y_norm, split_spec = load_panel_for_block(panel_name, block_indicators)
        print(f"    Panel: U={len(split_spec)}, T_mean={np.mean(split_spec):.1f}, N={Y_norm.shape[1]}")

        # Compute parametric AR(1) null ONCE for this block (shared across methods)
        print(f"\n    Computing parametric AR(1) null (B={NULL_B})...")
        pr_null_param = parametric_ar1_null(Y_norm, split_spec, B=NULL_B, seed=SEED, verbose=True)
        null_mean_param = float(pr_null_param.mean())
        print(f"    Parametric null PR: mean={null_mean_param:.4f}, std={pr_null_param.std():.4f}")

        # Per method: compute ΔPR
        block_methods = method_outputs_by_block[b_idx]
        for method_name in ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears']:
            if method_name not in block_methods:
                continue
            W_hat = np.asarray(block_methods[method_name]['W_hat'], dtype=float)
            nnz = int((np.abs(W_hat) > 1e-12).sum())

            pr_obs = pr_metric(W_hat)
            dpr_param = pr_obs - null_mean_param
            p_param = compute_p_value(pr_obs, pr_null_param)

            # Row-permuting null is per-method
            pr_null_rowperm = row_perm_null(W_hat, B=NULL_B, seed=SEED)
            null_mean_rowperm = float(pr_null_rowperm.mean())
            dpr_rowperm = pr_obs - null_mean_rowperm
            p_rowperm = compute_p_value(pr_obs, pr_null_rowperm)

            # CRPS from sweep_raw
            method_row_match = sweep[(sweep['method']==method_name) & (sweep['family']=='method')]
            if len(method_row_match) == 0:
                # method missing from sweep_raw — record placeholder
                crps_unc = float('nan')
                crps_con = float('nan')
                rb_beats = 0
            else:
                method_row = method_row_match.iloc[0]
                crps_unc = float(method_row['CRPS_unconstrained'])
                crps_con = float(method_row['CRPS_constrained'])

                # rb-beats: count of (protocol, family) cells where p_emp <= 0.10
                rb_beats = 0
                for protocol in ['CRPS_unconstrained', 'CRPS_constrained']:
                    crps_method = method_row[protocol]
                    for family in ['iid_density_matched', 'perm_rows', 'perm_full']:
                        base = sweep[(sweep['method']==method_name) & (sweep['family']==family)][protocol].dropna()
                        if len(base) == 0 or not np.isfinite(crps_method):
                            continue
                        n_beats = int((base <= crps_method).sum())
                        p_emp = (1 + n_beats) / (1 + len(base))
                        if p_emp <= 0.10:
                            rb_beats += 1

            print(f"\n    {method_name:22s}  nnz={nnz:>4d}  PR_obs={pr_obs:.4f}  "
                  f"ΔPR_param={dpr_param:+.4f} (p={p_param:.4f})  "
                  f"ΔPR_rowperm={dpr_rowperm:+.4f} (p={p_rowperm:.4f})")

            summary_rows.append({
                'panel':       panel_name,
                'block_idx':   b_idx,
                'block_size':  len(block_indicators),
                'T_cov':       t_cov,
                'method':      method_name,
                'nnz':         nnz,
                'CRPS_unc':    crps_unc,
                'CRPS_con':    crps_con,
                'rb_beats':    f'{rb_beats}/6',
                'PR_obs':      pr_obs,
                'PR_null_param_mean': null_mean_param,
                'dPR_param':   dpr_param,
                'p_param':     p_param,
                'PR_null_rowperm_mean': null_mean_rowperm,
                'dPR_rowperm': dpr_rowperm,
                'p_rowperm':   p_rowperm,
                'verdict_param':  '✓' if (dpr_param > 0 and p_param < 0.05) else '✗',
                'verdict_rowperm': '✓' if (dpr_rowperm > 0 and p_rowperm < 0.05) else '✗',
            })

    # Save summary for this panel
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUT_DIR, 'multi_criteria_summary.csv')
    summary_df.to_csv(summary_path, index=False)

    print(f"\n  {'='*70}")
    print(f"    MULTI-CRITERIA SUMMARY: {PANEL_NAME}  (panel runtime: {time.time()-t_panel:.1f}s)")
    print(f"  {'='*70}\n")
    display_cols = ['method', 'nnz', 'CRPS_unc', 'rb_beats',
                    'dPR_param', 'p_param', 'dPR_rowperm', 'p_rowperm',
                    'verdict_param']
    if len(summary_df) > 0:
        print(summary_df[display_cols].to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print(f"\n  Saved to {summary_path}")

    all_summaries[PANEL_NAME] = summary_df



# §5. Final overall summary


print(f"\n\n{'='*75}")
print(f"  ALL PANELS COMPLETE  (total runtime: {time.time()-t_overall:.1f}s)")
print(f"{'='*75}\n")
for panel_name, df in all_summaries.items():
    n_rows = len(df) if df is not None else 0
    n_pass_param = int((df['verdict_param'] == '✓').sum()) if df is not None and n_rows > 0 else 0
    n_pass_rp = int((df['verdict_rowperm'] == '✓').sum()) if df is not None and n_rows > 0 else 0
    print(f"  {panel_name:20s} {n_rows} method-rows, "
          f"{n_pass_param}/{n_rows} pass parametric, {n_pass_rp}/{n_rows} pass row-permuting")

panels_skipped = [p for p in PANELS_TO_RUN if p not in all_summaries]
if panels_skipped:
    print(f"\n  Skipped panels (no §5.6 outputs): {panels_skipped}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###########################################################################
#  PANEL: wb_wpp
###########################################################################
  Loaded wb_wpp: 1 block(s)

  Block 0 (size 6, T_cov=0.853)
    Indicators: ['wb_agri_land_share', 'kofpogidf', 'kofpogidj', 'kofipgidj', 'wb_life_expectancy', 'wpp_median_age_jul']
    Panel: U=173, T_mean=51.0, N=6

    Computing parametric AR(1) null (B=1000)...
      parametric null draw 100/1000 (4.7s elapsed)
      parametric null draw 200/1000 (9.5s elapsed)
      parametric null draw 300/1000 (14.2s elapsed)
      parametric null draw 400/1000 (18.9s elapsed)
      parametric null draw 500/1000 (23.7s elapsed)
      parametric null draw 600/1000 (28.4s elapsed)
      parametric null draw 700/1000 (33.1s elapsed)
      parametric null draw 800/1000 (37.9s elapsed)
      parametric nu

## Section 6 — Appendix Q: EDA conditions verification

Verifies the four EDA prerequisites on V-Dem-16, V-Dem-60, and WB-WPP: (1) regime-shift detection via Andrews sup-Wald; (2) saturation (>30% near-constant within-country); (3) outlier detection via EVT-based |z|-ratio threshold; (4) collinear-pair detection via |r| > 0.99.

### Cell §6.1 — EDA conditions verification (v1): superseded

**Paper sections:** Superseded by §6.2 (which is the version cited in Appendix Q).

Initial per-indicator EDA verification on all three substrates. The v1 sup-Wald criterion uses statistical significance alone; with 6000+ observations the test has enormous power and flags 12/16 V-Dem indices and 45/70 V-Dem indicators as regime-shifted (most of those flags are practically negligible β shifts). Superseded by §6.2 which adds an effect-size requirement.

For each substrate, computes per-indicator and panel-level metrics
under each condition, applies the literature-backed (or working)
threshold, and reports which indicators trigger flags or drops.

The output is a conditions × substrates table that becomes the
empirical core of the EDA appendix.

Conditions tested:
  1. Within-indicator regime shift (Andrews 1993 sup-Wald, CV=8.85)
  2. Saturation (>30% of countries near-constant)
  3. Outliers (max |z| relative to expected √(2 log N))
  4. Collinearity (pairwise |r| > 0.99)
  5. Cross-indicator correlation regime shift (max abs diff > 0.30 — heuristic)
  6. Within/total variance ratio (< 0.10 — diagnostic flag)
  7. T/N power (T/N ≥ 4 needed for stable verdicts — diagnostic)

**Inputs:** All three substrate CSVs.

**Outputs:** `eda_per_indicator_*.csv`, `eda_summary.json`.

**Runtime:** ~5-7 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'eda_verification_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

INDICES_CSV    = os.path.join(DATA_DIR, 'my_data_75_years.csv')
INDICATORS_CSV = os.path.join(DATA_DIR, 'my_data_VDem_Ind.csv')
WBWPP_CSV      = os.path.join(DATA_DIR, 'my_data_WB.csv')



# §1. Bounds and parameters


SUP_WALD_CV_ONE_PARAM = 8.85   # Andrews (1993) α=0.05, 15% trim, 1 param
TRIM_FRACTION = 0.15
SATURATION_THRESHOLD = 0.30    # drop if >30% of countries near-constant
NEAR_CONSTANT_EPS = 1e-6
OUTLIER_FLAG_RATIO = 1.5       # observed max|z| / expected max|z|
OUTLIER_DROP_RATIO = 2.0
COLLINEARITY_THRESHOLD = 0.99
CORR_REGIME_SHIFT_THRESHOLD = 0.30
WITHIN_TOTAL_RATIO_THRESHOLD = 0.10
TN_POWER_THRESHOLD = 4.0



# §2. Utility functions


def expected_max_abs_z(N):
    """Cramér's formula for expected max |z| of N standard normal samples."""
    if N < 2:
        return np.nan
    return np.sqrt(2 * np.log(N)) - (np.log(np.log(N)) + np.log(4 * np.pi)) / (2 * np.sqrt(2 * np.log(N)))


def sup_wald_ar1(values_by_country, trim_frac=0.15):
    """Sup-Wald test for structural break in AR(1) β across the panel.

    For each candidate breakpoint t in trim range, fit AR(1) on
    pre-break and post-break sub-panels, compute Wald statistic for
    H0: β_pre = β_post. Return supremum over candidate breakpoints.

    values_by_country: dict {country_name: array of values sorted by year}
    Returns: (sup_wald_stat, best_breakpoint_index)
    """
    # Stack pooled lag pairs by country; track time index
    all_y = []
    all_ylag = []
    all_t = []  # time index 0..T-1 within country
    for country, vals in values_by_country.items():
        if len(vals) < 5:
            continue
        for i in range(1, len(vals)):
            all_y.append(vals[i])
            all_ylag.append(vals[i-1])
            all_t.append(i)
    y = np.array(all_y)
    ylag = np.array(all_ylag)
    t = np.array(all_t)

    if len(y) < 20:
        return np.nan, np.nan

    T_max = t.max()
    trim_lo = int(np.ceil(trim_frac * T_max))
    trim_hi = int(np.floor((1 - trim_frac) * T_max))

    # Demean lag (intercept absorbed)
    sup_w = -np.inf
    best_t = None
    for t_break in range(trim_lo, trim_hi + 1):
        mask_pre = t <= t_break
        mask_post = t > t_break
        if mask_pre.sum() < 5 or mask_post.sum() < 5:
            continue
        # OLS with intercept on each sub-panel: y = α + β*ylag + ε
        def ols_simple(yy, xx):
            xc = xx - xx.mean()
            yc = yy - yy.mean()
            denom = (xc**2).sum()
            if denom < 1e-12:
                return np.nan, np.nan
            b = (xc * yc).sum() / denom
            resid = yc - b * xc
            sigma2 = (resid**2).sum() / max(len(yy) - 2, 1)
            se_b = np.sqrt(sigma2 / denom) if denom > 0 else np.inf
            return b, se_b

        b_pre, se_pre = ols_simple(y[mask_pre], ylag[mask_pre])
        b_post, se_post = ols_simple(y[mask_post], ylag[mask_post])
        if not (np.isfinite(b_pre) and np.isfinite(b_post)):
            continue
        if not (np.isfinite(se_pre) and np.isfinite(se_post)):
            continue
        var_diff = se_pre**2 + se_post**2
        if var_diff < 1e-12:
            continue
        wald = (b_pre - b_post)**2 / var_diff
        if wald > sup_w:
            sup_w = wald
            best_t = t_break

    return float(sup_w) if np.isfinite(sup_w) else np.nan, best_t


def fraction_constant_within_country(df, indicator, country_col='country_name', eps=NEAR_CONSTANT_EPS):
    by_country_std = df.groupby(country_col)[indicator].std()
    return float((by_country_std < eps).mean())


def within_total_variance_ratio(df, indicator, country_col='country_name'):
    total_var = df[indicator].var()
    if total_var < 1e-12:
        return np.nan
    within_var = df.groupby(country_col)[indicator].var().mean()
    return float(within_var / total_var)


def max_abs_z_after_standardize(df, indicator):
    vals = df[indicator].values
    z = (vals - vals.mean()) / (vals.std() + 1e-12)
    return float(np.abs(z).max())



# §3. Per-substrate evaluation


def evaluate_substrate(df, indicators, substrate_name, T_per_country, country_col='country_name'):
    """Run all 7 EDA conditions on the substrate. Returns dict of results."""
    print(f"\n{'='*80}")
    print(f"Substrate: {substrate_name}")
    print(f"{'='*80}")
    n_countries = df[country_col].nunique()
    n_total = len(df)
    print(f"  N rows: {n_total}, countries: {n_countries}, T/country: {T_per_country}")
    print(f"  Expected max |z| under normality: {expected_max_abs_z(n_total):.3f}")

    rows = []
    for ind in indicators:
        if ind not in df.columns:
            continue
        # Build per-country values for sup-Wald
        vals_by_country = {}
        for c, sub in df.groupby(country_col):
            v = sub.sort_values('year')[ind].values
            if len(v) >= 5 and np.std(v) > 1e-8:
                vals_by_country[c] = v

        # Cond 1: sup-Wald
        sup_w, best_t = sup_wald_ar1(vals_by_country, trim_frac=TRIM_FRACTION)

        # Cond 2: saturation
        sat_frac = fraction_constant_within_country(df, ind, country_col)

        # Cond 3: max |z|
        max_z = max_abs_z_after_standardize(df, ind)
        exp_max_z = expected_max_abs_z(n_total)
        z_ratio = max_z / exp_max_z if exp_max_z > 0 else np.nan

        # Cond 6: within/total variance
        wt_ratio = within_total_variance_ratio(df, ind, country_col)

        rows.append({
            'indicator': ind,
            'sup_wald': sup_w,
            'sup_wald_break_t': best_t,
            'flag_regime_shift': sup_w > SUP_WALD_CV_ONE_PARAM if np.isfinite(sup_w) else False,
            'saturation_frac': sat_frac,
            'flag_saturation': sat_frac > SATURATION_THRESHOLD,
            'max_abs_z': max_z,
            'z_ratio_to_expected': z_ratio,
            'flag_outlier': z_ratio > OUTLIER_FLAG_RATIO,
            'drop_outlier': z_ratio > OUTLIER_DROP_RATIO,
            'within_total_ratio': wt_ratio,
            'flag_low_within': wt_ratio < WITHIN_TOTAL_RATIO_THRESHOLD if np.isfinite(wt_ratio) else False,
        })

    per_indicator_df = pd.DataFrame(rows)

    # Cond 4: collinearity (panel-level)
    corr_matrix = df[indicators].corr().values
    n = len(indicators)
    high_corr_pairs = []
    for i in range(n):
        for j in range(i+1, n):
            if abs(corr_matrix[i,j]) > COLLINEARITY_THRESHOLD:
                high_corr_pairs.append((indicators[i], indicators[j], corr_matrix[i,j]))

    # Cond 5: cross-indicator regime shift (split panel at midpoint)
    if 'year' in df.columns:
        median_year = df['year'].median()
        df_pre = df[df['year'] < median_year]
        df_post = df[df['year'] >= median_year]
        if len(df_pre) >= 50 and len(df_post) >= 50:
            corr_pre = df_pre[indicators].corr()
            corr_post = df_post[indicators].corr()
            corr_diff = corr_post - corr_pre
            upper_tri = np.triu(np.ones(n, dtype=bool), k=1)
            max_abs_corr_shift = float(np.abs(corr_diff.values[upper_tri]).max())
        else:
            max_abs_corr_shift = np.nan
    else:
        max_abs_corr_shift = np.nan

    # Cond 7: T/N power
    N_block = len(indicators)
    tn_ratio = T_per_country / N_block

    # Aggregate counts
    summary = {
        'substrate': substrate_name,
        'n_countries': n_countries,
        'T_per_country': T_per_country,
        'n_indicators': len(indicators),
        'expected_max_z': expected_max_abs_z(n_total),
        # Per-indicator counts
        'n_flag_regime_shift': int(per_indicator_df['flag_regime_shift'].sum()),
        'n_flag_saturation': int(per_indicator_df['flag_saturation'].sum()),
        'n_flag_outlier': int(per_indicator_df['flag_outlier'].sum()),
        'n_drop_outlier': int(per_indicator_df['drop_outlier'].sum()),
        'n_flag_low_within': int(per_indicator_df['flag_low_within'].sum()),
        # Panel-level
        'n_high_corr_pairs': len(high_corr_pairs),
        'high_corr_pairs': high_corr_pairs,
        'max_abs_corr_shift_pre_post_median': max_abs_corr_shift,
        'flag_corr_regime_shift': max_abs_corr_shift > CORR_REGIME_SHIFT_THRESHOLD if np.isfinite(max_abs_corr_shift) else False,
        'tn_ratio': tn_ratio,
        'flag_low_power': tn_ratio < TN_POWER_THRESHOLD,
    }

    # Print per-indicator detail
    print("\n  Per-indicator EDA scores:")
    print(f"    {'indicator':30s} {'supW':>7s} {'sat':>5s} {'maxZ':>6s} {'zRatio':>7s} {'w/t':>6s}  flags")
    print("    " + "-" * 85)
    for r in per_indicator_df.itertuples():
        flags = []
        if r.flag_regime_shift: flags.append('REGIME')
        if r.flag_saturation:   flags.append('SAT')
        if r.drop_outlier:      flags.append('OUTLIER-DROP')
        elif r.flag_outlier:    flags.append('OUTLIER-FLAG')
        if r.flag_low_within:   flags.append('LOW-W')
        flag_str = ', '.join(flags) if flags else '-'
        sw_str = f"{r.sup_wald:7.2f}" if np.isfinite(r.sup_wald) else f"{'NaN':>7s}"
        wt_str = f"{r.within_total_ratio:6.3f}" if np.isfinite(r.within_total_ratio) else f"{'NaN':>6s}"
        print(f"    {r.indicator:30s} {sw_str} {r.saturation_frac:>5.2f} "
              f"{r.max_abs_z:>6.2f} {r.z_ratio_to_expected:>7.2f} {wt_str}  {flag_str}")

    print(f"\n  Panel-level diagnostics:")
    print(f"    High-correlation pairs (|r|>{COLLINEARITY_THRESHOLD}): {len(high_corr_pairs)}")
    for a, b, r in high_corr_pairs[:5]:
        print(f"      {a:30s} <-> {b:30s} r={r:+.3f}")
    if len(high_corr_pairs) > 5:
        print(f"      ... ({len(high_corr_pairs) - 5} more)")
    print(f"    Max abs cross-indicator correlation shift (pre/post median year): "
          f"{max_abs_corr_shift:.3f} {'[FLAG]' if max_abs_corr_shift > CORR_REGIME_SHIFT_THRESHOLD else ''}")
    print(f"    T/N ratio: {tn_ratio:.2f} {'[LOW POWER]' if tn_ratio < TN_POWER_THRESHOLD else '[OK]'}")

    print(f"\n  Aggregate flag counts:")
    print(f"    Regime shift:            {summary['n_flag_regime_shift']:>3d} indicators")
    print(f"    Saturation:              {summary['n_flag_saturation']:>3d} indicators")
    print(f"    Outlier flag (>1.5×):    {summary['n_flag_outlier']:>3d} indicators")
    print(f"    Outlier drop (>2.0×):    {summary['n_drop_outlier']:>3d} indicators")
    print(f"    Low within-variance:     {summary['n_flag_low_within']:>3d} indicators")

    return per_indicator_df, summary



# §4. Load substrates and run


# V-Dem indices (long_16var)
df_indices = pd.read_csv(INDICES_CSV).rename(columns={'country_id': 'country_name'})
df_indices = df_indices.sort_values(['country_name', 'year']).reset_index(drop=True)
INDICES_COLS = [c for c in df_indices.columns if c not in ['country_name', 'year']]

# V-Dem indicators (test on the FULL pool, not the cleaned one — we want to see
# what EDA flags BEFORE manual cleanup)
df_indicators = pd.read_csv(INDICATORS_CSV).sort_values(['country_name', 'year']).reset_index(drop=True)
INDICATORS_COLS_FULL = [c for c in df_indicators.columns
                        if c not in ['country_id', 'country_name', 'year']]

# WB-WPP (test on the FULL pool — same logic)
df_wbwpp = pd.read_csv(WBWPP_CSV).sort_values(['country_name', 'year']).reset_index(drop=True)
# WB-WPP file may have different ID column structure; check
WBWPP_COLS_FULL = [c for c in df_wbwpp.columns
                    if c not in ['country_name', 'year', 'iso3', 'country_code', 'ISO']]


print("Loaded:")
print(f"  V-Dem indices:    {df_indices.shape}, {len(INDICES_COLS)} indices")
print(f"  V-Dem indicators: {df_indicators.shape}, {len(INDICATORS_COLS_FULL)} indicator columns")
print(f"  WB-WPP:           {df_wbwpp.shape}, {len(WBWPP_COLS_FULL)} indicator columns")


# Run on each substrate
results = {}
T_indices = df_indices.groupby('country_name').size().iloc[0]
T_indicators = df_indicators.groupby('country_name').size().iloc[0]
T_wbwpp = df_wbwpp.groupby('country_name').size().iloc[0]

per_ind_indices, summary_indices = evaluate_substrate(
    df_indices, INDICES_COLS, 'V-Dem indices (long_16var)', T_indices)
results['indices'] = (per_ind_indices, summary_indices)

per_ind_indicators, summary_indicators = evaluate_substrate(
    df_indicators, INDICATORS_COLS_FULL, 'V-Dem indicators (full 70)', T_indicators)
results['indicators'] = (per_ind_indicators, summary_indicators)

per_ind_wbwpp, summary_wbwpp = evaluate_substrate(
    df_wbwpp, WBWPP_COLS_FULL, 'WB-WPP (full)', T_wbwpp)
results['wbwpp'] = (per_ind_wbwpp, summary_wbwpp)



# §5. Conditions × substrates summary table


print("\n\n" + "=" * 100)
print("CONDITIONS × SUBSTRATES SUMMARY")
print("=" * 100)
substrates_order = [
    ('V-Dem indices', summary_indices),
    ('V-Dem indicators', summary_indicators),
    ('WB-WPP', summary_wbwpp),
]
print(f"{'condition':50s}  " + "  ".join(f"{name:>22s}" for name, _ in substrates_order))
print("-" * 130)

def fmt_count(n, total):
    return f"{n}/{total}"

# Condition 1: regime shift
print(f"{'1. Regime shifts (sup-Wald > 8.85)':50s}  " +
      "  ".join(f"{fmt_count(s['n_flag_regime_shift'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
# Condition 2: saturation
print(f"{'2. Saturation (>30% countries flat)':50s}  " +
      "  ".join(f"{fmt_count(s['n_flag_saturation'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
# Condition 3a: outlier flag
print(f"{'3a. Outlier flag (max|z| > 1.5× expected)':50s}  " +
      "  ".join(f"{fmt_count(s['n_flag_outlier'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
# Condition 3b: outlier drop
print(f"{'3b. Outlier drop (max|z| > 2× expected)':50s}  " +
      "  ".join(f"{fmt_count(s['n_drop_outlier'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
# Condition 4: collinearity
print(f"{'4. Collinear pairs (|r| > 0.99)':50s}  " +
      "  ".join(f"{s['n_high_corr_pairs']:>22d}" for _, s in substrates_order))
# Condition 5: cross-indicator regime shift
print(f"{'5. Max corr shift (pre/post median year)':50s}  " +
      "  ".join(f"{s['max_abs_corr_shift_pre_post_median']:>22.3f}"
                for _, s in substrates_order))
# Condition 6: low within variance
print(f"{'6. Low within/total var (<0.10)':50s}  " +
      "  ".join(f"{fmt_count(s['n_flag_low_within'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
# Condition 7: T/N power
print(f"{'7. T/N ratio (need ≥4)':50s}  " +
      "  ".join(f"{s['tn_ratio']:>22.2f}" for _, s in substrates_order))



# §6. Save raw outputs


for key, (per_ind, summary) in results.items():
    per_ind.to_csv(os.path.join(OUT_DIR, f'eda_per_indicator_{key}.csv'), index=False)

with open(os.path.join(OUT_DIR, 'eda_summary.json'), 'w') as f:
    summary_clean = {k: {kk: vv for kk, vv in s.items() if kk != 'high_corr_pairs'}
                     for k, (_, s) in results.items()}
    # Add high_corr_pairs as clean tuples
    for k, (_, s) in results.items():
        summary_clean[k]['high_corr_pairs'] = [
            {'a': a, 'b': b, 'r': float(r)} for a, b, r in s['high_corr_pairs'][:20]
        ]
    json.dump(summary_clean, f, indent=2, default=str)

print(f"\nSaved per-indicator scores and summary to {OUT_DIR}")

### Cell §6.2 — EDA conditions verification v2 (CANONICAL)

**Paper section:** Appendix Q.

The canonical EDA verification used in the paper. Combines statistical significance (sup-Wald > 8.85 at α=0.05) with practical magnitude (|β_post − β_pre| > 0.05 at the best breakpoint, OR σ-ratio max/min > X across pre/post). Produces the EDA drops reported in Appendix Q including the `Elected_officials` saturation finding (sup-Wald = 10.79) that motivates the size-13 → size-12 verdict flip on V-Dem-16.

**Inputs:** All three substrate CSVs.

**Outputs:** `eda_v2_per_indicator_*.csv`, `eda_v2_summary.json`.

**Runtime:** ~1 minute on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'eda_verification_v2_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

INDICES_CSV    = os.path.join(DATA_DIR, 'my_data_75_years.csv')
INDICATORS_CSV = os.path.join(DATA_DIR, 'my_data_VDem_Ind.csv')
WBWPP_CSV      = os.path.join(DATA_DIR, 'my_data_WB.csv')



# §1. Bounds and parameters


SUP_WALD_CV_ONE_PARAM    = 8.85    # Andrews 1993, α=0.05, 15% trim, 1 param
TRIM_FRACTION            = 0.15
BETA_SHIFT_THRESHOLD     = 0.05    # practical: |Δβ| at best breakpoint
SIGMA_RATIO_THRESHOLD    = 1.5     # practical: max/min σ across pre and post
SATURATION_THRESHOLD     = 0.30
NEAR_CONSTANT_EPS        = 1e-6
OUTLIER_FLAG_RATIO       = 1.5
OUTLIER_DROP_RATIO       = 2.0
COLLINEARITY_THRESHOLD   = 0.99
CORR_REGIME_SHIFT_THRESHOLD = 0.30
WITHIN_TOTAL_RATIO_THRESHOLD = 0.10
TN_POWER_THRESHOLD       = 4.0



# §2. Utilities


def expected_max_abs_z(N):
    if N < 2:
        return np.nan
    return np.sqrt(2 * np.log(N)) - (np.log(np.log(N)) + np.log(4 * np.pi)) / (2 * np.sqrt(2 * np.log(N)))


def sup_wald_ar1_with_effect_size(values_by_country, trim_frac=0.15):
    """Sup-Wald test for AR(1) β stability AND return effect-size measures
    at the best breakpoint.

    Returns dict with: sup_wald, best_t, beta_pre, beta_post, sigma_pre,
    sigma_post, abs_beta_shift, sigma_ratio.
    """
    all_y, all_ylag, all_t = [], [], []
    for country, vals in values_by_country.items():
        if len(vals) < 5:
            continue
        for i in range(1, len(vals)):
            all_y.append(vals[i])
            all_ylag.append(vals[i-1])
            all_t.append(i)
    y = np.array(all_y); ylag = np.array(all_ylag); t = np.array(all_t)

    if len(y) < 20:
        return {'sup_wald': np.nan, 'best_t': np.nan, 'beta_pre': np.nan,
                'beta_post': np.nan, 'sigma_pre': np.nan, 'sigma_post': np.nan,
                'abs_beta_shift': np.nan, 'sigma_ratio': np.nan}

    T_max = t.max()
    trim_lo = int(np.ceil(trim_frac * T_max))
    trim_hi = int(np.floor((1 - trim_frac) * T_max))

    def ols_simple(yy, xx):
        xc = xx - xx.mean(); yc = yy - yy.mean()
        denom = (xc**2).sum()
        if denom < 1e-12:
            return np.nan, np.nan, np.nan
        b = (xc * yc).sum() / denom
        resid = yc - b * xc
        sigma2 = (resid**2).sum() / max(len(yy) - 2, 1)
        sigma = np.sqrt(sigma2) if sigma2 > 0 else np.nan
        se_b = np.sqrt(sigma2 / denom) if denom > 0 else np.inf
        return b, se_b, sigma

    sup_w = -np.inf; best = None
    for t_break in range(trim_lo, trim_hi + 1):
        mask_pre = t <= t_break
        mask_post = t > t_break
        if mask_pre.sum() < 5 or mask_post.sum() < 5:
            continue
        b_pre, se_pre, s_pre = ols_simple(y[mask_pre], ylag[mask_pre])
        b_post, se_post, s_post = ols_simple(y[mask_post], ylag[mask_post])
        if not (np.isfinite(b_pre) and np.isfinite(b_post)
                and np.isfinite(se_pre) and np.isfinite(se_post)):
            continue
        var_diff = se_pre**2 + se_post**2
        if var_diff < 1e-12:
            continue
        wald = (b_pre - b_post)**2 / var_diff
        if wald > sup_w:
            sup_w = wald
            best = {'best_t': t_break, 'beta_pre': b_pre, 'beta_post': b_post,
                    'sigma_pre': s_pre, 'sigma_post': s_post}

    if best is None:
        return {'sup_wald': np.nan, 'best_t': np.nan, 'beta_pre': np.nan,
                'beta_post': np.nan, 'sigma_pre': np.nan, 'sigma_post': np.nan,
                'abs_beta_shift': np.nan, 'sigma_ratio': np.nan}

    abs_beta_shift = abs(best['beta_pre'] - best['beta_post'])
    if np.isfinite(best['sigma_pre']) and np.isfinite(best['sigma_post']):
        s_min = min(best['sigma_pre'], best['sigma_post'])
        s_max = max(best['sigma_pre'], best['sigma_post'])
        sigma_ratio = s_max / s_min if s_min > 1e-9 else np.inf
    else:
        sigma_ratio = np.nan

    return {'sup_wald': float(sup_w), 'best_t': best['best_t'],
            'beta_pre': float(best['beta_pre']),
            'beta_post': float(best['beta_post']),
            'sigma_pre': float(best['sigma_pre']),
            'sigma_post': float(best['sigma_post']),
            'abs_beta_shift': float(abs_beta_shift),
            'sigma_ratio': float(sigma_ratio)}


def fraction_constant_within_country(df, indicator, country_col='country_name', eps=NEAR_CONSTANT_EPS):
    by_country_std = df.groupby(country_col)[indicator].std()
    return float((by_country_std < eps).mean())


def within_total_variance_ratio(df, indicator, country_col='country_name'):
    total_var = df[indicator].var()
    if total_var < 1e-12:
        return np.nan
    within_var = df.groupby(country_col)[indicator].var().mean()
    return float(within_var / total_var)


def max_abs_z_after_standardize(df, indicator):
    vals = df[indicator].values
    z = (vals - vals.mean()) / (vals.std() + 1e-12)
    return float(np.abs(z).max())



# §3. Per-substrate evaluation


def evaluate_substrate(df, indicators, substrate_name, T_per_country, country_col='country_name'):
    print(f"\n{'='*80}")
    print(f"Substrate: {substrate_name}")
    print(f"{'='*80}")
    n_countries = df[country_col].nunique()
    n_total = len(df)
    print(f"  N rows: {n_total}, countries: {n_countries}, T/country: {T_per_country}")
    print(f"  Expected max |z| under normality: {expected_max_abs_z(n_total):.3f}")

    rows = []
    for ind in indicators:
        if ind not in df.columns:
            continue
        vals_by_country = {}
        for c, sub in df.groupby(country_col):
            v = sub.sort_values('year')[ind].values
            if len(v) >= 5 and np.std(v) > 1e-8:
                vals_by_country[c] = v

        # Cond 1: sup-Wald + effect size
        rs = sup_wald_ar1_with_effect_size(vals_by_country, trim_frac=TRIM_FRACTION)

        # Statistical significance
        stat_sig = (rs['sup_wald'] > SUP_WALD_CV_ONE_PARAM) if np.isfinite(rs['sup_wald']) else False
        # Effect size: either |Δβ| or σ ratio above threshold
        beta_mag = (rs['abs_beta_shift'] > BETA_SHIFT_THRESHOLD) if np.isfinite(rs['abs_beta_shift']) else False
        sigma_mag = (rs['sigma_ratio'] > SIGMA_RATIO_THRESHOLD) if np.isfinite(rs['sigma_ratio']) else False
        practical = beta_mag or sigma_mag
        flag_regime = stat_sig and practical

        # Cond 2: saturation
        sat_frac = fraction_constant_within_country(df, ind, country_col)

        # Cond 3: max |z|
        max_z = max_abs_z_after_standardize(df, ind)
        exp_max_z = expected_max_abs_z(n_total)
        z_ratio = max_z / exp_max_z if exp_max_z > 0 else np.nan

        # Cond 6: within/total variance
        wt_ratio = within_total_variance_ratio(df, ind, country_col)

        rows.append({
            'indicator': ind,
            'sup_wald': rs['sup_wald'],
            'best_t': rs['best_t'],
            'beta_pre': rs['beta_pre'],
            'beta_post': rs['beta_post'],
            'abs_beta_shift': rs['abs_beta_shift'],
            'sigma_ratio': rs['sigma_ratio'],
            'stat_sig_regime_shift': stat_sig,
            'practical_regime_shift': practical,
            'flag_regime_shift': flag_regime,
            'saturation_frac': sat_frac,
            'flag_saturation': sat_frac > SATURATION_THRESHOLD,
            'max_abs_z': max_z,
            'z_ratio_to_expected': z_ratio,
            'flag_outlier': z_ratio > OUTLIER_FLAG_RATIO,
            'drop_outlier': z_ratio > OUTLIER_DROP_RATIO,
            'within_total_ratio': wt_ratio,
            'flag_low_within': wt_ratio < WITHIN_TOTAL_RATIO_THRESHOLD if np.isfinite(wt_ratio) else False,
        })

    per_indicator_df = pd.DataFrame(rows)

    # Cond 4: collinearity
    corr_matrix = df[indicators].corr().values
    n = len(indicators)
    high_corr_pairs = []
    for i in range(n):
        for j in range(i+1, n):
            if abs(corr_matrix[i,j]) > COLLINEARITY_THRESHOLD:
                high_corr_pairs.append((indicators[i], indicators[j], corr_matrix[i,j]))

    # Cond 5: cross-indicator regime shift
    if 'year' in df.columns:
        median_year = df['year'].median()
        df_pre = df[df['year'] < median_year]
        df_post = df[df['year'] >= median_year]
        if len(df_pre) >= 50 and len(df_post) >= 50:
            corr_pre = df_pre[indicators].corr()
            corr_post = df_post[indicators].corr()
            corr_diff = corr_post - corr_pre
            upper_tri = np.triu(np.ones(n, dtype=bool), k=1)
            max_abs_corr_shift = float(np.abs(corr_diff.values[upper_tri]).max())
        else:
            max_abs_corr_shift = np.nan
    else:
        max_abs_corr_shift = np.nan

    # Cond 7: T/N
    N_block = len(indicators)
    tn_ratio = T_per_country / N_block

    summary = {
        'substrate': substrate_name,
        'n_countries': n_countries, 'T_per_country': T_per_country,
        'n_indicators': len(indicators),
        'expected_max_z': expected_max_abs_z(n_total),
        'n_stat_sig_regime_shift': int(per_indicator_df['stat_sig_regime_shift'].sum()),
        'n_practical_regime_shift': int(per_indicator_df['practical_regime_shift'].sum()),
        'n_flag_regime_shift_joint': int(per_indicator_df['flag_regime_shift'].sum()),
        'n_flag_saturation': int(per_indicator_df['flag_saturation'].sum()),
        'n_flag_outlier': int(per_indicator_df['flag_outlier'].sum()),
        'n_drop_outlier': int(per_indicator_df['drop_outlier'].sum()),
        'n_flag_low_within': int(per_indicator_df['flag_low_within'].sum()),
        'n_high_corr_pairs': len(high_corr_pairs),
        'high_corr_pairs': high_corr_pairs,
        'max_abs_corr_shift_pre_post_median': max_abs_corr_shift,
        'flag_corr_regime_shift': max_abs_corr_shift > CORR_REGIME_SHIFT_THRESHOLD if np.isfinite(max_abs_corr_shift) else False,
        'tn_ratio': tn_ratio,
        'flag_low_power': tn_ratio < TN_POWER_THRESHOLD,
    }

    # Print per-indicator detail (sorted by sup-Wald)
    print("\n  Per-indicator EDA scores (sorted by sup-Wald):")
    print(f"    {'indicator':30s}  {'supW':>8s}  {'|Δβ|':>5s}  {'σratio':>7s}  "
          f"{'sat':>5s}  {'maxZ':>6s}  {'zRatio':>7s}  flags")
    print("    " + "-" * 110)
    sorted_df = per_indicator_df.sort_values('sup_wald', ascending=False)
    for r in sorted_df.itertuples():
        flags = []
        if r.flag_regime_shift:    flags.append('REGIME')
        elif r.stat_sig_regime_shift: flags.append('regime-stat-only')
        if r.flag_saturation:      flags.append('SAT')
        if r.drop_outlier:         flags.append('OUTLIER-DROP')
        elif r.flag_outlier:       flags.append('OUTLIER-FLAG')
        if r.flag_low_within:      flags.append('LOW-W')
        flag_str = ', '.join(flags) if flags else '-'
        sw_str = f"{r.sup_wald:8.2f}" if np.isfinite(r.sup_wald) else f"{'NaN':>8s}"
        bs_str = f"{r.abs_beta_shift:5.2f}" if np.isfinite(r.abs_beta_shift) else f"{'NaN':>5s}"
        sr_str = f"{r.sigma_ratio:7.2f}" if (np.isfinite(r.sigma_ratio) and r.sigma_ratio < 100) else f"{'>100':>7s}"
        print(f"    {r.indicator:30s}  {sw_str}  {bs_str}  {sr_str}  "
              f"{r.saturation_frac:>5.2f}  {r.max_abs_z:>6.2f}  {r.z_ratio_to_expected:>7.2f}  {flag_str}")

    print(f"\n  Panel-level diagnostics:")
    print(f"    High-corr pairs (|r|>{COLLINEARITY_THRESHOLD}): {len(high_corr_pairs)}")
    for a, b, r in high_corr_pairs[:5]:
        print(f"      {a:30s} <-> {b:30s} r={r:+.3f}")
    if len(high_corr_pairs) > 5:
        print(f"      ... ({len(high_corr_pairs) - 5} more)")
    print(f"    Cross-indicator corr regime shift (pre/post median year): "
          f"{max_abs_corr_shift:.3f} {'[FLAG]' if max_abs_corr_shift > CORR_REGIME_SHIFT_THRESHOLD else ''}")
    print(f"    T/N ratio: {tn_ratio:.2f} {'[LOW POWER]' if tn_ratio < TN_POWER_THRESHOLD else '[OK]'}")

    print(f"\n  Aggregate flag counts:")
    print(f"    Regime shift (joint stat+practical):    {summary['n_flag_regime_shift_joint']:>3d} indicators")
    print(f"    Regime shift (stat sig only, sup-W):    {summary['n_stat_sig_regime_shift']:>3d} indicators")
    print(f"    Regime shift (practical only):          {summary['n_practical_regime_shift']:>3d} indicators")
    print(f"    Saturation:                             {summary['n_flag_saturation']:>3d} indicators")
    print(f"    Outlier flag (>1.5×):                   {summary['n_flag_outlier']:>3d} indicators")
    print(f"    Outlier drop (>2.0×):                   {summary['n_drop_outlier']:>3d} indicators")
    print(f"    Low within-variance:                    {summary['n_flag_low_within']:>3d} indicators")

    return per_indicator_df, summary



# §4. Load substrates and run


df_indices = pd.read_csv(INDICES_CSV).rename(columns={'country_id': 'country_name'})
df_indices = df_indices.sort_values(['country_name', 'year']).reset_index(drop=True)
INDICES_COLS = [c for c in df_indices.columns if c not in ['country_name', 'year']]

df_indicators = pd.read_csv(INDICATORS_CSV).sort_values(['country_name', 'year']).reset_index(drop=True)
INDICATORS_COLS_FULL = [c for c in df_indicators.columns
                        if c not in ['country_id', 'country_name', 'year']]

df_wbwpp = pd.read_csv(WBWPP_CSV).sort_values(['country_name', 'year']).reset_index(drop=True)
WBWPP_COLS_FULL = [c for c in df_wbwpp.columns
                    if c not in ['country_name', 'year', 'iso3', 'country_code', 'ISO']]

print(f"Loaded:")
print(f"  V-Dem indices:    {df_indices.shape}, {len(INDICES_COLS)} indices")
print(f"  V-Dem indicators: {df_indicators.shape}, {len(INDICATORS_COLS_FULL)} indicators")
print(f"  WB-WPP:           {df_wbwpp.shape}, {len(WBWPP_COLS_FULL)} indicators")

T_indices = df_indices.groupby('country_name').size().iloc[0]
T_indicators = df_indicators.groupby('country_name').size().iloc[0]
T_wbwpp = df_wbwpp.groupby('country_name').size().iloc[0]

results = {}
per_ind_indices, summary_indices = evaluate_substrate(
    df_indices, INDICES_COLS, 'V-Dem indices (long_16var)', T_indices)
results['indices'] = (per_ind_indices, summary_indices)

per_ind_indicators, summary_indicators = evaluate_substrate(
    df_indicators, INDICATORS_COLS_FULL, 'V-Dem indicators (full 70)', T_indicators)
results['indicators'] = (per_ind_indicators, summary_indicators)

per_ind_wbwpp, summary_wbwpp = evaluate_substrate(
    df_wbwpp, WBWPP_COLS_FULL, 'WB-WPP (full)', T_wbwpp)
results['wbwpp'] = (per_ind_wbwpp, summary_wbwpp)



# §5. Conditions × substrates summary


print("\n\n" + "=" * 110)
print("CONDITIONS × SUBSTRATES SUMMARY (v2 with effect-size)")
print("=" * 110)
substrates_order = [
    ('V-Dem indices', summary_indices),
    ('V-Dem indicators', summary_indicators),
    ('WB-WPP', summary_wbwpp),
]
print(f"{'condition':55s}  " + "  ".join(f"{name:>22s}" for name, _ in substrates_order))
print("-" * 130)

def fmt_count(n, total):
    return f"{n}/{total}"

print(f"{'1. Regime shift (sup-W>8.85 AND |Δβ|>.05 OR σr>1.5)':55s}  " +
      "  ".join(f"{fmt_count(s['n_flag_regime_shift_joint'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'   - of which stat-sig only (sup-W alone)':55s}  " +
      "  ".join(f"{fmt_count(s['n_stat_sig_regime_shift'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'2. Saturation (>30% countries flat)':55s}  " +
      "  ".join(f"{fmt_count(s['n_flag_saturation'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'3a. Outlier flag (max|z| > 1.5× expected)':55s}  " +
      "  ".join(f"{fmt_count(s['n_flag_outlier'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'3b. Outlier drop (max|z| > 2× expected)':55s}  " +
      "  ".join(f"{fmt_count(s['n_drop_outlier'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'4. Collinear pairs (|r| > 0.99)':55s}  " +
      "  ".join(f"{s['n_high_corr_pairs']:>22d}" for _, s in substrates_order))
print(f"{'5. Max corr shift (pre/post median year)':55s}  " +
      "  ".join(f"{s['max_abs_corr_shift_pre_post_median']:>22.3f}"
                for _, s in substrates_order))
print(f"{'6. Low within/total var (<0.10)':55s}  " +
      "  ".join(f"{fmt_count(s['n_flag_low_within'], s['n_indicators']):>22s}"
                for _, s in substrates_order))
print(f"{'7. T/N ratio (need ≥4)':55s}  " +
      "  ".join(f"{s['tn_ratio']:>22.2f}" for _, s in substrates_order))



# §6. List the actually-flagged indicators per substrate


print("\n\n" + "=" * 80)
print("INDICATORS FLAGGED FOR DROPPING/INSPECTION (joint criteria)")
print("=" * 80)
for key, (per_ind, _) in results.items():
    print(f"\n  {results[key][1]['substrate']}:")
    flagged = per_ind[per_ind['flag_regime_shift'] | per_ind['flag_saturation']
                       | per_ind['drop_outlier'] | per_ind['flag_low_within']]
    for r in flagged.itertuples():
        reasons = []
        if r.flag_regime_shift:
            reasons.append(f"REGIME (supW={r.sup_wald:.1f}, |Δβ|={r.abs_beta_shift:.2f}, σr={r.sigma_ratio:.2f})")
        if r.flag_saturation:
            reasons.append(f"SAT ({r.saturation_frac:.0%} const)")
        if r.drop_outlier:
            reasons.append(f"OUTLIER-DROP (z={r.max_abs_z:.1f}, ratio={r.z_ratio_to_expected:.2f})")
        if r.flag_low_within:
            reasons.append(f"LOW-W (w/t={r.within_total_ratio:.3f})")
        print(f"    {r.indicator:30s} -> {'; '.join(reasons)}")
    if len(flagged) == 0:
        print("    (none flagged for drop)")



# §7. Save


for key, (per_ind, summary) in results.items():
    per_ind.to_csv(os.path.join(OUT_DIR, f'eda_v2_per_indicator_{key}.csv'), index=False)

with open(os.path.join(OUT_DIR, 'eda_v2_summary.json'), 'w') as f:
    summary_clean = {k: {kk: vv for kk, vv in s.items() if kk != 'high_corr_pairs'}
                     for k, (_, s) in results.items()}
    for k, (_, s) in results.items():
        summary_clean[k]['high_corr_pairs'] = [
            {'a': a, 'b': b, 'r': float(r)} for a, b, r in s['high_corr_pairs'][:20]
        ]
    json.dump(summary_clean, f, indent=2, default=str)

print(f"\nSaved to {OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded:
  V-Dem indices:    (6675, 18), 16 indices
  V-Dem indicators: (8512, 73), 70 indicators
  WB-WPP:           (9688, 23), 21 indicators

Substrate: V-Dem indices (long_16var)
  N rows: 6675, countries: 89, T/country: 75
  Expected max |z| under normality: 3.636

  Per-indicator EDA scores (sorted by sup-Wald):
    indicator                           supW   |Δβ|   σratio    sat    maxZ   zRatio  flags
    --------------------------------------------------------------------------------------------------------------
    Suffrage                        19560.04   0.92    56.90   0.46    3.77     1.04  REGIME, SAT
    Freedom_of_association             67.11   0.03     1.15   0.00    1.71     0.47  regime-stat-only
    Freedom_of_expression              62.32   0.03     1.28   0.00    1.72     0.47  regime-stat-only
    Civil_participation                55

#### Cell §6.2b — V-Dem variance ratios for R1-hetero design parameter

This cell computes the V-Dem indicator variance heterogeneity metric used as the calibration target for R1-hetero (Appendix B / Appendix N.1).

We compute three definitions on each of three blocks (size-13 reflective,
size-12 post-Layer-3-repair, full size-16 indicator pool) so a downstream
reader can see which definition the R1-hetero generator targets and how
alternative definitions would shift the R1-hetero parameter:

-   raw       : pooled indicator variance (across all country-years)
-   within    : within-country demeaned variance
-   ar1_resid : AR(1) residual variance per indicator

**Output:** vdem_variance_ratios_outputs/
-   vdem_variance_ratios.csv  -- 9 rows (3 blocks x 3 metrics)

**Runtime:** ~1 minute on Colab Pro CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_PATH = os.path.join(DRIVE_DIR, 'data', 'my_data_75_years.csv')
OUT_DIR   = os.path.join(DRIVE_DIR, 'vdem_variance_ratios_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")


# §1. Configuration

# Size-13 reflective block per groups_identified.json (the §7.1 canonical block,
# with Elected_officials and Equal_protection)
SIZE_13_BLOCK = [
    'Freedom_of_expression', 'Freedom_of_association', 'Clean_elections',
    'Elected_officials', 'Individual_liberty', 'Judicial_constraints',
    'Legislative_constraints', 'Civil_participation', 'Local_government',
    'Deliberative', 'Equal_access', 'Equal_distribution', 'Equal_protection',
]

# Size-12 = size-13 minus Elected_officials (Layer 3 repair block)
SIZE_12_BLOCK = [c for c in SIZE_13_BLOCK if c != 'Elected_officials']


# §2. Variance computation utilities

def variance_metrics(panel_df, indicator_cols):
    """Returns (raw_var, within_var, ar1_resid_var) per indicator."""
    out = {}

    # Raw pooled variance
    raw_var = panel_df[indicator_cols].var(ddof=0)
    out['raw'] = raw_var

    # Within-country demeaned variance
    demeaned = panel_df.copy()
    for col in indicator_cols:
        country_mean = demeaned.groupby('country_id')[col].transform('mean')
        demeaned[col] = demeaned[col] - country_mean
    within_var = demeaned[indicator_cols].var(ddof=0)
    out['within'] = within_var

    # AR(1) residual variance per indicator (panel-safe within-country fit)
    ar1_var = {}
    for col in indicator_cols:
        ys, ylags = [], []
        for cid, g in panel_df.sort_values(['country_id', 'year']).groupby('country_id'):
            v = g[col].values.astype(float)
            mask = ~np.isnan(v)
            v = v[mask]
            if len(v) >= 2:
                ys.append(v[1:])
                ylags.append(v[:-1])
        if not ys:
            ar1_var[col] = np.nan
            continue
        Y = np.concatenate(ys)
        Yl = np.concatenate(ylags)
        X = np.column_stack([Yl, np.ones_like(Yl)])
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        resid = Y - X @ beta
        ar1_var[col] = float(resid.var(ddof=0))
    out['ar1_resid'] = pd.Series(ar1_var)
    return out


def ratio_summary(metrics, label):
    rows = []
    for kind, vals in metrics.items():
        v = vals.dropna()
        if len(v) == 0:
            continue
        v_max, v_min = v.max(), v.min()
        ratio = v_max / v_min if v_min > 0 else np.nan
        rows.append({
            'block': label,
            'metric': kind,
            'min': float(v_min),
            'max': float(v_max),
            'ratio_max_over_min': float(ratio),
            'argmax_indicator': str(v.idxmax()),
            'argmin_indicator': str(v.idxmin()),
        })
    return rows


# §3. Compute for the three blocks

print(f"\nReading: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"Panel shape: {df.shape}")

for col in SIZE_13_BLOCK:
    assert col in df.columns, f"Missing column: {col}"

print("\nComputing variance metrics...")
rows = []

print("  size-13 reflective block (with Elected_officials)...")
metrics_13 = variance_metrics(df, SIZE_13_BLOCK)
rows += ratio_summary(metrics_13, 'size_13')

print("  size-12 (Layer 3 repair, Elected_officials removed)...")
metrics_12 = variance_metrics(df, SIZE_12_BLOCK)
rows += ratio_summary(metrics_12, 'size_12')

SIZE_16_BLOCK = [c for c in df.columns if c not in ['country_id', 'year']]
print(f"  size-{len(SIZE_16_BLOCK)} (full indicator pool)...")
metrics_16 = variance_metrics(df, SIZE_16_BLOCK)
rows += ratio_summary(metrics_16, f'size_{len(SIZE_16_BLOCK)}')


# §4. Save and summarize

result = pd.DataFrame(rows)
out_csv = os.path.join(OUT_DIR, 'vdem_variance_ratios.csv')
result.to_csv(out_csv, index=False)
print(f"\nWrote: {out_csv}")


# §5. Print paper-ready output

print()
print("=" * 70)
print("V-Dem max/min variance ratios across blocks and metrics")
print("=" * 70)
print(result.to_string(index=False))
print()

# Specifically report the size-12 AR(1) residual variance ratio,
# which is what the paper cites as the R1-hetero design parameter.
canonical = result[(result['block'] == 'size_12') &
                   (result['metric'] == 'ar1_resid')]
if len(canonical):
    r = canonical.iloc[0]
    print("Paper-cited value (R1-hetero design parameter, Appendix B):")
    print(f"  size-12 AR(1) residual variance ratio = {r['ratio_max_over_min']:.4f}")
    print(f"  argmax: {r['argmax_indicator']}")
    print(f"  argmin: {r['argmin_indicator']}")
    print(f"  Paper rounds to ~4.6.")

#### Cell §6.2c — EDA pre/post-median-year max correlation shift

This cell computes the cross-indicator correlation regime-shift metric cited in the EDA appendix (Condition 5): "Element-wise difference between indicator correlation matrices computed on PRE- AND POST-MEDIAN-YEAR subsets of the panel; report the maximum absolute off-diagonal difference."

Each substrate uses ITS OWN median year (V-Dem indices spans 1950-2024 -> median 1987; V-Dem indicators spans 1970-2025 -> median 1997; WB-WPP spans 1970-2025 -> median 1997). All three substrates are processed in one pass.

The values cited in the paper (EDA appendix Table 15 row 5) are:
-   V-Dem indices (size-16):  0.43  (Suffrage vs Equal_protection)
-   V-Dem indicators (~70):   0.31  (v2ddyrci vs v2ddyror)
-   WB-WPP indicators (~21):  0.26  (wb_pop_growth vs wb_urban_pop_growth)

**Output:** eda_corr_shift_outputs/
-   eda_corr_shift_summary.csv                        -- 3-row summary
-   eda_corr_shift_<substrate>.csv                    -- per-indicator detail

**Runtime:** 7-8 minutes on Colab CPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
OUT_DIR   = os.path.join(DRIVE_DIR, 'eda_corr_shift_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"DRIVE_DIR = {DRIVE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")


# §1. Configuration

SUBSTRATE_FILES = {
    'vdem_indices_size_16':         'my_data_75_years.csv',
    'vdem_indicators_size_70_pool': 'my_data_VDem_Ind.csv',
    'wbwpp_indicators_size_21_pool':'my_data_WB.csv',
}

# Per EDA-appendix Condition 5: split at panel median year.
SPLIT_MODE = 'median_year'

# Columns to exclude from indicator set (metadata)
META_COLS = {'country_id', 'year', 'country', 'iso3', 'country_name',
             'country_text_id', 'country_code', 'iso_code', 'iso2', 'COWcode'}


# §2. Compute max corr shift

def max_corr_shift(panel_df, indicator_cols, split_mode=SPLIT_MODE):
    """For each indicator, compute max |delta_corr| against any other indicator
    when the panel is split at its median year. Returns:
      (per_indicator_df sorted by max_abs_corr_shift desc, split_year_used)
    """
    # Coerce indicator columns to numeric; drop columns that are entirely
    # non-numeric (string country names, ISO codes, etc.)
    numeric_cols = []
    for c in indicator_cols:
        coerced = pd.to_numeric(panel_df[c], errors='coerce')
        if coerced.notna().sum() >= 10:
            numeric_cols.append(c)

    panel_numeric = panel_df.copy()
    for c in numeric_cols:
        panel_numeric[c] = pd.to_numeric(panel_numeric[c], errors='coerce')

    if split_mode == 'median_year':
        split_year = int(panel_numeric['year'].median())
    else:
        split_year = int(split_mode)

    pre = panel_numeric[panel_numeric['year'] < split_year][numeric_cols]
    post = panel_numeric[panel_numeric['year'] >= split_year][numeric_cols]
    if len(pre) < 10 or len(post) < 10:
        raise ValueError(f"Insufficient pre/post split: pre={len(pre)}, post={len(post)}")

    corr_pre = pre.corr()
    corr_post = post.corr()
    common = [c for c in numeric_cols
              if c in corr_pre.columns and c in corr_post.columns]
    corr_pre = corr_pre.loc[common, common].values
    corr_post = corr_post.loc[common, common].values

    abs_shift = np.abs(corr_pre - corr_post)
    np.fill_diagonal(abs_shift, 0.0)

    rows = []
    for i, ind in enumerate(common):
        row_shifts = abs_shift[i, :].copy()
        finite_mask = np.isfinite(row_shifts)
        if not finite_mask.any():
            continue
        masked = np.where(finite_mask, row_shifts, -np.inf)
        partner_idx = int(np.argmax(masked))
        max_val = float(row_shifts[partner_idx])
        if not np.isfinite(max_val):
            continue
        rows.append({
            'indicator': ind,
            'max_abs_corr_shift': max_val,
            'partner_indicator': common[partner_idx],
        })
    per_ind = pd.DataFrame(rows).sort_values('max_abs_corr_shift', ascending=False)
    return per_ind, split_year


# §3. Sweep the three substrates

print()
summary_rows = []
for label, filename in SUBSTRATE_FILES.items():
    print("=" * 70)
    print(f"Substrate: {label}")
    print("=" * 70)
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing data file: {path}")

    print(f"  Reading: {path}")
    df = pd.read_csv(path)
    print(f"  Shape: {df.shape}")
    if 'year' not in df.columns:
        raise ValueError(f"No 'year' column in {filename}")

    indicator_cols = [c for c in df.columns if c not in META_COLS]
    print(f"  Indicators: {len(indicator_cols)}")

    per_ind, split_year_used = max_corr_shift(df, indicator_cols)
    out_csv = os.path.join(OUT_DIR, f'eda_corr_shift_{label}.csv')
    per_ind.to_csv(out_csv, index=False)
    print(f"  Split year:  {split_year_used} "
          f"(panel range [{df['year'].min()}, {df['year'].max()}])")
    print(f"  Wrote per-indicator file: {out_csv}")

    if len(per_ind):
        top = per_ind.iloc[0]
        print(f"  Max corr shift: {top['max_abs_corr_shift']:.4f} "
              f"({top['indicator']} vs {top['partner_indicator']})")
        summary_rows.append({
            'substrate': label,
            'data_file': filename,
            'split_year': split_year_used,
            'max_abs_corr_shift': float(top['max_abs_corr_shift']),
            'argmax_indicator': top['indicator'],
            'partner_indicator': top['partner_indicator'],
            'n_indicators': len(per_ind),
        })


# §4. Save summary and print paper-ready output

summary = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUT_DIR, 'eda_corr_shift_summary.csv')
summary.to_csv(summary_csv, index=False)

print()
print("=" * 70)
print(f"Summary written: {summary_csv}")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("Paper-cited values (EDA appendix Table 15, Condition 5 row):")
print("  V-Dem indices (size-16):    paper cites 0.43")
print("  V-Dem indicators (~70):     paper cites 0.31")
print("  WB-WPP indicators (~21):    paper cites 0.26")

## Section 7 — Paper §6.4 adversarial robustness

The defense-Pareto follow-up to the §3.8 main adversary sweep. Uses the paper's actual tvNAR-based CRPS scoring protocol (rather than the naive Y_t = W·Y_{t-1} regression) and reports both unconstrained and constrained protocols. Produces the Pareto-frontier analysis in §6.4 and Appendix P.

### Cell §7.1 — Adversary defense Pareto (CANONICAL)

**Paper sections:** §6.4, Appendix P.

The canonical adversary defense analysis. Computes ΔPR and CRPS-excess for both honest methods and the rank-1 adversary across λ ∈ {0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0}, under both unconstrained and constrained CRPS protocols. Reports whether honest methods that pass ΔPR pay strictly more CRPS-excess than the adversary at matching ΔPR (Defense 2 verdict).

Uses the corrected scoring protocol (`score_W_against_panel` from `random_baselines_4method_corrected.py`): each method's W is treated as a structural mask, tvNAR fits the actual predictor with W constraining the structure, and CRPS is computed via bootstrap samples. The §3.8 main sweep used the simpler naive protocol; this cell is the canonical defense check.

**Inputs:** Adversary sweep CSV from §3.8; V-Dem panel.
 - V-Dem panel:           /content/drive/MyDrive/NeurIPS2026_1296/data/my_data_75_years.csv
-  V-Dem groups:          /content/drive/MyDrive/NeurIPS2026_1296/lsc_diagnostic_outputs/long_16var/groups_identified.json
-  Method outputs pickle: /content/drive/MyDrive/NeurIPS2026_1296/random_baselines_4method_outputs/long_16var/method_outputs.pkl
                         (keys: 'config_name', 'method_outputs', 'panel_summary'
                         — we unwrap to access 'method_outputs')
-  Adversary results CSV: /content/drive/MyDrive/NeurIPS2026_1296/adversary_rank1_block_8_outputs/adversary_sweep_aggregate.csv
-  Adversary fits pickle: /content/drive/MyDrive/NeurIPS2026_1296/adversary_rank1_block_8_outputs/adversary_fits.pkl
                         (saved per-λ A matrices — see note below)
-  Source of paper's CRPS pipeline:
                         /content/drive/MyDrive/NeurIPS2026_1296/src/random_baselines_4method_corrected.py
                         (we exec the function-definitions block in a
                         controlled namespace to call score_W_against_panel)

This notebook does the right defense check:
  1. Apply the paper's CRPS protocol (`score_W_against_panel`) to:
     - The 5 honest methods' fitted W on V-Dem (loaded from the
       `random_baselines_4method_outputs/long_16var/method_outputs.pkl`)
     - The adversary at each λ from Block 8
  2. Compute ΔPR + parametric null for each.
  3. Plot Pareto picture: (CRPS, ΔPR) for honest methods (points) and
     adversary (curve). Defense holds if honest methods sit *above* the
     adversary curve in CRPS at matching ΔPR.

**Outputs:** `combined_csv` (all methods + adversary), `defense_csv` (per-method defense verdicts).

**Runtime:** ~10 minutes on Colab Pro G4 GPU.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json, pickle
import numpy as np
import pandas as pd
from scipy import stats as scstats

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
LSC_DIR   = os.path.join(DRIVE_DIR, 'lsc_diagnostic_outputs', 'long_16var')
B8_DIR    = os.path.join(DRIVE_DIR, 'adversary_rank1_block_8_outputs')
OUT_DIR   = os.path.join(DRIVE_DIR, 'adversary_defense_pareto_block_8a_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Possible paths for method_outputs.pkl (matches lsc_diagnostic_vdem_long's logic)
METHOD_OUTPUTS_PATHS = [
    os.path.join(DRIVE_DIR, 'p2_outputs/method_outputs/method_outputs_long_16var.pkl'),
    os.path.join(DRIVE_DIR, 'method_outputs_long_16var.pkl'),
    os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs/long_16var/method_outputs_long_16var.pkl'),
    os.path.join(DRIVE_DIR, 'random_baselines_4method_outputs/long_16var/method_outputs.pkl'),
]



# §1. Configuration


B_PERMS = 1000
ALPHA = 0.05

# Methods we want to evaluate (those listed in the paper's V-Dem analysis)
HONEST_METHODS = ['ols_var', 'lvg', 'pcmci', 'cmlp', 'navar', 'dynotears']



# §2. Core utilities (same as Block 8)


def fit_ols_var_raw(Y, split_spec):
    """OLS VAR fit, KEEPS diagonal. Use for null-simulated panels.

    This matches §5.7's `fit_ols_var` convention used in the canonical
    multi-criteria pipeline for WB-WPP and V-Dem-60. Keeping the diagonal
    in the null fit is asymmetric with PR_obs (where method-output W has
    zero diagonal by construction), but it is the protocol that produced
    the paper's headline ΔPR values for the non-V-Dem-16 substrates.
    Using it here makes V-Dem-16 ΔPR values comparable across panels.
    """
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    return A_T.T


def fit_ols_var_signed(Y, split_spec):
    """OLS VAR fit, ZEROS diagonal. Use for the OLS VAR baseline on observed data.

    Causal-discovery method outputs W have zero diagonal by construction
    (no self-loops). For an apples-to-apples PR comparison, the OLS VAR
    baseline on the same observed panel must also be reported with zero
    diagonal. Do NOT use this on null-simulated panels — that path goes
    through fit_ols_var_raw.
    """
    A = fit_ols_var_raw(Y, split_spec)
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric_with_A(A_obs, Y, split_spec, ref_indices,
                                        B, seed, U_resample, T_resample):
    """Use a pre-computed A as the observed coefficient matrix; null
    distribution is OLS VAR refits on AR(1) sims (the standard parametric
    null procedure used throughout the paper)."""
    W_block = A_obs[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        # Protocol B: keep diagonal in null OLS VAR fit (matches §5.7 convention,
        # which produced the canonical multi_criteria_summary.csv files for
        # WB-WPP and V-Dem-60). DO NOT replace with fit_ols_var_signed here.
        A_sim = fit_ols_var_raw(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)


# Adversary fit. We need to re-fit per λ here because Block 8's
# CSV stored only the per-run summary stats, not the fitted A matrices. Refits
# are fast (~1-2s per λ on V-Dem) so this is not a bottleneck.
def fit_olsvar_rank1_regularized(Y, split_spec, ref_indices, lam,
                                  max_iters=200, tol=1e-7):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    n_obs, N = Yt.shape
    XtX = Ylag.T @ Ylag
    XtY = Ylag.T @ Yt
    ref = np.array(ref_indices, dtype=int)
    n_ref = len(ref)
    off_mask = ~np.eye(n_ref, dtype=bool)
    XtX_scale = float(np.trace(XtX) / N)
    lam_eff = lam * XtX_scale

    try:
        A_T = np.linalg.solve(XtX, XtY)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(XtX) @ XtY
    A = A_T.T
    np.fill_diagonal(A, 0.0)

    if lam == 0:
        return A

    Aref = A[np.ix_(ref, ref)].copy()
    np.fill_diagonal(Aref, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Aref, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0])
        v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        u = np.zeros(n_ref); v = np.zeros(n_ref)

    prev_loss = np.inf
    for it in range(max_iters):
        new_A = np.zeros_like(A)
        for j in range(N):
            if j not in ref:
                try:
                    a_j = np.linalg.solve(XtX, XtY[:, j])
                except np.linalg.LinAlgError:
                    a_j = np.linalg.pinv(XtX) @ XtY[:, j]
            else:
                j_pos = np.where(ref == j)[0][0]
                target_full = np.zeros(N)
                reg_diag = np.zeros(N)
                for k in range(n_ref):
                    if k == j_pos:
                        continue
                    target_full[ref[k]] = u[j_pos] * v[k]
                    reg_diag[ref[k]] = lam_eff
                M = XtX + np.diag(reg_diag)
                b = XtY[:, j] + reg_diag * target_full
                try:
                    a_j = np.linalg.solve(M, b)
                except np.linalg.LinAlgError:
                    a_j = np.linalg.pinv(M) @ b
            new_A[j, :] = a_j
        np.fill_diagonal(new_A, 0.0)

        Aref = new_A[np.ix_(ref, ref)].copy()
        np.fill_diagonal(Aref, 0.0)
        for _ in range(20):
            nu_ = (Aref * off_mask) @ v
            du_ = off_mask @ (v * v)
            u_new = np.where(du_ > 1e-12, nu_ / np.maximum(du_, 1e-12), 0.0)
            nv_ = (Aref * off_mask).T @ u_new
            dv_ = off_mask.T @ (u_new * u_new)
            v_new = np.where(dv_ > 1e-12, nv_ / np.maximum(dv_, 1e-12), 0.0)
            u, v = u_new, v_new

        resid = Yt - Ylag @ new_A.T
        pred_loss = float(np.sum(resid ** 2))
        Aref_post = new_A[np.ix_(ref, ref)].copy()
        np.fill_diagonal(Aref_post, 0.0)
        reg_loss = float(np.sum(((Aref_post - np.outer(u, v)) * off_mask) ** 2))
        total_loss = pred_loss + lam_eff * reg_loss

        A = new_A
        if abs(prev_loss - total_loss) / max(abs(prev_loss), 1.0) < tol:
            return A
        prev_loss = total_loss
    return A


print("Utilities defined.")



# §3. Source the paper's CRPS pipeline
# Function-definitions block.


PIPELINE_PATH = os.path.join(DRIVE_DIR, 'src', 'random_baselines_4method_corrected.py')
if not os.path.exists(PIPELINE_PATH):
    # Fallback: maybe it's at a different location
    fallbacks = [
        os.path.join(DRIVE_DIR, 'random_baselines_4method_corrected.py'),
        os.path.join(DRIVE_DIR, 'src/p2/random_baselines_4method_corrected.py'),
    ]
    for fb in fallbacks:
        if os.path.exists(fb):
            PIPELINE_PATH = fb
            break
    else:
        raise FileNotFoundError(
            f"random_baselines_4method_corrected.py not found at:\n  "
            + "\n  ".join([os.path.join(DRIVE_DIR, 'src', 'random_baselines_4method_corrected.py')] + fallbacks)
        )

print(f"Reading paper's CRPS pipeline from: {PIPELINE_PATH}")

with open(PIPELINE_PATH) as f:
    src_lines = f.readlines()
print(f"  total lines: {len(src_lines)}")

# Find the function-definitions block boundaries.
# Start: just before "def _as_2d_float" (function defs we need start there)
# End: just before "## 6. Method runners" markdown header (function defs end there)
start_idx = None
end_idx = None
for i, line in enumerate(src_lines):
    if 'def _as_2d_float' in line and start_idx is None:
        start_idx = i
    if '## 6. Method runners' in line and end_idx is None:
        end_idx = i
        break
if start_idx is None or end_idx is None:
    raise RuntimeError(f"Could not locate function-definitions block in {PIPELINE_PATH}")

fn_block = ''.join(src_lines[start_idx:end_idx])
# Prepend the necessary imports manually
preamble = (
    "from __future__ import annotations\n"
    "from dataclasses import dataclass\n"
    "from typing import Dict, List, Optional, Tuple, Union, Any\n"
    "import numpy as np\n"
    "ArrayLike = Union[np.ndarray]\n"
)
fn_block = preamble + fn_block
print(f"  function-definitions block: lines {start_idx+1}-{end_idx} "
      f"({end_idx - start_idx} lines)")

# Execute in a controlled namespace
crps_ns = {'__name__': '__crps_pipeline__'}
try:
    exec(fn_block, crps_ns)
    print(f"  Successfully sourced CRPS pipeline functions.")
except Exception as e:
    raise RuntimeError(f"Failed to exec CRPS pipeline source: {e}")

# Verify the functions we need are present
required_fns = ['score_W_against_panel', 'fit_tvNAR_lambda_paths',
                'fit_tvNAR_lambda_paths_constrained', 'mean_crps',
                'crps_1d_from_samples', 'tvnar_mean_one_step',
                'build_panel_lag_pairs']
missing = [fn for fn in required_fns if fn not in crps_ns]
if missing:
    raise RuntimeError(f"Missing required functions after exec: {missing}")

score_W_against_panel = crps_ns['score_W_against_panel']
print(f"  score_W_against_panel signature verified.")



# §4. Load V-Dem panel (same as Block 8)


print("\n" + "=" * 70)
print("Loading V-Dem")
print("=" * 70)

df_vdem = pd.read_csv(os.path.join(DATA_DIR, 'my_data_75_years.csv'))
with open(os.path.join(LSC_DIR, 'groups_identified.json')) as f:
    groups_data = json.load(f)
all_var_names = groups_data['variable_names']
ref_block = groups_data['groups'][0]
vdem_ref_indices = ref_block['indices']

mask = ~df_vdem[all_var_names].isna().any(axis=1)
df_vdem = df_vdem[mask].sort_values(['country_id', 'year']).reset_index(drop=True)
split_spec_vdem = (df_vdem.groupby('country_id', sort=False)['year']
                          .count().tolist())
Y_vdem = df_vdem[all_var_names].values
Y_vdem = (Y_vdem - Y_vdem.mean(0)) / (Y_vdem.std(0) + 1e-12)
T_vdem = max(split_spec_vdem); U_vdem = len(split_spec_vdem)
print(f"V-Dem: shape={Y_vdem.shape}, U={U_vdem}, T={T_vdem}, "
      f"ref_block_size={len(vdem_ref_indices)}")



# §5. Load saved method outputs (W matrices for the 5 honest methods)
#
# The pickle structure: top-level keys are
# {'config_name', 'method_outputs', 'panel_summary'}. We unwrap to the
# 'method_outputs' inner dict, which is keyed by method name with each
# value containing the fitted W_hat plus metadata.


print("\n" + "=" * 70)
print("Loading method_outputs.pkl")
print("=" * 70)

method_outputs_pickle = None
methods_path_used = None
for path in METHOD_OUTPUTS_PATHS:
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                method_outputs_pickle = pickle.load(f)
            methods_path_used = path
            print(f"  Loaded: {path}")
            break
        except Exception as e:
            print(f"  Failed to load {path}: {e}")

if method_outputs_pickle is None:
    raise FileNotFoundError(
        f"Could not find method_outputs.pkl at any of:\n  "
        + "\n  ".join(METHOD_OUTPUTS_PATHS)
    )

# Unwrap: pickle has structure {'config_name', 'method_outputs', 'panel_summary'}
print(f"\nTop-level keys in pickle: {list(method_outputs_pickle.keys())}")
if 'method_outputs' in method_outputs_pickle:
    method_outputs = method_outputs_pickle['method_outputs']
    print(f"  Unwrapped: pickle['method_outputs'] has {len(method_outputs)} entries")
else:
    # Fallback: maybe it's already at the top level
    method_outputs = method_outputs_pickle
    print(f"  No 'method_outputs' wrapper key; using top-level dict directly")

print(f"\nMethod names available: {list(method_outputs.keys())}")

# Extract W_hat per method
method_W_hats = {}
for method_name, method_data in method_outputs.items():
    if isinstance(method_data, dict) and 'W_hat' in method_data:
        W = method_data['W_hat']
        if W is not None and hasattr(W, 'shape'):
            method_W_hats[method_name] = np.array(W, dtype=float)

print(f"\nMethods with valid W_hat: {list(method_W_hats.keys())}")
N_vdem = len(all_var_names)
for m, W in list(method_W_hats.items()):
    if W.shape != (N_vdem, N_vdem):
        print(f"  WARNING: {m} W_hat shape {W.shape} != ({N_vdem},{N_vdem}); dropping")
        method_W_hats.pop(m)
        continue
    nnz = int((W != 0).sum())
    max_abs = float(np.abs(W).max())
    print(f"  {m}: shape={W.shape}, nnz={nnz}, max|W|={max_abs:.4f}")

if not method_W_hats:
    raise RuntimeError(
        "No valid W_hat entries found in method_outputs.pkl. "
        "Cannot proceed with defense check."
    )



# §6. Refit adversary at each λ and score with the paper's CRPS pipeline



print("\n" + "=" * 70)
print("Loading adversary λ grid + refitting adversary A at each λ")
print("=" * 70)

adv_csv = os.path.join(B8_DIR, 'adversary_sweep_aggregate.csv')
if not os.path.exists(adv_csv):
    raise FileNotFoundError(
        f"Block 8 adversary results not found at {adv_csv}. "
        f"Run adversary_rank1_block_8.py first."
    )
df_adv = pd.read_csv(adv_csv)
print(f"  Loaded adversary CSV: {adv_csv}")
print(f"  λ values: {df_adv['lambda'].tolist()}")

# Refit the adversary at each λ to get the A matrices for paper's CRPS scoring
adversary_A_per_lambda = {}
print(f"\nRefitting adversary at each λ on V-Dem (this is fast — ~1-2s per λ):")
for lam in df_adv['lambda'].tolist():
    t0 = time.time()
    A_adv = fit_olsvar_rank1_regularized(Y_vdem, split_spec_vdem, vdem_ref_indices, lam)
    adversary_A_per_lambda[lam] = A_adv
    print(f"  λ={lam:>5.2f}: refit in {time.time()-t0:.1f}s, "
          f"PR_obs(A_ref) = {offdiag_rank1_fit(A_adv[np.ix_(vdem_ref_indices, vdem_ref_indices)]):.4f}")



# §7. Score honest methods AND adversary using the paper's CRPS pipeline
# The function returns dict with 'CRPS', 'RMSE', 'MAE', 'MSE', 'nnz', plus
# diagnostic fields. We pull out 'CRPS' as the comparable metric.



print("\n" + "=" * 70)
print("Scoring all candidates with paper's CRPS pipeline")
print("=" * 70)

# Paper's exact scoring config (lines 93-98 of random_baselines_4method_corrected.py)
PAPER_H = 20
PAPER_B_BOOT = 300
PAPER_GRID_SIZE = 50
PAPER_BANDWIDTH = 0.2
PAPER_RIDGE = 1e-4
PAPER_EPSILON = 0.01
print(f"\nPaper config: H={PAPER_H}, B_boot={PAPER_B_BOOT}, "
      f"grid_size={PAPER_GRID_SIZE}, bandwidth={PAPER_BANDWIDTH}")
print(f"  We score under BOTH 'unconstrained' and 'constrained' protocols, "
      f"as the paper does.")
print(f"  Paper's Table 4 OLS VAR CRPS = 0.038 — we'll match this in one of "
      f"the two protocols.")


def score_one(W, Y, ss, seed=0):
    """Score W under both unconstrained and constrained protocols, with the
    paper's exact parameter values."""
    out = {}
    for protocol in ('unconstrained', 'constrained'):
        constrain = (protocol == 'constrained')
        try:
            res = score_W_against_panel(
                W=W, Y=Y, split_spec=ss,
                H=PAPER_H, B=PAPER_B_BOOT, grid_size=PAPER_GRID_SIZE,
                constrain=constrain, epsilon=PAPER_EPSILON,
                bandwidth=PAPER_BANDWIDTH, ridge=PAPER_RIDGE,
                rng=np.random.default_rng(seed),
            )
            out[f'crps_{protocol}'] = float(res['CRPS'])
            out[f'status_{protocol}'] = res.get('status', 'ok')
            out[f'n_eval_pairs_{protocol}'] = int(res.get('n_eval_pairs', 0))
        except Exception as e:
            out[f'crps_{protocol}'] = float('nan')
            out[f'status_{protocol}'] = f'{type(e).__name__}: {e}'
            out[f'n_eval_pairs_{protocol}'] = 0
    return out


# OLS VAR baseline
A_olsvar = fit_ols_var_signed(Y_vdem, split_spec_vdem)
print(f"\nOLS VAR baseline (fitted fresh):")
t0 = time.time()
score_olsvar = score_one(A_olsvar, Y_vdem, split_spec_vdem, seed=1)
elapsed = time.time() - t0
print(f"  status (unc): {score_olsvar['status_unconstrained']}")
print(f"  status (con): {score_olsvar['status_constrained']}")
print(f"  CRPS unconstrained: {score_olsvar['crps_unconstrained']:.6f}")
print(f"  CRPS constrained:   {score_olsvar['crps_constrained']:.6f}")
print(f"  paper Table-4 reference: 0.038 (one of these should match)")
print(f"  ({elapsed:.1f}s)")


crps_olsvar_unc = score_olsvar['crps_unconstrained']
crps_olsvar_con = score_olsvar['crps_constrained']
crps_olsvar = crps_olsvar_unc   # primary

# Honest methods
print(f"\nHonest methods:")
honest_results = []

# Add OLS VAR
d_pa, p_pa, pr_obs, mu_pa, sd_pa = compute_delta_pr_parametric_with_A(
    A_olsvar, Y_vdem, split_spec_vdem, vdem_ref_indices,
    B=B_PERMS, seed=42, U_resample=U_vdem, T_resample=T_vdem)
honest_results.append({
    'method': 'ols_var', 'kind': 'honest',
    'PR_obs': pr_obs, 'parametric_delta': d_pa, 'parametric_p': p_pa,
    'crps_unconstrained': crps_olsvar_unc,
    'crps_constrained': crps_olsvar_con,
    'crps_excess_unc': 0.0,
    'crps_excess_con': 0.0,
    'passes_dpr': bool(np.isfinite(d_pa) and d_pa > 0 and p_pa < ALPHA),
})
print(f"  ols_var: PR_obs={pr_obs:.4f}, ΔPR={d_pa:+.4f}, p={p_pa:.4f}")

# Other honest methods
for method_name, W_hat in method_W_hats.items():
    if method_name == 'ols_var':
        continue
    print(f"\n  {method_name}:")
    t0 = time.time()
    # ΔPR + parametric null
    d_pa, p_pa, pr_obs, mu_pa, sd_pa = compute_delta_pr_parametric_with_A(
        W_hat, Y_vdem, split_spec_vdem, vdem_ref_indices,
        B=B_PERMS, seed=hash(method_name) % 100000,
        U_resample=U_vdem, T_resample=T_vdem)
    # CRPS via paper's pipeline (both protocols, paper config)
    score_method = score_one(W_hat, Y_vdem, split_spec_vdem,
                              seed=hash(method_name) % 100000 + 1)
    crps_unc = score_method['crps_unconstrained']
    crps_con = score_method['crps_constrained']
    crps_excess_unc = crps_unc - crps_olsvar_unc
    crps_excess_con = crps_con - crps_olsvar_con
    passes = bool(np.isfinite(d_pa) and d_pa > 0 and p_pa < ALPHA)
    elapsed = time.time() - t0

    honest_results.append({
        'method': method_name, 'kind': 'honest',
        'PR_obs': pr_obs, 'parametric_delta': d_pa, 'parametric_p': p_pa,
        'crps_unconstrained': crps_unc,
        'crps_constrained': crps_con,
        'crps_excess_unc': crps_excess_unc,
        'crps_excess_con': crps_excess_con,
        'passes_dpr': passes,
    })
    print(f"    PR_obs={pr_obs:.4f}, ΔPR={d_pa:+.4f}, p={p_pa:.4f}, pass={'✓' if passes else '✗'}")
    print(f"    CRPS_unc={crps_unc:.6f} (excess={crps_excess_unc:+.6f})")
    print(f"    CRPS_con={crps_con:.6f} (excess={crps_excess_con:+.6f})")
    print(f"    [{elapsed:.1f}s]")

# Adversary at each λ
print(f"\nAdversary λ sweep:")
adversary_results = []
for lam in df_adv['lambda'].tolist():
    print(f"\n  λ={lam:>5.2f}:")
    t0 = time.time()
    A_adv = adversary_A_per_lambda[lam]
    # ΔPR
    d_pa, p_pa, pr_obs, mu_pa, sd_pa = compute_delta_pr_parametric_with_A(
        A_adv, Y_vdem, split_spec_vdem, vdem_ref_indices,
        B=B_PERMS, seed=int(7919 + lam * 1000),
        U_resample=U_vdem, T_resample=T_vdem)
    # CRPS via paper's pipeline (both protocols)
    score_adv = score_one(A_adv, Y_vdem, split_spec_vdem,
                          seed=int(7919 + lam * 1000) + 1)
    crps_unc = score_adv['crps_unconstrained']
    crps_con = score_adv['crps_constrained']
    crps_excess_unc = crps_unc - crps_olsvar_unc
    crps_excess_con = crps_con - crps_olsvar_con
    passes = bool(np.isfinite(d_pa) and d_pa > 0 and p_pa < ALPHA)
    elapsed = time.time() - t0

    adversary_results.append({
        'method': f'adv_lambda_{lam}', 'kind': 'adversary', 'lambda': lam,
        'PR_obs': pr_obs, 'parametric_delta': d_pa, 'parametric_p': p_pa,
        'crps_unconstrained': crps_unc,
        'crps_constrained': crps_con,
        'crps_excess_unc': crps_excess_unc,
        'crps_excess_con': crps_excess_con,
        'passes_dpr': passes,
    })
    print(f"    PR_obs={pr_obs:.4f}, ΔPR={d_pa:+.4f}, p={p_pa:.4f}, pass={'✓' if passes else '✗'}")
    print(f"    CRPS_unc={crps_unc:.6f} (excess={crps_excess_unc:+.6f})")
    print(f"    CRPS_con={crps_con:.6f} (excess={crps_excess_con:+.6f})")
    print(f"    [{elapsed:.1f}s]")

# Save consolidated results
df_honest = pd.DataFrame(honest_results)
df_adv_scored = pd.DataFrame(adversary_results)
df_combined = pd.concat([df_honest, df_adv_scored], ignore_index=True, sort=False)
combined_csv = os.path.join(OUT_DIR, 'all_evaluations.csv')
df_combined.to_csv(combined_csv, index=False)
print(f"\nSaved: {combined_csv}")



# §8. Defense check: efficiency comparison
#
# For each honest method, the adversary's CRPS_excess at matching ΔPR is
# the lower bound on what an honest method "ought" to pay. If honest methods
# sit ABOVE the adversary's CRPS-vs-ΔPR curve at matching ΔPR, the
# adversary's curve is a defensible diagnostic baseline.


print("\n" + "=" * 70)
print("DEFENSE CHECK: honest methods vs adversary trajectory")
print("(All CRPS values now from paper's `score_W_against_panel`,")
print(" using paper's H=20, B=300, grid_size=50, both protocols.)")
print("=" * 70)

print(f"\n{'method':14s} | {'PR_obs':>7s} {'ΔPR':>8s} {'p':>7s} {'pass':>4s} | "
      f"{'CRPS_unc':>8s} {'exc_unc':>8s} | {'CRPS_con':>8s} {'exc_con':>8s}")
print("-" * 100)

# Honest methods
for r in honest_results:
    print(f"{r['method']:14s} | "
          f"{r['PR_obs']:>7.4f} {r['parametric_delta']:>+8.4f} {r['parametric_p']:>7.4f}  "
          f"{'✓' if r['passes_dpr'] else '✗':>3s} | "
          f"{r['crps_unconstrained']:>8.4f} {r['crps_excess_unc']:>+8.4f} | "
          f"{r['crps_constrained']:>8.4f} {r['crps_excess_con']:>+8.4f}")

print()
# Adversary trajectory
for r in adversary_results:
    if r['lambda'] == 0:
        continue
    print(f"adv λ={r['lambda']:>5.2f}    | "
          f"{r['PR_obs']:>7.4f} {r['parametric_delta']:>+8.4f} {r['parametric_p']:>7.4f}  "
          f"{'✓' if r['passes_dpr'] else '✗':>3s} | "
          f"{r['crps_unconstrained']:>8.4f} {r['crps_excess_unc']:>+8.4f} | "
          f"{r['crps_constrained']:>8.4f} {r['crps_excess_con']:>+8.4f}")



# §9. Pareto-frontier analysis
#
# For each honest method that passes ΔPR, compute the adversary's
# CRPS_excess at matching ΔPR (via interpolation along the adversary
# trajectory). All CRPS values come from the paper's `score_W_against_panel`,
# so the comparison is apples-to-apples.


print("\n" + "=" * 70)
print("PARETO-FRONTIER ANALYSIS (under both CRPS protocols)")
print("=" * 70)

# Sort adversary trajectory by ΔPR
df_adv_for_interp = pd.DataFrame(adversary_results)
df_adv_sorted = df_adv_for_interp.sort_values('parametric_delta').reset_index(drop=True)

print(f"\nAdversary trajectory (sorted by ΔPR):")
print(f"{'λ':>6s} {'ΔPR':>9s} {'CRPS_exc_unc':>14s} {'CRPS_exc_con':>14s}")
for _, r in df_adv_sorted.iterrows():
    print(f"  {r['lambda']:>5.2f} {r['parametric_delta']:>+9.4f} "
          f"{r['crps_excess_unc']:>+14.6f} {r['crps_excess_con']:>+14.6f}")

print(f"\nFor each honest method that passes ΔPR, interpolate adversary's "
      f"CRPS_excess at matching ΔPR (under each protocol):")

defense_summary = []
for r in honest_results:
    if not r['passes_dpr']:
        continue
    if r['method'] == 'ols_var':
        continue

    target_dpr = r['parametric_delta']

    def interp_excess(col):
        """Interpolate adversary CRPS_excess at target_dpr."""
        vals = df_adv_sorted[col].values
        deltas = df_adv_sorted['parametric_delta'].values
        if target_dpr < deltas.min():
            return float(vals[0]), 'extrapolated below'
        elif target_dpr > deltas.max():
            return float(vals[-1]), 'extrapolated above'
        else:
            return float(np.interp(target_dpr, deltas, vals)), 'interpolated'

    adv_exc_unc, note_unc = interp_excess('crps_excess_unc')
    adv_exc_con, note_con = interp_excess('crps_excess_con')

    honest_exc_unc = r['crps_excess_unc']
    honest_exc_con = r['crps_excess_con']

    pays_more_unc = honest_exc_unc > adv_exc_unc
    pays_more_con = honest_exc_con > adv_exc_con

    defense_summary.append({
        'method': r['method'],
        'parametric_delta': r['parametric_delta'],
        # Unconstrained
        'honest_crps_excess_unc': honest_exc_unc,
        'adv_crps_excess_unc_at_match': adv_exc_unc,
        'pays_more_unc': pays_more_unc,
        # Constrained
        'honest_crps_excess_con': honest_exc_con,
        'adv_crps_excess_con_at_match': adv_exc_con,
        'pays_more_con': pays_more_con,
        # Notes
        'interp_note_unc': note_unc,
        'interp_note_con': note_con,
    })

    print(f"\n  {r['method']}: ΔPR={target_dpr:+.4f}")
    print(f"    Unconstrained protocol:")
    print(f"      honest CRPS_excess:    {honest_exc_unc:+.6f}")
    print(f"      adv at matching ΔPR:   {adv_exc_unc:+.6f} ({note_unc})")
    print(f"      Pays more than adv?    {'YES' if pays_more_unc else 'NO'}")
    print(f"    Constrained protocol:")
    print(f"      honest CRPS_excess:    {honest_exc_con:+.6f}")
    print(f"      adv at matching ΔPR:   {adv_exc_con:+.6f} ({note_con})")
    print(f"      Pays more than adv?    {'YES' if pays_more_con else 'NO'}")

df_defense = pd.DataFrame(defense_summary)
defense_csv = os.path.join(OUT_DIR, 'defense_summary.csv')
df_defense.to_csv(defense_csv, index=False)
print(f"\nSaved: {defense_csv}")



# §10. Final verdict


print("\n" + "=" * 70)
print("FINAL VERDICT")
print("=" * 70)

if len(defense_summary) == 0:
    print("\nNo honest methods passed ΔPR; defense check not applicable.")
else:
    n_holds_unc = sum(1 for d in defense_summary if d['pays_more_unc'])
    n_holds_con = sum(1 for d in defense_summary if d['pays_more_con'])
    n_total = len(defense_summary)

    print(f"\nDefense 2 (predictive efficiency):")
    print(f"  Unconstrained protocol: {n_holds_unc}/{n_total} honest methods pay more than adversary at matching ΔPR")
    print(f"  Constrained protocol:   {n_holds_con}/{n_total} honest methods pay more than adversary at matching ΔPR")

    if n_holds_unc == n_total and n_holds_con == n_total:
        print(f"\n⇒ DEFENSE 2 HOLDS UNIVERSALLY under both protocols.")
    elif n_holds_unc == 0 and n_holds_con == 0:
        print(f"\n⇒ DEFENSE 2 FAILS under both protocols. Need a different defense.")
    else:
        print(f"\n⇒ DEFENSE 2 PARTIALLY HOLDS: results vary by method and protocol.")

    # PR_obs ceiling diagnostic — this catches high-λ adversaries
    print(f"\n--- Additional diagnostic: PR_obs ceiling ---")
    honest_passing = [r for r in honest_results if r['passes_dpr'] and r['method'] != 'ols_var']
    if honest_passing:
        honest_pr_max = max(r['PR_obs'] for r in honest_passing)
        honest_pr_min = min(r['PR_obs'] for r in honest_passing)
        print(f"  Honest methods passing ΔPR have PR_obs in [{honest_pr_min:.3f}, {honest_pr_max:.3f}]")
        adv_passing = [r for r in adversary_results if r['passes_dpr'] and r['lambda'] > 0]
        adv_above_honest = [r for r in adv_passing if r['PR_obs'] > honest_pr_max + 0.05]
        adv_within_honest = [r for r in adv_passing if r['PR_obs'] <= honest_pr_max + 0.05]
        print(f"  Adversary configs passing ΔPR: {len(adv_passing)}")
        print(f"    PR_obs > {honest_pr_max + 0.05:.3f} (clearly above honest range): "
              f"{len(adv_above_honest)} configs (λ = {[r['lambda'] for r in adv_above_honest]})")
        print(f"    PR_obs ≤ {honest_pr_max + 0.05:.3f} (mimics honest range):           "
              f"{len(adv_within_honest)} configs (λ = {[r['lambda'] for r in adv_within_honest]})")
        print(f"\n  Interpretation:")
        print(f"  - High-λ adversaries (PR_obs > {honest_pr_max + 0.05:.3f}) are detectable via PR_obs ceiling.")
        print(f"  - Low/mid-λ adversaries can mimic honest methods' (PR_obs, ΔPR, CRPS) jointly.")
        print(f"  - This is a real limitation: framework cannot perfectly distinguish a determined")
        print(f"    rank-1-regularized adversary from honest methods that incidentally produce")
        print(f"    rank-1-like W on data with strong common variance (V-Dem T_cov=0.987).")

print(f"\nDone.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Utilities defined.
Reading paper's CRPS pipeline from: /content/drive/MyDrive/NeurIPS2026_1296/src/random_baselines_4method_corrected.py
  total lines: 1527
  function-definitions block: lines 237-642 (406 lines)
  Successfully sourced CRPS pipeline functions.
  score_W_against_panel signature verified.

Loading V-Dem
V-Dem: shape=(6675, 16), U=89, T=75, ref_block_size=13

Loading method_outputs.pkl
  Loaded: /content/drive/MyDrive/NeurIPS2026_1296/random_baselines_4method_outputs/long_16var/method_outputs.pkl

Top-level keys in pickle: ['config_name', 'method_outputs', 'panel_summary']
  Unwrapped: pickle['method_outputs'] has 5 entries

Method names available: ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears']

Methods with valid W_hat: ['linear_var_granger', 'pcmci', 'cmlp', 'navar', 'dynotears']
  linear_var_granger: shape=(16, 16), nnz=38, max

### Cell §7.2 - σ_Λ sweep extension to {0.50, 0.75, 1.00}

 This cell reproduces the canonical sweep cell's R1-heterogen DGP byte-for-byte. Pre-flight verifies σ_lam=0.10 seed=1000 reproduces existing PR_obs=0.959805 before extending. Differences from v2's R1-vdemlike approach:
   - DGP: additive Gaussian per-unit λ perturbation (not multiplicative log-normal)
   - N=12 indicators (not 15 — no singletons)
   - U=89 panel units (not 139 — matches V-Dem panel post-balance)
   - Linear measurement (no tanh on indicators, no heteroskedastic σ_ε)
   - realized_lam_unit_std = Lam_unit.std(axis=0).mean() — mean across
     indicators of per-unit std
 - Null seed: seed*7+3 (matches canonical, not v2's seed*1000+7)

**Run time:** ~70 minutes on Colab Pro G4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NeurIPS2026_1296'
OUT_DIR   = os.path.join(DRIVE_DIR, 'sigma_lam_sweep_block_7_outputs')
INCR_CSV  = os.path.join(OUT_DIR, 'sigma_lam_sweep_incremental.csv')
AGG_CSV   = os.path.join(OUT_DIR, 'sigma_lam_sweep_aggregate.csv')

# --- Configuration ---
SIGMA_LAM_VALUES_TO_RUN = [0.50, 0.75, 1.00]    # extension only
N_SEEDS                 = 10
SEEDS                   = list(range(1000, 1000 + N_SEEDS))   # match existing convention
B_PERMS                 = 1000
ALPHA                   = 0.05

# Canonical sigma_lam_sweep_block_7 DGP defaults — DO NOT CHANGE
U_UNITS          = 89          # NOT 139 — matches V-Dem balanced panel
T_PER_UNIT       = 75
N_INDICATORS     = 12          # NOT 15 — pure R1, no singletons
RHO              = 0.95
SIGMA_ZETA       = 0.6
TANH_SCALE       = 2.0
IDIOSYNCRATIC_SD = 0.3
BURN_IN          = 50
LAM_LOW, LAM_HIGH = 0.6, 0.9



# CANONICAL R1-HETEROGEN DGP — pasted byte-for-byte from sigma_lam_sweep_block_7
# canonical notebook cell. DO NOT modify.


def generate_R1_heterogen(seed, sigma_lam, U=U_UNITS, T=T_PER_UNIT, N=N_INDICATORS,
                           rho=RHO, sigma_zeta=SIGMA_ZETA, tanh_scale=TANH_SCALE,
                           idiosyncratic_sd=IDIOSYNCRATIC_SD, burn_in=BURN_IN,
                           lam_low=LAM_LOW, lam_high=LAM_HIGH):
    """R1-heterogen: 1-factor reflective with PER-UNIT loading dispersion.

    σ_lam controls how much each unit's loadings deviate from the
    population mean λ_i. σ_lam=0 reduces exactly to R1.
    """
    rng = np.random.default_rng(seed)
    Lam_pop = rng.uniform(lam_low, lam_high, size=N)        # population mean λ_i
    Z_unit = rng.standard_normal(size=(U, N))               # (U, N)
    Lam_unit = Lam_pop[None, :] + sigma_lam * Z_unit        # (U, N)

    unit_paths = []
    for u in range(U):
        lam_u = Lam_unit[u]                                 # this unit's loadings
        eta = 0.0
        for _ in range(burn_in):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
        path = np.zeros((T, N))
        for t in range(T):
            eta = rho * np.tanh(tanh_scale * eta) + rng.normal(0, sigma_zeta)
            path[t] = lam_u * eta + rng.normal(0, idiosyncratic_sd, size=N)
        unit_paths.append(path)

    Y = np.vstack(unit_paths)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    split_spec = [T] * U
    indices = list(range(N))

    return {
        'Y': Y, 'split_spec': split_spec,
        'Lam_pop': Lam_pop, 'Lam_unit': Lam_unit,
        'sigma_lam': sigma_lam, 'seed': seed,
        'reflective_indices': indices,
    }



# Canonical inference helpers (byte-for-byte from sigma_lam_sweep_block_7)


def fit_ols_var_signed(Y, split_spec):
    Yt_l, Yl_l = [], []
    cursor = 0
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            Yt_l.append(unit[1:]); Yl_l.append(unit[:-1])
    Yt, Ylag = np.vstack(Yt_l), np.vstack(Yl_l)
    try:
        A_T = np.linalg.solve(Ylag.T @ Ylag, Ylag.T @ Yt)
    except np.linalg.LinAlgError:
        A_T = np.linalg.pinv(Ylag.T @ Ylag) @ (Ylag.T @ Yt)
    A = A_T.T
    np.fill_diagonal(A, 0.0)
    return A


def offdiag_rank1_fit(W_block, max_iters=50, tol=1e-7):
    W = W_block.astype(float).copy()
    m = W.shape[0]
    mask = (1 - np.eye(m)).astype(bool)
    tss = float(np.sum(W[mask] ** 2))
    if tss <= 0.0:
        return np.nan
    Wz = W.copy(); np.fill_diagonal(Wz, 0.0)
    try:
        U_, S_, Vt_ = np.linalg.svd(Wz, full_matrices=False)
        u = U_[:, 0] * np.sqrt(S_[0]); v = Vt_[0, :] * np.sqrt(S_[0])
    except np.linalg.LinAlgError:
        rng = np.random.default_rng(0)
        u = rng.standard_normal(m); v = rng.standard_normal(m)
    pr = np.inf
    for _ in range(max_iters):
        nu = (W * mask) @ v; du = mask @ (v * v)
        u_new = np.where(du > 1e-12, nu / np.maximum(du, 1e-12), 0.0)
        nv = (W * mask).T @ u_new; dv = mask.T @ (u_new * u_new)
        v_new = np.where(dv > 1e-12, nv / np.maximum(dv, 1e-12), 0.0)
        u, v = u_new, v_new
        r = float(np.sum(((W - np.outer(u, v))[mask]) ** 2))
        if pr < np.inf and abs(pr - r) / max(pr, 1e-12) < tol:
            break
        pr = r
    return max(0.0, min(1.0, 1.0 - r / tss))


def fit_per_indicator_ar1(Y, split_spec):
    cursor = 0; yt_l, yl_l = [], []
    for n_u in split_spec:
        unit = Y[cursor:cursor + n_u]; cursor += n_u
        if n_u >= 2:
            yt_l.append(unit[1:]); yl_l.append(unit[:-1])
    yt = np.vstack(yt_l); yl = np.vstack(yl_l)
    N = yt.shape[1]
    betas = np.zeros(N); sigmas = np.zeros(N)
    for i in range(N):
        b = np.sum(yt[:, i] * yl[:, i]) / np.sum(yl[:, i] ** 2)
        resid = yt[:, i] - b * yl[:, i]
        betas[i] = b; sigmas[i] = resid.std(ddof=1)
    return betas, sigmas


def simulate_ar1_panel_from_params(betas, sigmas, U, T, rng):
    N = len(betas)
    stationary_var = sigmas**2 / (1 - betas**2 + 1e-12)
    stationary_var = np.where(np.abs(betas) < 1, stationary_var, sigmas**2)
    stationary_sd = np.sqrt(stationary_var)
    x = rng.standard_normal((U, N)) * stationary_sd[None, :]
    Y = np.zeros((U, T, N))
    eps = rng.standard_normal((U, T, N))
    for t in range(T):
        x = betas[None, :] * x + eps[:, t, :] * sigmas[None, :]
        Y[:, t, :] = x
    Y = Y.reshape(U * T, N)
    Y = (Y - Y.mean(0)) / (Y.std(0) + 1e-12)
    return Y


def compute_delta_pr_parametric(Y, split_spec, ref_indices, B, seed, U_resample, T_resample):
    """Parametric AR(1) null. Returns (ΔPR, p, PR_obs, null_mu, null_sd)."""
    A = fit_ols_var_signed(Y, split_spec)
    W_block = A[np.ix_(ref_indices, ref_indices)].copy()
    PR_obs = offdiag_rank1_fit(W_block)
    if not np.isfinite(PR_obs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    betas, sigmas = fit_per_indicator_ar1(Y, split_spec)
    rng = np.random.default_rng(seed)
    null_PRs = np.zeros(B)
    for b in range(B):
        Y_sim = simulate_ar1_panel_from_params(betas, sigmas, U_resample, T_resample, rng)
        ss_sim = [T_resample] * U_resample
        A_sim = fit_ols_var_signed(Y_sim, ss_sim)
        W_sim_block = A_sim[np.ix_(ref_indices, ref_indices)].copy()
        null_PRs[b] = offdiag_rank1_fit(W_sim_block)
    null_PRs = null_PRs[np.isfinite(null_PRs)]
    if not len(null_PRs):
        return (np.nan, np.nan, PR_obs, np.nan, np.nan)
    mu = float(null_PRs.mean()); sd = float(null_PRs.std())
    delta = PR_obs - mu
    p = (1 + np.sum(null_PRs >= PR_obs)) / (1 + len(null_PRs))
    return (float(delta), float(p), float(PR_obs), mu, sd)



# Per-(σ, seed) iteration matching canonical cell exactly


def run_one_seed(sigma_lam, seed):
    """Run one (sigma_lam, seed) iteration; return per-row dict matching existing CSV format."""
    regime = generate_R1_heterogen(seed=seed, sigma_lam=sigma_lam)
    Y = regime['Y']
    ss = regime['split_spec']
    ref_idx = regime['reflective_indices']

    # Realized λ-unit std: mean across indicators of per-unit std (canonical formula)
    lam_unit_std = float(regime['Lam_unit'].std(axis=0).mean())

    # Parametric null with canonical seed convention: seed*7+3
    delta, p, pr_obs, null_mu, null_sd = compute_delta_pr_parametric(
        Y, ss, ref_idx, B=B_PERMS, seed=seed * 7 + 3,
        U_resample=U_UNITS, T_resample=T_PER_UNIT
    )

    return {
        'sigma_lam':              sigma_lam,
        'seed':                   seed,
        'PR_obs':                 pr_obs,
        'parametric_delta':       delta,
        'parametric_p':           p,
        'parametric_null_mean':   null_mu,
        'parametric_null_sd':     null_sd,
        'passes':                 bool(np.isfinite(delta) and delta > 0 and p < ALPHA),
        'realized_lam_unit_std':  lam_unit_std,
        'elapsed_sec':            np.nan,
    }



# Pre-flight: verify canonical reproduction at σ_lam=0.10 seed=1000

print("=" * 70)
print("Pre-flight: reproduce existing σ_lam=0.10 seed=1000")
print("Target: PR_obs=0.959805, dPR=+0.6560, null_sd=0.0329")
print("=" * 70)

if os.path.exists(INCR_CSV):
    df_existing = pd.read_csv(INCR_CSV)
    existing_010_seed1000 = df_existing[
        (abs(df_existing['sigma_lam'] - 0.10) < 1e-6) & (df_existing['seed'] == 1000)
    ]
    if len(existing_010_seed1000) > 0:
        ref = existing_010_seed1000.iloc[0]
        print(f"Existing row: PR={ref['PR_obs']:.6f}, dPR={ref['parametric_delta']:+.4f}, "
              f"p={ref['parametric_p']:.4f}, null_sd={ref['parametric_null_sd']:.4f}, "
              f"realized_lam_std={ref['realized_lam_unit_std']:.4f}")
        print()
        print("Reproducing with R1-heterogen DGP...")
        t0 = time.time()
        repro = run_one_seed(0.10, 1000)
        elapsed = time.time() - t0
        print(f"Reproduced:   PR={repro['PR_obs']:.6f}, dPR={repro['parametric_delta']:+.4f}, "
              f"p={repro['parametric_p']:.4f}, null_sd={repro['parametric_null_sd']:.4f}, "
              f"realized_lam_std={repro['realized_lam_unit_std']:.4f}")
        print(f"Single-iteration time: {elapsed:.1f} sec")
        print()
        # Tighter tolerance now: any non-trivial diff means the canonical paste has a bug
        pr_match = abs(repro['PR_obs'] - float(ref['PR_obs'])) < 0.001
        delta_match = abs(repro['parametric_delta'] - float(ref['parametric_delta'])) < 0.005
        sd_match = abs(repro['parametric_null_sd'] - float(ref['parametric_null_sd'])) < 0.005
        lam_match = abs(repro['realized_lam_unit_std'] - float(ref['realized_lam_unit_std'])) < 0.001
        all_match = pr_match and delta_match and sd_match and lam_match
        if all_match:
            print("✓ Canonical reproduction successful — DGP is correct")
        else:
            print(f"✗ Reproduction MISMATCH — DO NOT proceed.")
            print(f"  PR_obs match: {pr_match}")
            print(f"  dPR match: {delta_match}")
            print(f"  null_sd match: {sd_match}")
            print(f"  realized_lam_std match: {lam_match}")
            raise RuntimeError("Pre-flight verification failed")
        print()
        total_iters = len(SIGMA_LAM_VALUES_TO_RUN) * N_SEEDS
        est_total_sec = elapsed * total_iters
        print(f"Estimated total runtime: {est_total_sec:.0f} sec = {est_total_sec/60:.1f} min")
        print(f"  ({total_iters} iterations × {elapsed:.1f} sec/iteration)")
    else:
        print("⚠ No existing row at σ_lam=0.10 seed=1000 to verify against.")
else:
    print(f"⚠ Incremental CSV not found at {INCR_CSV}. Skipping pre-flight.")



# Pre-sweep state check

print()
print("=" * 70)
print("Pre-sweep state check")
print("=" * 70)
if os.path.exists(INCR_CSV):
    existing_sigmas = sorted(df_existing['sigma_lam'].unique())
    print(f"Existing σ_lam values: {existing_sigmas}")
    overlap = [s for s in SIGMA_LAM_VALUES_TO_RUN if s in existing_sigmas]
    if overlap:
        print(f"⚠ COLLISION: σ_lam {overlap} already exist; their rows will be REPLACED")
    else:
        print(f"No collision; σ_lam {SIGMA_LAM_VALUES_TO_RUN} are new")
else:
    print(f"No existing incremental file; will create fresh")



# Sweep

print()
print("=" * 70)
print(f"σ_Λ extension sweep: σ_Λ ∈ {SIGMA_LAM_VALUES_TO_RUN}")
print(f"  {N_SEEDS} seeds (1000-{1000+N_SEEDS-1}), B={B_PERMS}, α={ALPHA}")
print("=" * 70)

new_rows = []
overall_t0 = time.time()
for sigma_lam in SIGMA_LAM_VALUES_TO_RUN:
    print(f"\nσ_Λ = {sigma_lam:.2f} ...", flush=True)
    sigma_t0 = time.time()
    for seed in SEEDS:
        seed_t0 = time.time()
        row = run_one_seed(sigma_lam, seed)
        row['elapsed_sec'] = time.time() - seed_t0
        new_rows.append(row)
        print(f"  σ={sigma_lam:.2f} seed={seed}: PR={row['PR_obs']:.4f}, "
              f"ΔPR={row['parametric_delta']:+.4f}, p={row['parametric_p']:.4f}, "
              f"passes={row['passes']}  [{row['elapsed_sec']:.0f}s]", flush=True)
    print(f"  → σ_Λ={sigma_lam:.2f} done in {time.time()-sigma_t0:.0f}s "
          f"(total: {(time.time()-overall_t0)/60:.1f} min)", flush=True)



# Append + recompute aggregate

if os.path.exists(INCR_CSV):
    df_existing = pd.read_csv(INCR_CSV)
else:
    df_existing = pd.DataFrame()
df_new = pd.DataFrame(new_rows)
if len(df_existing) > 0:
    n_dropped = int(df_existing['sigma_lam'].isin(df_new['sigma_lam'].unique()).sum())
    df_existing = df_existing[~df_existing['sigma_lam'].isin(df_new['sigma_lam'].unique())]
    print(f"\nDedup dropped {n_dropped} rows at σ_lam ∈ {SIGMA_LAM_VALUES_TO_RUN}")
df_full = pd.concat([df_existing, df_new], ignore_index=True)
df_full = df_full.sort_values(['sigma_lam', 'seed']).reset_index(drop=True)
df_full.to_csv(INCR_CSV, index=False)
print(f"Wrote {INCR_CSV}: {len(df_full)} rows")

agg_rows = []
for sl, grp in df_full.groupby('sigma_lam'):
    valid = grp[grp['parametric_delta'].notna()]
    if len(valid) == 0:
        continue
    agg_rows.append({
        'sigma_lam':                    sl,
        'n_seeds':                      len(grp),
        'PR_obs_median':                valid['PR_obs'].median(),
        'PR_obs_p25':                   valid['PR_obs'].quantile(0.25),
        'PR_obs_p75':                   valid['PR_obs'].quantile(0.75),
        'delta_median':                 valid['parametric_delta'].median(),
        'delta_p25':                    valid['parametric_delta'].quantile(0.25),
        'delta_p75':                    valid['parametric_delta'].quantile(0.75),
        'p_median':                     valid['parametric_p'].median(),
        'p_p25':                        valid['parametric_p'].quantile(0.25),
        'p_p75':                        valid['parametric_p'].quantile(0.75),
        'pass_rate':                    grp['passes'].mean(),
        'realized_lam_unit_std_median': valid['realized_lam_unit_std'].median(),
    })
df_agg = pd.DataFrame(agg_rows).sort_values('sigma_lam').reset_index(drop=True)
df_agg.to_csv(AGG_CSV, index=False)
print(f"Wrote {AGG_CSV}")
print()
print("Final aggregate:")
print(df_agg[['sigma_lam','n_seeds','PR_obs_median','delta_median','p_median','pass_rate']].to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pre-flight: reproduce existing σ_lam=0.10 seed=1000
Target: PR_obs=0.959805, dPR=+0.6560, null_sd=0.0329
Existing row: PR=0.959805, dPR=+0.6560, p=0.0010, null_sd=0.0329, realized_lam_std=0.0951

Reproducing with R1-heterogen DGP...
Reproduced:   PR=0.959805, dPR=+0.6560, p=0.0010, null_sd=0.0329, realized_lam_std=0.0951
Single-iteration time: 2.1 sec

✓ Canonical reproduction successful — DGP is correct

Estimated total runtime: 62 sec = 1.0 min
  (30 iterations × 2.1 sec/iteration)

Pre-sweep state check
Existing σ_lam values: [np.float64(0.0), np.float64(0.05), np.float64(0.1), np.float64(0.15), np.float64(0.2), np.float64(0.25)]
No collision; σ_lam [0.5, 0.75, 1.0] are new

σ_Λ extension sweep: σ_Λ ∈ [0.5, 0.75, 1.0]
  10 seeds (1000-1009), B=1000, α=0.05

σ_Λ = 0.50 ...
  σ=0.50 seed=1000: PR=0.5657, ΔPR=+0.2617, p=0.0010, passes=True  [2s]
  σ=0.50 seed